In [ ]:
# ============================================================
# STEP 1: Mount Google Drive and create permanent project folders
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import json
import datetime

# Permanent project location in Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

# Permanent folder structure
DIRS = {
    "code": PROJECT_DIR / "00_code",
    "data_original": PROJECT_DIR / "01_data_original",
    "data_processed": PROJECT_DIR / "02_data_processed",
    "embeddings": PROJECT_DIR / "03_embeddings",
    "models": PROJECT_DIR / "04_models",
    "predictions": PROJECT_DIR / "05_predictions",
    "xai": PROJECT_DIR / "06_xai",
    "results": PROJECT_DIR / "07_results",
    "checkpoints": PROJECT_DIR / "08_checkpoints",
    "logs": PROJECT_DIR / "09_logs",
}

# Create every folder
for name, folder in DIRS.items():
    folder.mkdir(parents=True, exist_ok=True)

# Subfolders for each protein language model
MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5"
]

for model_name in MODEL_NAMES:
    (DIRS["embeddings"] / model_name).mkdir(parents=True, exist_ok=True)
    (DIRS["models"] / model_name).mkdir(parents=True, exist_ok=True)
    (DIRS["predictions"] / model_name).mkdir(parents=True, exist_ok=True)
    (DIRS["xai"] / model_name).mkdir(parents=True, exist_ok=True)

# Results subfolders
RESULT_SUBDIRS = [
    "figures_main",
    "figures_SI",
    "tables_main",
    "tables_SI",
    "reports"
]

for subdir in RESULT_SUBDIRS:
    (DIRS["results"] / subdir).mkdir(parents=True, exist_ok=True)

# Permanent pipeline status file
STATUS_FILE = DIRS["checkpoints"] / "pipeline_status.json"

if STATUS_FILE.exists():
    with open(STATUS_FILE, "r", encoding="utf-8") as handle:
        PIPELINE_STATUS = json.load(handle)
else:
    PIPELINE_STATUS = {}

def save_pipeline_status():
    """Safely save pipeline progress to Google Drive."""
    temporary_file = STATUS_FILE.with_suffix(".tmp")

    with open(temporary_file, "w", encoding="utf-8") as handle:
        json.dump(PIPELINE_STATUS, handle, indent=2)

    os.replace(temporary_file, STATUS_FILE)

def mark_step_complete(step_name, output_files=None, details=None):
    """Record that a pipeline step has finished."""
    PIPELINE_STATUS[step_name] = {
        "completed": True,
        "completed_at": datetime.datetime.now().isoformat(),
        "output_files": [
            str(path) for path in (output_files or [])
        ],
        "details": details or {}
    }

    save_pipeline_status()
    print(f"[SAVED] {step_name}")

def step_is_complete(step_name, required_files=None):
    """Check whether a step and its expected outputs already exist."""
    record = PIPELINE_STATUS.get(step_name, {})

    if not record.get("completed", False):
        return False

    if required_files:
        return all(Path(path).exists() for path in required_files)

    return True

# Save this setup step
mark_step_complete(
    "01_project_directory_setup",
    output_files=[STATUS_FILE],
    details={
        "project_directory": str(PROJECT_DIR),
        "models": MODEL_NAMES
    }
)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nCreated folders:")
for name, folder in DIRS.items():
    print(f"{name:16s}: {folder}")

print("\nPreviously completed steps:")
for step in PIPELINE_STATUS:
    print(" -", step)

In [ ]:
# ============================================================
# STEP 2: Check GPU, CUDA, Python, and PyTorch
# ============================================================

import sys
import platform
import subprocess
import torch

print("Python version :", sys.version.split()[0])
print("Platform       :", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name       :", torch.cuda.get_device_name(0))
    print("CUDA version   :", torch.version.cuda)

    gpu_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )
    print(f"GPU memory     : {gpu_memory_gb:.2f} GB")
else:
    print("\nWARNING: GPU is not enabled.")
    print("Go to Runtime → Change runtime type → T4 GPU.")

print("\nNVIDIA GPU details:")
subprocess.run(["nvidia-smi"])

In [ ]:
# ============================================================
# STEP 3: Install packages required for the pipeline
# ============================================================

import subprocess
import sys

packages = [
    "fair-esm==2.0.0",
    "transformers",
    "sentencepiece",
    "accelerate",
    "captum",
    "biopython",
    "openpyxl",
    "pandas",
    "scikit-learn",
    "scipy",
    "umap-learn",
    "matplotlib",
    "seaborn",
    "logomaker",
    "tqdm",
    "joblib"
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade"
] + packages

print("Installing required packages...")
subprocess.check_call(command)

print("\nInstallation completed successfully.")

In [ ]:
# ============================================================
# STEP 3B: Test package imports and save environment versions
# ============================================================

import sys
import json
import datetime
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import sklearn
import scipy
import torch
import transformers
import esm
import captum
import Bio
import umap
import matplotlib
import seaborn
import logomaker
import sentencepiece
import accelerate
import openpyxl
import joblib

packages_to_record = [
    "torch",
    "fair-esm",
    "transformers",
    "sentencepiece",
    "accelerate",
    "captum",
    "biopython",
    "openpyxl",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "umap-learn",
    "matplotlib",
    "seaborn",
    "logomaker",
    "tqdm",
    "joblib"
]

environment_info = {
    "recorded_at": datetime.datetime.now().isoformat(),
    "python_version": sys.version,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "packages": {}
}

for package in packages_to_record:
    try:
        environment_info["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        environment_info["packages"][package] = "NOT FOUND"

environment_file = (
    DIRS["checkpoints"] / "environment_versions.json"
)

with open(environment_file, "w", encoding="utf-8") as handle:
    json.dump(environment_info, handle, indent=2)

mark_step_complete(
    "02_environment_setup",
    output_files=[environment_file],
    details={
        "python_version": sys.version.split()[0],
        "pytorch_version": torch.__version__,
        "gpu": environment_info["gpu_name"]
    }
)

print("All required packages imported successfully.\n")

print("Important versions:")
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("fair-esm     :", metadata.version("fair-esm"))
print("Captum       :", captum.__version__)
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("GPU          :", environment_info["gpu_name"])

print("\nEnvironment record saved to:")
print(environment_file)

In [ ]:
# ============================================================
# STEP 4: Upload both datasets one by one and save permanently
# ============================================================

from google.colab import files
from pathlib import Path
import shutil
import pandas as pd

# Permanent Google Drive folder already created in Step 1
DATA_DIR = DIRS["data_original"]

# ------------------------------------------------------------
# Upload internal dataset
# ------------------------------------------------------------

print("=" * 70)
print("UPLOAD FILE 1")
print("Select: Final_non_redundant_sequences.xlsx")
print("=" * 70)

uploaded_internal = files.upload()

if len(uploaded_internal) == 0:
    raise RuntimeError("No internal dataset was selected.")

internal_uploaded_name = list(uploaded_internal.keys())[0]
internal_source = Path("/content") / internal_uploaded_name
internal_destination = DATA_DIR / "Final_non_redundant_sequences.xlsx"

shutil.copy2(internal_source, internal_destination)

print("\nSaved internal dataset permanently:")
print(internal_destination)


# ------------------------------------------------------------
# Upload KELM external dataset
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UPLOAD FILE 2")
print("Select: kelm_dataset.csv")
print("=" * 70)

uploaded_kelm = files.upload()

if len(uploaded_kelm) == 0:
    raise RuntimeError("No KELM dataset was selected.")

kelm_uploaded_name = list(uploaded_kelm.keys())[0]
kelm_source = Path("/content") / kelm_uploaded_name

# Preserve CSV or Excel format automatically
kelm_suffix = Path(kelm_uploaded_name).suffix.lower()

if kelm_suffix == ".csv":
    kelm_destination = DATA_DIR / "kelm_dataset.csv"
elif kelm_suffix in [".xlsx", ".xls"]:
    kelm_destination = DATA_DIR / "kelm_dataset.xlsx"
else:
    raise ValueError(
        f"Unsupported KELM file format: {kelm_suffix}. "
        "Please upload a CSV or Excel file."
    )

shutil.copy2(kelm_source, kelm_destination)

print("\nSaved KELM dataset permanently:")
print(kelm_destination)


# ------------------------------------------------------------
# Verify that both permanent files exist
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PERMANENT DATA FILES")
print("=" * 70)

for file_path in sorted(DATA_DIR.iterdir()):
    if file_path.is_file():
        size_kb = file_path.stat().st_size / 1024
        print(f"{file_path.name:45s} {size_kb:10.2f} KB")


# ------------------------------------------------------------
# Read both datasets
# ------------------------------------------------------------

internal_raw = pd.read_excel(internal_destination)

if kelm_destination.suffix.lower() == ".csv":
    kelm_raw = pd.read_csv(kelm_destination)
else:
    kelm_raw = pd.read_excel(kelm_destination)


# ------------------------------------------------------------
# Display dataset information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INTERNAL DATASET")
print("=" * 70)
print("File   :", internal_destination.name)
print("Rows   :", internal_raw.shape[0])
print("Columns:", internal_raw.shape[1])
print("Column names:", internal_raw.columns.tolist())

display(internal_raw.head())


print("\n" + "=" * 70)
print("KELM EXTERNAL DATASET")
print("=" * 70)
print("File   :", kelm_destination.name)
print("Rows   :", kelm_raw.shape[0])
print("Columns:", kelm_raw.shape[1])
print("Column names:", kelm_raw.columns.tolist())

display(kelm_raw.head())


# ------------------------------------------------------------
# Record completion
# ------------------------------------------------------------

mark_step_complete(
    "03_original_datasets_uploaded",
    output_files=[
        internal_destination,
        kelm_destination
    ],
    details={
        "internal_shape": list(internal_raw.shape),
        "kelm_shape": list(kelm_raw.shape),
        "internal_columns": internal_raw.columns.tolist(),
        "kelm_columns": kelm_raw.columns.tolist()
    }
)

print("\nBoth datasets are now saved permanently in Google Drive.")
print("You will not need to upload them again tomorrow.")

In [ ]:
# ============================================================
# Remove old datasets (we won't use them anymore)
# ============================================================

from pathlib import Path

files_to_delete = [
    DIRS["data_original"] / "Final_non_redundant_sequences.xlsx",
    DIRS["data_original"] / "kelm_dataset.csv",
]

for f in files_to_delete:
    if f.exists():
        f.unlink()
        print("Deleted:", f.name)

print("\nRemaining files:")

for f in sorted(DIRS["data_original"].iterdir()):
    print("✓", f.name)

In [ ]:
# ============================================================
# STEP 4D: Correctly merge CPP and non-CPP datasets
# ============================================================

from google.colab import files
from pathlib import Path
import pandas as pd
import shutil

DATA_DIR = DIRS["data_original"]
TEMP_DIR = Path("/content")

required_files = {
    "internal_cpp": "pLM4CPPs_dataset_CPP.xlsx",
    "internal_noncpp": "pLM4CPPs_dataset_Non-CPP.xlsx",
    "kelm_cpp": "kelm_dataset_CPP.csv",
    "kelm_noncpp": "kelm_dataset_Non-CPP.csv"
}

# ------------------------------------------------------------
# Locate files already present in /content or permanent Drive
# ------------------------------------------------------------

located_files = {}

for key, filename in required_files.items():
    content_path = TEMP_DIR / filename
    drive_path = DATA_DIR / filename

    if content_path.exists():
        located_files[key] = content_path
    elif drive_path.exists():
        located_files[key] = drive_path


# ------------------------------------------------------------
# Upload any files that are missing
# ------------------------------------------------------------

missing_keys = [
    key for key in required_files
    if key not in located_files
]

if missing_keys:
    print("The following files are missing:")
    for key in missing_keys:
        print(" -", required_files[key])

    print("\nPlease upload all missing files in the upload window.")
    uploaded = files.upload()

    for uploaded_name in uploaded.keys():
        uploaded_path = TEMP_DIR / uploaded_name

        for key, expected_name in required_files.items():
            if uploaded_name == expected_name:
                located_files[key] = uploaded_path

# Verify again
still_missing = [
    required_files[key]
    for key in required_files
    if key not in located_files
]

if still_missing:
    raise FileNotFoundError(
        "These required files are still missing:\n"
        + "\n".join(still_missing)
    )


# ------------------------------------------------------------
# Copy all four original files permanently to Google Drive
# ------------------------------------------------------------

for key, source_path in located_files.items():
    destination_path = DATA_DIR / required_files[key]

    if source_path.resolve() != destination_path.resolve():
        shutil.copy2(source_path, destination_path)

    located_files[key] = destination_path

    print(f"[SAVED ORIGINAL] {destination_path.name}")


# ------------------------------------------------------------
# Read the internal CPP and non-CPP datasets
# ------------------------------------------------------------

internal_cpp = pd.read_excel(
    located_files["internal_cpp"]
)

internal_noncpp = pd.read_excel(
    located_files["internal_noncpp"]
)

print("\nInternal CPP columns:", internal_cpp.columns.tolist())
print("Internal non-CPP columns:", internal_noncpp.columns.tolist())


# ------------------------------------------------------------
# Standardize internal datasets
# ------------------------------------------------------------

def find_sequence_column(df):
    possible_names = [
        "sequence",
        "Sequence",
        "seq",
        "Seq",
        "peptide",
        "Peptide"
    ]

    for name in possible_names:
        if name in df.columns:
            return name

    raise ValueError(
        f"No sequence column found. Available columns: {df.columns.tolist()}"
    )


internal_cpp_seq_col = find_sequence_column(internal_cpp)
internal_noncpp_seq_col = find_sequence_column(internal_noncpp)

internal_cpp_clean = pd.DataFrame({
    "sequence": internal_cpp[internal_cpp_seq_col].astype(str),
    "label": 1
})

internal_noncpp_clean = pd.DataFrame({
    "sequence": internal_noncpp[internal_noncpp_seq_col].astype(str),
    "label": 0
})

internal_combined = pd.concat(
    [internal_cpp_clean, internal_noncpp_clean],
    ignore_index=True
)


# ------------------------------------------------------------
# Read and standardize KELM datasets
# ------------------------------------------------------------

kelm_cpp = pd.read_csv(
    located_files["kelm_cpp"]
)

kelm_noncpp = pd.read_csv(
    located_files["kelm_noncpp"]
)

print("\nKELM CPP columns:", kelm_cpp.columns.tolist())
print("KELM non-CPP columns:", kelm_noncpp.columns.tolist())

kelm_cpp_seq_col = find_sequence_column(kelm_cpp)
kelm_noncpp_seq_col = find_sequence_column(kelm_noncpp)

kelm_cpp_clean = pd.DataFrame({
    "sequence": kelm_cpp[kelm_cpp_seq_col].astype(str),
    "label": 1
})

kelm_noncpp_clean = pd.DataFrame({
    "sequence": kelm_noncpp[kelm_noncpp_seq_col].astype(str),
    "label": 0
})

kelm_combined = pd.concat(
    [kelm_cpp_clean, kelm_noncpp_clean],
    ignore_index=True
)


# ------------------------------------------------------------
# Basic sequence cleanup
# ------------------------------------------------------------

def basic_clean(df):
    cleaned = df.copy()

    cleaned["sequence"] = (
        cleaned["sequence"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(" ", "", regex=False)
    )

    cleaned = cleaned[
        cleaned["sequence"].notna()
        & cleaned["sequence"].ne("")
        & cleaned["sequence"].ne("NAN")
    ].copy()

    return cleaned.reset_index(drop=True)


internal_combined = basic_clean(internal_combined)
kelm_combined = basic_clean(kelm_combined)


# ------------------------------------------------------------
# Save canonical combined datasets
# ------------------------------------------------------------

internal_final_file = (
    DATA_DIR / "Final_non_redundant_sequences.xlsx"
)

kelm_final_file = (
    DATA_DIR / "KELM_external_dataset.csv"
)

internal_combined.to_excel(
    internal_final_file,
    index=False
)

kelm_combined.to_csv(
    kelm_final_file,
    index=False
)


# ------------------------------------------------------------
# Display final counts
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL INTERNAL DATASET")
print("=" * 70)

print("Rows:", len(internal_combined))
print("\nClass counts:")
print(internal_combined["label"].value_counts().sort_index())

display(internal_combined.head())


print("\n" + "=" * 70)
print("FINAL KELM EXTERNAL DATASET")
print("=" * 70)

print("Rows:", len(kelm_combined))
print("\nClass counts:")
print(kelm_combined["label"].value_counts().sort_index())

display(kelm_combined.head())


# ------------------------------------------------------------
# Save corrected progress
# ------------------------------------------------------------

mark_step_complete(
    "03_correct_combined_datasets",
    output_files=[
        internal_final_file,
        kelm_final_file
    ],
    details={
        "internal_rows": len(internal_combined),
        "internal_class_counts": (
            internal_combined["label"]
            .value_counts()
            .sort_index()
            .to_dict()
        ),
        "kelm_rows": len(kelm_combined),
        "kelm_class_counts": (
            kelm_combined["label"]
            .value_counts()
            .sort_index()
            .to_dict()
        )
    }
)

print("\nCorrect combined datasets saved permanently:")
print(internal_final_file)
print(kelm_final_file)

In [ ]:
# ============================================================
# STEP 4D: Preserve four original files and create processed datasets
# ============================================================

from google.colab import files
from pathlib import Path
import pandas as pd
import shutil
import re

ORIGINAL_DIR = DIRS["data_original"]
PROCESSED_DIR = DIRS["data_processed"]
TEMP_DIR = Path("/content")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = {
    "internal_cpp": "pLM4CPPs_dataset_CPP.xlsx",
    "internal_noncpp": "pLM4CPPs_dataset_Non-CPP.xlsx",
    "kelm_cpp": "kelm_dataset_CPP.csv",
    "kelm_noncpp": "kelm_dataset_Non-CPP.csv",
}

# ------------------------------------------------------------
# 1. Remove only previously created incorrect combined files
# ------------------------------------------------------------

incorrect_files = [
    ORIGINAL_DIR / "Final_non_redundant_sequences.xlsx",
    ORIGINAL_DIR / "kelm_dataset.csv",
    ORIGINAL_DIR / "KELM_external_dataset.csv",
]

for file_path in incorrect_files:
    if file_path.exists():
        file_path.unlink()
        print("[DELETED INCORRECT FILE]", file_path.name)

# ------------------------------------------------------------
# 2. Locate the four original files
# ------------------------------------------------------------

located_files = {}

for key, filename in required_files.items():
    drive_path = ORIGINAL_DIR / filename
    content_path = TEMP_DIR / filename

    if drive_path.exists():
        located_files[key] = drive_path
    elif content_path.exists():
        located_files[key] = content_path

missing_keys = [
    key for key in required_files
    if key not in located_files
]

# ------------------------------------------------------------
# 3. Upload missing originals if necessary
# ------------------------------------------------------------

if missing_keys:
    print("\nPlease upload these missing files:")

    for key in missing_keys:
        print(" -", required_files[key])

    uploaded = files.upload()

    for uploaded_name in uploaded:
        uploaded_path = TEMP_DIR / uploaded_name

        for key, expected_name in required_files.items():
            if uploaded_name == expected_name:
                located_files[key] = uploaded_path

still_missing = [
    required_files[key]
    for key in required_files
    if key not in located_files
]

if still_missing:
    raise FileNotFoundError(
        "These files are still missing:\n"
        + "\n".join(still_missing)
    )

# ------------------------------------------------------------
# 4. Save the four untouched originals permanently
# ------------------------------------------------------------

for key, source_path in located_files.items():
    destination_path = ORIGINAL_DIR / required_files[key]

    if source_path.resolve() != destination_path.resolve():
        shutil.copy2(source_path, destination_path)

    located_files[key] = destination_path
    print("[ORIGINAL SAVED]", destination_path.name)

# ------------------------------------------------------------
# 5. Functions for reading and cleaning
# ------------------------------------------------------------

def find_sequence_column(df):
    normalized_columns = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    possible_names = [
        "sequence",
        "seq",
        "peptide",
        "peptide sequence",
        "protein sequence",
    ]

    for name in possible_names:
        if name in normalized_columns:
            return normalized_columns[name]

    raise ValueError(
        "Sequence column not found. Available columns: "
        f"{df.columns.tolist()}"
    )


def clean_sequences(df, label):
    sequence_column = find_sequence_column(df)

    cleaned = pd.DataFrame({
        "sequence": df[sequence_column],
        "label": label,
    })

    cleaned["sequence"] = (
        cleaned["sequence"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
    )

    cleaned = cleaned[
        cleaned["sequence"].notna()
        & cleaned["sequence"].ne("")
        & cleaned["sequence"].ne("NAN")
    ].copy()

    # Retain only the 20 standard amino-acid letters
    standard_pattern = re.compile(r"^[ACDEFGHIKLMNPQRSTVWY]+$")

    cleaned["valid_standard_sequence"] = (
        cleaned["sequence"]
        .apply(lambda sequence: bool(standard_pattern.fullmatch(sequence)))
    )

    invalid = cleaned[
        ~cleaned["valid_standard_sequence"]
    ].copy()

    cleaned = cleaned[
        cleaned["valid_standard_sequence"]
    ].drop(columns="valid_standard_sequence")

    cleaned = cleaned.drop_duplicates(
        subset=["sequence", "label"]
    ).reset_index(drop=True)

    return cleaned, invalid

# ------------------------------------------------------------
# 6. Read the four original files
# ------------------------------------------------------------

internal_cpp_raw = pd.read_excel(
    located_files["internal_cpp"]
)

internal_noncpp_raw = pd.read_excel(
    located_files["internal_noncpp"]
)

kelm_cpp_raw = pd.read_csv(
    located_files["kelm_cpp"]
)

kelm_noncpp_raw = pd.read_csv(
    located_files["kelm_noncpp"]
)

print("\nOriginal columns:")
print("Internal CPP    :", internal_cpp_raw.columns.tolist())
print("Internal non-CPP:", internal_noncpp_raw.columns.tolist())
print("KELM CPP        :", kelm_cpp_raw.columns.tolist())
print("KELM non-CPP    :", kelm_noncpp_raw.columns.tolist())

# ------------------------------------------------------------
# 7. Clean each class separately
# ------------------------------------------------------------

internal_cpp, internal_cpp_invalid = clean_sequences(
    internal_cpp_raw,
    label=1,
)

internal_noncpp, internal_noncpp_invalid = clean_sequences(
    internal_noncpp_raw,
    label=0,
)

kelm_cpp, kelm_cpp_invalid = clean_sequences(
    kelm_cpp_raw,
    label=1,
)

kelm_noncpp, kelm_noncpp_invalid = clean_sequences(
    kelm_noncpp_raw,
    label=0,
)

# ------------------------------------------------------------
# 8. Merge CPP and non-CPP datasets
# ------------------------------------------------------------

internal_processed = pd.concat(
    [internal_cpp, internal_noncpp],
    ignore_index=True,
)

kelm_processed = pd.concat(
    [kelm_cpp, kelm_noncpp],
    ignore_index=True,
)

# Add stable sequence IDs
internal_processed.insert(
    0,
    "sequence_id",
    [
        f"INT_{index:05d}"
        for index in range(1, len(internal_processed) + 1)
    ],
)

kelm_processed.insert(
    0,
    "sequence_id",
    [
        f"KELM_{index:04d}"
        for index in range(1, len(kelm_processed) + 1)
    ],
)

# Add sequence length
internal_processed["length"] = (
    internal_processed["sequence"].str.len()
)

kelm_processed["length"] = (
    kelm_processed["sequence"].str.len()
)

# ------------------------------------------------------------
# 9. Check conflicting labels within each dataset
# ------------------------------------------------------------

internal_conflicts = (
    internal_processed
    .groupby("sequence")["label"]
    .nunique()
)

internal_conflicting_sequences = internal_conflicts[
    internal_conflicts > 1
].index.tolist()

kelm_conflicts = (
    kelm_processed
    .groupby("sequence")["label"]
    .nunique()
)

kelm_conflicting_sequences = kelm_conflicts[
    kelm_conflicts > 1
].index.tolist()

if internal_conflicting_sequences:
    print(
        "\n[WARNING] Internal sequences with conflicting labels:",
        len(internal_conflicting_sequences),
    )

if kelm_conflicting_sequences:
    print(
        "[WARNING] KELM sequences with conflicting labels:",
        len(kelm_conflicting_sequences),
    )

# Do not silently remove conflicting sequences yet.
# Save them for inspection.
internal_conflict_file = (
    PROCESSED_DIR / "internal_conflicting_labels.csv"
)

kelm_conflict_file = (
    PROCESSED_DIR / "kelm_conflicting_labels.csv"
)

internal_processed[
    internal_processed["sequence"].isin(
        internal_conflicting_sequences
    )
].to_csv(internal_conflict_file, index=False)

kelm_processed[
    kelm_processed["sequence"].isin(
        kelm_conflicting_sequences
    )
].to_csv(kelm_conflict_file, index=False)

# ------------------------------------------------------------
# 10. Save processed datasets
# ------------------------------------------------------------

internal_processed_file = (
    PROCESSED_DIR / "internal_dataset_cleaned.csv"
)

kelm_processed_file = (
    PROCESSED_DIR / "kelm_external_dataset_cleaned.csv"
)

internal_processed.to_csv(
    internal_processed_file,
    index=False,
)

kelm_processed.to_csv(
    kelm_processed_file,
    index=False,
)

# Save rejected non-standard sequences
invalid_sequences = pd.concat(
    [
        internal_cpp_invalid.assign(source="internal_CPP"),
        internal_noncpp_invalid.assign(source="internal_nonCPP"),
        kelm_cpp_invalid.assign(source="KELM_CPP"),
        kelm_noncpp_invalid.assign(source="KELM_nonCPP"),
    ],
    ignore_index=True,
)

invalid_file = (
    PROCESSED_DIR / "rejected_nonstandard_sequences.csv"
)

invalid_sequences.to_csv(
    invalid_file,
    index=False,
)

# ------------------------------------------------------------
# 11. Display final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INTERNAL PROCESSED DATASET")
print("=" * 70)

print("Rows:", len(internal_processed))
print("Class counts:")
print(
    internal_processed["label"]
    .value_counts()
    .sort_index()
)
print(
    "Length range:",
    internal_processed["length"].min(),
    "to",
    internal_processed["length"].max(),
)

display(internal_processed.head())

print("\n" + "=" * 70)
print("KELM PROCESSED DATASET")
print("=" * 70)

print("Rows:", len(kelm_processed))
print("Class counts:")
print(
    kelm_processed["label"]
    .value_counts()
    .sort_index()
)
print(
    "Length range:",
    kelm_processed["length"].min(),
    "to",
    kelm_processed["length"].max(),
)

display(kelm_processed.head())

print("\nRejected non-standard sequences:", len(invalid_sequences))

# ------------------------------------------------------------
# 12. Record completion
# ------------------------------------------------------------

mark_step_complete(
    "03_correct_combined_datasets",
    output_files=[
        internal_processed_file,
        kelm_processed_file,
        invalid_file,
        internal_conflict_file,
        kelm_conflict_file,
    ],
    details={
        "internal_rows": int(len(internal_processed)),
        "internal_class_counts": {
            str(key): int(value)
            for key, value in (
                internal_processed["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "kelm_rows": int(len(kelm_processed)),
        "kelm_class_counts": {
            str(key): int(value)
            for key, value in (
                kelm_processed["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "rejected_nonstandard_sequences": int(
            len(invalid_sequences)
        ),
        "internal_conflicting_sequences": int(
            len(internal_conflicting_sequences)
        ),
        "kelm_conflicting_sequences": int(
            len(kelm_conflicting_sequences)
        ),
    },
)

print("\nSaved processed internal dataset:")
print(internal_processed_file)

print("\nSaved processed KELM dataset:")
print(kelm_processed_file)

print("\nOriginal files preserved in:")
print(ORIGINAL_DIR)

In [ ]:
# ============================================================
# STEP 5: Dataset QC, duplicates, conflicts, and KELM leakage
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

PROCESSED_DIR = DIRS["data_processed"]
CHECKPOINT_DIR = DIRS["checkpoints"]

internal_file = (
    PROCESSED_DIR / "internal_dataset_cleaned.csv"
)

kelm_file = (
    PROCESSED_DIR / "kelm_external_dataset_cleaned.csv"
)

rejected_file = (
    PROCESSED_DIR / "rejected_nonstandard_sequences.csv"
)

# ------------------------------------------------------------
# 1. Load processed datasets
# ------------------------------------------------------------

internal_df = pd.read_csv(internal_file)
kelm_df = pd.read_csv(kelm_file)

if rejected_file.exists():
    rejected_df = pd.read_csv(rejected_file)
else:
    rejected_df = pd.DataFrame()

print("=" * 70)
print("REJECTED NON-STANDARD SEQUENCES")
print("=" * 70)

print("Rejected rows:", len(rejected_df))

if not rejected_df.empty:
    display(rejected_df)

# ------------------------------------------------------------
# 2. Check exact duplicate rows
# ------------------------------------------------------------

internal_exact_duplicates = internal_df.duplicated(
    subset=["sequence", "label"],
    keep=False,
)

kelm_exact_duplicates = kelm_df.duplicated(
    subset=["sequence", "label"],
    keep=False,
)

print("\n" + "=" * 70)
print("EXACT DUPLICATE CHECK")
print("=" * 70)

print(
    "Internal duplicate rows:",
    int(internal_exact_duplicates.sum())
)

print(
    "KELM duplicate rows:",
    int(kelm_exact_duplicates.sum())
)

# ------------------------------------------------------------
# 3. Check conflicting labels
# ------------------------------------------------------------

internal_label_counts = (
    internal_df
    .groupby("sequence")["label"]
    .nunique()
)

kelm_label_counts = (
    kelm_df
    .groupby("sequence")["label"]
    .nunique()
)

internal_conflict_sequences = (
    internal_label_counts[
        internal_label_counts > 1
    ]
    .index
    .tolist()
)

kelm_conflict_sequences = (
    kelm_label_counts[
        kelm_label_counts > 1
    ]
    .index
    .tolist()
)

print("\n" + "=" * 70)
print("CONFLICTING LABEL CHECK")
print("=" * 70)

print(
    "Internal sequences with conflicting labels:",
    len(internal_conflict_sequences)
)

print(
    "KELM sequences with conflicting labels:",
    len(kelm_conflict_sequences)
)

if internal_conflict_sequences:
    display(
        internal_df[
            internal_df["sequence"].isin(
                internal_conflict_sequences
            )
        ].sort_values("sequence")
    )

if kelm_conflict_sequences:
    display(
        kelm_df[
            kelm_df["sequence"].isin(
                kelm_conflict_sequences
            )
        ].sort_values("sequence")
    )

# ------------------------------------------------------------
# 4. Check overlap between internal and KELM datasets
# ------------------------------------------------------------

internal_sequences = set(internal_df["sequence"])
kelm_sequences = set(kelm_df["sequence"])

overlap_sequences = sorted(
    internal_sequences.intersection(kelm_sequences)
)

overlap_df = kelm_df[
    kelm_df["sequence"].isin(overlap_sequences)
].copy()

if len(overlap_df) > 0:
    internal_labels = (
        internal_df[
            internal_df["sequence"].isin(overlap_sequences)
        ][["sequence", "label"]]
        .drop_duplicates()
        .rename(columns={"label": "internal_label"})
    )

    overlap_df = overlap_df.merge(
        internal_labels,
        on="sequence",
        how="left"
    )

    overlap_df = overlap_df.rename(
        columns={"label": "kelm_label"}
    )

    overlap_df["same_label"] = (
        overlap_df["kelm_label"]
        == overlap_df["internal_label"]
    )

print("\n" + "=" * 70)
print("INTERNAL–KELM EXACT OVERLAP")
print("=" * 70)

print("Exact overlapping sequences:", len(overlap_sequences))

if len(overlap_df) > 0:
    print("\nOverlap by KELM label:")
    print(
        overlap_df["kelm_label"]
        .value_counts()
        .sort_index()
    )

    print("\nLabel agreement:")
    print(
        overlap_df["same_label"]
        .value_counts()
    )

    display(overlap_df.head(20))

# Save overlap report
overlap_report_file = (
    PROCESSED_DIR / "internal_kelm_exact_overlap.csv"
)

overlap_df.to_csv(
    overlap_report_file,
    index=False
)

# ------------------------------------------------------------
# 5. Create leakage-free datasets
# ------------------------------------------------------------

# Remove conflicting-label sequences from internal dataset
internal_qc = internal_df[
    ~internal_df["sequence"].isin(
        internal_conflict_sequences
    )
].copy()

# Remove conflicting-label sequences from KELM dataset
kelm_qc = kelm_df[
    ~kelm_df["sequence"].isin(
        kelm_conflict_sequences
    )
].copy()

# Remove from KELM any sequence already found in internal data
kelm_qc = kelm_qc[
    ~kelm_qc["sequence"].isin(
        set(internal_qc["sequence"])
    )
].copy()

# Remove any remaining exact duplicates
internal_qc = internal_qc.drop_duplicates(
    subset=["sequence"],
    keep="first"
).reset_index(drop=True)

kelm_qc = kelm_qc.drop_duplicates(
    subset=["sequence"],
    keep="first"
).reset_index(drop=True)

# Reassign stable IDs after QC
internal_qc["sequence_id"] = [
    f"INTQC_{index:05d}"
    for index in range(1, len(internal_qc) + 1)
]

kelm_qc["sequence_id"] = [
    f"KELMQC_{index:04d}"
    for index in range(1, len(kelm_qc) + 1)
]

# Put columns in consistent order
internal_qc = internal_qc[
    ["sequence_id", "sequence", "label", "length"]
]

kelm_qc = kelm_qc[
    ["sequence_id", "sequence", "label", "length"]
]

# ------------------------------------------------------------
# 6. Save QC datasets
# ------------------------------------------------------------

internal_qc_file = (
    PROCESSED_DIR / "internal_dataset_qc.csv"
)

kelm_qc_file = (
    PROCESSED_DIR / "kelm_external_dataset_qc.csv"
)

internal_qc.to_csv(
    internal_qc_file,
    index=False
)

kelm_qc.to_csv(
    kelm_qc_file,
    index=False
)

# ------------------------------------------------------------
# 7. Create dataset fingerprints
# ------------------------------------------------------------

def dataframe_sha256(df):
    csv_text = df.to_csv(
        index=False,
        lineterminator="\n"
    )

    return hashlib.sha256(
        csv_text.encode("utf-8")
    ).hexdigest()

dataset_manifest = {
    "internal_dataset": {
        "file": str(internal_qc_file),
        "rows": int(len(internal_qc)),
        "class_counts": {
            str(key): int(value)
            for key, value in (
                internal_qc["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "minimum_length": int(
            internal_qc["length"].min()
        ),
        "maximum_length": int(
            internal_qc["length"].max()
        ),
        "sha256": dataframe_sha256(
            internal_qc
        ),
    },
    "kelm_external_dataset": {
        "file": str(kelm_qc_file),
        "rows": int(len(kelm_qc)),
        "class_counts": {
            str(key): int(value)
            for key, value in (
                kelm_qc["label"]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        },
        "minimum_length": int(
            kelm_qc["length"].min()
        ),
        "maximum_length": int(
            kelm_qc["length"].max()
        ),
        "sha256": dataframe_sha256(
            kelm_qc
        ),
    },
    "removed": {
        "internal_conflicting_sequences": int(
            len(internal_conflict_sequences)
        ),
        "kelm_conflicting_sequences": int(
            len(kelm_conflict_sequences)
        ),
        "internal_kelm_exact_overlap": int(
            len(overlap_sequences)
        ),
        "rejected_nonstandard_sequences": int(
            len(rejected_df)
        ),
    },
}

manifest_file = (
    CHECKPOINT_DIR / "dataset_manifest.json"
)

with open(
    manifest_file,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        dataset_manifest,
        handle,
        indent=2
    )

# ------------------------------------------------------------
# 8. Display final QC summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL INTERNAL QC DATASET")
print("=" * 70)

print("Rows:", len(internal_qc))
print("Class counts:")
print(
    internal_qc["label"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 70)
print("FINAL LEAKAGE-FREE KELM DATASET")
print("=" * 70)

print("Rows:", len(kelm_qc))
print("Class counts:")
print(
    kelm_qc["label"]
    .value_counts()
    .sort_index()
)

print("\nSaved files:")
print(internal_qc_file)
print(kelm_qc_file)
print(overlap_report_file)
print(manifest_file)

# ------------------------------------------------------------
# 9. Save checkpoint
# ------------------------------------------------------------

mark_step_complete(
    "04_dataset_quality_control",
    output_files=[
        internal_qc_file,
        kelm_qc_file,
        overlap_report_file,
        manifest_file,
    ],
    details={
        "internal_final_rows": int(
            len(internal_qc)
        ),
        "kelm_final_rows": int(
            len(kelm_qc)
        ),
        "exact_internal_kelm_overlap": int(
            len(overlap_sequences)
        ),
        "internal_conflicts": int(
            len(internal_conflict_sequences)
        ),
        "kelm_conflicts": int(
            len(kelm_conflict_sequences)
        ),
    },
)

print("\nDataset quality-control step completed.")

In [ ]:
# ============================================================
# STEP 6: Create permanent stratified train/validation/test splits
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split

PROCESSED_DIR = DIRS["data_processed"]
CHECKPOINT_DIR = DIRS["checkpoints"]

RANDOM_SEED = 42

internal_qc_file = (
    PROCESSED_DIR / "internal_dataset_qc.csv"
)

kelm_qc_file = (
    PROCESSED_DIR / "kelm_external_dataset_qc.csv"
)

# ------------------------------------------------------------
# 1. Load final QC datasets
# ------------------------------------------------------------

internal_df = pd.read_csv(internal_qc_file)
kelm_df = pd.read_csv(kelm_qc_file)

print("Internal dataset shape:", internal_df.shape)
print("KELM dataset shape    :", kelm_df.shape)

# ------------------------------------------------------------
# 2. Create train and temporary split
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    internal_df,
    test_size=0.30,
    stratify=internal_df["label"],
    random_state=RANDOM_SEED,
    shuffle=True,
)

# Split temporary set equally into validation and test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED,
    shuffle=True,
)

# ------------------------------------------------------------
# 3. Reset indices
# ------------------------------------------------------------

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Add split-specific row IDs
train_df.insert(
    0,
    "split_row_id",
    [
        f"TRAIN_{i:05d}"
        for i in range(1, len(train_df) + 1)
    ],
)

val_df.insert(
    0,
    "split_row_id",
    [
        f"VAL_{i:05d}"
        for i in range(1, len(val_df) + 1)
    ],
)

test_df.insert(
    0,
    "split_row_id",
    [
        f"TEST_{i:05d}"
        for i in range(1, len(test_df) + 1)
    ],
)

# ------------------------------------------------------------
# 4. Verify that no sequence appears in multiple splits
# ------------------------------------------------------------

train_sequences = set(train_df["sequence"])
val_sequences = set(val_df["sequence"])
test_sequences = set(test_df["sequence"])

train_val_overlap = train_sequences.intersection(val_sequences)
train_test_overlap = train_sequences.intersection(test_sequences)
val_test_overlap = val_sequences.intersection(test_sequences)

if train_val_overlap:
    raise ValueError(
        f"Train-validation overlap found: {len(train_val_overlap)}"
    )

if train_test_overlap:
    raise ValueError(
        f"Train-test overlap found: {len(train_test_overlap)}"
    )

if val_test_overlap:
    raise ValueError(
        f"Validation-test overlap found: {len(val_test_overlap)}"
    )

# ------------------------------------------------------------
# 5. Save permanent split files
# ------------------------------------------------------------

train_file = PROCESSED_DIR / "train_split.csv"
val_file = PROCESSED_DIR / "validation_split.csv"
test_file = PROCESSED_DIR / "internal_test_split.csv"

train_df.to_csv(train_file, index=False)
val_df.to_csv(val_file, index=False)
test_df.to_csv(test_file, index=False)

# Save one combined split-assignment file
split_assignment_df = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)

split_assignment_file = (
    PROCESSED_DIR / "internal_split_assignments.csv"
)

split_assignment_df.to_csv(
    split_assignment_file,
    index=False,
)

# ------------------------------------------------------------
# 6. Create split summary
# ------------------------------------------------------------

def summarize_split(df, name):
    counts = (
        df["label"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    return {
        "name": name,
        "rows": int(len(df)),
        "class_0": int(counts.get(0, 0)),
        "class_1": int(counts.get(1, 0)),
        "minimum_length": int(df["length"].min()),
        "maximum_length": int(df["length"].max()),
        "mean_length": float(df["length"].mean()),
    }

split_summary = {
    "random_seed": RANDOM_SEED,
    "split_ratio": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "train": summarize_split(train_df, "train"),
    "validation": summarize_split(val_df, "validation"),
    "test": summarize_split(test_df, "test"),
    "kelm_external": summarize_split(kelm_df, "kelm_external"),
    "overlap_checks": {
        "train_validation": len(train_val_overlap),
        "train_test": len(train_test_overlap),
        "validation_test": len(val_test_overlap),
    },
}

split_summary_file = (
    CHECKPOINT_DIR / "split_summary.json"
)

with open(
    split_summary_file,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        split_summary,
        handle,
        indent=2
    )

# ------------------------------------------------------------
# 7. Display summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAIN SPLIT")
print("=" * 70)

print("Rows:", len(train_df))
print(train_df["label"].value_counts().sort_index())
print(
    "Length range:",
    train_df["length"].min(),
    "to",
    train_df["length"].max(),
)

print("\n" + "=" * 70)
print("VALIDATION SPLIT")
print("=" * 70)

print("Rows:", len(val_df))
print(val_df["label"].value_counts().sort_index())
print(
    "Length range:",
    val_df["length"].min(),
    "to",
    val_df["length"].max(),
)

print("\n" + "=" * 70)
print("INTERNAL TEST SPLIT")
print("=" * 70)

print("Rows:", len(test_df))
print(test_df["label"].value_counts().sort_index())
print(
    "Length range:",
    test_df["length"].min(),
    "to",
    test_df["length"].max(),
)

print("\n" + "=" * 70)
print("OVERLAP CHECK")
print("=" * 70)

print("Train-validation overlap:", len(train_val_overlap))
print("Train-test overlap      :", len(train_test_overlap))
print("Validation-test overlap :", len(val_test_overlap))

print("\nSaved files:")
print(train_file)
print(val_file)
print(test_file)
print(split_assignment_file)
print(split_summary_file)

# ------------------------------------------------------------
# 8. Save checkpoint
# ------------------------------------------------------------

mark_step_complete(
    "05_fixed_internal_data_splits",
    output_files=[
        train_file,
        val_file,
        test_file,
        split_assignment_file,
        split_summary_file,
    ],
    details={
        "random_seed": RANDOM_SEED,
        "train_rows": int(len(train_df)),
        "validation_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_validation_overlap": int(
            len(train_val_overlap)
        ),
        "train_test_overlap": int(
            len(train_test_overlap)
        ),
        "validation_test_overlap": int(
            len(val_test_overlap)
        ),
    },
)

print("\nPermanent train/validation/test splits created successfully.")

In [ ]:
# ============================================================
# STEP 7: MASTER RESTART-SAFE EMBEDDING PIPELINE
#
# Models:
#   1. ESM2-320
#   2. ESM2-640
#   3. ESM2-1280
#   4. ProtT5
#
# Datasets:
#   1. Train
#   2. Validation
#   3. Internal test
#   4. KELM external
#
# Outputs:
#   Per-residue embeddings: float16 .npy
#   Residue masks: uint8 .npy
#   Metadata: CSV
#   Restart-safe batch chunks: compressed .npz
# ============================================================

from pathlib import Path
import gc
import json
import math
import os
import time
import traceback

import numpy as np
import pandas as pd
import torch
import esm

from transformers import T5Tokenizer, T5EncoderModel


# ============================================================
# 1. GLOBAL CONFIGURATION
# ============================================================

SEED = 42
MAX_LEN = 61

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PROCESSED_DIR = DIRS["data_processed"]
EMBEDDING_ROOT = DIRS["embeddings"]
CHECKPOINT_DIR = DIRS["checkpoints"]

EMBEDDING_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# Exact PLMs from the original pLM4CPP-XAI pipeline
MODEL_CONFIGS = {
    "ESM2_320": {
        "type": "esm",
        "loader": "esm2_t6_8M_UR50D",
        "layer": 6,
        "embedding_dimension": 320,
        "batch_size": 64,
    },

    "ESM2_640": {
        "type": "esm",
        "loader": "esm2_t30_150M_UR50D",
        "layer": 30,
        "embedding_dimension": 640,
        "batch_size": 24,
    },

    "ESM2_1280": {
        "type": "esm",
        "loader": "esm2_t33_650M_UR50D",
        "layer": 33,
        "embedding_dimension": 1280,
        "batch_size": 8,
    },

    "ProtT5": {
        "type": "prott5",
        "loader": "Rostlab/prot_t5_xl_uniref50",
        "layer": None,
        "embedding_dimension": 1024,
        "batch_size": 8,
    },
}


DATASET_FILES = {
    "train": (
        PROCESSED_DIR / "train_split.csv"
    ),

    "validation": (
        PROCESSED_DIR / "validation_split.csv"
    ),

    "internal_test": (
        PROCESSED_DIR / "internal_test_split.csv"
    ),

    "kelm_external": (
        PROCESSED_DIR / "kelm_external_dataset_qc.csv"
    ),
}


print("=" * 75)
print("MASTER PLM EMBEDDING PIPELINE")
print("=" * 75)

print("Device :", DEVICE)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

print("Models :", list(MODEL_CONFIGS))
print("Datasets:", list(DATASET_FILES))
print("Maximum sequence length:", MAX_LEN)


# ============================================================
# 2. VERIFY DATASET FILES
# ============================================================

for dataset_name, dataset_file in DATASET_FILES.items():

    if not dataset_file.exists():
        raise FileNotFoundError(
            f"Missing dataset file for {dataset_name}:\n"
            f"{dataset_file}"
        )

    dataset_check = pd.read_csv(dataset_file)

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "length",
    }

    missing_columns = (
        required_columns
        - set(dataset_check.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    if dataset_check["length"].max() > MAX_LEN:
        raise ValueError(
            f"{dataset_name} contains a sequence longer "
            f"than MAX_LEN={MAX_LEN}."
        )

    print(
        f"[DATASET FOUND] {dataset_name:15s} "
        f"rows={len(dataset_check)}"
    )


# ============================================================
# 3. GENERAL HELPER FUNCTIONS
# ============================================================

def clear_memory():
    """Release CPU and GPU memory."""

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def atomic_save_npz(final_path, **arrays):
    """
    Save an NPZ file using a temporary file first.

    This reduces the possibility of retaining a partially
    written file after a Colab interruption.
    """

    final_path = Path(final_path)

    temporary_path = Path(
        str(final_path) + ".temporary.npz"
    )

    np.savez_compressed(
        temporary_path,
        **arrays,
    )

    temporary_path.replace(final_path)


def validate_saved_chunk(
    chunk_file,
    expected_ids,
    expected_rows,
    embedding_dimension,
):
    """Check whether an existing batch chunk is valid."""

    if not chunk_file.exists():
        return False

    try:
        chunk_data = np.load(
            chunk_file,
            allow_pickle=True,
        )

        saved_ids = (
            chunk_data["sequence_id"]
            .astype(str)
        )

        saved_embeddings = (
            chunk_data["embeddings"]
        )

        saved_masks = (
            chunk_data["masks"]
        )

        valid = (
            np.array_equal(
                saved_ids,
                expected_ids.astype(str),
            )
            and saved_embeddings.shape
            == (
                expected_rows,
                MAX_LEN,
                embedding_dimension,
            )
            and saved_masks.shape
            == (
                expected_rows,
                MAX_LEN,
            )
        )

        chunk_data.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 4. ESM MODEL LOADING
# ============================================================

def load_esm_model(config):
    """Load one ESM2 model."""

    loader_name = config["loader"]

    print(f"\nLoading {loader_name}...")

    loader_function = getattr(
        esm.pretrained,
        loader_name,
    )

    model, alphabet = loader_function()

    model = model.to(DEVICE)
    model.eval()

    batch_converter = (
        alphabet.get_batch_converter()
    )

    print(
        f"Loaded {loader_name} on {DEVICE}"
    )

    return model, alphabet, batch_converter


# ============================================================
# 5. PROTT5 MODEL LOADING
# ============================================================

def load_prott5_model(config):
    """Load ProtT5 tokenizer and encoder."""

    loader_name = config["loader"]

    print(f"\nLoading {loader_name} tokenizer...")

    tokenizer = T5Tokenizer.from_pretrained(
        loader_name,
        do_lower_case=False,
    )

    print(f"Loading {loader_name} encoder...")

    model_dtype = (
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )

    model = T5EncoderModel.from_pretrained(
        loader_name,
        torch_dtype=model_dtype,
        low_cpu_mem_usage=True,
    )

    model = model.to(DEVICE)
    model.eval()

    print(
        f"Loaded {loader_name} on {DEVICE}"
    )

    return model, tokenizer


# ============================================================
# 6. ESM PER-RESIDUE EMBEDDINGS
# ============================================================

@torch.no_grad()
def generate_esm_batch(
    batch_df,
    model,
    batch_converter,
    layer,
    embedding_dimension,
):
    """
    Generate padded per-residue ESM2 embeddings.

    Output:
        embeddings:
            shape = batch × MAX_LEN × embedding_dimension

        masks:
            shape = batch × MAX_LEN
    """

    batch_items = [
        (
            str(row.sequence_id),
            str(row.sequence),
        )
        for row in batch_df.itertuples(index=False)
    ]

    _, sequences, tokens = (
        batch_converter(batch_items)
    )

    tokens = tokens.to(DEVICE)

    autocast_enabled = (
        torch.cuda.is_available()
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=autocast_enabled,
    ):
        results = model(
            tokens,
            repr_layers=[layer],
            return_contacts=False,
        )

    token_representations = (
        results["representations"][layer]
    )

    batch_size_actual = len(batch_df)

    padded_embeddings = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )

    masks = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    for row_index, sequence in enumerate(
        sequences
    ):
        sequence_length = min(
            len(sequence),
            MAX_LEN,
        )

        # ESM:
        # token 0 is BOS.
        # Residue tokens begin at position 1.
        residue_embedding = (
            token_representations[
                row_index,
                1:sequence_length + 1,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(np.float16)
        )

        padded_embeddings[
            row_index,
            :sequence_length,
            :
        ] = residue_embedding

        masks[
            row_index,
            :sequence_length
        ] = 1

    del tokens
    del results
    del token_representations

    return padded_embeddings, masks


# ============================================================
# 7. PROTT5 PER-RESIDUE EMBEDDINGS
# ============================================================

@torch.no_grad()
def generate_prott5_batch(
    batch_df,
    model,
    tokenizer,
    embedding_dimension,
):
    """
    Generate padded per-residue ProtT5 embeddings.

    ProtT5 requires amino acids separated by spaces.
    """

    raw_sequences = (
        batch_df["sequence"]
        .astype(str)
        .tolist()
    )

    spaced_sequences = [
        " ".join(list(sequence))
        for sequence in raw_sequences
    ]

    encoded = tokenizer(
        spaced_sequences,
        add_special_tokens=True,
        padding=True,
        return_tensors="pt",
    )

    input_ids = (
        encoded["input_ids"]
        .to(DEVICE)
    )

    attention_mask = (
        encoded["attention_mask"]
        .to(DEVICE)
    )

    autocast_enabled = (
        torch.cuda.is_available()
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=autocast_enabled,
    ):
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    hidden_states = output.last_hidden_state

    batch_size_actual = len(batch_df)

    padded_embeddings = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )

    masks = np.zeros(
        (
            batch_size_actual,
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    for row_index, sequence in enumerate(
        raw_sequences
    ):
        sequence_length = min(
            len(sequence),
            MAX_LEN,
        )

        # ProtT5 places residue tokens first and EOS last.
        residue_embedding = (
            hidden_states[
                row_index,
                :sequence_length,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(np.float16)
        )

        padded_embeddings[
            row_index,
            :sequence_length,
            :
        ] = residue_embedding

        masks[
            row_index,
            :sequence_length
        ] = 1

    del input_ids
    del attention_mask
    del encoded
    del output
    del hidden_states

    return padded_embeddings, masks


# ============================================================
# 8. PROCESS ONE DATASET FOR ONE MODEL
# ============================================================

def process_dataset_for_model(
    model_key,
    config,
    dataset_name,
    dataset_file,
    model,
    tokenizer_or_converter,
):
    """
    Generate restart-safe batch chunks and then create
    final memory-mapped NPY files.
    """

    dataset_df = pd.read_csv(
        dataset_file
    )

    dataset_df["sequence_id"] = (
        dataset_df["sequence_id"]
        .astype(str)
    )

    number_of_rows = len(dataset_df)

    embedding_dimension = (
        config["embedding_dimension"]
    )

    batch_size = config["batch_size"]

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    output_directory = (
        EMBEDDING_ROOT
        / model_key
        / dataset_name
    )

    chunk_directory = (
        output_directory
        / "batch_chunks"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    chunk_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_embedding_file = (
        output_directory
        / "X_per_residue.npy"
    )

    final_mask_file = (
        output_directory
        / "M_masks.npy"
    )

    final_metadata_file = (
        output_directory
        / "metadata.csv"
    )

    completion_file = (
        output_directory
        / "COMPLETE.json"
    )

    print("\n" + "=" * 75)
    print(
        f"{model_key} — {dataset_name}"
    )
    print("=" * 75)

    print("Rows              :", number_of_rows)
    print("Embedding dimension:", embedding_dimension)
    print("Batch size         :", batch_size)
    print("Number of batches  :", number_of_batches)

    # --------------------------------------------------------
    # Check whether final output is already complete
    # --------------------------------------------------------

    if (
        completion_file.exists()
        and final_embedding_file.exists()
        and final_mask_file.exists()
        and final_metadata_file.exists()
    ):
        try:
            existing_embeddings = np.load(
                final_embedding_file,
                mmap_mode="r",
            )

            existing_masks = np.load(
                final_mask_file,
                mmap_mode="r",
            )

            valid_final_output = (
                existing_embeddings.shape
                == (
                    number_of_rows,
                    MAX_LEN,
                    embedding_dimension,
                )
                and existing_masks.shape
                == (
                    number_of_rows,
                    MAX_LEN,
                )
            )

            del existing_embeddings
            del existing_masks

            if valid_final_output:
                print(
                    "[SKIP COMPLETE] Final files "
                    "already exist and are valid."
                )

                return {
                    "model": model_key,
                    "dataset": dataset_name,
                    "rows": number_of_rows,
                    "embedding_dimension": (
                        embedding_dimension
                    ),
                    "embedding_file": str(
                        final_embedding_file
                    ),
                    "mask_file": str(
                        final_mask_file
                    ),
                    "metadata_file": str(
                        final_metadata_file
                    ),
                    "status": "previously_complete",
                }

        except Exception:
            print(
                "[REBUILD] Existing final files "
                "failed validation."
            )

    # --------------------------------------------------------
    # Generate batch chunks
    # --------------------------------------------------------

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        batch_df = dataset_df.iloc[
            start_index:end_index
        ].copy()

        expected_ids = (
            batch_df["sequence_id"]
            .astype(str)
            .to_numpy()
        )

        chunk_file = (
            chunk_directory
            / f"batch_{batch_number:05d}.npz"
        )

        valid_chunk = validate_saved_chunk(
            chunk_file=chunk_file,
            expected_ids=expected_ids,
            expected_rows=len(batch_df),
            embedding_dimension=(
                embedding_dimension
            ),
        )

        if valid_chunk:
            print(
                f"[SKIP CHUNK] {model_key} | "
                f"{dataset_name} | "
                f"{batch_number + 1}/"
                f"{number_of_batches}"
            )

            continue

        if chunk_file.exists():
            chunk_file.unlink()

        batch_start_time = time.time()

        try:
            if config["type"] == "esm":
                batch_embeddings, batch_masks = (
                    generate_esm_batch(
                        batch_df=batch_df,
                        model=model,
                        batch_converter=(
                            tokenizer_or_converter
                        ),
                        layer=config["layer"],
                        embedding_dimension=(
                            embedding_dimension
                        ),
                    )
                )

            elif config["type"] == "prott5":
                batch_embeddings, batch_masks = (
                    generate_prott5_batch(
                        batch_df=batch_df,
                        model=model,
                        tokenizer=(
                            tokenizer_or_converter
                        ),
                        embedding_dimension=(
                            embedding_dimension
                        ),
                    )
                )

            else:
                raise ValueError(
                    f"Unknown model type: "
                    f"{config['type']}"
                )

        except torch.cuda.OutOfMemoryError:

            clear_memory()

            raise RuntimeError(
                f"GPU memory error for {model_key}. "
                f"Reduce its batch_size in MODEL_CONFIGS "
                f"and rerun this same cell. Completed "
                f"chunks will be skipped."
            )

        atomic_save_npz(
            chunk_file,
            sequence_id=expected_ids,
            embeddings=batch_embeddings,
            masks=batch_masks,
        )

        elapsed_seconds = (
            time.time() - batch_start_time
        )

        print(
            f"[SAVED CHUNK] {model_key} | "
            f"{dataset_name} | "
            f"{batch_number + 1}/"
            f"{number_of_batches} | "
            f"rows {start_index}:{end_index} | "
            f"{elapsed_seconds:.1f} seconds"
        )

        del batch_embeddings
        del batch_masks

        clear_memory()

    # --------------------------------------------------------
    # Assemble final NPY arrays without loading all data
    # into RAM simultaneously
    # --------------------------------------------------------

    print(
        "\nAssembling final memory-mapped files..."
    )

    temporary_embedding_file = (
        output_directory
        / "X_per_residue.temporary.npy"
    )

    temporary_mask_file = (
        output_directory
        / "M_masks.temporary.npy"
    )

    embedding_memmap = (
        np.lib.format.open_memmap(
            temporary_embedding_file,
            mode="w+",
            dtype=np.float16,
            shape=(
                number_of_rows,
                MAX_LEN,
                embedding_dimension,
            ),
        )
    )

    mask_memmap = (
        np.lib.format.open_memmap(
            temporary_mask_file,
            mode="w+",
            dtype=np.uint8,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        chunk_file = (
            chunk_directory
            / f"batch_{batch_number:05d}.npz"
        )

        if not chunk_file.exists():
            raise FileNotFoundError(
                f"Missing chunk:\n{chunk_file}"
            )

        chunk_data = np.load(
            chunk_file,
            allow_pickle=True,
        )

        embedding_memmap[
            start_index:end_index
        ] = chunk_data["embeddings"]

        mask_memmap[
            start_index:end_index
        ] = chunk_data["masks"]

        chunk_data.close()

    embedding_memmap.flush()
    mask_memmap.flush()

    del embedding_memmap
    del mask_memmap

    os.replace(
        temporary_embedding_file,
        final_embedding_file,
    )

    os.replace(
        temporary_mask_file,
        final_mask_file,
    )

    # Metadata preserves exact row order
    metadata_df = dataset_df[
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    metadata_df.insert(
        0,
        "row_index",
        np.arange(len(metadata_df)),
    )

    metadata_df["dataset"] = (
        dataset_name
    )

    metadata_df["model"] = model_key

    metadata_df.to_csv(
        final_metadata_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validate final output
    # --------------------------------------------------------

    final_embeddings_check = np.load(
        final_embedding_file,
        mmap_mode="r",
    )

    final_masks_check = np.load(
        final_mask_file,
        mmap_mode="r",
    )

    expected_embedding_shape = (
        number_of_rows,
        MAX_LEN,
        embedding_dimension,
    )

    expected_mask_shape = (
        number_of_rows,
        MAX_LEN,
    )

    if (
        final_embeddings_check.shape
        != expected_embedding_shape
    ):
        raise ValueError(
            f"Incorrect final embedding shape for "
            f"{model_key}/{dataset_name}: "
            f"{final_embeddings_check.shape}"
        )

    if (
        final_masks_check.shape
        != expected_mask_shape
    ):
        raise ValueError(
            f"Incorrect mask shape for "
            f"{model_key}/{dataset_name}: "
            f"{final_masks_check.shape}"
        )

    del final_embeddings_check
    del final_masks_check

    completion_data = {
        "model": model_key,
        "loader": config["loader"],
        "model_type": config["type"],
        "dataset": dataset_name,
        "source_dataset_file": str(
            dataset_file
        ),
        "rows": int(number_of_rows),
        "maximum_length": int(MAX_LEN),
        "embedding_dimension": int(
            embedding_dimension
        ),
        "embedding_dtype": "float16",
        "mask_dtype": "uint8",
        "embedding_shape": list(
            expected_embedding_shape
        ),
        "mask_shape": list(
            expected_mask_shape
        ),
        "batch_size": int(batch_size),
        "number_of_batches": int(
            number_of_batches
        ),
        "embedding_file": str(
            final_embedding_file
        ),
        "mask_file": str(
            final_mask_file
        ),
        "metadata_file": str(
            final_metadata_file
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    temporary_completion_file = Path(
        str(completion_file) + ".temporary"
    )

    with open(
        temporary_completion_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            completion_data,
            handle,
            indent=2,
        )

    temporary_completion_file.replace(
        completion_file
    )

    print(
        f"[DATASET COMPLETE] {model_key} | "
        f"{dataset_name}"
    )

    print(
        "Embedding shape:",
        expected_embedding_shape,
    )

    return {
        "model": model_key,
        "dataset": dataset_name,
        "rows": number_of_rows,
        "embedding_dimension": (
            embedding_dimension
        ),
        "embedding_file": str(
            final_embedding_file
        ),
        "mask_file": str(
            final_mask_file
        ),
        "metadata_file": str(
            final_metadata_file
        ),
        "status": "completed_now",
    }


# ============================================================
# 9. PROCESS ONE COMPLETE PLM
# ============================================================

def process_complete_model(
    model_key,
    config,
):
    """Process all four datasets using one PLM."""

    print("\n\n" + "#" * 75)
    print(f"STARTING MODEL: {model_key}")
    print("#" * 75)

    model = None
    tokenizer_or_converter = None
    model_results = []

    try:
        if config["type"] == "esm":

            model, alphabet, batch_converter = (
                load_esm_model(config)
            )

            tokenizer_or_converter = (
                batch_converter
            )

        elif config["type"] == "prott5":

            model, tokenizer = (
                load_prott5_model(config)
            )

            tokenizer_or_converter = tokenizer

        else:
            raise ValueError(
                f"Unsupported model type: "
                f"{config['type']}"
            )

        for dataset_name, dataset_file in (
            DATASET_FILES.items()
        ):
            dataset_result = (
                process_dataset_for_model(
                    model_key=model_key,
                    config=config,
                    dataset_name=dataset_name,
                    dataset_file=dataset_file,
                    model=model,
                    tokenizer_or_converter=(
                        tokenizer_or_converter
                    ),
                )
            )

            model_results.append(
                dataset_result
            )

        model_summary_file = (
            EMBEDDING_ROOT
            / model_key
            / "model_embedding_summary.json"
        )

        with open(
            model_summary_file,
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                {
                    "model": model_key,
                    "configuration": config,
                    "datasets": model_results,
                    "completed_at": (
                        pd.Timestamp.now()
                        .isoformat()
                    ),
                },
                handle,
                indent=2,
            )

        mark_step_complete(
            f"embeddings_{model_key}",
            output_files=[
                model_summary_file
            ],
            details={
                "model": model_key,
                "loader": config["loader"],
                "datasets": [
                    result["dataset"]
                    for result in model_results
                ],
            },
        )

        print("\n" + "#" * 75)
        print(f"MODEL COMPLETE: {model_key}")
        print("#" * 75)

        return model_results

    finally:
        if model is not None:
            del model

        if tokenizer_or_converter is not None:
            del tokenizer_or_converter

        if "alphabet" in locals():
            del alphabet

        clear_memory()


# ============================================================
# 10. RUN ALL FOUR PLMS
# ============================================================

all_model_results = []
failed_models = []

for model_key, config in (
    MODEL_CONFIGS.items()
):
    try:
        model_results = (
            process_complete_model(
                model_key=model_key,
                config=config,
            )
        )

        all_model_results.extend(
            model_results
        )

    except Exception as error:
        print("\n" + "!" * 75)
        print(f"MODEL FAILED: {model_key}")
        print("!" * 75)
        print(type(error).__name__, ":", error)

        traceback.print_exc()

        failed_models.append(
            {
                "model": model_key,
                "error_type": (
                    type(error).__name__
                ),
                "error_message": str(error),
            }
        )

        failure_file = (
            CHECKPOINT_DIR
            / "embedding_failures.json"
        )

        with open(
            failure_file,
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                failed_models,
                handle,
                indent=2,
            )

        clear_memory()

        # Continue to the next model rather than losing
        # all progress.
        continue


# ============================================================
# 11. FINAL VERIFICATION OF ALL OUTPUTS
# ============================================================

print("\n\n" + "=" * 75)
print("FINAL EMBEDDING VERIFICATION")
print("=" * 75)

verification_rows = []

for model_key, config in (
    MODEL_CONFIGS.items()
):

    for dataset_name, dataset_file in (
        DATASET_FILES.items()
    ):
        output_directory = (
            EMBEDDING_ROOT
            / model_key
            / dataset_name
        )

        embedding_file = (
            output_directory
            / "X_per_residue.npy"
        )

        mask_file = (
            output_directory
            / "M_masks.npy"
        )

        metadata_file = (
            output_directory
            / "metadata.csv"
        )

        complete_file = (
            output_directory
            / "COMPLETE.json"
        )

        expected_rows = len(
            pd.read_csv(dataset_file)
        )

        expected_shape = (
            expected_rows,
            MAX_LEN,
            config["embedding_dimension"],
        )

        status = "MISSING"

        actual_shape = None

        if (
            embedding_file.exists()
            and mask_file.exists()
            and metadata_file.exists()
            and complete_file.exists()
        ):
            try:
                embedding_array = np.load(
                    embedding_file,
                    mmap_mode="r",
                )

                mask_array = np.load(
                    mask_file,
                    mmap_mode="r",
                )

                actual_shape = tuple(
                    embedding_array.shape
                )

                if (
                    actual_shape == expected_shape
                    and mask_array.shape
                    == (
                        expected_rows,
                        MAX_LEN,
                    )
                ):
                    status = "COMPLETE"
                else:
                    status = "SHAPE_ERROR"

                del embedding_array
                del mask_array

            except Exception:
                status = "READ_ERROR"

        verification_rows.append(
            {
                "model": model_key,
                "dataset": dataset_name,
                "expected_rows": (
                    expected_rows
                ),
                "embedding_dimension": (
                    config[
                        "embedding_dimension"
                    ]
                ),
                "expected_shape": str(
                    expected_shape
                ),
                "actual_shape": str(
                    actual_shape
                ),
                "status": status,
                "embedding_file": str(
                    embedding_file
                ),
            }
        )

        print(
            f"{model_key:12s} | "
            f"{dataset_name:15s} | "
            f"{status:12s} | "
            f"{actual_shape}"
        )


verification_df = pd.DataFrame(
    verification_rows
)

verification_file = (
    CHECKPOINT_DIR
    / "all_plm_embedding_verification.csv"
)

verification_df.to_csv(
    verification_file,
    index=False,
)


# ============================================================
# 12. FINAL MASTER CHECKPOINT
# ============================================================

all_complete = bool(
    (
        verification_df["status"]
        == "COMPLETE"
    ).all()
)

master_summary = {
    "all_complete": all_complete,
    "models": list(
        MODEL_CONFIGS.keys()
    ),
    "datasets": list(
        DATASET_FILES.keys()
    ),
    "maximum_sequence_length": (
        MAX_LEN
    ),
    "failed_models": failed_models,
    "verification_file": str(
        verification_file
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

master_summary_file = (
    CHECKPOINT_DIR
    / "all_plm_embedding_master_summary.json"
)

with open(
    master_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        master_summary,
        handle,
        indent=2,
    )


if all_complete:
    mark_step_complete(
        "06_all_PLM_embeddings",
        output_files=[
            verification_file,
            master_summary_file,
        ],
        details=master_summary,
    )

    print("\n" + "=" * 75)
    print("ALL FOUR PLM EMBEDDINGS COMPLETED")
    print("=" * 75)

else:
    print("\n" + "=" * 75)
    print("PIPELINE PARTIALLY COMPLETED")
    print("=" * 75)

    print(
        "Rerun this same cell. Valid completed "
        "models and batches will be skipped."
    )


print("\nVerification table saved to:")
print(verification_file)

print("\nMaster summary saved to:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 8A: Check TensorFlow compatibility
# ============================================================

import sys
import subprocess
import importlib.util

# Install TensorFlow only if it is missing
if importlib.util.find_spec("tensorflow") is None:
    print("TensorFlow is not installed. Installing it now...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tensorflow"
    ])

import tensorflow as tf
import numpy as np

print("=" * 70)
print("TENSORFLOW ENVIRONMENT")
print("=" * 70)

print("TensorFlow version:", tf.__version__)
print("NumPy version     :", np.__version__)
print("TensorFlow GPUs   :", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    print("\nTensorFlow can access the GPU.")
else:
    print("\nWARNING: TensorFlow cannot currently access the GPU.")

# Prevent TensorFlow from reserving all GPU memory immediately
gpu_devices = tf.config.list_physical_devices("GPU")

for gpu in gpu_devices:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )
    except RuntimeError as error:
        print("Memory-growth warning:", error)

# Reproducibility
SEED = 42

tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic TensorFlow operations enabled.")
except Exception as error:
    print(
        "Deterministic operations could not be fully enabled:",
        error
    )

# Save the environment check permanently
tensorflow_environment_file = (
    DIRS["checkpoints"]
    / "tensorflow_environment.json"
)

tensorflow_environment = {
    "tensorflow_version": tf.__version__,
    "numpy_version": np.__version__,
    "gpu_available": bool(
        tf.config.list_physical_devices("GPU")
    ),
    "gpu_devices": [
        str(device)
        for device in tf.config.list_physical_devices("GPU")
    ],
    "seed": SEED,
}

import json

with open(
    tensorflow_environment_file,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        tensorflow_environment,
        handle,
        indent=2
    )

mark_step_complete(
    "07_tensorflow_environment",
    output_files=[
        tensorflow_environment_file
    ],
    details=tensorflow_environment
)

print("\nEnvironment record saved:")
print(tensorflow_environment_file)

In [ ]:
# ============================================================
# RESUME PROJECT AFTER A NEW COLAB SESSION
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

DIRS = {
    "code": PROJECT_DIR / "00_code",
    "data_original": PROJECT_DIR / "01_data_original",
    "data_processed": PROJECT_DIR / "02_data_processed",
    "embeddings": PROJECT_DIR / "03_embeddings",
    "models": PROJECT_DIR / "04_models",
    "predictions": PROJECT_DIR / "05_predictions",
    "xai": PROJECT_DIR / "06_xai",
    "results": PROJECT_DIR / "07_results",
    "checkpoints": PROJECT_DIR / "08_checkpoints",
    "logs": PROJECT_DIR / "09_logs",
}

STATUS_FILE = (
    DIRS["checkpoints"] / "pipeline_status.json"
)

if STATUS_FILE.exists():
    with open(
        STATUS_FILE,
        "r",
        encoding="utf-8"
    ) as handle:
        PIPELINE_STATUS = json.load(handle)
else:
    PIPELINE_STATUS = {}

def save_pipeline_status():
    temporary_file = STATUS_FILE.with_suffix(".tmp")

    with open(
        temporary_file,
        "w",
        encoding="utf-8"
    ) as handle:
        json.dump(
            PIPELINE_STATUS,
            handle,
            indent=2
        )

    os.replace(
        temporary_file,
        STATUS_FILE
    )

def mark_step_complete(
    step_name,
    output_files=None,
    details=None
):
    PIPELINE_STATUS[step_name] = {
        "completed": True,
        "output_files": [
            str(path)
            for path in (output_files or [])
        ],
        "details": details or {}
    }

    save_pipeline_status()
    print(f"[SAVED] {step_name}")

def step_is_complete(
    step_name,
    required_files=None
):
    record = PIPELINE_STATUS.get(
        step_name,
        {}
    )

    if not record.get(
        "completed",
        False
    ):
        return False

    if required_files:
        return all(
            Path(path).exists()
            for path in required_files
        )

    return True

print("Project restored:", PROJECT_DIR)

print("\nCompleted steps:")
for step_name in PIPELINE_STATUS:
    print("✓", step_name)

In [ ]:
# ============================================================
# STEP 8B: RESTART-SAFE ATTENTION CLASSIFIER TRAINING
#
# Trains one masked residue-attention classifier for each PLM:
#   ESM2-320, ESM2-640, ESM2-1280, ProtT5
#
# Uses:
#   Train set       -> model fitting
#   Validation set  -> early stopping + threshold selection
#   Internal test   -> final internal evaluation
#   KELM external   -> independent external evaluation
# ============================================================

from pathlib import Path
import gc
import json
import os
import random
import shutil
import traceback

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
)
from sklearn.utils.class_weight import compute_class_weight


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
MAX_LEN = 61
MAX_EPOCHS = 100
PATIENCE = 12
LEARNING_RATE = 1e-3

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# Prevent TensorFlow from taking all GPU memory immediately
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Mixed precision is efficient on a Tesla T4
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow version:", tf.__version__)
print("Precision policy  :", mixed_precision.global_policy())
print("TensorFlow GPUs   :", tf.config.list_physical_devices("GPU"))


MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 64,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 48,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 32,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 32,
    },
}

DATASET_NAMES = [
    "train",
    "validation",
    "internal_test",
    "kelm_external",
]

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
PREDICTION_ROOT = DIRS["predictions"]
RESULT_ROOT = DIRS["results"]
CHECKPOINT_ROOT = DIRS["checkpoints"]
LOG_ROOT = DIRS["logs"]

for folder in [
    MODEL_ROOT,
    PREDICTION_ROOT,
    RESULT_ROOT,
    CHECKPOINT_ROOT,
    LOG_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. CUSTOM MASKED ATTENTION LAYER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):
    """
    Learn one attention score per residue and calculate a
    mask-aware weighted peptide representation.

    Inputs:
        residue_features: batch × length × features
        residue_mask:     batch × length

    Returns:
        pooled_vector:     batch × features
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def call(self, inputs):
        residue_features, residue_mask = inputs

        # batch × length × 1
        logits = self.attention_dense(
            residue_features
        )

        # batch × length
        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        # Assign extremely negative scores to padded positions
        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        attention_weights = tf.expand_dims(
            attention_weights,
            axis=-1,
        )

        weighted_features = (
            residue_features
            * attention_weights
        )

        pooled_vector = tf.reduce_sum(
            weighted_features,
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. BUILD ATTENTION CLASSIFIER
# ============================================================

def build_attention_classifier(
    embedding_dimension
):
    residue_input = tf.keras.Input(
        shape=(
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=tf.float16,
        name="residue_embeddings",
    )

    mask_input = tf.keras.Input(
        shape=(MAX_LEN,),
        dtype=tf.uint8,
        name="residue_mask",
    )

    # Layer normalization stabilizes PLM embeddings
    x = tf.keras.layers.LayerNormalization(
        epsilon=1e-6,
        name="embedding_layer_norm",
    )(residue_input)

    # Residue-level nonlinear projection
    x = tf.keras.layers.Dense(
        128,
        activation="gelu",
        name="residue_projection",
    )(x)

    x = tf.keras.layers.Dropout(
        0.20,
        name="residue_dropout",
    )(x)

    pooled = MaskedAttentionPooling(
        name="masked_attention_pooling",
    )([x, mask_input])

    x = tf.keras.layers.Dense(
        128,
        activation="gelu",
        name="peptide_dense_1",
    )(pooled)

    x = tf.keras.layers.Dropout(
        0.30,
        name="peptide_dropout_1",
    )(x)

    x = tf.keras.layers.Dense(
        32,
        activation="gelu",
        name="peptide_dense_2",
    )(x)

    x = tf.keras.layers.Dropout(
        0.20,
        name="peptide_dropout_2",
    )(x)

    # Force final probability to float32 under mixed precision
    output = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="cpp_probability",
    )(x)

    model = tf.keras.Model(
        inputs=[
            residue_input,
            mask_input,
        ],
        outputs=output,
        name="pLM4CPP_attention_classifier",
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
    )

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.AUC(
                name="roc_auc",
                curve="ROC",
            ),
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR",
            ),
        ],
    )

    return model


# ============================================================
# 4. LOAD ONE MODEL'S SAVED EMBEDDINGS
# ============================================================

def load_embedding_dataset(
    model_key,
    dataset_name
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_key
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n"
                f"{required_file}"
            )

    # Memory mapping avoids unnecessary initial RAM copies
    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    labels = (
        metadata["label"]
        .astype(np.float32)
        .to_numpy()
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Row mismatch for "
            f"{model_key}/{dataset_name}"
        )

    return (
        embeddings,
        masks,
        labels,
        metadata,
    )


# ============================================================
# 5. VALIDATION-BASED THRESHOLD SELECTION
# ============================================================

def choose_validation_threshold(
    y_true,
    probabilities
):
    """
    Choose the probability threshold that maximizes MCC
    on the validation set only.
    """

    thresholds = np.linspace(
        0.05,
        0.95,
        181,
    )

    records = []

    best_threshold = 0.5
    best_mcc = -1.0

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            predictions,
        )

        records.append({
            "threshold": float(threshold),
            "mcc": float(mcc),
            "accuracy": float(
                accuracy_score(
                    y_true,
                    predictions,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    predictions,
                )
            ),
            "f1": float(
                f1_score(
                    y_true,
                    predictions,
                    zero_division=0,
                )
            ),
        })

        if mcc > best_mcc:
            best_mcc = mcc
            best_threshold = float(
                threshold
            )

    return (
        best_threshold,
        pd.DataFrame(records),
    )


# ============================================================
# 6. METRIC CALCULATION
# ============================================================

def calculate_binary_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            specificity
        ),
        "negative_predictive_value": float(
            npv
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


# ============================================================
# 7. SAVE PREDICTIONS
# ============================================================

def create_prediction_table(
    metadata,
    probabilities,
    threshold,
    dataset_name,
    model_key,
):
    result = metadata.copy()

    result["probability_CPP"] = (
        probabilities.astype(float)
    )

    result["predicted_label"] = (
        probabilities >= threshold
    ).astype(int)

    result["correct_prediction"] = (
        result["predicted_label"]
        == result["label"]
    )

    result["decision_threshold"] = (
        threshold
    )

    result["model"] = model_key
    result["evaluation_dataset"] = (
        dataset_name
    )

    return result


# ============================================================
# 8. TRAIN ONE PLM CLASSIFIER
# ============================================================

def train_one_plm_classifier(
    model_key,
    config,
):
    print("\n\n" + "#" * 76)
    print(f"TRAINING CLASSIFIER: {model_key}")
    print("#" * 76)

    embedding_dimension = (
        config["embedding_dimension"]
    )

    batch_size = config["batch_size"]

    model_dir = (
        MODEL_ROOT / model_key
    )

    prediction_dir = (
        PREDICTION_ROOT / model_key
    )

    log_dir = (
        LOG_ROOT / model_key
    )

    backup_dir = (
        model_dir / "training_backup"
    )

    for folder in [
        model_dir,
        prediction_dir,
        log_dir,
    ]:
        folder.mkdir(
            parents=True,
            exist_ok=True,
        )

    final_model_file = (
        model_dir
        / "final_attention_classifier.keras"
    )

    best_model_file = (
        model_dir
        / "best_attention_classifier.keras"
    )

    completion_file = (
        model_dir
        / "TRAINING_COMPLETE.json"
    )

    metrics_file = (
        model_dir
        / "evaluation_metrics.json"
    )

    threshold_file = (
        model_dir
        / "selected_threshold.json"
    )

    # --------------------------------------------------------
    # Skip a fully completed model
    # --------------------------------------------------------

    if (
        completion_file.exists()
        and final_model_file.exists()
        and metrics_file.exists()
        and threshold_file.exists()
    ):
        print(
            "[SKIP COMPLETE] Model and evaluation "
            "outputs already exist."
        )

        with open(
            metrics_file,
            "r",
            encoding="utf-8",
        ) as handle:
            existing_metrics = json.load(
                handle
            )

        return existing_metrics

    # --------------------------------------------------------
    # Load permanent embedding files
    # --------------------------------------------------------

    (
        X_train,
        M_train,
        y_train,
        meta_train,
    ) = load_embedding_dataset(
        model_key,
        "train",
    )

    (
        X_val,
        M_val,
        y_val,
        meta_val,
    ) = load_embedding_dataset(
        model_key,
        "validation",
    )

    (
        X_test,
        M_test,
        y_test,
        meta_test,
    ) = load_embedding_dataset(
        model_key,
        "internal_test",
    )

    (
        X_kelm,
        M_kelm,
        y_kelm,
        meta_kelm,
    ) = load_embedding_dataset(
        model_key,
        "kelm_external",
    )

    print("Training shape  :", X_train.shape)
    print("Validation shape:", X_val.shape)
    print("Test shape      :", X_test.shape)
    print("KELM shape      :", X_kelm.shape)

    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------

    unique_classes = np.unique(
        y_train.astype(int)
    )

    calculated_weights = (
        compute_class_weight(
            class_weight="balanced",
            classes=unique_classes,
            y=y_train.astype(int),
        )
    )

    class_weight = {
        int(class_label): float(weight)
        for class_label, weight in zip(
            unique_classes,
            calculated_weights,
        )
    }

    print("Class weights:", class_weight)

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    model = build_attention_classifier(
        embedding_dimension
    )

    model.summary()

    history_file = (
        log_dir / "training_history.csv"
    )

    callbacks = [
        # Restores optimizer/model state after interruption
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(backup_dir),
            save_freq="epoch",
            delete_checkpoint=False,
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_file),
            monitor="val_roc_auc",
            mode="max",
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_roc_auc",
            mode="max",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),

        tf.keras.callbacks.CSVLogger(
            filename=str(history_file),
            append=True,
        ),

        tf.keras.callbacks.TerminateOnNaN(),
    ]

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = model.fit(
        x={
            "residue_embeddings": X_train,
            "residue_mask": M_train,
        },
        y=y_train,
        validation_data=(
            {
                "residue_embeddings": X_val,
                "residue_mask": M_val,
            },
            y_val,
        ),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=callbacks,
        shuffle=True,
        verbose=1,
    )

    # Save final in-memory best-restored model
    model.save(
        final_model_file
    )

    # --------------------------------------------------------
    # Generate probabilities
    # --------------------------------------------------------

    validation_probabilities = (
        model.predict(
            {
                "residue_embeddings": X_val,
                "residue_mask": M_val,
            },
            batch_size=batch_size,
            verbose=0,
        )
        .reshape(-1)
        .astype(float)
    )

    test_probabilities = (
        model.predict(
            {
                "residue_embeddings": X_test,
                "residue_mask": M_test,
            },
            batch_size=batch_size,
            verbose=0,
        )
        .reshape(-1)
        .astype(float)
    )

    kelm_probabilities = (
        model.predict(
            {
                "residue_embeddings": X_kelm,
                "residue_mask": M_kelm,
            },
            batch_size=batch_size,
            verbose=0,
        )
        .reshape(-1)
        .astype(float)
    )

    # --------------------------------------------------------
    # Select threshold using validation data only
    # --------------------------------------------------------

    (
        selected_threshold,
        threshold_scan_df,
    ) = choose_validation_threshold(
        y_true=y_val.astype(int),
        probabilities=validation_probabilities,
    )

    print(
        "Selected validation threshold:",
        selected_threshold
    )

    threshold_scan_file = (
        model_dir
        / "validation_threshold_scan.csv"
    )

    threshold_scan_df.to_csv(
        threshold_scan_file,
        index=False,
    )

    threshold_information = {
        "selection_dataset": "validation",
        "selection_metric": (
            "Matthews correlation coefficient"
        ),
        "selected_threshold": float(
            selected_threshold
        ),
        "default_threshold_for_comparison": 0.5,
    }

    with open(
        threshold_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            threshold_information,
            handle,
            indent=2,
        )

    # --------------------------------------------------------
    # Evaluate validation, internal test, and KELM
    # --------------------------------------------------------

    validation_metrics = (
        calculate_binary_metrics(
            y_true=y_val.astype(int),
            probabilities=validation_probabilities,
            threshold=selected_threshold,
        )
    )

    internal_test_metrics = (
        calculate_binary_metrics(
            y_true=y_test.astype(int),
            probabilities=test_probabilities,
            threshold=selected_threshold,
        )
    )

    kelm_metrics = (
        calculate_binary_metrics(
            y_true=y_kelm.astype(int),
            probabilities=kelm_probabilities,
            threshold=selected_threshold,
        )
    )

    # Also retain default 0.5 metrics for transparency
    internal_test_default_metrics = (
        calculate_binary_metrics(
            y_true=y_test.astype(int),
            probabilities=test_probabilities,
            threshold=0.5,
        )
    )

    kelm_default_metrics = (
        calculate_binary_metrics(
            y_true=y_kelm.astype(int),
            probabilities=kelm_probabilities,
            threshold=0.5,
        )
    )

    all_metrics = {
        "model": model_key,
        "embedding_dimension": int(
            embedding_dimension
        ),
        "selected_threshold": float(
            selected_threshold
        ),
        "class_weights": {
            str(key): float(value)
            for key, value in class_weight.items()
        },
        "validation_selected_threshold": (
            validation_metrics
        ),
        "internal_test_selected_threshold": (
            internal_test_metrics
        ),
        "kelm_external_selected_threshold": (
            kelm_metrics
        ),
        "internal_test_default_0.5": (
            internal_test_default_metrics
        ),
        "kelm_external_default_0.5": (
            kelm_default_metrics
        ),
    }

    with open(
        metrics_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            all_metrics,
            handle,
            indent=2,
        )

    # --------------------------------------------------------
    # Save prediction tables
    # --------------------------------------------------------

    validation_predictions = (
        create_prediction_table(
            metadata=meta_val,
            probabilities=validation_probabilities,
            threshold=selected_threshold,
            dataset_name="validation",
            model_key=model_key,
        )
    )

    test_predictions = (
        create_prediction_table(
            metadata=meta_test,
            probabilities=test_probabilities,
            threshold=selected_threshold,
            dataset_name="internal_test",
            model_key=model_key,
        )
    )

    kelm_predictions = (
        create_prediction_table(
            metadata=meta_kelm,
            probabilities=kelm_probabilities,
            threshold=selected_threshold,
            dataset_name="kelm_external",
            model_key=model_key,
        )
    )

    validation_prediction_file = (
        prediction_dir
        / "validation_predictions.csv"
    )

    test_prediction_file = (
        prediction_dir
        / "internal_test_predictions.csv"
    )

    kelm_prediction_file = (
        prediction_dir
        / "kelm_external_predictions.csv"
    )

    validation_predictions.to_csv(
        validation_prediction_file,
        index=False,
    )

    test_predictions.to_csv(
        test_prediction_file,
        index=False,
    )

    kelm_predictions.to_csv(
        kelm_prediction_file,
        index=False,
    )

    # --------------------------------------------------------
    # Mark complete atomically
    # --------------------------------------------------------

    completion_information = {
        "model": model_key,
        "embedding_dimension": int(
            embedding_dimension
        ),
        "batch_size": int(batch_size),
        "maximum_epochs": int(MAX_EPOCHS),
        "epochs_run_this_session": int(
            len(history.history.get("loss", []))
        ),
        "selected_threshold": float(
            selected_threshold
        ),
        "final_model_file": str(
            final_model_file
        ),
        "best_model_file": str(
            best_model_file
        ),
        "metrics_file": str(
            metrics_file
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    temporary_completion_file = Path(
        str(completion_file) + ".temporary"
    )

    with open(
        temporary_completion_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            completion_information,
            handle,
            indent=2,
        )

    temporary_completion_file.replace(
        completion_file
    )

    mark_step_complete(
        f"attention_classifier_{model_key}",
        output_files=[
            final_model_file,
            best_model_file,
            metrics_file,
            threshold_file,
            threshold_scan_file,
            validation_prediction_file,
            test_prediction_file,
            kelm_prediction_file,
            history_file,
            completion_file,
        ],
        details=completion_information,
    )

    print("\nInternal test metrics:")
    print(
        json.dumps(
            internal_test_metrics,
            indent=2,
        )
    )

    print("\nKELM external metrics:")
    print(
        json.dumps(
            kelm_metrics,
            indent=2,
        )
    )

    # Release one model before loading the next one
    del model
    del history

    del X_train, M_train, y_train, meta_train
    del X_val, M_val, y_val, meta_val
    del X_test, M_test, y_test, meta_test
    del X_kelm, M_kelm, y_kelm, meta_kelm

    gc.collect()
    tf.keras.backend.clear_session()

    return all_metrics


# ============================================================
# 9. RUN ALL FOUR PLM CLASSIFIERS
# ============================================================

all_results = {}
failed_models = []

for model_key, config in MODEL_CONFIGS.items():
    try:
        model_metrics = (
            train_one_plm_classifier(
                model_key=model_key,
                config=config,
            )
        )

        all_results[model_key] = (
            model_metrics
        )

    except tf.errors.ResourceExhaustedError as error:
        print("\n" + "!" * 76)
        print(f"GPU MEMORY ERROR: {model_key}")
        print("!" * 76)

        print(
            "Reduce the batch size for this model "
            "and rerun the same cell."
        )

        print(error)

        failed_models.append({
            "model": model_key,
            "error": str(error),
        })

        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as error:
        print("\n" + "!" * 76)
        print(f"MODEL FAILED: {model_key}")
        print("!" * 76)

        print(
            type(error).__name__,
            ":",
            error,
        )

        traceback.print_exc()

        failed_models.append({
            "model": model_key,
            "error_type": (
                type(error).__name__
            ),
            "error": str(error),
        })

        tf.keras.backend.clear_session()
        gc.collect()


# ============================================================
# 10. CREATE MASTER PERFORMANCE TABLE
# ============================================================

performance_rows = []

for model_key, model_metrics in (
    all_results.items()
):
    for evaluation_name in [
        "validation_selected_threshold",
        "internal_test_selected_threshold",
        "kelm_external_selected_threshold",
    ]:
        metrics = model_metrics[
            evaluation_name
        ]

        performance_rows.append({
            "model": model_key,
            "evaluation": evaluation_name,
            **metrics,
        })

performance_df = pd.DataFrame(
    performance_rows
)

performance_file = (
    RESULT_ROOT
    / "tables_main"
    / "attention_classifier_performance.csv"
)

performance_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)

performance_df.to_csv(
    performance_file,
    index=False,
)

failure_file = (
    CHECKPOINT_ROOT
    / "attention_classifier_failures.json"
)

with open(
    failure_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        failed_models,
        handle,
        indent=2,
    )

print("\n\n" + "=" * 76)
print("ATTENTION CLASSIFIER TRAINING SUMMARY")
print("=" * 76)

if not performance_df.empty:
    display(
        performance_df[
            [
                "model",
                "evaluation",
                "threshold",
                "accuracy",
                "balanced_accuracy",
                "f1",
                "mcc",
                "roc_auc",
                "pr_auc",
            ]
        ]
    )

print("\nFailed models:", failed_models)
print("\nPerformance table saved to:")
print(performance_file)

if len(failed_models) == 0 and len(all_results) == 4:
    mark_step_complete(
        "08_all_attention_classifiers",
        output_files=[
            performance_file,
            failure_file,
        ],
        details={
            "models_completed": list(
                all_results.keys()
            ),
            "failed_models": [],
        },
    )

    print("\n" + "=" * 76)
    print("ALL FOUR ATTENTION CLASSIFIERS COMPLETED")
    print("=" * 76)

else:
    print("\nSome models are incomplete.")
    print(
        "Rerun this same cell. Completed models "
        "will be skipped and training backups "
        "will restore interrupted models."
    )

In [ ]:
# ============================================================
# STEP 9: FOUR-PLM PROBABILITY ENSEMBLE
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASET_FILES = {
    "validation": "validation_predictions.csv",
    "internal_test": "internal_test_predictions.csv",
    "kelm_external": "kelm_external_predictions.csv",
}

PREDICTION_ROOT = DIRS["predictions"]
RESULT_TABLE_DIR = DIRS["results"] / "tables_main"
ENSEMBLE_DIR = PREDICTION_ROOT / "ensemble"

RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. LOAD AND ALIGN MODEL PREDICTIONS
# ============================================================

def load_aligned_predictions(dataset_name, filename):

    model_tables = {}

    for model_name in MODEL_NAMES:

        prediction_file = (
            PREDICTION_ROOT
            / model_name
            / filename
        )

        if not prediction_file.exists():
            raise FileNotFoundError(
                f"Missing prediction file:\n{prediction_file}"
            )

        table = pd.read_csv(prediction_file)

        required_columns = {
            "sequence_id",
            "sequence",
            "label",
            "probability_CPP",
        }

        missing_columns = (
            required_columns - set(table.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{model_name}/{dataset_name} is missing: "
                f"{sorted(missing_columns)}"
            )

        table["sequence_id"] = (
            table["sequence_id"].astype(str)
        )

        model_tables[model_name] = table.copy()

    reference_model = MODEL_NAMES[0]

    reference = model_tables[
        reference_model
    ][
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    reference = reference.reset_index(drop=True)

    merged = reference.copy()

    for model_name in MODEL_NAMES:

        current = (
            model_tables[model_name]
            .set_index("sequence_id")
        )

        expected_ids = (
            merged["sequence_id"]
            .astype(str)
            .tolist()
        )

        missing_ids = [
            sequence_id
            for sequence_id in expected_ids
            if sequence_id not in current.index
        ]

        if missing_ids:
            raise ValueError(
                f"{model_name}/{dataset_name} is missing "
                f"{len(missing_ids)} sequence IDs."
            )

        current = current.loc[
            expected_ids
        ].reset_index()

        if not np.array_equal(
            current["label"].to_numpy(),
            merged["label"].to_numpy(),
        ):
            raise ValueError(
                f"Label mismatch for {model_name}/{dataset_name}"
            )

        merged[
            f"probability_{model_name}"
        ] = current[
            "probability_CPP"
        ].astype(float).to_numpy()

    return merged


ensemble_tables = {}

for dataset_name, filename in DATASET_FILES.items():

    ensemble_tables[dataset_name] = (
        load_aligned_predictions(
            dataset_name=dataset_name,
            filename=filename,
        )
    )

    print(
        f"[LOADED] {dataset_name}: "
        f"{len(ensemble_tables[dataset_name])} rows"
    )


# ============================================================
# 2. CALCULATE ENSEMBLE PROBABILITIES AND AGREEMENT
# ============================================================

probability_columns = [
    f"probability_{model_name}"
    for model_name in MODEL_NAMES
]

for dataset_name, table in ensemble_tables.items():

    probability_matrix = table[
        probability_columns
    ].to_numpy(dtype=float)

    table[
        "ensemble_probability_mean"
    ] = probability_matrix.mean(axis=1)

    table[
        "ensemble_probability_median"
    ] = np.median(
        probability_matrix,
        axis=1,
    )

    table[
        "model_probability_std"
    ] = probability_matrix.std(
        axis=1,
        ddof=0,
    )

    table[
        "model_probability_range"
    ] = (
        probability_matrix.max(axis=1)
        - probability_matrix.min(axis=1)
    )

    table[
        "minimum_model_probability"
    ] = probability_matrix.min(axis=1)

    table[
        "maximum_model_probability"
    ] = probability_matrix.max(axis=1)


# ============================================================
# 3. THRESHOLD OPTIMIZATION USING VALIDATION ONLY
# ============================================================

def select_threshold_by_mcc(
    y_true,
    probabilities,
):

    thresholds = np.linspace(
        0.01,
        0.99,
        197,
    )

    rows = []

    best_threshold = 0.5
    best_mcc = -np.inf

    for threshold in thresholds:

        predicted = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            predicted,
        )

        rows.append({
            "threshold": float(threshold),
            "mcc": float(mcc),
            "accuracy": float(
                accuracy_score(
                    y_true,
                    predicted,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    predicted,
                )
            ),
            "f1": float(
                f1_score(
                    y_true,
                    predicted,
                    zero_division=0,
                )
            ),
        })

        if mcc > best_mcc:
            best_mcc = mcc
            best_threshold = float(
                threshold
            )

    return (
        best_threshold,
        pd.DataFrame(rows),
    )


validation_table = ensemble_tables[
    "validation"
]

y_validation = (
    validation_table["label"]
    .astype(int)
    .to_numpy()
)

mean_threshold, mean_threshold_scan = (
    select_threshold_by_mcc(
        y_true=y_validation,
        probabilities=(
            validation_table[
                "ensemble_probability_mean"
            ].to_numpy()
        ),
    )
)

median_threshold, median_threshold_scan = (
    select_threshold_by_mcc(
        y_true=y_validation,
        probabilities=(
            validation_table[
                "ensemble_probability_median"
            ].to_numpy()
        ),
    )
)

print("\nSelected mean-ensemble threshold  :", mean_threshold)
print("Selected median-ensemble threshold:", median_threshold)

mean_threshold_scan.to_csv(
    ENSEMBLE_DIR
    / "validation_mean_threshold_scan.csv",
    index=False,
)

median_threshold_scan.to_csv(
    ENSEMBLE_DIR
    / "validation_median_threshold_scan.csv",
    index=False,
)


# ============================================================
# 4. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):

    predicted = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predicted,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "specificity": float(
            specificity
        ),
        "negative_predictive_value": float(
            npv
        ),
        "f1": float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predicted,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


# ============================================================
# 5. EVALUATE MEAN AND MEDIAN ENSEMBLES
# ============================================================

all_metrics = {}
performance_rows = []

for dataset_name, table in ensemble_tables.items():

    y_true = (
        table["label"]
        .astype(int)
        .to_numpy()
    )

    mean_probabilities = (
        table[
            "ensemble_probability_mean"
        ].to_numpy()
    )

    median_probabilities = (
        table[
            "ensemble_probability_median"
        ].to_numpy()
    )

    mean_metrics = calculate_metrics(
        y_true=y_true,
        probabilities=mean_probabilities,
        threshold=mean_threshold,
    )

    median_metrics = calculate_metrics(
        y_true=y_true,
        probabilities=median_probabilities,
        threshold=median_threshold,
    )

    all_metrics[dataset_name] = {
        "mean_ensemble": mean_metrics,
        "median_ensemble": median_metrics,
    }

    performance_rows.append({
        "method": "mean_ensemble",
        "dataset": dataset_name,
        **mean_metrics,
    })

    performance_rows.append({
        "method": "median_ensemble",
        "dataset": dataset_name,
        **median_metrics,
    })

    table[
        "mean_ensemble_predicted_label"
    ] = (
        mean_probabilities >= mean_threshold
    ).astype(int)

    table[
        "median_ensemble_predicted_label"
    ] = (
        median_probabilities >= median_threshold
    ).astype(int)

    # Number of individual PLMs voting CPP at threshold 0.5
    model_votes = (
        table[probability_columns]
        .to_numpy()
        >= 0.5
    ).astype(int)

    table[
        "number_of_CPP_votes"
    ] = model_votes.sum(axis=1)

    table[
        "unanimous_model_vote"
    ] = (
        (table["number_of_CPP_votes"] == 0)
        | (table["number_of_CPP_votes"] == 4)
    )

    output_file = (
        ENSEMBLE_DIR
        / f"{dataset_name}_ensemble_predictions.csv"
    )

    table.to_csv(
        output_file,
        index=False,
    )

    print("\n" + "=" * 70)
    print(dataset_name.upper())
    print("=" * 70)

    print("Mean ensemble:")
    print(json.dumps(mean_metrics, indent=2))

    print("\nMedian ensemble:")
    print(json.dumps(median_metrics, indent=2))


# ============================================================
# 6. SAVE MASTER PERFORMANCE TABLE
# ============================================================

ensemble_performance_df = pd.DataFrame(
    performance_rows
)

ensemble_performance_file = (
    RESULT_TABLE_DIR
    / "four_plm_ensemble_performance.csv"
)

ensemble_performance_df.to_csv(
    ensemble_performance_file,
    index=False,
)


# ============================================================
# 7. ADD INDIVIDUAL MODEL PERFORMANCE FOR COMPARISON
# ============================================================

individual_performance_file = (
    RESULT_TABLE_DIR
    / "attention_classifier_performance.csv"
)

individual_performance = pd.read_csv(
    individual_performance_file
)

comparison_rows = []

for _, row in individual_performance.iterrows():

    if row["evaluation"] == (
        "validation_selected_threshold"
    ):
        dataset_name = "validation"

    elif row["evaluation"] == (
        "internal_test_selected_threshold"
    ):
        dataset_name = "internal_test"

    elif row["evaluation"] == (
        "kelm_external_selected_threshold"
    ):
        dataset_name = "kelm_external"

    else:
        continue

    comparison_rows.append({
        "method": row["model"],
        "dataset": dataset_name,
        "threshold": row["threshold"],
        "accuracy": row["accuracy"],
        "balanced_accuracy": row[
            "balanced_accuracy"
        ],
        "f1": row["f1"],
        "mcc": row["mcc"],
        "roc_auc": row["roc_auc"],
        "pr_auc": row["pr_auc"],
    })

for _, row in ensemble_performance_df.iterrows():

    comparison_rows.append({
        "method": row["method"],
        "dataset": row["dataset"],
        "threshold": row["threshold"],
        "accuracy": row["accuracy"],
        "balanced_accuracy": row[
            "balanced_accuracy"
        ],
        "f1": row["f1"],
        "mcc": row["mcc"],
        "roc_auc": row["roc_auc"],
        "pr_auc": row["pr_auc"],
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_file = (
    RESULT_TABLE_DIR
    / "individual_vs_ensemble_comparison.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False,
)


# ============================================================
# 8. SAVE JSON SUMMARY
# ============================================================

summary = {
    "models": MODEL_NAMES,
    "mean_ensemble_threshold": float(
        mean_threshold
    ),
    "median_ensemble_threshold": float(
        median_threshold
    ),
    "metrics": all_metrics,
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    ENSEMBLE_DIR
    / "ensemble_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 9. SAVE CHECKPOINT
# ============================================================

output_files = [
    ensemble_performance_file,
    comparison_file,
    summary_file,
    ENSEMBLE_DIR
    / "validation_ensemble_predictions.csv",
    ENSEMBLE_DIR
    / "internal_test_ensemble_predictions.csv",
    ENSEMBLE_DIR
    / "kelm_external_ensemble_predictions.csv",
]

mark_step_complete(
    "09_four_PLM_ensemble",
    output_files=output_files,
    details={
        "models": MODEL_NAMES,
        "mean_threshold": float(
            mean_threshold
        ),
        "median_threshold": float(
            median_threshold
        ),
        "metrics": all_metrics,
    },
)

print("\n" + "=" * 70)
print("INDIVIDUAL AND ENSEMBLE COMPARISON")
print("=" * 70)

display(
    comparison_df[
        [
            "method",
            "dataset",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "mcc",
            "roc_auc",
            "pr_auc",
        ]
    ].sort_values(
        ["dataset", "mcc"],
        ascending=[True, False],
    )
)

print("\nSaved ensemble performance:")
print(ensemble_performance_file)

print("\nSaved comparison table:")
print(comparison_file)

print("\nSaved ensemble summary:")
print(summary_file)

In [ ]:
# ============================================================
# STEP 10 FIX: Extract attention using eager batch computation
# Compatible with TensorFlow 2.20 / Keras 3
# ============================================================

from pathlib import Path
import gc
import json
import math

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 128,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 96,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 64,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 64,
    },
}

DATASETS_TO_EXPLAIN = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]


# ============================================================
# 2. CUSTOM LAYER FOR LOADING SAVED MODELS
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD EMBEDDING DATASET
# ============================================================

def load_xai_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing file:\n{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    return embeddings, masks, metadata


# ============================================================
# 4. CALCULATE ATTENTION FOR ONE REAL BATCH
# ============================================================

def calculate_attention_batch(
    trained_model,
    embedding_batch,
    mask_batch,
):
    """
    Calculate attention from actual tensors rather than
    symbolic KerasTensor objects.
    """

    embedding_batch = tf.convert_to_tensor(
        embedding_batch,
        dtype=tf.float16,
    )

    mask_batch = tf.convert_to_tensor(
        mask_batch,
    )

    layer_norm = trained_model.get_layer(
        "embedding_layer_norm"
    )

    projection = trained_model.get_layer(
        "residue_projection"
    )

    dropout = trained_model.get_layer(
        "residue_dropout"
    )

    attention_pooling = trained_model.get_layer(
        "masked_attention_pooling"
    )

    # Reproduce the residue features used during inference
    residue_features = layer_norm(
        embedding_batch,
        training=False,
    )

    residue_features = projection(
        residue_features,
        training=False,
    )

    residue_features = dropout(
        residue_features,
        training=False,
    )

    # Learned raw residue-attention scores
    logits = attention_pooling.attention_dense(
        residue_features,
        training=False,
    )

    logits = tf.squeeze(
        logits,
        axis=-1,
    )

    mask_float = tf.cast(
        mask_batch,
        logits.dtype,
    )

    masked_logits = (
        logits
        + (1.0 - mask_float)
        * tf.cast(-1e4, logits.dtype)
    )

    attention_weights = tf.nn.softmax(
        masked_logits,
        axis=1,
    )

    # Ensure padded positions are exactly zero
    attention_weights = (
        attention_weights * mask_float
    )

    row_sums = tf.reduce_sum(
        attention_weights,
        axis=1,
        keepdims=True,
    )

    attention_weights = tf.math.divide_no_nan(
        attention_weights,
        row_sums,
    )

    return (
        attention_weights
        .numpy()
        .astype(np.float32)
    )


# ============================================================
# 5. EXTRACT ONE MODEL / DATASET
# ============================================================

def extract_attention_scores(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        XAI_ROOT
        / model_name
        / "attention"
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    residue_output_file = (
        output_dir
        / "residue_attention_scores.csv"
    )

    matrix_output_file = (
        output_dir
        / "attention_matrix.npy"
    )

    summary_output_file = (
        output_dir
        / "attention_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    # Skip valid completed result
    if (
        complete_file.exists()
        and residue_output_file.exists()
        and matrix_output_file.exists()
        and summary_output_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] {model_name} | "
            f"{dataset_name}"
        )

        return [
            residue_output_file,
            matrix_output_file,
            summary_output_file,
            complete_file,
        ]

    print("\n" + "=" * 75)
    print(f"{model_name} — {dataset_name}")
    print("=" * 75)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Model not found:\n{model_file}"
        )

    trained_model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings, masks, metadata = (
        load_xai_dataset(
            model_name,
            dataset_name,
        )
    )

    number_of_rows = len(metadata)

    temporary_matrix_file = (
        output_dir
        / "attention_matrix.temporary.npy"
    )

    attention_memmap = (
        np.lib.format.open_memmap(
            temporary_matrix_file,
            mode="w+",
            dtype=np.float32,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        embedding_batch = np.asarray(
            embeddings[
                start_index:end_index
            ]
        )

        mask_batch = np.asarray(
            masks[
                start_index:end_index
            ]
        )

        batch_attention = (
            calculate_attention_batch(
                trained_model=trained_model,
                embedding_batch=embedding_batch,
                mask_batch=mask_batch,
            )
        )

        attention_memmap[
            start_index:end_index
        ] = batch_attention

        attention_memmap.flush()

        print(
            f"[ATTENTION] {model_name} | "
            f"{dataset_name} | "
            f"batch {batch_number + 1}/"
            f"{number_of_batches}"
        )

        del embedding_batch
        del mask_batch
        del batch_attention

        gc.collect()

    attention_memmap.flush()
    del attention_memmap

    temporary_matrix_file.replace(
        matrix_output_file
    )

    attention_matrix = np.load(
        matrix_output_file,
        mmap_mode="r",
    )

    # --------------------------------------------------------
    # Convert matrix into one row per residue
    # --------------------------------------------------------

    residue_rows = []

    for row_index, row in metadata.iterrows():
        sequence = str(row["sequence"])
        sequence_length = len(sequence)

        for position_index, residue in enumerate(
            sequence
        ):
            residue_rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "row_index": int(row_index),
                "sequence_id": str(
                    row["sequence_id"]
                ),
                "sequence": sequence,
                "label": int(row["label"]),
                "sequence_length": int(
                    sequence_length
                ),
                "position": int(
                    position_index + 1
                ),
                "normalized_position": float(
                    (
                        position_index + 1
                    )
                    / sequence_length
                ),
                "residue": residue,
                "attention_score": float(
                    attention_matrix[
                        row_index,
                        position_index,
                    ]
                ),
            })

    residue_df = pd.DataFrame(
        residue_rows
    )

    residue_df.to_csv(
        residue_output_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    attention_sums = (
        residue_df
        .groupby("sequence_id")[
            "attention_score"
        ]
        .sum()
        .to_numpy()
    )

    maximum_sum_error = float(
        np.max(
            np.abs(
                attention_sums - 1.0
            )
        )
    )

    if maximum_sum_error > 1e-3:
        raise ValueError(
            "Attention normalization failed. "
            f"Maximum error = {maximum_sum_error}"
        )

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            number_of_rows
        ),
        "number_of_residues": int(
            len(residue_df)
        ),
        "attention_matrix_shape": [
            int(number_of_rows),
            int(MAX_LEN),
        ],
        "maximum_sequence_sum_error": (
            maximum_sum_error
        ),
        "mean_attention_score": float(
            residue_df[
                "attention_score"
            ].mean()
        ),
        "maximum_attention_score": float(
            residue_df[
                "attention_score"
            ].max()
        ),
        "minimum_attention_score": float(
            residue_df[
                "attention_score"
            ].min()
        ),
        "completed_at": (
            pd.Timestamp.now()
            .isoformat()
        ),
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    with open(
        complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    print(
        f"[COMPLETE] {model_name} | "
        f"{dataset_name}"
    )

    print(
        "Sequences:",
        number_of_rows,
        "| Residues:",
        len(residue_df),
        "| Maximum normalization error:",
        maximum_sum_error,
    )

    del attention_matrix
    del embeddings
    del masks
    del trained_model

    tf.keras.backend.clear_session()
    gc.collect()

    return [
        residue_output_file,
        matrix_output_file,
        summary_output_file,
        complete_file,
    ]


# ============================================================
# 6. RUN ALL MODELS AND DATASETS
# ============================================================

all_output_files = []
attention_summary_rows = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in (
        DATASETS_TO_EXPLAIN
    ):
        output_files = (
            extract_attention_scores(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=(
                    config["batch_size"]
                ),
            )
        )

        all_output_files.extend(
            output_files
        )

        summary_file = (
            XAI_ROOT
            / model_name
            / "attention"
            / dataset_name
            / "attention_summary.json"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            attention_summary_rows.append(
                json.load(handle)
            )


# ============================================================
# 7. SAVE MASTER SUMMARY
# ============================================================

attention_summary_df = pd.DataFrame(
    attention_summary_rows
)

master_summary_file = (
    CHECKPOINT_ROOT
    / "attention_extraction_summary.csv"
)

attention_summary_df.to_csv(
    master_summary_file,
    index=False,
)

all_output_files.append(
    master_summary_file
)

mark_step_complete(
    "10_all_attention_scores",
    output_files=all_output_files,
    details={
        "models": list(
            MODEL_CONFIGS.keys()
        ),
        "datasets": (
            DATASETS_TO_EXPLAIN
        ),
        "completed_outputs": int(
            len(attention_summary_rows)
        ),
    },
)

print("\n" + "=" * 75)
print("ATTENTION EXTRACTION SUMMARY")
print("=" * 75)

display(
    attention_summary_df[
        [
            "model",
            "dataset",
            "number_of_sequences",
            "number_of_residues",
            "maximum_sequence_sum_error",
            "maximum_attention_score",
        ]
    ]
)

print("\nSaved master summary:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 11: GRADIENT × INPUT ATTRIBUTION
# TensorFlow 2.20 / Keras 3 compatible
# Restart-safe for all four PLMs
# ============================================================

from pathlib import Path
import gc
import json
import math
import os

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 64,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 48,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 24,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 24,
    },
}

DATASETS_TO_EXPLAIN = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]


# ============================================================
# 2. CUSTOM LAYER REQUIRED FOR MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD EMBEDDINGS, MASKS, AND METADATA
# ============================================================

def load_attribution_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Row mismatch for {model_name}/{dataset_name}"
        )

    return embeddings, masks, metadata


# ============================================================
# 4. FORWARD PASS RETURNING THE PRE-SIGMOID CPP LOGIT
# ============================================================

def forward_cpp_logit(
    trained_model,
    embedding_tensor,
    mask_tensor,
):
    """
    Reproduce the trained classifier forward pass and return
    the pre-sigmoid CPP logit.

    Using the logit rather than probability reduces gradient
    saturation for highly confident predictions.
    """

    layer_norm = trained_model.get_layer(
        "embedding_layer_norm"
    )

    residue_projection = trained_model.get_layer(
        "residue_projection"
    )

    residue_dropout = trained_model.get_layer(
        "residue_dropout"
    )

    attention_pooling = trained_model.get_layer(
        "masked_attention_pooling"
    )

    peptide_dense_1 = trained_model.get_layer(
        "peptide_dense_1"
    )

    peptide_dropout_1 = trained_model.get_layer(
        "peptide_dropout_1"
    )

    peptide_dense_2 = trained_model.get_layer(
        "peptide_dense_2"
    )

    peptide_dropout_2 = trained_model.get_layer(
        "peptide_dropout_2"
    )

    output_layer = trained_model.get_layer(
        "cpp_probability"
    )

    x = tf.cast(
        embedding_tensor,
        tf.float16,
    )

    x = layer_norm(
        x,
        training=False,
    )

    x = residue_projection(
        x,
        training=False,
    )

    x = residue_dropout(
        x,
        training=False,
    )

    pooled = attention_pooling(
        [x, mask_tensor],
        training=False,
    )

    x = peptide_dense_1(
        pooled,
        training=False,
    )

    x = peptide_dropout_1(
        x,
        training=False,
    )

    x = peptide_dense_2(
        x,
        training=False,
    )

    x = peptide_dropout_2(
        x,
        training=False,
    )

    # The saved output layer has sigmoid activation.
    # Apply its trained kernel and bias manually to obtain logits.
    kernel = tf.cast(
        output_layer.kernel,
        x.dtype,
    )

    bias = tf.cast(
        output_layer.bias,
        x.dtype,
    )

    logits = tf.linalg.matmul(
        x,
        kernel,
    ) + bias

    return tf.squeeze(
        logits,
        axis=-1,
    )


# ============================================================
# 5. CALCULATE GRADIENT × INPUT FOR ONE BATCH
# ============================================================

def calculate_gradient_input_batch(
    trained_model,
    embedding_batch,
    mask_batch,
):
    """
    Returns normalized residue scores for one batch.

    Raw residue score:
        sum(abs(gradient * embedding), embedding_dimension)
    """

    # Use float32 watched inputs for more stable gradients.
    input_tensor = tf.convert_to_tensor(
        embedding_batch,
        dtype=tf.float32,
    )

    mask_tensor = tf.convert_to_tensor(
        mask_batch,
        dtype=tf.uint8,
    )

    with tf.GradientTape() as tape:
        tape.watch(input_tensor)

        logits = forward_cpp_logit(
            trained_model=trained_model,
            embedding_tensor=input_tensor,
            mask_tensor=mask_tensor,
        )

        # Sum gives an independent gradient for every batch row,
        # because examples do not interact during inference.
        objective = tf.reduce_sum(
            tf.cast(logits, tf.float32)
        )

    gradients = tape.gradient(
        objective,
        input_tensor,
    )

    if gradients is None:
        raise RuntimeError(
            "Gradient calculation returned None."
        )

    gradient_input = tf.abs(
        gradients * input_tensor
    )

    residue_scores = tf.reduce_sum(
        gradient_input,
        axis=-1,
    )

    mask_float = tf.cast(
        mask_tensor,
        tf.float32,
    )

    residue_scores = (
        residue_scores * mask_float
    )

    # Normalize separately within every peptide.
    row_sums = tf.reduce_sum(
        residue_scores,
        axis=1,
        keepdims=True,
    )

    normalized_scores = tf.math.divide_no_nan(
        residue_scores,
        row_sums,
    )

    return (
        normalized_scores
        .numpy()
        .astype(np.float32)
    )


# ============================================================
# 6. VALIDATE A SAVED BATCH CHUNK
# ============================================================

def valid_saved_chunk(
    chunk_file,
    expected_ids,
    expected_rows,
):
    if not chunk_file.exists():
        return False

    try:
        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        saved_ids = (
            chunk["sequence_id"]
            .astype(str)
        )

        saved_scores = chunk[
            "gradient_input_scores"
        ]

        valid = (
            np.array_equal(
                saved_ids,
                expected_ids.astype(str),
            )
            and saved_scores.shape
            == (
                expected_rows,
                MAX_LEN,
            )
        )

        chunk.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 7. PROCESS ONE MODEL / DATASET
# ============================================================

def extract_gradient_input(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        XAI_ROOT
        / model_name
        / "gradient_input"
        / dataset_name
    )

    chunk_dir = (
        output_dir / "batch_chunks"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    chunk_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    matrix_output_file = (
        output_dir
        / "gradient_input_matrix.npy"
    )

    residue_output_file = (
        output_dir
        / "residue_gradient_input_scores.csv"
    )

    summary_output_file = (
        output_dir
        / "gradient_input_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        complete_file.exists()
        and matrix_output_file.exists()
        and residue_output_file.exists()
        and summary_output_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] {model_name} | "
            f"{dataset_name}"
        )

        return [
            matrix_output_file,
            residue_output_file,
            summary_output_file,
            complete_file,
        ]

    print("\n" + "=" * 75)
    print(
        f"GRADIENT × INPUT: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 75)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Model not found:\n{model_file}"
        )

    trained_model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings, masks, metadata = (
        load_attribution_dataset(
            model_name,
            dataset_name,
        )
    )

    number_of_rows = len(metadata)

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    # --------------------------------------------------------
    # Generate restart-safe chunks
    # --------------------------------------------------------

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        batch_metadata = metadata.iloc[
            start_index:end_index
        ]

        expected_ids = (
            batch_metadata[
                "sequence_id"
            ]
            .astype(str)
            .to_numpy()
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        if valid_saved_chunk(
            chunk_file=chunk_file,
            expected_ids=expected_ids,
            expected_rows=len(batch_metadata),
        ):
            print(
                f"[SKIP CHUNK] {model_name} | "
                f"{dataset_name} | "
                f"{batch_number + 1}/"
                f"{number_of_batches}"
            )

            continue

        if chunk_file.exists():
            chunk_file.unlink()

        embedding_batch = np.asarray(
            embeddings[
                start_index:end_index
            ],
            dtype=np.float32,
        )

        mask_batch = np.asarray(
            masks[
                start_index:end_index
            ],
            dtype=np.uint8,
        )

        try:
            batch_scores = (
                calculate_gradient_input_batch(
                    trained_model=trained_model,
                    embedding_batch=embedding_batch,
                    mask_batch=mask_batch,
                )
            )

        except tf.errors.ResourceExhaustedError:
            raise RuntimeError(
                f"GPU memory error for {model_name}. "
                f"Reduce its batch_size and rerun. "
                f"Completed chunks will be skipped."
            )

        temporary_chunk = Path(
            str(chunk_file) + ".temporary.npz"
        )

        np.savez_compressed(
            temporary_chunk,
            sequence_id=expected_ids,
            gradient_input_scores=batch_scores,
        )

        temporary_chunk.replace(
            chunk_file
        )

        print(
            f"[SAVED CHUNK] {model_name} | "
            f"{dataset_name} | "
            f"{batch_number + 1}/"
            f"{number_of_batches}"
        )

        del embedding_batch
        del mask_batch
        del batch_scores

        gc.collect()

    # --------------------------------------------------------
    # Assemble final attribution matrix
    # --------------------------------------------------------

    temporary_matrix_file = (
        output_dir
        / "gradient_input_matrix.temporary.npy"
    )

    matrix_memmap = (
        np.lib.format.open_memmap(
            temporary_matrix_file,
            mode="w+",
            dtype=np.float32,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        matrix_memmap[
            start_index:end_index
        ] = chunk[
            "gradient_input_scores"
        ]

        chunk.close()

    matrix_memmap.flush()
    del matrix_memmap

    os.replace(
        temporary_matrix_file,
        matrix_output_file,
    )

    score_matrix = np.load(
        matrix_output_file,
        mmap_mode="r",
    )

    # --------------------------------------------------------
    # Create one row per real residue
    # --------------------------------------------------------

    residue_rows = []

    for row_index, row in metadata.iterrows():
        sequence = str(row["sequence"])
        sequence_length = len(sequence)

        for position_index, residue in enumerate(
            sequence
        ):
            residue_rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "row_index": int(row_index),
                "sequence_id": str(
                    row["sequence_id"]
                ),
                "sequence": sequence,
                "label": int(row["label"]),
                "sequence_length": int(
                    sequence_length
                ),
                "position": int(
                    position_index + 1
                ),
                "normalized_position": float(
                    (
                        position_index + 1
                    )
                    / sequence_length
                ),
                "residue": residue,
                "gradient_input_score": float(
                    score_matrix[
                        row_index,
                        position_index,
                    ]
                ),
            })

    residue_df = pd.DataFrame(
        residue_rows
    )

    residue_df.to_csv(
        residue_output_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    sequence_score_sums = (
        residue_df
        .groupby("sequence_id")[
            "gradient_input_score"
        ]
        .sum()
        .to_numpy()
    )

    zero_sum_sequences = int(
        np.sum(
            np.isclose(
                sequence_score_sums,
                0.0,
            )
        )
    )

    nonzero_sums = sequence_score_sums[
        ~np.isclose(
            sequence_score_sums,
            0.0,
        )
    ]

    if len(nonzero_sums) > 0:
        maximum_sum_error = float(
            np.max(
                np.abs(
                    nonzero_sums - 1.0
                )
            )
        )
    else:
        maximum_sum_error = None

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            number_of_rows
        ),
        "number_of_residues": int(
            len(residue_df)
        ),
        "score_matrix_shape": [
            int(number_of_rows),
            int(MAX_LEN),
        ],
        "zero_sum_sequences": int(
            zero_sum_sequences
        ),
        "maximum_nonzero_sequence_sum_error": (
            maximum_sum_error
        ),
        "mean_score": float(
            residue_df[
                "gradient_input_score"
            ].mean()
        ),
        "maximum_score": float(
            residue_df[
                "gradient_input_score"
            ].max()
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file = Path(
        str(complete_file) + ".temporary"
    )

    with open(
        temporary_complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file.replace(
        complete_file
    )

    print(
        f"[COMPLETE] {model_name} | "
        f"{dataset_name}"
    )

    print(
        "Sequences:",
        number_of_rows,
        "| Residues:",
        len(residue_df),
        "| Zero-sum sequences:",
        zero_sum_sequences,
        "| Max normalization error:",
        maximum_sum_error,
    )

    del score_matrix
    del embeddings
    del masks
    del trained_model

    tf.keras.backend.clear_session()
    gc.collect()

    return [
        matrix_output_file,
        residue_output_file,
        summary_output_file,
        complete_file,
    ]


# ============================================================
# 8. RUN ALL MODELS AND DATASETS
# ============================================================

all_output_files = []
summary_rows = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in (
        DATASETS_TO_EXPLAIN
    ):
        output_files = (
            extract_gradient_input(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=config[
                    "batch_size"
                ],
            )
        )

        all_output_files.extend(
            output_files
        )

        summary_file = (
            XAI_ROOT
            / model_name
            / "gradient_input"
            / dataset_name
            / "gradient_input_summary.json"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            summary_rows.append(
                json.load(handle)
            )


# ============================================================
# 9. SAVE MASTER SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

master_summary_file = (
    CHECKPOINT_ROOT
    / "gradient_input_extraction_summary.csv"
)

summary_df.to_csv(
    master_summary_file,
    index=False,
)

all_output_files.append(
    master_summary_file
)

mark_step_complete(
    "11_all_gradient_input_scores",
    output_files=all_output_files,
    details={
        "models": list(
            MODEL_CONFIGS.keys()
        ),
        "datasets": (
            DATASETS_TO_EXPLAIN
        ),
        "completed_outputs": int(
            len(summary_rows)
        ),
    },
)

print("\n" + "=" * 75)
print("GRADIENT × INPUT SUMMARY")
print("=" * 75)

display(
    summary_df[
        [
            "model",
            "dataset",
            "number_of_sequences",
            "number_of_residues",
            "zero_sum_sequences",
            "maximum_nonzero_sequence_sum_error",
            "maximum_score",
        ]
    ]
)

print("\nMaster summary saved to:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 12: INTEGRATED GRADIENTS ATTRIBUTION
# TensorFlow 2.20 / Keras 3 compatible
# Restart-safe for all four PLMs
# ============================================================

from pathlib import Path
import gc
import json
import math
import os

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_CONFIGS = {
    "ESM2_320": {
        "embedding_dimension": 320,
        "batch_size": 16,
    },
    "ESM2_640": {
        "embedding_dimension": 640,
        "batch_size": 12,
    },
    "ESM2_1280": {
        "embedding_dimension": 1280,
        "batch_size": 6,
    },
    "ProtT5": {
        "embedding_dimension": 1024,
        "batch_size": 6,
    },
}

DATASETS_TO_EXPLAIN = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61
IG_STEPS = 32

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]


# ============================================================
# 2. CUSTOM LAYER FOR MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD EMBEDDINGS AND METADATA
# ============================================================

def load_ig_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Row mismatch for {model_name}/{dataset_name}"
        )

    return embeddings, masks, metadata


# ============================================================
# 4. FORWARD PASS TO PRE-SIGMOID CPP LOGIT
# ============================================================

def forward_cpp_logit(
    trained_model,
    embedding_tensor,
    mask_tensor,
):
    layer_norm = trained_model.get_layer(
        "embedding_layer_norm"
    )

    residue_projection = trained_model.get_layer(
        "residue_projection"
    )

    residue_dropout = trained_model.get_layer(
        "residue_dropout"
    )

    attention_pooling = trained_model.get_layer(
        "masked_attention_pooling"
    )

    peptide_dense_1 = trained_model.get_layer(
        "peptide_dense_1"
    )

    peptide_dropout_1 = trained_model.get_layer(
        "peptide_dropout_1"
    )

    peptide_dense_2 = trained_model.get_layer(
        "peptide_dense_2"
    )

    peptide_dropout_2 = trained_model.get_layer(
        "peptide_dropout_2"
    )

    output_layer = trained_model.get_layer(
        "cpp_probability"
    )

    x = tf.cast(
        embedding_tensor,
        tf.float16,
    )

    x = layer_norm(
        x,
        training=False,
    )

    x = residue_projection(
        x,
        training=False,
    )

    x = residue_dropout(
        x,
        training=False,
    )

    pooled = attention_pooling(
        [x, mask_tensor],
        training=False,
    )

    x = peptide_dense_1(
        pooled,
        training=False,
    )

    x = peptide_dropout_1(
        x,
        training=False,
    )

    x = peptide_dense_2(
        x,
        training=False,
    )

    x = peptide_dropout_2(
        x,
        training=False,
    )

    kernel = tf.cast(
        output_layer.kernel,
        x.dtype,
    )

    bias = tf.cast(
        output_layer.bias,
        x.dtype,
    )

    logits = tf.linalg.matmul(
        x,
        kernel,
    ) + bias

    return tf.squeeze(
        logits,
        axis=-1,
    )


# ============================================================
# 5. CALCULATE INTEGRATED GRADIENTS FOR ONE BATCH
# ============================================================

def calculate_integrated_gradients_batch(
    trained_model,
    embedding_batch,
    mask_batch,
    number_of_steps=IG_STEPS,
):
    """
    Integrated Gradients from a zero-embedding baseline.

    For every interpolation point alpha:
        x_alpha = baseline + alpha * (input - baseline)

    The target is the CPP pre-sigmoid logit.
    """

    input_tensor = tf.convert_to_tensor(
        embedding_batch,
        dtype=tf.float32,
    )

    mask_tensor = tf.convert_to_tensor(
        mask_batch,
        dtype=tf.uint8,
    )

    baseline_tensor = tf.zeros_like(
        input_tensor,
        dtype=tf.float32,
    )

    input_difference = (
        input_tensor - baseline_tensor
    )

    accumulated_gradients = tf.zeros_like(
        input_tensor,
        dtype=tf.float32,
    )

    # Midpoint Riemann approximation is more stable than
    # including only the endpoints.
    alpha_values = (
        (
            tf.range(
                number_of_steps,
                dtype=tf.float32,
            )
            + 0.5
        )
        / float(number_of_steps)
    )

    for alpha in alpha_values:
        interpolated_input = (
            baseline_tensor
            + alpha * input_difference
        )

        with tf.GradientTape() as tape:
            tape.watch(
                interpolated_input
            )

            logits = forward_cpp_logit(
                trained_model=trained_model,
                embedding_tensor=interpolated_input,
                mask_tensor=mask_tensor,
            )

            objective = tf.reduce_sum(
                tf.cast(
                    logits,
                    tf.float32,
                )
            )

        gradients = tape.gradient(
            objective,
            interpolated_input,
        )

        if gradients is None:
            raise RuntimeError(
                "Integrated Gradients returned None."
            )

        accumulated_gradients += gradients

    average_gradients = (
        accumulated_gradients
        / float(number_of_steps)
    )

    integrated_gradients = (
        input_difference
        * average_gradients
    )

    # Absolute attribution summed across embedding dimensions.
    residue_scores = tf.reduce_sum(
        tf.abs(
            integrated_gradients
        ),
        axis=-1,
    )

    mask_float = tf.cast(
        mask_tensor,
        tf.float32,
    )

    residue_scores = (
        residue_scores
        * mask_float
    )

    row_sums = tf.reduce_sum(
        residue_scores,
        axis=1,
        keepdims=True,
    )

    normalized_scores = tf.math.divide_no_nan(
        residue_scores,
        row_sums,
    )

    return (
        normalized_scores
        .numpy()
        .astype(np.float32)
    )


# ============================================================
# 6. VALIDATE EXISTING CHUNKS
# ============================================================

def valid_saved_ig_chunk(
    chunk_file,
    expected_ids,
    expected_rows,
):
    if not chunk_file.exists():
        return False

    try:
        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        saved_ids = (
            chunk["sequence_id"]
            .astype(str)
        )

        saved_scores = (
            chunk[
                "integrated_gradients_scores"
            ]
        )

        saved_steps = int(
            chunk["ig_steps"][0]
        )

        valid = (
            np.array_equal(
                saved_ids,
                expected_ids.astype(str),
            )
            and saved_scores.shape
            == (
                expected_rows,
                MAX_LEN,
            )
            and saved_steps == IG_STEPS
        )

        chunk.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 7. PROCESS ONE MODEL / DATASET
# ============================================================

def extract_integrated_gradients(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        XAI_ROOT
        / model_name
        / "integrated_gradients"
        / dataset_name
    )

    chunk_dir = (
        output_dir / "batch_chunks"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    chunk_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    matrix_output_file = (
        output_dir
        / "integrated_gradients_matrix.npy"
    )

    residue_output_file = (
        output_dir
        / "residue_integrated_gradients_scores.csv"
    )

    summary_output_file = (
        output_dir
        / "integrated_gradients_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        complete_file.exists()
        and matrix_output_file.exists()
        and residue_output_file.exists()
        and summary_output_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] {model_name} | "
            f"{dataset_name}"
        )

        return [
            matrix_output_file,
            residue_output_file,
            summary_output_file,
            complete_file,
        ]

    print("\n" + "=" * 75)
    print(
        f"INTEGRATED GRADIENTS: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 75)

    print("Integration steps:", IG_STEPS)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Model not found:\n{model_file}"
        )

    trained_model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings, masks, metadata = (
        load_ig_dataset(
            model_name,
            dataset_name,
        )
    )

    number_of_rows = len(metadata)

    number_of_batches = math.ceil(
        number_of_rows / batch_size
    )

    # --------------------------------------------------------
    # Generate restart-safe batch chunks
    # --------------------------------------------------------

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        batch_metadata = metadata.iloc[
            start_index:end_index
        ]

        expected_ids = (
            batch_metadata[
                "sequence_id"
            ]
            .astype(str)
            .to_numpy()
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        if valid_saved_ig_chunk(
            chunk_file=chunk_file,
            expected_ids=expected_ids,
            expected_rows=len(batch_metadata),
        ):
            print(
                f"[SKIP CHUNK] {model_name} | "
                f"{dataset_name} | "
                f"{batch_number + 1}/"
                f"{number_of_batches}"
            )

            continue

        if chunk_file.exists():
            chunk_file.unlink()

        embedding_batch = np.asarray(
            embeddings[
                start_index:end_index
            ],
            dtype=np.float32,
        )

        mask_batch = np.asarray(
            masks[
                start_index:end_index
            ],
            dtype=np.uint8,
        )

        try:
            batch_scores = (
                calculate_integrated_gradients_batch(
                    trained_model=trained_model,
                    embedding_batch=embedding_batch,
                    mask_batch=mask_batch,
                    number_of_steps=IG_STEPS,
                )
            )

        except tf.errors.ResourceExhaustedError:
            raise RuntimeError(
                f"GPU memory error for {model_name}. "
                f"Reduce batch_size and rerun. "
                f"Completed chunks will be skipped."
            )

        temporary_chunk = Path(
            str(chunk_file) + ".temporary.npz"
        )

        np.savez_compressed(
            temporary_chunk,
            sequence_id=expected_ids,
            integrated_gradients_scores=(
                batch_scores
            ),
            ig_steps=np.array(
                [IG_STEPS],
                dtype=np.int32,
            ),
        )

        temporary_chunk.replace(
            chunk_file
        )

        print(
            f"[SAVED CHUNK] {model_name} | "
            f"{dataset_name} | "
            f"{batch_number + 1}/"
            f"{number_of_batches}"
        )

        del embedding_batch
        del mask_batch
        del batch_scores

        gc.collect()

    # --------------------------------------------------------
    # Assemble final matrix
    # --------------------------------------------------------

    temporary_matrix_file = (
        output_dir
        / "integrated_gradients_matrix.temporary.npy"
    )

    matrix_memmap = (
        np.lib.format.open_memmap(
            temporary_matrix_file,
            mode="w+",
            dtype=np.float32,
            shape=(
                number_of_rows,
                MAX_LEN,
            ),
        )
    )

    for batch_number in range(
        number_of_batches
    ):
        start_index = (
            batch_number * batch_size
        )

        end_index = min(
            start_index + batch_size,
            number_of_rows,
        )

        chunk_file = (
            chunk_dir
            / f"batch_{batch_number:05d}.npz"
        )

        chunk = np.load(
            chunk_file,
            allow_pickle=True,
        )

        matrix_memmap[
            start_index:end_index
        ] = chunk[
            "integrated_gradients_scores"
        ]

        chunk.close()

    matrix_memmap.flush()
    del matrix_memmap

    os.replace(
        temporary_matrix_file,
        matrix_output_file,
    )

    score_matrix = np.load(
        matrix_output_file,
        mmap_mode="r",
    )

    # --------------------------------------------------------
    # Create residue-level table
    # --------------------------------------------------------

    residue_rows = []

    for row_index, row in metadata.iterrows():
        sequence = str(row["sequence"])
        sequence_length = len(sequence)

        for position_index, residue in enumerate(
            sequence
        ):
            residue_rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "row_index": int(row_index),
                "sequence_id": str(
                    row["sequence_id"]
                ),
                "sequence": sequence,
                "label": int(row["label"]),
                "sequence_length": int(
                    sequence_length
                ),
                "position": int(
                    position_index + 1
                ),
                "normalized_position": float(
                    (
                        position_index + 1
                    )
                    / sequence_length
                ),
                "residue": residue,
                "integrated_gradients_score": float(
                    score_matrix[
                        row_index,
                        position_index,
                    ]
                ),
            })

    residue_df = pd.DataFrame(
        residue_rows
    )

    residue_df.to_csv(
        residue_output_file,
        index=False,
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    sequence_score_sums = (
        residue_df
        .groupby("sequence_id")[
            "integrated_gradients_score"
        ]
        .sum()
        .to_numpy()
    )

    zero_sum_sequences = int(
        np.sum(
            np.isclose(
                sequence_score_sums,
                0.0,
            )
        )
    )

    nonzero_sums = sequence_score_sums[
        ~np.isclose(
            sequence_score_sums,
            0.0,
        )
    ]

    if len(nonzero_sums) > 0:
        maximum_sum_error = float(
            np.max(
                np.abs(
                    nonzero_sums - 1.0
                )
            )
        )
    else:
        maximum_sum_error = None

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            number_of_rows
        ),
        "number_of_residues": int(
            len(residue_df)
        ),
        "ig_steps": int(
            IG_STEPS
        ),
        "baseline": (
            "all-zero residue embedding tensor"
        ),
        "target": (
            "pre-sigmoid CPP logit"
        ),
        "score_matrix_shape": [
            int(number_of_rows),
            int(MAX_LEN),
        ],
        "zero_sum_sequences": int(
            zero_sum_sequences
        ),
        "maximum_nonzero_sequence_sum_error": (
            maximum_sum_error
        ),
        "mean_score": float(
            residue_df[
                "integrated_gradients_score"
            ].mean()
        ),
        "maximum_score": float(
            residue_df[
                "integrated_gradients_score"
            ].max()
        ),
        "completed_at": (
            pd.Timestamp.now().isoformat()
        ),
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file = Path(
        str(complete_file) + ".temporary"
    )

    with open(
        temporary_complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    temporary_complete_file.replace(
        complete_file
    )

    print(
        f"[COMPLETE] {model_name} | "
        f"{dataset_name}"
    )

    print(
        "Sequences:",
        number_of_rows,
        "| Residues:",
        len(residue_df),
        "| Zero-sum sequences:",
        zero_sum_sequences,
        "| Maximum normalization error:",
        maximum_sum_error,
    )

    del score_matrix
    del embeddings
    del masks
    del trained_model

    tf.keras.backend.clear_session()
    gc.collect()

    return [
        matrix_output_file,
        residue_output_file,
        summary_output_file,
        complete_file,
    ]


# ============================================================
# 8. RUN ALL MODELS AND DATASETS
# ============================================================

all_output_files = []
summary_rows = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in (
        DATASETS_TO_EXPLAIN
    ):
        output_files = (
            extract_integrated_gradients(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=config[
                    "batch_size"
                ],
            )
        )

        all_output_files.extend(
            output_files
        )

        summary_file = (
            XAI_ROOT
            / model_name
            / "integrated_gradients"
            / dataset_name
            / "integrated_gradients_summary.json"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            summary_rows.append(
                json.load(handle)
            )


# ============================================================
# 9. SAVE MASTER SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

master_summary_file = (
    CHECKPOINT_ROOT
    / "integrated_gradients_extraction_summary.csv"
)

summary_df.to_csv(
    master_summary_file,
    index=False,
)

all_output_files.append(
    master_summary_file
)

mark_step_complete(
    "12_all_integrated_gradients_scores",
    output_files=all_output_files,
    details={
        "models": list(
            MODEL_CONFIGS.keys()
        ),
        "datasets": (
            DATASETS_TO_EXPLAIN
        ),
        "integration_steps": int(
            IG_STEPS
        ),
        "baseline": (
            "all-zero residue embeddings"
        ),
        "completed_outputs": int(
            len(summary_rows)
        ),
    },
)

print("\n" + "=" * 75)
print("INTEGRATED GRADIENTS SUMMARY")
print("=" * 75)

display(
    summary_df[
        [
            "model",
            "dataset",
            "number_of_sequences",
            "number_of_residues",
            "ig_steps",
            "zero_sum_sequences",
            "maximum_nonzero_sequence_sum_error",
            "maximum_score",
        ]
    ]
)

print("\nMaster summary saved to:")
print(master_summary_file)

In [ ]:
# ============================================================
# STEP 13: CONSENSUS XAI ACROSS METHODS AND PLMS
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]
RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)
CHECKPOINT_DIR = DIRS["checkpoints"]

CONSENSUS_ROOT = (
    XAI_ROOT / "consensus"
)

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONSENSUS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. RANK NORMALIZATION WITHIN EACH PEPTIDE
# ============================================================

def rank_normalize_within_sequence(
    dataframe,
    score_column,
):
    """
    Convert residue scores to percentile ranks within
    each peptide.

    Highest-scoring residue approaches 1.0.
    """

    ranked = (
        dataframe
        .groupby("sequence_id")[
            score_column
        ]
        .rank(
            method="average",
            pct=True,
        )
    )

    return ranked.astype(float)


# ============================================================
# 3. LOAD THREE XAI METHODS FOR ONE MODEL/DATASET
# ============================================================

def load_model_xai(
    model_name,
    dataset_name,
):

    attention_file = (
        XAI_ROOT
        / model_name
        / "attention"
        / dataset_name
        / "residue_attention_scores.csv"
    )

    gradient_file = (
        XAI_ROOT
        / model_name
        / "gradient_input"
        / dataset_name
        / "residue_gradient_input_scores.csv"
    )

    ig_file = (
        XAI_ROOT
        / model_name
        / "integrated_gradients"
        / dataset_name
        / "residue_integrated_gradients_scores.csv"
    )

    for required_file in [
        attention_file,
        gradient_file,
        ig_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing XAI file:\n{required_file}"
            )

    attention = pd.read_csv(
        attention_file
    )

    gradient = pd.read_csv(
        gradient_file
    )

    integrated = pd.read_csv(
        ig_file
    )

    identity_columns = [
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
    ]

    merged = attention[
        identity_columns
        + ["attention_score"]
    ].copy()

    merged = merged.merge(
        gradient[
            identity_columns
            + ["gradient_input_score"]
        ],
        on=identity_columns,
        how="inner",
        validate="one_to_one",
    )

    merged = merged.merge(
        integrated[
            identity_columns
            + ["integrated_gradients_score"]
        ],
        on=identity_columns,
        how="inner",
        validate="one_to_one",
    )

    expected_rows = len(attention)

    if len(merged) != expected_rows:
        raise ValueError(
            f"Merge row mismatch for "
            f"{model_name}/{dataset_name}: "
            f"{len(merged)} versus {expected_rows}"
        )

    merged["model"] = model_name
    merged["dataset"] = dataset_name

    return merged


# ============================================================
# 4. CREATE PER-MODEL CONSENSUS
# ============================================================

def calculate_model_consensus(
    model_name,
    dataset_name,
):

    merged = load_model_xai(
        model_name,
        dataset_name,
    )

    merged["attention_rank"] = (
        rank_normalize_within_sequence(
            merged,
            "attention_score",
        )
    )

    merged["gradient_input_rank"] = (
        rank_normalize_within_sequence(
            merged,
            "gradient_input_score",
        )
    )

    merged["integrated_gradients_rank"] = (
        rank_normalize_within_sequence(
            merged,
            "integrated_gradients_score",
        )
    )

    method_rank_columns = [
        "attention_rank",
        "gradient_input_rank",
        "integrated_gradients_rank",
    ]

    merged[
        "model_consensus_score"
    ] = merged[
        method_rank_columns
    ].mean(axis=1)

    merged[
        "method_rank_std"
    ] = merged[
        method_rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    # Number of methods placing residue in top 20%
    merged[
        "methods_top20_count"
    ] = (
        merged[
            method_rank_columns
        ] >= 0.80
    ).sum(axis=1)

    merged[
        "methods_top15_count"
    ] = (
        merged[
            method_rank_columns
        ] >= 0.85
    ).sum(axis=1)

    merged[
        "methods_top10_count"
    ] = (
        merged[
            method_rank_columns
        ] >= 0.90
    ).sum(axis=1)

    merged[
        "model_consensus_rank"
    ] = (
        rank_normalize_within_sequence(
            merged,
            "model_consensus_score",
        )
    )

    return merged


# ============================================================
# 5. METHOD AGREEMENT
# ============================================================

def calculate_method_agreement(
    dataframe,
    model_name,
    dataset_name,
):

    rows = []

    method_pairs = [
        (
            "attention_rank",
            "gradient_input_rank",
            "Attention vs Gradient×Input",
        ),
        (
            "attention_rank",
            "integrated_gradients_rank",
            "Attention vs Integrated Gradients",
        ),
        (
            "gradient_input_rank",
            "integrated_gradients_rank",
            "Gradient×Input vs Integrated Gradients",
        ),
    ]

    for (
        method_a,
        method_b,
        pair_name,
    ) in method_pairs:

        sequence_correlations = []

        for _, group in dataframe.groupby(
            "sequence_id"
        ):
            if len(group) < 3:
                continue

            correlation, _ = spearmanr(
                group[method_a],
                group[method_b],
            )

            if np.isfinite(correlation):
                sequence_correlations.append(
                    correlation
                )

        rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "method_pair": pair_name,
            "number_of_sequences": int(
                len(sequence_correlations)
            ),
            "mean_sequence_spearman": float(
                np.mean(
                    sequence_correlations
                )
            ),
            "median_sequence_spearman": float(
                np.median(
                    sequence_correlations
                )
            ),
            "standard_deviation": float(
                np.std(
                    sequence_correlations,
                    ddof=0,
                )
            ),
        })

    return rows


# ============================================================
# 6. RUN PER-MODEL CONSENSUS
# ============================================================

all_model_consensus = {}
method_agreement_rows = []
all_output_files = []

for dataset_name in DATASETS:

    all_model_consensus[
        dataset_name
    ] = {}

    for model_name in MODEL_NAMES:

        print(
            f"[CONSENSUS] {model_name} | "
            f"{dataset_name}"
        )

        model_consensus = (
            calculate_model_consensus(
                model_name,
                dataset_name,
            )
        )

        model_output_dir = (
            CONSENSUS_ROOT
            / model_name
            / dataset_name
        )

        model_output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model_output_file = (
            model_output_dir
            / "model_method_consensus.csv"
        )

        model_consensus.to_csv(
            model_output_file,
            index=False,
        )

        all_output_files.append(
            model_output_file
        )

        all_model_consensus[
            dataset_name
        ][model_name] = (
            model_consensus
        )

        method_agreement_rows.extend(
            calculate_method_agreement(
                dataframe=model_consensus,
                model_name=model_name,
                dataset_name=dataset_name,
            )
        )


# ============================================================
# 7. FOUR-PLM GLOBAL CONSENSUS
# ============================================================

global_consensus_tables = {}

identity_columns = [
    "sequence_id",
    "sequence",
    "label",
    "sequence_length",
    "position",
    "normalized_position",
    "residue",
]

for dataset_name in DATASETS:

    reference = (
        all_model_consensus[
            dataset_name
        ][MODEL_NAMES[0]][
            identity_columns
        ]
        .copy()
    )

    global_table = reference.copy()

    model_score_columns = []
    model_rank_columns = []

    for model_name in MODEL_NAMES:

        current = (
            all_model_consensus[
                dataset_name
            ][model_name][
                identity_columns
                + [
                    "model_consensus_score",
                    "model_consensus_rank",
                    "method_rank_std",
                    "methods_top20_count",
                ]
            ]
            .copy()
        )

        rename_map = {
            "model_consensus_score":
                f"consensus_score_{model_name}",
            "model_consensus_rank":
                f"consensus_rank_{model_name}",
            "method_rank_std":
                f"method_rank_std_{model_name}",
            "methods_top20_count":
                f"methods_top20_count_{model_name}",
        }

        current = current.rename(
            columns=rename_map
        )

        global_table = global_table.merge(
            current,
            on=identity_columns,
            how="inner",
            validate="one_to_one",
        )

        model_score_columns.append(
            f"consensus_score_{model_name}"
        )

        model_rank_columns.append(
            f"consensus_rank_{model_name}"
        )

    global_table[
        "global_consensus_score"
    ] = global_table[
        model_rank_columns
    ].mean(axis=1)

    global_table[
        "cross_model_rank_std"
    ] = global_table[
        model_rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    global_table[
        "models_top20_count"
    ] = (
        global_table[
            model_rank_columns
        ] >= 0.80
    ).sum(axis=1)

    global_table[
        "models_top15_count"
    ] = (
        global_table[
            model_rank_columns
        ] >= 0.85
    ).sum(axis=1)

    global_table[
        "models_top10_count"
    ] = (
        global_table[
            model_rank_columns
        ] >= 0.90
    ).sum(axis=1)

    global_table[
        "global_consensus_rank"
    ] = (
        rank_normalize_within_sequence(
            global_table,
            "global_consensus_score",
        )
    )

    global_table[
        "hotspot_top20"
    ] = (
        global_table[
            "global_consensus_rank"
        ] >= 0.80
    )

    global_table[
        "hotspot_top15"
    ] = (
        global_table[
            "global_consensus_rank"
        ] >= 0.85
    )

    global_table[
        "hotspot_top10"
    ] = (
        global_table[
            "global_consensus_rank"
        ] >= 0.90
    )

    # Strict consensus: top 20% in at least 3/4 models
    global_table[
        "strict_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ] >= 3
    )

    # Very strict: top 20% in all four models
    global_table[
        "unanimous_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ] == 4
    )

    output_dir = (
        CONSENSUS_ROOT
        / "global"
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    global_output_file = (
        output_dir
        / "global_consensus_residue_scores.csv"
    )

    global_table.to_csv(
        global_output_file,
        index=False,
    )

    all_output_files.append(
        global_output_file
    )

    global_consensus_tables[
        dataset_name
    ] = global_table


# ============================================================
# 8. GLOBAL CONSENSUS SUMMARY
# ============================================================

summary_rows = []

for dataset_name, table in (
    global_consensus_tables.items()
):

    total_residues = len(table)

    for label_value, label_name in [
        (0, "non_CPP"),
        (1, "CPP"),
    ]:

        subset = table[
            table["label"] == label_value
        ]

        summary_rows.append({
            "dataset": dataset_name,
            "class": label_name,
            "number_of_sequences": int(
                subset[
                    "sequence_id"
                ].nunique()
            ),
            "number_of_residues": int(
                len(subset)
            ),
            "top20_hotspots": int(
                subset[
                    "hotspot_top20"
                ].sum()
            ),
            "strict_cross_model_hotspots": int(
                subset[
                    "strict_cross_model_hotspot"
                ].sum()
            ),
            "unanimous_cross_model_hotspots": int(
                subset[
                    "unanimous_cross_model_hotspot"
                ].sum()
            ),
            "mean_cross_model_rank_std": float(
                subset[
                    "cross_model_rank_std"
                ].mean()
            ),
            "mean_global_consensus_score": float(
                subset[
                    "global_consensus_score"
                ].mean()
            ),
        })


summary_df = pd.DataFrame(
    summary_rows
)

summary_file = (
    RESULT_TABLE_DIR
    / "consensus_xai_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
)

all_output_files.append(
    summary_file
)


# ============================================================
# 9. METHOD AGREEMENT TABLE
# ============================================================

method_agreement_df = pd.DataFrame(
    method_agreement_rows
)

method_agreement_file = (
    RESULT_TABLE_DIR
    / "xai_method_agreement.csv"
)

method_agreement_df.to_csv(
    method_agreement_file,
    index=False,
)

all_output_files.append(
    method_agreement_file
)


# ============================================================
# 10. CROSS-MODEL AGREEMENT
# ============================================================

cross_model_rows = []

for dataset_name, table in (
    global_consensus_tables.items()
):

    rank_columns = {
        model_name:
            f"consensus_rank_{model_name}"
        for model_name in MODEL_NAMES
    }

    for index_a in range(
        len(MODEL_NAMES)
    ):
        for index_b in range(
            index_a + 1,
            len(MODEL_NAMES),
        ):

            model_a = MODEL_NAMES[index_a]
            model_b = MODEL_NAMES[index_b]

            sequence_correlations = []

            for _, group in table.groupby(
                "sequence_id"
            ):
                if len(group) < 3:
                    continue

                correlation, _ = spearmanr(
                    group[
                        rank_columns[model_a]
                    ],
                    group[
                        rank_columns[model_b]
                    ],
                )

                if np.isfinite(correlation):
                    sequence_correlations.append(
                        correlation
                    )

            cross_model_rows.append({
                "dataset": dataset_name,
                "model_a": model_a,
                "model_b": model_b,
                "number_of_sequences": int(
                    len(sequence_correlations)
                ),
                "mean_sequence_spearman": float(
                    np.mean(
                        sequence_correlations
                    )
                ),
                "median_sequence_spearman": float(
                    np.median(
                        sequence_correlations
                    )
                ),
                "standard_deviation": float(
                    np.std(
                        sequence_correlations,
                        ddof=0,
                    )
                ),
            })


cross_model_df = pd.DataFrame(
    cross_model_rows
)

cross_model_file = (
    RESULT_TABLE_DIR
    / "cross_model_xai_agreement.csv"
)

cross_model_df.to_csv(
    cross_model_file,
    index=False,
)

all_output_files.append(
    cross_model_file
)


# ============================================================
# 11. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "13_consensus_XAI",
    output_files=all_output_files,
    details={
        "models": MODEL_NAMES,
        "methods": [
            "attention",
            "gradient_input",
            "integrated_gradients",
        ],
        "datasets": DATASETS,
        "hotspot_thresholds": [
            0.80,
            0.85,
            0.90,
        ],
    },
)


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("CONSENSUS XAI SUMMARY")
print("=" * 75)

display(summary_df)

print("\n" + "=" * 75)
print("METHOD AGREEMENT")
print("=" * 75)

display(
    method_agreement_df[
        [
            "model",
            "dataset",
            "method_pair",
            "mean_sequence_spearman",
            "median_sequence_spearman",
        ]
    ]
)

print("\n" + "=" * 75)
print("CROSS-MODEL AGREEMENT")
print("=" * 75)

display(
    cross_model_df[
        [
            "dataset",
            "model_a",
            "model_b",
            "mean_sequence_spearman",
            "median_sequence_spearman",
        ]
    ]
)

print("\nSaved consensus summary:")
print(summary_file)

print("\nSaved method agreement:")
print(method_agreement_file)

print("\nSaved cross-model agreement:")
print(cross_model_file)

In [ ]:
# ============================================================
# STEP 13B: REDUNDANCY-ADJUSTED CONSENSUS XAI
#
# Rationale:
# Gradient × Input and Integrated Gradients were almost
# perfectly correlated. They are therefore combined into one
# gradient-family score before integration with Attention.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]
RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)
CHECKPOINT_DIR = DIRS["checkpoints"]

ADJUSTED_ROOT = (
    XAI_ROOT / "consensus_adjusted"
)

ADJUSTED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. HELPER: RANK WITHIN EACH PEPTIDE
# ============================================================

def rank_within_sequence(
    dataframe,
    score_column,
):
    return (
        dataframe
        .groupby("sequence_id")[
            score_column
        ]
        .rank(
            method="average",
            pct=True,
        )
        .astype(float)
    )


# ============================================================
# 3. LOAD ORIGINAL METHOD-LEVEL CONSENSUS FILE
# ============================================================

def load_original_model_consensus(
    model_name,
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus"
        / model_name
        / dataset_name
        / "model_method_consensus.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Missing original consensus file:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "attention_rank",
        "gradient_input_rank",
        "integrated_gradients_rank",
    }

    missing = (
        required_columns
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            f"{model_name}/{dataset_name} "
            f"is missing columns: {sorted(missing)}"
        )

    return dataframe


# ============================================================
# 4. CREATE ADJUSTED PER-MODEL CONSENSUS
# ============================================================

adjusted_model_tables = {}
output_files = []

for dataset_name in DATASETS:

    adjusted_model_tables[
        dataset_name
    ] = {}

    for model_name in MODEL_NAMES:

        print(
            f"[ADJUSTED CONSENSUS] "
            f"{model_name} | {dataset_name}"
        )

        dataframe = (
            load_original_model_consensus(
                model_name,
                dataset_name,
            )
        )

        # Combine the highly redundant gradient methods.
        dataframe[
            "gradient_family_rank"
        ] = dataframe[
            [
                "gradient_input_rank",
                "integrated_gradients_rank",
            ]
        ].mean(axis=1)

        # Give equal weight to Attention and gradient family.
        dataframe[
            "adjusted_model_consensus_score"
        ] = (
            dataframe["attention_rank"]
            + dataframe[
                "gradient_family_rank"
            ]
        ) / 2.0

        dataframe[
            "adjusted_model_consensus_rank"
        ] = rank_within_sequence(
            dataframe,
            "adjusted_model_consensus_score",
        )

        # Difference between the two attribution families.
        dataframe[
            "attention_gradient_disagreement"
        ] = np.abs(
            dataframe["attention_rank"]
            - dataframe[
                "gradient_family_rank"
            ]
        )

        dataframe[
            "attention_top20"
        ] = (
            dataframe["attention_rank"]
            >= 0.80
        )

        dataframe[
            "gradient_family_top20"
        ] = (
            dataframe[
                "gradient_family_rank"
            ]
            >= 0.80
        )

        dataframe[
            "both_families_top20"
        ] = (
            dataframe["attention_top20"]
            & dataframe[
                "gradient_family_top20"
            ]
        )

        dataframe[
            "adjusted_hotspot_top20"
        ] = (
            dataframe[
                "adjusted_model_consensus_rank"
            ]
            >= 0.80
        )

        dataframe[
            "adjusted_hotspot_top15"
        ] = (
            dataframe[
                "adjusted_model_consensus_rank"
            ]
            >= 0.85
        )

        dataframe[
            "adjusted_hotspot_top10"
        ] = (
            dataframe[
                "adjusted_model_consensus_rank"
            ]
            >= 0.90
        )

        output_dir = (
            ADJUSTED_ROOT
            / model_name
            / dataset_name
        )

        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        output_file = (
            output_dir
            / "adjusted_model_consensus.csv"
        )

        dataframe.to_csv(
            output_file,
            index=False,
        )

        adjusted_model_tables[
            dataset_name
        ][model_name] = dataframe

        output_files.append(
            output_file
        )


# ============================================================
# 5. CREATE ADJUSTED GLOBAL CROSS-MODEL CONSENSUS
# ============================================================

identity_columns = [
    "sequence_id",
    "sequence",
    "label",
    "sequence_length",
    "position",
    "normalized_position",
    "residue",
]

global_tables = {}

for dataset_name in DATASETS:

    reference = (
        adjusted_model_tables[
            dataset_name
        ][MODEL_NAMES[0]][
            identity_columns
        ]
        .copy()
    )

    global_table = reference.copy()

    model_rank_columns = []
    model_score_columns = []

    for model_name in MODEL_NAMES:

        current = (
            adjusted_model_tables[
                dataset_name
            ][model_name][
                identity_columns
                + [
                    "adjusted_model_consensus_score",
                    "adjusted_model_consensus_rank",
                    "attention_gradient_disagreement",
                    "both_families_top20",
                ]
            ]
            .copy()
        )

        current = current.rename(
            columns={
                "adjusted_model_consensus_score":
                    f"adjusted_score_{model_name}",
                "adjusted_model_consensus_rank":
                    f"adjusted_rank_{model_name}",
                "attention_gradient_disagreement":
                    f"family_disagreement_{model_name}",
                "both_families_top20":
                    f"both_families_top20_{model_name}",
            }
        )

        global_table = global_table.merge(
            current,
            on=identity_columns,
            how="inner",
            validate="one_to_one",
        )

        model_rank_columns.append(
            f"adjusted_rank_{model_name}"
        )

        model_score_columns.append(
            f"adjusted_score_{model_name}"
        )

    # Equal weight to each PLM.
    global_table[
        "adjusted_global_consensus_score"
    ] = global_table[
        model_rank_columns
    ].mean(axis=1)

    global_table[
        "adjusted_global_consensus_rank"
    ] = rank_within_sequence(
        global_table,
        "adjusted_global_consensus_score",
    )

    global_table[
        "cross_model_rank_std"
    ] = global_table[
        model_rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    global_table[
        "models_top20_count"
    ] = (
        global_table[
            model_rank_columns
        ]
        >= 0.80
    ).sum(axis=1)

    global_table[
        "models_top15_count"
    ] = (
        global_table[
            model_rank_columns
        ]
        >= 0.85
    ).sum(axis=1)

    global_table[
        "models_top10_count"
    ] = (
        global_table[
            model_rank_columns
        ]
        >= 0.90
    ).sum(axis=1)

    global_table[
        "adjusted_hotspot_top20"
    ] = (
        global_table[
            "adjusted_global_consensus_rank"
        ]
        >= 0.80
    )

    global_table[
        "adjusted_hotspot_top15"
    ] = (
        global_table[
            "adjusted_global_consensus_rank"
        ]
        >= 0.85
    )

    global_table[
        "adjusted_hotspot_top10"
    ] = (
        global_table[
            "adjusted_global_consensus_rank"
        ]
        >= 0.90
    )

    global_table[
        "strict_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ]
        >= 3
    )

    global_table[
        "unanimous_cross_model_hotspot"
    ] = (
        global_table[
            "models_top20_count"
        ]
        == 4
    )

    global_table[
        "mean_family_disagreement"
    ] = global_table[
        [
            f"family_disagreement_{model}"
            for model in MODEL_NAMES
        ]
    ].mean(axis=1)

    global_output_dir = (
        ADJUSTED_ROOT
        / "global"
        / dataset_name
    )

    global_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    global_output_file = (
        global_output_dir
        / "adjusted_global_consensus_residue_scores.csv"
    )

    global_table.to_csv(
        global_output_file,
        index=False,
    )

    output_files.append(
        global_output_file
    )

    global_tables[
        dataset_name
    ] = global_table


# ============================================================
# 6. COMPARE ORIGINAL AND ADJUSTED CONSENSUS
# ============================================================

comparison_rows = []

for dataset_name in DATASETS:

    original_file = (
        XAI_ROOT
        / "consensus"
        / "global"
        / dataset_name
        / "global_consensus_residue_scores.csv"
    )

    original = pd.read_csv(
        original_file
    )

    adjusted = global_tables[
        dataset_name
    ]

    comparison = original[
        identity_columns
        + [
            "global_consensus_rank",
            "hotspot_top20",
            "strict_cross_model_hotspot",
        ]
    ].merge(
        adjusted[
            identity_columns
            + [
                "adjusted_global_consensus_rank",
                "adjusted_hotspot_top20",
                "strict_cross_model_hotspot",
            ]
        ],
        on=identity_columns,
        suffixes=(
            "_original",
            "_adjusted",
        ),
        validate="one_to_one",
    )

    overall_correlation, _ = spearmanr(
        comparison[
            "global_consensus_rank"
        ],
        comparison[
            "adjusted_global_consensus_rank"
        ],
    )

    original_hotspots = (
        comparison[
            "hotspot_top20"
        ].astype(bool)
    )

    adjusted_hotspots = (
        comparison[
            "adjusted_hotspot_top20"
        ].astype(bool)
    )

    intersection = int(
        (
            original_hotspots
            & adjusted_hotspots
        ).sum()
    )

    union = int(
        (
            original_hotspots
            | adjusted_hotspots
        ).sum()
    )

    jaccard = (
        intersection / union
        if union > 0
        else np.nan
    )

    sequence_correlations = []

    for _, group in comparison.groupby(
        "sequence_id"
    ):
        if len(group) < 3:
            continue

        correlation, _ = spearmanr(
            group[
                "global_consensus_rank"
            ],
            group[
                "adjusted_global_consensus_rank"
            ],
        )

        if np.isfinite(correlation):
            sequence_correlations.append(
                correlation
            )

    comparison_rows.append({
        "dataset": dataset_name,
        "number_of_residues": int(
            len(comparison)
        ),
        "overall_rank_spearman": float(
            overall_correlation
        ),
        "mean_within_sequence_spearman": float(
            np.mean(
                sequence_correlations
            )
        ),
        "median_within_sequence_spearman": float(
            np.median(
                sequence_correlations
            )
        ),
        "original_top20_residues": int(
            original_hotspots.sum()
        ),
        "adjusted_top20_residues": int(
            adjusted_hotspots.sum()
        ),
        "top20_intersection": int(
            intersection
        ),
        "top20_jaccard": float(
            jaccard
        ),
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_file = (
    RESULT_TABLE_DIR
    / "original_vs_adjusted_consensus.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False,
)

output_files.append(
    comparison_file
)


# ============================================================
# 7. ADJUSTED CONSENSUS SUMMARY
# ============================================================

summary_rows = []

for dataset_name, table in (
    global_tables.items()
):

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
    ]:

        subset = table[
            table["label"]
            == label_value
        ]

        summary_rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "number_of_sequences": int(
                subset[
                    "sequence_id"
                ].nunique()
            ),
            "number_of_residues": int(
                len(subset)
            ),
            "adjusted_top20_hotspots": int(
                subset[
                    "adjusted_hotspot_top20"
                ].sum()
            ),
            "strict_cross_model_hotspots": int(
                subset[
                    "strict_cross_model_hotspot"
                ].sum()
            ),
            "unanimous_cross_model_hotspots": int(
                subset[
                    "unanimous_cross_model_hotspot"
                ].sum()
            ),
            "mean_cross_model_rank_std": float(
                subset[
                    "cross_model_rank_std"
                ].mean()
            ),
            "mean_family_disagreement": float(
                subset[
                    "mean_family_disagreement"
                ].mean()
            ),
        })


summary_df = pd.DataFrame(
    summary_rows
)

summary_file = (
    RESULT_TABLE_DIR
    / "adjusted_consensus_xai_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
)

output_files.append(
    summary_file
)


# ============================================================
# 8. SAVE METHODOLOGICAL NOTE
# ============================================================

method_note = {
    "reason_for_adjustment": (
        "Gradient × Input and Integrated Gradients "
        "showed near-perfect residue-rank agreement and were "
        "therefore treated as one gradient-attribution family."
    ),
    "gradient_family_definition": (
        "mean of Gradient × Input percentile rank and "
        "Integrated Gradients percentile rank"
    ),
    "per_model_consensus_definition": (
        "equal-weight mean of Attention rank and "
        "gradient-family rank"
    ),
    "global_consensus_definition": (
        "equal-weight mean of adjusted per-model ranks "
        "across four PLMs"
    ),
    "models": MODEL_NAMES,
    "datasets": DATASETS,
}

method_note_file = (
    CHECKPOINT_DIR
    / "adjusted_consensus_method.json"
)

with open(
    method_note_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        method_note,
        handle,
        indent=2,
    )

output_files.append(
    method_note_file
)


# ============================================================
# 9. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "13B_redundancy_adjusted_consensus",
    output_files=output_files,
    details=method_note,
)


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("ORIGINAL VS ADJUSTED CONSENSUS")
print("=" * 75)

display(comparison_df)

print("\n" + "=" * 75)
print("ADJUSTED CONSENSUS SUMMARY")
print("=" * 75)

display(summary_df)

print("\nAdjusted global consensus files:")

for dataset_name in DATASETS:
    print(
        ADJUSTED_ROOT
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

print("\nComparison table:")
print(comparison_file)

print("\nAdjusted summary:")
print(summary_file)

In [ ]:
# ============================================================
# STEP 14: RESIDUE ENRICHMENT IN ADJUSTED CONSENSUS HOTSPOTS
#
# Analyses:
# 1. Within CPPs:
#    hotspot residues vs non-hotspot CPP residues
#
# 2. Between classes:
#    CPP hotspot residues vs non-CPP hotspot residues
#
# Thresholds:
#    Top 10%, 15%, and 20%
#
# Statistics:
#    Fisher exact test
#    Odds ratio
#    Benjamini-Hochberg FDR correction
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from scipy.stats import fisher_exact


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

AMINO_ACIDS = list(
    "ACDEFGHIKLMNPQRSTVWY"
)

HOTSPOT_DEFINITIONS = {
    "top10": "adjusted_hotspot_top10",
    "top15": "adjusted_hotspot_top15",
    "top20": "adjusted_hotspot_top20",
    "strict_cross_model": (
        "strict_cross_model_hotspot"
    ),
    "unanimous_cross_model": (
        "unanimous_cross_model_hotspot"
    ),
}

XAI_ROOT = DIRS["xai"]
RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)
RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)
CHECKPOINT_DIR = DIRS["checkpoints"]

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_SI_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. BENJAMINI-HOCHBERG FDR CORRECTION
# ============================================================

def benjamini_hochberg(
    p_values
):
    """
    Return Benjamini-Hochberg adjusted p-values.
    """

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(p_values)

    order = np.argsort(
        p_values
    )

    ranked_p_values = (
        p_values[order]
    )

    adjusted_ranked = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ranked_p_values[
                reverse_index
            ]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ranked[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


# ============================================================
# 3. SAFE ODDS RATIO WITH HALDANE-ANSCOMBE CORRECTION
# ============================================================

def corrected_odds_ratio(
    a,
    b,
    c,
    d,
):
    """
    Calculate an odds ratio using a 0.5 continuity correction.

    Table:
                 Residue AA   Other residues
        Group 1      a             b
        Group 2      c             d
    """

    return (
        (a + 0.5)
        * (d + 0.5)
        / (
            (b + 0.5)
            * (c + 0.5)
        )
    )


# ============================================================
# 4. LOAD ADJUSTED GLOBAL CONSENSUS
# ============================================================

def load_adjusted_consensus(
    dataset_name
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file not found:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "position",
        "residue",
        "adjusted_global_consensus_rank",
        *HOTSPOT_DEFINITIONS.values(),
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    dataframe["residue"] = (
        dataframe["residue"]
        .astype(str)
        .str.upper()
    )

    invalid_residues = sorted(
        set(dataframe["residue"])
        - set(AMINO_ACIDS)
    )

    if invalid_residues:
        raise ValueError(
            "Unexpected amino-acid symbols: "
            f"{invalid_residues}"
        )

    return dataframe, input_file


# ============================================================
# 5. WITHIN-CPP HOTSPOT ENRICHMENT
# ============================================================

def calculate_within_cpp_enrichment(
    dataframe,
    dataset_name,
    hotspot_name,
    hotspot_column,
):
    """
    Compare CPP hotspot residues against non-hotspot
    residues from the same CPP sequences.
    """

    cpp_data = dataframe[
        dataframe["label"] == 1
    ].copy()

    hotspot_data = cpp_data[
        cpp_data[hotspot_column].astype(bool)
    ]

    non_hotspot_data = cpp_data[
        ~cpp_data[hotspot_column].astype(bool)
    ]

    hotspot_total = len(
        hotspot_data
    )

    non_hotspot_total = len(
        non_hotspot_data
    )

    rows = []

    for amino_acid in AMINO_ACIDS:

        hotspot_aa = int(
            (
                hotspot_data["residue"]
                == amino_acid
            ).sum()
        )

        hotspot_other = int(
            hotspot_total
            - hotspot_aa
        )

        non_hotspot_aa = int(
            (
                non_hotspot_data[
                    "residue"
                ]
                == amino_acid
            ).sum()
        )

        non_hotspot_other = int(
            non_hotspot_total
            - non_hotspot_aa
        )

        contingency_table = [
            [
                hotspot_aa,
                hotspot_other,
            ],
            [
                non_hotspot_aa,
                non_hotspot_other,
            ],
        ]

        scipy_odds_ratio, p_value = (
            fisher_exact(
                contingency_table,
                alternative="two-sided",
            )
        )

        corrected_or = (
            corrected_odds_ratio(
                hotspot_aa,
                hotspot_other,
                non_hotspot_aa,
                non_hotspot_other,
            )
        )

        hotspot_frequency = (
            hotspot_aa
            / hotspot_total
            if hotspot_total > 0
            else np.nan
        )

        non_hotspot_frequency = (
            non_hotspot_aa
            / non_hotspot_total
            if non_hotspot_total > 0
            else np.nan
        )

        log2_enrichment = np.log2(
            (
                hotspot_frequency
                + 1e-12
            )
            / (
                non_hotspot_frequency
                + 1e-12
            )
        )

        rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_CPP_nonhotspot"
            ),
            "hotspot_definition": (
                hotspot_name
            ),
            "residue": amino_acid,
            "group1_count": hotspot_aa,
            "group1_total": hotspot_total,
            "group1_frequency": float(
                hotspot_frequency
            ),
            "group2_count": (
                non_hotspot_aa
            ),
            "group2_total": (
                non_hotspot_total
            ),
            "group2_frequency": float(
                non_hotspot_frequency
            ),
            "odds_ratio_scipy": float(
                scipy_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_or
            ),
            "log2_enrichment": float(
                log2_enrichment
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    result["fdr_bh"] = (
        benjamini_hochberg(
            result["p_value"]
            .to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "enrichment_direction"
    ] = np.where(
        result[
            "odds_ratio_corrected"
        ] > 1.0,
        "enriched_in_hotspots",
        "depleted_in_hotspots",
    )

    return result


# ============================================================
# 6. CPP VS NON-CPP HOTSPOT ENRICHMENT
# ============================================================

def calculate_between_class_enrichment(
    dataframe,
    dataset_name,
    hotspot_name,
    hotspot_column,
):
    """
    Compare residue composition of CPP hotspots against
    residue composition of non-CPP hotspots.
    """

    hotspot_data = dataframe[
        dataframe[
            hotspot_column
        ].astype(bool)
    ].copy()

    cpp_hotspots = hotspot_data[
        hotspot_data["label"] == 1
    ]

    noncpp_hotspots = hotspot_data[
        hotspot_data["label"] == 0
    ]

    cpp_total = len(
        cpp_hotspots
    )

    noncpp_total = len(
        noncpp_hotspots
    )

    rows = []

    for amino_acid in AMINO_ACIDS:

        cpp_aa = int(
            (
                cpp_hotspots["residue"]
                == amino_acid
            ).sum()
        )

        cpp_other = int(
            cpp_total
            - cpp_aa
        )

        noncpp_aa = int(
            (
                noncpp_hotspots[
                    "residue"
                ]
                == amino_acid
            ).sum()
        )

        noncpp_other = int(
            noncpp_total
            - noncpp_aa
        )

        contingency_table = [
            [
                cpp_aa,
                cpp_other,
            ],
            [
                noncpp_aa,
                noncpp_other,
            ],
        ]

        scipy_odds_ratio, p_value = (
            fisher_exact(
                contingency_table,
                alternative="two-sided",
            )
        )

        corrected_or = (
            corrected_odds_ratio(
                cpp_aa,
                cpp_other,
                noncpp_aa,
                noncpp_other,
            )
        )

        cpp_frequency = (
            cpp_aa / cpp_total
            if cpp_total > 0
            else np.nan
        )

        noncpp_frequency = (
            noncpp_aa
            / noncpp_total
            if noncpp_total > 0
            else np.nan
        )

        log2_enrichment = np.log2(
            (
                cpp_frequency
                + 1e-12
            )
            / (
                noncpp_frequency
                + 1e-12
            )
        )

        rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            "hotspot_definition": (
                hotspot_name
            ),
            "residue": amino_acid,
            "group1_count": cpp_aa,
            "group1_total": cpp_total,
            "group1_frequency": float(
                cpp_frequency
            ),
            "group2_count": (
                noncpp_aa
            ),
            "group2_total": (
                noncpp_total
            ),
            "group2_frequency": float(
                noncpp_frequency
            ),
            "odds_ratio_scipy": float(
                scipy_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_or
            ),
            "log2_enrichment": float(
                log2_enrichment
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    result["fdr_bh"] = (
        benjamini_hochberg(
            result["p_value"]
            .to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "enrichment_direction"
    ] = np.where(
        result[
            "odds_ratio_corrected"
        ] > 1.0,
        "enriched_in_CPP_hotspots",
        "enriched_in_nonCPP_hotspots",
    )

    return result


# ============================================================
# 7. RUN ALL ENRICHMENT ANALYSES
# ============================================================

all_results = []
source_files = []

for dataset_name in DATASETS:

    consensus_df, source_file = (
        load_adjusted_consensus(
            dataset_name
        )
    )

    source_files.append(
        source_file
    )

    print("\n" + "=" * 75)
    print(
        f"RESIDUE ENRICHMENT: "
        f"{dataset_name}"
    )
    print("=" * 75)

    for (
        hotspot_name,
        hotspot_column,
    ) in HOTSPOT_DEFINITIONS.items():

        print(
            f"[ANALYSIS] {dataset_name} | "
            f"{hotspot_name}"
        )

        within_cpp = (
            calculate_within_cpp_enrichment(
                dataframe=consensus_df,
                dataset_name=dataset_name,
                hotspot_name=hotspot_name,
                hotspot_column=(
                    hotspot_column
                ),
            )
        )

        between_classes = (
            calculate_between_class_enrichment(
                dataframe=consensus_df,
                dataset_name=dataset_name,
                hotspot_name=hotspot_name,
                hotspot_column=(
                    hotspot_column
                ),
            )
        )

        all_results.append(
            within_cpp
        )

        all_results.append(
            between_classes
        )


enrichment_df = pd.concat(
    all_results,
    ignore_index=True,
)


# ============================================================
# 8. SAVE COMPLETE TABLE
# ============================================================

complete_output_file = (
    RESULT_SI_DIR
    / "residue_enrichment_all_thresholds.csv"
)

enrichment_df.to_csv(
    complete_output_file,
    index=False,
)


# ============================================================
# 9. CREATE MAIN TOP-20 SUMMARY
# ============================================================

main_summary = enrichment_df[
    enrichment_df[
        "hotspot_definition"
    ] == "top20"
].copy()

main_summary = main_summary.sort_values(
    [
        "dataset",
        "analysis",
        "fdr_bh",
        "odds_ratio_corrected",
    ],
    ascending=[
        True,
        True,
        True,
        False,
    ],
)

main_summary_file = (
    RESULT_TABLE_DIR
    / "residue_enrichment_top20.csv"
)

main_summary.to_csv(
    main_summary_file,
    index=False,
)


# ============================================================
# 10. SIGNIFICANT RESULTS ONLY
# ============================================================

significant_df = enrichment_df[
    enrichment_df[
        "significant_fdr_0_05"
    ]
].copy()

significant_df = significant_df.sort_values(
    [
        "dataset",
        "analysis",
        "hotspot_definition",
        "fdr_bh",
    ]
)

significant_file = (
    RESULT_TABLE_DIR
    / "significant_residue_enrichment.csv"
)

significant_df.to_csv(
    significant_file,
    index=False,
)


# ============================================================
# 11. CROSS-DATASET REPLICATION
# ============================================================

replication_source = enrichment_df[
    (
        enrichment_df[
            "hotspot_definition"
        ] == "top20"
    )
    & (
        enrichment_df[
            "analysis"
        ] == (
            "CPP_hotspot_vs_CPP_nonhotspot"
        )
    )
][
    [
        "dataset",
        "residue",
        "odds_ratio_corrected",
        "log2_enrichment",
        "fdr_bh",
        "significant_fdr_0_05",
    ]
].copy()

internal_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "internal_test"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)

kelm_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "kelm_external"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)

replication_df = (
    internal_replication.merge(
        kelm_replication,
        on="residue",
        how="outer",
        validate="one_to_one",
    )
)

replication_df[
    "same_enrichment_direction"
] = (
    np.sign(
        replication_df[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replication_df[
            "kelm_log2_enrichment"
        ]
    )
)

replication_df[
    "significant_in_both"
] = (
    replication_df[
        "internal_significant"
    ].fillna(False)
    & replication_df[
        "kelm_significant"
    ].fillna(False)
)

replication_df = (
    replication_df.sort_values(
        [
            "significant_in_both",
            "same_enrichment_direction",
            "internal_log2_enrichment",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
)

replication_file = (
    RESULT_TABLE_DIR
    / "residue_enrichment_replication.csv"
)

replication_df.to_csv(
    replication_file,
    index=False,
)


# ============================================================
# 12. SUMMARY JSON
# ============================================================

summary = {
    "datasets": DATASETS,
    "amino_acids": AMINO_ACIDS,
    "hotspot_definitions": (
        HOTSPOT_DEFINITIONS
    ),
    "number_of_tests": int(
        len(enrichment_df)
    ),
    "number_significant_fdr_0_05": int(
        enrichment_df[
            "significant_fdr_0_05"
        ].sum()
    ),
    "top20_number_significant": int(
        main_summary[
            "significant_fdr_0_05"
        ].sum()
    ),
    "replicated_significant_residues": (
        replication_df[
            replication_df[
                "significant_in_both"
            ]
        ]["residue"]
        .tolist()
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "residue_enrichment_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 13. SAVE CHECKPOINT
# ============================================================

output_files = [
    complete_output_file,
    main_summary_file,
    significant_file,
    replication_file,
    summary_file,
]

mark_step_complete(
    "14_residue_enrichment",
    output_files=output_files,
    details=summary,
)


# ============================================================
# 14. DISPLAY IMPORTANT RESULTS
# ============================================================

print("\n" + "=" * 75)
print("TOP-20% CPP HOTSPOT ENRICHMENT")
print("=" * 75)

display(
    main_summary[
        (
            main_summary[
                "analysis"
            ]
            == (
                "CPP_hotspot_vs_CPP_nonhotspot"
            )
        )
    ][
        [
            "dataset",
            "residue",
            "group1_frequency",
            "group2_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "p_value",
            "fdr_bh",
            "significant_fdr_0_05",
            "enrichment_direction",
        ]
    ].sort_values(
        [
            "dataset",
            "fdr_bh",
        ]
    )
)

print("\n" + "=" * 75)
print("INTERNAL–KELM REPLICATION")
print("=" * 75)

display(
    replication_df[
        [
            "residue",
            "internal_odds_ratio",
            "internal_log2_enrichment",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_log2_enrichment",
            "kelm_fdr",
            "same_enrichment_direction",
            "significant_in_both",
        ]
    ]
)

print("\nSignificant replicated residues:")
print(
    replication_df[
        replication_df[
            "significant_in_both"
        ]
    ]["residue"].tolist()
)

print("\nComplete SI table:")
print(complete_output_file)

print("\nMain top-20 table:")
print(main_summary_file)

print("\nReplication table:")
print(replication_file)

In [ ]:
# ============================================================
# STEP 15: HOTSPOT-CENTERED MOTIF DISCOVERY
#
# Extracts motifs around adjusted consensus hotspots and tests
# whether motifs are enriched in CPP hotspot neighborhoods.
#
# Motif lengths: 2 to 5 residues
# Datasets: internal test and KELM external
# ============================================================

from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd

from scipy.stats import fisher_exact


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

MIN_MOTIF_LENGTH = 2
MAX_MOTIF_LENGTH = 5

# Require a motif to occur in at least this many sequences
MIN_CPP_SEQUENCE_SUPPORT = {
    "internal_test": 5,
    "kelm_external": 3,
}

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

RESULT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_SI_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. BENJAMINI-HOCHBERG CORRECTION
# ============================================================

def benjamini_hochberg(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(p_values)

    if number_of_tests == 0:
        return np.array([])

    order = np.argsort(p_values)

    ranked = p_values[order]

    adjusted_ranked = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ranked[reverse_index]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ranked[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


# ============================================================
# 3. LOAD ADJUSTED CONSENSUS
# ============================================================

def load_consensus(dataset_name):

    consensus_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not consensus_file.exists():
        raise FileNotFoundError(
            f"Consensus file missing:\n{consensus_file}"
        )

    dataframe = pd.read_csv(
        consensus_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "position",
        "residue",
        "adjusted_hotspot_top20",
        "adjusted_global_consensus_rank",
    }

    missing = (
        required_columns
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    return dataframe


# ============================================================
# 4. EXTRACT MOTIFS THAT CONTAIN AT LEAST ONE HOTSPOT
# ============================================================

def extract_sequence_motifs(
    sequence,
    hotspot_positions,
    motif_length,
):
    """
    Return unique motifs of a given length that overlap at
    least one top-20% hotspot.

    hotspot_positions are 1-based.
    """

    sequence = str(sequence)

    zero_based_hotspots = {
        position - 1
        for position in hotspot_positions
    }

    motifs = set()

    maximum_start = (
        len(sequence) - motif_length
    )

    for start in range(
        maximum_start + 1
    ):
        end = start + motif_length

        motif_positions = set(
            range(start, end)
        )

        if motif_positions.intersection(
            zero_based_hotspots
        ):
            motif = sequence[
                start:end
            ]

            motifs.add(motif)

    return motifs


# ============================================================
# 5. BUILD SEQUENCE-LEVEL MOTIF PRESENCE TABLE
# ============================================================

def build_motif_presence(
    consensus_df,
    motif_length,
):

    sequence_records = []

    for sequence_id, group in (
        consensus_df.groupby(
            "sequence_id",
            sort=False,
        )
    ):
        sequence = str(
            group["sequence"].iloc[0]
        )

        label = int(
            group["label"].iloc[0]
        )

        hotspot_positions = (
            group.loc[
                group[
                    "adjusted_hotspot_top20"
                ].astype(bool),
                "position",
            ]
            .astype(int)
            .tolist()
        )

        motifs = extract_sequence_motifs(
            sequence=sequence,
            hotspot_positions=hotspot_positions,
            motif_length=motif_length,
        )

        sequence_records.append({
            "sequence_id": sequence_id,
            "sequence": sequence,
            "label": label,
            "motifs": motifs,
        })

    return sequence_records


# ============================================================
# 6. TEST MOTIF ENRICHMENT BETWEEN CPP AND NON-CPP SEQUENCES
# ============================================================

def calculate_motif_enrichment(
    consensus_df,
    dataset_name,
    motif_length,
):

    sequence_records = build_motif_presence(
        consensus_df=consensus_df,
        motif_length=motif_length,
    )

    cpp_records = [
        record
        for record in sequence_records
        if record["label"] == 1
    ]

    noncpp_records = [
        record
        for record in sequence_records
        if record["label"] == 0
    ]

    cpp_total = len(cpp_records)
    noncpp_total = len(noncpp_records)

    cpp_motif_counter = Counter()

    noncpp_motif_counter = Counter()

    for record in cpp_records:
        cpp_motif_counter.update(
            record["motifs"]
        )

    for record in noncpp_records:
        noncpp_motif_counter.update(
            record["motifs"]
        )

    all_motifs = sorted(
        set(cpp_motif_counter)
        | set(noncpp_motif_counter)
    )

    rows = []

    minimum_support = (
        MIN_CPP_SEQUENCE_SUPPORT[
            dataset_name
        ]
    )

    for motif in all_motifs:

        cpp_present = int(
            cpp_motif_counter.get(
                motif,
                0,
            )
        )

        if cpp_present < minimum_support:
            continue

        noncpp_present = int(
            noncpp_motif_counter.get(
                motif,
                0,
            )
        )

        cpp_absent = (
            cpp_total - cpp_present
        )

        noncpp_absent = (
            noncpp_total
            - noncpp_present
        )

        contingency = [
            [
                cpp_present,
                cpp_absent,
            ],
            [
                noncpp_present,
                noncpp_absent,
            ],
        ]

        scipy_odds_ratio, p_value = (
            fisher_exact(
                contingency,
                alternative="two-sided",
            )
        )

        corrected_odds_ratio = (
            (cpp_present + 0.5)
            * (noncpp_absent + 0.5)
            / (
                (cpp_absent + 0.5)
                * (noncpp_present + 0.5)
            )
        )

        cpp_frequency = (
            cpp_present / cpp_total
        )

        noncpp_frequency = (
            noncpp_present / noncpp_total
        )

        rows.append({
            "dataset": dataset_name,
            "motif_length": int(
                motif_length
            ),
            "motif": motif,
            "cpp_sequence_count": int(
                cpp_present
            ),
            "cpp_sequence_total": int(
                cpp_total
            ),
            "cpp_sequence_frequency": float(
                cpp_frequency
            ),
            "noncpp_sequence_count": int(
                noncpp_present
            ),
            "noncpp_sequence_total": int(
                noncpp_total
            ),
            "noncpp_sequence_frequency": float(
                noncpp_frequency
            ),
            "odds_ratio_scipy": float(
                scipy_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_odds_ratio
            ),
            "log2_enrichment": float(
                np.log2(
                    (
                        cpp_frequency
                        + 1e-12
                    )
                    / (
                        noncpp_frequency
                        + 1e-12
                    )
                )
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    if len(result) == 0:
        return result

    result["fdr_bh"] = (
        benjamini_hochberg(
            result["p_value"]
            .to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "contains_K_or_R"
    ] = result["motif"].apply(
        lambda motif: (
            "K" in motif
            or "R" in motif
        )
    )

    result[
        "basic_residue_fraction"
    ] = result["motif"].apply(
        lambda motif: (
            sum(
                residue in {"K", "R"}
                for residue in motif
            )
            / len(motif)
        )
    )

    return result


# ============================================================
# 7. RUN ALL MOTIF LENGTHS AND DATASETS
# ============================================================

all_motif_results = []

for dataset_name in DATASETS:

    consensus_df = load_consensus(
        dataset_name
    )

    print("\n" + "=" * 75)
    print(
        f"MOTIF DISCOVERY: {dataset_name}"
    )
    print("=" * 75)

    for motif_length in range(
        MIN_MOTIF_LENGTH,
        MAX_MOTIF_LENGTH + 1,
    ):
        print(
            f"[MOTIFS] length={motif_length}"
        )

        motif_result = (
            calculate_motif_enrichment(
                consensus_df=consensus_df,
                dataset_name=dataset_name,
                motif_length=motif_length,
            )
        )

        if len(motif_result) > 0:
            all_motif_results.append(
                motif_result
            )


if not all_motif_results:
    raise RuntimeError(
        "No motifs passed the minimum-support criteria."
    )

motif_df = pd.concat(
    all_motif_results,
    ignore_index=True,
)


# ============================================================
# 8. SAVE COMPLETE MOTIF TABLE
# ============================================================

complete_motif_file = (
    RESULT_SI_DIR
    / "hotspot_motif_enrichment_all.csv"
)

motif_df.to_csv(
    complete_motif_file,
    index=False,
)


# ============================================================
# 9. SIGNIFICANT MOTIFS
# ============================================================

significant_motifs = motif_df[
    motif_df[
        "significant_fdr_0_05"
    ]
].copy()

significant_motifs = (
    significant_motifs.sort_values(
        [
            "dataset",
            "fdr_bh",
            "odds_ratio_corrected",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)

significant_motif_file = (
    RESULT_TABLE_DIR
    / "significant_hotspot_motifs.csv"
)

significant_motifs.to_csv(
    significant_motif_file,
    index=False,
)


# ============================================================
# 10. INTERNAL–KELM MOTIF REPLICATION
# ============================================================

internal_motifs = (
    motif_df[
        motif_df["dataset"]
        == "internal_test"
    ][
        [
            "motif",
            "motif_length",
            "cpp_sequence_count",
            "cpp_sequence_frequency",
            "noncpp_sequence_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
            "contains_K_or_R",
            "basic_residue_fraction",
        ]
    ]
    .rename(
        columns={
            "cpp_sequence_count":
                "internal_cpp_count",
            "cpp_sequence_frequency":
                "internal_cpp_frequency",
            "noncpp_sequence_frequency":
                "internal_noncpp_frequency",
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)

kelm_motifs = (
    motif_df[
        motif_df["dataset"]
        == "kelm_external"
    ][
        [
            "motif",
            "motif_length",
            "cpp_sequence_count",
            "cpp_sequence_frequency",
            "noncpp_sequence_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
    .rename(
        columns={
            "cpp_sequence_count":
                "kelm_cpp_count",
            "cpp_sequence_frequency":
                "kelm_cpp_frequency",
            "noncpp_sequence_frequency":
                "kelm_noncpp_frequency",
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)

replicated_motifs = internal_motifs.merge(
    kelm_motifs,
    on=[
        "motif",
        "motif_length",
    ],
    how="inner",
    validate="one_to_one",
)

replicated_motifs[
    "same_enrichment_direction"
] = (
    np.sign(
        replicated_motifs[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replicated_motifs[
            "kelm_log2_enrichment"
        ]
    )
)

replicated_motifs[
    "significant_in_both"
] = (
    replicated_motifs[
        "internal_significant"
    ]
    & replicated_motifs[
        "kelm_significant"
    ]
)

replicated_motifs[
    "mean_log2_enrichment"
] = (
    replicated_motifs[
        [
            "internal_log2_enrichment",
            "kelm_log2_enrichment",
        ]
    ].mean(axis=1)
)

replicated_motifs = (
    replicated_motifs.sort_values(
        [
            "significant_in_both",
            "same_enrichment_direction",
            "mean_log2_enrichment",
            "internal_cpp_count",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
)

replication_file = (
    RESULT_TABLE_DIR
    / "hotspot_motif_replication.csv"
)

replicated_motifs.to_csv(
    replication_file,
    index=False,
)


# ============================================================
# 11. STRICT REPLICATED MOTIFS
# ============================================================

strict_replicated = replicated_motifs[
    replicated_motifs[
        "significant_in_both"
    ]
    & replicated_motifs[
        "same_enrichment_direction"
    ]
    & (
        replicated_motifs[
            "internal_log2_enrichment"
        ] > 0
    )
    & (
        replicated_motifs[
            "kelm_log2_enrichment"
        ] > 0
    )
].copy()

strict_replicated_file = (
    RESULT_TABLE_DIR
    / "strict_replicated_CPP_hotspot_motifs.csv"
)

strict_replicated.to_csv(
    strict_replicated_file,
    index=False,
)


# ============================================================
# 12. SUMMARY
# ============================================================

summary = {
    "motif_lengths": list(
        range(
            MIN_MOTIF_LENGTH,
            MAX_MOTIF_LENGTH + 1,
        )
    ),
    "total_tested_motifs": int(
        len(motif_df)
    ),
    "significant_motifs": int(
        len(significant_motifs)
    ),
    "strict_replicated_enriched_motifs": (
        strict_replicated[
            "motif"
        ].tolist()
    ),
    "number_strict_replicated": int(
        len(strict_replicated)
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "hotspot_motif_discovery_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 13. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "15_hotspot_motif_discovery",
    output_files=[
        complete_motif_file,
        significant_motif_file,
        replication_file,
        strict_replicated_file,
        summary_file,
    ],
    details=summary,
)


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("STRICT REPLICATED ENRICHED MOTIFS")
print("=" * 75)

if len(strict_replicated) > 0:

    display(
        strict_replicated[
            [
                "motif",
                "motif_length",
                "internal_cpp_count",
                "internal_odds_ratio",
                "internal_fdr",
                "kelm_cpp_count",
                "kelm_odds_ratio",
                "kelm_fdr",
                "contains_K_or_R",
                "basic_residue_fraction",
            ]
        ]
    )

else:
    print(
        "No motif was significant in both datasets "
        "under the current criteria."
    )

print("\nTop replicated motifs regardless of significance:")

display(
    replicated_motifs[
        [
            "motif",
            "motif_length",
            "internal_cpp_count",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_cpp_count",
            "kelm_odds_ratio",
            "kelm_fdr",
            "same_enrichment_direction",
            "significant_in_both",
        ]
    ].head(30)
)

print("\nComplete motif table:")
print(complete_motif_file)

print("\nReplication table:")
print(replication_file)

print("\nStrict replicated motifs:")
print(strict_replicated_file)

In [ ]:
# ============================================================
# STEP 16: HOTSPOT FAITHFULNESS AND PERTURBATION VALIDATION
#
# Tests:
# 1. Comprehensiveness:
#    Remove top-20% adjusted consensus hotspots.
#
# 2. Random matched control:
#    Remove the same number of randomly selected residues.
#
# 3. Sufficiency:
#    Retain hotspot residues and remove all other residues.
#
# Uses saved residue embeddings; no PLM recomputation required.
# ============================================================

from pathlib import Path
import gc
import json
import math

import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.stats import (
    wilcoxon,
    mannwhitneyu,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
RANDOM_REPEATS = 20

MODEL_CONFIGS = {
    "ESM2_320": {
        "batch_size": 128,
    },
    "ESM2_640": {
        "batch_size": 96,
    },
    "ESM2_1280": {
        "batch_size": 64,
    },
    "ProtT5": {
        "batch_size": 64,
    },
}

DATASETS = [
    "internal_test",
    "kelm_external",
]

MAX_LEN = 61

rng_master = np.random.default_rng(
    SEED
)

EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

CHECKPOINT_ROOT = DIRS["checkpoints"]

FAITHFULNESS_ROOT = (
    XAI_ROOT / "faithfulness"
)

for folder in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FAITHFULNESS_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. CUSTOM LAYER FOR MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = (
            tf.keras.layers.Dense(
                1,
                use_bias=True,
                name="residue_attention_score",
            )
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        return tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. LOAD CONSENSUS HOTSPOT MASKS
# ============================================================

def load_consensus_hotspot_masks(
    dataset_name,
    metadata,
):
    consensus_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not consensus_file.exists():
        raise FileNotFoundError(
            f"Consensus file missing:\n"
            f"{consensus_file}"
        )

    consensus = pd.read_csv(
        consensus_file
    )

    hotspot_masks = np.zeros(
        (
            len(metadata),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    row_lookup = {
        str(sequence_id): row_index
        for row_index, sequence_id in enumerate(
            metadata["sequence_id"]
            .astype(str)
        )
    }

    hotspot_rows = consensus[
        consensus[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    for row in hotspot_rows.itertuples(
        index=False
    ):
        sequence_id = str(
            row.sequence_id
        )

        position_index = int(
            row.position
        ) - 1

        if sequence_id not in row_lookup:
            raise ValueError(
                f"Sequence ID absent from metadata: "
                f"{sequence_id}"
            )

        if not (
            0 <= position_index < MAX_LEN
        ):
            raise ValueError(
                f"Invalid position for {sequence_id}: "
                f"{position_index + 1}"
            )

        hotspot_masks[
            row_lookup[sequence_id],
            position_index,
        ] = 1

    hotspot_counts = (
        hotspot_masks.sum(axis=1)
    )

    if np.any(hotspot_counts == 0):
        raise ValueError(
            "At least one sequence has no top-20% "
            "consensus hotspot."
        )

    return hotspot_masks


# ============================================================
# 4. LOAD ONE MODEL/DATASET
# ============================================================

def load_model_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if (
        len(embeddings) != len(metadata)
        or len(masks) != len(metadata)
    ):
        raise ValueError(
            f"Row mismatch for "
            f"{model_name}/{dataset_name}"
        )

    return embeddings, masks, metadata


# ============================================================
# 5. PREDICT PROBABILITIES
# ============================================================

def predict_probabilities(
    model,
    embeddings,
    masks,
    batch_size,
):
    probabilities = model.predict(
        {
            "residue_embeddings": embeddings,
            "residue_mask": masks,
        },
        batch_size=batch_size,
        verbose=0,
    )

    return (
        probabilities
        .reshape(-1)
        .astype(np.float32)
    )


# ============================================================
# 6. CREATE RANDOM MATCHED ABLATION MASK
# ============================================================

def create_random_ablation_mask(
    valid_mask,
    hotspot_mask,
    seed,
):
    """
    Select the same number of random valid positions as
    top-20% hotspots for every sequence.

    Hotspot positions are excluded from the random pool where
    sufficient non-hotspot positions are available.
    """

    rng = np.random.default_rng(
        seed
    )

    random_mask = np.zeros_like(
        valid_mask,
        dtype=np.uint8,
    )

    for row_index in range(
        len(valid_mask)
    ):
        valid_positions = np.flatnonzero(
            valid_mask[row_index]
        )

        hotspot_positions = np.flatnonzero(
            hotspot_mask[row_index]
        )

        number_to_select = len(
            hotspot_positions
        )

        non_hotspot_positions = (
            np.setdiff1d(
                valid_positions,
                hotspot_positions,
                assume_unique=True,
            )
        )

        if (
            len(non_hotspot_positions)
            >= number_to_select
        ):
            candidate_positions = (
                non_hotspot_positions
            )
        else:
            candidate_positions = (
                valid_positions
            )

        selected = rng.choice(
            candidate_positions,
            size=number_to_select,
            replace=False,
        )

        random_mask[
            row_index,
            selected,
        ] = 1

    return random_mask


# ============================================================
# 7. PERTURB EMBEDDINGS
# ============================================================

def ablate_positions(
    embeddings,
    position_mask,
):
    """
    Set selected residue embeddings to zero.
    """

    return (
        embeddings
        * (
            1.0
            - position_mask[
                :,
                :,
                None,
            ].astype(
                embeddings.dtype
            )
        )
    )


def retain_positions_only(
    embeddings,
    position_mask,
):
    """
    Retain only selected residue embeddings.
    """

    return (
        embeddings
        * position_mask[
            :,
            :,
            None,
        ].astype(
            embeddings.dtype
        )
    )


# ============================================================
# 8. SAFE STATISTICAL TESTS
# ============================================================

def safe_wilcoxon(
    values_a,
    values_b,
):
    difference = (
        np.asarray(values_a)
        - np.asarray(values_b)
    )

    if np.allclose(
        difference,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        values_a,
        values_b,
        alternative="two-sided",
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


# ============================================================
# 9. PROCESS ONE MODEL/DATASET
# ============================================================

def run_faithfulness_analysis(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        FAITHFULNESS_ROOT
        / model_name
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    per_sequence_file = (
        output_dir
        / "per_sequence_faithfulness.csv"
    )

    random_repeat_file = (
        output_dir
        / "random_ablation_repeats.csv"
    )

    summary_file = (
        output_dir
        / "faithfulness_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        per_sequence_file.exists()
        and random_repeat_file.exists()
        and summary_file.exists()
        and complete_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] "
            f"{model_name} | {dataset_name}"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            return json.load(handle)

    print("\n" + "=" * 75)
    print(
        f"FAITHFULNESS: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 75)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings_memmap, masks, metadata = (
        load_model_dataset(
            model_name,
            dataset_name,
        )
    )

    # Copy one dataset into RAM for efficient repeated testing.
    embeddings = np.asarray(
        embeddings_memmap,
        dtype=np.float16,
    )

    valid_masks = np.asarray(
        masks,
        dtype=np.uint8,
    )

    hotspot_masks = (
        load_consensus_hotspot_masks(
            dataset_name=dataset_name,
            metadata=metadata,
        )
    )

    # --------------------------------------------------------
    # Original predictions
    # --------------------------------------------------------

    original_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    # --------------------------------------------------------
    # Consensus hotspot ablation
    # --------------------------------------------------------

    hotspot_ablated_embeddings = (
        ablate_positions(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_ablated_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=(
                hotspot_ablated_embeddings
            ),
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    hotspot_probability_drop = (
        original_probabilities
        - hotspot_ablated_probabilities
    )

    del hotspot_ablated_embeddings

    # --------------------------------------------------------
    # Hotspot-only sufficiency
    # --------------------------------------------------------

    hotspot_only_embeddings = (
        retain_positions_only(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_only_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=(
                hotspot_only_embeddings
            ),
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    del hotspot_only_embeddings

    # --------------------------------------------------------
    # Random matched controls
    # --------------------------------------------------------

    random_repeat_rows = []

    random_drop_matrix = np.zeros(
        (
            len(metadata),
            RANDOM_REPEATS,
        ),
        dtype=np.float32,
    )

    for repeat_index in range(
        RANDOM_REPEATS
    ):
        random_mask = (
            create_random_ablation_mask(
                valid_mask=valid_masks,
                hotspot_mask=hotspot_masks,
                seed=(
                    SEED
                    + repeat_index
                    + 1000
                ),
            )
        )

        random_ablated_embeddings = (
            ablate_positions(
                embeddings,
                random_mask,
            )
        )

        random_probabilities = (
            predict_probabilities(
                model=model,
                embeddings=(
                    random_ablated_embeddings
                ),
                masks=valid_masks,
                batch_size=batch_size,
            )
        )

        random_drops = (
            original_probabilities
            - random_probabilities
        )

        random_drop_matrix[
            :,
            repeat_index
        ] = random_drops

        random_repeat_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "repeat": int(
                repeat_index + 1
            ),
            "mean_probability_drop_all": float(
                random_drops.mean()
            ),
            "mean_probability_drop_CPP": float(
                random_drops[
                    metadata["label"]
                    .to_numpy()
                    == 1
                ].mean()
            ),
            "mean_probability_drop_nonCPP": float(
                random_drops[
                    metadata["label"]
                    .to_numpy()
                    == 0
                ].mean()
            ),
        })

        print(
            f"[RANDOM CONTROL] "
            f"{repeat_index + 1}/"
            f"{RANDOM_REPEATS}"
        )

        del random_mask
        del random_ablated_embeddings
        del random_probabilities
        del random_drops

        gc.collect()

    random_mean_drop = (
        random_drop_matrix.mean(
            axis=1
        )
    )

    random_drop_std = (
        random_drop_matrix.std(
            axis=1,
            ddof=0,
        )
    )

    # --------------------------------------------------------
    # Per-sequence results
    # --------------------------------------------------------

    per_sequence = metadata[
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    per_sequence.insert(
        0,
        "model",
        model_name,
    )

    per_sequence.insert(
        1,
        "dataset",
        dataset_name,
    )

    per_sequence[
        "number_of_hotspots"
    ] = hotspot_masks.sum(
        axis=1
    )

    per_sequence[
        "hotspot_fraction"
    ] = (
        per_sequence[
            "number_of_hotspots"
        ]
        / per_sequence["length"]
    )

    per_sequence[
        "original_probability"
    ] = original_probabilities

    per_sequence[
        "hotspot_ablated_probability"
    ] = hotspot_ablated_probabilities

    per_sequence[
        "hotspot_probability_drop"
    ] = hotspot_probability_drop

    per_sequence[
        "random_mean_probability_drop"
    ] = random_mean_drop

    per_sequence[
        "random_drop_std"
    ] = random_drop_std

    per_sequence[
        "hotspot_minus_random_drop"
    ] = (
        hotspot_probability_drop
        - random_mean_drop
    )

    per_sequence[
        "hotspot_only_probability"
    ] = hotspot_only_probabilities

    per_sequence[
        "sufficiency_probability_loss"
    ] = (
        original_probabilities
        - hotspot_only_probabilities
    )

    per_sequence[
        "hotspot_ablation_stronger_than_random"
    ] = (
        per_sequence[
            "hotspot_probability_drop"
        ]
        > per_sequence[
            "random_mean_probability_drop"
        ]
    )

    per_sequence.to_csv(
        per_sequence_file,
        index=False,
    )

    random_repeat_df = pd.DataFrame(
        random_repeat_rows
    )

    random_repeat_df.to_csv(
        random_repeat_file,
        index=False,
    )

    # --------------------------------------------------------
    # Statistical summaries
    # --------------------------------------------------------

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            len(per_sequence)
        ),
        "number_of_CPPs": int(
            (per_sequence["label"] == 1).sum()
        ),
        "number_of_nonCPPs": int(
            (per_sequence["label"] == 0).sum()
        ),
        "random_repeats": int(
            RANDOM_REPEATS
        ),
        "classes": {},
    }

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
        ("all", "all"),
    ]:
        if label_value == "all":
            subset = per_sequence
        else:
            subset = per_sequence[
                per_sequence["label"]
                == label_value
            ]

        wilcoxon_statistic, wilcoxon_p = (
            safe_wilcoxon(
                subset[
                    "hotspot_probability_drop"
                ],
                subset[
                    "random_mean_probability_drop"
                ],
            )
        )

        summary["classes"][
            class_name
        ] = {
            "n": int(
                len(subset)
            ),
            "mean_original_probability": float(
                subset[
                    "original_probability"
                ].mean()
            ),
            "mean_hotspot_ablated_probability": float(
                subset[
                    "hotspot_ablated_probability"
                ].mean()
            ),
            "mean_hotspot_probability_drop": float(
                subset[
                    "hotspot_probability_drop"
                ].mean()
            ),
            "median_hotspot_probability_drop": float(
                subset[
                    "hotspot_probability_drop"
                ].median()
            ),
            "mean_random_probability_drop": float(
                subset[
                    "random_mean_probability_drop"
                ].mean()
            ),
            "median_random_probability_drop": float(
                subset[
                    "random_mean_probability_drop"
                ].median()
            ),
            "mean_hotspot_minus_random_drop": float(
                subset[
                    "hotspot_minus_random_drop"
                ].mean()
            ),
            "fraction_hotspot_stronger_than_random": float(
                subset[
                    "hotspot_ablation_stronger_than_random"
                ].mean()
            ),
            "mean_hotspot_only_probability": float(
                subset[
                    "hotspot_only_probability"
                ].mean()
            ),
            "wilcoxon_hotspot_vs_random_statistic": (
                wilcoxon_statistic
            ),
            "wilcoxon_hotspot_vs_random_p": (
                wilcoxon_p
            ),
        }

    with open(
        summary_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    with open(
        complete_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
        )

    print(
        "[COMPLETE]",
        model_name,
        dataset_name,
    )

    cpp_summary = summary[
        "classes"
    ]["CPP"]

    print(
        "CPP hotspot drop:",
        cpp_summary[
            "mean_hotspot_probability_drop"
        ],
    )

    print(
        "CPP random drop:",
        cpp_summary[
            "mean_random_probability_drop"
        ],
    )

    print(
        "CPP hotspot-minus-random:",
        cpp_summary[
            "mean_hotspot_minus_random_drop"
        ],
    )

    del model
    del embeddings_memmap
    del embeddings
    del masks
    del valid_masks
    del hotspot_masks
    del random_drop_matrix

    tf.keras.backend.clear_session()
    gc.collect()

    return summary


# ============================================================
# 10. RUN ALL MODELS AND DATASETS
# ============================================================

summary_records = []
all_output_files = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in DATASETS:
        summary = (
            run_faithfulness_analysis(
                model_name=model_name,
                dataset_name=dataset_name,
                batch_size=config[
                    "batch_size"
                ],
            )
        )

        for class_name, values in (
            summary["classes"].items()
        ):
            summary_records.append({
                "model": model_name,
                "dataset": dataset_name,
                "class": class_name,
                **values,
            })

        output_dir = (
            FAITHFULNESS_ROOT
            / model_name
            / dataset_name
        )

        all_output_files.extend([
            output_dir
            / "per_sequence_faithfulness.csv",
            output_dir
            / "random_ablation_repeats.csv",
            output_dir
            / "faithfulness_summary.json",
            output_dir
            / "COMPLETE.json",
        ])


# ============================================================
# 11. MASTER SUMMARY TABLE
# ============================================================

faithfulness_summary_df = (
    pd.DataFrame(
        summary_records
    )
)

summary_output_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_summary.csv"
)

faithfulness_summary_df.to_csv(
    summary_output_file,
    index=False,
)

all_output_files.append(
    summary_output_file
)


# ============================================================
# 12. CROSS-MODEL AVERAGE
# ============================================================

numeric_columns = [
    "mean_original_probability",
    "mean_hotspot_ablated_probability",
    "mean_hotspot_probability_drop",
    "median_hotspot_probability_drop",
    "mean_random_probability_drop",
    "median_random_probability_drop",
    "mean_hotspot_minus_random_drop",
    "fraction_hotspot_stronger_than_random",
    "mean_hotspot_only_probability",
]

cross_model_summary = (
    faithfulness_summary_df
    .groupby(
        [
            "dataset",
            "class",
        ],
        as_index=False,
    )[numeric_columns]
    .mean()
)

cross_model_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_cross_model_summary.csv"
)

cross_model_summary.to_csv(
    cross_model_file,
    index=False,
)

all_output_files.append(
    cross_model_file
)


# ============================================================
# 13. SAVE CHECKPOINT
# ============================================================

checkpoint_details = {
    "models": list(
        MODEL_CONFIGS.keys()
    ),
    "datasets": DATASETS,
    "hotspot_definition": (
        "adjusted global consensus top 20%"
    ),
    "random_control_repeats": int(
        RANDOM_REPEATS
    ),
    "ablation_method": (
        "set selected fixed PLM residue embeddings to zero"
    ),
    "sufficiency_method": (
        "retain only selected hotspot residue embeddings"
    ),
}

mark_step_complete(
    "16_hotspot_faithfulness",
    output_files=all_output_files,
    details=checkpoint_details,
)


# ============================================================
# 14. DISPLAY CPP RESULTS
# ============================================================

print("\n" + "=" * 75)
print("CPP HOTSPOT FAITHFULNESS BY MODEL")
print("=" * 75)

display(
    faithfulness_summary_df[
        faithfulness_summary_df[
            "class"
        ] == "CPP"
    ][
        [
            "model",
            "dataset",
            "n",
            "mean_original_probability",
            "mean_hotspot_ablated_probability",
            "mean_hotspot_probability_drop",
            "mean_random_probability_drop",
            "mean_hotspot_minus_random_drop",
            "fraction_hotspot_stronger_than_random",
            "mean_hotspot_only_probability",
            "wilcoxon_hotspot_vs_random_p",
        ]
    ]
)

print("\n" + "=" * 75)
print("CROSS-MODEL FAITHFULNESS SUMMARY")
print("=" * 75)

display(
    cross_model_summary[
        cross_model_summary[
            "class"
        ] == "CPP"
    ]
)

print("\nMaster faithfulness table:")
print(summary_output_file)

print("\nCross-model summary:")
print(cross_model_file)

In [ ]:
# ============================================================
# STEP 16: PUBLICATION-READY HOTSPOT FAITHFULNESS ANALYSIS
#
# Tests:
# 1. Comprehensiveness:
#    Set consensus-hotspot residue embeddings to zero.
#
# 2. Matched random controls:
#    Remove the same number of non-hotspot residues,
#    repeated RANDOM_REPEATS times.
#
# 3. Sufficiency:
#    Retain only consensus-hotspot residue embeddings.
#
# Outputs:
# - Per-sequence results
# - Restart-safe random repeat checkpoints
# - Statistical tests and effect sizes
# - Bootstrap confidence intervals
# - Manuscript-ready summary tables
# - PNG, PDF, and SVG figures
# ============================================================

from pathlib import Path
import gc
import json
import math
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from scipy.stats import wilcoxon


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
RANDOM_REPEATS = 20
BOOTSTRAP_REPEATS = 2000
CONFIDENCE_LEVEL = 0.95
MAX_LEN = 61

MODEL_CONFIGS = {
    "ESM2_320": {
        "batch_size": 128,
    },
    "ESM2_640": {
        "batch_size": 96,
    },
    "ESM2_1280": {
        "batch_size": 64,
    },
    "ProtT5": {
        "batch_size": 64,
    },
}

DATASETS = [
    "internal_test",
    "kelm_external",
]

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass


EMBEDDING_ROOT = DIRS["embeddings"]
MODEL_ROOT = DIRS["models"]
XAI_ROOT = DIRS["xai"]
CHECKPOINT_ROOT = DIRS["checkpoints"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

FAITHFULNESS_ROOT = (
    XAI_ROOT / "faithfulness"
)

for folder in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
    FAITHFULNESS_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("=" * 78)
print("PUBLICATION-READY HOTSPOT FAITHFULNESS ANALYSIS")
print("=" * 78)
print("TensorFlow version :", tf.__version__)
print("TensorFlow GPUs    :", tf.config.list_physical_devices("GPU"))
print("Random repeats     :", RANDOM_REPEATS)
print("Bootstrap repeats  :", BOOTSTRAP_REPEATS)


# ============================================================
# 2. CUSTOM LAYER FOR SAVED MODEL LOADING
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.attention_dense = tf.keras.layers.Dense(
            1,
            use_bias=True,
            name="residue_attention_score",
        )

    def build(self, input_shape):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(input_shape)

    def call(self, inputs):
        residue_features, residue_mask = inputs

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (1.0 - residue_mask)
            * tf.cast(-1e4, logits.dtype)
        )

        attention_weights = tf.nn.softmax(
            masked_logits,
            axis=1,
        )

        pooled_vector = tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

        return pooled_vector

    def get_config(self):
        return super().get_config()


# ============================================================
# 3. GENERAL HELPERS
# ============================================================

def clear_memory():
    gc.collect()

    tf.keras.backend.clear_session()

    if tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.reset_memory_stats(
                "GPU:0"
            )
        except Exception:
            pass


def atomic_save_json(
    output_file,
    data,
):
    output_file = Path(output_file)

    temporary_file = Path(
        str(output_file) + ".temporary"
    )

    with open(
        temporary_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            data,
            handle,
            indent=2,
        )

    temporary_file.replace(
        output_file
    )


def atomic_save_csv(
    dataframe,
    output_file,
):
    output_file = Path(output_file)

    temporary_file = Path(
        str(output_file) + ".temporary"
    )

    dataframe.to_csv(
        temporary_file,
        index=False,
    )

    temporary_file.replace(
        output_file
    )


# ============================================================
# 4. LOAD MODEL DATASET
# ============================================================

def load_model_dataset(
    model_name,
    dataset_name,
):
    dataset_dir = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    embedding_file = (
        dataset_dir / "X_per_residue.npy"
    )

    mask_file = (
        dataset_dir / "M_masks.npy"
    )

    metadata_file = (
        dataset_dir / "metadata.csv"
    )

    for required_file in [
        embedding_file,
        mask_file,
        metadata_file,
    ]:
        if not required_file.exists():
            raise FileNotFoundError(
                f"Missing required file:\n"
                f"{required_file}"
            )

    embeddings = np.load(
        embedding_file,
        mmap_mode="r",
    )

    masks = np.load(
        mask_file,
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        metadata_file
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Embedding/metadata row mismatch for "
            f"{model_name}/{dataset_name}"
        )

    if len(masks) != len(metadata):
        raise ValueError(
            f"Mask/metadata row mismatch for "
            f"{model_name}/{dataset_name}"
        )

    return (
        embeddings,
        masks,
        metadata,
    )


# ============================================================
# 5. LOAD GLOBAL CONSENSUS HOTSPOTS
# ============================================================

def load_consensus_hotspot_masks(
    dataset_name,
    metadata,
):
    consensus_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not consensus_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file missing:\n"
            f"{consensus_file}"
        )

    consensus = pd.read_csv(
        consensus_file
    )

    required_columns = {
        "sequence_id",
        "position",
        "adjusted_hotspot_top20",
    }

    missing_columns = (
        required_columns
        - set(consensus.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Consensus file is missing columns: "
            f"{sorted(missing_columns)}"
        )

    hotspot_masks = np.zeros(
        (
            len(metadata),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )

    row_lookup = {
        str(sequence_id): row_index
        for row_index, sequence_id in enumerate(
            metadata["sequence_id"].astype(str)
        )
    }

    hotspot_rows = consensus[
        consensus[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    for row in hotspot_rows.itertuples(
        index=False
    ):
        sequence_id = str(
            row.sequence_id
        )

        position_index = int(
            row.position
        ) - 1

        if sequence_id not in row_lookup:
            raise ValueError(
                f"Consensus sequence not found in metadata: "
                f"{sequence_id}"
            )

        if not (
            0 <= position_index < MAX_LEN
        ):
            raise ValueError(
                f"Invalid hotspot position "
                f"{position_index + 1} for {sequence_id}"
            )

        hotspot_masks[
            row_lookup[sequence_id],
            position_index,
        ] = 1

    hotspot_counts = (
        hotspot_masks.sum(axis=1)
    )

    zero_hotspot_sequences = int(
        np.sum(hotspot_counts == 0)
    )

    if zero_hotspot_sequences > 0:
        raise ValueError(
            f"{zero_hotspot_sequences} sequences have no "
            f"top-20% consensus hotspot."
        )

    return (
        hotspot_masks,
        consensus_file,
    )


# ============================================================
# 6. PREDICTION
# ============================================================

def predict_probabilities(
    model,
    embeddings,
    masks,
    batch_size,
):
    probabilities = model.predict(
        {
            "residue_embeddings": embeddings,
            "residue_mask": masks,
        },
        batch_size=batch_size,
        verbose=0,
    )

    return (
        probabilities
        .reshape(-1)
        .astype(np.float32)
    )


# ============================================================
# 7. PERTURBATIONS
# ============================================================

def ablate_positions(
    embeddings,
    position_mask,
):
    keep_mask = (
        1.0
        - position_mask[
            :,
            :,
            None,
        ].astype(
            embeddings.dtype
        )
    )

    return embeddings * keep_mask


def retain_positions_only(
    embeddings,
    position_mask,
):
    retain_mask = (
        position_mask[
            :,
            :,
            None,
        ].astype(
            embeddings.dtype
        )
    )

    return embeddings * retain_mask


def create_random_ablation_mask(
    valid_mask,
    hotspot_mask,
    seed,
):
    """
    Select the same number of valid non-hotspot positions
    as consensus hotspots for each sequence.

    For very short sequences where insufficient non-hotspot
    positions exist, sample from all valid positions.
    """

    rng = np.random.default_rng(
        seed
    )

    random_mask = np.zeros_like(
        valid_mask,
        dtype=np.uint8,
    )

    for row_index in range(
        len(valid_mask)
    ):
        valid_positions = np.flatnonzero(
            valid_mask[row_index]
        )

        hotspot_positions = np.flatnonzero(
            hotspot_mask[row_index]
        )

        number_to_select = len(
            hotspot_positions
        )

        non_hotspot_positions = np.setdiff1d(
            valid_positions,
            hotspot_positions,
            assume_unique=True,
        )

        if (
            len(non_hotspot_positions)
            >= number_to_select
        ):
            candidate_positions = (
                non_hotspot_positions
            )
        else:
            candidate_positions = (
                valid_positions
            )

        selected_positions = rng.choice(
            candidate_positions,
            size=number_to_select,
            replace=False,
        )

        random_mask[
            row_index,
            selected_positions,
        ] = 1

    return random_mask


# ============================================================
# 8. STATISTICAL HELPERS
# ============================================================

def safe_wilcoxon(
    values_a,
    values_b,
    alternative="two-sided",
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    differences = (
        values_a - values_b
    )

    finite_mask = np.isfinite(
        differences
    )

    differences = differences[
        finite_mask
    ]

    if len(differences) == 0:
        return np.nan, np.nan

    if np.allclose(
        differences,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        differences,
        alternative=alternative,
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


def paired_cohens_dz(
    values_a,
    values_b,
):
    differences = (
        np.asarray(values_a, dtype=float)
        - np.asarray(values_b, dtype=float)
    )

    differences = differences[
        np.isfinite(differences)
    ]

    if len(differences) < 2:
        return np.nan

    standard_deviation = (
        differences.std(ddof=1)
    )

    if np.isclose(
        standard_deviation,
        0.0,
    ):
        return np.nan

    return float(
        differences.mean()
        / standard_deviation
    )


def rank_biserial_effect(
    values_a,
    values_b,
):
    """
    Matched-pairs rank-biserial effect size.

    Positive values indicate values_a > values_b.
    """

    differences = (
        np.asarray(values_a, dtype=float)
        - np.asarray(values_b, dtype=float)
    )

    differences = differences[
        np.isfinite(differences)
        & ~np.isclose(differences, 0.0)
    ]

    if len(differences) == 0:
        return 0.0

    absolute_differences = np.abs(
        differences
    )

    ranks = pd.Series(
        absolute_differences
    ).rank(
        method="average"
    ).to_numpy()

    positive_rank_sum = ranks[
        differences > 0
    ].sum()

    negative_rank_sum = ranks[
        differences < 0
    ].sum()

    denominator = (
        positive_rank_sum
        + negative_rank_sum
    )

    if denominator == 0:
        return 0.0

    return float(
        (
            positive_rank_sum
            - negative_rank_sum
        )
        / denominator
    )


def bootstrap_mean_confidence_interval(
    values,
    repeats=BOOTSTRAP_REPEATS,
    confidence_level=CONFIDENCE_LEVEL,
    seed=SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    rng = np.random.default_rng(
        seed
    )

    sample_indices = rng.integers(
        low=0,
        high=len(values),
        size=(
            repeats,
            len(values),
        ),
    )

    bootstrap_means = values[
        sample_indices
    ].mean(axis=1)

    alpha = (
        1.0 - confidence_level
    ) / 2.0

    lower = np.quantile(
        bootstrap_means,
        alpha,
    )

    upper = np.quantile(
        bootstrap_means,
        1.0 - alpha,
    )

    return (
        float(values.mean()),
        float(lower),
        float(upper),
    )


def bootstrap_paired_difference_ci(
    values_a,
    values_b,
    repeats=BOOTSTRAP_REPEATS,
    confidence_level=CONFIDENCE_LEVEL,
    seed=SEED,
):
    differences = (
        np.asarray(values_a, dtype=float)
        - np.asarray(values_b, dtype=float)
    )

    return bootstrap_mean_confidence_interval(
        values=differences,
        repeats=repeats,
        confidence_level=confidence_level,
        seed=seed,
    )


# ============================================================
# 9. RANDOM REPEAT CHECKPOINT HELPERS
# ============================================================

def random_repeat_file_path(
    repeat_directory,
    repeat_index,
):
    return (
        repeat_directory
        / f"random_repeat_{repeat_index:03d}.npz"
    )


def validate_random_repeat_file(
    repeat_file,
    expected_sequence_ids,
):
    if not repeat_file.exists():
        return False

    try:
        data = np.load(
            repeat_file,
            allow_pickle=True,
        )

        saved_ids = (
            data["sequence_id"]
            .astype(str)
        )

        saved_probabilities = data[
            "random_ablated_probability"
        ]

        valid = (
            np.array_equal(
                saved_ids,
                expected_sequence_ids.astype(str),
            )
            and len(saved_probabilities)
            == len(expected_sequence_ids)
        )

        data.close()

        return bool(valid)

    except Exception:
        return False


# ============================================================
# 10. ANALYZE ONE MODEL/DATASET
# ============================================================

def run_faithfulness_analysis(
    model_name,
    dataset_name,
    batch_size,
):
    output_dir = (
        FAITHFULNESS_ROOT
        / model_name
        / dataset_name
    )

    repeat_directory = (
        output_dir
        / "random_repeat_checkpoints"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    repeat_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    per_sequence_file = (
        output_dir
        / "per_sequence_faithfulness.csv"
    )

    random_repeat_summary_file = (
        output_dir
        / "random_ablation_repeat_summary.csv"
    )

    summary_file = (
        output_dir
        / "faithfulness_summary.json"
    )

    complete_file = (
        output_dir
        / "COMPLETE.json"
    )

    if (
        per_sequence_file.exists()
        and random_repeat_summary_file.exists()
        and summary_file.exists()
        and complete_file.exists()
    ):
        print(
            f"[SKIP COMPLETE] "
            f"{model_name} | {dataset_name}"
        )

        with open(
            summary_file,
            "r",
            encoding="utf-8",
        ) as handle:
            return json.load(handle)

    print("\n" + "=" * 78)
    print(
        f"FAITHFULNESS: "
        f"{model_name} — {dataset_name}"
    )
    print("=" * 78)

    model_file = (
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras"
    )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Trained model missing:\n"
            f"{model_file}"
        )

    model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    (
        embeddings_memmap,
        masks_memmap,
        metadata,
    ) = load_model_dataset(
        model_name,
        dataset_name,
    )

    embeddings = np.asarray(
        embeddings_memmap,
        dtype=np.float16,
    )

    valid_masks = np.asarray(
        masks_memmap,
        dtype=np.uint8,
    )

    (
        hotspot_masks,
        consensus_file,
    ) = load_consensus_hotspot_masks(
        dataset_name=dataset_name,
        metadata=metadata,
    )

    sequence_ids = (
        metadata["sequence_id"]
        .astype(str)
        .to_numpy()
    )

    labels = (
        metadata["label"]
        .astype(int)
        .to_numpy()
    )

    # --------------------------------------------------------
    # A. Original predictions
    # --------------------------------------------------------

    print("[PREDICT] Original embeddings")

    original_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    # --------------------------------------------------------
    # B. Hotspot ablation
    # --------------------------------------------------------

    print("[PREDICT] Consensus hotspot ablation")

    hotspot_ablated_embeddings = (
        ablate_positions(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_ablated_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=hotspot_ablated_embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    hotspot_probability_drop = (
        original_probabilities
        - hotspot_ablated_probabilities
    )

    del hotspot_ablated_embeddings
    gc.collect()

    # --------------------------------------------------------
    # C. Hotspot-only sufficiency
    # --------------------------------------------------------

    print("[PREDICT] Hotspot-only embeddings")

    hotspot_only_embeddings = (
        retain_positions_only(
            embeddings,
            hotspot_masks,
        )
    )

    hotspot_only_probabilities = (
        predict_probabilities(
            model=model,
            embeddings=hotspot_only_embeddings,
            masks=valid_masks,
            batch_size=batch_size,
        )
    )

    sufficiency_probability_loss = (
        original_probabilities
        - hotspot_only_probabilities
    )

    del hotspot_only_embeddings
    gc.collect()

    # --------------------------------------------------------
    # D. Restart-safe matched random controls
    # --------------------------------------------------------

    random_probability_matrix = np.zeros(
        (
            len(metadata),
            RANDOM_REPEATS,
        ),
        dtype=np.float32,
    )

    random_repeat_rows = []

    for repeat_index in range(
        1,
        RANDOM_REPEATS + 1,
    ):
        repeat_file = random_repeat_file_path(
            repeat_directory,
            repeat_index,
        )

        if validate_random_repeat_file(
            repeat_file=repeat_file,
            expected_sequence_ids=sequence_ids,
        ):
            repeat_data = np.load(
                repeat_file,
                allow_pickle=True,
            )

            random_probabilities = (
                repeat_data[
                    "random_ablated_probability"
                ].astype(np.float32)
            )

            repeat_data.close()

            print(
                f"[SKIP RANDOM] "
                f"{repeat_index}/{RANDOM_REPEATS}"
            )

        else:
            random_mask = (
                create_random_ablation_mask(
                    valid_mask=valid_masks,
                    hotspot_mask=hotspot_masks,
                    seed=(
                        SEED
                        + 1000
                        + repeat_index
                    ),
                )
            )

            random_ablated_embeddings = (
                ablate_positions(
                    embeddings,
                    random_mask,
                )
            )

            random_probabilities = (
                predict_probabilities(
                    model=model,
                    embeddings=random_ablated_embeddings,
                    masks=valid_masks,
                    batch_size=batch_size,
                )
            )

            temporary_repeat_file = Path(
                str(repeat_file)
                + ".temporary.npz"
            )

            np.savez_compressed(
                temporary_repeat_file,
                sequence_id=sequence_ids,
                random_ablated_probability=(
                    random_probabilities
                ),
                repeat=np.array(
                    [repeat_index],
                    dtype=np.int32,
                ),
                seed=np.array(
                    [
                        SEED
                        + 1000
                        + repeat_index
                    ],
                    dtype=np.int32,
                ),
            )

            temporary_repeat_file.replace(
                repeat_file
            )

            print(
                f"[SAVED RANDOM] "
                f"{repeat_index}/{RANDOM_REPEATS}"
            )

            del random_mask
            del random_ablated_embeddings
            gc.collect()

        random_probability_matrix[
            :,
            repeat_index - 1
        ] = random_probabilities

        random_drops = (
            original_probabilities
            - random_probabilities
        )

        cpp_mask = (
            labels == 1
        )

        noncpp_mask = (
            labels == 0
        )

        random_repeat_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "repeat": int(
                repeat_index
            ),
            "mean_random_drop_all": float(
                random_drops.mean()
            ),
            "mean_random_drop_CPP": float(
                random_drops[
                    cpp_mask
                ].mean()
            ),
            "mean_random_drop_nonCPP": float(
                random_drops[
                    noncpp_mask
                ].mean()
            ),
            "median_random_drop_CPP": float(
                np.median(
                    random_drops[
                        cpp_mask
                    ]
                )
            ),
            "median_random_drop_nonCPP": float(
                np.median(
                    random_drops[
                        noncpp_mask
                    ]
                )
            ),
        })

        del random_probabilities
        del random_drops
        gc.collect()

    random_repeat_df = pd.DataFrame(
        random_repeat_rows
    )

    atomic_save_csv(
        random_repeat_df,
        random_repeat_summary_file,
    )

    random_mean_probabilities = (
        random_probability_matrix.mean(
            axis=1
        )
    )

    random_probability_std = (
        random_probability_matrix.std(
            axis=1,
            ddof=0,
        )
    )

    random_mean_drop = (
        original_probabilities
        - random_mean_probabilities
    )

    # --------------------------------------------------------
    # E. Per-sequence table
    # --------------------------------------------------------

    per_sequence = metadata[
        [
            "sequence_id",
            "sequence",
            "label",
            "length",
        ]
    ].copy()

    per_sequence.insert(
        0,
        "model",
        model_name,
    )

    per_sequence.insert(
        1,
        "dataset",
        dataset_name,
    )

    per_sequence[
        "number_of_hotspots"
    ] = hotspot_masks.sum(
        axis=1
    )

    per_sequence[
        "hotspot_fraction"
    ] = (
        per_sequence[
            "number_of_hotspots"
        ]
        / per_sequence["length"]
    )

    per_sequence[
        "original_probability"
    ] = original_probabilities

    per_sequence[
        "hotspot_ablated_probability"
    ] = hotspot_ablated_probabilities

    per_sequence[
        "hotspot_probability_drop"
    ] = hotspot_probability_drop

    per_sequence[
        "random_mean_ablated_probability"
    ] = random_mean_probabilities

    per_sequence[
        "random_mean_probability_drop"
    ] = random_mean_drop

    per_sequence[
        "random_probability_std"
    ] = random_probability_std

    per_sequence[
        "hotspot_minus_random_drop"
    ] = (
        hotspot_probability_drop
        - random_mean_drop
    )

    per_sequence[
        "hotspot_ablation_stronger_than_random"
    ] = (
        per_sequence[
            "hotspot_probability_drop"
        ]
        > per_sequence[
            "random_mean_probability_drop"
        ]
    )

    per_sequence[
        "hotspot_only_probability"
    ] = hotspot_only_probabilities

    per_sequence[
        "sufficiency_probability_loss"
    ] = sufficiency_probability_loss

    per_sequence[
        "hotspot_only_probability_fraction"
    ] = np.divide(
        hotspot_only_probabilities,
        original_probabilities,
        out=np.full_like(
            hotspot_only_probabilities,
            np.nan,
            dtype=np.float32,
        ),
        where=(
            original_probabilities > 1e-8
        ),
    )

    atomic_save_csv(
        per_sequence,
        per_sequence_file,
    )

    # --------------------------------------------------------
    # F. Class-specific statistical summaries
    # --------------------------------------------------------

    summary = {
        "model": model_name,
        "dataset": dataset_name,
        "number_of_sequences": int(
            len(metadata)
        ),
        "number_of_CPPs": int(
            np.sum(labels == 1)
        ),
        "number_of_nonCPPs": int(
            np.sum(labels == 0)
        ),
        "hotspot_definition": (
            "adjusted global consensus top 20%"
        ),
        "random_repeats": int(
            RANDOM_REPEATS
        ),
        "bootstrap_repeats": int(
            BOOTSTRAP_REPEATS
        ),
        "consensus_file": str(
            consensus_file
        ),
        "classes": {},
    }

    class_definitions = [
        ("all", np.ones(
            len(metadata),
            dtype=bool,
        )),
        ("CPP", labels == 1),
        ("non_CPP", labels == 0),
    ]

    for class_name, class_mask in (
        class_definitions
    ):
        subset = per_sequence.loc[
            class_mask
        ].copy()

        hotspot_drops = subset[
            "hotspot_probability_drop"
        ].to_numpy()

        random_drops = subset[
            "random_mean_probability_drop"
        ].to_numpy()

        hotspot_minus_random = subset[
            "hotspot_minus_random_drop"
        ].to_numpy()

        hotspot_only_values = subset[
            "hotspot_only_probability"
        ].to_numpy()

        original_values = subset[
            "original_probability"
        ].to_numpy()

        wilcoxon_statistic, wilcoxon_p = (
            safe_wilcoxon(
                hotspot_drops,
                random_drops,
                alternative="greater",
            )
        )

        (
            hotspot_drop_mean,
            hotspot_drop_ci_low,
            hotspot_drop_ci_high,
        ) = bootstrap_mean_confidence_interval(
            hotspot_drops,
            seed=(
                SEED
                + len(class_name)
            ),
        )

        (
            random_drop_mean,
            random_drop_ci_low,
            random_drop_ci_high,
        ) = bootstrap_mean_confidence_interval(
            random_drops,
            seed=(
                SEED
                + 100
                + len(class_name)
            ),
        )

        (
            difference_mean,
            difference_ci_low,
            difference_ci_high,
        ) = bootstrap_paired_difference_ci(
            hotspot_drops,
            random_drops,
            seed=(
                SEED
                + 200
                + len(class_name)
            ),
        )

        summary[
            "classes"
        ][class_name] = {
            "n": int(
                len(subset)
            ),
            "mean_original_probability": float(
                original_values.mean()
            ),
            "median_original_probability": float(
                np.median(
                    original_values
                )
            ),
            "mean_hotspot_ablated_probability": float(
                subset[
                    "hotspot_ablated_probability"
                ].mean()
            ),
            "mean_hotspot_probability_drop": float(
                hotspot_drop_mean
            ),
            "hotspot_drop_ci95_low": float(
                hotspot_drop_ci_low
            ),
            "hotspot_drop_ci95_high": float(
                hotspot_drop_ci_high
            ),
            "median_hotspot_probability_drop": float(
                np.median(
                    hotspot_drops
                )
            ),
            "mean_random_probability_drop": float(
                random_drop_mean
            ),
            "random_drop_ci95_low": float(
                random_drop_ci_low
            ),
            "random_drop_ci95_high": float(
                random_drop_ci_high
            ),
            "median_random_probability_drop": float(
                np.median(
                    random_drops
                )
            ),
            "mean_hotspot_minus_random_drop": float(
                difference_mean
            ),
            "difference_ci95_low": float(
                difference_ci_low
            ),
            "difference_ci95_high": float(
                difference_ci_high
            ),
            "fraction_hotspot_stronger_than_random": float(
                subset[
                    "hotspot_ablation_stronger_than_random"
                ].mean()
            ),
            "paired_cohens_dz": float(
                paired_cohens_dz(
                    hotspot_drops,
                    random_drops,
                )
            ),
            "rank_biserial_effect_size": float(
                rank_biserial_effect(
                    hotspot_drops,
                    random_drops,
                )
            ),
            "wilcoxon_alternative": (
                "hotspot drop > random drop"
            ),
            "wilcoxon_statistic": float(
                wilcoxon_statistic
            ),
            "wilcoxon_p_value": float(
                wilcoxon_p
            ),
            "mean_hotspot_only_probability": float(
                hotspot_only_values.mean()
            ),
            "median_hotspot_only_probability": float(
                np.median(
                    hotspot_only_values
                )
            ),
            "mean_sufficiency_probability_loss": float(
                subset[
                    "sufficiency_probability_loss"
                ].mean()
            ),
            "mean_hotspot_only_probability_fraction": float(
                np.nanmean(
                    subset[
                        "hotspot_only_probability_fraction"
                    ]
                )
            ),
        }

    atomic_save_json(
        summary_file,
        summary,
    )

    atomic_save_json(
        complete_file,
        summary,
    )

    cpp_summary = (
        summary["classes"]["CPP"]
    )

    print(
        "[COMPLETE]",
        model_name,
        dataset_name,
    )

    print(
        "CPP hotspot drop       :",
        cpp_summary[
            "mean_hotspot_probability_drop"
        ],
    )

    print(
        "CPP random drop        :",
        cpp_summary[
            "mean_random_probability_drop"
        ],
    )

    print(
        "CPP hotspot - random   :",
        cpp_summary[
            "mean_hotspot_minus_random_drop"
        ],
    )

    print(
        "CPP Wilcoxon p-value   :",
        cpp_summary[
            "wilcoxon_p_value"
        ],
    )

    print(
        "CPP paired Cohen's dz  :",
        cpp_summary[
            "paired_cohens_dz"
        ],
    )

    del model
    del embeddings_memmap
    del masks_memmap
    del embeddings
    del valid_masks
    del hotspot_masks
    del random_probability_matrix

    clear_memory()

    return summary


# ============================================================
# 11. RUN ALL MODEL/DATASET COMBINATIONS
# ============================================================

summary_records = []
all_output_files = []

for model_name, config in (
    MODEL_CONFIGS.items()
):
    for dataset_name in DATASETS:
        summary = run_faithfulness_analysis(
            model_name=model_name,
            dataset_name=dataset_name,
            batch_size=config[
                "batch_size"
            ],
        )

        for class_name, values in (
            summary["classes"].items()
        ):
            summary_records.append({
                "model": model_name,
                "dataset": dataset_name,
                "class": class_name,
                **values,
            })

        output_dir = (
            FAITHFULNESS_ROOT
            / model_name
            / dataset_name
        )

        all_output_files.extend([
            output_dir
            / "per_sequence_faithfulness.csv",
            output_dir
            / "random_ablation_repeat_summary.csv",
            output_dir
            / "faithfulness_summary.json",
            output_dir
            / "COMPLETE.json",
        ])


# ============================================================
# 12. MASTER SUMMARY TABLE
# ============================================================

faithfulness_summary_df = pd.DataFrame(
    summary_records
)

master_summary_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_summary.csv"
)

atomic_save_csv(
    faithfulness_summary_df,
    master_summary_file,
)

all_output_files.append(
    master_summary_file
)


# ============================================================
# 13. CPP-ONLY MANUSCRIPT TABLE
# ============================================================

cpp_summary_df = (
    faithfulness_summary_df[
        faithfulness_summary_df[
            "class"
        ] == "CPP"
    ]
    .copy()
)

cpp_manuscript_columns = [
    "model",
    "dataset",
    "n",
    "mean_original_probability",
    "mean_hotspot_probability_drop",
    "hotspot_drop_ci95_low",
    "hotspot_drop_ci95_high",
    "mean_random_probability_drop",
    "random_drop_ci95_low",
    "random_drop_ci95_high",
    "mean_hotspot_minus_random_drop",
    "difference_ci95_low",
    "difference_ci95_high",
    "fraction_hotspot_stronger_than_random",
    "paired_cohens_dz",
    "rank_biserial_effect_size",
    "wilcoxon_p_value",
    "mean_hotspot_only_probability",
    "mean_hotspot_only_probability_fraction",
]

cpp_manuscript_table = (
    cpp_summary_df[
        cpp_manuscript_columns
    ]
    .copy()
)

cpp_manuscript_file = (
    RESULT_TABLE_DIR
    / "CPP_hotspot_faithfulness_manuscript_table.csv"
)

atomic_save_csv(
    cpp_manuscript_table,
    cpp_manuscript_file,
)

all_output_files.append(
    cpp_manuscript_file
)


# ============================================================
# 14. CROSS-MODEL SUMMARY
# ============================================================

cross_model_numeric_columns = [
    "mean_original_probability",
    "mean_hotspot_probability_drop",
    "mean_random_probability_drop",
    "mean_hotspot_minus_random_drop",
    "fraction_hotspot_stronger_than_random",
    "paired_cohens_dz",
    "rank_biserial_effect_size",
    "mean_hotspot_only_probability",
    "mean_hotspot_only_probability_fraction",
]

cross_model_summary = (
    faithfulness_summary_df
    .groupby(
        [
            "dataset",
            "class",
        ],
        as_index=False,
    )[cross_model_numeric_columns]
    .mean()
)

cross_model_file = (
    RESULT_TABLE_DIR
    / "hotspot_faithfulness_cross_model_summary.csv"
)

atomic_save_csv(
    cross_model_summary,
    cross_model_file,
)

all_output_files.append(
    cross_model_file
)


# ============================================================
# 15. COMBINE ALL PER-SEQUENCE OUTPUTS
# ============================================================

all_per_sequence_tables = []

for model_name in MODEL_CONFIGS:
    for dataset_name in DATASETS:
        per_sequence_file = (
            FAITHFULNESS_ROOT
            / model_name
            / dataset_name
            / "per_sequence_faithfulness.csv"
        )

        all_per_sequence_tables.append(
            pd.read_csv(
                per_sequence_file
            )
        )

combined_per_sequence_df = pd.concat(
    all_per_sequence_tables,
    ignore_index=True,
)

combined_per_sequence_file = (
    RESULT_SI_DIR
    / "faithfulness_all_sequences_all_models.csv"
)

atomic_save_csv(
    combined_per_sequence_df,
    combined_per_sequence_file,
)

all_output_files.append(
    combined_per_sequence_file
)


# ============================================================
# 16. FIGURE 1 — HOTSPOT VS RANDOM DROP BY MODEL
# ============================================================

cpp_plot_data = (
    combined_per_sequence_df[
        combined_per_sequence_df[
            "label"
        ] == 1
    ]
    .copy()
)

models = list(
    MODEL_CONFIGS.keys()
)

datasets_display = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

for dataset_name in DATASETS:
    dataset_subset = cpp_plot_data[
        cpp_plot_data["dataset"]
        == dataset_name
    ]

    hotspot_values = [
        dataset_subset[
            dataset_subset["model"]
            == model_name
        ][
            "hotspot_probability_drop"
        ].to_numpy()
        for model_name in models
    ]

    random_values = [
        dataset_subset[
            dataset_subset["model"]
            == model_name
        ][
            "random_mean_probability_drop"
        ].to_numpy()
        for model_name in models
    ]

    positions_hotspot = (
        np.arange(
            len(models)
        ) * 3.0
    )

    positions_random = (
        positions_hotspot + 1.0
    )

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    hotspot_box = ax.boxplot(
        hotspot_values,
        positions=positions_hotspot,
        widths=0.75,
        patch_artist=False,
        showfliers=False,
        medianprops={
            "linewidth": 1.5,
        },
    )

    random_box = ax.boxplot(
        random_values,
        positions=positions_random,
        widths=0.75,
        patch_artist=False,
        showfliers=False,
        medianprops={
            "linewidth": 1.5,
        },
    )

    ax.axhline(
        0.0,
        linewidth=1.0,
        linestyle="--",
    )

    ax.set_xticks(
        positions_hotspot + 0.5
    )

    ax.set_xticklabels(
        models,
        rotation=20,
        ha="right",
    )

    ax.set_ylabel(
        "Decrease in predicted CPP probability"
    )

    ax.set_title(
        f"Consensus-hotspot ablation versus "
        f"matched random ablation\n"
        f"{datasets_display[dataset_name]} CPPs"
    )

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            linewidth=2,
            label="Consensus hotspots",
        ),
        plt.Line2D(
            [0],
            [0],
            linewidth=2,
            linestyle="--",
            label="Matched random residues",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        frameon=False,
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / f"faithfulness_hotspot_vs_random_{dataset_name}"
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    all_output_files.extend([
        figure_base.with_suffix(".png"),
        figure_base.with_suffix(".pdf"),
        figure_base.with_suffix(".svg"),
    ])


# ============================================================
# 17. FIGURE 2 — MEAN DROPS WITH 95% CI
# ============================================================

for dataset_name in DATASETS:
    table_subset = cpp_manuscript_table[
        cpp_manuscript_table[
            "dataset"
        ] == dataset_name
    ].copy()

    table_subset["model"] = pd.Categorical(
        table_subset["model"],
        categories=models,
        ordered=True,
    )

    table_subset = table_subset.sort_values(
        "model"
    )

    x_positions = np.arange(
        len(table_subset)
    )

    hotspot_means = table_subset[
        "mean_hotspot_probability_drop"
    ].to_numpy()

    hotspot_lower = (
        hotspot_means
        - table_subset[
            "hotspot_drop_ci95_low"
        ].to_numpy()
    )

    hotspot_upper = (
        table_subset[
            "hotspot_drop_ci95_high"
        ].to_numpy()
        - hotspot_means
    )

    random_means = table_subset[
        "mean_random_probability_drop"
    ].to_numpy()

    random_lower = (
        random_means
        - table_subset[
            "random_drop_ci95_low"
        ].to_numpy()
    )

    random_upper = (
        table_subset[
            "random_drop_ci95_high"
        ].to_numpy()
        - random_means
    )

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    ax.errorbar(
        x_positions - 0.12,
        hotspot_means,
        yerr=np.vstack([
            hotspot_lower,
            hotspot_upper,
        ]),
        fmt="o",
        capsize=4,
        label="Consensus hotspot ablation",
    )

    ax.errorbar(
        x_positions + 0.12,
        random_means,
        yerr=np.vstack([
            random_lower,
            random_upper,
        ]),
        fmt="s",
        capsize=4,
        label="Matched random ablation",
    )

    ax.axhline(
        0.0,
        linewidth=1.0,
        linestyle="--",
    )

    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        table_subset[
            "model"
        ].astype(str),
        rotation=20,
        ha="right",
    )

    ax.set_ylabel(
        "Mean decrease in CPP probability"
    )

    ax.set_title(
        f"Faithfulness of consensus hotspots "
        f"across PLMs\n"
        f"{datasets_display[dataset_name]}"
    )

    ax.legend(
        frameon=False,
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / f"faithfulness_mean_CI_{dataset_name}"
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    all_output_files.extend([
        figure_base.with_suffix(".png"),
        figure_base.with_suffix(".pdf"),
        figure_base.with_suffix(".svg"),
    ])


# ============================================================
# 18. FIGURE 3 — ORIGINAL VS HOTSPOT-ONLY SUFFICIENCY
# ============================================================

for dataset_name in DATASETS:
    dataset_subset = cpp_plot_data[
        cpp_plot_data[
            "dataset"
        ] == dataset_name
    ]

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    for model_name in models:
        model_subset = dataset_subset[
            dataset_subset[
                "model"
            ] == model_name
        ]

        ax.scatter(
            model_subset[
                "original_probability"
            ],
            model_subset[
                "hotspot_only_probability"
            ],
            s=16,
            alpha=0.45,
            label=model_name,
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_xlim(
        0,
        1.02,
    )

    ax.set_ylim(
        0,
        1.02,
    )

    ax.set_xlabel(
        "Original predicted CPP probability"
    )

    ax.set_ylabel(
        "Hotspot-only predicted CPP probability"
    )

    ax.set_title(
        f"Predictive sufficiency of consensus hotspots\n"
        f"{datasets_display[dataset_name]} CPPs"
    )

    ax.legend(
        frameon=False,
        fontsize=8,
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_SI_DIR
        / f"faithfulness_sufficiency_scatter_{dataset_name}"
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    all_output_files.extend([
        figure_base.with_suffix(".png"),
        figure_base.with_suffix(".pdf"),
        figure_base.with_suffix(".svg"),
    ])


# ============================================================
# 19. FIGURE 4 — EFFECT SIZE BY MODEL
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

for dataset_index, dataset_name in enumerate(
    DATASETS
):
    subset = cpp_manuscript_table[
        cpp_manuscript_table[
            "dataset"
        ] == dataset_name
    ].copy()

    subset["model"] = pd.Categorical(
        subset["model"],
        categories=models,
        ordered=True,
    )

    subset = subset.sort_values(
        "model"
    )

    x_positions = (
        np.arange(
            len(models)
        )
        + dataset_index * 0.16
        - 0.08
    )

    ax.scatter(
        x_positions,
        subset[
            "paired_cohens_dz"
        ],
        s=55,
        label=datasets_display[
            dataset_name
        ],
    )

ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

ax.axhline(
    0.8,
    linestyle=":",
    linewidth=1.0,
)

ax.set_xticks(
    np.arange(
        len(models)
    )
)

ax.set_xticklabels(
    models,
    rotation=20,
    ha="right",
)

ax.set_ylabel(
    "Paired Cohen's $d_z$\n"
    "(hotspot drop − random drop)"
)

ax.set_title(
    "Effect size of consensus-hotspot ablation"
)

ax.legend(
    frameon=False,
)

fig.tight_layout()

effect_figure_base = (
    FIGURE_MAIN_DIR
    / "faithfulness_effect_sizes"
)

fig.savefig(
    effect_figure_base.with_suffix(".png"),
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    effect_figure_base.with_suffix(".pdf"),
    bbox_inches="tight",
)

fig.savefig(
    effect_figure_base.with_suffix(".svg"),
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

all_output_files.extend([
    effect_figure_base.with_suffix(".png"),
    effect_figure_base.with_suffix(".pdf"),
    effect_figure_base.with_suffix(".svg"),
])


# ============================================================
# 20. SAVE ANALYSIS DESCRIPTION
# ============================================================

analysis_description = {
    "analysis": (
        "Consensus-hotspot faithfulness and "
        "matched random perturbation"
    ),
    "hotspot_definition": (
        "Top 20% redundancy-adjusted global "
        "consensus residue ranks"
    ),
    "ablation_operation": (
        "Selected fixed PLM residue embeddings "
        "were replaced by zero vectors while "
        "sequence masks remained unchanged."
    ),
    "random_control": (
        "The same number of valid non-hotspot "
        "residues was randomly selected per sequence."
    ),
    "random_repeats": int(
        RANDOM_REPEATS
    ),
    "sufficiency_operation": (
        "Only hotspot residue embeddings were "
        "retained; all other residue embeddings "
        "were replaced by zero vectors."
    ),
    "statistical_test": (
        "One-sided paired Wilcoxon signed-rank test: "
        "hotspot probability drop > mean matched "
        "random probability drop."
    ),
    "effect_sizes": [
        "paired Cohen's dz",
        "matched-pairs rank-biserial effect size",
    ],
    "confidence_intervals": (
        f"{int(CONFIDENCE_LEVEL * 100)}% percentile "
        f"bootstrap intervals with "
        f"{BOOTSTRAP_REPEATS} resamples"
    ),
    "important_limitation": (
        "This analysis tests downstream-classifier "
        "faithfulness using fixed PLM residue embeddings. "
        "It is not equivalent to amino-acid mutation followed "
        "by re-embedding with the original PLM."
    ),
    "models": list(
        MODEL_CONFIGS.keys()
    ),
    "datasets": DATASETS,
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

analysis_description_file = (
    CHECKPOINT_ROOT
    / "faithfulness_analysis_method.json"
)

atomic_save_json(
    analysis_description_file,
    analysis_description,
)

all_output_files.append(
    analysis_description_file
)


# ============================================================
# 21. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "16_publication_ready_hotspot_faithfulness",
    output_files=all_output_files,
    details=analysis_description,
)


# ============================================================
# 22. DISPLAY FINAL TABLES
# ============================================================

print("\n" + "=" * 78)
print("CPP HOTSPOT FAITHFULNESS — MANUSCRIPT TABLE")
print("=" * 78)

display(
    cpp_manuscript_table[
        [
            "model",
            "dataset",
            "n",
            "mean_original_probability",
            "mean_hotspot_probability_drop",
            "hotspot_drop_ci95_low",
            "hotspot_drop_ci95_high",
            "mean_random_probability_drop",
            "random_drop_ci95_low",
            "random_drop_ci95_high",
            "mean_hotspot_minus_random_drop",
            "difference_ci95_low",
            "difference_ci95_high",
            "fraction_hotspot_stronger_than_random",
            "paired_cohens_dz",
            "rank_biserial_effect_size",
            "wilcoxon_p_value",
            "mean_hotspot_only_probability",
        ]
    ]
)

print("\n" + "=" * 78)
print("CROSS-MODEL CPP SUMMARY")
print("=" * 78)

display(
    cross_model_summary[
        cross_model_summary[
            "class"
        ] == "CPP"
    ]
)

print("\nSaved main summary:")
print(master_summary_file)

print("\nSaved manuscript table:")
print(cpp_manuscript_file)

print("\nSaved combined per-sequence SI table:")
print(combined_per_sequence_file)

print("\nSaved cross-model summary:")
print(cross_model_file)

print("\n" + "=" * 78)
print("STEP 16 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 17: CROSS-PLM HOTSPOT CONSERVATION
#
# Uses adjusted per-model top-20% hotspots.
#
# Analyses:
# 1. Pairwise Jaccard similarity between PLM hotspot sets
# 2. Residue support by 1, 2, 3, or all 4 PLMs
# 3. Per-sequence cross-model conservation
# 4. Unique versus shared hotspots
# 5. CPP versus non-CPP comparison
# 6. Internal versus KELM replication
# ============================================================

from pathlib import Path
from itertools import combinations
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    mannwhitneyu,
    fisher_exact,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAMES = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

HOTSPOT_RANK_THRESHOLD = 0.80

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

CONSERVATION_ROOT = (
    XAI_ROOT / "hotspot_conservation"
)

for folder in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
    CONSERVATION_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. LOAD ADJUSTED PER-MODEL CONSENSUS
# ============================================================

def load_model_hotspots(
    model_name,
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / model_name
        / dataset_name
        / "adjusted_model_consensus.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Missing adjusted model consensus:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "adjusted_model_consensus_rank",
    }

    missing = (
        required_columns
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            f"{model_name}/{dataset_name} "
            f"is missing columns: {sorted(missing)}"
        )

    dataframe[
        f"hotspot_{model_name}"
    ] = (
        dataframe[
            "adjusted_model_consensus_rank"
        ]
        >= HOTSPOT_RANK_THRESHOLD
    )

    return dataframe


# ============================================================
# 3. MERGE ALL FOUR PLMS
# ============================================================

identity_columns = [
    "sequence_id",
    "sequence",
    "label",
    "sequence_length",
    "position",
    "normalized_position",
    "residue",
]

merged_dataset_tables = {}

for dataset_name in DATASETS:

    merged = None

    for model_name in MODEL_NAMES:

        current = load_model_hotspots(
            model_name=model_name,
            dataset_name=dataset_name,
        )

        current = current[
            identity_columns
            + [
                "adjusted_model_consensus_rank",
                f"hotspot_{model_name}",
            ]
        ].copy()

        current = current.rename(
            columns={
                "adjusted_model_consensus_rank":
                    f"rank_{model_name}"
            }
        )

        if merged is None:
            merged = current
        else:
            merged = merged.merge(
                current,
                on=identity_columns,
                how="inner",
                validate="one_to_one",
            )

    hotspot_columns = [
        f"hotspot_{model_name}"
        for model_name in MODEL_NAMES
    ]

    merged[
        "model_support_count"
    ] = (
        merged[
            hotspot_columns
        ]
        .astype(int)
        .sum(axis=1)
    )

    merged[
        "supported_by_at_least_2"
    ] = (
        merged["model_support_count"] >= 2
    )

    merged[
        "supported_by_at_least_3"
    ] = (
        merged["model_support_count"] >= 3
    )

    merged[
        "supported_by_all_4"
    ] = (
        merged["model_support_count"] == 4
    )

    merged[
        "model_unique_hotspot"
    ] = (
        merged["model_support_count"] == 1
    )

    merged[
        "any_model_hotspot"
    ] = (
        merged["model_support_count"] >= 1
    )

    rank_columns = [
        f"rank_{model_name}"
        for model_name in MODEL_NAMES
    ]

    merged[
        "mean_model_rank"
    ] = merged[
        rank_columns
    ].mean(axis=1)

    merged[
        "model_rank_std"
    ] = merged[
        rank_columns
    ].std(
        axis=1,
        ddof=0,
    )

    output_dir = (
        CONSERVATION_ROOT
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    residue_output_file = (
        output_dir
        / "cross_plm_residue_support.csv"
    )

    merged.to_csv(
        residue_output_file,
        index=False,
    )

    merged_dataset_tables[
        dataset_name
    ] = merged

    print(
        f"[MERGED] {dataset_name}: "
        f"{len(merged)} residues"
    )


# ============================================================
# 4. PAIRWISE JACCARD PER SEQUENCE
# ============================================================

pairwise_rows = []
per_sequence_pairwise_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for model_a, model_b in combinations(
        MODEL_NAMES,
        2,
    ):

        column_a = (
            f"hotspot_{model_a}"
        )

        column_b = (
            f"hotspot_{model_b}"
        )

        sequence_jaccards = []

        cpp_jaccards = []
        noncpp_jaccards = []

        for sequence_id, group in (
            dataframe.groupby(
                "sequence_id",
                sort=False,
            )
        ):

            set_a = set(
                group.loc[
                    group[column_a].astype(bool),
                    "position",
                ].astype(int)
            )

            set_b = set(
                group.loc[
                    group[column_b].astype(bool),
                    "position",
                ].astype(int)
            )

            union = (
                set_a | set_b
            )

            intersection = (
                set_a & set_b
            )

            if len(union) == 0:
                jaccard = 1.0
            else:
                jaccard = (
                    len(intersection)
                    / len(union)
                )

            label = int(
                group["label"].iloc[0]
            )

            sequence_jaccards.append(
                jaccard
            )

            if label == 1:
                cpp_jaccards.append(
                    jaccard
                )
            else:
                noncpp_jaccards.append(
                    jaccard
                )

            per_sequence_pairwise_rows.append({
                "dataset": dataset_name,
                "sequence_id": sequence_id,
                "label": label,
                "model_a": model_a,
                "model_b": model_b,
                "intersection_size": int(
                    len(intersection)
                ),
                "union_size": int(
                    len(union)
                ),
                "jaccard_similarity": float(
                    jaccard
                ),
            })

        pairwise_rows.append({
            "dataset": dataset_name,
            "model_a": model_a,
            "model_b": model_b,
            "number_of_sequences": int(
                len(sequence_jaccards)
            ),
            "mean_jaccard_all": float(
                np.mean(
                    sequence_jaccards
                )
            ),
            "median_jaccard_all": float(
                np.median(
                    sequence_jaccards
                )
            ),
            "mean_jaccard_CPP": float(
                np.mean(
                    cpp_jaccards
                )
            ),
            "median_jaccard_CPP": float(
                np.median(
                    cpp_jaccards
                )
            ),
            "mean_jaccard_nonCPP": float(
                np.mean(
                    noncpp_jaccards
                )
            ),
            "median_jaccard_nonCPP": float(
                np.median(
                    noncpp_jaccards
                )
            ),
        })


pairwise_df = pd.DataFrame(
    pairwise_rows
)

per_sequence_pairwise_df = (
    pd.DataFrame(
        per_sequence_pairwise_rows
    )
)

pairwise_file = (
    RESULT_TABLE_DIR
    / "cross_plm_pairwise_hotspot_jaccard.csv"
)

pairwise_sequence_file = (
    RESULT_SI_DIR
    / "cross_plm_pairwise_jaccard_per_sequence.csv"
)

pairwise_df.to_csv(
    pairwise_file,
    index=False,
)

per_sequence_pairwise_df.to_csv(
    pairwise_sequence_file,
    index=False,
)


# ============================================================
# 5. SUPPORT DISTRIBUTION
# ============================================================

support_distribution_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
        ("all", "all"),
    ]:

        if label_value == "all":
            subset = dataframe
        else:
            subset = dataframe[
                dataframe["label"]
                == label_value
            ]

        total_residues = len(
            subset
        )

        hotspot_union = subset[
            subset[
                "model_support_count"
            ] >= 1
        ]

        union_total = len(
            hotspot_union
        )

        for support_count in [
            0,
            1,
            2,
            3,
            4,
        ]:

            count = int(
                (
                    subset[
                        "model_support_count"
                    ]
                    == support_count
                ).sum()
            )

            support_distribution_rows.append({
                "dataset": dataset_name,
                "class": class_name,
                "model_support_count": (
                    support_count
                ),
                "residue_count": count,
                "fraction_all_residues": float(
                    count / total_residues
                ),
                "fraction_hotspot_union": (
                    float(
                        count / union_total
                    )
                    if (
                        support_count > 0
                        and union_total > 0
                    )
                    else 0.0
                ),
            })


support_distribution_df = pd.DataFrame(
    support_distribution_rows
)

support_distribution_file = (
    RESULT_TABLE_DIR
    / "cross_plm_hotspot_support_distribution.csv"
)

support_distribution_df.to_csv(
    support_distribution_file,
    index=False,
)


# ============================================================
# 6. PER-SEQUENCE CONSERVATION SUMMARY
# ============================================================

per_sequence_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for sequence_id, group in (
        dataframe.groupby(
            "sequence_id",
            sort=False,
        )
    ):

        sequence_length = int(
            group[
                "sequence_length"
            ].iloc[0]
        )

        support_counts = group[
            "model_support_count"
        ].to_numpy()

        any_hotspot_count = int(
            np.sum(
                support_counts >= 1
            )
        )

        at_least_2_count = int(
            np.sum(
                support_counts >= 2
            )
        )

        at_least_3_count = int(
            np.sum(
                support_counts >= 3
            )
        )

        unanimous_count = int(
            np.sum(
                support_counts == 4
            )
        )

        unique_count = int(
            np.sum(
                support_counts == 1
            )
        )

        mean_pairwise_jaccard = (
            per_sequence_pairwise_df[
                (
                    per_sequence_pairwise_df[
                        "dataset"
                    ] == dataset_name
                )
                & (
                    per_sequence_pairwise_df[
                        "sequence_id"
                    ] == sequence_id
                )
            ][
                "jaccard_similarity"
            ].mean()
        )

        per_sequence_rows.append({
            "dataset": dataset_name,
            "sequence_id": sequence_id,
            "sequence": str(
                group["sequence"].iloc[0]
            ),
            "label": int(
                group["label"].iloc[0]
            ),
            "sequence_length": (
                sequence_length
            ),
            "any_model_hotspot_count": (
                any_hotspot_count
            ),
            "at_least_2_models_count": (
                at_least_2_count
            ),
            "at_least_3_models_count": (
                at_least_3_count
            ),
            "all_4_models_count": (
                unanimous_count
            ),
            "model_unique_count": (
                unique_count
            ),
            "fraction_supported_by_at_least_2": float(
                at_least_2_count
                / sequence_length
            ),
            "fraction_supported_by_at_least_3": float(
                at_least_3_count
                / sequence_length
            ),
            "fraction_supported_by_all_4": float(
                unanimous_count
                / sequence_length
            ),
            "fraction_unique_to_one_model": float(
                unique_count
                / sequence_length
            ),
            "mean_pairwise_jaccard": float(
                mean_pairwise_jaccard
            ),
        })


per_sequence_df = pd.DataFrame(
    per_sequence_rows
)

per_sequence_file = (
    RESULT_SI_DIR
    / "cross_plm_hotspot_conservation_per_sequence.csv"
)

per_sequence_df.to_csv(
    per_sequence_file,
    index=False,
)


# ============================================================
# 7. CPP VS NON-CPP CONSERVATION STATISTICS
# ============================================================

class_comparison_rows = []

metrics_to_compare = [
    "fraction_supported_by_at_least_2",
    "fraction_supported_by_at_least_3",
    "fraction_supported_by_all_4",
    "fraction_unique_to_one_model",
    "mean_pairwise_jaccard",
]

for dataset_name in DATASETS:

    dataset_table = (
        per_sequence_df[
            per_sequence_df["dataset"]
            == dataset_name
        ]
    )

    cpp = dataset_table[
        dataset_table["label"] == 1
    ]

    noncpp = dataset_table[
        dataset_table["label"] == 0
    ]

    for metric in metrics_to_compare:

        statistic, p_value = (
            mannwhitneyu(
                cpp[metric],
                noncpp[metric],
                alternative="two-sided",
            )
        )

        class_comparison_rows.append({
            "dataset": dataset_name,
            "metric": metric,
            "CPP_mean": float(
                cpp[metric].mean()
            ),
            "CPP_median": float(
                cpp[metric].median()
            ),
            "nonCPP_mean": float(
                noncpp[metric].mean()
            ),
            "nonCPP_median": float(
                noncpp[metric].median()
            ),
            "mannwhitney_statistic": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })


class_comparison_df = pd.DataFrame(
    class_comparison_rows
)

class_comparison_file = (
    RESULT_TABLE_DIR
    / "CPP_vs_nonCPP_hotspot_conservation.csv"
)

class_comparison_df.to_csv(
    class_comparison_file,
    index=False,
)


# ============================================================
# 8. RESIDUE-LEVEL CPP VS NON-CPP CONSERVATION
# ============================================================

residue_class_rows = []

for dataset_name, dataframe in (
    merged_dataset_tables.items()
):

    for minimum_support in [
        2,
        3,
        4,
    ]:

        cpp = dataframe[
            dataframe["label"] == 1
        ]

        noncpp = dataframe[
            dataframe["label"] == 0
        ]

        cpp_supported = int(
            (
                cpp[
                    "model_support_count"
                ] >= minimum_support
            ).sum()
        )

        cpp_not_supported = int(
            len(cpp) - cpp_supported
        )

        noncpp_supported = int(
            (
                noncpp[
                    "model_support_count"
                ] >= minimum_support
            ).sum()
        )

        noncpp_not_supported = int(
            len(noncpp)
            - noncpp_supported
        )

        odds_ratio, p_value = (
            fisher_exact(
                [
                    [
                        cpp_supported,
                        cpp_not_supported,
                    ],
                    [
                        noncpp_supported,
                        noncpp_not_supported,
                    ],
                ],
                alternative="two-sided",
            )
        )

        corrected_odds_ratio = (
            (cpp_supported + 0.5)
            * (
                noncpp_not_supported
                + 0.5
            )
            / (
                (cpp_not_supported + 0.5)
                * (
                    noncpp_supported
                    + 0.5
                )
            )
        )

        residue_class_rows.append({
            "dataset": dataset_name,
            "minimum_model_support": (
                minimum_support
            ),
            "CPP_supported_count": (
                cpp_supported
            ),
            "CPP_total_residues": int(
                len(cpp)
            ),
            "CPP_supported_fraction": float(
                cpp_supported / len(cpp)
            ),
            "nonCPP_supported_count": (
                noncpp_supported
            ),
            "nonCPP_total_residues": int(
                len(noncpp)
            ),
            "nonCPP_supported_fraction": float(
                noncpp_supported
                / len(noncpp)
            ),
            "odds_ratio_scipy": float(
                odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_odds_ratio
            ),
            "p_value": float(
                p_value
            ),
        })


residue_class_df = pd.DataFrame(
    residue_class_rows
)

residue_class_file = (
    RESULT_TABLE_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

residue_class_df.to_csv(
    residue_class_file,
    index=False,
)


# ============================================================
# 9. JACCARD MATRICES
# ============================================================

jaccard_matrix_files = []

for dataset_name in DATASETS:

    for class_column, class_name in [
        ("mean_jaccard_all", "all"),
        ("mean_jaccard_CPP", "CPP"),
        ("mean_jaccard_nonCPP", "nonCPP"),
    ]:

        matrix = pd.DataFrame(
            np.eye(
                len(MODEL_NAMES)
            ),
            index=MODEL_NAMES,
            columns=MODEL_NAMES,
        )

        subset = pairwise_df[
            pairwise_df["dataset"]
            == dataset_name
        ]

        for row in subset.itertuples(
            index=False
        ):

            matrix.loc[
                row.model_a,
                row.model_b,
            ] = getattr(
                row,
                class_column
            )

            matrix.loc[
                row.model_b,
                row.model_a,
            ] = getattr(
                row,
                class_column
            )

        matrix_file = (
            RESULT_TABLE_DIR
            / (
                f"hotspot_jaccard_matrix_"
                f"{dataset_name}_{class_name}.csv"
            )
        )

        matrix.to_csv(
            matrix_file
        )

        jaccard_matrix_files.append(
            matrix_file
        )

        # Plot matrix
        fig, ax = plt.subplots(
            figsize=(6.5, 5.5)
        )

        image = ax.imshow(
            matrix.to_numpy(),
            vmin=0,
            vmax=1,
            aspect="equal",
        )

        ax.set_xticks(
            np.arange(
                len(MODEL_NAMES)
            )
        )

        ax.set_yticks(
            np.arange(
                len(MODEL_NAMES)
            )
        )

        ax.set_xticklabels(
            MODEL_NAMES,
            rotation=35,
            ha="right",
        )

        ax.set_yticklabels(
            MODEL_NAMES
        )

        for row_index in range(
            len(MODEL_NAMES)
        ):
            for column_index in range(
                len(MODEL_NAMES)
            ):

                ax.text(
                    column_index,
                    row_index,
                    f"{matrix.iloc[row_index, column_index]:.2f}",
                    ha="center",
                    va="center",
                )

        ax.set_title(
            f"Cross-PLM hotspot Jaccard similarity\n"
            f"{dataset_name} — {class_name}"
        )

        figure_colorbar = fig.colorbar(
            image,
            ax=ax,
        )

        figure_colorbar.set_label(
            "Mean per-sequence Jaccard similarity"
        )

        fig.tight_layout()

        figure_base = (
            FIGURE_MAIN_DIR
            / (
                f"hotspot_jaccard_"
                f"{dataset_name}_{class_name}"
            )
        )

        fig.savefig(
            figure_base.with_suffix(".png"),
            dpi=600,
            bbox_inches="tight",
        )

        fig.savefig(
            figure_base.with_suffix(".pdf"),
            bbox_inches="tight",
        )

        fig.savefig(
            figure_base.with_suffix(".svg"),
            bbox_inches="tight",
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 10. SUPPORT-COUNT BAR PLOTS
# ============================================================

for dataset_name in DATASETS:

    plot_subset = (
        support_distribution_df[
            (
                support_distribution_df[
                    "dataset"
                ] == dataset_name
            )
            & (
                support_distribution_df[
                    "class"
                ].isin(
                    [
                        "CPP",
                        "non_CPP",
                    ]
                )
            )
            & (
                support_distribution_df[
                    "model_support_count"
                ] > 0
            )
        ]
    )

    support_counts = [
        1,
        2,
        3,
        4,
    ]

    cpp_values = [
        float(
            plot_subset[
                (
                    plot_subset["class"]
                    == "CPP"
                )
                & (
                    plot_subset[
                        "model_support_count"
                    ] == support
                )
            ][
                "fraction_hotspot_union"
            ].iloc[0]
        )
        for support in support_counts
    ]

    noncpp_values = [
        float(
            plot_subset[
                (
                    plot_subset["class"]
                    == "non_CPP"
                )
                & (
                    plot_subset[
                        "model_support_count"
                    ] == support
                )
            ][
                "fraction_hotspot_union"
            ].iloc[0]
        )
        for support in support_counts
    ]

    x = np.arange(
        len(support_counts)
    )

    width = 0.36

    fig, ax = plt.subplots(
        figsize=(8, 5.5)
    )

    ax.bar(
        x - width / 2,
        cpp_values,
        width=width,
        label="CPP",
    )

    ax.bar(
        x + width / 2,
        noncpp_values,
        width=width,
        label="Non-CPP",
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        [
            "1 PLM",
            "2 PLMs",
            "3 PLMs",
            "4 PLMs",
        ]
    )

    ax.set_ylabel(
        "Fraction of hotspot-union residues"
    )

    ax.set_title(
        f"Cross-PLM support of hotspot residues\n"
        f"{dataset_name}"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            f"hotspot_model_support_"
            f"{dataset_name}"
        )
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 11. DATASET-LEVEL SUMMARY
# ============================================================

dataset_summary_rows = []

for dataset_name in DATASETS:

    residue_table = (
        merged_dataset_tables[
            dataset_name
        ]
    )

    sequence_table = (
        per_sequence_df[
            per_sequence_df["dataset"]
            == dataset_name
        ]
    )

    for label_value, class_name in [
        (0, "non_CPP"),
        (1, "CPP"),
    ]:

        residue_subset = (
            residue_table[
                residue_table["label"]
                == label_value
            ]
        )

        sequence_subset = (
            sequence_table[
                sequence_table["label"]
                == label_value
            ]
        )

        dataset_summary_rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "number_of_sequences": int(
                len(sequence_subset)
            ),
            "number_of_residues": int(
                len(residue_subset)
            ),
            "fraction_any_model_hotspot": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] >= 1
                ).mean()
            ),
            "fraction_supported_by_at_least_2": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] >= 2
                ).mean()
            ),
            "fraction_supported_by_at_least_3": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] >= 3
                ).mean()
            ),
            "fraction_supported_by_all_4": float(
                (
                    residue_subset[
                        "model_support_count"
                    ] == 4
                ).mean()
            ),
            "mean_sequence_pairwise_jaccard": float(
                sequence_subset[
                    "mean_pairwise_jaccard"
                ].mean()
            ),
            "median_sequence_pairwise_jaccard": float(
                sequence_subset[
                    "mean_pairwise_jaccard"
                ].median()
            ),
        })


dataset_summary_df = pd.DataFrame(
    dataset_summary_rows
)

dataset_summary_file = (
    RESULT_TABLE_DIR
    / "cross_plm_hotspot_conservation_summary.csv"
)

dataset_summary_df.to_csv(
    dataset_summary_file,
    index=False,
)


# ============================================================
# 12. SAVE CHECKPOINT
# ============================================================

analysis_summary = {
    "models": MODEL_NAMES,
    "datasets": DATASETS,
    "hotspot_threshold": float(
        HOTSPOT_RANK_THRESHOLD
    ),
    "hotspot_definition": (
        "Adjusted per-model consensus percentile rank "
        "greater than or equal to 0.80"
    ),
    "pairwise_similarity": (
        "Per-sequence Jaccard similarity between "
        "model-specific hotspot position sets"
    ),
    "support_definition": (
        "Number of PLMs independently classifying a "
        "residue as a top-20% hotspot"
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

analysis_summary_file = (
    CHECKPOINT_DIR
    / "cross_plm_hotspot_conservation_method.json"
)

with open(
    analysis_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        analysis_summary,
        handle,
        indent=2,
    )

output_files = [
    pairwise_file,
    pairwise_sequence_file,
    support_distribution_file,
    per_sequence_file,
    class_comparison_file,
    residue_class_file,
    dataset_summary_file,
    analysis_summary_file,
    *jaccard_matrix_files,
]

for dataset_name in DATASETS:
    output_files.append(
        CONSERVATION_ROOT
        / dataset_name
        / "cross_plm_residue_support.csv"
    )

mark_step_complete(
    "17_cross_PLM_hotspot_conservation",
    output_files=output_files,
    details=analysis_summary,
)


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 78)
print("PAIRWISE CROSS-PLM HOTSPOT JACCARD")
print("=" * 78)

display(
    pairwise_df[
        [
            "dataset",
            "model_a",
            "model_b",
            "mean_jaccard_CPP",
            "median_jaccard_CPP",
            "mean_jaccard_nonCPP",
            "median_jaccard_nonCPP",
        ]
    ]
)

print("\n" + "=" * 78)
print("CROSS-PLM CONSERVATION SUMMARY")
print("=" * 78)

display(
    dataset_summary_df
)

print("\n" + "=" * 78)
print("CPP VS NON-CPP CONSERVATION")
print("=" * 78)

display(
    class_comparison_df
)

print("\n" + "=" * 78)
print("RESIDUE-LEVEL CONSERVATION ENRICHMENT")
print("=" * 78)

display(
    residue_class_df
)

print("\nSaved pairwise Jaccard table:")
print(pairwise_file)

print("\nSaved conservation summary:")
print(dataset_summary_file)

print("\nSaved per-sequence SI table:")
print(per_sequence_file)

print("\n" + "=" * 78)
print("STEP 17 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 18: PHYSICOCHEMICAL ENRICHMENT OF CONSENSUS HOTSPOTS
#
# Analyses:
# 1. Categorical residue-property enrichment
#    - CPP hotspots vs CPP non-hotspots
#    - CPP hotspots vs non-CPP hotspots
#
# 2. Sequence-level continuous-property comparison
#    - paired hotspot vs non-hotspot comparison within CPPs
#    - CPP hotspot vs non-CPP hotspot comparison
#
# 3. Internal-test vs KELM replication
#
# Primary hotspot definition:
#    redundancy-adjusted global top-20% residues
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    fisher_exact,
    wilcoxon,
    mannwhitneyu,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

for directory in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. NON-OVERLAPPING PHYSICOCHEMICAL CLASSES
#
# Every standard amino acid belongs to exactly one class.
# Histidine is assigned to the basic class.
# ============================================================

PHYSICOCHEMICAL_CLASSES = {
    "basic_positive": set("KRH"),
    "acidic_negative": set("DE"),
    "polar_uncharged": set("STNQ"),
    "aromatic": set("FWY"),
    "aliphatic_hydrophobic": set("AILMV"),
    "structure_special": set("GPC"),
}

AMINO_ACIDS = set(
    "ACDEFGHIKLMNPQRSTVWY"
)

all_classified_residues = set().union(
    *PHYSICOCHEMICAL_CLASSES.values()
)

if all_classified_residues != AMINO_ACIDS:
    missing = (
        AMINO_ACIDS
        - all_classified_residues
    )

    duplicated = []

    for residue in AMINO_ACIDS:
        number_of_classes = sum(
            residue in residues
            for residues
            in PHYSICOCHEMICAL_CLASSES.values()
        )

        if number_of_classes != 1:
            duplicated.append(residue)

    raise ValueError(
        f"Physicochemical classes are invalid. "
        f"Missing={missing}, non-unique={duplicated}"
    )


def residue_to_class(residue):
    for class_name, residue_set in (
        PHYSICOCHEMICAL_CLASSES.items()
    ):
        if residue in residue_set:
            return class_name

    raise ValueError(
        f"Unknown residue: {residue}"
    )


# ============================================================
# 3. RESIDUE-LEVEL PROPERTY LOOKUPS
#
# Charge is a simplified physiological-pH proxy.
# Hydropathy uses the Kyte-Doolittle scale.
# ============================================================

CHARGE_SCORE = {
    "A": 0.0,
    "C": 0.0,
    "D": -1.0,
    "E": -1.0,
    "F": 0.0,
    "G": 0.0,
    "H": 0.1,
    "I": 0.0,
    "K": 1.0,
    "L": 0.0,
    "M": 0.0,
    "N": 0.0,
    "P": 0.0,
    "Q": 0.0,
    "R": 1.0,
    "S": 0.0,
    "T": 0.0,
    "V": 0.0,
    "W": 0.0,
    "Y": 0.0,
}

KYTE_DOOLITTLE = {
    "A": 1.8,
    "C": 2.5,
    "D": -3.5,
    "E": -3.5,
    "F": 2.8,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "K": -3.9,
    "L": 3.8,
    "M": 1.9,
    "N": -3.5,
    "P": -1.6,
    "Q": -3.5,
    "R": -4.5,
    "S": -0.8,
    "T": -0.7,
    "V": 4.2,
    "W": -0.9,
    "Y": -1.3,
}

AROMATIC_SCORE = {
    residue: float(
        residue in set("FWY")
    )
    for residue in AMINO_ACIDS
}

BASIC_SCORE = {
    residue: float(
        residue in set("KRH")
    )
    for residue in AMINO_ACIDS
}

ACIDIC_SCORE = {
    residue: float(
        residue in set("DE")
    )
    for residue in AMINO_ACIDS
}

POLAR_SCORE = {
    residue: float(
        residue in set("STNQ")
    )
    for residue in AMINO_ACIDS
}


CONTINUOUS_PROPERTIES = {
    "charge_score": CHARGE_SCORE,
    "hydropathy_KD": KYTE_DOOLITTLE,
    "aromatic_fraction": AROMATIC_SCORE,
    "basic_fraction": BASIC_SCORE,
    "acidic_fraction": ACIDIC_SCORE,
    "polar_fraction": POLAR_SCORE,
}


# ============================================================
# 4. BENJAMINI-HOCHBERG FDR
# ============================================================

def benjamini_hochberg(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    if number_of_tests == 0:
        return np.array(
            [],
            dtype=float,
        )

    order = np.argsort(
        p_values
    )

    ordered_p_values = (
        p_values[order]
    )

    adjusted_ordered = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ordered_p_values[
                reverse_index
            ]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ordered[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ordered,
        1.0,
    )

    return adjusted


# ============================================================
# 5. ODDS RATIO WITH CONTINUITY CORRECTION
# ============================================================

def corrected_odds_ratio(
    a,
    b,
    c,
    d,
):
    return float(
        (
            (a + 0.5)
            * (d + 0.5)
        )
        / (
            (b + 0.5)
            * (c + 0.5)
        )
    )


# ============================================================
# 6. SAFE STATISTICAL TESTS
# ============================================================

def safe_wilcoxon(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    valid = (
        np.isfinite(values_a)
        & np.isfinite(values_b)
    )

    values_a = values_a[
        valid
    ]

    values_b = values_b[
        valid
    ]

    differences = (
        values_a - values_b
    )

    if len(differences) == 0:
        return np.nan, np.nan

    if np.allclose(
        differences,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        values_a,
        values_b,
        alternative="two-sided",
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


def safe_mannwhitney(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    values_a = values_a[
        np.isfinite(values_a)
    ]

    values_b = values_b[
        np.isfinite(values_b)
    ]

    if (
        len(values_a) == 0
        or len(values_b) == 0
    ):
        return np.nan, np.nan

    statistic, p_value = mannwhitneyu(
        values_a,
        values_b,
        alternative="two-sided",
    )

    return (
        float(statistic),
        float(p_value),
    )


# ============================================================
# 7. LOAD ADJUSTED GLOBAL CONSENSUS
# ============================================================

def load_consensus(
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file not found:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "adjusted_hotspot_top20",
        "strict_cross_model_hotspot",
        "unanimous_cross_model_hotspot",
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    dataframe["residue"] = (
        dataframe["residue"]
        .astype(str)
        .str.upper()
    )

    invalid_residues = (
        set(dataframe["residue"])
        - AMINO_ACIDS
    )

    if invalid_residues:
        raise ValueError(
            f"Invalid residues detected: "
            f"{sorted(invalid_residues)}"
        )

    dataframe[
        "physicochemical_class"
    ] = dataframe[
        "residue"
    ].map(
        residue_to_class
    )

    for property_name, lookup in (
        CONTINUOUS_PROPERTIES.items()
    ):
        dataframe[
            property_name
        ] = dataframe[
            "residue"
        ].map(
            lookup
        ).astype(float)

    return (
        dataframe,
        input_file,
    )


# ============================================================
# 8. CATEGORICAL ENRICHMENT
# ============================================================

def categorical_enrichment(
    group_1,
    group_2,
    dataset_name,
    analysis_name,
    group_1_name,
    group_2_name,
):
    rows = []

    group_1_total = len(
        group_1
    )

    group_2_total = len(
        group_2
    )

    if (
        group_1_total == 0
        or group_2_total == 0
    ):
        raise ValueError(
            f"Empty comparison group for "
            f"{dataset_name}/{analysis_name}"
        )

    for category_name in (
        PHYSICOCHEMICAL_CLASSES
    ):
        group_1_count = int(
            (
                group_1[
                    "physicochemical_class"
                ]
                == category_name
            ).sum()
        )

        group_2_count = int(
            (
                group_2[
                    "physicochemical_class"
                ]
                == category_name
            ).sum()
        )

        group_1_other = (
            group_1_total
            - group_1_count
        )

        group_2_other = (
            group_2_total
            - group_2_count
        )

        contingency_table = [
            [
                group_1_count,
                group_1_other,
            ],
            [
                group_2_count,
                group_2_other,
            ],
        ]

        raw_odds_ratio, p_value = (
            fisher_exact(
                contingency_table,
                alternative="two-sided",
            )
        )

        corrected_or = (
            corrected_odds_ratio(
                group_1_count,
                group_1_other,
                group_2_count,
                group_2_other,
            )
        )

        group_1_frequency = (
            group_1_count
            / group_1_total
        )

        group_2_frequency = (
            group_2_count
            / group_2_total
        )

        log2_enrichment = np.log2(
            (
                group_1_frequency
                + 1e-12
            )
            / (
                group_2_frequency
                + 1e-12
            )
        )

        rows.append({
            "dataset": dataset_name,
            "analysis": analysis_name,
            "group_1": group_1_name,
            "group_2": group_2_name,
            "property_class": (
                category_name
            ),
            "group_1_count": (
                group_1_count
            ),
            "group_1_total": (
                group_1_total
            ),
            "group_1_frequency": float(
                group_1_frequency
            ),
            "group_2_count": (
                group_2_count
            ),
            "group_2_total": (
                group_2_total
            ),
            "group_2_frequency": float(
                group_2_frequency
            ),
            "odds_ratio_raw": float(
                raw_odds_ratio
            ),
            "odds_ratio_corrected": float(
                corrected_or
            ),
            "log2_enrichment": float(
                log2_enrichment
            ),
            "p_value": float(
                p_value
            ),
        })

    result = pd.DataFrame(
        rows
    )

    result["fdr_bh"] = (
        benjamini_hochberg(
            result[
                "p_value"
            ].to_numpy()
        )
    )

    result[
        "significant_fdr_0_05"
    ] = (
        result["fdr_bh"] < 0.05
    )

    result[
        "direction"
    ] = np.where(
        result[
            "odds_ratio_corrected"
        ] > 1.0,
        f"enriched_in_{group_1_name}",
        f"enriched_in_{group_2_name}",
    )

    return result


# ============================================================
# 9. CREATE SEQUENCE-LEVEL PROPERTY TABLE
# ============================================================

def build_sequence_property_table(
    dataframe,
    dataset_name,
):
    rows = []

    for sequence_id, group in (
        dataframe.groupby(
            "sequence_id",
            sort=False,
        )
    ):
        hotspot_group = group[
            group[
                "adjusted_hotspot_top20"
            ].astype(bool)
        ]

        non_hotspot_group = group[
            ~group[
                "adjusted_hotspot_top20"
            ].astype(bool)
        ]

        record = {
            "dataset": dataset_name,
            "sequence_id": (
                sequence_id
            ),
            "sequence": str(
                group[
                    "sequence"
                ].iloc[0]
            ),
            "label": int(
                group[
                    "label"
                ].iloc[0]
            ),
            "sequence_length": int(
                group[
                    "sequence_length"
                ].iloc[0]
            ),
            "number_of_hotspots": int(
                len(hotspot_group)
            ),
            "number_of_nonhotspots": int(
                len(non_hotspot_group)
            ),
        }

        for property_name in (
            CONTINUOUS_PROPERTIES
        ):
            record[
                f"hotspot_{property_name}"
            ] = float(
                hotspot_group[
                    property_name
                ].mean()
            )

            record[
                f"nonhotspot_{property_name}"
            ] = float(
                non_hotspot_group[
                    property_name
                ].mean()
            )

            record[
                f"hotspot_minus_nonhotspot_{property_name}"
            ] = (
                record[
                    f"hotspot_{property_name}"
                ]
                - record[
                    f"nonhotspot_{property_name}"
                ]
            )

        rows.append(
            record
        )

    return pd.DataFrame(
        rows
    )


# ============================================================
# 10. RUN DATASET ANALYSES
# ============================================================

categorical_results = []
sequence_property_tables = []
source_files = []

for dataset_name in DATASETS:
    dataframe, source_file = (
        load_consensus(
            dataset_name
        )
    )

    source_files.append(
        source_file
    )

    print("\n" + "=" * 78)
    print(
        f"PHYSICOCHEMICAL ANALYSIS: "
        f"{dataset_name}"
    )
    print("=" * 78)

    cpp = dataframe[
        dataframe["label"] == 1
    ]

    noncpp = dataframe[
        dataframe["label"] == 0
    ]

    cpp_hotspots = cpp[
        cpp[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    cpp_nonhotspots = cpp[
        ~cpp[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    noncpp_hotspots = noncpp[
        noncpp[
            "adjusted_hotspot_top20"
        ].astype(bool)
    ]

    # Analysis A:
    # CPP hotspots vs non-hotspots from CPPs
    within_cpp_result = (
        categorical_enrichment(
            group_1=cpp_hotspots,
            group_2=cpp_nonhotspots,
            dataset_name=dataset_name,
            analysis_name=(
                "CPP_hotspot_vs_CPP_nonhotspot"
            ),
            group_1_name="CPP_hotspot",
            group_2_name="CPP_nonhotspot",
        )
    )

    categorical_results.append(
        within_cpp_result
    )

    # Analysis B:
    # CPP hotspots vs non-CPP hotspots
    between_class_result = (
        categorical_enrichment(
            group_1=cpp_hotspots,
            group_2=noncpp_hotspots,
            dataset_name=dataset_name,
            analysis_name=(
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            group_1_name="CPP_hotspot",
            group_2_name="nonCPP_hotspot",
        )
    )

    categorical_results.append(
        between_class_result
    )

    sequence_property_table = (
        build_sequence_property_table(
            dataframe=dataframe,
            dataset_name=dataset_name,
        )
    )

    sequence_property_tables.append(
        sequence_property_table
    )


categorical_df = pd.concat(
    categorical_results,
    ignore_index=True,
)

sequence_property_df = pd.concat(
    sequence_property_tables,
    ignore_index=True,
)


# ============================================================
# 11. SEQUENCE-LEVEL STATISTICAL TESTS
# ============================================================

sequence_test_rows = []

for dataset_name in DATASETS:
    dataset_table = (
        sequence_property_df[
            sequence_property_df[
                "dataset"
            ] == dataset_name
        ]
    )

    cpp_table = dataset_table[
        dataset_table["label"] == 1
    ]

    noncpp_table = dataset_table[
        dataset_table["label"] == 0
    ]

    for property_name in (
        CONTINUOUS_PROPERTIES
    ):
        # Paired within-CPP comparison
        hotspot_column = (
            f"hotspot_{property_name}"
        )

        nonhotspot_column = (
            f"nonhotspot_{property_name}"
        )

        wilcoxon_statistic, wilcoxon_p = (
            safe_wilcoxon(
                cpp_table[
                    hotspot_column
                ],
                cpp_table[
                    nonhotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "paired_CPP_hotspot_vs_CPP_nonhotspot"
            ),
            "property": property_name,
            "group_1": "CPP_hotspot",
            "group_2": "CPP_nonhotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(cpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_1_median": float(
                cpp_table[
                    hotspot_column
                ].median()
            ),
            "group_2_mean": float(
                cpp_table[
                    nonhotspot_column
                ].mean()
            ),
            "group_2_median": float(
                cpp_table[
                    nonhotspot_column
                ].median()
            ),
            "mean_difference": float(
                (
                    cpp_table[
                        hotspot_column
                    ]
                    - cpp_table[
                        nonhotspot_column
                    ]
                ).mean()
            ),
            "test": (
                "paired Wilcoxon signed-rank"
            ),
            "statistic": float(
                wilcoxon_statistic
            ),
            "p_value": float(
                wilcoxon_p
            ),
        })

        # CPP hotspot vs non-CPP hotspot
        mannwhitney_statistic, mannwhitney_p = (
            safe_mannwhitney(
                cpp_table[
                    hotspot_column
                ],
                noncpp_table[
                    hotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            "property": property_name,
            "group_1": "CPP_hotspot",
            "group_2": "nonCPP_hotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(noncpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_1_median": float(
                cpp_table[
                    hotspot_column
                ].median()
            ),
            "group_2_mean": float(
                noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_2_median": float(
                noncpp_table[
                    hotspot_column
                ].median()
            ),
            "mean_difference": float(
                cpp_table[
                    hotspot_column
                ].mean()
                - noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "test": (
                "Mann-Whitney U"
            ),
            "statistic": float(
                mannwhitney_statistic
            ),
            "p_value": float(
                mannwhitney_p
            ),
        })


sequence_tests_df = pd.DataFrame(
    sequence_test_rows
)

sequence_tests_df["fdr_bh"] = (
    sequence_tests_df.groupby(
        [
            "dataset",
            "analysis",
        ]
    )[
        "p_value"
    ].transform(
        lambda values:
            benjamini_hochberg(
                values.to_numpy()
            )
    )
)

sequence_tests_df[
    "significant_fdr_0_05"
] = (
    sequence_tests_df[
        "fdr_bh"
    ] < 0.05
)


# ============================================================
# 12. INTERNAL–KELM CATEGORY REPLICATION
# ============================================================

replication_source = categorical_df[
    categorical_df[
        "analysis"
    ] == (
        "CPP_hotspot_vs_CPP_nonhotspot"
    )
][
    [
        "dataset",
        "property_class",
        "odds_ratio_corrected",
        "log2_enrichment",
        "fdr_bh",
        "significant_fdr_0_05",
    ]
].copy()


internal_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "internal_test"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)


kelm_replication = (
    replication_source[
        replication_source[
            "dataset"
        ] == "kelm_external"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)


replication_df = (
    internal_replication.merge(
        kelm_replication,
        on="property_class",
        how="outer",
        validate="one_to_one",
    )
)

replication_df[
    "same_direction"
] = (
    np.sign(
        replication_df[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replication_df[
            "kelm_log2_enrichment"
        ]
    )
)

replication_df[
    "significant_in_both"
] = (
    replication_df[
        "internal_significant"
    ].fillna(False)
    & replication_df[
        "kelm_significant"
    ].fillna(False)
)

replication_df = (
    replication_df.sort_values(
        [
            "significant_in_both",
            "same_direction",
            "internal_log2_enrichment",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
)


# ============================================================
# 13. SAVE TABLES
# ============================================================

categorical_file = (
    RESULT_TABLE_DIR
    / "physicochemical_category_enrichment.csv"
)

sequence_property_file = (
    RESULT_SI_DIR
    / "physicochemical_properties_per_sequence.csv"
)

sequence_tests_file = (
    RESULT_TABLE_DIR
    / "physicochemical_sequence_level_tests.csv"
)

replication_file = (
    RESULT_TABLE_DIR
    / "physicochemical_enrichment_replication.csv"
)

significant_categorical_file = (
    RESULT_TABLE_DIR
    / "significant_physicochemical_enrichment.csv"
)

categorical_df.to_csv(
    categorical_file,
    index=False,
)

sequence_property_df.to_csv(
    sequence_property_file,
    index=False,
)

sequence_tests_df.to_csv(
    sequence_tests_file,
    index=False,
)

replication_df.to_csv(
    replication_file,
    index=False,
)

categorical_df[
    categorical_df[
        "significant_fdr_0_05"
    ]
].to_csv(
    significant_categorical_file,
    index=False,
)


# ============================================================
# 14. FIGURE — CATEGORY LOG2 ENRICHMENT
# ============================================================

category_order = list(
    PHYSICOCHEMICAL_CLASSES.keys()
)

for analysis_name, analysis_title in [
    (
        "CPP_hotspot_vs_CPP_nonhotspot",
        "CPP hotspots versus CPP non-hotspots",
    ),
    (
        "CPP_hotspot_vs_nonCPP_hotspot",
        "CPP hotspots versus non-CPP hotspots",
    ),
]:
    plot_table = categorical_df[
        categorical_df[
            "analysis"
        ] == analysis_name
    ].copy()

    plot_table[
        "property_class"
    ] = pd.Categorical(
        plot_table[
            "property_class"
        ],
        categories=category_order,
        ordered=True,
    )

    plot_table = plot_table.sort_values(
        [
            "property_class",
            "dataset",
        ]
    )

    x = np.arange(
        len(category_order)
    )

    width = 0.36

    internal_values = []

    kelm_values = []

    internal_significance = []

    kelm_significance = []

    for category_name in category_order:
        internal_row = plot_table[
            (
                plot_table["dataset"]
                == "internal_test"
            )
            & (
                plot_table[
                    "property_class"
                ] == category_name
            )
        ].iloc[0]

        kelm_row = plot_table[
            (
                plot_table["dataset"]
                == "kelm_external"
            )
            & (
                plot_table[
                    "property_class"
                ] == category_name
            )
        ].iloc[0]

        internal_values.append(
            internal_row[
                "log2_enrichment"
            ]
        )

        kelm_values.append(
            kelm_row[
                "log2_enrichment"
            ]
        )

        internal_significance.append(
            bool(
                internal_row[
                    "significant_fdr_0_05"
                ]
            )
        )

        kelm_significance.append(
            bool(
                kelm_row[
                    "significant_fdr_0_05"
                ]
            )
        )

    fig, ax = plt.subplots(
        figsize=(10, 5.8)
    )

    internal_bars = ax.bar(
        x - width / 2,
        internal_values,
        width=width,
        label="Internal test",
    )

    kelm_bars = ax.bar(
        x + width / 2,
        kelm_values,
        width=width,
        label="KELM external",
    )

    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        [
            category.replace(
                "_",
                "\n",
            )
            for category
            in category_order
        ],
        rotation=0,
    )

    ax.set_ylabel(
        "log$_2$ enrichment"
    )

    ax.set_title(
        analysis_title
    )

    ax.legend(
        frameon=False,
    )

    for bar, significant in zip(
        internal_bars,
        internal_significance,
    ):
        if significant:
            height = bar.get_height()

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height
                + (
                    0.08
                    if height >= 0
                    else -0.18
                ),
                "*",
                ha="center",
                va="center",
                fontsize=13,
            )

    for bar, significant in zip(
        kelm_bars,
        kelm_significance,
    ):
        if significant:
            height = bar.get_height()

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height
                + (
                    0.08
                    if height >= 0
                    else -0.18
                ),
                "*",
                ha="center",
                va="center",
                fontsize=13,
            )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            "physicochemical_enrichment_"
            + analysis_name
        )
    )

    fig.savefig(
        figure_base.with_suffix(
            ".png"
        ),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".pdf"
        ),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".svg"
        ),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 15. FIGURE — SEQUENCE-LEVEL CONTINUOUS PROPERTIES
# ============================================================

property_display_names = {
    "charge_score": "Mean charge score",
    "hydropathy_KD": "Mean hydropathy",
    "aromatic_fraction": "Aromatic fraction",
    "basic_fraction": "Basic-residue fraction",
    "acidic_fraction": "Acidic-residue fraction",
    "polar_fraction": "Polar-residue fraction",
}

cpp_property_table = (
    sequence_property_df[
        sequence_property_df[
            "label"
        ] == 1
    ]
)

for property_name in (
    CONTINUOUS_PROPERTIES
):
    internal_cpp = cpp_property_table[
        cpp_property_table[
            "dataset"
        ] == "internal_test"
    ]

    kelm_cpp = cpp_property_table[
        cpp_property_table[
            "dataset"
        ] == "kelm_external"
    ]

    plot_groups = [
        internal_cpp[
            f"hotspot_{property_name}"
        ].to_numpy(),
        internal_cpp[
            f"nonhotspot_{property_name}"
        ].to_numpy(),
        kelm_cpp[
            f"hotspot_{property_name}"
        ].to_numpy(),
        kelm_cpp[
            f"nonhotspot_{property_name}"
        ].to_numpy(),
    ]

    fig, ax = plt.subplots(
        figsize=(8.5, 5.5)
    )

    ax.boxplot(
        plot_groups,
        showfliers=False,
    )

    ax.set_xticks(
        [
            1,
            2,
            3,
            4,
        ]
    )

    ax.set_xticklabels(
        [
            "Internal\nhotspots",
            "Internal\nnon-hotspots",
            "KELM\nhotspots",
            "KELM\nnon-hotspots",
        ]
    )

    ax.set_ylabel(
        property_display_names[
            property_name
        ]
    )

    ax.set_title(
        f"CPP hotspot physicochemical property:\n"
        f"{property_display_names[property_name]}"
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_SI_DIR
        / (
            "physicochemical_sequence_"
            + property_name
        )
    )

    fig.savefig(
        figure_base.with_suffix(
            ".png"
        ),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".pdf"
        ),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(
            ".svg"
        ),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 16. SUMMARY JSON
# ============================================================

summary = {
    "datasets": DATASETS,
    "hotspot_definition": (
        "Top 20% redundancy-adjusted "
        "global consensus ranks"
    ),
    "physicochemical_classes": {
        class_name: sorted(
            residue_set
        )
        for class_name, residue_set
        in PHYSICOCHEMICAL_CLASSES.items()
    },
    "continuous_properties": list(
        CONTINUOUS_PROPERTIES.keys()
    ),
    "categorical_tests": int(
        len(categorical_df)
    ),
    "significant_categorical_tests": int(
        categorical_df[
            "significant_fdr_0_05"
        ].sum()
    ),
    "sequence_level_tests": int(
        len(sequence_tests_df)
    ),
    "significant_sequence_level_tests": int(
        sequence_tests_df[
            "significant_fdr_0_05"
        ].sum()
    ),
    "replicated_significant_categories": (
        replication_df[
            replication_df[
                "significant_in_both"
            ]
        ][
            "property_class"
        ].tolist()
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "physicochemical_enrichment_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 17. SAVE CHECKPOINT
# ============================================================

output_files = [
    categorical_file,
    sequence_property_file,
    sequence_tests_file,
    replication_file,
    significant_categorical_file,
    summary_file,
]

mark_step_complete(
    "18_physicochemical_hotspot_enrichment",
    output_files=output_files,
    details=summary,
)


# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 78)
print("PHYSICOCHEMICAL CATEGORY ENRICHMENT")
print("=" * 78)

display(
    categorical_df[
        [
            "dataset",
            "analysis",
            "property_class",
            "group_1_frequency",
            "group_2_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
            "direction",
        ]
    ].sort_values(
        [
            "analysis",
            "dataset",
            "fdr_bh",
        ]
    )
)

print("\n" + "=" * 78)
print("INTERNAL–KELM PHYSICOCHEMICAL REPLICATION")
print("=" * 78)

display(
    replication_df[
        [
            "property_class",
            "internal_odds_ratio",
            "internal_log2_enrichment",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_log2_enrichment",
            "kelm_fdr",
            "same_direction",
            "significant_in_both",
        ]
    ]
)

print("\n" + "=" * 78)
print("SEQUENCE-LEVEL PROPERTY TESTS")
print("=" * 78)

display(
    sequence_tests_df[
        [
            "dataset",
            "analysis",
            "property",
            "group_1_mean",
            "group_2_mean",
            "mean_difference",
            "test",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
)

print("\nReplicated significant property classes:")
print(
    summary[
        "replicated_significant_categories"
    ]
)

print("\nSaved category enrichment table:")
print(categorical_file)

print("\nSaved sequence-level tests:")
print(sequence_tests_file)

print("\nSaved replication table:")
print(replication_file)

print("\n" + "=" * 78)
print("STEP 18 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 19: POSITIONAL PREFERENCE OF CONSENSUS HOTSPOTS
#
# Primary questions:
# 1. Are CPP hotspots preferentially located near the
#    N-terminus, middle, or C-terminus?
# 2. Does positional preference differ between CPPs and
#    non-CPPs?
# 3. Does the pattern replicate in KELM?
#
# Primary hotspot definition:
#    Top 20% redundancy-adjusted global consensus ranks
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    fisher_exact,
    mannwhitneyu,
    wilcoxon,
    kstest,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASETS = [
    "internal_test",
    "kelm_external",
]

XAI_ROOT = DIRS["xai"]

RESULT_TABLE_DIR = (
    DIRS["results"] / "tables_main"
)

RESULT_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

CHECKPOINT_DIR = DIRS["checkpoints"]

for directory in [
    RESULT_TABLE_DIR,
    RESULT_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Coarse positional regions
# normalized_position is in (0, 1]
REGION_BOUNDARIES = {
    "N_terminal": (0.0, 1.0 / 3.0),
    "Middle": (1.0 / 3.0, 2.0 / 3.0),
    "C_terminal": (2.0 / 3.0, 1.0),
}

# Detailed positional bins
NUMBER_OF_POSITION_BINS = 10


# ============================================================
# 2. BENJAMINI-HOCHBERG FDR
# ============================================================

def benjamini_hochberg(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    if number_of_tests == 0:
        return np.array(
            [],
            dtype=float,
        )

    order = np.argsort(
        p_values
    )

    ordered = p_values[
        order
    ]

    adjusted_ordered = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_minimum = 1.0

    for reverse_index in range(
        number_of_tests - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        adjusted_value = (
            ordered[reverse_index]
            * number_of_tests
            / rank
        )

        running_minimum = min(
            running_minimum,
            adjusted_value,
        )

        adjusted_ordered[
            reverse_index
        ] = running_minimum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[order] = np.minimum(
        adjusted_ordered,
        1.0,
    )

    return adjusted


# ============================================================
# 3. SAFE STATISTICAL HELPERS
# ============================================================

def corrected_odds_ratio(
    a,
    b,
    c,
    d,
):
    return float(
        (
            (a + 0.5)
            * (d + 0.5)
        )
        / (
            (b + 0.5)
            * (c + 0.5)
        )
    )


def safe_mannwhitney(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    values_a = values_a[
        np.isfinite(values_a)
    ]

    values_b = values_b[
        np.isfinite(values_b)
    ]

    if (
        len(values_a) == 0
        or len(values_b) == 0
    ):
        return np.nan, np.nan

    statistic, p_value = mannwhitneyu(
        values_a,
        values_b,
        alternative="two-sided",
    )

    return (
        float(statistic),
        float(p_value),
    )


def safe_wilcoxon(
    values_a,
    values_b,
):
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    valid = (
        np.isfinite(values_a)
        & np.isfinite(values_b)
    )

    values_a = values_a[
        valid
    ]

    values_b = values_b[
        valid
    ]

    if len(values_a) == 0:
        return np.nan, np.nan

    differences = (
        values_a - values_b
    )

    if np.allclose(
        differences,
        0.0,
    ):
        return 0.0, 1.0

    statistic, p_value = wilcoxon(
        values_a,
        values_b,
        alternative="two-sided",
        zero_method="wilcox",
    )

    return (
        float(statistic),
        float(p_value),
    )


# ============================================================
# 4. LOAD ADJUSTED CONSENSUS
# ============================================================

def load_adjusted_consensus(
    dataset_name,
):
    input_file = (
        XAI_ROOT
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Adjusted consensus file missing:\n"
            f"{input_file}"
        )

    dataframe = pd.read_csv(
        input_file
    )

    required_columns = {
        "sequence_id",
        "sequence",
        "label",
        "sequence_length",
        "position",
        "normalized_position",
        "residue",
        "adjusted_global_consensus_rank",
        "adjusted_hotspot_top20",
        "strict_cross_model_hotspot",
        "unanimous_cross_model_hotspot",
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    dataframe[
        "normalized_position"
    ] = (
        dataframe["position"]
        / dataframe["sequence_length"]
    )

    return (
        dataframe,
        input_file,
    )


# ============================================================
# 5. ASSIGN POSITIONAL REGION
# ============================================================

def assign_position_region(
    normalized_position,
):
    if normalized_position <= (
        1.0 / 3.0
    ):
        return "N_terminal"

    if normalized_position <= (
        2.0 / 3.0
    ):
        return "Middle"

    return "C_terminal"


def assign_position_bin(
    normalized_position,
):
    # Convert positions in (0, 1] into bins 1–10.
    bin_number = int(
        np.ceil(
            normalized_position
            * NUMBER_OF_POSITION_BINS
        )
    )

    return min(
        max(
            bin_number,
            1,
        ),
        NUMBER_OF_POSITION_BINS,
    )


# ============================================================
# 6. PREPARE RESIDUE TABLES
# ============================================================

residue_tables = {}
source_files = []

for dataset_name in DATASETS:

    dataframe, source_file = (
        load_adjusted_consensus(
            dataset_name
        )
    )

    source_files.append(
        source_file
    )

    dataframe[
        "position_region"
    ] = dataframe[
        "normalized_position"
    ].apply(
        assign_position_region
    )

    dataframe[
        "position_bin"
    ] = dataframe[
        "normalized_position"
    ].apply(
        assign_position_bin
    )

    dataframe[
        "is_hotspot"
    ] = dataframe[
        "adjusted_hotspot_top20"
    ].astype(bool)

    residue_tables[
        dataset_name
    ] = dataframe

    print(
        f"[LOADED] {dataset_name}: "
        f"{len(dataframe)} residues"
    )


# ============================================================
# 7. RESIDUE-LEVEL REGION ENRICHMENT
#
# Within each class:
# hotspot positions vs non-hotspot positions.
# ============================================================

region_enrichment_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for label_value, class_name in [
        (1, "CPP"),
        (0, "non_CPP"),
    ]:

        class_data = dataframe[
            dataframe["label"]
            == label_value
        ]

        hotspots = class_data[
            class_data[
                "is_hotspot"
            ]
        ]

        nonhotspots = class_data[
            ~class_data[
                "is_hotspot"
            ]
        ]

        hotspot_total = len(
            hotspots
        )

        nonhotspot_total = len(
            nonhotspots
        )

        for region_name in (
            REGION_BOUNDARIES
        ):

            hotspot_region_count = int(
                (
                    hotspots[
                        "position_region"
                    ]
                    == region_name
                ).sum()
            )

            nonhotspot_region_count = int(
                (
                    nonhotspots[
                        "position_region"
                    ]
                    == region_name
                ).sum()
            )

            hotspot_other = (
                hotspot_total
                - hotspot_region_count
            )

            nonhotspot_other = (
                nonhotspot_total
                - nonhotspot_region_count
            )

            raw_odds_ratio, p_value = (
                fisher_exact(
                    [
                        [
                            hotspot_region_count,
                            hotspot_other,
                        ],
                        [
                            nonhotspot_region_count,
                            nonhotspot_other,
                        ],
                    ],
                    alternative="two-sided",
                )
            )

            corrected_or = (
                corrected_odds_ratio(
                    hotspot_region_count,
                    hotspot_other,
                    nonhotspot_region_count,
                    nonhotspot_other,
                )
            )

            hotspot_frequency = (
                hotspot_region_count
                / hotspot_total
            )

            nonhotspot_frequency = (
                nonhotspot_region_count
                / nonhotspot_total
            )

            region_enrichment_rows.append({
                "dataset": dataset_name,
                "class": class_name,
                "region": region_name,
                "hotspot_count": int(
                    hotspot_region_count
                ),
                "hotspot_total": int(
                    hotspot_total
                ),
                "hotspot_frequency": float(
                    hotspot_frequency
                ),
                "nonhotspot_count": int(
                    nonhotspot_region_count
                ),
                "nonhotspot_total": int(
                    nonhotspot_total
                ),
                "nonhotspot_frequency": float(
                    nonhotspot_frequency
                ),
                "odds_ratio_raw": float(
                    raw_odds_ratio
                ),
                "odds_ratio_corrected": float(
                    corrected_or
                ),
                "log2_enrichment": float(
                    np.log2(
                        (
                            hotspot_frequency
                            + 1e-12
                        )
                        / (
                            nonhotspot_frequency
                            + 1e-12
                        )
                    )
                ),
                "p_value": float(
                    p_value
                ),
            })


region_enrichment_df = pd.DataFrame(
    region_enrichment_rows
)

region_enrichment_df["fdr_bh"] = (
    region_enrichment_df.groupby(
        [
            "dataset",
            "class",
        ]
    )[
        "p_value"
    ].transform(
        lambda values:
            benjamini_hochberg(
                values.to_numpy()
            )
    )
)

region_enrichment_df[
    "significant_fdr_0_05"
] = (
    region_enrichment_df[
        "fdr_bh"
    ] < 0.05
)

region_enrichment_df[
    "direction"
] = np.where(
    region_enrichment_df[
        "odds_ratio_corrected"
    ] > 1.0,
    "enriched_in_hotspots",
    "depleted_in_hotspots",
)


# ============================================================
# 8. DETAILED 10-BIN DISTRIBUTION
# ============================================================

position_bin_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for label_value, class_name in [
        (1, "CPP"),
        (0, "non_CPP"),
    ]:

        class_data = dataframe[
            dataframe["label"]
            == label_value
        ]

        hotspots = class_data[
            class_data["is_hotspot"]
        ]

        nonhotspots = class_data[
            ~class_data["is_hotspot"]
        ]

        for position_bin in range(
            1,
            NUMBER_OF_POSITION_BINS + 1,
        ):

            hotspot_count = int(
                (
                    hotspots["position_bin"]
                    == position_bin
                ).sum()
            )

            nonhotspot_count = int(
                (
                    nonhotspots[
                        "position_bin"
                    ]
                    == position_bin
                ).sum()
            )

            position_bin_rows.append({
                "dataset": dataset_name,
                "class": class_name,
                "position_bin": int(
                    position_bin
                ),
                "bin_start": float(
                    (
                        position_bin - 1
                    )
                    / NUMBER_OF_POSITION_BINS
                ),
                "bin_end": float(
                    position_bin
                    / NUMBER_OF_POSITION_BINS
                ),
                "hotspot_count": int(
                    hotspot_count
                ),
                "hotspot_frequency": float(
                    hotspot_count
                    / len(hotspots)
                ),
                "nonhotspot_count": int(
                    nonhotspot_count
                ),
                "nonhotspot_frequency": float(
                    nonhotspot_count
                    / len(nonhotspots)
                ),
            })


position_bin_df = pd.DataFrame(
    position_bin_rows
)

position_bin_df[
    "frequency_difference"
] = (
    position_bin_df[
        "hotspot_frequency"
    ]
    - position_bin_df[
        "nonhotspot_frequency"
    ]
)

position_bin_df[
    "hotspot_to_nonhotspot_ratio"
] = np.divide(
    position_bin_df[
        "hotspot_frequency"
    ],
    position_bin_df[
        "nonhotspot_frequency"
    ],
    out=np.full(
        len(position_bin_df),
        np.nan,
    ),
    where=(
        position_bin_df[
            "nonhotspot_frequency"
        ].to_numpy()
        > 0
    ),
)


# ============================================================
# 9. PER-SEQUENCE POSITIONAL SUMMARY
# ============================================================

per_sequence_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for sequence_id, group in (
        dataframe.groupby(
            "sequence_id",
            sort=False,
        )
    ):

        hotspots = group[
            group["is_hotspot"]
        ]

        nonhotspots = group[
            ~group["is_hotspot"]
        ]

        record = {
            "dataset": dataset_name,
            "sequence_id": (
                sequence_id
            ),
            "sequence": str(
                group["sequence"].iloc[0]
            ),
            "label": int(
                group["label"].iloc[0]
            ),
            "sequence_length": int(
                group[
                    "sequence_length"
                ].iloc[0]
            ),
            "number_of_hotspots": int(
                len(hotspots)
            ),
            "mean_hotspot_position": float(
                hotspots[
                    "normalized_position"
                ].mean()
            ),
            "median_hotspot_position": float(
                hotspots[
                    "normalized_position"
                ].median()
            ),
            "mean_nonhotspot_position": float(
                nonhotspots[
                    "normalized_position"
                ].mean()
            ),
            "median_nonhotspot_position": float(
                nonhotspots[
                    "normalized_position"
                ].median()
            ),
        }

        for region_name in (
            REGION_BOUNDARIES
        ):
            hotspot_region_fraction = float(
                (
                    hotspots[
                        "position_region"
                    ]
                    == region_name
                ).mean()
            )

            nonhotspot_region_fraction = float(
                (
                    nonhotspots[
                        "position_region"
                    ]
                    == region_name
                ).mean()
            )

            record[
                f"hotspot_fraction_{region_name}"
            ] = hotspot_region_fraction

            record[
                f"nonhotspot_fraction_{region_name}"
            ] = nonhotspot_region_fraction

            record[
                f"hotspot_minus_nonhotspot_{region_name}"
            ] = (
                hotspot_region_fraction
                - nonhotspot_region_fraction
            )

        per_sequence_rows.append(
            record
        )


per_sequence_df = pd.DataFrame(
    per_sequence_rows
)


# ============================================================
# 10. SEQUENCE-LEVEL PAIRED TESTS WITHIN CPPs
# ============================================================

sequence_test_rows = []

for dataset_name in DATASETS:

    dataset_table = per_sequence_df[
        per_sequence_df["dataset"]
        == dataset_name
    ]

    cpp_table = dataset_table[
        dataset_table["label"] == 1
    ]

    noncpp_table = dataset_table[
        dataset_table["label"] == 0
    ]

    # Paired hotspot vs non-hotspot positional fractions
    # within CPP sequences.
    for region_name in (
        REGION_BOUNDARIES
    ):

        hotspot_column = (
            f"hotspot_fraction_{region_name}"
        )

        nonhotspot_column = (
            f"nonhotspot_fraction_{region_name}"
        )

        statistic, p_value = (
            safe_wilcoxon(
                cpp_table[
                    hotspot_column
                ],
                cpp_table[
                    nonhotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "paired_CPP_hotspot_vs_CPP_nonhotspot"
            ),
            "metric": (
                f"fraction_{region_name}"
            ),
            "group_1": "CPP_hotspot",
            "group_2": "CPP_nonhotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(cpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_2_mean": float(
                cpp_table[
                    nonhotspot_column
                ].mean()
            ),
            "mean_difference": float(
                (
                    cpp_table[
                        hotspot_column
                    ]
                    - cpp_table[
                        nonhotspot_column
                    ]
                ).mean()
            ),
            "test": (
                "paired Wilcoxon signed-rank"
            ),
            "statistic": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })

        # CPP vs non-CPP hotspot positional preference.
        statistic, p_value = (
            safe_mannwhitney(
                cpp_table[
                    hotspot_column
                ],
                noncpp_table[
                    hotspot_column
                ],
            )
        )

        sequence_test_rows.append({
            "dataset": dataset_name,
            "analysis": (
                "CPP_hotspot_vs_nonCPP_hotspot"
            ),
            "metric": (
                f"fraction_{region_name}"
            ),
            "group_1": "CPP_hotspot",
            "group_2": "nonCPP_hotspot",
            "group_1_n": int(
                len(cpp_table)
            ),
            "group_2_n": int(
                len(noncpp_table)
            ),
            "group_1_mean": float(
                cpp_table[
                    hotspot_column
                ].mean()
            ),
            "group_2_mean": float(
                noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "mean_difference": float(
                cpp_table[
                    hotspot_column
                ].mean()
                - noncpp_table[
                    hotspot_column
                ].mean()
            ),
            "test": "Mann-Whitney U",
            "statistic": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })

    # Mean normalized hotspot position:
    # CPP vs non-CPP.
    statistic, p_value = (
        safe_mannwhitney(
            cpp_table[
                "mean_hotspot_position"
            ],
            noncpp_table[
                "mean_hotspot_position"
            ],
        )
    )

    sequence_test_rows.append({
        "dataset": dataset_name,
        "analysis": (
            "CPP_hotspot_vs_nonCPP_hotspot"
        ),
        "metric": (
            "mean_normalized_hotspot_position"
        ),
        "group_1": "CPP_hotspot",
        "group_2": "nonCPP_hotspot",
        "group_1_n": int(
            len(cpp_table)
        ),
        "group_2_n": int(
            len(noncpp_table)
        ),
        "group_1_mean": float(
            cpp_table[
                "mean_hotspot_position"
            ].mean()
        ),
        "group_2_mean": float(
            noncpp_table[
                "mean_hotspot_position"
            ].mean()
        ),
        "mean_difference": float(
            cpp_table[
                "mean_hotspot_position"
            ].mean()
            - noncpp_table[
                "mean_hotspot_position"
            ].mean()
        ),
        "test": "Mann-Whitney U",
        "statistic": float(
            statistic
        ),
        "p_value": float(
            p_value
        ),
    })


sequence_tests_df = pd.DataFrame(
    sequence_test_rows
)

sequence_tests_df["fdr_bh"] = (
    sequence_tests_df.groupby(
        [
            "dataset",
            "analysis",
        ]
    )[
        "p_value"
    ].transform(
        lambda values:
            benjamini_hochberg(
                values.to_numpy()
            )
    )
)

sequence_tests_df[
    "significant_fdr_0_05"
] = (
    sequence_tests_df[
        "fdr_bh"
    ] < 0.05
)


# ============================================================
# 11. HOTSPOT POSITION DISTRIBUTION VS UNIFORM
#
# This is supplementary because residues in finite peptides
# are discrete rather than perfectly continuous.
# ============================================================

uniformity_rows = []

for dataset_name, dataframe in (
    residue_tables.items()
):

    for label_value, class_name in [
        (1, "CPP"),
        (0, "non_CPP"),
    ]:

        hotspot_positions = dataframe[
            (
                dataframe["label"]
                == label_value
            )
            & dataframe["is_hotspot"]
        ][
            "normalized_position"
        ].to_numpy()

        statistic, p_value = kstest(
            hotspot_positions,
            "uniform",
            args=(0, 1),
        )

        uniformity_rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "number_of_hotspots": int(
                len(hotspot_positions)
            ),
            "mean_normalized_position": float(
                np.mean(
                    hotspot_positions
                )
            ),
            "median_normalized_position": float(
                np.median(
                    hotspot_positions
                )
            ),
            "ks_statistic_vs_uniform": float(
                statistic
            ),
            "p_value": float(
                p_value
            ),
        })


uniformity_df = pd.DataFrame(
    uniformity_rows
)

uniformity_df["fdr_bh"] = (
    benjamini_hochberg(
        uniformity_df[
            "p_value"
        ].to_numpy()
    )
)


# ============================================================
# 12. INTERNAL–KELM REPLICATION
# ============================================================

replication_source = (
    region_enrichment_df[
        region_enrichment_df[
            "class"
        ] == "CPP"
    ][
        [
            "dataset",
            "region",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
)


internal_replication = (
    replication_source[
        replication_source["dataset"]
        == "internal_test"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "internal_odds_ratio",
            "log2_enrichment":
                "internal_log2_enrichment",
            "fdr_bh":
                "internal_fdr",
            "significant_fdr_0_05":
                "internal_significant",
        }
    )
)


kelm_replication = (
    replication_source[
        replication_source["dataset"]
        == "kelm_external"
    ]
    .drop(
        columns="dataset"
    )
    .rename(
        columns={
            "odds_ratio_corrected":
                "kelm_odds_ratio",
            "log2_enrichment":
                "kelm_log2_enrichment",
            "fdr_bh":
                "kelm_fdr",
            "significant_fdr_0_05":
                "kelm_significant",
        }
    )
)


replication_df = (
    internal_replication.merge(
        kelm_replication,
        on="region",
        how="outer",
        validate="one_to_one",
    )
)

replication_df[
    "same_direction"
] = (
    np.sign(
        replication_df[
            "internal_log2_enrichment"
        ]
    )
    == np.sign(
        replication_df[
            "kelm_log2_enrichment"
        ]
    )
)

replication_df[
    "significant_in_both"
] = (
    replication_df[
        "internal_significant"
    ].fillna(False)
    & replication_df[
        "kelm_significant"
    ].fillna(False)
)


# ============================================================
# 13. SAVE TABLES
# ============================================================

region_enrichment_file = (
    RESULT_TABLE_DIR
    / "hotspot_positional_region_enrichment.csv"
)

position_bin_file = (
    RESULT_TABLE_DIR
    / "hotspot_position_10bin_distribution.csv"
)

per_sequence_file = (
    RESULT_SI_DIR
    / "hotspot_positional_metrics_per_sequence.csv"
)

sequence_tests_file = (
    RESULT_TABLE_DIR
    / "hotspot_positional_sequence_tests.csv"
)

uniformity_file = (
    RESULT_SI_DIR
    / "hotspot_position_uniformity_tests.csv"
)

replication_file = (
    RESULT_TABLE_DIR
    / "hotspot_positional_replication.csv"
)


region_enrichment_df.to_csv(
    region_enrichment_file,
    index=False,
)

position_bin_df.to_csv(
    position_bin_file,
    index=False,
)

per_sequence_df.to_csv(
    per_sequence_file,
    index=False,
)

sequence_tests_df.to_csv(
    sequence_tests_file,
    index=False,
)

uniformity_df.to_csv(
    uniformity_file,
    index=False,
)

replication_df.to_csv(
    replication_file,
    index=False,
)


# ============================================================
# 14. FIGURE — HOTSPOT POSITION DENSITY
# ============================================================

for dataset_name in DATASETS:

    dataframe = residue_tables[
        dataset_name
    ]

    cpp_hotspots = dataframe[
        (
            dataframe["label"] == 1
        )
        & dataframe["is_hotspot"]
    ][
        "normalized_position"
    ].to_numpy()

    cpp_nonhotspots = dataframe[
        (
            dataframe["label"] == 1
        )
        & ~dataframe["is_hotspot"]
    ][
        "normalized_position"
    ].to_numpy()

    bins = np.linspace(
        0,
        1,
        21,
    )

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    ax.hist(
        cpp_hotspots,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        label="CPP hotspots",
    )

    ax.hist(
        cpp_nonhotspots,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        linestyle="--",
        label="CPP non-hotspots",
    )

    ax.axvline(
        1.0 / 3.0,
        linestyle=":",
        linewidth=1,
    )

    ax.axvline(
        2.0 / 3.0,
        linestyle=":",
        linewidth=1,
    )

    ax.set_xlim(
        0,
        1,
    )

    ax.set_xlabel(
        "Normalized residue position"
    )

    ax.set_ylabel(
        "Density"
    )

    ax.set_title(
        f"Positional distribution of CPP hotspots\n"
        f"{dataset_name}"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            f"CPP_hotspot_position_density_"
            f"{dataset_name}"
        )
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 15. FIGURE — 10-BIN HOTSPOT ENRICHMENT
# ============================================================

for dataset_name in DATASETS:

    plot_table = position_bin_df[
        (
            position_bin_df["dataset"]
            == dataset_name
        )
        & (
            position_bin_df["class"]
            == "CPP"
        )
    ].sort_values(
        "position_bin"
    )

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    ax.plot(
        plot_table[
            "position_bin"
        ],
        plot_table[
            "hotspot_frequency"
        ],
        marker="o",
        label="CPP hotspots",
    )

    ax.plot(
        plot_table[
            "position_bin"
        ],
        plot_table[
            "nonhotspot_frequency"
        ],
        marker="s",
        linestyle="--",
        label="CPP non-hotspots",
    )

    ax.set_xticks(
        range(
            1,
            NUMBER_OF_POSITION_BINS + 1,
        )
    )

    ax.set_xticklabels(
        [
            f"{(index - 1) * 10}–{index * 10}%"
            for index in range(
                1,
                NUMBER_OF_POSITION_BINS + 1
            )
        ],
        rotation=35,
        ha="right",
    )

    ax.set_xlabel(
        "Relative sequence position"
    )

    ax.set_ylabel(
        "Fraction of residues"
    )

    ax.set_title(
        f"CPP hotspot positional profile\n"
        f"{dataset_name}"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    figure_base = (
        FIGURE_MAIN_DIR
        / (
            f"CPP_hotspot_position_10bin_"
            f"{dataset_name}"
        )
    )

    fig.savefig(
        figure_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        figure_base.with_suffix(".svg"),
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 16. FIGURE — N/MIDDLE/C ENRICHMENT
# ============================================================

cpp_region_plot = region_enrichment_df[
    region_enrichment_df[
        "class"
    ] == "CPP"
].copy()

region_order = [
    "N_terminal",
    "Middle",
    "C_terminal",
]

x = np.arange(
    len(region_order)
)

width = 0.36

internal_values = []
kelm_values = []

for region_name in region_order:

    internal_values.append(
        float(
            cpp_region_plot[
                (
                    cpp_region_plot["dataset"]
                    == "internal_test"
                )
                & (
                    cpp_region_plot["region"]
                    == region_name
                )
            ][
                "log2_enrichment"
            ].iloc[0]
        )
    )

    kelm_values.append(
        float(
            cpp_region_plot[
                (
                    cpp_region_plot["dataset"]
                    == "kelm_external"
                )
                & (
                    cpp_region_plot["region"]
                    == region_name
                )
            ][
                "log2_enrichment"
            ].iloc[0]
        )
    )


fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

ax.bar(
    x - width / 2,
    internal_values,
    width=width,
    label="Internal test",
)

ax.bar(
    x + width / 2,
    kelm_values,
    width=width,
    label="KELM external",
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        "N-terminal",
        "Middle",
        "C-terminal",
    ]
)

ax.set_ylabel(
    "log$_2$ enrichment in CPP hotspots"
)

ax.set_title(
    "Regional preference of CPP consensus hotspots"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

regional_figure_base = (
    FIGURE_MAIN_DIR
    / "CPP_hotspot_regional_enrichment"
)

fig.savefig(
    regional_figure_base.with_suffix(".png"),
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    regional_figure_base.with_suffix(".pdf"),
    bbox_inches="tight",
)

fig.savefig(
    regional_figure_base.with_suffix(".svg"),
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ============================================================
# 17. SUMMARY JSON
# ============================================================

replicated_significant_regions = (
    replication_df[
        replication_df[
            "significant_in_both"
        ]
    ][
        "region"
    ].tolist()
)

same_direction_regions = (
    replication_df[
        replication_df[
            "same_direction"
        ]
    ][
        "region"
    ].tolist()
)

summary = {
    "datasets": DATASETS,
    "hotspot_definition": (
        "Top 20% redundancy-adjusted global "
        "consensus residue ranks"
    ),
    "coarse_regions": {
        "N_terminal": (
            "normalized position <= 1/3"
        ),
        "Middle": (
            "1/3 < normalized position <= 2/3"
        ),
        "C_terminal": (
            "normalized position > 2/3"
        ),
    },
    "number_of_detailed_bins": int(
        NUMBER_OF_POSITION_BINS
    ),
    "replicated_significant_regions": (
        replicated_significant_regions
    ),
    "same_direction_regions": (
        same_direction_regions
    ),
    "completed_at": (
        pd.Timestamp.now().isoformat()
    ),
}

summary_file = (
    CHECKPOINT_DIR
    / "hotspot_positional_preference_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )


# ============================================================
# 18. SAVE CHECKPOINT
# ============================================================

mark_step_complete(
    "19_hotspot_positional_preference",
    output_files=[
        region_enrichment_file,
        position_bin_file,
        per_sequence_file,
        sequence_tests_file,
        uniformity_file,
        replication_file,
        summary_file,
        regional_figure_base.with_suffix(
            ".png"
        ),
        regional_figure_base.with_suffix(
            ".pdf"
        ),
        regional_figure_base.with_suffix(
            ".svg"
        ),
    ],
    details=summary,
)


# ============================================================
# 19. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 78)
print("CPP HOTSPOT REGIONAL ENRICHMENT")
print("=" * 78)

display(
    region_enrichment_df[
        region_enrichment_df[
            "class"
        ] == "CPP"
    ][
        [
            "dataset",
            "region",
            "hotspot_frequency",
            "nonhotspot_frequency",
            "odds_ratio_corrected",
            "log2_enrichment",
            "fdr_bh",
            "significant_fdr_0_05",
            "direction",
        ]
    ]
)

print("\n" + "=" * 78)
print("INTERNAL–KELM POSITIONAL REPLICATION")
print("=" * 78)

display(
    replication_df[
        [
            "region",
            "internal_odds_ratio",
            "internal_log2_enrichment",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_log2_enrichment",
            "kelm_fdr",
            "same_direction",
            "significant_in_both",
        ]
    ]
)

print("\n" + "=" * 78)
print("SEQUENCE-LEVEL POSITIONAL TESTS")
print("=" * 78)

display(
    sequence_tests_df[
        [
            "dataset",
            "analysis",
            "metric",
            "group_1_mean",
            "group_2_mean",
            "mean_difference",
            "test",
            "fdr_bh",
            "significant_fdr_0_05",
        ]
    ]
)

print("\n" + "=" * 78)
print("DETAILED CPP POSITION PROFILE")
print("=" * 78)

display(
    position_bin_df[
        position_bin_df[
            "class"
        ] == "CPP"
    ][
        [
            "dataset",
            "position_bin",
            "bin_start",
            "bin_end",
            "hotspot_frequency",
            "nonhotspot_frequency",
            "frequency_difference",
            "hotspot_to_nonhotspot_ratio",
        ]
    ]
)

print("\nReplicated significant regions:")
print(
    replicated_significant_regions
)

print("\nRegions with the same direction:")
print(
    same_direction_regions
)

print("\nSaved regional enrichment:")
print(region_enrichment_file)

print("\nSaved detailed position profile:")
print(position_bin_file)

print("\nSaved positional replication:")
print(replication_file)

print("\n" + "=" * 78)
print("STEP 19 COMPLETED")
print("=" * 78)

In [ ]:
# ============================================================
# STEP 20: FINAL RESULTS AUDIT AND MANUSCRIPT CONTENT PLAN
#
# Outputs:
# 1. Complete file audit
# 2. Pipeline checkpoint audit
# 3. Key-result consolidation
# 4. Statistical consistency checks
# 5. Proposed main/SI figures
# 6. Proposed main/SI tables
# 7. Manuscript section roadmap
# 8. Excel audit workbook
# ============================================================

from pathlib import Path
import json
import math
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

DIRS = {
    "code": PROJECT_DIR / "00_code",
    "data_original": PROJECT_DIR / "01_data_original",
    "data_processed": PROJECT_DIR / "02_data_processed",
    "embeddings": PROJECT_DIR / "03_embeddings",
    "models": PROJECT_DIR / "04_models",
    "predictions": PROJECT_DIR / "05_predictions",
    "xai": PROJECT_DIR / "06_xai",
    "results": PROJECT_DIR / "07_results",
    "checkpoints": PROJECT_DIR / "08_checkpoints",
    "logs": PROJECT_DIR / "09_logs",
}

TABLE_MAIN_DIR = (
    DIRS["results"] / "tables_main"
)

TABLE_SI_DIR = (
    DIRS["results"] / "tables_SI"
)

FIGURE_MAIN_DIR = (
    DIRS["results"] / "figures_main"
)

FIGURE_SI_DIR = (
    DIRS["results"] / "figures_SI"
)

AUDIT_DIR = (
    DIRS["results"] / "manuscript_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for folder in [
    TABLE_MAIN_DIR,
    TABLE_SI_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SI_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. EXPECTED PIPELINE STEPS
# ============================================================

EXPECTED_STEPS = [
    "01_project_directory_setup",
    "02_environment_setup",
    "03_correct_combined_datasets",
    "04_dataset_quality_control",
    "05_fixed_internal_data_splits",
    "06_all_PLM_embeddings",
    "07_tensorflow_environment",
    "08_all_attention_classifiers",
    "09_four_PLM_ensemble",
    "10_all_attention_scores",
    "11_all_gradient_input_scores",
    "12_all_integrated_gradients_scores",
    "13_consensus_XAI",
    "13B_redundancy_adjusted_consensus",
    "14_residue_enrichment",
    "15_hotspot_motif_discovery",
    "16_publication_ready_hotspot_faithfulness",
    "17_cross_PLM_hotspot_conservation",
    "18_physicochemical_hotspot_enrichment",
    "19_hotspot_positional_preference",
]


# ============================================================
# 3. EXPECTED CRITICAL FILES
# ============================================================

EXPECTED_FILES = {
    # Data
    "internal_qc_dataset":
        DIRS["data_processed"]
        / "internal_dataset_qc.csv",

    "kelm_qc_dataset":
        DIRS["data_processed"]
        / "kelm_external_dataset_qc.csv",

    "train_split":
        DIRS["data_processed"]
        / "train_split.csv",

    "validation_split":
        DIRS["data_processed"]
        / "validation_split.csv",

    "internal_test_split":
        DIRS["data_processed"]
        / "internal_test_split.csv",

    # Performance
    "individual_model_performance":
        TABLE_MAIN_DIR
        / "attention_classifier_performance.csv",

    "ensemble_performance":
        TABLE_MAIN_DIR
        / "four_plm_ensemble_performance.csv",

    "individual_vs_ensemble":
        TABLE_MAIN_DIR
        / "individual_vs_ensemble_comparison.csv",

    # XAI
    "consensus_summary":
        TABLE_MAIN_DIR
        / "consensus_xai_summary.csv",

    "method_agreement":
        TABLE_MAIN_DIR
        / "xai_method_agreement.csv",

    "cross_model_agreement":
        TABLE_MAIN_DIR
        / "cross_model_xai_agreement.csv",

    "adjusted_consensus_summary":
        TABLE_MAIN_DIR
        / "adjusted_consensus_xai_summary.csv",

    "original_vs_adjusted_consensus":
        TABLE_MAIN_DIR
        / "original_vs_adjusted_consensus.csv",

    # Biological analyses
    "residue_enrichment":
        TABLE_MAIN_DIR
        / "residue_enrichment_top20.csv",

    "residue_replication":
        TABLE_MAIN_DIR
        / "residue_enrichment_replication.csv",

    "significant_motifs":
        TABLE_MAIN_DIR
        / "strict_replicated_CPP_hotspot_motifs.csv",

    "motif_replication":
        TABLE_MAIN_DIR
        / "hotspot_motif_replication.csv",

    "faithfulness":
        TABLE_MAIN_DIR
        / "CPP_hotspot_faithfulness_manuscript_table.csv",

    "faithfulness_cross_model":
        TABLE_MAIN_DIR
        / "hotspot_faithfulness_cross_model_summary.csv",

    "conservation_pairwise":
        TABLE_MAIN_DIR
        / "cross_plm_pairwise_hotspot_jaccard.csv",

    "conservation_summary":
        TABLE_MAIN_DIR
        / "cross_plm_hotspot_conservation_summary.csv",

    "conservation_class_comparison":
        TABLE_MAIN_DIR
        / "CPP_vs_nonCPP_hotspot_conservation.csv",

    "conservation_residue_level":
        TABLE_MAIN_DIR
        / "residue_level_cross_plm_conservation_enrichment.csv",

    "physicochemical_enrichment":
        TABLE_MAIN_DIR
        / "physicochemical_category_enrichment.csv",

    "physicochemical_replication":
        TABLE_MAIN_DIR
        / "physicochemical_enrichment_replication.csv",

    "physicochemical_sequence_tests":
        TABLE_MAIN_DIR
        / "physicochemical_sequence_level_tests.csv",

    "positional_enrichment":
        TABLE_MAIN_DIR
        / "hotspot_positional_region_enrichment.csv",

    "positional_replication":
        TABLE_MAIN_DIR
        / "hotspot_positional_replication.csv",

    "positional_sequence_tests":
        TABLE_MAIN_DIR
        / "hotspot_positional_sequence_tests.csv",

    "positional_10bin":
        TABLE_MAIN_DIR
        / "hotspot_position_10bin_distribution.csv",

    # Consensus residue-level outputs
    "internal_adjusted_consensus":
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / "internal_test"
        / "adjusted_global_consensus_residue_scores.csv",

    "kelm_adjusted_consensus":
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / "kelm_external"
        / "adjusted_global_consensus_residue_scores.csv",
}


# ============================================================
# 4. LOAD PIPELINE STATUS
# ============================================================

STATUS_FILE = (
    DIRS["checkpoints"]
    / "pipeline_status.json"
)

if STATUS_FILE.exists():
    with open(
        STATUS_FILE,
        "r",
        encoding="utf-8",
    ) as handle:
        PIPELINE_STATUS = json.load(
            handle
        )
else:
    PIPELINE_STATUS = {}


# ============================================================
# 5. PIPELINE STEP AUDIT
# ============================================================

step_audit_rows = []

for step_name in EXPECTED_STEPS:

    status_record = PIPELINE_STATUS.get(
        step_name,
        {}
    )

    step_audit_rows.append({
        "step": step_name,
        "recorded_in_pipeline_status": bool(
            step_name in PIPELINE_STATUS
        ),
        "completed_flag": bool(
            status_record.get(
                "completed",
                False,
            )
        ),
        "number_of_recorded_output_files": int(
            len(
                status_record.get(
                    "output_files",
                    [],
                )
            )
        ),
        "details_present": bool(
            status_record.get(
                "details"
            )
        ),
    })


step_audit_df = pd.DataFrame(
    step_audit_rows
)


# ============================================================
# 6. FILE AUDIT
# ============================================================

file_audit_rows = []

for file_key, file_path in (
    EXPECTED_FILES.items()
):

    exists = file_path.exists()

    size_bytes = (
        file_path.stat().st_size
        if exists
        else 0
    )

    row_count = None
    column_count = None
    read_status = (
        "missing"
        if not exists
        else "not_checked"
    )

    if exists:
        try:
            if file_path.suffix.lower() == ".csv":
                dataframe = pd.read_csv(
                    file_path
                )

                row_count = int(
                    len(dataframe)
                )

                column_count = int(
                    len(dataframe.columns)
                )

                if len(dataframe) == 0:
                    read_status = "empty"
                else:
                    read_status = "readable"

            elif file_path.suffix.lower() == ".json":
                with open(
                    file_path,
                    "r",
                    encoding="utf-8",
                ) as handle:
                    json.load(handle)

                read_status = "readable"

            else:
                read_status = "exists"

        except Exception as error:
            read_status = (
                f"read_error: "
                f"{type(error).__name__}"
            )

    file_audit_rows.append({
        "file_key": file_key,
        "path": str(file_path),
        "exists": exists,
        "size_bytes": int(
            size_bytes
        ),
        "size_MB": float(
            size_bytes / (1024 ** 2)
        ),
        "row_count": row_count,
        "column_count": column_count,
        "read_status": read_status,
    })


file_audit_df = pd.DataFrame(
    file_audit_rows
)


# ============================================================
# 7. GENERAL CSV VALIDATION
# ============================================================

def load_required_csv(
    file_key,
):
    file_path = EXPECTED_FILES[
        file_key
    ]

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing required file: "
            f"{file_path}"
        )

    return pd.read_csv(
        file_path
    )


validation_issue_rows = []


def add_validation_issue(
    analysis,
    severity,
    issue,
    recommendation,
):
    validation_issue_rows.append({
        "analysis": analysis,
        "severity": severity,
        "issue": issue,
        "recommendation": recommendation,
    })


# ============================================================
# 8. DATASET CONSISTENCY AUDIT
# ============================================================

try:
    train_df = load_required_csv(
        "train_split"
    )

    validation_df = load_required_csv(
        "validation_split"
    )

    internal_test_df = load_required_csv(
        "internal_test_split"
    )

    kelm_df = load_required_csv(
        "kelm_qc_dataset"
    )

    dataset_summary_rows = []

    for dataset_name, dataframe in [
        ("train", train_df),
        ("validation", validation_df),
        ("internal_test", internal_test_df),
        ("kelm_external", kelm_df),
    ]:

        dataset_summary_rows.append({
            "dataset": dataset_name,
            "rows": int(
                len(dataframe)
            ),
            "CPP_count": int(
                (
                    dataframe["label"]
                    == 1
                ).sum()
            ),
            "nonCPP_count": int(
                (
                    dataframe["label"]
                    == 0
                ).sum()
            ),
            "minimum_length": int(
                dataframe["length"].min()
            ),
            "maximum_length": int(
                dataframe["length"].max()
            ),
            "mean_length": float(
                dataframe["length"].mean()
            ),
            "duplicate_sequence_ids": int(
                dataframe[
                    "sequence_id"
                ].duplicated().sum()
            ),
            "duplicate_sequences": int(
                dataframe[
                    "sequence"
                ].duplicated().sum()
            ),
        })

    dataset_summary_df = pd.DataFrame(
        dataset_summary_rows
    )

    split_sets = {
        "train": set(
            train_df["sequence"]
        ),
        "validation": set(
            validation_df["sequence"]
        ),
        "internal_test": set(
            internal_test_df["sequence"]
        ),
        "kelm_external": set(
            kelm_df["sequence"]
        ),
    }

    overlap_rows = []

    split_names = list(
        split_sets.keys()
    )

    for index_a in range(
        len(split_names)
    ):
        for index_b in range(
            index_a + 1,
            len(split_names),
        ):
            name_a = split_names[
                index_a
            ]

            name_b = split_names[
                index_b
            ]

            overlap_count = len(
                split_sets[name_a]
                & split_sets[name_b]
            )

            overlap_rows.append({
                "dataset_a": name_a,
                "dataset_b": name_b,
                "exact_sequence_overlap": int(
                    overlap_count
                ),
            })

            if overlap_count > 0:
                add_validation_issue(
                    analysis="dataset_splits",
                    severity="high",
                    issue=(
                        f"{overlap_count} exact sequence "
                        f"overlaps between {name_a} "
                        f"and {name_b}"
                    ),
                    recommendation=(
                        "Inspect and remove overlap before "
                        "manuscript submission."
                    ),
                )

    split_overlap_df = pd.DataFrame(
        overlap_rows
    )

except Exception as error:

    dataset_summary_df = pd.DataFrame()
    split_overlap_df = pd.DataFrame()

    add_validation_issue(
        analysis="dataset_splits",
        severity="high",
        issue=(
            f"Dataset audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check processed dataset and split files."
        ),
    )


# ============================================================
# 9. MODEL PERFORMANCE CONSOLIDATION
# ============================================================

try:
    model_performance_df = (
        load_required_csv(
            "individual_vs_ensemble"
        )
    )

    performance_summary_rows = []

    for dataset_name in [
        "validation",
        "internal_test",
        "kelm_external",
    ]:

        subset = model_performance_df[
            model_performance_df[
                "dataset"
            ] == dataset_name
        ].copy()

        if len(subset) == 0:
            continue

        for metric in [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "mcc",
            "roc_auc",
            "pr_auc",
        ]:

            best_row = subset.loc[
                subset[metric].idxmax()
            ]

            performance_summary_rows.append({
                "dataset": dataset_name,
                "metric": metric,
                "best_method": (
                    best_row["method"]
                ),
                "best_value": float(
                    best_row[metric]
                ),
                "threshold": float(
                    best_row[
                        "threshold"
                    ]
                ),
            })

    performance_summary_df = pd.DataFrame(
        performance_summary_rows
    )

    duplicate_performance_rows = (
        model_performance_df.duplicated(
            subset=[
                "method",
                "dataset",
            ]
        ).sum()
    )

    if duplicate_performance_rows > 0:
        add_validation_issue(
            analysis="model_performance",
            severity="medium",
            issue=(
                f"{duplicate_performance_rows} "
                f"duplicated method-dataset rows."
            ),
            recommendation=(
                "Remove duplicated rows before final tables."
            ),
        )

except Exception as error:

    model_performance_df = pd.DataFrame()
    performance_summary_df = pd.DataFrame()

    add_validation_issue(
        analysis="model_performance",
        severity="high",
        issue=(
            f"Performance audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check individual_vs_ensemble_comparison.csv."
        ),
    )


# ============================================================
# 10. XAI AGREEMENT CONSOLIDATION
# ============================================================

try:
    method_agreement_df = (
        load_required_csv(
            "method_agreement"
        )
    )

    cross_model_agreement_df = (
        load_required_csv(
            "cross_model_agreement"
        )
    )

    original_adjusted_df = (
        load_required_csv(
            "original_vs_adjusted_consensus"
        )
    )

    xai_summary_rows = []

    gradient_pairs = method_agreement_df[
        method_agreement_df[
            "method_pair"
        ].str.contains(
            "Gradient×Input vs Integrated",
            regex=False,
        )
    ]

    attention_pairs = method_agreement_df[
        method_agreement_df[
            "method_pair"
        ].str.contains(
            "Attention vs",
            regex=False,
        )
    ]

    for dataset_name in [
        "internal_test",
        "kelm_external",
    ]:

        gradient_subset = gradient_pairs[
            gradient_pairs[
                "dataset"
            ] == dataset_name
        ]

        attention_subset = attention_pairs[
            attention_pairs[
                "dataset"
            ] == dataset_name
        ]

        cross_model_subset = (
            cross_model_agreement_df[
                cross_model_agreement_df[
                    "dataset"
                ] == dataset_name
            ]
        )

        adjusted_subset = (
            original_adjusted_df[
                original_adjusted_df[
                    "dataset"
                ] == dataset_name
            ]
        )

        xai_summary_rows.append({
            "dataset": dataset_name,
            "mean_attention_vs_gradient_spearman": float(
                attention_subset[
                    "mean_sequence_spearman"
                ].mean()
            ),
            "mean_gradient_vs_IG_spearman": float(
                gradient_subset[
                    "mean_sequence_spearman"
                ].mean()
            ),
            "mean_cross_model_spearman": float(
                cross_model_subset[
                    "mean_sequence_spearman"
                ].mean()
            ),
            "maximum_cross_model_spearman": float(
                cross_model_subset[
                    "mean_sequence_spearman"
                ].max()
            ),
            "original_vs_adjusted_rank_spearman": float(
                adjusted_subset[
                    "overall_rank_spearman"
                ].iloc[0]
            ),
            "original_vs_adjusted_top20_jaccard": float(
                adjusted_subset[
                    "top20_jaccard"
                ].iloc[0]
            ),
        })

    xai_audit_summary_df = pd.DataFrame(
        xai_summary_rows
    )

    if (
        method_agreement_df[
            method_agreement_df[
                "method_pair"
            ].str.contains(
                "Gradient×Input vs Integrated",
                regex=False,
            )
        ][
            "mean_sequence_spearman"
        ].mean()
        > 0.99
    ):
        add_validation_issue(
            analysis="XAI_consensus",
            severity="informational",
            issue=(
                "Gradient × Input and Integrated "
                "Gradients are nearly redundant."
            ),
            recommendation=(
                "Use the redundancy-adjusted consensus "
                "in all main-text analyses."
            ),
        )

except Exception as error:

    method_agreement_df = pd.DataFrame()
    cross_model_agreement_df = pd.DataFrame()
    original_adjusted_df = pd.DataFrame()
    xai_audit_summary_df = pd.DataFrame()

    add_validation_issue(
        analysis="XAI_consensus",
        severity="high",
        issue=(
            f"XAI audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check XAI agreement and consensus files."
        ),
    )


# ============================================================
# 11. RESIDUE ENRICHMENT CONSOLIDATION
# ============================================================

try:
    residue_replication_df = (
        load_required_csv(
            "residue_replication"
        )
    )

    residue_result_rows = []

    for _, row in (
        residue_replication_df.iterrows()
    ):

        internal_direction = (
            "enriched"
            if row[
                "internal_log2_enrichment"
            ] > 0
            else "depleted"
        )

        kelm_direction = (
            "enriched"
            if row[
                "kelm_log2_enrichment"
            ] > 0
            else "depleted"
        )

        residue_result_rows.append({
            "residue": row["residue"],
            "internal_direction": (
                internal_direction
            ),
            "kelm_direction": (
                kelm_direction
            ),
            "same_direction": bool(
                row[
                    "same_enrichment_direction"
                ]
            ),
            "significant_in_both": bool(
                row[
                    "significant_in_both"
                ]
            ),
            "internal_odds_ratio": float(
                row[
                    "internal_odds_ratio"
                ]
            ),
            "kelm_odds_ratio": float(
                row[
                    "kelm_odds_ratio"
                ]
            ),
            "internal_fdr": float(
                row[
                    "internal_fdr"
                ]
            ),
            "kelm_fdr": float(
                row[
                    "kelm_fdr"
                ]
            ),
        })

    residue_key_results_df = (
        pd.DataFrame(
            residue_result_rows
        )
    )

except Exception as error:

    residue_replication_df = pd.DataFrame()
    residue_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="residue_enrichment",
        severity="high",
        issue=(
            f"Residue audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check residue enrichment replication file."
        ),
    )


# ============================================================
# 12. MOTIF CONSOLIDATION
# ============================================================

try:
    motifs_df = load_required_csv(
        "significant_motifs"
    )

    motif_key_results_df = motifs_df[
        [
            "motif",
            "motif_length",
            "internal_cpp_count",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_cpp_count",
            "kelm_odds_ratio",
            "kelm_fdr",
            "contains_K_or_R",
            "basic_residue_fraction",
        ]
    ].copy()

except Exception as error:

    motifs_df = pd.DataFrame()
    motif_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="motif_discovery",
        severity="high",
        issue=(
            f"Motif audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check strict replicated motif file."
        ),
    )


# ============================================================
# 13. FAITHFULNESS CONSOLIDATION
# ============================================================

try:
    faithfulness_df = load_required_csv(
        "faithfulness"
    )

    faithfulness_key_results_df = (
        faithfulness_df[
            [
                "model",
                "dataset",
                "n",
                "mean_original_probability",
                "mean_hotspot_probability_drop",
                "hotspot_drop_ci95_low",
                "hotspot_drop_ci95_high",
                "mean_random_probability_drop",
                "mean_hotspot_minus_random_drop",
                "difference_ci95_low",
                "difference_ci95_high",
                "fraction_hotspot_stronger_than_random",
                "paired_cohens_dz",
                "rank_biserial_effect_size",
                "wilcoxon_p_value",
                "mean_hotspot_only_probability",
            ]
        ].copy()
    )

    suspicious_ratio_column = (
        "mean_hotspot_only_probability_fraction"
    )

    if suspicious_ratio_column in (
        faithfulness_df.columns
    ):
        suspicious_values = (
            faithfulness_df[
                suspicious_ratio_column
            ]
        )

        if (
            suspicious_values.replace(
                [np.inf, -np.inf],
                np.nan,
            ).max()
            > 2.0
        ):
            add_validation_issue(
                analysis="faithfulness",
                severity="medium",
                issue=(
                    "Hotspot-only/original probability "
                    "ratio contains unstable extreme values."
                ),
                recommendation=(
                    "Exclude this ratio from the manuscript; "
                    "report hotspot-only probability and "
                    "sufficiency loss instead."
                ),
            )

except Exception as error:

    faithfulness_df = pd.DataFrame()
    faithfulness_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="faithfulness",
        severity="high",
        issue=(
            f"Faithfulness audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check faithfulness manuscript table."
        ),
    )


# ============================================================
# 14. CONSERVATION CONSOLIDATION
# ============================================================

try:
    conservation_summary_df = (
        load_required_csv(
            "conservation_summary"
        )
    )

    conservation_class_df = (
        load_required_csv(
            "conservation_class_comparison"
        )
    )

    conservation_residue_df = (
        load_required_csv(
            "conservation_residue_level"
        )
    )

    conservation_key_results_df = (
        conservation_residue_df[
            [
                "dataset",
                "minimum_model_support",
                "CPP_supported_fraction",
                "nonCPP_supported_fraction",
                "odds_ratio_corrected",
                "p_value",
            ]
        ].copy()
    )

except Exception as error:

    conservation_summary_df = pd.DataFrame()
    conservation_class_df = pd.DataFrame()
    conservation_residue_df = pd.DataFrame()
    conservation_key_results_df = pd.DataFrame()

    add_validation_issue(
        analysis="cross_PLM_conservation",
        severity="high",
        issue=(
            f"Conservation audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check hotspot conservation tables."
        ),
    )


# ============================================================
# 15. PHYSICOCHEMICAL CONSOLIDATION
# ============================================================

try:
    physicochemical_replication_df = (
        load_required_csv(
            "physicochemical_replication"
        )
    )

    physicochemical_key_results_df = (
        physicochemical_replication_df[
            [
                "property_class",
                "internal_odds_ratio",
                "internal_log2_enrichment",
                "internal_fdr",
                "kelm_odds_ratio",
                "kelm_log2_enrichment",
                "kelm_fdr",
                "same_direction",
                "significant_in_both",
            ]
        ].copy()
    )

except Exception as error:

    physicochemical_replication_df = (
        pd.DataFrame()
    )

    physicochemical_key_results_df = (
        pd.DataFrame()
    )

    add_validation_issue(
        analysis="physicochemical_enrichment",
        severity="high",
        issue=(
            f"Physicochemical audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check physicochemical replication file."
        ),
    )


# ============================================================
# 16. POSITIONAL CONSOLIDATION
# ============================================================

try:
    positional_replication_df = (
        load_required_csv(
            "positional_replication"
        )
    )

    positional_key_results_df = (
        positional_replication_df[
            [
                "region",
                "internal_odds_ratio",
                "internal_log2_enrichment",
                "internal_fdr",
                "kelm_odds_ratio",
                "kelm_log2_enrichment",
                "kelm_fdr",
                "same_direction",
                "significant_in_both",
            ]
        ].copy()
    )

except Exception as error:

    positional_replication_df = (
        pd.DataFrame()
    )

    positional_key_results_df = (
        pd.DataFrame()
    )

    add_validation_issue(
        analysis="positional_preference",
        severity="high",
        issue=(
            f"Positional audit failed: "
            f"{type(error).__name__}: {error}"
        ),
        recommendation=(
            "Check positional replication file."
        ),
    )


# ============================================================
# 17. CREATE MANUSCRIPT KEY-FINDING TABLE
# ============================================================

key_finding_rows = [
    {
        "result_id": "R1",
        "section": "Predictive performance",
        "finding": (
            "Median four-PLM ensemble produced the "
            "strongest internal-test MCC."
        ),
        "internal_evidence": (
            "MCC 0.814; ROC-AUC 0.971"
        ),
        "external_evidence": (
            "Mean ensemble ROC-AUC 0.954; "
            "PR-AUC 0.960"
        ),
        "recommended_claim_strength": (
            "Strong, but note ESM2-1280 had the "
            "best threshold-dependent KELM accuracy."
        ),
    },
    {
        "result_id": "R2",
        "section": "Attribution agreement",
        "finding": (
            "Attention agreed strongly with "
            "gradient-based methods within each PLM."
        ),
        "internal_evidence": (
            "Mean sequence Spearman approximately "
            "0.92–0.96"
        ),
        "external_evidence": (
            "Mean sequence Spearman approximately "
            "0.93–0.97"
        ),
        "recommended_claim_strength": (
            "Strong within-model agreement."
        ),
    },
    {
        "result_id": "R3",
        "section": "Attribution redundancy",
        "finding": (
            "Gradient × Input and Integrated "
            "Gradients were nearly identical."
        ),
        "internal_evidence": (
            "Spearman approximately 0.999–1.000"
        ),
        "external_evidence": (
            "Spearman approximately 0.999–1.000"
        ),
        "recommended_claim_strength": (
            "Treat as one gradient-attribution family."
        ),
    },
    {
        "result_id": "R4",
        "section": "Cross-PLM agreement",
        "finding": (
            "Different PLMs showed modest residue-rank "
            "agreement, motivating consensus XAI."
        ),
        "internal_evidence": (
            "Mean pairwise rank correlations "
            "approximately 0.10–0.28"
        ),
        "external_evidence": (
            "Mean pairwise rank correlations "
            "approximately 0.07–0.29"
        ),
        "recommended_claim_strength": (
            "Do not claim identical explanations."
        ),
    },
    {
        "result_id": "R5",
        "section": "Residue enrichment",
        "finding": (
            "Lysine and arginine were reproducibly "
            "enriched in CPP hotspots."
        ),
        "internal_evidence": (
            "K OR 3.51; R OR 3.63"
        ),
        "external_evidence": (
            "K OR 4.26; R OR 2.79"
        ),
        "recommended_claim_strength": (
            "Strong replicated cationic signal."
        ),
    },
    {
        "result_id": "R6",
        "section": "Motif discovery",
        "finding": (
            "RR and aromatic-basic motifs were "
            "reproducibly enriched."
        ),
        "internal_evidence": (
            "RR OR 25.77; WK OR 27.08"
        ),
        "external_evidence": (
            "RR OR 106.54; WK OR 31.20"
        ),
        "recommended_claim_strength": (
            "Report counts with odds ratios."
        ),
    },
    {
        "result_id": "R7",
        "section": "Faithfulness",
        "finding": (
            "Ablating consensus hotspots generally "
            "reduced CPP probabilities more than "
            "matched random ablation."
        ),
        "internal_evidence": (
            "Significant for all four PLMs"
        ),
        "external_evidence": (
            "Strongest for ProtT5; model-dependent "
            "for other PLMs"
        ),
        "recommended_claim_strength": (
            "State that external faithfulness was "
            "model dependent."
        ),
    },
    {
        "result_id": "R8",
        "section": "Cross-PLM conservation",
        "finding": (
            "Residues supported by three or four PLMs "
            "were enriched in CPPs."
        ),
        "internal_evidence": (
            "≥3 PLMs OR 1.63; all 4 OR 4.02"
        ),
        "external_evidence": (
            "≥3 PLMs OR 1.41; all 4 OR 2.26"
        ),
        "recommended_claim_strength": (
            "Strong externally replicated result."
        ),
    },
    {
        "result_id": "R9",
        "section": "Physicochemical grammar",
        "finding": (
            "CPP hotspots were strongly basic and "
            "comparatively hydrophilic."
        ),
        "internal_evidence": (
            "Basic class OR 5.96; charge increase "
            "0.395"
        ),
        "external_evidence": (
            "Basic class OR 3.87; charge increase "
            "0.287"
        ),
        "recommended_claim_strength": (
            "Strong replicated physicochemical signal."
        ),
    },
    {
        "result_id": "R10",
        "section": "Positional grammar",
        "finding": (
            "CPP hotspots were enriched centrally and "
            "depleted near the C-terminus."
        ),
        "internal_evidence": (
            "Middle OR 1.36; C-terminal OR 0.75"
        ),
        "external_evidence": (
            "Middle OR 1.53; C-terminal OR 0.57"
        ),
        "recommended_claim_strength": (
            "Strong replicated positional signal."
        ),
    },
]

key_findings_df = pd.DataFrame(
    key_finding_rows
)


# ============================================================
# 18. PROPOSED MAIN FIGURES
# ============================================================

main_figure_rows = [
    {
        "figure": "Figure 1",
        "title": (
            "Study design and consensus-XAI workflow"
        ),
        "panels": (
            "A Dataset curation and fixed splits; "
            "B four PLMs and attention classifiers; "
            "C three XAI methods; "
            "D redundancy-adjusted consensus; "
            "E external validation and biological analyses"
        ),
        "source_outputs": (
            "Dataset summary, embedding/model pipeline, "
            "consensus workflow"
        ),
        "main_message": (
            "A reproducible multi-PLM pipeline decodes "
            "and validates CPP-associated sequence grammar."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 2",
        "title": (
            "Predictive performance across PLMs and ensemble"
        ),
        "panels": (
            "A Internal ROC/PR curves; "
            "B KELM ROC/PR curves; "
            "C MCC/accuracy comparison; "
            "D confusion matrices for best models"
        ),
        "source_outputs": (
            "attention_classifier_performance.csv; "
            "individual_vs_ensemble_comparison.csv; "
            "prediction files"
        ),
        "main_message": (
            "Multiple PLMs generalize well, while ensemble "
            "benefits are metric dependent."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 3",
        "title": (
            "Consensus-XAI construction and agreement"
        ),
        "panels": (
            "A Attribution heatmap for representative CPPs; "
            "B within-model method agreement; "
            "C cross-model agreement; "
            "D original versus adjusted consensus stability"
        ),
        "source_outputs": (
            "xai_method_agreement.csv; "
            "cross_model_xai_agreement.csv; "
            "original_vs_adjusted_consensus.csv"
        ),
        "main_message": (
            "Attribution methods agree within PLMs, whereas "
            "PLMs provide complementary residue priorities."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 4",
        "title": (
            "Residue and motif grammar of CPP hotspots"
        ),
        "panels": (
            "A residue enrichment forest plot; "
            "B internal–KELM replication; "
            "C replicated motif odds ratios; "
            "D representative hotspot sequence maps"
        ),
        "source_outputs": (
            "residue_enrichment_replication.csv; "
            "strict_replicated_CPP_hotspot_motifs.csv"
        ),
        "main_message": (
            "CPP hotspots are dominated by reproducible "
            "arginine/lysine-centered local patterns."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 5",
        "title": (
            "Faithfulness of consensus hotspots"
        ),
        "panels": (
            "A hotspot versus random probability drop; "
            "B mean drop with 95% CI; "
            "C paired effect sizes; "
            "D hotspot-only sufficiency"
        ),
        "source_outputs": (
            "CPP_hotspot_faithfulness_manuscript_table.csv; "
            "faithfulness figures"
        ),
        "main_message": (
            "Consensus hotspots contribute more strongly "
            "to predictions than matched random residues."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 6",
        "title": (
            "Cross-PLM conservation of CPP hotspots"
        ),
        "panels": (
            "A pairwise Jaccard matrix; "
            "B support by 1–4 PLMs; "
            "C conservation enrichment odds ratios; "
            "D CPP versus non-CPP conservation distributions"
        ),
        "source_outputs": (
            "cross_plm_pairwise_hotspot_jaccard.csv; "
            "cross_plm_hotspot_support_distribution.csv; "
            "residue_level_cross_plm_conservation_enrichment.csv"
        ),
        "main_message": (
            "High-confidence hotspots are more reproducible "
            "across PLMs in CPPs than in non-CPPs."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 7",
        "title": (
            "Physicochemical and positional organization"
        ),
        "panels": (
            "A physicochemical class enrichment; "
            "B charge and hydropathy distributions; "
            "C N/middle/C enrichment; "
            "D normalized 10-bin positional profile"
        ),
        "source_outputs": (
            "physicochemical_category_enrichment.csv; "
            "physicochemical_sequence_level_tests.csv; "
            "hotspot_positional_region_enrichment.csv; "
            "hotspot_position_10bin_distribution.csv"
        ),
        "main_message": (
            "CPP hotspots form a basic, hydrophilic, "
            "centrally positioned sequence grammar."
        ),
        "priority": "Essential",
    },
    {
        "figure": "Figure 8",
        "title": (
            "Integrated biological model of CPP sequence grammar"
        ),
        "panels": (
            "Graphical synthesis of cationic residues, "
            "arginine-centered motifs, central positioning, "
            "cross-PLM conservation, and faithfulness"
        ),
        "source_outputs": (
            "All major biological analyses"
        ),
        "main_message": (
            "Consensus PLM-XAI reveals a conserved sequence "
            "grammar associated with CPP activity."
        ),
        "priority": "Optional graphical summary",
    },
]

main_figures_df = pd.DataFrame(
    main_figure_rows
)


# ============================================================
# 19. PROPOSED SUPPLEMENTARY FIGURES
# ============================================================

si_figure_rows = [
    {
        "figure": "Figure S1",
        "content": (
            "Dataset length distributions, class balance, "
            "and split validation."
        ),
    },
    {
        "figure": "Figure S2",
        "content": (
            "Training histories for all four PLMs."
        ),
    },
    {
        "figure": "Figure S3",
        "content": (
            "ROC and PR curves for every model and dataset."
        ),
    },
    {
        "figure": "Figure S4",
        "content": (
            "Threshold scans and default versus optimized "
            "threshold comparison."
        ),
    },
    {
        "figure": "Figure S5",
        "content": (
            "Representative attention, Gradient × Input, "
            "and Integrated Gradients residue maps."
        ),
    },
    {
        "figure": "Figure S6",
        "content": (
            "Method-agreement distributions for each PLM."
        ),
    },
    {
        "figure": "Figure S7",
        "content": (
            "Cross-model agreement distributions."
        ),
    },
    {
        "figure": "Figure S8",
        "content": (
            "Original versus redundancy-adjusted consensus."
        ),
    },
    {
        "figure": "Figure S9",
        "content": (
            "Residue enrichment at top 10%, 15%, 20%, "
            "strict, and unanimous thresholds."
        ),
    },
    {
        "figure": "Figure S10",
        "content": (
            "Complete motif enrichment by motif length."
        ),
    },
    {
        "figure": "Figure S11",
        "content": (
            "Faithfulness per-sequence distributions."
        ),
    },
    {
        "figure": "Figure S12",
        "content": (
            "Hotspot-only sufficiency scatter plots."
        ),
    },
    {
        "figure": "Figure S13",
        "content": (
            "All pairwise PLM hotspot Jaccard matrices."
        ),
    },
    {
        "figure": "Figure S14",
        "content": (
            "Sequence-level physicochemical property boxplots."
        ),
    },
    {
        "figure": "Figure S15",
        "content": (
            "Detailed positional density and uniformity tests."
        ),
    },
]

si_figures_df = pd.DataFrame(
    si_figure_rows
)


# ============================================================
# 20. PROPOSED TABLES
# ============================================================

main_table_rows = [
    {
        "table": "Table 1",
        "title": (
            "Datasets, class composition, and fixed splits"
        ),
        "source": (
            "Dataset summary and QC files"
        ),
    },
    {
        "table": "Table 2",
        "title": (
            "Internal and external predictive performance"
        ),
        "source": (
            "individual_vs_ensemble_comparison.csv"
        ),
    },
    {
        "table": "Table 3",
        "title": (
            "Replicated residue and motif determinants"
        ),
        "source": (
            "residue_enrichment_replication.csv and "
            "strict_replicated_CPP_hotspot_motifs.csv"
        ),
    },
    {
        "table": "Table 4",
        "title": (
            "Faithfulness of consensus hotspots"
        ),
        "source": (
            "CPP_hotspot_faithfulness_manuscript_table.csv"
        ),
    },
    {
        "table": "Table 5",
        "title": (
            "Cross-PLM conservation, physicochemical, "
            "and positional replication"
        ),
        "source": (
            "Conservation, physicochemical, and positional "
            "replication tables"
        ),
    },
]

main_tables_df = pd.DataFrame(
    main_table_rows
)


si_table_rows = [
    {
        "table": "Table S1",
        "content": "Complete dataset QC and rejected sequences",
    },
    {
        "table": "Table S2",
        "content": "Model hyperparameters and training settings",
    },
    {
        "table": "Table S3",
        "content": "Complete individual model metrics",
    },
    {
        "table": "Table S4",
        "content": "All prediction-level outputs",
    },
    {
        "table": "Table S5",
        "content": "XAI method agreement",
    },
    {
        "table": "Table S6",
        "content": "Cross-model XAI agreement",
    },
    {
        "table": "Table S7",
        "content": "All residue enrichment thresholds",
    },
    {
        "table": "Table S8",
        "content": "Complete motif enrichment results",
    },
    {
        "table": "Table S9",
        "content": "Per-sequence faithfulness results",
    },
    {
        "table": "Table S10",
        "content": "Per-sequence hotspot conservation",
    },
    {
        "table": "Table S11",
        "content": "Sequence-level physicochemical properties",
    },
    {
        "table": "Table S12",
        "content": "Sequence-level positional metrics",
    },
]

si_tables_df = pd.DataFrame(
    si_table_rows
)


# ============================================================
# 21. PROPOSED RESULTS SECTION STRUCTURE
# ============================================================

results_section_rows = [
    {
        "section_number": "3.1",
        "section_title": (
            "Dataset curation and leakage-free evaluation design"
        ),
        "core_content": (
            "Dataset sizes, class balance, sequence-length "
            "range, fixed train/validation/test splits, and "
            "independent KELM dataset."
        ),
    },
    {
        "section_number": "3.2",
        "section_title": (
            "Predictive performance of four PLM classifiers"
        ),
        "core_content": (
            "Individual internal and external metrics, "
            "threshold selection, and performance trade-offs."
        ),
    },
    {
        "section_number": "3.3",
        "section_title": (
            "Four-PLM ensemble performance"
        ),
        "core_content": (
            "Median ensemble improves internal MCC; mean "
            "ensemble improves external ranking metrics."
        ),
    },
    {
        "section_number": "3.4",
        "section_title": (
            "Residue-level attribution and consensus-XAI design"
        ),
        "core_content": (
            "Attention, Gradient × Input, IG, method "
            "agreement, redundancy correction, and "
            "cross-model complementarity."
        ),
    },
    {
        "section_number": "3.5",
        "section_title": (
            "Consensus hotspots encode reproducible residue "
            "and motif determinants"
        ),
        "core_content": (
            "K/R enrichment, depleted residues, RR and "
            "aromatic-basic motif replication."
        ),
    },
    {
        "section_number": "3.6",
        "section_title": (
            "Perturbation analysis validates hotspot faithfulness"
        ),
        "core_content": (
            "Hotspot ablation, matched random controls, "
            "effect sizes, and hotspot-only sufficiency."
        ),
    },
    {
        "section_number": "3.7",
        "section_title": (
            "CPP hotspots exhibit enhanced cross-PLM conservation"
        ),
        "core_content": (
            "Jaccard agreement, support by multiple PLMs, "
            "and CPP-versus-non-CPP conservation."
        ),
    },
    {
        "section_number": "3.8",
        "section_title": (
            "CPP hotspots possess a conserved "
            "physicochemical grammar"
        ),
        "core_content": (
            "Basic charge enrichment, hydrophilicity, and "
            "depletion of hydrophobic and structure-special "
            "residues."
        ),
    },
    {
        "section_number": "3.9",
        "section_title": (
            "CPP hotspots show reproducible positional organization"
        ),
        "core_content": (
            "Central enrichment, C-terminal depletion, and "
            "detailed 10-bin profile."
        ),
    },
    {
        "section_number": "3.10",
        "section_title": (
            "Integrated sequence grammar associated with CPP activity"
        ),
        "core_content": (
            "Synthesis of cationic composition, motif context, "
            "position, conservation, and model faithfulness."
        ),
    },
]

results_sections_df = pd.DataFrame(
    results_section_rows
)


# ============================================================
# 22. EXISTING FIGURE INVENTORY
# ============================================================

figure_inventory_rows = []

for figure_category, figure_directory in [
    ("main", FIGURE_MAIN_DIR),
    ("SI", FIGURE_SI_DIR),
]:

    for file_path in sorted(
        figure_directory.glob("*")
    ):

        if file_path.suffix.lower() not in [
            ".png",
            ".pdf",
            ".svg",
            ".jpg",
            ".jpeg",
        ]:
            continue

        figure_inventory_rows.append({
            "category": figure_category,
            "filename": file_path.name,
            "path": str(file_path),
            "extension": (
                file_path.suffix.lower()
            ),
            "size_MB": float(
                file_path.stat().st_size
                / (1024 ** 2)
            ),
        })


figure_inventory_df = pd.DataFrame(
    figure_inventory_rows
)


# ============================================================
# 23. TABLE INVENTORY
# ============================================================

table_inventory_rows = []

for table_category, table_directory in [
    ("main", TABLE_MAIN_DIR),
    ("SI", TABLE_SI_DIR),
]:

    for file_path in sorted(
        table_directory.glob("*.csv")
    ):

        try:
            dataframe = pd.read_csv(
                file_path
            )

            rows = len(
                dataframe
            )

            columns = len(
                dataframe.columns
            )

            duplicate_rows = int(
                dataframe.duplicated().sum()
            )

        except Exception:
            rows = None
            columns = None
            duplicate_rows = None

        table_inventory_rows.append({
            "category": table_category,
            "filename": file_path.name,
            "path": str(file_path),
            "rows": rows,
            "columns": columns,
            "duplicate_rows": (
                duplicate_rows
            ),
            "size_MB": float(
                file_path.stat().st_size
                / (1024 ** 2)
            ),
        })


table_inventory_df = pd.DataFrame(
    table_inventory_rows
)


# ============================================================
# 24. FINAL AUDIT SUMMARY
# ============================================================

validation_issues_df = pd.DataFrame(
    validation_issue_rows
)

if len(validation_issues_df) == 0:
    validation_issues_df = pd.DataFrame(
        columns=[
            "analysis",
            "severity",
            "issue",
            "recommendation",
        ]
    )


critical_missing_files = file_audit_df[
    ~file_audit_df["exists"]
]

unreadable_files = file_audit_df[
    file_audit_df[
        "read_status"
    ].astype(str).str.contains(
        "error|empty",
        regex=True,
    )
]

incomplete_steps = step_audit_df[
    ~step_audit_df[
        "completed_flag"
    ]
]


audit_summary = {
    "audit_date": (
        pd.Timestamp.now().isoformat()
    ),
    "project_directory": str(
        PROJECT_DIR
    ),
    "expected_pipeline_steps": int(
        len(EXPECTED_STEPS)
    ),
    "completed_pipeline_steps": int(
        step_audit_df[
            "completed_flag"
        ].sum()
    ),
    "incomplete_pipeline_steps": (
        incomplete_steps[
            "step"
        ].tolist()
    ),
    "expected_critical_files": int(
        len(EXPECTED_FILES)
    ),
    "existing_critical_files": int(
        file_audit_df[
            "exists"
        ].sum()
    ),
    "missing_critical_files": (
        critical_missing_files[
            "file_key"
        ].tolist()
    ),
    "unreadable_or_empty_files": (
        unreadable_files[
            "file_key"
        ].tolist()
    ),
    "validation_issue_count": int(
        len(validation_issues_df)
    ),
    "high_severity_issue_count": int(
        (
            validation_issues_df[
                "severity"
            ] == "high"
        ).sum()
    ),
    "main_figures_proposed": int(
        len(main_figures_df)
    ),
    "SI_figures_proposed": int(
        len(si_figures_df)
    ),
    "main_tables_proposed": int(
        len(main_tables_df)
    ),
    "SI_tables_proposed": int(
        len(si_tables_df)
    ),
}


# ============================================================
# 25. SAVE CSV OUTPUTS
# ============================================================

csv_outputs = {
    "pipeline_step_audit.csv":
        step_audit_df,

    "critical_file_audit.csv":
        file_audit_df,

    "dataset_summary.csv":
        dataset_summary_df,

    "split_overlap_audit.csv":
        split_overlap_df,

    "performance_best_results.csv":
        performance_summary_df,

    "xai_audit_summary.csv":
        xai_audit_summary_df,

    "residue_key_results.csv":
        residue_key_results_df,

    "motif_key_results.csv":
        motif_key_results_df,

    "faithfulness_key_results.csv":
        faithfulness_key_results_df,

    "conservation_key_results.csv":
        conservation_key_results_df,

    "physicochemical_key_results.csv":
        physicochemical_key_results_df,

    "positional_key_results.csv":
        positional_key_results_df,

    "manuscript_key_findings.csv":
        key_findings_df,

    "proposed_main_figures.csv":
        main_figures_df,

    "proposed_SI_figures.csv":
        si_figures_df,

    "proposed_main_tables.csv":
        main_tables_df,

    "proposed_SI_tables.csv":
        si_tables_df,

    "results_section_plan.csv":
        results_sections_df,

    "figure_inventory.csv":
        figure_inventory_df,

    "table_inventory.csv":
        table_inventory_df,

    "validation_issues.csv":
        validation_issues_df,
}


for filename, dataframe in (
    csv_outputs.items()
):

    output_path = (
        AUDIT_DIR / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )


# ============================================================
# 26. SAVE JSON SUMMARY
# ============================================================

audit_summary_file = (
    AUDIT_DIR
    / "final_audit_summary.json"
)

with open(
    audit_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        audit_summary,
        handle,
        indent=2,
    )


# ============================================================
# 27. SAVE EXCEL WORKBOOK
# ============================================================

audit_workbook_file = (
    AUDIT_DIR
    / "pLM4CPP_XAI_final_manuscript_audit.xlsx"
)

with pd.ExcelWriter(
    audit_workbook_file,
    engine="openpyxl",
) as writer:

    step_audit_df.to_excel(
        writer,
        sheet_name="Pipeline Steps",
        index=False,
    )

    file_audit_df.to_excel(
        writer,
        sheet_name="Critical Files",
        index=False,
    )

    dataset_summary_df.to_excel(
        writer,
        sheet_name="Datasets",
        index=False,
    )

    split_overlap_df.to_excel(
        writer,
        sheet_name="Split Overlap",
        index=False,
    )

    performance_summary_df.to_excel(
        writer,
        sheet_name="Best Performance",
        index=False,
    )

    xai_audit_summary_df.to_excel(
        writer,
        sheet_name="XAI Summary",
        index=False,
    )

    residue_key_results_df.to_excel(
        writer,
        sheet_name="Residues",
        index=False,
    )

    motif_key_results_df.to_excel(
        writer,
        sheet_name="Motifs",
        index=False,
    )

    faithfulness_key_results_df.to_excel(
        writer,
        sheet_name="Faithfulness",
        index=False,
    )

    conservation_key_results_df.to_excel(
        writer,
        sheet_name="Conservation",
        index=False,
    )

    physicochemical_key_results_df.to_excel(
        writer,
        sheet_name="Physicochemical",
        index=False,
    )

    positional_key_results_df.to_excel(
        writer,
        sheet_name="Positional",
        index=False,
    )

    key_findings_df.to_excel(
        writer,
        sheet_name="Key Findings",
        index=False,
    )

    main_figures_df.to_excel(
        writer,
        sheet_name="Main Figures",
        index=False,
    )

    si_figures_df.to_excel(
        writer,
        sheet_name="SI Figures",
        index=False,
    )

    main_tables_df.to_excel(
        writer,
        sheet_name="Main Tables",
        index=False,
    )

    si_tables_df.to_excel(
        writer,
        sheet_name="SI Tables",
        index=False,
    )

    results_sections_df.to_excel(
        writer,
        sheet_name="Results Plan",
        index=False,
    )

    figure_inventory_df.to_excel(
        writer,
        sheet_name="Figure Inventory",
        index=False,
    )

    table_inventory_df.to_excel(
        writer,
        sheet_name="Table Inventory",
        index=False,
    )

    validation_issues_df.to_excel(
        writer,
        sheet_name="Issues",
        index=False,
    )


# ============================================================
# 28. FORMAT EXCEL WORKBOOK
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
)
from openpyxl.utils import (
    get_column_letter,
)

workbook = load_workbook(
    audit_workbook_file
)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7",
)

warning_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC",
)

error_fill = PatternFill(
    fill_type="solid",
    fgColor="F4CCCC",
)

for worksheet in workbook.worksheets:

    worksheet.freeze_panes = "A2"

    for cell in worksheet[1]:
        cell.font = Font(
            bold=True
        )

        cell.fill = header_fill

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for column_cells in worksheet.columns:

        maximum_length = 0

        column_letter = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

            cell_value = (
                ""
                if cell.value is None
                else str(cell.value)
            )

            maximum_length = max(
                maximum_length,
                len(cell_value),
            )

        worksheet.column_dimensions[
            column_letter
        ].width = min(
            max(
                maximum_length + 2,
                12,
            ),
            55,
        )


if "Issues" in workbook.sheetnames:

    issue_sheet = workbook[
        "Issues"
    ]

    for row in range(
        2,
        issue_sheet.max_row + 1,
    ):

        severity = str(
            issue_sheet.cell(
                row=row,
                column=2,
            ).value
        ).lower()

        fill = None

        if severity == "high":
            fill = error_fill

        elif severity in [
            "medium",
            "informational",
        ]:
            fill = warning_fill

        if fill is not None:
            for column in range(
                1,
                issue_sheet.max_column + 1,
            ):
                issue_sheet.cell(
                    row=row,
                    column=column,
                ).fill = fill


workbook.save(
    audit_workbook_file
)


# ============================================================
# 29. SAVE CHECKPOINT
# ============================================================

output_files = [
    audit_summary_file,
    audit_workbook_file,
]

output_files.extend(
    [
        AUDIT_DIR / filename
        for filename in csv_outputs
    ]
)

mark_step_complete(
    "20_final_results_audit",
    output_files=output_files,
    details=audit_summary,
)


# ============================================================
# 30. DISPLAY AUDIT RESULTS
# ============================================================

print("\n" + "=" * 80)
print("FINAL MANUSCRIPT AUDIT SUMMARY")
print("=" * 80)

for key, value in audit_summary.items():
    print(f"{key}: {value}")


print("\n" + "=" * 80)
print("INCOMPLETE PIPELINE STEPS")
print("=" * 80)

if len(incomplete_steps) == 0:
    print("None")
else:
    display(
        incomplete_steps
    )


print("\n" + "=" * 80)
print("MISSING OR UNREADABLE CRITICAL FILES")
print("=" * 80)

problem_files = file_audit_df[
    (
        ~file_audit_df["exists"]
    )
    | (
        file_audit_df[
            "read_status"
        ].isin(
            [
                "empty",
            ]
        )
    )
    | (
        file_audit_df[
            "read_status"
        ].astype(str).str.contains(
            "read_error",
            regex=False,
        )
    )
]

if len(problem_files) == 0:
    print("None")
else:
    display(
        problem_files
    )


print("\n" + "=" * 80)
print("VALIDATION ISSUES")
print("=" * 80)

if len(validation_issues_df) == 0:
    print("No issues identified.")
else:
    display(
        validation_issues_df
    )


print("\n" + "=" * 80)
print("KEY MANUSCRIPT FINDINGS")
print("=" * 80)

display(
    key_findings_df
)


print("\n" + "=" * 80)
print("PROPOSED MAIN FIGURES")
print("=" * 80)

display(
    main_figures_df
)


print("\n" + "=" * 80)
print("PROPOSED MAIN TABLES")
print("=" * 80)

display(
    main_tables_df
)


print("\nAudit workbook saved to:")
print(audit_workbook_file)

print("\nAudit directory:")
print(AUDIT_DIR)

print("\n" + "=" * 80)
print("STEP 20 COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 21: FINAL MANUSCRIPT TABLES AND FIGURE MANIFEST
#
# Creates:
# 1. Final Table 1 — datasets and splits
# 2. Final Table 2 — model performance
# 3. Final Table 3 — replicated residues and motifs
# 4. Final Table 4 — biological and faithfulness validation
# 5. Main/SI figure manifest
# 6. Final manuscript-results directory
# ============================================================

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side,
)
from openpyxl.utils import get_column_letter


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

TABLE_SI_DIR = (
    RESULTS_DIR / "tables_SI"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FIGURE_SI_DIR = (
    RESULTS_DIR / "figures_SI"
)

FINAL_DIR = (
    RESULTS_DIR / "final_manuscript_package"
)

FINAL_TABLE_DIR = (
    FINAL_DIR / "01_main_tables"
)

FINAL_SI_TABLE_DIR = (
    FINAL_DIR / "02_SI_tables"
)

FINAL_FIGURE_DIR = (
    FINAL_DIR / "03_main_figures"
)

FINAL_SI_FIGURE_DIR = (
    FINAL_DIR / "04_SI_figures"
)

FINAL_MANIFEST_DIR = (
    FINAL_DIR / "05_manifests"
)

for directory in [
    FINAL_TABLE_DIR,
    FINAL_SI_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_SI_FIGURE_DIR,
    FINAL_MANIFEST_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. HELPERS
# ============================================================

def require_file(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )

    return path


def format_p_value(value):
    if pd.isna(value):
        return ""

    value = float(value)

    if value < 0.001:
        return f"{value:.2e}"

    return f"{value:.3f}"


def format_value_ci(
    mean,
    lower,
    upper,
    decimals=3,
):
    return (
        f"{mean:.{decimals}f} "
        f"({lower:.{decimals}f}–"
        f"{upper:.{decimals}f})"
    )


def format_number(
    value,
    decimals=3,
):
    if pd.isna(value):
        return ""

    return f"{float(value):.{decimals}f}"


def save_csv_and_excel(
    dataframe,
    base_filename,
    sheet_name,
):
    csv_file = (
        FINAL_TABLE_DIR
        / f"{base_filename}.csv"
    )

    excel_file = (
        FINAL_TABLE_DIR
        / f"{base_filename}.xlsx"
    )

    dataframe.to_csv(
        csv_file,
        index=False,
    )

    with pd.ExcelWriter(
        excel_file,
        engine="openpyxl",
    ) as writer:
        dataframe.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False,
        )

    format_excel_file(
        excel_file
    )

    return csv_file, excel_file


def format_excel_file(excel_file):

    workbook = load_workbook(
        excel_file
    )

    worksheet = workbook.active

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )

    thin_border = Border(
        bottom=Side(
            style="thin",
            color="B7B7B7",
        )
    )

    worksheet.freeze_panes = "A2"

    for cell in worksheet[1]:
        cell.font = Font(
            bold=True
        )

        cell.fill = header_fill

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

        cell.border = thin_border

    for row in worksheet.iter_rows(
        min_row=2
    ):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

    for column_cells in worksheet.columns:

        maximum_length = max(
            len(
                str(cell.value)
                if cell.value is not None
                else ""
            )
            for cell in column_cells
        )

        column_letter = get_column_letter(
            column_cells[0].column
        )

        worksheet.column_dimensions[
            column_letter
        ].width = min(
            max(
                maximum_length + 2,
                12,
            ),
            42,
        )

    workbook.save(
        excel_file
    )


# ============================================================
# 3. TABLE 1 — DATASETS AND FIXED SPLITS
# ============================================================

dataset_files = {
    "Training": (
        PROJECT_DIR
        / "02_data_processed"
        / "train_split.csv"
    ),
    "Validation": (
        PROJECT_DIR
        / "02_data_processed"
        / "validation_split.csv"
    ),
    "Internal test": (
        PROJECT_DIR
        / "02_data_processed"
        / "internal_test_split.csv"
    ),
    "KELM external": (
        PROJECT_DIR
        / "02_data_processed"
        / "kelm_external_dataset_qc.csv"
    ),
}

table1_rows = []

for dataset_name, file_path in (
    dataset_files.items()
):
    dataframe = pd.read_csv(
        require_file(file_path)
    )

    table1_rows.append({
        "Dataset": dataset_name,
        "Total sequences": int(
            len(dataframe)
        ),
        "CPP, n": int(
            (
                dataframe["label"] == 1
            ).sum()
        ),
        "Non-CPP, n": int(
            (
                dataframe["label"] == 0
            ).sum()
        ),
        "CPP fraction": round(
            (
                dataframe["label"] == 1
            ).mean(),
            3,
        ),
        "Minimum length": int(
            dataframe["length"].min()
        ),
        "Maximum length": int(
            dataframe["length"].max()
        ),
        "Mean length": round(
            dataframe["length"].mean(),
            2,
        ),
        "Role": {
            "Training":
                "Classifier fitting",
            "Validation":
                "Early stopping and threshold selection",
            "Internal test":
                "Held-out internal evaluation",
            "KELM external":
                "Independent external validation",
        }[dataset_name],
    })


table1_df = pd.DataFrame(
    table1_rows
)

table1_csv, table1_excel = (
    save_csv_and_excel(
        table1_df,
        "Table_1_Datasets_and_splits",
        "Table 1",
    )
)


# ============================================================
# 4. TABLE 2 — PREDICTIVE PERFORMANCE
# ============================================================

performance_file = require_file(
    TABLE_MAIN_DIR
    / "individual_vs_ensemble_comparison.csv"
)

performance = pd.read_csv(
    performance_file
)

method_order = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
    "mean_ensemble",
    "median_ensemble",
]

dataset_order = [
    "internal_test",
    "kelm_external",
]

performance = performance[
    performance["dataset"].isin(
        dataset_order
    )
].copy()

performance["method"] = pd.Categorical(
    performance["method"],
    categories=method_order,
    ordered=True,
)

performance["dataset"] = pd.Categorical(
    performance["dataset"],
    categories=dataset_order,
    ordered=True,
)

performance = performance.sort_values(
    [
        "dataset",
        "method",
    ]
)

dataset_display = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

method_display = {
    "ESM2_320": "ESM2-320",
    "ESM2_640": "ESM2-640",
    "ESM2_1280": "ESM2-1280",
    "ProtT5": "ProtT5",
    "mean_ensemble": "Mean ensemble",
    "median_ensemble": "Median ensemble",
}

table2_rows = []

for row in performance.itertuples(
    index=False
):

    table2_rows.append({
        "Dataset": dataset_display[
            str(row.dataset)
        ],
        "Model": method_display[
            str(row.method)
        ],
        "Threshold": round(
            float(row.threshold),
            3,
        ),
        "Accuracy": round(
            float(row.accuracy),
            3,
        ),
        "Balanced accuracy": round(
            float(row.balanced_accuracy),
            3,
        ),
        "F1 score": round(
            float(row.f1),
            3,
        ),
        "MCC": round(
            float(row.mcc),
            3,
        ),
        "ROC-AUC": round(
            float(row.roc_auc),
            3,
        ),
        "PR-AUC": round(
            float(row.pr_auc),
            3,
        ),
    })


table2_df = pd.DataFrame(
    table2_rows
)

table2_csv, table2_excel = (
    save_csv_and_excel(
        table2_df,
        "Table_2_Predictive_performance",
        "Table 2",
    )
)


# ============================================================
# 5. TABLE 3A — REPLICATED RESIDUE DETERMINANTS
# ============================================================

residue_file = require_file(
    TABLE_MAIN_DIR
    / "residue_enrichment_replication.csv"
)

residue_df = pd.read_csv(
    residue_file
)

replicated_residues = residue_df[
    residue_df[
        "significant_in_both"
    ].astype(bool)
].copy()

replicated_residues[
    "Direction"
] = np.where(
    replicated_residues[
        "internal_log2_enrichment"
    ] > 0,
    "Enriched in CPP hotspots",
    "Depleted in CPP hotspots",
)

table3a_df = pd.DataFrame({
    "Residue":
        replicated_residues[
            "residue"
        ],

    "Direction":
        replicated_residues[
            "Direction"
        ],

    "Internal odds ratio":
        replicated_residues[
            "internal_odds_ratio"
        ].round(2),

    "Internal FDR":
        replicated_residues[
            "internal_fdr"
        ].apply(
            format_p_value
        ),

    "KELM odds ratio":
        replicated_residues[
            "kelm_odds_ratio"
        ].round(2),

    "KELM FDR":
        replicated_residues[
            "kelm_fdr"
        ].apply(
            format_p_value
        ),
})

table3a_df = table3a_df.sort_values(
    [
        "Direction",
        "Internal odds ratio",
    ],
    ascending=[
        True,
        False,
    ],
)

table3a_csv, table3a_excel = (
    save_csv_and_excel(
        table3a_df,
        "Table_3A_Replicated_residue_determinants",
        "Table 3A",
    )
)


# ============================================================
# 6. TABLE 3B — REPLICATED MOTIFS
# ============================================================

motif_file = require_file(
    TABLE_MAIN_DIR
    / "strict_replicated_CPP_hotspot_motifs.csv"
)

motif_df = pd.read_csv(
    motif_file
)

table3b_df = pd.DataFrame({
    "Motif":
        motif_df["motif"],

    "Length":
        motif_df[
            "motif_length"
        ].astype(int),

    "Internal CPP sequences":
        motif_df[
            "internal_cpp_count"
        ].astype(int),

    "Internal odds ratio":
        motif_df[
            "internal_odds_ratio"
        ].round(2),

    "Internal FDR":
        motif_df[
            "internal_fdr"
        ].apply(
            format_p_value
        ),

    "KELM CPP sequences":
        motif_df[
            "kelm_cpp_count"
        ].astype(int),

    "KELM odds ratio":
        motif_df[
            "kelm_odds_ratio"
        ].round(2),

    "KELM FDR":
        motif_df[
            "kelm_fdr"
        ].apply(
            format_p_value
        ),
})

table3b_df = table3b_df.sort_values(
    [
        "Internal FDR",
        "Internal odds ratio",
    ],
    ascending=[
        True,
        False,
    ],
)

table3b_csv, table3b_excel = (
    save_csv_and_excel(
        table3b_df,
        "Table_3B_Replicated_hotspot_motifs",
        "Table 3B",
    )
)


# ============================================================
# 7. TABLE 4A — FAITHFULNESS
# ============================================================

faithfulness_file = require_file(
    TABLE_MAIN_DIR
    / "CPP_hotspot_faithfulness_manuscript_table.csv"
)

faithfulness = pd.read_csv(
    faithfulness_file
)

faithfulness = faithfulness[
    faithfulness[
        "dataset"
    ].isin(
        dataset_order
    )
].copy()

faithfulness[
    "dataset"
] = pd.Categorical(
    faithfulness["dataset"],
    categories=dataset_order,
    ordered=True,
)

faithfulness[
    "model"
] = pd.Categorical(
    faithfulness["model"],
    categories=[
        "ESM2_320",
        "ESM2_640",
        "ESM2_1280",
        "ProtT5",
    ],
    ordered=True,
)

faithfulness = faithfulness.sort_values(
    [
        "dataset",
        "model",
    ]
)

table4a_rows = []

for row in faithfulness.itertuples(
    index=False
):

    table4a_rows.append({
        "Dataset": dataset_display[
            str(row.dataset)
        ],
        "Model": method_display[
            str(row.model)
        ],
        "CPPs, n": int(
            row.n
        ),
        "Hotspot probability drop, mean (95% CI)":
            format_value_ci(
                row.mean_hotspot_probability_drop,
                row.hotspot_drop_ci95_low,
                row.hotspot_drop_ci95_high,
            ),
        "Random probability drop, mean":
            round(
                row.mean_random_probability_drop,
                3,
            ),
        "Hotspot − random drop":
            round(
                row.mean_hotspot_minus_random_drop,
                3,
            ),
        "Fraction hotspot > random":
            round(
                row.fraction_hotspot_stronger_than_random,
                3,
            ),
        "Paired Cohen's dz":
            round(
                row.paired_cohens_dz,
                3,
            ),
        "Rank-biserial effect":
            round(
                row.rank_biserial_effect_size,
                3,
            ),
        "Wilcoxon p":
            format_p_value(
                row.wilcoxon_p_value
            ),
        "Hotspot-only probability":
            round(
                row.mean_hotspot_only_probability,
                3,
            ),
    })


table4a_df = pd.DataFrame(
    table4a_rows
)

table4a_csv, table4a_excel = (
    save_csv_and_excel(
        table4a_df,
        "Table_4A_Hotspot_faithfulness",
        "Table 4A",
    )
)


# ============================================================
# 8. TABLE 4B — CONSERVATION, PHYSICOCHEMICAL AND POSITIONAL
# ============================================================

conservation_file = require_file(
    TABLE_MAIN_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

conservation = pd.read_csv(
    conservation_file
)

physicochemical_file = require_file(
    TABLE_MAIN_DIR
    / "physicochemical_enrichment_replication.csv"
)

physicochemical = pd.read_csv(
    physicochemical_file
)

positional_file = require_file(
    TABLE_MAIN_DIR
    / "hotspot_positional_replication.csv"
)

positional = pd.read_csv(
    positional_file
)

table4b_rows = []

# Cross-model conservation
for row in conservation.itertuples(
    index=False
):
    if int(
        row.minimum_model_support
    ) not in [
        3,
        4,
    ]:
        continue

    support_label = (
        "Supported by ≥3 PLMs"
        if int(
            row.minimum_model_support
        ) == 3
        else "Supported by all 4 PLMs"
    )

    table4b_rows.append({
        "Analysis domain":
            "Cross-PLM conservation",
        "Feature":
            support_label,
        "Internal effect":
            (
                f"OR {row.odds_ratio_corrected:.2f}; "
                f"p={format_p_value(row.p_value)}"
            ),
        "KELM effect":
            "",
        "Replicated direction":
            "Yes",
        "Interpretation":
            (
                "Multi-PLM-supported residues are "
                "enriched in CPPs."
            ),
    })


# Physicochemical replication
for row in physicochemical.itertuples(
    index=False
):

    if not bool(
        row.significant_in_both
    ):
        continue

    direction = (
        "Enriched"
        if row.internal_log2_enrichment > 0
        else "Depleted"
    )

    table4b_rows.append({
        "Analysis domain":
            "Physicochemical grammar",
        "Feature":
            str(
                row.property_class
            ).replace(
                "_",
                " ",
            ),
        "Internal effect":
            (
                f"{direction}; OR "
                f"{row.internal_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.internal_fdr)}"
            ),
        "KELM effect":
            (
                f"{direction}; OR "
                f"{row.kelm_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.kelm_fdr)}"
            ),
        "Replicated direction":
            "Yes",
        "Interpretation":
            (
                "Basic residues are enriched; "
                "hydrophobic/aromatic/special classes "
                "are depleted."
            ),
    })


# Positional replication
for row in positional.itertuples(
    index=False
):

    if not bool(
        row.significant_in_both
    ):
        continue

    direction = (
        "Enriched"
        if row.internal_log2_enrichment > 0
        else "Depleted"
    )

    table4b_rows.append({
        "Analysis domain":
            "Positional grammar",
        "Feature":
            str(
                row.region
            ).replace(
                "_",
                " ",
            ),
        "Internal effect":
            (
                f"{direction}; OR "
                f"{row.internal_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.internal_fdr)}"
            ),
        "KELM effect":
            (
                f"{direction}; OR "
                f"{row.kelm_odds_ratio:.2f}; "
                f"FDR={format_p_value(row.kelm_fdr)}"
            ),
        "Replicated direction":
            "Yes",
        "Interpretation":
            (
                "CPP hotspots are centrally enriched "
                "and C-terminally depleted."
            ),
    })


table4b_df = pd.DataFrame(
    table4b_rows
)

table4b_csv, table4b_excel = (
    save_csv_and_excel(
        table4b_df,
        "Table_4B_Replicated_biological_validation",
        "Table 4B",
    )
)


# ============================================================
# 9. CREATE COMBINED MASTER EXCEL WORKBOOK
# ============================================================

master_workbook = (
    FINAL_TABLE_DIR
    / "pLM4CPP_XAI_Main_Tables.xlsx"
)

with pd.ExcelWriter(
    master_workbook,
    engine="openpyxl",
) as writer:

    table1_df.to_excel(
        writer,
        sheet_name="Table 1",
        index=False,
    )

    table2_df.to_excel(
        writer,
        sheet_name="Table 2",
        index=False,
    )

    table3a_df.to_excel(
        writer,
        sheet_name="Table 3A",
        index=False,
    )

    table3b_df.to_excel(
        writer,
        sheet_name="Table 3B",
        index=False,
    )

    table4a_df.to_excel(
        writer,
        sheet_name="Table 4A",
        index=False,
    )

    table4b_df.to_excel(
        writer,
        sheet_name="Table 4B",
        index=False,
    )


format_excel_file(
    master_workbook
)


# ============================================================
# 10. COPY IMPORTANT SI TABLES
# ============================================================

si_table_sources = {
    "Table_S1_dataset_QC.csv":
        PROJECT_DIR
        / "02_data_processed"
        / "internal_dataset_qc.csv",

    "Table_S2_complete_model_performance.csv":
        TABLE_MAIN_DIR
        / "attention_classifier_performance.csv",

    "Table_S3_XAI_method_agreement.csv":
        TABLE_MAIN_DIR
        / "xai_method_agreement.csv",

    "Table_S4_cross_model_XAI_agreement.csv":
        TABLE_MAIN_DIR
        / "cross_model_xai_agreement.csv",

    "Table_S5_residue_enrichment_all_thresholds.csv":
        TABLE_SI_DIR
        / "residue_enrichment_all_thresholds.csv",

    "Table_S6_complete_motif_enrichment.csv":
        TABLE_SI_DIR
        / "hotspot_motif_enrichment_all.csv",

    "Table_S7_faithfulness_all_sequences.csv":
        TABLE_SI_DIR
        / "faithfulness_all_sequences_all_models.csv",

    "Table_S8_conservation_per_sequence.csv":
        TABLE_SI_DIR
        / "cross_plm_hotspot_conservation_per_sequence.csv",

    "Table_S9_physicochemical_per_sequence.csv":
        TABLE_SI_DIR
        / "physicochemical_properties_per_sequence.csv",

    "Table_S10_positional_metrics_per_sequence.csv":
        TABLE_SI_DIR
        / "hotspot_positional_metrics_per_sequence.csv",
}

si_copy_rows = []

for output_name, source_file in (
    si_table_sources.items()
):
    source_file = Path(
        source_file
    )

    exists = source_file.exists()

    destination_file = (
        FINAL_SI_TABLE_DIR
        / output_name
    )

    if exists:
        shutil.copy2(
            source_file,
            destination_file,
        )

    si_copy_rows.append({
        "SI table": output_name,
        "source": str(
            source_file
        ),
        "copied": bool(
            exists
        ),
        "destination": str(
            destination_file
        ),
    })


si_copy_df = pd.DataFrame(
    si_copy_rows
)


# ============================================================
# 11. FIGURE MANIFEST
# ============================================================

figure_manifest_rows = [
    {
        "Figure": "Figure 1",
        "Title": (
            "Study design and multi-PLM consensus-XAI workflow"
        ),
        "Panels": (
            "A datasets and splits; B four PLMs; "
            "C attention classifiers; D attribution methods; "
            "E adjusted consensus and validation"
        ),
        "Status": "Must create",
        "Main or SI": "Main",
        "Priority": 1,
    },
    {
        "Figure": "Figure 2",
        "Title": (
            "Predictive performance across individual PLMs "
            "and ensembles"
        ),
        "Panels": (
            "A internal ROC; B internal PR; "
            "C KELM ROC; D KELM PR; "
            "E MCC comparison"
        ),
        "Status": "Must create/finalize",
        "Main or SI": "Main",
        "Priority": 2,
    },
    {
        "Figure": "Figure 3",
        "Title": (
            "Construction and stability of consensus XAI"
        ),
        "Panels": (
            "A representative residue map; "
            "B method agreement; C cross-model agreement; "
            "D adjusted consensus stability"
        ),
        "Status": "Must create",
        "Main or SI": "Main",
        "Priority": 3,
    },
    {
        "Figure": "Figure 4",
        "Title": (
            "Replicated residue and motif grammar"
        ),
        "Panels": (
            "A residue enrichment; "
            "B internal–KELM replication; "
            "C replicated motifs; "
            "D representative sequence hotspots"
        ),
        "Status": "Must create",
        "Main or SI": "Main",
        "Priority": 4,
    },
    {
        "Figure": "Figure 5",
        "Title": (
            "Faithfulness of consensus hotspots"
        ),
        "Panels": (
            "A hotspot vs random drops; "
            "B mean and 95% CI; "
            "C effect sizes; D hotspot-only sufficiency"
        ),
        "Status": "Partially available",
        "Main or SI": "Main",
        "Priority": 5,
    },
    {
        "Figure": "Figure 6",
        "Title": (
            "Cross-PLM conservation of CPP hotspots"
        ),
        "Panels": (
            "A pairwise Jaccard; "
            "B support by 1–4 PLMs; "
            "C conservation odds ratios; "
            "D CPP vs non-CPP comparison"
        ),
        "Status": "Partially available",
        "Main or SI": "Main",
        "Priority": 6,
    },
    {
        "Figure": "Figure 7",
        "Title": (
            "Physicochemical and positional organization "
            "of CPP hotspots"
        ),
        "Panels": (
            "A physicochemical enrichment; "
            "B charge/hydropathy; "
            "C N/middle/C enrichment; "
            "D 10-bin positional profile"
        ),
        "Status": "Partially available",
        "Main or SI": "Main",
        "Priority": 7,
    },
    {
        "Figure": "Graphical abstract",
        "Title": (
            "Consensus PLM-XAI reveals a conserved "
            "CPP-associated sequence grammar"
        ),
        "Panels": (
            "Four PLMs → consensus hotspots → "
            "K/R and RR motifs → central localization → "
            "faithfulness and external validation"
        ),
        "Status": "Create after main figures",
        "Main or SI": "Graphical abstract",
        "Priority": 8,
    },
]


figure_manifest_df = pd.DataFrame(
    figure_manifest_rows
)

figure_manifest_file = (
    FINAL_MANIFEST_DIR
    / "main_figure_manifest.csv"
)

figure_manifest_df.to_csv(
    figure_manifest_file,
    index=False,
)


# ============================================================
# 12. INVENTORY EXISTING FIGURES
# ============================================================

existing_figure_rows = []

for category, figure_directory in [
    ("Main", FIGURE_MAIN_DIR),
    ("SI", FIGURE_SI_DIR),
]:

    for file_path in sorted(
        figure_directory.glob("*")
    ):

        if file_path.suffix.lower() not in {
            ".png",
            ".pdf",
            ".svg",
            ".jpg",
            ".jpeg",
        }:
            continue

        existing_figure_rows.append({
            "Category": category,
            "Filename": (
                file_path.name
            ),
            "Extension": (
                file_path.suffix.lower()
            ),
            "Size MB": round(
                file_path.stat().st_size
                / (1024 ** 2),
                3,
            ),
            "Source path": str(
                file_path
            ),
        })


existing_figures_df = pd.DataFrame(
    existing_figure_rows
)

existing_figures_file = (
    FINAL_MANIFEST_DIR
    / "existing_figure_inventory.csv"
)

existing_figures_df.to_csv(
    existing_figures_file,
    index=False,
)


# ============================================================
# 13. MANUSCRIPT RESULT CLAIMS
# ============================================================

claim_rows = [
    {
        "Claim ID": "C1",
        "Claim": (
            "All four PLM classifiers achieved strong "
            "internal and external discrimination."
        ),
        "Use in": "Results and Discussion",
        "Qualification": (
            "Performance differed by metric and threshold."
        ),
    },
    {
        "Claim ID": "C2",
        "Claim": (
            "The median ensemble improved internal MCC, "
            "whereas the mean ensemble improved external "
            "probability-ranking metrics."
        ),
        "Use in": "Results",
        "Qualification": (
            "Do not claim universal ensemble superiority."
        ),
    },
    {
        "Claim ID": "C3",
        "Claim": (
            "Attention and gradient-based attributions "
            "showed strong within-model agreement."
        ),
        "Use in": "Results",
        "Qualification": (
            "Gradient × Input and IG were treated as one "
            "attribution family because of redundancy."
        ),
    },
    {
        "Claim ID": "C4",
        "Claim": (
            "Different PLMs prioritized complementary "
            "residue positions."
        ),
        "Use in": "Results and Discussion",
        "Qualification": (
            "Cross-model agreement was modest rather "
            "than absent."
        ),
    },
    {
        "Claim ID": "C5",
        "Claim": (
            "Lysine and arginine were reproducibly "
            "enriched in CPP hotspots."
        ),
        "Use in": "Abstract, Results, Discussion",
        "Qualification": (
            "Describe association with CPP activity, "
            "not a complete uptake mechanism."
        ),
    },
    {
        "Claim ID": "C6",
        "Claim": (
            "RR, WK, WR, RW, RL, and LR motifs were "
            "replicated in the external dataset."
        ),
        "Use in": "Results",
        "Qualification": (
            "Report motif counts alongside odds ratios."
        ),
    },
    {
        "Claim ID": "C7",
        "Claim": (
            "Consensus hotspot ablation reduced CPP "
            "probabilities more than matched random ablation."
        ),
        "Use in": "Results",
        "Qualification": (
            "External faithfulness was model dependent."
        ),
    },
    {
        "Claim ID": "C8",
        "Claim": (
            "Residues independently supported by three "
            "or four PLMs were enriched in CPPs."
        ),
        "Use in": "Abstract, Results",
        "Qualification": (
            "This indicates explanation reproducibility, "
            "not necessarily experimental causality."
        ),
    },
    {
        "Claim ID": "C9",
        "Claim": (
            "CPP hotspots were positively charged, "
            "basic, and comparatively hydrophilic."
        ),
        "Use in": "Abstract, Results, Discussion",
        "Qualification": (
            "Aromatic behavior depended on the comparison."
        ),
    },
    {
        "Claim ID": "C10",
        "Claim": (
            "CPP hotspots were enriched in the central "
            "sequence region and depleted near the "
            "C-terminus."
        ),
        "Use in": "Results and Discussion",
        "Qualification": (
            "No consistent overall N-terminal enrichment "
            "was detected."
        ),
    },
]


claims_df = pd.DataFrame(
    claim_rows
)

claims_file = (
    FINAL_MANIFEST_DIR
    / "approved_manuscript_claims.csv"
)

claims_df.to_csv(
    claims_file,
    index=False,
)


# ============================================================
# 14. RESULTS SECTION PLAN
# ============================================================

results_plan_rows = [
    {
        "Section": "3.1",
        "Title": (
            "Dataset curation and evaluation design"
        ),
        "Primary table/figure": (
            "Table 1; Figure 1"
        ),
    },
    {
        "Section": "3.2",
        "Title": (
            "Predictive performance of individual PLMs"
        ),
        "Primary table/figure": (
            "Table 2; Figure 2"
        ),
    },
    {
        "Section": "3.3",
        "Title": (
            "Ensemble performance and generalization"
        ),
        "Primary table/figure": (
            "Table 2; Figure 2"
        ),
    },
    {
        "Section": "3.4",
        "Title": (
            "Residue attribution and adjusted consensus XAI"
        ),
        "Primary table/figure": (
            "Figure 3"
        ),
    },
    {
        "Section": "3.5",
        "Title": (
            "Replicated residue and motif determinants"
        ),
        "Primary table/figure": (
            "Table 3A–B; Figure 4"
        ),
    },
    {
        "Section": "3.6",
        "Title": (
            "Perturbation analysis validates hotspot faithfulness"
        ),
        "Primary table/figure": (
            "Table 4A; Figure 5"
        ),
    },
    {
        "Section": "3.7",
        "Title": (
            "CPP hotspots exhibit enhanced "
            "cross-PLM conservation"
        ),
        "Primary table/figure": (
            "Table 4B; Figure 6"
        ),
    },
    {
        "Section": "3.8",
        "Title": (
            "Physicochemical organization of CPP hotspots"
        ),
        "Primary table/figure": (
            "Table 4B; Figure 7"
        ),
    },
    {
        "Section": "3.9",
        "Title": (
            "Positional organization of CPP hotspots"
        ),
        "Primary table/figure": (
            "Table 4B; Figure 7"
        ),
    },
]


results_plan_df = pd.DataFrame(
    results_plan_rows
)

results_plan_file = (
    FINAL_MANIFEST_DIR
    / "final_results_section_plan.csv"
)

results_plan_df.to_csv(
    results_plan_file,
    index=False,
)


# ============================================================
# 15. PACKAGE SUMMARY
# ============================================================

package_summary = {
    "created_at": (
        pd.Timestamp.now().isoformat()
    ),
    "main_tables": [
        str(table1_excel),
        str(table2_excel),
        str(table3a_excel),
        str(table3b_excel),
        str(table4a_excel),
        str(table4b_excel),
        str(master_workbook),
    ],
    "SI_tables_copied": int(
        si_copy_df["copied"].sum()
    ),
    "main_figures_planned": int(
        (
            figure_manifest_df[
                "Main or SI"
            ] == "Main"
        ).sum()
    ),
    "graphical_abstract_planned": True,
    "approved_claims": int(
        len(claims_df)
    ),
    "important_exclusions": [
        (
            "Do not report "
            "mean_hotspot_only_probability_fraction."
        ),
        (
            "Do not treat Gradient × Input and "
            "Integrated Gradients as independent "
            "consensus families."
        ),
        (
            "Do not claim all external faithfulness "
            "tests were significant."
        ),
    ],
}

package_summary_file = (
    FINAL_MANIFEST_DIR
    / "final_package_summary.json"
)

with open(
    package_summary_file,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        package_summary,
        handle,
        indent=2,
    )


# ============================================================
# 16. SAVE CHECKPOINT
# ============================================================

output_files = [
    table1_csv,
    table1_excel,
    table2_csv,
    table2_excel,
    table3a_csv,
    table3a_excel,
    table3b_csv,
    table3b_excel,
    table4a_csv,
    table4a_excel,
    table4b_csv,
    table4b_excel,
    master_workbook,
    figure_manifest_file,
    existing_figures_file,
    claims_file,
    results_plan_file,
    package_summary_file,
]

mark_step_complete(
    "21_final_manuscript_tables_and_manifest",
    output_files=output_files,
    details=package_summary,
)


# ============================================================
# 17. DISPLAY FINAL OUTPUTS
# ============================================================

print("\n" + "=" * 80)
print("FINAL TABLE 1 — DATASETS AND SPLITS")
print("=" * 80)

display(table1_df)


print("\n" + "=" * 80)
print("FINAL TABLE 2 — PREDICTIVE PERFORMANCE")
print("=" * 80)

display(table2_df)


print("\n" + "=" * 80)
print("FINAL TABLE 3A — REPLICATED RESIDUES")
print("=" * 80)

display(table3a_df)


print("\n" + "=" * 80)
print("FINAL TABLE 3B — REPLICATED MOTIFS")
print("=" * 80)

display(table3b_df)


print("\n" + "=" * 80)
print("FINAL TABLE 4A — FAITHFULNESS")
print("=" * 80)

display(table4a_df)


print("\n" + "=" * 80)
print("FINAL TABLE 4B — BIOLOGICAL VALIDATION")
print("=" * 80)

display(table4b_df)


print("\n" + "=" * 80)
print("FIGURE MANIFEST")
print("=" * 80)

display(
    figure_manifest_df
)


print("\n" + "=" * 80)
print("FINAL RESULTS SECTION PLAN")
print("=" * 80)

display(
    results_plan_df
)


print("\nFinal package directory:")
print(FINAL_DIR)

print("\nMaster table workbook:")
print(master_workbook)

print("\nFigure manifest:")
print(figure_manifest_file)

print("\nApproved manuscript claims:")
print(claims_file)

print("\n" + "=" * 80)
print("STEP 21 COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 22: PUBLICATION FIGURE STYLE + FIGURE 1 WORKFLOW
#
# Output:
# Figure 1 — Study design and multi-PLM consensus-XAI workflow
#
# Exports:
# PNG: 600 dpi
# PDF: vector
# SVG: vector
# ============================================================

from pathlib import Path
import shutil

import matplotlib.pyplot as plt
from matplotlib.patches import (
    FancyBboxPatch,
    FancyArrowPatch,
    Circle,
)
import matplotlib as mpl


# ============================================================
# 1. PROJECT DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

FIGURE_MAIN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. GLOBAL PUBLICATION STYLE
# ============================================================

mpl.rcParams.update({
    # Font
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "font.weight": "bold",

    # Axes
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    # Ticks
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    # Legend
    "legend.fontsize": 10,
    "legend.frameon": False,

    # Lines
    "lines.linewidth": 2.0,
    "lines.markersize": 6,

    # Saving
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,

    # PDF/SVG text remains editable
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# ============================================================
# 3. FIGURE HELPERS
# ============================================================

def add_box(
    axis,
    x,
    y,
    width,
    height,
    title,
    lines,
    title_size=12,
    body_size=9.5,
    linewidth=1.5,
):
    """
    Add a compact rounded workflow box.
    """

    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle=(
            "round,pad=0.012,"
            "rounding_size=0.018"
        ),
        linewidth=linewidth,
        edgecolor="black",
        facecolor="white",
        zorder=2,
    )

    axis.add_patch(box)

    axis.text(
        x + width / 2,
        y + height * 0.77,
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold",
        zorder=3,
    )

    body_text = "\n".join(lines)

    axis.text(
        x + width / 2,
        y + height * 0.40,
        body_text,
        ha="center",
        va="center",
        fontsize=body_size,
        fontweight="bold",
        linespacing=1.22,
        zorder=3,
    )

    return box


def add_arrow(
    axis,
    start,
    end,
    linewidth=1.8,
):
    """
    Add a compact workflow arrow.
    """

    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=14,
        linewidth=linewidth,
        color="black",
        shrinkA=2,
        shrinkB=2,
        zorder=1,
    )

    axis.add_patch(arrow)


def add_panel_label(
    axis,
    x,
    y,
    label,
):
    axis.text(
        x,
        y,
        label,
        ha="left",
        va="top",
        fontsize=16,
        fontweight="bold",
    )


def save_figure(
    figure,
    filename_stem,
):
    """
    Save publication formats in both figure directories.
    """

    output_paths = []

    for directory in [
        FIGURE_MAIN_DIR,
        FINAL_FIGURE_DIR,
    ]:
        base_path = (
            directory / filename_stem
        )

        png_file = base_path.with_suffix(
            ".png"
        )

        pdf_file = base_path.with_suffix(
            ".pdf"
        )

        svg_file = base_path.with_suffix(
            ".svg"
        )

        figure.savefig(
            png_file,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.04,
        )

        figure.savefig(
            pdf_file,
            bbox_inches="tight",
            pad_inches=0.04,
        )

        figure.savefig(
            svg_file,
            bbox_inches="tight",
            pad_inches=0.04,
        )

        output_paths.extend([
            png_file,
            pdf_file,
            svg_file,
        ])

    return output_paths


# ============================================================
# 4. CREATE FIGURE 1
# ============================================================

fig, ax = plt.subplots(
    figsize=(15.0, 7.2)
)

ax.set_xlim(
    0,
    1,
)

ax.set_ylim(
    0,
    1,
)

ax.axis(
    "off"
)


# ------------------------------------------------------------
# Main title
# ------------------------------------------------------------

ax.text(
    0.5,
    0.965,
    "Multi-PLM Consensus Explainable AI Framework "
    "for Decoding CPP-Associated Sequence Grammar",
    ha="center",
    va="top",
    fontsize=18,
    fontweight="bold",
)


# ============================================================
# PANEL A — DATASETS
# ============================================================

add_panel_label(
    ax,
    0.018,
    0.895,
    "A"
)

add_box(
    axis=ax,
    x=0.035,
    y=0.57,
    width=0.155,
    height=0.26,
    title="Dataset curation",
    lines=[
        "Internal: 5,459 peptides",
        "CPP = 1,370",
        "Non-CPP = 4,089",
        "",
        "External KELM:",
        "96 CPP + 96 non-CPP",
    ],
)

ax.text(
    0.1125,
    0.525,
    "Cleaning • QC • leakage check",
    ha="center",
    va="center",
    fontsize=9.5,
    fontweight="bold",
)


# Small split nodes
split_centers = [
    (0.065, 0.405, "Train\n3,821"),
    (0.1125, 0.405, "Validation\n819"),
    (0.160, 0.405, "Test\n819"),
]

for cx, cy, label in split_centers:

    circle = Circle(
        (cx, cy),
        radius=0.034,
        edgecolor="black",
        facecolor="white",
        linewidth=1.3,
    )

    ax.add_patch(
        circle
    )

    ax.text(
        cx,
        cy,
        label,
        ha="center",
        va="center",
        fontsize=7.8,
        fontweight="bold",
    )


# ============================================================
# PANEL B — PLM EMBEDDINGS
# ============================================================

add_panel_label(
    ax,
    0.214,
    0.895,
    "B"
)

add_box(
    axis=ax,
    x=0.23,
    y=0.57,
    width=0.155,
    height=0.26,
    title="Four protein language models",
    lines=[
        "ESM2-320",
        "ESM2-640",
        "ESM2-1280",
        "ProtT5",
        "",
        "Per-residue embeddings",
    ],
)

ax.text(
    0.3075,
    0.49,
    "Fixed embeddings saved permanently",
    ha="center",
    va="center",
    fontsize=9.2,
    fontweight="bold",
)


# ============================================================
# PANEL C — CLASSIFICATION
# ============================================================

add_panel_label(
    ax,
    0.409,
    0.895,
    "C"
)

add_box(
    axis=ax,
    x=0.425,
    y=0.57,
    width=0.155,
    height=0.26,
    title="Attention classifiers",
    lines=[
        "Masked residue attention",
        "Peptide-level classification",
        "",
        "Internal test",
        "KELM validation",
        "Four-PLM ensembles",
    ],
)

ax.text(
    0.5025,
    0.49,
    "Thresholds selected on validation set",
    ha="center",
    va="center",
    fontsize=9.2,
    fontweight="bold",
)


# ============================================================
# PANEL D — XAI AND CONSENSUS
# ============================================================

add_panel_label(
    ax,
    0.604,
    0.895,
    "D"
)

add_box(
    axis=ax,
    x=0.62,
    y=0.57,
    width=0.155,
    height=0.26,
    title="Residue-level explainability",
    lines=[
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "",
        "Rank normalization",
        "Method-level consensus",
    ],
)

ax.text(
    0.6975,
    0.49,
    "Gradient-family redundancy adjusted",
    ha="center",
    va="center",
    fontsize=9.2,
    fontweight="bold",
)


# ============================================================
# PANEL E — BIOLOGICAL VALIDATION
# ============================================================

add_panel_label(
    ax,
    0.799,
    0.895,
    "E"
)

add_box(
    axis=ax,
    x=0.815,
    y=0.57,
    width=0.155,
    height=0.26,
    title="Consensus hotspot validation",
    lines=[
        "Residue enrichment",
        "Motif discovery",
        "Faithfulness testing",
        "Cross-PLM conservation",
        "Physicochemical grammar",
        "Positional preference",
    ],
)


# ============================================================
# 5. MAIN ARROWS
# ============================================================

arrow_y = 0.70

add_arrow(
    ax,
    (0.194, arrow_y),
    (0.226, arrow_y),
)

add_arrow(
    ax,
    (0.389, arrow_y),
    (0.421, arrow_y),
)

add_arrow(
    ax,
    (0.584, arrow_y),
    (0.616, arrow_y),
)

add_arrow(
    ax,
    (0.779, arrow_y),
    (0.811, arrow_y),
)


# ============================================================
# 6. BOTTOM INTEGRATED FINDINGS
# ============================================================

ax.plot(
    [0.22, 0.95],
    [0.335, 0.335],
    linewidth=1.1,
    color="black",
)

ax.text(
    0.5,
    0.305,
    "Externally Replicated CPP-Associated Sequence Grammar",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
)


finding_boxes = [
    (
        0.225,
        "K/R enrichment",
        "Basic and\npositively charged",
    ),
    (
        0.375,
        "RR-centered motifs",
        "RR • WK • WR\nRW • RL • LR",
    ),
    (
        0.525,
        "Faithful hotspots",
        "Greater probability drop\nthan random ablation",
    ),
    (
        0.675,
        "Cross-PLM conservation",
        "Enriched support by\n3–4 PLMs",
    ),
    (
        0.825,
        "Central localization",
        "Middle enrichment\nC-terminal depletion",
    ),
]

for center_x, title, body in finding_boxes:

    add_box(
        axis=ax,
        x=center_x - 0.065,
        y=0.075,
        width=0.13,
        height=0.16,
        title=title,
        lines=[
            body
        ],
        title_size=10.5,
        body_size=8.5,
        linewidth=1.25,
    )


# Arrow from analysis to final findings
add_arrow(
    ax,
    (0.8925, 0.565),
    (0.8925, 0.35),
    linewidth=1.6,
)


# ============================================================
# 7. FINAL COMPACT LAYOUT
# ============================================================

fig.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.97,
    bottom=0.02,
)


# ============================================================
# 8. SAVE FIGURE
# ============================================================

figure_files = save_figure(
    fig,
    "Figure_1_Multi_PLM_Consensus_XAI_Workflow",
)

plt.show()
plt.close(fig)


# ============================================================
# 9. CHECKPOINT
# ============================================================

try:
    mark_step_complete(
        "22_figure_1_workflow",
        output_files=figure_files,
        details={
            "figure": "Figure 1",
            "title": (
                "Study design and multi-PLM "
                "consensus-XAI workflow"
            ),
            "formats": [
                "PNG 600 dpi",
                "PDF",
                "SVG",
            ],
            "font_family": (
                "DejaVu Sans"
            ),
            "font_style": (
                "Bold, compact, publication-ready"
            ),
        },
    )
except NameError:
    print(
        "mark_step_complete() was not found, "
        "but the figure files were saved successfully."
    )


print("\n" + "=" * 78)
print("FIGURE 1 COMPLETED")
print("=" * 78)

for file_path in figure_files:
    print(file_path)

In [ ]:
# ============================================================
# STEP 23: FIGURE 2 — PREDICTIVE PERFORMANCE
#
# Panels:
# A. Internal-test ROC curves
# B. Internal-test precision-recall curves
# C. KELM ROC curves
# D. KELM precision-recall curves
# E. MCC comparison
# F. Balanced-accuracy comparison
#
# Exports:
# PNG 600 dpi
# PDF vector
# SVG vector
# ============================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
)


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

PREDICTION_DIR = (
    PROJECT_DIR / "05_predictions"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

CHECKPOINT_DIR = (
    PROJECT_DIR / "08_checkpoints"
)

for directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. GLOBAL FIGURE STYLE
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "font.weight": "bold",

    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.2,
    "legend.frameon": False,

    "lines.linewidth": 2.0,
    "lines.markersize": 5,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
})


# ============================================================
# 3. CONFIGURATION
# ============================================================

METHOD_ORDER = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
    "mean_ensemble",
    "median_ensemble",
]

METHOD_DISPLAY = {
    "ESM2_320": "ESM2-320",
    "ESM2_640": "ESM2-640",
    "ESM2_1280": "ESM2-1280",
    "ProtT5": "ProtT5",
    "mean_ensemble": "Mean ensemble",
    "median_ensemble": "Median ensemble",
}

DATASET_DISPLAY = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

# Explicit color choices are useful here because six curves
# must remain visually distinguishable across all panels.
METHOD_COLORS = {
    "ESM2_320": "#1f77b4",
    "ESM2_640": "#ff7f0e",
    "ESM2_1280": "#2ca02c",
    "ProtT5": "#d62728",
    "mean_ensemble": "#9467bd",
    "median_ensemble": "#111111",
}

METHOD_LINESTYLES = {
    "ESM2_320": "-",
    "ESM2_640": "-",
    "ESM2_1280": "-",
    "ProtT5": "-",
    "mean_ensemble": "--",
    "median_ensemble": "-.",
}


# ============================================================
# 4. LOAD PERFORMANCE TABLE
# ============================================================

performance_file = (
    TABLE_MAIN_DIR
    / "individual_vs_ensemble_comparison.csv"
)

if not performance_file.exists():
    raise FileNotFoundError(
        f"Performance table not found:\n{performance_file}"
    )

performance_df = pd.read_csv(
    performance_file
)

required_performance_columns = {
    "method",
    "dataset",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "mcc",
    "roc_auc",
    "pr_auc",
}

missing_performance_columns = (
    required_performance_columns
    - set(performance_df.columns)
)

if missing_performance_columns:
    raise ValueError(
        "Performance table is missing columns: "
        f"{sorted(missing_performance_columns)}"
    )


# ============================================================
# 5. PREDICTION FILE DISCOVERY
# ============================================================

def normalize_method_name(text):
    """
    Infer the model/method name from a path or filename.
    """

    text_lower = str(text).lower()

    if (
        "median" in text_lower
        and "ensemble" in text_lower
    ):
        return "median_ensemble"

    if (
        "mean" in text_lower
        and "ensemble" in text_lower
    ):
        return "mean_ensemble"

    if (
        "esm2_1280" in text_lower
        or "esm2-1280" in text_lower
        or "1280" in text_lower
    ):
        return "ESM2_1280"

    if (
        "esm2_640" in text_lower
        or "esm2-640" in text_lower
        or "640" in text_lower
    ):
        return "ESM2_640"

    if (
        "esm2_320" in text_lower
        or "esm2-320" in text_lower
        or "320" in text_lower
    ):
        return "ESM2_320"

    if (
        "prott5" in text_lower
        or "prot_t5" in text_lower
        or "prot-t5" in text_lower
    ):
        return "ProtT5"

    return None


def normalize_dataset_name(text):
    """
    Infer evaluation dataset from a path or filename.
    """

    text_lower = str(text).lower()

    if (
        "kelm" in text_lower
        or "external" in text_lower
    ):
        return "kelm_external"

    if (
        "internal_test" in text_lower
        or "internal-test" in text_lower
        or "test" in text_lower
    ):
        return "internal_test"

    if "validation" in text_lower:
        return "validation"

    return None


def find_column(
    dataframe,
    candidates,
):
    """
    Find a column using exact and normalized names.
    """

    normalized_lookup = {
        str(column).lower().strip():
            column
        for column in dataframe.columns
    }

    for candidate in candidates:

        candidate_lower = (
            candidate.lower().strip()
        )

        if candidate_lower in normalized_lookup:
            return normalized_lookup[
                candidate_lower
            ]

    for column in dataframe.columns:

        normalized_column = (
            str(column)
            .lower()
            .strip()
            .replace(" ", "_")
            .replace("-", "_")
        )

        for candidate in candidates:

            normalized_candidate = (
                candidate
                .lower()
                .strip()
                .replace(" ", "_")
                .replace("-", "_")
            )

            if normalized_column == normalized_candidate:
                return column

    return None


LABEL_COLUMN_CANDIDATES = [
    "label",
    "true_label",
    "y_true",
    "target",
    "actual",
    "class",
]

PROBABILITY_COLUMN_CANDIDATES = [
    "probability",
    "predicted_probability",
    "prediction_probability",
    "cpp_probability",
    "positive_probability",
    "y_probability",
    "y_prob",
    "score",
    "ensemble_probability",
    "mean_probability",
    "median_probability",
]


def inspect_prediction_file(
    file_path,
):
    """
    Read a candidate CSV and identify true labels and
    predicted CPP probabilities.
    """

    try:
        dataframe = pd.read_csv(
            file_path
        )
    except Exception:
        return None

    if len(dataframe) == 0:
        return None

    label_column = find_column(
        dataframe,
        LABEL_COLUMN_CANDIDATES,
    )

    probability_column = find_column(
        dataframe,
        PROBABILITY_COLUMN_CANDIDATES,
    )

    if (
        label_column is None
        or probability_column is None
    ):
        return None

    labels = pd.to_numeric(
        dataframe[label_column],
        errors="coerce",
    )

    probabilities = pd.to_numeric(
        dataframe[probability_column],
        errors="coerce",
    )

    valid = (
        labels.notna()
        & probabilities.notna()
    )

    labels = (
        labels[valid]
        .astype(int)
        .to_numpy()
    )

    probabilities = (
        probabilities[valid]
        .astype(float)
        .to_numpy()
    )

    if len(labels) == 0:
        return None

    if not set(
        np.unique(labels)
    ).issubset({0, 1}):
        return None

    if (
        np.nanmin(probabilities) < -1e-6
        or np.nanmax(probabilities) > 1 + 1e-6
    ):
        return None

    return {
        "dataframe": dataframe,
        "labels": labels,
        "probabilities": probabilities,
        "label_column": label_column,
        "probability_column":
            probability_column,
    }


def discover_prediction_files():
    """
    Search prediction directories and select the most likely
    file for every method-dataset combination.
    """

    candidate_files = sorted(
        PREDICTION_DIR.rglob("*.csv")
    )

    discovered = {}

    audit_rows = []

    for file_path in candidate_files:

        method = normalize_method_name(
            file_path
        )

        dataset = normalize_dataset_name(
            file_path
        )

        inspected = inspect_prediction_file(
            file_path
        )

        if inspected is None:
            continue

        if (
            method is None
            or dataset not in {
                "internal_test",
                "kelm_external",
            }
        ):
            continue

        key = (
            method,
            dataset,
        )

        # Prefer filenames containing selected/final/prediction.
        path_text = str(
            file_path
        ).lower()

        score = 0

        for keyword, weight in [
            ("final", 5),
            ("selected", 4),
            ("prediction", 3),
            ("probability", 2),
            ("test", 1),
        ]:
            if keyword in path_text:
                score += weight

        score += min(
            len(inspected["labels"])
            / 1000,
            2,
        )

        audit_rows.append({
            "method": method,
            "dataset": dataset,
            "file": str(file_path),
            "rows": int(
                len(inspected["labels"])
            ),
            "label_column":
                inspected["label_column"],
            "probability_column":
                inspected[
                    "probability_column"
                ],
            "selection_score": float(
                score
            ),
        })

        if (
            key not in discovered
            or score
            > discovered[key]["score"]
        ):
            discovered[key] = {
                "file": file_path,
                "score": score,
                **inspected,
            }

    return (
        discovered,
        pd.DataFrame(audit_rows),
    )


prediction_data, prediction_audit_df = (
    discover_prediction_files()
)


# ============================================================
# 6. CREATE ENSEMBLE PROBABILITIES IF FILES ARE ABSENT
# ============================================================

def build_ensemble_from_individuals(
    dataset_name,
    method_name,
):
    """
    Construct ensemble probabilities from the four individual
    prediction arrays when no saved ensemble file is found.
    """

    individual_keys = [
        (
            method,
            dataset_name,
        )
        for method in [
            "ESM2_320",
            "ESM2_640",
            "ESM2_1280",
            "ProtT5",
        ]
    ]

    if not all(
        key in prediction_data
        for key in individual_keys
    ):
        return None

    label_arrays = [
        prediction_data[key]["labels"]
        for key in individual_keys
    ]

    probability_arrays = [
        prediction_data[key][
            "probabilities"
        ]
        for key in individual_keys
    ]

    lengths = {
        len(array)
        for array in label_arrays
        + probability_arrays
    }

    if len(lengths) != 1:
        raise ValueError(
            f"Prediction row mismatch for "
            f"{dataset_name} ensemble."
        )

    reference_labels = label_arrays[0]

    for label_array in label_arrays[1:]:

        if not np.array_equal(
            reference_labels,
            label_array,
        ):
            raise ValueError(
                f"Label order mismatch while creating "
                f"{dataset_name} ensemble."
            )

    probability_matrix = np.column_stack(
        probability_arrays
    )

    if method_name == "mean_ensemble":

        probabilities = probability_matrix.mean(
            axis=1
        )

    elif method_name == "median_ensemble":

        probabilities = np.median(
            probability_matrix,
            axis=1,
        )

    else:
        raise ValueError(
            f"Unknown ensemble: {method_name}"
        )

    return {
        "labels": reference_labels,
        "probabilities": probabilities,
        "file": "constructed_from_individual_predictions",
        "score": np.nan,
    }


for dataset_name in [
    "internal_test",
    "kelm_external",
]:

    for ensemble_name in [
        "mean_ensemble",
        "median_ensemble",
    ]:

        key = (
            ensemble_name,
            dataset_name,
        )

        if key not in prediction_data:

            ensemble_result = (
                build_ensemble_from_individuals(
                    dataset_name,
                    ensemble_name,
                )
            )

            if ensemble_result is not None:
                prediction_data[key] = (
                    ensemble_result
                )


# ============================================================
# 7. VERIFY REQUIRED CURVE DATA
# ============================================================

required_curve_keys = [
    (
        method,
        dataset,
    )
    for dataset in [
        "internal_test",
        "kelm_external",
    ]
    for method in METHOD_ORDER
]

missing_curve_keys = [
    key
    for key in required_curve_keys
    if key not in prediction_data
]

if missing_curve_keys:

    print("\nMissing prediction combinations:")

    for key in missing_curve_keys:
        print(" -", key)

    print(
        "\nCurve panels will show all available methods. "
        "Metric comparison panels will still include all "
        "methods from the saved performance table."
    )


# ============================================================
# 8. PANEL HELPERS
# ============================================================

def add_panel_label(
    axis,
    label,
):
    axis.text(
        -0.16,
        1.09,
        label,
        transform=axis.transAxes,
        fontsize=15,
        fontweight="bold",
        ha="left",
        va="top",
    )


def style_axis(
    axis,
):
    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        width=1.1,
        length=4,
    )

    axis.grid(
        False
    )


def plot_roc_panel(
    axis,
    dataset_name,
    panel_label,
):
    for method_name in METHOD_ORDER:

        key = (
            method_name,
            dataset_name,
        )

        if key not in prediction_data:
            continue

        labels = prediction_data[
            key
        ]["labels"]

        probabilities = prediction_data[
            key
        ]["probabilities"]

        false_positive_rate, true_positive_rate, _ = (
            roc_curve(
                labels,
                probabilities,
            )
        )

        auc_value = roc_auc_score(
            labels,
            probabilities,
        )

        linewidth = (
            2.6
            if "ensemble" in method_name
            else 1.9
        )

        axis.plot(
            false_positive_rate,
            true_positive_rate,
            color=METHOD_COLORS[
                method_name
            ],
            linestyle=METHOD_LINESTYLES[
                method_name
            ],
            linewidth=linewidth,
            label=(
                f"{METHOD_DISPLAY[method_name]} "
                f"({auc_value:.3f})"
            ),
        )

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle=":",
        linewidth=1.2,
        color="black",
    )

    axis.set_xlim(
        0,
        1,
    )

    axis.set_ylim(
        0,
        1.02,
    )

    axis.set_xlabel(
        "False-positive rate"
    )

    axis.set_ylabel(
        "True-positive rate"
    )

    axis.set_title(
        f"{DATASET_DISPLAY[dataset_name]} ROC"
    )

    axis.legend(
        title="Model (ROC-AUC)",
        title_fontsize=8.5,
        loc="lower right",
        handlelength=2.3,
        labelspacing=0.25,
    )

    add_panel_label(
        axis,
        panel_label,
    )

    style_axis(
        axis
    )


def plot_pr_panel(
    axis,
    dataset_name,
    panel_label,
):
    prevalence = None

    for method_name in METHOD_ORDER:

        key = (
            method_name,
            dataset_name,
        )

        if key not in prediction_data:
            continue

        labels = prediction_data[
            key
        ]["labels"]

        probabilities = prediction_data[
            key
        ]["probabilities"]

        precision, recall, _ = (
            precision_recall_curve(
                labels,
                probabilities,
            )
        )

        average_precision = (
            average_precision_score(
                labels,
                probabilities,
            )
        )

        prevalence = float(
            np.mean(labels)
        )

        linewidth = (
            2.6
            if "ensemble" in method_name
            else 1.9
        )

        axis.plot(
            recall,
            precision,
            color=METHOD_COLORS[
                method_name
            ],
            linestyle=METHOD_LINESTYLES[
                method_name
            ],
            linewidth=linewidth,
            label=(
                f"{METHOD_DISPLAY[method_name]} "
                f"({average_precision:.3f})"
            ),
        )

    if prevalence is not None:

        axis.axhline(
            prevalence,
            linestyle=":",
            linewidth=1.2,
            color="black",
            label=(
                f"Class prevalence "
                f"({prevalence:.3f})"
            ),
        )

    axis.set_xlim(
        0,
        1,
    )

    axis.set_ylim(
        0,
        1.02,
    )

    axis.set_xlabel(
        "Recall"
    )

    axis.set_ylabel(
        "Precision"
    )

    axis.set_title(
        f"{DATASET_DISPLAY[dataset_name]} "
        f"precision–recall"
    )

    axis.legend(
        title="Model (PR-AUC)",
        title_fontsize=8.5,
        loc="lower left",
        handlelength=2.3,
        labelspacing=0.25,
    )

    add_panel_label(
        axis,
        panel_label,
    )

    style_axis(
        axis
    )


def plot_metric_comparison(
    axis,
    metric_column,
    metric_label,
    panel_label,
):
    comparison = performance_df[
        performance_df[
            "dataset"
        ].isin(
            [
                "internal_test",
                "kelm_external",
            ]
        )
        & performance_df[
            "method"
        ].isin(
            METHOD_ORDER
        )
    ].copy()

    internal_values = []
    external_values = []

    for method_name in METHOD_ORDER:

        internal_row = comparison[
            (
                comparison["method"]
                == method_name
            )
            & (
                comparison["dataset"]
                == "internal_test"
            )
        ]

        external_row = comparison[
            (
                comparison["method"]
                == method_name
            )
            & (
                comparison["dataset"]
                == "kelm_external"
            )
        ]

        internal_values.append(
            float(
                internal_row[
                    metric_column
                ].iloc[0]
            )
            if len(internal_row) > 0
            else np.nan
        )

        external_values.append(
            float(
                external_row[
                    metric_column
                ].iloc[0]
            )
            if len(external_row) > 0
            else np.nan
        )

    x = np.arange(
        len(METHOD_ORDER)
    )

    width = 0.36

    internal_bars = axis.bar(
        x - width / 2,
        internal_values,
        width=width,
        label="Internal test",
        color="#4C78A8",
        edgecolor="black",
        linewidth=0.7,
    )

    external_bars = axis.bar(
        x + width / 2,
        external_values,
        width=width,
        label="KELM external",
        color="#F58518",
        edgecolor="black",
        linewidth=0.7,
    )

    axis.set_xticks(
        x
    )

    axis.set_xticklabels(
        [
            METHOD_DISPLAY[method]
            for method in METHOD_ORDER
        ],
        rotation=30,
        ha="right",
        fontsize=8.4,
        fontweight="bold",
    )

    axis.set_ylabel(
        metric_label
    )

    axis.set_title(
        f"{metric_label} comparison"
    )

    minimum_value = np.nanmin(
        internal_values
        + external_values
    )

    lower_limit = max(
        0,
        minimum_value - 0.12,
    )

    axis.set_ylim(
        lower_limit,
        1.02,
    )

    axis.legend(
        loc="lower left",
        fontsize=8.5,
    )

    all_bars = list(
        internal_bars
    ) + list(
        external_bars
    )

    all_values = (
        internal_values
        + external_values
    )

    for bar, value in zip(
        all_bars,
        all_values,
    ):
        if not np.isfinite(value):
            continue

        axis.text(
            bar.get_x()
            + bar.get_width() / 2,
            value + 0.012,
            f"{value:.2f}",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=7.2,
            fontweight="bold",
        )

    add_panel_label(
        axis,
        panel_label,
    )

    style_axis(
        axis
    )


# ============================================================
# 9. CREATE FIGURE 2
# ============================================================

fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(15.2, 9.0),
)

plot_roc_panel(
    axes[0, 0],
    dataset_name="internal_test",
    panel_label="A",
)

plot_pr_panel(
    axes[0, 1],
    dataset_name="internal_test",
    panel_label="B",
)

plot_metric_comparison(
    axes[0, 2],
    metric_column="mcc",
    metric_label="Matthews correlation coefficient",
    panel_label="E",
)

plot_roc_panel(
    axes[1, 0],
    dataset_name="kelm_external",
    panel_label="C",
)

plot_pr_panel(
    axes[1, 1],
    dataset_name="kelm_external",
    panel_label="D",
)

plot_metric_comparison(
    axes[1, 2],
    metric_column="balanced_accuracy",
    metric_label="Balanced accuracy",
    panel_label="F",
)


fig.suptitle(
    "Predictive Performance of Individual Protein "
    "Language Models and Four-PLM Ensembles",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)

fig.subplots_adjust(
    left=0.065,
    right=0.985,
    bottom=0.09,
    top=0.93,
    wspace=0.33,
    hspace=0.34,
)


# ============================================================
# 10. SAVE FIGURE
# ============================================================

figure_stem = (
    "Figure_2_Predictive_Performance"
)

saved_files = []

for output_directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:

    base_file = (
        output_directory
        / figure_stem
    )

    png_file = base_file.with_suffix(
        ".png"
    )

    pdf_file = base_file.with_suffix(
        ".pdf"
    )

    svg_file = base_file.with_suffix(
        ".svg"
    )

    fig.savefig(
        png_file,
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.04,
    )

    fig.savefig(
        pdf_file,
        bbox_inches="tight",
        pad_inches=0.04,
    )

    fig.savefig(
        svg_file,
        bbox_inches="tight",
        pad_inches=0.04,
    )

    saved_files.extend([
        png_file,
        pdf_file,
        svg_file,
    ])

plt.show()
plt.close(fig)


# ============================================================
# 11. SAVE PREDICTION FILE AUDIT
# ============================================================

prediction_audit_output = (
    CHECKPOINT_DIR
    / "figure_2_prediction_file_audit.csv"
)

prediction_audit_df.to_csv(
    prediction_audit_output,
    index=False,
)


selected_prediction_rows = []

for (
    method_name,
    dataset_name,
), record in sorted(
    prediction_data.items()
):

    selected_prediction_rows.append({
        "method": method_name,
        "dataset": dataset_name,
        "source_file": str(
            record.get(
                "file",
                "",
            )
        ),
        "number_of_predictions": int(
            len(
                record["labels"]
            )
        ),
        "roc_auc_recalculated": float(
            roc_auc_score(
                record["labels"],
                record[
                    "probabilities"
                ],
            )
        ),
        "pr_auc_recalculated": float(
            average_precision_score(
                record["labels"],
                record[
                    "probabilities"
                ],
            )
        ),
    })


selected_prediction_df = pd.DataFrame(
    selected_prediction_rows
)

selected_prediction_output = (
    CHECKPOINT_DIR
    / "figure_2_selected_prediction_files.csv"
)

selected_prediction_df.to_csv(
    selected_prediction_output,
    index=False,
)


# ============================================================
# 12. VERIFY RECALCULATED AUC VALUES AGAINST SAVED TABLE
# ============================================================

auc_check_rows = []

for row in selected_prediction_df.itertuples(
    index=False
):

    saved_row = performance_df[
        (
            performance_df["method"]
            == row.method
        )
        & (
            performance_df["dataset"]
            == row.dataset
        )
    ]

    if len(saved_row) == 0:
        continue

    saved_roc_auc = float(
        saved_row[
            "roc_auc"
        ].iloc[0]
    )

    saved_pr_auc = float(
        saved_row[
            "pr_auc"
        ].iloc[0]
    )

    auc_check_rows.append({
        "method": row.method,
        "dataset": row.dataset,
        "saved_ROC_AUC": saved_roc_auc,
        "recalculated_ROC_AUC":
            row.roc_auc_recalculated,
        "ROC_AUC_absolute_difference": abs(
            saved_roc_auc
            - row.roc_auc_recalculated
        ),
        "saved_PR_AUC": saved_pr_auc,
        "recalculated_PR_AUC":
            row.pr_auc_recalculated,
        "PR_AUC_absolute_difference": abs(
            saved_pr_auc
            - row.pr_auc_recalculated
        ),
    })


auc_check_df = pd.DataFrame(
    auc_check_rows
)

auc_check_output = (
    CHECKPOINT_DIR
    / "figure_2_auc_consistency_check.csv"
)

auc_check_df.to_csv(
    auc_check_output,
    index=False,
)


# ============================================================
# 13. CHECKPOINT
# ============================================================

checkpoint_details = {
    "figure": "Figure 2",
    "title": (
        "Predictive performance across individual "
        "PLMs and ensembles"
    ),
    "panels": {
        "A": "Internal-test ROC curves",
        "B": (
            "Internal-test precision-recall curves"
        ),
        "C": "KELM external ROC curves",
        "D": (
            "KELM external precision-recall curves"
        ),
        "E": "MCC comparison",
        "F": "Balanced-accuracy comparison",
    },
    "formats": [
        "PNG 600 dpi",
        "PDF",
        "SVG",
    ],
    "missing_curve_combinations": [
        f"{method}/{dataset}"
        for method, dataset
        in missing_curve_keys
    ],
}

try:
    mark_step_complete(
        "23_figure_2_predictive_performance",
        output_files=(
            saved_files
            + [
                prediction_audit_output,
                selected_prediction_output,
                auc_check_output,
            ]
        ),
        details=checkpoint_details,
    )

except NameError:
    print(
        "mark_step_complete() was not found, "
        "but all Figure 2 outputs were saved."
    )


# ============================================================
# 14. DISPLAY AUDIT TABLES
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 2 SELECTED PREDICTION FILES")
print("=" * 78)

display(
    selected_prediction_df
)


print("\n" + "=" * 78)
print("AUC CONSISTENCY CHECK")
print("=" * 78)

display(
    auc_check_df
)


print("\n" + "=" * 78)
print("FIGURE 2 COMPLETED")
print("=" * 78)

for file_path in saved_files:
    print(file_path)

In [ ]:
# ============================================================
# STEP 23B: RECOVER PREDICTIONS AND REBUILD FIGURE 2
#
# No PLM embedding generation
# No classifier retraining
#
# Uses:
# - Saved per-residue embeddings
# - Saved masks and metadata
# - Saved final attention classifiers
#
# Generates:
# - Canonical prediction CSVs
# - Mean and median ensembles
# - ROC and PR curves
# - MCC and balanced-accuracy panels
# ============================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import tensorflow as tf

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
)


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

EMBEDDING_ROOT = (
    PROJECT_DIR / "03_embeddings"
)

MODEL_ROOT = (
    PROJECT_DIR / "04_models"
)

PREDICTION_ROOT = (
    PROJECT_DIR
    / "05_predictions"
    / "figure2_canonical_predictions"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

CHECKPOINT_DIR = (
    PROJECT_DIR / "08_checkpoints"
)

for directory in [
    PREDICTION_ROOT,
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_CONFIGS = {
    "ESM2_320": {
        "batch_size": 128,
    },
    "ESM2_640": {
        "batch_size": 96,
    },
    "ESM2_1280": {
        "batch_size": 64,
    },
    "ProtT5": {
        "batch_size": 64,
    },
}

MODEL_ORDER = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

METHOD_ORDER = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
    "mean_ensemble",
    "median_ensemble",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

METHOD_DISPLAY = {
    "ESM2_320": "ESM2-320",
    "ESM2_640": "ESM2-640",
    "ESM2_1280": "ESM2-1280",
    "ProtT5": "ProtT5",
    "mean_ensemble": "Mean ensemble",
    "median_ensemble": "Median ensemble",
}

DATASET_DISPLAY = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

METHOD_COLORS = {
    "ESM2_320": "#1f77b4",
    "ESM2_640": "#ff7f0e",
    "ESM2_1280": "#2ca02c",
    "ProtT5": "#d62728",
    "mean_ensemble": "#9467bd",
    "median_ensemble": "#111111",
}

METHOD_LINESTYLES = {
    "ESM2_320": "-",
    "ESM2_640": "-",
    "ESM2_1280": "-",
    "ProtT5": "-",
    "mean_ensemble": "--",
    "median_ensemble": "-.",
}


# ============================================================
# 3. PUBLICATION FIGURE STYLE
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "font.weight": "bold",

    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.0,
    "legend.frameon": False,

    "lines.linewidth": 2.0,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})


# ============================================================
# 4. CUSTOM ATTENTION LAYER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        **kwargs,
    ):
        super().__init__(
            **kwargs
        )

        self.attention_dense = (
            tf.keras.layers.Dense(
                1,
                use_bias=True,
                name="residue_attention_score",
            )
        )

    def build(
        self,
        input_shape,
    ):
        residue_shape, _ = input_shape

        self.attention_dense.build(
            residue_shape
        )

        super().build(
            input_shape
        )

    def call(
        self,
        inputs,
    ):
        residue_features, residue_mask = (
            inputs
        )

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            + (
                1.0
                - residue_mask
            )
            * tf.cast(
                -1e4,
                logits.dtype,
            )
        )

        attention_weights = (
            tf.nn.softmax(
                masked_logits,
                axis=1,
            )
        )

        return tf.reduce_sum(
            residue_features
            * tf.expand_dims(
                attention_weights,
                axis=-1,
            ),
            axis=1,
        )

    def get_config(
        self,
    ):
        return super().get_config()


# ============================================================
# 5. FIND MODEL FILE
# ============================================================

def locate_model_file(
    model_name,
):
    preferred_files = [
        MODEL_ROOT
        / model_name
        / "final_attention_classifier.keras",

        MODEL_ROOT
        / model_name
        / "best_attention_classifier.keras",

        MODEL_ROOT
        / model_name
        / "attention_classifier.keras",
    ]

    for file_path in preferred_files:

        if file_path.exists():
            return file_path

    candidates = sorted(
        (
            MODEL_ROOT
            / model_name
        ).rglob("*.keras")
    )

    if not candidates:
        raise FileNotFoundError(
            f"No .keras model found for "
            f"{model_name} under:\n"
            f"{MODEL_ROOT / model_name}"
        )

    # Prefer names containing final or best.
    candidates = sorted(
        candidates,
        key=lambda path: (
            "final" not in path.name.lower(),
            "best" not in path.name.lower(),
            len(str(path)),
        ),
    )

    return candidates[0]


# ============================================================
# 6. FIND EMBEDDING DATASET FILES
# ============================================================

def locate_embedding_dataset(
    model_name,
    dataset_name,
):
    dataset_directory = (
        EMBEDDING_ROOT
        / model_name
        / dataset_name
    )

    if not dataset_directory.exists():
        raise FileNotFoundError(
            f"Embedding directory missing:\n"
            f"{dataset_directory}"
        )

    preferred_embedding_files = [
        dataset_directory
        / "X_per_residue.npy",

        dataset_directory
        / "embeddings.npy",

        dataset_directory
        / "residue_embeddings.npy",
    ]

    preferred_mask_files = [
        dataset_directory
        / "M_masks.npy",

        dataset_directory
        / "masks.npy",

        dataset_directory
        / "residue_masks.npy",
    ]

    preferred_metadata_files = [
        dataset_directory
        / "metadata.csv",

        dataset_directory
        / "sequences.csv",
    ]

    embedding_file = next(
        (
            path
            for path
            in preferred_embedding_files
            if path.exists()
        ),
        None,
    )

    mask_file = next(
        (
            path
            for path
            in preferred_mask_files
            if path.exists()
        ),
        None,
    )

    metadata_file = next(
        (
            path
            for path
            in preferred_metadata_files
            if path.exists()
        ),
        None,
    )

    if embedding_file is None:

        candidates = sorted(
            dataset_directory.rglob(
                "*.npy"
            )
        )

        embedding_candidates = [
            path
            for path in candidates
            if (
                "mask"
                not in path.name.lower()
            )
        ]

        if embedding_candidates:
            embedding_file = (
                embedding_candidates[0]
            )

    if mask_file is None:

        candidates = sorted(
            dataset_directory.rglob(
                "*.npy"
            )
        )

        mask_candidates = [
            path
            for path in candidates
            if (
                "mask"
                in path.name.lower()
            )
        ]

        if mask_candidates:
            mask_file = (
                mask_candidates[0]
            )

    if metadata_file is None:

        csv_candidates = sorted(
            dataset_directory.rglob(
                "*.csv"
            )
        )

        if csv_candidates:
            metadata_file = (
                csv_candidates[0]
            )

    missing = []

    if embedding_file is None:
        missing.append(
            "embedding .npy"
        )

    if mask_file is None:
        missing.append(
            "mask .npy"
        )

    if metadata_file is None:
        missing.append(
            "metadata .csv"
        )

    if missing:
        raise FileNotFoundError(
            f"Missing {missing} for "
            f"{model_name}/{dataset_name}\n"
            f"Directory: {dataset_directory}"
        )

    return {
        "embedding_file":
            embedding_file,
        "mask_file":
            mask_file,
        "metadata_file":
            metadata_file,
    }


# ============================================================
# 7. PREDICTION FUNCTION
# ============================================================

def predict_one_model_dataset(
    model_name,
    dataset_name,
    batch_size,
):
    output_directory = (
        PREDICTION_ROOT
        / model_name
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_file = (
        output_directory
        / f"{dataset_name}_predictions.csv"
    )

    if output_file.exists():

        existing = pd.read_csv(
            output_file
        )

        required_columns = {
            "sequence_id",
            "label",
            "probability",
        }

        if (
            required_columns
            .issubset(
                existing.columns
            )
            and len(existing) > 0
        ):
            print(
                f"[SKIP SAVED] "
                f"{model_name} | "
                f"{dataset_name}"
            )

            return existing

    print(
        "\n" + "=" * 72
    )

    print(
        f"PREDICTING: "
        f"{model_name} — "
        f"{dataset_name}"
    )

    print(
        "=" * 72
    )

    model_file = (
        locate_model_file(
            model_name
        )
    )

    dataset_files = (
        locate_embedding_dataset(
            model_name,
            dataset_name,
        )
    )

    print(
        "Model:",
        model_file
    )

    print(
        "Embeddings:",
        dataset_files[
            "embedding_file"
        ]
    )

    print(
        "Masks:",
        dataset_files[
            "mask_file"
        ]
    )

    print(
        "Metadata:",
        dataset_files[
            "metadata_file"
        ]
    )

    model = tf.keras.models.load_model(
        model_file,
        custom_objects={
            "MaskedAttentionPooling":
                MaskedAttentionPooling
        },
        compile=False,
    )

    embeddings = np.load(
        dataset_files[
            "embedding_file"
        ],
        mmap_mode="r",
    )

    masks = np.load(
        dataset_files[
            "mask_file"
        ],
        mmap_mode="r",
    )

    metadata = pd.read_csv(
        dataset_files[
            "metadata_file"
        ]
    )

    if len(embeddings) != len(metadata):
        raise ValueError(
            f"Embedding/metadata mismatch: "
            f"{len(embeddings)} vs "
            f"{len(metadata)}"
        )

    if len(masks) != len(metadata):
        raise ValueError(
            f"Mask/metadata mismatch: "
            f"{len(masks)} vs "
            f"{len(metadata)}"
        )

    probabilities = model.predict(
        {
            "residue_embeddings":
                embeddings,
            "residue_mask":
                masks,
        },
        batch_size=batch_size,
        verbose=0,
    ).reshape(-1)

    required_metadata_columns = [
        "sequence_id",
        "sequence",
        "label",
    ]

    missing_columns = [
        column
        for column
        in required_metadata_columns
        if column
        not in metadata.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Metadata missing columns: "
            f"{missing_columns}"
        )

    prediction_table = metadata[
        required_metadata_columns
    ].copy()

    prediction_table[
        "probability"
    ] = probabilities.astype(
        float
    )

    prediction_table[
        "model"
    ] = model_name

    prediction_table[
        "dataset"
    ] = dataset_name

    prediction_table.to_csv(
        output_file,
        index=False,
    )

    print(
        f"[SAVED] {output_file}"
    )

    del model
    del embeddings
    del masks
    del probabilities

    tf.keras.backend.clear_session()
    gc.collect()

    return prediction_table


# ============================================================
# 8. GENERATE INDIVIDUAL MODEL PREDICTIONS
# ============================================================

prediction_tables = {}

for model_name, config in (
    MODEL_CONFIGS.items()
):

    for dataset_name in DATASETS:

        prediction_tables[
            (
                model_name,
                dataset_name,
            )
        ] = (
            predict_one_model_dataset(
                model_name=model_name,
                dataset_name=(
                    dataset_name
                ),
                batch_size=config[
                    "batch_size"
                ],
            )
        )


# ============================================================
# 9. BUILD ENSEMBLES BY SEQUENCE ID
# ============================================================

def build_ensemble_predictions(
    dataset_name,
):
    merged = None

    for model_name in MODEL_ORDER:

        table = prediction_tables[
            (
                model_name,
                dataset_name,
            )
        ][
            [
                "sequence_id",
                "sequence",
                "label",
                "probability",
            ]
        ].copy()

        table = table.rename(
            columns={
                "probability":
                    f"probability_{model_name}"
            }
        )

        if merged is None:
            merged = table
        else:
            merged = merged.merge(
                table,
                on=[
                    "sequence_id",
                    "sequence",
                    "label",
                ],
                how="inner",
                validate="one_to_one",
            )

    if len(merged) == 0:
        raise RuntimeError(
            f"Ensemble merge produced zero rows "
            f"for {dataset_name}."
        )

    probability_columns = [
        f"probability_{model_name}"
        for model_name in MODEL_ORDER
    ]

    probability_matrix = merged[
        probability_columns
    ].to_numpy(
        dtype=float
    )

    ensemble_tables = {}

    for ensemble_name in [
        "mean_ensemble",
        "median_ensemble",
    ]:

        ensemble_table = merged[
            [
                "sequence_id",
                "sequence",
                "label",
            ]
        ].copy()

        if (
            ensemble_name
            == "mean_ensemble"
        ):
            ensemble_probability = (
                probability_matrix.mean(
                    axis=1
                )
            )

        else:
            ensemble_probability = (
                np.median(
                    probability_matrix,
                    axis=1,
                )
            )

        ensemble_table[
            "probability"
        ] = ensemble_probability

        ensemble_table[
            "model"
        ] = ensemble_name

        ensemble_table[
            "dataset"
        ] = dataset_name

        output_directory = (
            PREDICTION_ROOT
            / "ensemble"
        )

        output_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        output_file = (
            output_directory
            / (
                f"{dataset_name}_"
                f"{ensemble_name}_"
                f"predictions.csv"
            )
        )

        ensemble_table.to_csv(
            output_file,
            index=False,
        )

        ensemble_tables[
            ensemble_name
        ] = ensemble_table

        print(
            f"[SAVED ENSEMBLE] "
            f"{output_file}"
        )

    return ensemble_tables


for dataset_name in DATASETS:

    ensemble_tables = (
        build_ensemble_predictions(
            dataset_name
        )
    )

    for (
        ensemble_name,
        ensemble_table,
    ) in ensemble_tables.items():

        prediction_tables[
            (
                ensemble_name,
                dataset_name,
            )
        ] = ensemble_table


# ============================================================
# 10. LOAD FINAL PERFORMANCE TABLE
# ============================================================

performance_file = (
    TABLE_MAIN_DIR
    / "individual_vs_ensemble_comparison.csv"
)

if not performance_file.exists():
    raise FileNotFoundError(
        f"Performance file missing:\n"
        f"{performance_file}"
    )

performance_df = pd.read_csv(
    performance_file
)


# ============================================================
# 11. CHECK RECALCULATED AUCS
# ============================================================

auc_check_rows = []

for method_name in METHOD_ORDER:

    for dataset_name in DATASETS:

        table = prediction_tables[
            (
                method_name,
                dataset_name,
            )
        ]

        labels = table[
            "label"
        ].to_numpy(
            dtype=int
        )

        probabilities = table[
            "probability"
        ].to_numpy(
            dtype=float
        )

        recalculated_roc = (
            roc_auc_score(
                labels,
                probabilities,
            )
        )

        recalculated_pr = (
            average_precision_score(
                labels,
                probabilities,
            )
        )

        saved_row = performance_df[
            (
                performance_df[
                    "method"
                ] == method_name
            )
            & (
                performance_df[
                    "dataset"
                ] == dataset_name
            )
        ]

        if len(saved_row) == 1:

            saved_roc = float(
                saved_row[
                    "roc_auc"
                ].iloc[0]
            )

            saved_pr = float(
                saved_row[
                    "pr_auc"
                ].iloc[0]
            )

        else:

            saved_roc = np.nan
            saved_pr = np.nan

        auc_check_rows.append({
            "method": method_name,
            "dataset": dataset_name,
            "number_of_sequences": int(
                len(table)
            ),
            "recalculated_ROC_AUC": float(
                recalculated_roc
            ),
            "saved_ROC_AUC": saved_roc,
            "ROC_difference": (
                abs(
                    recalculated_roc
                    - saved_roc
                )
                if np.isfinite(
                    saved_roc
                )
                else np.nan
            ),
            "recalculated_PR_AUC": float(
                recalculated_pr
            ),
            "saved_PR_AUC": saved_pr,
            "PR_difference": (
                abs(
                    recalculated_pr
                    - saved_pr
                )
                if np.isfinite(
                    saved_pr
                )
                else np.nan
            ),
        })


auc_check_df = pd.DataFrame(
    auc_check_rows
)

auc_check_file = (
    CHECKPOINT_DIR
    / "figure_2_recovered_auc_check.csv"
)

auc_check_df.to_csv(
    auc_check_file,
    index=False,
)


# ============================================================
# 12. FIGURE HELPERS
# ============================================================

def style_axis(
    axis,
):
    axis.spines[
        "top"
    ].set_visible(
        False
    )

    axis.spines[
        "right"
    ].set_visible(
        False
    )

    axis.tick_params(
        width=1.1,
        length=4,
    )


def add_panel_label(
    axis,
    label,
):
    axis.text(
        -0.15,
        1.08,
        label,
        transform=axis.transAxes,
        fontsize=15,
        fontweight="bold",
        ha="left",
        va="top",
    )


def plot_roc(
    axis,
    dataset_name,
    panel_label,
):
    for method_name in METHOD_ORDER:

        table = prediction_tables[
            (
                method_name,
                dataset_name,
            )
        ]

        labels = table[
            "label"
        ].to_numpy(
            dtype=int
        )

        probabilities = table[
            "probability"
        ].to_numpy(
            dtype=float
        )

        fpr, tpr, _ = roc_curve(
            labels,
            probabilities,
        )

        auc_value = roc_auc_score(
            labels,
            probabilities,
        )

        axis.plot(
            fpr,
            tpr,
            color=METHOD_COLORS[
                method_name
            ],
            linestyle=(
                METHOD_LINESTYLES[
                    method_name
                ]
            ),
            linewidth=(
                2.6
                if "ensemble"
                in method_name
                else 1.9
            ),
            label=(
                f"{METHOD_DISPLAY[method_name]} "
                f"({auc_value:.3f})"
            ),
        )

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle=":",
        linewidth=1.1,
        color="black",
    )

    axis.set_xlim(
        0,
        1
    )

    axis.set_ylim(
        0,
        1.02
    )

    axis.set_xlabel(
        "False-positive rate"
    )

    axis.set_ylabel(
        "True-positive rate"
    )

    axis.set_title(
        f"{DATASET_DISPLAY[dataset_name]} ROC"
    )

    axis.legend(
        title="Model (ROC-AUC)",
        title_fontsize=8.2,
        loc="lower right",
        handlelength=2.2,
        labelspacing=0.22,
    )

    add_panel_label(
        axis,
        panel_label,
    )

    style_axis(
        axis
    )


def plot_pr(
    axis,
    dataset_name,
    panel_label,
):
    prevalence = None

    for method_name in METHOD_ORDER:

        table = prediction_tables[
            (
                method_name,
                dataset_name,
            )
        ]

        labels = table[
            "label"
        ].to_numpy(
            dtype=int
        )

        probabilities = table[
            "probability"
        ].to_numpy(
            dtype=float
        )

        precision, recall, _ = (
            precision_recall_curve(
                labels,
                probabilities,
            )
        )

        pr_auc = (
            average_precision_score(
                labels,
                probabilities,
            )
        )

        prevalence = float(
            labels.mean()
        )

        axis.plot(
            recall,
            precision,
            color=METHOD_COLORS[
                method_name
            ],
            linestyle=(
                METHOD_LINESTYLES[
                    method_name
                ]
            ),
            linewidth=(
                2.6
                if "ensemble"
                in method_name
                else 1.9
            ),
            label=(
                f"{METHOD_DISPLAY[method_name]} "
                f"({pr_auc:.3f})"
            ),
        )

    axis.axhline(
        prevalence,
        linestyle=":",
        linewidth=1.1,
        color="black",
        label=(
            f"Prevalence "
            f"({prevalence:.3f})"
        ),
    )

    axis.set_xlim(
        0,
        1
    )

    axis.set_ylim(
        0,
        1.02
    )

    axis.set_xlabel(
        "Recall"
    )

    axis.set_ylabel(
        "Precision"
    )

    axis.set_title(
        f"{DATASET_DISPLAY[dataset_name]} "
        f"precision–recall"
    )

    axis.legend(
        title="Model (PR-AUC)",
        title_fontsize=8.2,
        loc="lower left",
        handlelength=2.2,
        labelspacing=0.22,
    )

    add_panel_label(
        axis,
        panel_label,
    )

    style_axis(
        axis
    )


def plot_metric(
    axis,
    metric_column,
    metric_label,
    panel_label,
):
    internal_values = []
    kelm_values = []

    for method_name in METHOD_ORDER:

        internal_row = performance_df[
            (
                performance_df[
                    "method"
                ] == method_name
            )
            & (
                performance_df[
                    "dataset"
                ] == "internal_test"
            )
        ]

        kelm_row = performance_df[
            (
                performance_df[
                    "method"
                ] == method_name
            )
            & (
                performance_df[
                    "dataset"
                ] == "kelm_external"
            )
        ]

        internal_values.append(
            float(
                internal_row[
                    metric_column
                ].iloc[0]
            )
        )

        kelm_values.append(
            float(
                kelm_row[
                    metric_column
                ].iloc[0]
            )
        )

    x = np.arange(
        len(METHOD_ORDER)
    )

    width = 0.36

    bars_internal = axis.bar(
        x - width / 2,
        internal_values,
        width=width,
        label="Internal test",
        edgecolor="black",
        linewidth=0.7,
    )

    bars_external = axis.bar(
        x + width / 2,
        kelm_values,
        width=width,
        label="KELM external",
        edgecolor="black",
        linewidth=0.7,
    )

    axis.set_xticks(
        x
    )

    axis.set_xticklabels(
        [
            METHOD_DISPLAY[
                method_name
            ]
            for method_name
            in METHOD_ORDER
        ],
        rotation=30,
        ha="right",
        fontsize=8.2,
        fontweight="bold",
    )

    axis.set_ylabel(
        metric_label
    )

    axis.set_title(
        f"{metric_label} comparison"
    )

    combined_values = (
        internal_values
        + kelm_values
    )

    axis.set_ylim(
        max(
            0,
            min(combined_values)
            - 0.13,
        ),
        min(
            1.05,
            max(combined_values)
            + 0.10,
        ),
    )

    axis.legend(
        loc="lower left"
    )

    for bar, value in zip(
        list(
            bars_internal
        )
        + list(
            bars_external
        ),
        combined_values,
    ):
        axis.text(
            bar.get_x()
            + bar.get_width()
            / 2,
            value + 0.012,
            f"{value:.2f}",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=7,
            fontweight="bold",
        )

    add_panel_label(
        axis,
        panel_label,
    )

    style_axis(
        axis
    )


# ============================================================
# 13. GENERATE CORRECTED FIGURE 2
# ============================================================

fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(15.2, 9.0),
)

plot_roc(
    axes[0, 0],
    "internal_test",
    "A",
)

plot_pr(
    axes[0, 1],
    "internal_test",
    "B",
)

plot_metric(
    axes[0, 2],
    metric_column="mcc",
    metric_label="Matthews correlation coefficient",
    panel_label="E",
)

plot_roc(
    axes[1, 0],
    "kelm_external",
    "C",
)

plot_pr(
    axes[1, 1],
    "kelm_external",
    "D",
)

plot_metric(
    axes[1, 2],
    metric_column="balanced_accuracy",
    metric_label="Balanced accuracy",
    panel_label="F",
)

fig.suptitle(
    "Predictive Performance of Individual Protein "
    "Language Models and Four-PLM Ensembles",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)

fig.subplots_adjust(
    left=0.065,
    right=0.985,
    bottom=0.09,
    top=0.93,
    wspace=0.33,
    hspace=0.34,
)


# ============================================================
# 14. SAVE CORRECTED FIGURE
# ============================================================

figure_stem = (
    "Figure_2_Predictive_Performance"
)

saved_files = []

for output_directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:

    base_file = (
        output_directory
        / figure_stem
    )

    for extension in [
        ".png",
        ".pdf",
        ".svg",
    ]:

        output_file = (
            base_file.with_suffix(
                extension
            )
        )

        if extension == ".png":

            fig.savefig(
                output_file,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        else:

            fig.savefig(
                output_file,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        saved_files.append(
            output_file
        )

plt.show()
plt.close(fig)


# ============================================================
# 15. SAVE CHECKPOINT
# ============================================================

checkpoint_details = {
    "figure": "Figure 2",
    "prediction_source": (
        "Predictions regenerated from saved "
        "embeddings and saved trained classifiers"
    ),
    "PLM_embeddings_recomputed": False,
    "classifiers_retrained": False,
    "panels": {
        "A": "Internal-test ROC",
        "B": "Internal-test precision–recall",
        "C": "KELM ROC",
        "D": "KELM precision–recall",
        "E": "MCC comparison",
        "F": "Balanced-accuracy comparison",
    },
}

try:
    mark_step_complete(
        "23B_corrected_figure_2_predictions_and_performance",
        output_files=(
            saved_files
            + [
                auc_check_file,
            ]
        ),
        details=checkpoint_details,
    )

except NameError:
    print(
        "mark_step_complete() not found; "
        "the predictions and figure were still saved."
    )


# ============================================================
# 16. DISPLAY VERIFICATION
# ============================================================

print(
    "\n" + "=" * 78
)

print(
    "RECOVERED PREDICTION AUC CHECK"
)

print(
    "=" * 78
)

display(
    auc_check_df
)


print(
    "\nMaximum ROC-AUC difference:",
    auc_check_df[
        "ROC_difference"
    ].max()
)

print(
    "Maximum PR-AUC difference:",
    auc_check_df[
        "PR_difference"
    ].max()
)


print(
    "\n" + "=" * 78
)

print(
    "CORRECTED FIGURE 2 COMPLETED"
)

print(
    "=" * 78
)

for file_path in saved_files:
    print(
        file_path
    )

In [ ]:
# ============================================================
# STEP 24: FIGURE 3 — CONSENSUS XAI CONSTRUCTION AND STABILITY
#
# Panels:
# A. Representative CPP residue-attribution map
# B. Within-model XAI method agreement
# C. Cross-PLM attribution agreement
# D. Original vs redundancy-adjusted consensus stability
#
# Exports:
# PNG 600 dpi
# PDF vector
# SVG vector
# ============================================================

from pathlib import Path
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

XAI_ROOT = (
    PROJECT_DIR / "06_xai"
)

PREDICTION_ROOT = (
    PROJECT_DIR
    / "05_predictions"
    / "figure2_canonical_predictions"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

CHECKPOINT_DIR = (
    PROJECT_DIR / "08_checkpoints"
)

for directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. PUBLICATION STYLE
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "font.weight": "bold",

    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 8.5,
    "legend.frameon": False,

    "lines.linewidth": 2.0,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})


# ============================================================
# 3. CONFIGURATION
# ============================================================

REPRESENTATIVE_MODEL = "ESM2_1280"
REPRESENTATIVE_DATASET = "internal_test"

MODEL_ORDER = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_DISPLAY = {
    "ESM2_320": "ESM2-320",
    "ESM2_640": "ESM2-640",
    "ESM2_1280": "ESM2-1280",
    "ProtT5": "ProtT5",
}

DATASET_DISPLAY = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

METHOD_DISPLAY = {
    "attention": "Attention",
    "gradient_input": "Gradient × Input",
    "integrated_gradients": "Integrated Gradients",
    "adjusted_consensus": "Adjusted consensus",
}


# ============================================================
# 4. GENERAL HELPERS
# ============================================================

def add_panel_label(
    axis,
    label,
):
    axis.text(
        -0.12,
        1.08,
        label,
        transform=axis.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )


def style_axis(
    axis,
):
    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        width=1.1,
        length=4,
    )


def normalize_scores(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    finite = np.isfinite(
        values
    )

    if not finite.any():
        return np.zeros_like(
            values,
            dtype=float,
        )

    minimum = np.nanmin(
        values
    )

    maximum = np.nanmax(
        values
    )

    if np.isclose(
        maximum,
        minimum,
    ):
        return np.zeros_like(
            values,
            dtype=float,
        )

    normalized = (
        values - minimum
    ) / (
        maximum - minimum
    )

    return normalized


# ============================================================
# 5. LOAD MAIN AGREEMENT TABLES
# ============================================================

method_agreement_file = (
    TABLE_MAIN_DIR
    / "xai_method_agreement.csv"
)

cross_model_file = (
    TABLE_MAIN_DIR
    / "cross_model_xai_agreement.csv"
)

adjusted_stability_file = (
    TABLE_MAIN_DIR
    / "original_vs_adjusted_consensus.csv"
)

for required_file in [
    method_agreement_file,
    cross_model_file,
    adjusted_stability_file,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file missing:\n"
            f"{required_file}"
        )


method_agreement_df = pd.read_csv(
    method_agreement_file
)

cross_model_df = pd.read_csv(
    cross_model_file
)

adjusted_stability_df = pd.read_csv(
    adjusted_stability_file
)


# ============================================================
# 6. LOAD GLOBAL ADJUSTED CONSENSUS
# ============================================================

global_consensus_file = (
    XAI_ROOT
    / "consensus_adjusted"
    / "global"
    / REPRESENTATIVE_DATASET
    / "adjusted_global_consensus_residue_scores.csv"
)

if not global_consensus_file.exists():
    raise FileNotFoundError(
        f"Global adjusted consensus missing:\n"
        f"{global_consensus_file}"
    )

global_consensus_df = pd.read_csv(
    global_consensus_file
)


# ============================================================
# 7. FIND SAVED XAI RESIDUE FILES
# ============================================================

def path_matches_model_dataset(
    file_path,
    model_name,
    dataset_name,
):
    text = str(
        file_path
    ).lower()

    model_tokens = {
        "ESM2_320": [
            "esm2_320",
            "esm2-320",
        ],
        "ESM2_640": [
            "esm2_640",
            "esm2-640",
        ],
        "ESM2_1280": [
            "esm2_1280",
            "esm2-1280",
        ],
        "ProtT5": [
            "prott5",
            "prot_t5",
            "prot-t5",
        ],
    }[model_name]

    dataset_tokens = {
        "internal_test": [
            "internal_test",
            "internal-test",
        ],
        "kelm_external": [
            "kelm_external",
            "kelm-external",
            "kelm",
        ],
    }[dataset_name]

    return (
        any(
            token in text
            for token in model_tokens
        )
        and any(
            token in text
            for token in dataset_tokens
        )
    )


def infer_score_column(
    dataframe,
    method_name,
):
    method_keywords = {
        "attention": [
            "attention",
        ],
        "gradient_input": [
            "gradient",
            "grad",
            "input",
        ],
        "integrated_gradients": [
            "integrated",
            "ig",
            "gradient",
        ],
    }[method_name]

    excluded_keywords = [
        "position",
        "length",
        "label",
        "rank_std",
        "sequence_id",
    ]

    numeric_columns = []

    for column in dataframe.columns:

        if not pd.api.types.is_numeric_dtype(
            dataframe[column]
        ):
            continue

        column_lower = str(
            column
        ).lower()

        if any(
            keyword in column_lower
            for keyword in excluded_keywords
        ):
            continue

        numeric_columns.append(
            column
        )

    preferred_columns = []

    for column in numeric_columns:

        column_lower = str(
            column
        ).lower()

        keyword_score = sum(
            keyword in column_lower
            for keyword in method_keywords
        )

        normalized_bonus = int(
            "normalized" in column_lower
            or "norm" in column_lower
        )

        score_bonus = int(
            "score" in column_lower
            or "attribution" in column_lower
        )

        preferred_columns.append(
            (
                keyword_score
                + normalized_bonus
                + score_bonus,
                column,
            )
        )

    preferred_columns = sorted(
        preferred_columns,
        reverse=True,
    )

    if not preferred_columns:
        return None

    return preferred_columns[0][1]


def locate_method_residue_file(
    method_name,
    model_name,
    dataset_name,
):
    method_path_keywords = {
        "attention": [
            "attention",
        ],
        "gradient_input": [
            "gradient_input",
            "gradient-input",
            "gradient",
        ],
        "integrated_gradients": [
            "integrated_gradients",
            "integrated-gradients",
            "integrated",
        ],
    }[method_name]

    candidates = []

    for file_path in XAI_ROOT.rglob(
        "*.csv"
    ):

        if not path_matches_model_dataset(
            file_path,
            model_name,
            dataset_name,
        ):
            continue

        path_lower = str(
            file_path
        ).lower()

        if not any(
            keyword in path_lower
            for keyword in method_path_keywords
        ):
            continue

        try:
            dataframe = pd.read_csv(
                file_path,
                nrows=20,
            )
        except Exception:
            continue

        required_identity = {
            "sequence_id",
            "position",
        }

        if not required_identity.issubset(
            dataframe.columns
        ):
            continue

        score_column = infer_score_column(
            dataframe,
            method_name,
        )

        if score_column is None:
            continue

        score = 0

        if "residue" in dataframe.columns:
            score += 3

        if "sequence" in dataframe.columns:
            score += 2

        if "score" in str(
            score_column
        ).lower():
            score += 2

        if "normalized" in str(
            score_column
        ).lower():
            score += 2

        if method_name in path_lower:
            score += 3

        candidates.append(
            (
                score,
                file_path,
                score_column,
            )
        )

    if not candidates:
        raise FileNotFoundError(
            f"No residue-level {method_name} file found "
            f"for {model_name}/{dataset_name}"
        )

    candidates = sorted(
        candidates,
        key=lambda item: (
            item[0],
            item[1].stat().st_size,
        ),
        reverse=True,
    )

    selected_score, selected_file, score_column = (
        candidates[0]
    )

    return (
        selected_file,
        score_column,
    )


method_file_records = {}

for method_name in [
    "attention",
    "gradient_input",
    "integrated_gradients",
]:

    selected_file, score_column = (
        locate_method_residue_file(
            method_name=method_name,
            model_name=REPRESENTATIVE_MODEL,
            dataset_name=REPRESENTATIVE_DATASET,
        )
    )

    method_file_records[
        method_name
    ] = {
        "file": selected_file,
        "score_column": score_column,
    }

    print(
        f"[FOUND] {method_name}:"
    )

    print(
        f"  File  : {selected_file}"
    )

    print(
        f"  Column: {score_column}"
    )


# ============================================================
# 8. LOAD MODEL PREDICTIONS TO SELECT REPRESENTATIVE CPP
# ============================================================

prediction_tables = []

for model_name in MODEL_ORDER:

    prediction_file = (
        PREDICTION_ROOT
        / model_name
        / "internal_test_predictions.csv"
    )

    if not prediction_file.exists():
        raise FileNotFoundError(
            f"Canonical prediction file missing:\n"
            f"{prediction_file}"
        )

    prediction_table = pd.read_csv(
        prediction_file
    )[
        [
            "sequence_id",
            "sequence",
            "label",
            "probability",
        ]
    ].copy()

    prediction_table = prediction_table.rename(
        columns={
            "probability":
                f"probability_{model_name}"
        }
    )

    prediction_tables.append(
        prediction_table
    )


prediction_merged = prediction_tables[0]

for table in prediction_tables[1:]:

    prediction_merged = (
        prediction_merged.merge(
            table,
            on=[
                "sequence_id",
                "sequence",
                "label",
            ],
            how="inner",
            validate="one_to_one",
        )
    )


probability_columns = [
    f"probability_{model_name}"
    for model_name in MODEL_ORDER
]

prediction_merged[
    "mean_probability"
] = prediction_merged[
    probability_columns
].mean(axis=1)

prediction_merged[
    "minimum_probability"
] = prediction_merged[
    probability_columns
].min(axis=1)

prediction_merged[
    "sequence_length"
] = prediction_merged[
    "sequence"
].str.len()


# Count global hotspots per sequence
hotspot_summary = (
    global_consensus_df.groupby(
        "sequence_id",
        as_index=False,
    )
    .agg(
        number_of_hotspots=(
            "adjusted_hotspot_top20",
            "sum",
        ),
        strict_hotspots=(
            "strict_cross_model_hotspot",
            "sum",
        ),
        unanimous_hotspots=(
            "unanimous_cross_model_hotspot",
            "sum",
        ),
    )
)


candidate_table = (
    prediction_merged.merge(
        hotspot_summary,
        on="sequence_id",
        how="left",
    )
)


candidate_table = candidate_table[
    (
        candidate_table["label"] == 1
    )
    & (
        candidate_table[
            "minimum_probability"
        ] >= 0.70
    )
    & (
        candidate_table[
            "sequence_length"
        ].between(
            14,
            30,
        )
    )
    & (
        candidate_table[
            "strict_hotspots"
        ] >= 1
    )
].copy()


if len(candidate_table) == 0:

    candidate_table = (
        prediction_merged.merge(
            hotspot_summary,
            on="sequence_id",
            how="left",
        )
    )

    candidate_table = candidate_table[
        candidate_table["label"] == 1
    ].copy()


candidate_table[
    "selection_score"
] = (
    candidate_table[
        "mean_probability"
    ]
    + 0.03
    * candidate_table[
        "strict_hotspots"
    ]
    + 0.05
    * candidate_table[
        "unanimous_hotspots"
    ]
    - 0.003
    * np.abs(
        candidate_table[
            "sequence_length"
        ]
        - 21
    )
)


candidate_table = candidate_table.sort_values(
    "selection_score",
    ascending=False,
)


representative_row = candidate_table.iloc[
    0
]

REPRESENTATIVE_SEQUENCE_ID = str(
    representative_row[
        "sequence_id"
    ]
)

REPRESENTATIVE_SEQUENCE = str(
    representative_row[
        "sequence"
    ]
)

print("\n" + "=" * 72)
print("REPRESENTATIVE CPP")
print("=" * 72)

print(
    "Sequence ID:",
    REPRESENTATIVE_SEQUENCE_ID,
)

print(
    "Sequence:",
    REPRESENTATIVE_SEQUENCE,
)

print(
    "Length:",
    len(
        REPRESENTATIVE_SEQUENCE
    ),
)

print(
    "Mean probability:",
    representative_row[
        "mean_probability"
    ],
)

print(
    "Strict hotspots:",
    representative_row[
        "strict_hotspots"
    ],
)

print(
    "Unanimous hotspots:",
    representative_row[
        "unanimous_hotspots"
    ],
)


# ============================================================
# 9. BUILD REPRESENTATIVE RESIDUE SCORE MATRIX
# ============================================================

score_rows = []
position_reference = None
residue_reference = None

for method_name in [
    "attention",
    "gradient_input",
    "integrated_gradients",
]:

    record = method_file_records[
        method_name
    ]

    dataframe = pd.read_csv(
        record["file"]
    )

    sequence_rows = dataframe[
        dataframe[
            "sequence_id"
        ].astype(str)
        == REPRESENTATIVE_SEQUENCE_ID
    ].copy()

    if len(sequence_rows) == 0:
        raise ValueError(
            f"Representative sequence "
            f"{REPRESENTATIVE_SEQUENCE_ID} not found in "
            f"{record['file']}"
        )

    sequence_rows = sequence_rows.sort_values(
        "position"
    )

    positions = sequence_rows[
        "position"
    ].astype(int).to_numpy()

    if "residue" in sequence_rows.columns:
        residues = sequence_rows[
            "residue"
        ].astype(str).to_numpy()
    else:
        residues = np.array(
            list(
                REPRESENTATIVE_SEQUENCE
            )
        )

    scores = sequence_rows[
        record["score_column"]
    ].astype(float).to_numpy()

    scores = normalize_scores(
        scores
    )

    if position_reference is None:
        position_reference = positions
        residue_reference = residues
    else:
        if not np.array_equal(
            position_reference,
            positions,
        ):
            raise ValueError(
                f"Position mismatch for {method_name}"
            )

    score_rows.append(
        scores
    )


# Adjusted global consensus
adjusted_rows = global_consensus_df[
    global_consensus_df[
        "sequence_id"
    ].astype(str)
    == REPRESENTATIVE_SEQUENCE_ID
].copy()

adjusted_rows = adjusted_rows.sort_values(
    "position"
)

adjusted_score_column_candidates = [
    "adjusted_global_consensus_rank",
    "adjusted_global_consensus_score",
    "global_consensus_rank",
]

adjusted_score_column = next(
    (
        column
        for column
        in adjusted_score_column_candidates
        if column in adjusted_rows.columns
    ),
    None,
)

if adjusted_score_column is None:
    raise ValueError(
        "No adjusted global consensus score/rank column "
        "was found."
    )

adjusted_scores = adjusted_rows[
    adjusted_score_column
].astype(float).to_numpy()

adjusted_scores = normalize_scores(
    adjusted_scores
)

score_rows.append(
    adjusted_scores
)

score_matrix = np.vstack(
    score_rows
)

hotspot_mask = adjusted_rows[
    "adjusted_hotspot_top20"
].astype(bool).to_numpy()

strict_mask = adjusted_rows[
    "strict_cross_model_hotspot"
].astype(bool).to_numpy()

unanimous_mask = adjusted_rows[
    "unanimous_cross_model_hotspot"
].astype(bool).to_numpy()


# ============================================================
# 10. PREPARE METHOD AGREEMENT TABLE
# ============================================================

method_pair_order = [
    "Attention vs Gradient×Input",
    "Attention vs Integrated Gradients",
    "Gradient×Input vs Integrated Gradients",
]

method_pair_display = {
    "Attention vs Gradient×Input":
        "Attention vs\nGradient × Input",

    "Attention vs Integrated Gradients":
        "Attention vs\nIntegrated Gradients",

    "Gradient×Input vs Integrated Gradients":
        "Gradient × Input vs\nIntegrated Gradients",
}


method_agreement_summary = (
    method_agreement_df.groupby(
        [
            "dataset",
            "method_pair",
        ],
        as_index=False,
    )[
        "mean_sequence_spearman"
    ].mean()
)


# ============================================================
# 11. BUILD CROSS-MODEL AGREEMENT MATRICES
# ============================================================

def build_cross_model_matrix(
    dataset_name,
):
    matrix = pd.DataFrame(
        np.eye(
            len(
                MODEL_ORDER
            )
        ),
        index=MODEL_ORDER,
        columns=MODEL_ORDER,
    )

    subset = cross_model_df[
        cross_model_df[
            "dataset"
        ] == dataset_name
    ]

    for row in subset.itertuples(
        index=False
    ):

        matrix.loc[
            row.model_a,
            row.model_b,
        ] = row.mean_sequence_spearman

        matrix.loc[
            row.model_b,
            row.model_a,
        ] = row.mean_sequence_spearman

    return matrix


internal_matrix = build_cross_model_matrix(
    "internal_test"
)

kelm_matrix = build_cross_model_matrix(
    "kelm_external"
)


# ============================================================
# 12. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(15.0, 9.0)
)

grid = fig.add_gridspec(
    nrows=2,
    ncols=2,
    width_ratios=[
        1.25,
        1.0,
    ],
    height_ratios=[
        1.12,
        1.0,
    ],
    wspace=0.26,
    hspace=0.32,
)

ax_a = fig.add_subplot(
    grid[0, :]
)

ax_b = fig.add_subplot(
    grid[1, 0]
)

ax_c = fig.add_subplot(
    grid[1, 1]
)


# ============================================================
# PANEL A — REPRESENTATIVE ATTRIBUTION MAP
# ============================================================

image_a = ax_a.imshow(
    score_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1,
    cmap="viridis",
)

ax_a.set_yticks(
    np.arange(
        4
    )
)

ax_a.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus",
    ],
    fontsize=10,
    fontweight="bold",
)

ax_a.set_xticks(
    np.arange(
        len(
            residue_reference
        )
    )
)

ax_a.set_xticklabels(
    [
        f"{residue}\n{position}"
        for residue, position
        in zip(
            residue_reference,
            position_reference,
        )
    ],
    fontsize=7.5,
    fontweight="bold",
)

ax_a.tick_params(
    axis="x",
    length=0,
)

ax_a.tick_params(
    axis="y",
    length=0,
)

ax_a.set_xlabel(
    "Residue and sequence position"
)

ax_a.set_title(
    (
        f"Representative CPP: "
        f"{REPRESENTATIVE_SEQUENCE_ID}  |  "
        f"{REPRESENTATIVE_SEQUENCE}"
    ),
    pad=10,
)

# Highlight adjusted top-20% hotspot residues
for column_index in range(
    len(
        hotspot_mask
    )
):

    if hotspot_mask[
        column_index
    ]:

        rectangle = mpl.patches.Rectangle(
            (
                column_index - 0.5,
                3 - 0.5,
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.8,
        )

        ax_a.add_patch(
            rectangle
        )

    if unanimous_mask[
        column_index
    ]:

        ax_a.text(
            column_index,
            -0.72,
            "★",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
        )

    elif strict_mask[
        column_index
    ]:

        ax_a.text(
            column_index,
            -0.72,
            "●",
            ha="center",
            va="center",
            fontsize=6,
            fontweight="bold",
        )


ax_a.text(
    0.995,
    -0.19,
    "White border: adjusted top-20% hotspot   "
    "● strict cross-model hotspot   "
    "★ unanimous hotspot",
    transform=ax_a.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    fontweight="bold",
)

colorbar_a = fig.colorbar(
    image_a,
    ax=ax_a,
    fraction=0.018,
    pad=0.012,
)

colorbar_a.set_label(
    "Normalized residue importance",
    fontweight="bold",
)

add_panel_label(
    ax_a,
    "A",
)


# ============================================================
# PANEL B — METHOD AGREEMENT
# ============================================================

x = np.arange(
    len(
        method_pair_order
    )
)

width = 0.36

internal_values = []
kelm_values = []

for method_pair in method_pair_order:

    internal_row = (
        method_agreement_summary[
            (
                method_agreement_summary[
                    "dataset"
                ] == "internal_test"
            )
            & (
                method_agreement_summary[
                    "method_pair"
                ] == method_pair
            )
        ]
    )

    kelm_row = (
        method_agreement_summary[
            (
                method_agreement_summary[
                    "dataset"
                ] == "kelm_external"
            )
            & (
                method_agreement_summary[
                    "method_pair"
                ] == method_pair
            )
        ]
    )

    internal_values.append(
        float(
            internal_row[
                "mean_sequence_spearman"
            ].iloc[0]
        )
    )

    kelm_values.append(
        float(
            kelm_row[
                "mean_sequence_spearman"
            ].iloc[0]
        )
    )


bars_internal = ax_b.bar(
    x - width / 2,
    internal_values,
    width=width,
    label="Internal test",
    edgecolor="black",
    linewidth=0.7,
)

bars_kelm = ax_b.bar(
    x + width / 2,
    kelm_values,
    width=width,
    label="KELM external",
    edgecolor="black",
    linewidth=0.7,
)

ax_b.set_xticks(
    x
)

ax_b.set_xticklabels(
    [
        method_pair_display[
            pair
        ]
        for pair in method_pair_order
    ],
    fontsize=8,
    fontweight="bold",
)

ax_b.set_ylim(
    0.80,
    1.02,
)

ax_b.set_ylabel(
    "Mean sequence-level Spearman correlation"
)

ax_b.set_title(
    "Agreement among XAI methods"
)

ax_b.legend(
    loc="lower right"
)

for bar, value in zip(
    list(
        bars_internal
    )
    + list(
        bars_kelm
    ),
    internal_values
    + kelm_values,
):

    ax_b.text(
        bar.get_x()
        + bar.get_width() / 2,
        value + 0.004,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        rotation=90,
        fontsize=7,
        fontweight="bold",
    )

add_panel_label(
    ax_b,
    "B",
)

style_axis(
    ax_b
)


# ============================================================
# PANEL C — CROSS-PLM AGREEMENT + CONSENSUS STABILITY
# ============================================================

ax_c.axis(
    "off"
)

add_panel_label(
    ax_c,
    "C",
)

ax_c.set_title(
    "Cross-PLM agreement and adjusted-consensus stability",
    pad=10,
)


internal_ax = inset_axes(
    ax_c,
    width="43%",
    height="49%",
    loc="upper left",
    borderpad=1.2,
)

kelm_ax = inset_axes(
    ax_c,
    width="43%",
    height="49%",
    loc="upper right",
    borderpad=1.2,
)

stability_ax = inset_axes(
    ax_c,
    width="92%",
    height="37%",
    loc="lower center",
    borderpad=1.1,
)


for matrix_axis, matrix, title in [
    (
        internal_ax,
        internal_matrix,
        "Internal test",
    ),
    (
        kelm_ax,
        kelm_matrix,
        "KELM external",
    ),
]:

    matrix_image = matrix_axis.imshow(
        matrix.to_numpy(),
        vmin=0,
        vmax=1,
        cmap="Blues",
        aspect="equal",
    )

    matrix_axis.set_xticks(
        np.arange(
            len(
                MODEL_ORDER
            )
        )
    )

    matrix_axis.set_yticks(
        np.arange(
            len(
                MODEL_ORDER
            )
        )
    )

    matrix_axis.set_xticklabels(
        [
            MODEL_DISPLAY[
                model
            ]
            for model in MODEL_ORDER
        ],
        rotation=45,
        ha="right",
        fontsize=6.5,
        fontweight="bold",
    )

    matrix_axis.set_yticklabels(
        [
            MODEL_DISPLAY[
                model
            ]
            for model in MODEL_ORDER
        ],
        fontsize=6.5,
        fontweight="bold",
    )

    matrix_axis.set_title(
        title,
        fontsize=9,
        fontweight="bold",
        pad=3,
    )

    for row_index in range(
        len(
            MODEL_ORDER
        )
    ):

        for column_index in range(
            len(
                MODEL_ORDER
            )
        ):

            value = matrix.iloc[
                row_index,
                column_index,
            ]

            matrix_axis.text(
                column_index,
                row_index,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=6,
                fontweight="bold",
            )


# Consensus stability bars
stability_metrics = [
    "overall_rank_spearman",
    "top20_jaccard",
]

stability_labels = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard",
]

internal_stability = (
    adjusted_stability_df[
        adjusted_stability_df[
            "dataset"
        ] == "internal_test"
    ].iloc[0]
)

kelm_stability = (
    adjusted_stability_df[
        adjusted_stability_df[
            "dataset"
        ] == "kelm_external"
    ].iloc[0]
)

x_stability = np.arange(
    len(
        stability_metrics
    )
)

width_stability = 0.34

internal_stability_values = [
    float(
        internal_stability[
            metric
        ]
    )
    for metric in stability_metrics
]

kelm_stability_values = [
    float(
        kelm_stability[
            metric
        ]
    )
    for metric in stability_metrics
]


bars_1 = stability_ax.bar(
    x_stability
    - width_stability / 2,
    internal_stability_values,
    width=width_stability,
    label="Internal test",
    edgecolor="black",
    linewidth=0.6,
)

bars_2 = stability_ax.bar(
    x_stability
    + width_stability / 2,
    kelm_stability_values,
    width=width_stability,
    label="KELM external",
    edgecolor="black",
    linewidth=0.6,
)

stability_ax.set_xticks(
    x_stability
)

stability_ax.set_xticklabels(
    stability_labels,
    fontsize=7.5,
    fontweight="bold",
)

stability_ax.set_ylim(
    0.80,
    1.02,
)

stability_ax.set_ylabel(
    "Agreement",
    fontsize=8,
    fontweight="bold",
)

stability_ax.set_title(
    "Original vs redundancy-adjusted consensus",
    fontsize=9,
    fontweight="bold",
)

stability_ax.legend(
    fontsize=7,
    loc="lower left",
)

style_axis(
    stability_ax
)

for bar, value in zip(
    list(
        bars_1
    )
    + list(
        bars_2
    ),
    internal_stability_values
    + kelm_stability_values,
):

    stability_ax.text(
        bar.get_x()
        + bar.get_width() / 2,
        value + 0.005,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=6.8,
        fontweight="bold",
    )


# ============================================================
# 13. FINAL LAYOUT
# ============================================================

fig.suptitle(
    "Construction, Agreement, and Stability of "
    "Multi-PLM Consensus Explainable AI",
    fontsize=16,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    left=0.07,
    right=0.985,
    top=0.92,
    bottom=0.08,
)


# ============================================================
# 14. SAVE FIGURE
# ============================================================

figure_stem = (
    "Figure_3_Consensus_XAI_Construction_and_Stability"
)

saved_files = []

for output_directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:

    base_file = (
        output_directory
        / figure_stem
    )

    for extension in [
        ".png",
        ".pdf",
        ".svg",
    ]:

        output_file = (
            base_file.with_suffix(
                extension
            )
        )

        if extension == ".png":

            fig.savefig(
                output_file,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        else:

            fig.savefig(
                output_file,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        saved_files.append(
            output_file
        )


plt.show()
plt.close(fig)


# ============================================================
# 15. SAVE FIGURE AUDIT
# ============================================================

method_file_audit_df = pd.DataFrame([
    {
        "method": method_name,
        "model": REPRESENTATIVE_MODEL,
        "dataset": REPRESENTATIVE_DATASET,
        "file": str(
            record["file"]
        ),
        "score_column": (
            record["score_column"]
        ),
    }
    for method_name, record
    in method_file_records.items()
])


method_file_audit = (
    CHECKPOINT_DIR
    / "figure_3_method_file_audit.csv"
)

method_file_audit_df.to_csv(
    method_file_audit,
    index=False,
)


representative_sequence_file = (
    CHECKPOINT_DIR
    / "figure_3_representative_CPP.json"
)

with open(
    representative_sequence_file,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        {
            "sequence_id":
                REPRESENTATIVE_SEQUENCE_ID,

            "sequence":
                REPRESENTATIVE_SEQUENCE,

            "length": int(
                len(
                    REPRESENTATIVE_SEQUENCE
                )
            ),

            "mean_probability": float(
                representative_row[
                    "mean_probability"
                ]
            ),

            "minimum_model_probability": float(
                representative_row[
                    "minimum_probability"
                ]
            ),

            "number_of_hotspots": int(
                representative_row[
                    "number_of_hotspots"
                ]
            ),

            "strict_hotspots": int(
                representative_row[
                    "strict_hotspots"
                ]
            ),

            "unanimous_hotspots": int(
                representative_row[
                    "unanimous_hotspots"
                ]
            ),
        },
        handle,
        indent=2,
    )


# ============================================================
# 16. CHECKPOINT
# ============================================================

checkpoint_details = {
    "figure": "Figure 3",

    "title": (
        "Construction, agreement, and stability "
        "of multi-PLM consensus XAI"
    ),

    "representative_model":
        REPRESENTATIVE_MODEL,

    "representative_dataset":
        REPRESENTATIVE_DATASET,

    "representative_sequence_id":
        REPRESENTATIVE_SEQUENCE_ID,

    "representative_sequence":
        REPRESENTATIVE_SEQUENCE,

    "panels": {
        "A": (
            "Representative residue-level XAI map"
        ),
        "B": (
            "Within-model XAI method agreement"
        ),
        "C": (
            "Cross-PLM agreement and adjusted "
            "consensus stability"
        ),
    },

    "formats": [
        "PNG 600 dpi",
        "PDF",
        "SVG",
    ],
}


try:

    mark_step_complete(
        "24_figure_3_consensus_XAI",
        output_files=(
            saved_files
            + [
                method_file_audit,
                representative_sequence_file,
            ]
        ),
        details=checkpoint_details,
    )

except NameError:

    print(
        "mark_step_complete() was not found, "
        "but Figure 3 was saved successfully."
    )


# ============================================================
# 17. DISPLAY OUTPUT SUMMARY
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 3 METHOD FILES")
print("=" * 78)

display(
    method_file_audit_df
)


print("\n" + "=" * 78)
print("REPRESENTATIVE CPP")
print("=" * 78)

display(
    candidate_table[
        [
            "sequence_id",
            "sequence",
            "sequence_length",
            "mean_probability",
            "minimum_probability",
            "number_of_hotspots",
            "strict_hotspots",
            "unanimous_hotspots",
            "selection_score",
        ]
    ].head(10)
)


print("\n" + "=" * 78)
print("FIGURE 3 COMPLETED")
print("=" * 78)

for file_path in saved_files:
    print(
        file_path
    )

In [ ]:
# ============================================================
# STEP 25: FIGURE 4 — REPLICATED RESIDUE AND MOTIF GRAMMAR
#
# Panels:
# A. Replicated residue enrichment forest plot
# B. Internal vs KELM residue-effect replication
# C. Replicated hotspot motifs
# D. Representative motif-containing CPP hotspot maps
#
# Exports:
# PNG 600 dpi
# PDF vector
# SVG vector
# ============================================================

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.patches import (
    Rectangle,
    Patch,
)
from matplotlib.lines import Line2D


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

XAI_ROOT = (
    PROJECT_DIR / "06_xai"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

CHECKPOINT_DIR = (
    PROJECT_DIR / "08_checkpoints"
)

for directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. PUBLICATION STYLE
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "font.weight": "bold",

    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 8.2,
    "legend.frameon": False,

    "lines.linewidth": 2.0,
    "lines.markersize": 6,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})


# ============================================================
# 3. INPUT FILES
# ============================================================

RESIDUE_REPLICATION_FILE = (
    TABLE_MAIN_DIR
    / "residue_enrichment_replication.csv"
)

MOTIF_FILE = (
    TABLE_MAIN_DIR
    / "strict_replicated_CPP_hotspot_motifs.csv"
)

MOTIF_REPLICATION_FILE = (
    TABLE_MAIN_DIR
    / "hotspot_motif_replication.csv"
)

INTERNAL_CONSENSUS_FILE = (
    XAI_ROOT
    / "consensus_adjusted"
    / "global"
    / "internal_test"
    / "adjusted_global_consensus_residue_scores.csv"
)

KELM_CONSENSUS_FILE = (
    XAI_ROOT
    / "consensus_adjusted"
    / "global"
    / "kelm_external"
    / "adjusted_global_consensus_residue_scores.csv"
)

for required_file in [
    RESIDUE_REPLICATION_FILE,
    MOTIF_FILE,
    MOTIF_REPLICATION_FILE,
    INTERNAL_CONSENSUS_FILE,
    KELM_CONSENSUS_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file missing:\n{required_file}"
        )


# ============================================================
# 4. LOAD DATA
# ============================================================

residue_df = pd.read_csv(
    RESIDUE_REPLICATION_FILE
)

motif_df = pd.read_csv(
    MOTIF_FILE
)

motif_replication_df = pd.read_csv(
    MOTIF_REPLICATION_FILE
)

internal_consensus = pd.read_csv(
    INTERNAL_CONSENSUS_FILE
)

kelm_consensus = pd.read_csv(
    KELM_CONSENSUS_FILE
)


# ============================================================
# 5. GENERAL HELPERS
# ============================================================

def add_panel_label(
    axis,
    label,
    x=-0.13,
    y=1.08,
):
    axis.text(
        x,
        y,
        label,
        transform=axis.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )


def style_axis(
    axis,
):
    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        width=1.1,
        length=4,
    )


def safe_log2_odds_ratio(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = np.clip(
        values,
        1e-6,
        None,
    )

    return np.log2(
        values
    )


def abbreviated_p_value(
    value,
):
    value = float(
        value
    )

    if value < 1e-99:
        return "<1e−99"

    if value < 0.001:
        exponent = int(
            np.floor(
                np.log10(
                    value
                )
            )
        )

        coefficient = (
            value
            / (
                10 ** exponent
            )
        )

        return (
            f"{coefficient:.1f}"
            f"e{exponent}"
        )

    return f"{value:.3f}"


# ============================================================
# 6. PREPARE REPLICATED RESIDUE TABLE
# ============================================================

replicated_residues = residue_df[
    residue_df[
        "significant_in_both"
    ].astype(bool)
].copy()

replicated_residues[
    "direction"
] = np.where(
    replicated_residues[
        "internal_log2_enrichment"
    ] > 0,
    "Enriched",
    "Depleted",
)

replicated_residues[
    "mean_log2_enrichment"
] = (
    replicated_residues[
        [
            "internal_log2_enrichment",
            "kelm_log2_enrichment",
        ]
    ].mean(axis=1)
)

replicated_residues = (
    replicated_residues.sort_values(
        [
            "direction",
            "mean_log2_enrichment",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


# Keep enriched residues first, followed by depleted residues
enriched_residues = replicated_residues[
    replicated_residues[
        "direction"
    ] == "Enriched"
].copy()

depleted_residues = replicated_residues[
    replicated_residues[
        "direction"
    ] == "Depleted"
].copy()

depleted_residues = depleted_residues.sort_values(
    "mean_log2_enrichment",
    ascending=True,
)

replicated_residues = pd.concat(
    [
        enriched_residues,
        depleted_residues,
    ],
    ignore_index=True,
)


# ============================================================
# 7. PREPARE MOTIF TABLE
# ============================================================

motif_plot_df = motif_df.copy()

motif_plot_df[
    "mean_log2_odds_ratio"
] = (
    safe_log2_odds_ratio(
        motif_plot_df[
            "internal_odds_ratio"
        ]
    )
    + safe_log2_odds_ratio(
        motif_plot_df[
            "kelm_odds_ratio"
        ]
    )
) / 2.0

motif_plot_df = motif_plot_df.sort_values(
    "mean_log2_odds_ratio",
    ascending=True,
).reset_index(
    drop=True
)


# ============================================================
# 8. FIND REPRESENTATIVE MOTIF-CONTAINING CPP SEQUENCES
# ============================================================

TARGET_MOTIFS = [
    "RR",
    "WK",
    "WR",
    "RW",
    "RL",
    "LR",
]


def build_sequence_hotspot_table(
    consensus_df,
):
    sequence_records = []

    for sequence_id, group in (
        consensus_df.groupby(
            "sequence_id",
            sort=False,
        )
    ):
        sequence = str(
            group[
                "sequence"
            ].iloc[0]
        )

        label = int(
            group[
                "label"
            ].iloc[0]
        )

        positions = group[
            "position"
        ].astype(int).to_numpy()

        residues = group[
            "residue"
        ].astype(str).to_numpy()

        hotspots = group[
            "adjusted_hotspot_top20"
        ].astype(bool).to_numpy()

        strict = group[
            "strict_cross_model_hotspot"
        ].astype(bool).to_numpy()

        unanimous = group[
            "unanimous_cross_model_hotspot"
        ].astype(bool).to_numpy()

        sequence_records.append({
            "sequence_id": str(
                sequence_id
            ),
            "sequence": sequence,
            "label": label,
            "positions": positions,
            "residues": residues,
            "hotspots": hotspots,
            "strict": strict,
            "unanimous": unanimous,
            "length": len(sequence),
        })

    return sequence_records


internal_sequence_records = (
    build_sequence_hotspot_table(
        internal_consensus
    )
)

kelm_sequence_records = (
    build_sequence_hotspot_table(
        kelm_consensus
    )
)


def locate_motif_starts(
    sequence,
    motif,
):
    starts = []

    start = sequence.find(
        motif
    )

    while start != -1:
        starts.append(
            start
        )

        start = sequence.find(
            motif,
            start + 1,
        )

    return starts


def motif_overlaps_hotspot(
    motif_start,
    motif_length,
    hotspot_mask,
):
    motif_positions = range(
        motif_start,
        motif_start + motif_length,
    )

    return any(
        hotspot_mask[
            position
        ]
        for position in motif_positions
        if (
            0
            <= position
            < len(
                hotspot_mask
            )
        )
    )


def rank_representative_sequences(
    records,
    motif,
):
    candidate_rows = []

    for record in records:

        if record[
            "label"
        ] != 1:
            continue

        starts = locate_motif_starts(
            record[
                "sequence"
            ],
            motif,
        )

        if not starts:
            continue

        overlapping_starts = [
            start
            for start in starts
            if motif_overlaps_hotspot(
                motif_start=start,
                motif_length=len(motif),
                hotspot_mask=record[
                    "hotspots"
                ],
            )
        ]

        if not overlapping_starts:
            continue

        motif_positions = set()

        for start in overlapping_starts:

            motif_positions.update(
                range(
                    start,
                    start + len(
                        motif
                    ),
                )
            )

        strict_overlap = sum(
            record[
                "strict"
            ][position]
            for position in motif_positions
            if position
            < record[
                "length"
            ]
        )

        unanimous_overlap = sum(
            record[
                "unanimous"
            ][position]
            for position in motif_positions
            if position
            < record[
                "length"
            ]
        )

        hotspot_overlap = sum(
            record[
                "hotspots"
            ][position]
            for position in motif_positions
            if position
            < record[
                "length"
            ]
        )

        length_penalty = abs(
            record[
                "length"
            ] - 22
        )

        score = (
            3.0
            * unanimous_overlap
            + 2.0
            * strict_overlap
            + 1.0
            * hotspot_overlap
            + 0.5
            * len(
                overlapping_starts
            )
            - 0.03
            * length_penalty
        )

        candidate_rows.append({
            **record,
            "motif": motif,
            "motif_starts":
                overlapping_starts,
            "selection_score":
                score,
        })

    return sorted(
        candidate_rows,
        key=lambda item:
            item[
                "selection_score"
            ],
        reverse=True,
    )


representative_sequences = []

used_sequence_ids = set()

# Select three complementary examples
for motif in [
    "RR",
    "WK",
    "RW",
    "WR",
    "RL",
    "LR",
]:

    candidates = (
        rank_representative_sequences(
            internal_sequence_records,
            motif,
        )
        + rank_representative_sequences(
            kelm_sequence_records,
            motif,
        )
    )

    candidates = sorted(
        candidates,
        key=lambda item:
            item[
                "selection_score"
            ],
        reverse=True,
    )

    selected = None

    for candidate in candidates:

        if candidate[
            "sequence_id"
        ] in used_sequence_ids:
            continue

        selected = candidate
        break

    if selected is not None:

        selected[
            "dataset"
        ] = (
            "internal_test"
            if selected[
                "sequence_id"
            ].startswith(
                "INT"
            )
            else "kelm_external"
        )

        representative_sequences.append(
            selected
        )

        used_sequence_ids.add(
            selected[
                "sequence_id"
            ]
        )

    if len(
        representative_sequences
    ) >= 3:
        break


if len(
    representative_sequences
) < 3:
    raise RuntimeError(
        "Fewer than three representative "
        "motif-containing sequences were found."
    )


# ============================================================
# 9. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(15.2, 10.0)
)

grid = fig.add_gridspec(
    nrows=2,
    ncols=2,
    width_ratios=[
        1.05,
        0.95,
    ],
    height_ratios=[
        1.0,
        1.0,
    ],
    wspace=0.28,
    hspace=0.36,
)

ax_a = fig.add_subplot(
    grid[0, 0]
)

ax_b = fig.add_subplot(
    grid[0, 1]
)

ax_c = fig.add_subplot(
    grid[1, 0]
)

ax_d = fig.add_subplot(
    grid[1, 1]
)


# ============================================================
# PANEL A — RESIDUE ENRICHMENT FOREST PLOT
# ============================================================

y_positions = np.arange(
    len(
        replicated_residues
    )
)

internal_log2_or = safe_log2_odds_ratio(
    replicated_residues[
        "internal_odds_ratio"
    ]
)

kelm_log2_or = safe_log2_odds_ratio(
    replicated_residues[
        "kelm_odds_ratio"
    ]
)

offset = 0.14

ax_a.scatter(
    internal_log2_or,
    y_positions - offset,
    s=60,
    marker="o",
    edgecolor="black",
    linewidth=0.7,
    label="Internal test",
    zorder=3,
)

ax_a.scatter(
    kelm_log2_or,
    y_positions + offset,
    s=60,
    marker="s",
    edgecolor="black",
    linewidth=0.7,
    label="KELM external",
    zorder=3,
)

for row_index in range(
    len(
        replicated_residues
    )
):

    ax_a.plot(
        [
            internal_log2_or[
                row_index
            ],
            kelm_log2_or[
                row_index
            ],
        ],
        [
            y_positions[
                row_index
            ] - offset,
            y_positions[
                row_index
            ] + offset,
        ],
        linewidth=1.0,
        alpha=0.65,
        zorder=1,
    )

ax_a.axvline(
    0,
    linestyle="--",
    linewidth=1.2,
)

ax_a.set_yticks(
    y_positions
)

ax_a.set_yticklabels(
    replicated_residues[
        "residue"
    ],
    fontsize=10,
    fontweight="bold",
)

ax_a.set_xlabel(
    "log$_2$(odds ratio)"
)

ax_a.set_ylabel(
    "Amino-acid residue"
)

ax_a.set_title(
    "Replicated residue enrichment in CPP hotspots"
)

ax_a.legend(
    loc="lower right"
)

ax_a.invert_yaxis()

style_axis(
    ax_a
)

add_panel_label(
    ax_a,
    "A",
)


# Add direction labels
x_limits_a = ax_a.get_xlim()

ax_a.text(
    x_limits_a[0],
    len(
        replicated_residues
    ) - 0.2,
    "Depleted",
    ha="left",
    va="bottom",
    fontsize=8,
    fontweight="bold",
)

ax_a.text(
    x_limits_a[1],
    len(
        replicated_residues
    ) - 0.2,
    "Enriched",
    ha="right",
    va="bottom",
    fontsize=8,
    fontweight="bold",
)


# ============================================================
# PANEL B — INTERNAL–KELM RESIDUE REPLICATION
# ============================================================

x_values = replicated_residues[
    "internal_log2_enrichment"
].to_numpy()

y_values = replicated_residues[
    "kelm_log2_enrichment"
].to_numpy()

ax_b.scatter(
    x_values,
    y_values,
    s=75,
    edgecolor="black",
    linewidth=0.8,
    zorder=3,
)

for row in replicated_residues.itertuples(
    index=False
):

    ax_b.text(
        row.internal_log2_enrichment,
        row.kelm_log2_enrichment,
        f" {row.residue}",
        fontsize=9,
        fontweight="bold",
        ha="left",
        va="center",
    )

minimum_value = min(
    x_values.min(),
    y_values.min(),
)

maximum_value = max(
    x_values.max(),
    y_values.max(),
)

padding = 0.5

plot_minimum = (
    minimum_value - padding
)

plot_maximum = (
    maximum_value + padding
)

ax_b.plot(
    [
        plot_minimum,
        plot_maximum,
    ],
    [
        plot_minimum,
        plot_maximum,
    ],
    linestyle="--",
    linewidth=1.2,
)

ax_b.axhline(
    0,
    linestyle=":",
    linewidth=1.0,
)

ax_b.axvline(
    0,
    linestyle=":",
    linewidth=1.0,
)

ax_b.set_xlim(
    plot_minimum,
    plot_maximum,
)

ax_b.set_ylim(
    plot_minimum,
    plot_maximum,
)

ax_b.set_xlabel(
    "Internal log$_2$ enrichment"
)

ax_b.set_ylabel(
    "KELM log$_2$ enrichment"
)

ax_b.set_title(
    "External replication of residue effects"
)

style_axis(
    ax_b
)

add_panel_label(
    ax_b,
    "B",
)


# ============================================================
# PANEL C — REPLICATED MOTIF ODDS RATIOS
# ============================================================

motif_y = np.arange(
    len(
        motif_plot_df
    )
)

motif_internal_log2_or = (
    safe_log2_odds_ratio(
        motif_plot_df[
            "internal_odds_ratio"
        ]
    )
)

motif_kelm_log2_or = (
    safe_log2_odds_ratio(
        motif_plot_df[
            "kelm_odds_ratio"
        ]
    )
)

motif_offset = 0.14

ax_c.scatter(
    motif_internal_log2_or,
    motif_y - motif_offset,
    s=70,
    marker="o",
    edgecolor="black",
    linewidth=0.8,
    label="Internal test",
    zorder=3,
)

ax_c.scatter(
    motif_kelm_log2_or,
    motif_y + motif_offset,
    s=70,
    marker="s",
    edgecolor="black",
    linewidth=0.8,
    label="KELM external",
    zorder=3,
)

for row_index in range(
    len(
        motif_plot_df
    )
):

    ax_c.plot(
        [
            motif_internal_log2_or[
                row_index
            ],
            motif_kelm_log2_or[
                row_index
            ],
        ],
        [
            motif_y[
                row_index
            ] - motif_offset,
            motif_y[
                row_index
            ] + motif_offset,
        ],
        linewidth=1.0,
        alpha=0.65,
    )

ax_c.set_yticks(
    motif_y
)

ax_c.set_yticklabels(
    motif_plot_df[
        "motif"
    ],
    fontsize=10,
    fontweight="bold",
)

ax_c.set_xlabel(
    "log$_2$(odds ratio)"
)

ax_c.set_ylabel(
    "Replicated hotspot motif"
)

ax_c.set_title(
    "Externally replicated CPP hotspot motifs"
)

ax_c.legend(
    loc="lower right"
)

style_axis(
    ax_c
)

add_panel_label(
    ax_c,
    "C",
)


# Add counts beside motifs
x_right = ax_c.get_xlim()[1]

for row_index, row in (
    motif_plot_df.iterrows()
):

    ax_c.text(
        x_right,
        motif_y[
            row_index
        ],
        (
            f"  n={int(row['internal_cpp_count'])}/"
            f"{int(row['kelm_cpp_count'])}"
        ),
        ha="left",
        va="center",
        fontsize=7.5,
        fontweight="bold",
    )

ax_c.set_xlim(
    ax_c.get_xlim()[0],
    x_right + 1.1,
)


# ============================================================
# PANEL D — REPRESENTATIVE MOTIF-CENTERED HOTSPOT MAPS
# ============================================================

ax_d.set_xlim(
    0,
    1,
)

ax_d.set_ylim(
    0,
    1,
)

ax_d.axis(
    "off"
)

ax_d.set_title(
    "Representative motif-centered consensus hotspots",
    pad=10,
)

add_panel_label(
    ax_d,
    "D",
)


row_centers = [
    0.77,
    0.48,
    0.19,
]

maximum_display_length = max(
    sequence_record[
        "length"
    ]
    for sequence_record
    in representative_sequences
)

left_margin = 0.05
right_margin = 0.97

available_width = (
    right_margin
    - left_margin
)

residue_width = (
    available_width
    / maximum_display_length
)


representative_audit_rows = []

for sequence_index, (
    record,
    y_center,
) in enumerate(
    zip(
        representative_sequences,
        row_centers,
    )
):

    sequence = record[
        "sequence"
    ]

    motif = record[
        "motif"
    ]

    motif_starts = record[
        "motif_starts"
    ]

    motif_position_set = set()

    for motif_start in motif_starts:

        motif_position_set.update(
            range(
                motif_start,
                motif_start
                + len(
                    motif
                ),
            )
        )

    ax_d.text(
        left_margin,
        y_center + 0.095,
        (
            f"{record['sequence_id']} "
            f"({record['dataset'].replace('_', ' ')}) "
            f"— motif {motif}"
        ),
        ha="left",
        va="center",
        fontsize=9,
        fontweight="bold",
    )

    for residue_index, residue in enumerate(
        sequence
    ):

        x_left = (
            left_margin
            + residue_index
            * residue_width
        )

        is_hotspot = bool(
            record[
                "hotspots"
            ][residue_index]
        )

        is_strict = bool(
            record[
                "strict"
            ][residue_index]
        )

        is_unanimous = bool(
            record[
                "unanimous"
            ][residue_index]
        )

        is_motif = (
            residue_index
            in motif_position_set
        )

        facecolor = (
            "#D9D9D9"
        )

        if is_hotspot:
            facecolor = (
                "#F2B134"
            )

        if is_motif:
            facecolor = (
                "#E86A5A"
            )

        rectangle = Rectangle(
            (
                x_left,
                y_center - 0.035,
            ),
            residue_width * 0.96,
            0.07,
            facecolor=facecolor,
            edgecolor="black",
            linewidth=0.7,
        )

        ax_d.add_patch(
            rectangle
        )

        ax_d.text(
            x_left
            + residue_width
            * 0.48,
            y_center,
            residue,
            ha="center",
            va="center",
            fontsize=(
                7.5
                if maximum_display_length
                <= 30
                else 6.5
            ),
            fontweight="bold",
        )

        if is_unanimous:

            ax_d.text(
                x_left
                + residue_width
                * 0.48,
                y_center - 0.060,
                "★",
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
            )

        elif is_strict:

            ax_d.text(
                x_left
                + residue_width
                * 0.48,
                y_center - 0.060,
                "●",
                ha="center",
                va="center",
                fontsize=5,
                fontweight="bold",
            )

    representative_audit_rows.append({
        "sequence_id":
            record[
                "sequence_id"
            ],
        "dataset":
            record[
                "dataset"
            ],
        "sequence":
            sequence,
        "length":
            record[
                "length"
            ],
        "motif":
            motif,
        "motif_starts_0_based":
            motif_starts,
        "hotspot_count": int(
            np.sum(
                record[
                    "hotspots"
                ]
            )
        ),
        "strict_hotspot_count": int(
            np.sum(
                record[
                    "strict"
                ]
            )
        ),
        "unanimous_hotspot_count": int(
            np.sum(
                record[
                    "unanimous"
                ]
            )
        ),
        "selection_score": float(
            record[
                "selection_score"
            ]
        ),
    })


legend_handles = [
    Patch(
        facecolor="#D9D9D9",
        edgecolor="black",
        label="Other residue",
    ),
    Patch(
        facecolor="#F2B134",
        edgecolor="black",
        label="Consensus hotspot",
    ),
    Patch(
        facecolor="#E86A5A",
        edgecolor="black",
        label="Replicated motif",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        color="black",
        markersize=4,
        label="Strict hotspot",
    ),
    Line2D(
        [0],
        [0],
        marker="*",
        linestyle="None",
        color="black",
        markersize=7,
        label="Unanimous hotspot",
    ),
]

ax_d.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(
        0.5,
        -0.03,
    ),
    ncol=3,
    fontsize=7.5,
    frameon=False,
)


# ============================================================
# 10. FINAL LAYOUT
# ============================================================

fig.suptitle(
    "Replicated Residue and Motif Grammar of "
    "Consensus CPP Hotspots",
    fontsize=16,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    left=0.07,
    right=0.97,
    top=0.92,
    bottom=0.08,
)


# ============================================================
# 11. SAVE FIGURE
# ============================================================

figure_stem = (
    "Figure_4_Replicated_Residue_and_Motif_Grammar"
)

saved_files = []

for output_directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:

    base_file = (
        output_directory
        / figure_stem
    )

    for extension in [
        ".png",
        ".pdf",
        ".svg",
    ]:

        output_file = (
            base_file.with_suffix(
                extension
            )
        )

        if extension == ".png":

            fig.savefig(
                output_file,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        else:

            fig.savefig(
                output_file,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        saved_files.append(
            output_file
        )


plt.show()
plt.close(fig)


# ============================================================
# 12. SAVE FIGURE AUDIT TABLES
# ============================================================

residue_audit_file = (
    CHECKPOINT_DIR
    / "figure_4_replicated_residues.csv"
)

motif_audit_file = (
    CHECKPOINT_DIR
    / "figure_4_replicated_motifs.csv"
)

representative_audit_file = (
    CHECKPOINT_DIR
    / "figure_4_representative_sequences.csv"
)

replicated_residues.to_csv(
    residue_audit_file,
    index=False,
)

motif_plot_df.to_csv(
    motif_audit_file,
    index=False,
)

representative_audit_df = pd.DataFrame(
    representative_audit_rows
)

representative_audit_df.to_csv(
    representative_audit_file,
    index=False,
)


# ============================================================
# 13. CHECKPOINT
# ============================================================

checkpoint_details = {
    "figure": "Figure 4",

    "title": (
        "Replicated residue and motif grammar "
        "of consensus CPP hotspots"
    ),

    "panels": {
        "A": (
            "Replicated residue enrichment "
            "forest plot"
        ),
        "B": (
            "Internal–KELM residue-effect "
            "replication"
        ),
        "C": (
            "Replicated hotspot motif odds ratios"
        ),
        "D": (
            "Representative motif-containing "
            "consensus hotspot maps"
        ),
    },

    "replicated_residues": (
        replicated_residues[
            "residue"
        ].tolist()
    ),

    "replicated_motifs": (
        motif_plot_df[
            "motif"
        ].tolist()
    ),

    "representative_sequences": (
        representative_audit_df[
            "sequence_id"
        ].tolist()
    ),

    "formats": [
        "PNG 600 dpi",
        "PDF",
        "SVG",
    ],
}


try:

    mark_step_complete(
        "25_figure_4_residue_and_motif_grammar",
        output_files=(
            saved_files
            + [
                residue_audit_file,
                motif_audit_file,
                representative_audit_file,
            ]
        ),
        details=checkpoint_details,
    )

except NameError:

    print(
        "mark_step_complete() was not found, "
        "but Figure 4 was saved successfully."
    )


# ============================================================
# 14. DISPLAY OUTPUTS
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 4 REPLICATED RESIDUES")
print("=" * 78)

display(
    replicated_residues[
        [
            "residue",
            "direction",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_fdr",
            "same_enrichment_direction",
            "significant_in_both",
        ]
    ]
)


print("\n" + "=" * 78)
print("FIGURE 4 REPLICATED MOTIFS")
print("=" * 78)

display(
    motif_plot_df[
        [
            "motif",
            "motif_length",
            "internal_cpp_count",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_cpp_count",
            "kelm_odds_ratio",
            "kelm_fdr",
        ]
    ]
)


print("\n" + "=" * 78)
print("FIGURE 4 REPRESENTATIVE SEQUENCES")
print("=" * 78)

display(
    representative_audit_df
)


print("\n" + "=" * 78)
print("FIGURE 4 COMPLETED")
print("=" * 78)

for file_path in saved_files:
    print(
        file_path
    )

In [ ]:
# ============================================================
# STEP 26: FIGURE 5 — FAITHFULNESS OF CONSENSUS HOTSPOTS
#
# Panels:
# A. Internal CPP hotspot vs random ablation
# B. KELM CPP hotspot vs random ablation
# C. Hotspot-minus-random effect with 95% CI
# D. Hotspot-only sufficiency
#
# Exports:
# PNG 600 dpi
# PDF vector
# SVG vector
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

TABLE_SI_DIR = (
    RESULTS_DIR / "tables_SI"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

CHECKPOINT_DIR = (
    PROJECT_DIR / "08_checkpoints"
)

for directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. STYLE
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "font.weight": "bold",

    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 8.5,
    "legend.frameon": False,

    "lines.linewidth": 2.0,
    "lines.markersize": 6,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})


# ============================================================
# 3. INPUT FILES
# ============================================================

PER_SEQUENCE_FILE = (
    TABLE_SI_DIR
    / "faithfulness_all_sequences_all_models.csv"
)

SUMMARY_FILE = (
    TABLE_MAIN_DIR
    / "CPP_hotspot_faithfulness_manuscript_table.csv"
)

for required_file in [
    PER_SEQUENCE_FILE,
    SUMMARY_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file missing:\n{required_file}"
        )


per_sequence = pd.read_csv(
    PER_SEQUENCE_FILE
)

summary = pd.read_csv(
    SUMMARY_FILE
)


# ============================================================
# 4. CONFIGURATION
# ============================================================

MODEL_ORDER = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_DISPLAY = {
    "ESM2_320": "ESM2-320",
    "ESM2_640": "ESM2-640",
    "ESM2_1280": "ESM2-1280",
    "ProtT5": "ProtT5",
}

DATASET_DISPLAY = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}


# Keep CPP sequences only
cpp_data = per_sequence[
    per_sequence["label"] == 1
].copy()


# ============================================================
# 5. HELPERS
# ============================================================

def add_panel_label(
    axis,
    label,
):
    axis.text(
        -0.13,
        1.08,
        label,
        transform=axis.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )


def style_axis(
    axis,
):
    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        width=1.1,
        length=4,
    )


def add_jittered_points(
    axis,
    values,
    x_position,
    random_seed,
):
    rng = np.random.default_rng(
        random_seed
    )

    jitter = rng.normal(
        loc=0,
        scale=0.045,
        size=len(values),
    )

    axis.scatter(
        np.full(
            len(values),
            x_position,
        ) + jitter,
        values,
        s=8,
        alpha=0.20,
        linewidth=0,
        zorder=1,
    )


def plot_ablation_panel(
    axis,
    dataset_name,
    panel_label,
):
    dataset_table = cpp_data[
        cpp_data["dataset"]
        == dataset_name
    ]

    hotspot_values = []
    random_values = []

    for model_name in MODEL_ORDER:
        model_table = dataset_table[
            dataset_table["model"]
            == model_name
        ]

        hotspot_values.append(
            model_table[
                "hotspot_probability_drop"
            ].to_numpy()
        )

        random_values.append(
            model_table[
                "random_mean_probability_drop"
            ].to_numpy()
        )

    base_positions = np.arange(
        len(MODEL_ORDER)
    ) * 2.5

    hotspot_positions = (
        base_positions - 0.32
    )

    random_positions = (
        base_positions + 0.32
    )

    hotspot_boxes = axis.boxplot(
        hotspot_values,
        positions=hotspot_positions,
        widths=0.52,
        patch_artist=True,
        showfliers=False,
        medianprops={
            "linewidth": 1.6,
        },
        boxprops={
            "linewidth": 1.1,
        },
        whiskerprops={
            "linewidth": 1.0,
        },
        capprops={
            "linewidth": 1.0,
        },
    )

    random_boxes = axis.boxplot(
        random_values,
        positions=random_positions,
        widths=0.52,
        patch_artist=True,
        showfliers=False,
        medianprops={
            "linewidth": 1.6,
        },
        boxprops={
            "linewidth": 1.1,
        },
        whiskerprops={
            "linewidth": 1.0,
        },
        capprops={
            "linewidth": 1.0,
        },
    )

    for box in hotspot_boxes[
        "boxes"
    ]:
        box.set_facecolor(
            "#D9D9D9"
        )

        box.set_edgecolor(
            "black"
        )

    for box in random_boxes[
        "boxes"
    ]:
        box.set_facecolor(
            "white"
        )

        box.set_edgecolor(
            "black"
        )

    for model_index in range(
        len(MODEL_ORDER)
    ):
        add_jittered_points(
            axis,
            hotspot_values[
                model_index
            ],
            hotspot_positions[
                model_index
            ],
            random_seed=100
            + model_index,
        )

        add_jittered_points(
            axis,
            random_values[
                model_index
            ],
            random_positions[
                model_index
            ],
            random_seed=200
            + model_index,
        )

    axis.axhline(
        0,
        linestyle="--",
        linewidth=1.0,
    )

    axis.set_xticks(
        base_positions
    )

    axis.set_xticklabels(
        [
            MODEL_DISPLAY[
                model_name
            ]
            for model_name
            in MODEL_ORDER
        ],
        rotation=20,
        ha="right",
        fontweight="bold",
    )

    axis.set_ylabel(
        "Decrease in predicted CPP probability"
    )

    axis.set_title(
        f"{DATASET_DISPLAY[dataset_name]} CPPs"
    )

    legend_handles = [
        mpl.patches.Patch(
            facecolor="#D9D9D9",
            edgecolor="black",
            label="Consensus-hotspot ablation",
        ),
        mpl.patches.Patch(
            facecolor="white",
            edgecolor="black",
            label="Matched random ablation",
        ),
    ]

    axis.legend(
        handles=legend_handles,
        loc="upper right",
    )

    style_axis(
        axis
    )

    add_panel_label(
        axis,
        panel_label,
    )


# ============================================================
# 6. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(15.0, 9.0)
)

grid = fig.add_gridspec(
    nrows=2,
    ncols=2,
    width_ratios=[
        1.05,
        0.95,
    ],
    height_ratios=[
        1.0,
        1.0,
    ],
    wspace=0.28,
    hspace=0.34,
)

ax_a = fig.add_subplot(
    grid[0, 0]
)

ax_b = fig.add_subplot(
    grid[0, 1]
)

ax_c = fig.add_subplot(
    grid[1, 0]
)

ax_d = fig.add_subplot(
    grid[1, 1]
)


# ============================================================
# PANEL A — INTERNAL ABLATION
# ============================================================

plot_ablation_panel(
    ax_a,
    dataset_name="internal_test",
    panel_label="A",
)


# ============================================================
# PANEL B — KELM ABLATION
# ============================================================

plot_ablation_panel(
    ax_b,
    dataset_name="kelm_external",
    panel_label="B",
)


# ============================================================
# PANEL C — MEAN HOTSPOT-MINUS-RANDOM EFFECT
# ============================================================

effect_table = summary.copy()

effect_table["model"] = pd.Categorical(
    effect_table["model"],
    categories=MODEL_ORDER,
    ordered=True,
)

effect_table = effect_table.sort_values(
    [
        "dataset",
        "model",
    ]
)

x = np.arange(
    len(MODEL_ORDER)
)

width = 0.34

for dataset_index, dataset_name in enumerate(
    [
        "internal_test",
        "kelm_external",
    ]
):
    subset = effect_table[
        effect_table["dataset"]
        == dataset_name
    ]

    means = subset[
        "mean_hotspot_minus_random_drop"
    ].to_numpy()

    lower_errors = (
        means
        - subset[
            "difference_ci95_low"
        ].to_numpy()
    )

    upper_errors = (
        subset[
            "difference_ci95_high"
        ].to_numpy()
        - means
    )

    positions = (
        x
        + (
            dataset_index - 0.5
        )
        * width
    )

    ax_c.errorbar(
        positions,
        means,
        yerr=np.vstack([
            lower_errors,
            upper_errors,
        ]),
        fmt=(
            "o"
            if dataset_index == 0
            else "s"
        ),
        markersize=7,
        capsize=4,
        linewidth=1.6,
        label=DATASET_DISPLAY[
            dataset_name
        ],
    )

ax_c.axhline(
    0,
    linestyle="--",
    linewidth=1.0,
)

ax_c.set_xticks(
    x
)

ax_c.set_xticklabels(
    [
        MODEL_DISPLAY[
            model_name
        ]
        for model_name
        in MODEL_ORDER
    ],
    rotation=20,
    ha="right",
    fontweight="bold",
)

ax_c.set_ylabel(
    "Hotspot drop − random drop"
)

ax_c.set_title(
    "Faithfulness effect with 95% confidence intervals"
)

ax_c.legend(
    loc="upper left"
)

style_axis(
    ax_c
)

add_panel_label(
    ax_c,
    "C",
)


# ============================================================
# PANEL D — HOTSPOT-ONLY SUFFICIENCY
# ============================================================

marker_map = {
    "ESM2_320": "o",
    "ESM2_640": "s",
    "ESM2_1280": "^",
    "ProtT5": "D",
}

for dataset_name, fill_style in [
    (
        "internal_test",
        "full",
    ),
    (
        "kelm_external",
        "none",
    ),
]:
    dataset_table = cpp_data[
        cpp_data["dataset"]
        == dataset_name
    ]

    for model_name in MODEL_ORDER:
        model_table = dataset_table[
            dataset_table["model"]
            == model_name
        ]

        ax_d.scatter(
            model_table[
                "original_probability"
            ],
            model_table[
                "hotspot_only_probability"
            ],
            s=20,
            marker=marker_map[
                model_name
            ],
            facecolors=(
                "none"
                if fill_style == "none"
                else None
            ),
            edgecolors="black",
            linewidth=0.6,
            alpha=0.38,
        )

ax_d.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    linewidth=1.1,
)

ax_d.set_xlim(
    0,
    1.02,
)

ax_d.set_ylim(
    0,
    1.02,
)

ax_d.set_xlabel(
    "Original predicted CPP probability"
)

ax_d.set_ylabel(
    "Hotspot-only predicted CPP probability"
)

ax_d.set_title(
    "Predictive sufficiency of consensus hotspots"
)

model_handles = [
    mpl.lines.Line2D(
        [0],
        [0],
        marker=marker_map[
            model_name
        ],
        linestyle="None",
        color="black",
        markersize=6,
        label=MODEL_DISPLAY[
            model_name
        ],
    )
    for model_name in MODEL_ORDER
]

dataset_handles = [
    mpl.lines.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        color="black",
        markerfacecolor="black",
        markersize=6,
        label="Internal test",
    ),
    mpl.lines.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        color="black",
        markerfacecolor="white",
        markersize=6,
        label="KELM external",
    ),
]

legend_1 = ax_d.legend(
    handles=model_handles,
    loc="lower right",
    title="Model",
    title_fontsize=8.5,
)

ax_d.add_artist(
    legend_1
)

ax_d.legend(
    handles=dataset_handles,
    loc="upper left",
    title="Dataset",
    title_fontsize=8.5,
)

style_axis(
    ax_d
)

add_panel_label(
    ax_d,
    "D",
)


# ============================================================
# 7. FINAL LAYOUT
# ============================================================

fig.suptitle(
    "Perturbation-Based Faithfulness of "
    "Consensus CPP Hotspots",
    fontsize=16,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    left=0.075,
    right=0.98,
    top=0.92,
    bottom=0.09,
)


# ============================================================
# 8. SAVE FIGURE
# ============================================================

figure_stem = (
    "Figure_5_Consensus_Hotspot_Faithfulness"
)

saved_files = []

for output_directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:

    base_file = (
        output_directory
        / figure_stem
    )

    for extension in [
        ".png",
        ".pdf",
        ".svg",
    ]:

        output_file = (
            base_file.with_suffix(
                extension
            )
        )

        if extension == ".png":
            fig.savefig(
                output_file,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        else:
            fig.savefig(
                output_file,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        saved_files.append(
            output_file
        )


plt.show()
plt.close(fig)


# ============================================================
# 9. FIGURE AUDIT TABLE
# ============================================================

audit_columns = [
    "model",
    "dataset",
    "n",
    "mean_hotspot_probability_drop",
    "hotspot_drop_ci95_low",
    "hotspot_drop_ci95_high",
    "mean_random_probability_drop",
    "mean_hotspot_minus_random_drop",
    "difference_ci95_low",
    "difference_ci95_high",
    "fraction_hotspot_stronger_than_random",
    "paired_cohens_dz",
    "rank_biserial_effect_size",
    "wilcoxon_p_value",
    "mean_hotspot_only_probability",
]

figure_audit_df = summary[
    audit_columns
].copy()

figure_audit_file = (
    CHECKPOINT_DIR
    / "figure_5_faithfulness_audit.csv"
)

figure_audit_df.to_csv(
    figure_audit_file,
    index=False,
)


# ============================================================
# 10. CHECKPOINT
# ============================================================

checkpoint_details = {
    "figure": "Figure 5",

    "title": (
        "Perturbation-based faithfulness "
        "of consensus CPP hotspots"
    ),

    "panels": {
        "A": (
            "Internal CPP hotspot vs random ablation"
        ),
        "B": (
            "KELM CPP hotspot vs random ablation"
        ),
        "C": (
            "Mean hotspot-minus-random effect "
            "with 95% confidence intervals"
        ),
        "D": (
            "Hotspot-only predictive sufficiency"
        ),
    },

    "important_interpretation": (
        "Internal faithfulness was significant for "
        "all PLMs; external faithfulness was "
        "model dependent and strongest for ProtT5."
    ),

    "excluded_metric": (
        "mean_hotspot_only_probability_fraction"
    ),

    "formats": [
        "PNG 600 dpi",
        "PDF",
        "SVG",
    ],
}


try:
    mark_step_complete(
        "26_figure_5_hotspot_faithfulness",
        output_files=(
            saved_files
            + [
                figure_audit_file,
            ]
        ),
        details=checkpoint_details,
    )

except NameError:
    print(
        "mark_step_complete() was not found, "
        "but Figure 5 was saved successfully."
    )


# ============================================================
# 11. DISPLAY OUTPUTS
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 5 FAITHFULNESS SUMMARY")
print("=" * 78)

display(
    figure_audit_df
)


print("\n" + "=" * 78)
print("FIGURE 5 COMPLETED")
print("=" * 78)

for file_path in saved_files:
    print(
        file_path
    )

In [ ]:
# ============================================================
# STEP 27: FIGURE 6 — CROSS-PLM HOTSPOT CONSERVATION
#
# Panels:
# A. Pairwise PLM hotspot Jaccard agreement
# B. Residue support by 1–4 PLMs
# C. CPP enrichment of multi-PLM-supported residues
# D. Sequence-level hotspot conservation
#
# Exports:
# PNG 600 dpi
# PDF vector
# SVG vector
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

TABLE_SI_DIR = (
    RESULTS_DIR / "tables_SI"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

CHECKPOINT_DIR = (
    PROJECT_DIR / "08_checkpoints"
)

for directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. STYLE
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "font.weight": "bold",

    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 8.5,
    "legend.frameon": False,

    "lines.linewidth": 2.0,
    "lines.markersize": 6,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})


# ============================================================
# 3. INPUT FILES
# ============================================================

PAIRWISE_FILE = (
    TABLE_MAIN_DIR
    / "cross_plm_pairwise_hotspot_jaccard.csv"
)

SUPPORT_FILE = (
    TABLE_MAIN_DIR
    / "cross_plm_hotspot_support_distribution.csv"
)

RESIDUE_ENRICHMENT_FILE = (
    TABLE_MAIN_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

CLASS_COMPARISON_FILE = (
    TABLE_MAIN_DIR
    / "CPP_vs_nonCPP_hotspot_conservation.csv"
)

PER_SEQUENCE_FILE = (
    TABLE_SI_DIR
    / "cross_plm_hotspot_conservation_per_sequence.csv"
)

for required_file in [
    PAIRWISE_FILE,
    SUPPORT_FILE,
    RESIDUE_ENRICHMENT_FILE,
    CLASS_COMPARISON_FILE,
    PER_SEQUENCE_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file missing:\n{required_file}"
        )


pairwise = pd.read_csv(
    PAIRWISE_FILE
)

support = pd.read_csv(
    SUPPORT_FILE
)

residue_enrichment = pd.read_csv(
    RESIDUE_ENRICHMENT_FILE
)

class_comparison = pd.read_csv(
    CLASS_COMPARISON_FILE
)

per_sequence = pd.read_csv(
    PER_SEQUENCE_FILE
)


# ============================================================
# 4. CONFIGURATION
# ============================================================

MODEL_ORDER = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_DISPLAY = {
    "ESM2_320": "ESM2-320",
    "ESM2_640": "ESM2-640",
    "ESM2_1280": "ESM2-1280",
    "ProtT5": "ProtT5",
}

DATASET_DISPLAY = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}


# ============================================================
# 5. HELPERS
# ============================================================

def add_panel_label(
    axis,
    label,
):
    axis.text(
        -0.13,
        1.08,
        label,
        transform=axis.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )


def style_axis(
    axis,
):
    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        width=1.1,
        length=4,
    )


def build_jaccard_matrix(
    dataset_name,
):
    matrix = pd.DataFrame(
        np.eye(
            len(MODEL_ORDER)
        ),
        index=MODEL_ORDER,
        columns=MODEL_ORDER,
    )

    subset = pairwise[
        pairwise["dataset"]
        == dataset_name
    ]

    for row in subset.itertuples(
        index=False
    ):
        value = float(
            row.mean_jaccard_CPP
        )

        matrix.loc[
            row.model_a,
            row.model_b,
        ] = value

        matrix.loc[
            row.model_b,
            row.model_a,
        ] = value

    return matrix


internal_matrix = build_jaccard_matrix(
    "internal_test"
)

kelm_matrix = build_jaccard_matrix(
    "kelm_external"
)


# ============================================================
# 6. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(15.0, 9.5)
)

grid = fig.add_gridspec(
    nrows=2,
    ncols=2,
    width_ratios=[
        1.0,
        1.0,
    ],
    height_ratios=[
        1.0,
        1.0,
    ],
    wspace=0.28,
    hspace=0.34,
)

ax_a = fig.add_subplot(
    grid[0, 0]
)

ax_b = fig.add_subplot(
    grid[0, 1]
)

ax_c = fig.add_subplot(
    grid[1, 0]
)

ax_d = fig.add_subplot(
    grid[1, 1]
)


# ============================================================
# PANEL A — PAIRWISE CPP HOTSPOT JACCARD
# ============================================================

combined_matrix = np.block([
    [
        internal_matrix.to_numpy(),
        np.full(
            (
                len(MODEL_ORDER),
                1,
            ),
            np.nan,
        ),
        kelm_matrix.to_numpy(),
    ]
])

masked_matrix = np.ma.masked_invalid(
    combined_matrix
)

image_a = ax_a.imshow(
    masked_matrix,
    vmin=0,
    vmax=1,
    cmap="Blues",
    aspect="auto",
)

separator_index = len(
    MODEL_ORDER
)

ax_a.axvline(
    separator_index - 0.5,
    linewidth=2.0,
)

x_labels = (
    [
        MODEL_DISPLAY[model]
        for model in MODEL_ORDER
    ]
    + [""]
    + [
        MODEL_DISPLAY[model]
        for model in MODEL_ORDER
    ]
)

ax_a.set_xticks(
    np.arange(
        len(x_labels)
    )
)

ax_a.set_xticklabels(
    x_labels,
    rotation=45,
    ha="right",
    fontsize=7.5,
    fontweight="bold",
)

ax_a.set_yticks(
    np.arange(
        len(MODEL_ORDER)
    )
)

ax_a.set_yticklabels(
    [
        MODEL_DISPLAY[model]
        for model in MODEL_ORDER
    ],
    fontsize=8.5,
    fontweight="bold",
)

for row_index in range(
    len(MODEL_ORDER)
):
    for column_index in range(
        len(MODEL_ORDER)
    ):
        ax_a.text(
            column_index,
            row_index,
            f"{internal_matrix.iloc[row_index, column_index]:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            fontweight="bold",
        )

        ax_a.text(
            column_index
            + len(MODEL_ORDER)
            + 1,
            row_index,
            f"{kelm_matrix.iloc[row_index, column_index]:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            fontweight="bold",
        )

ax_a.text(
    1.5,
    -0.85,
    "Internal test",
    ha="center",
    va="center",
    fontsize=9.5,
    fontweight="bold",
)

ax_a.text(
    6.5,
    -0.85,
    "KELM external",
    ha="center",
    va="center",
    fontsize=9.5,
    fontweight="bold",
)

ax_a.set_title(
    "Pairwise CPP hotspot Jaccard agreement"
)

colorbar_a = fig.colorbar(
    image_a,
    ax=ax_a,
    fraction=0.046,
    pad=0.03,
)

colorbar_a.set_label(
    "Mean per-sequence Jaccard",
    fontweight="bold",
)

add_panel_label(
    ax_a,
    "A",
)


# ============================================================
# PANEL B — SUPPORT BY 1–4 PLMS
# ============================================================

support_plot = support[
    support["class"].isin(
        [
            "CPP",
            "non_CPP",
        ]
    )
    & (
        support["model_support_count"]
        > 0
    )
].copy()

support_counts = [
    1,
    2,
    3,
    4,
]

x = np.arange(
    len(support_counts)
)

width = 0.18

plot_specs = [
    (
        "internal_test",
        "CPP",
        -1.5 * width,
        "Internal CPP",
    ),
    (
        "internal_test",
        "non_CPP",
        -0.5 * width,
        "Internal non-CPP",
    ),
    (
        "kelm_external",
        "CPP",
        0.5 * width,
        "KELM CPP",
    ),
    (
        "kelm_external",
        "non_CPP",
        1.5 * width,
        "KELM non-CPP",
    ),
]

for dataset_name, class_name, offset, label in plot_specs:

    values = []

    for support_count in support_counts:

        row = support_plot[
            (
                support_plot["dataset"]
                == dataset_name
            )
            & (
                support_plot["class"]
                == class_name
            )
            & (
                support_plot[
                    "model_support_count"
                ] == support_count
            )
        ]

        values.append(
            float(
                row[
                    "fraction_hotspot_union"
                ].iloc[0]
            )
        )

    ax_b.bar(
        x + offset,
        values,
        width=width,
        label=label,
        edgecolor="black",
        linewidth=0.6,
    )

ax_b.set_xticks(
    x
)

ax_b.set_xticklabels(
    [
        "1 PLM",
        "2 PLMs",
        "3 PLMs",
        "4 PLMs",
    ],
    fontweight="bold",
)

ax_b.set_ylabel(
    "Fraction of hotspot-union residues"
)

ax_b.set_title(
    "Cross-PLM support of hotspot residues"
)

ax_b.legend(
    loc="upper right",
    ncol=2,
    fontsize=7.5,
)

style_axis(
    ax_b
)

add_panel_label(
    ax_b,
    "B",
)


# ============================================================
# PANEL C — RESIDUE-LEVEL CONSERVATION ENRICHMENT
# ============================================================

enrichment_plot = residue_enrichment[
    residue_enrichment[
        "minimum_model_support"
    ].isin(
        [
            2,
            3,
            4,
        ]
    )
].copy()

support_label_map = {
    2: "≥2 PLMs",
    3: "≥3 PLMs",
    4: "All 4 PLMs",
}

x = np.arange(
    3
)

width = 0.34

for dataset_index, dataset_name in enumerate(
    [
        "internal_test",
        "kelm_external",
    ]
):

    subset = enrichment_plot[
        enrichment_plot["dataset"]
        == dataset_name
    ].sort_values(
        "minimum_model_support"
    )

    odds_ratios = subset[
        "odds_ratio_corrected"
    ].to_numpy()

    positions = (
        x
        + (
            dataset_index - 0.5
        )
        * width
    )

    bars = ax_c.bar(
        positions,
        odds_ratios,
        width=width,
        label=DATASET_DISPLAY[
            dataset_name
        ],
        edgecolor="black",
        linewidth=0.7,
    )

    for bar, value, p_value in zip(
        bars,
        odds_ratios,
        subset["p_value"],
    ):

        significance = (
            "***"
            if p_value < 0.001
            else (
                "**"
                if p_value < 0.01
                else (
                    "*"
                    if p_value < 0.05
                    else "ns"
                )
            )
        )

        ax_c.text(
            bar.get_x()
            + bar.get_width() / 2,
            value + 0.08,
            significance,
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

ax_c.axhline(
    1,
    linestyle="--",
    linewidth=1.1,
)

ax_c.set_xticks(
    x
)

ax_c.set_xticklabels(
    [
        support_label_map[
            support_count
        ]
        for support_count in [
            2,
            3,
            4,
        ]
    ],
    fontweight="bold",
)

ax_c.set_ylabel(
    "CPP enrichment odds ratio"
)

ax_c.set_title(
    "CPP enrichment increases with PLM agreement"
)

ax_c.legend(
    loc="upper left"
)

style_axis(
    ax_c
)

add_panel_label(
    ax_c,
    "C",
)


# ============================================================
# PANEL D — SEQUENCE-LEVEL CONSERVATION
# ============================================================

metric_order = [
    "mean_pairwise_jaccard",
    "fraction_supported_by_at_least_3",
    "fraction_supported_by_all_4",
]

metric_display = {
    "mean_pairwise_jaccard":
        "Mean pairwise\nJaccard",

    "fraction_supported_by_at_least_3":
        "Fraction supported\nby ≥3 PLMs",

    "fraction_supported_by_all_4":
        "Fraction supported\nby all 4 PLMs",
}

group_specs = [
    (
        "internal_test",
        0,
        "Internal non-CPP",
    ),
    (
        "internal_test",
        1,
        "Internal CPP",
    ),
    (
        "kelm_external",
        0,
        "KELM non-CPP",
    ),
    (
        "kelm_external",
        1,
        "KELM CPP",
    ),
]

base_positions = np.arange(
    len(metric_order)
) * 5

box_positions = []

box_values = []

box_labels = []

for metric_index, metric_name in enumerate(
    metric_order
):

    for group_index, (
        dataset_name,
        label_value,
        group_label,
    ) in enumerate(
        group_specs
    ):

        values = per_sequence[
            (
                per_sequence["dataset"]
                == dataset_name
            )
            & (
                per_sequence["label"]
                == label_value
            )
        ][
            metric_name
        ].to_numpy()

        position = (
            base_positions[
                metric_index
            ]
            + group_index * 0.85
        )

        box_positions.append(
            position
        )

        box_values.append(
            values
        )

        box_labels.append(
            group_label
        )


boxplot = ax_d.boxplot(
    box_values,
    positions=box_positions,
    widths=0.62,
    patch_artist=True,
    showfliers=False,
    medianprops={
        "linewidth": 1.5,
    },
    boxprops={
        "linewidth": 0.9,
    },
    whiskerprops={
        "linewidth": 0.9,
    },
    capprops={
        "linewidth": 0.9,
    },
)

facecolors = [
    "#D9D9D9",
    "#A6A6A6",
    "white",
    "#BFBFBF",
]

for box_index, box in enumerate(
    boxplot["boxes"]
):
    box.set_facecolor(
        facecolors[
            box_index % 4
        ]
    )

    box.set_edgecolor(
        "black"
    )


metric_centers = [
    base_position + 1.275
    for base_position in base_positions
]

ax_d.set_xticks(
    metric_centers
)

ax_d.set_xticklabels(
    [
        metric_display[
            metric_name
        ]
        for metric_name in metric_order
    ],
    fontweight="bold",
)

ax_d.set_ylabel(
    "Sequence-level conservation"
)

ax_d.set_title(
    "CPP hotspots show greater sequence-level conservation"
)

legend_handles = [
    mpl.patches.Patch(
        facecolor=facecolors[index],
        edgecolor="black",
        label=group_specs[index][2],
    )
    for index in range(
        4
    )
]

ax_d.legend(
    handles=legend_handles,
    loc="upper right",
    fontsize=7.5,
    ncol=2,
)

style_axis(
    ax_d
)

add_panel_label(
    ax_d,
    "D",
)


# ============================================================
# 7. FINAL LAYOUT
# ============================================================

fig.suptitle(
    "Cross-PLM Conservation of Consensus CPP Hotspots",
    fontsize=16,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    left=0.075,
    right=0.98,
    top=0.92,
    bottom=0.09,
)


# ============================================================
# 8. SAVE FIGURE
# ============================================================

figure_stem = (
    "Figure_6_Cross_PLM_Hotspot_Conservation"
)

saved_files = []

for output_directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:

    base_file = (
        output_directory
        / figure_stem
    )

    for extension in [
        ".png",
        ".pdf",
        ".svg",
    ]:

        output_file = (
            base_file.with_suffix(
                extension
            )
        )

        if extension == ".png":
            fig.savefig(
                output_file,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04,
            )
        else:
            fig.savefig(
                output_file,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        saved_files.append(
            output_file
        )


plt.show()
plt.close(fig)


# ============================================================
# 9. SAVE FIGURE AUDIT
# ============================================================

figure_audit_file = (
    CHECKPOINT_DIR
    / "figure_6_conservation_audit.xlsx"
)

with pd.ExcelWriter(
    figure_audit_file,
    engine="openpyxl",
) as writer:

    pairwise.to_excel(
        writer,
        sheet_name="Pairwise Jaccard",
        index=False,
    )

    support.to_excel(
        writer,
        sheet_name="Support Distribution",
        index=False,
    )

    residue_enrichment.to_excel(
        writer,
        sheet_name="Residue Enrichment",
        index=False,
    )

    class_comparison.to_excel(
        writer,
        sheet_name="Class Comparison",
        index=False,
    )


# ============================================================
# 10. CHECKPOINT
# ============================================================

checkpoint_details = {
    "figure": "Figure 6",

    "title": (
        "Cross-PLM conservation of "
        "consensus CPP hotspots"
    ),

    "panels": {
        "A": (
            "Pairwise CPP hotspot Jaccard agreement"
        ),
        "B": (
            "Hotspot support by one to four PLMs"
        ),
        "C": (
            "CPP enrichment of multi-PLM-supported residues"
        ),
        "D": (
            "Sequence-level CPP versus non-CPP conservation"
        ),
    },

    "main_result": (
        "Residues supported by three or four PLMs "
        "were significantly enriched in CPPs in both "
        "the internal and external datasets."
    ),

    "formats": [
        "PNG 600 dpi",
        "PDF",
        "SVG",
    ],
}


try:
    mark_step_complete(
        "27_figure_6_cross_PLM_conservation",
        output_files=(
            saved_files
            + [
                figure_audit_file,
            ]
        ),
        details=checkpoint_details,
    )

except NameError:
    print(
        "mark_step_complete() was not found, "
        "but Figure 6 was saved successfully."
    )


# ============================================================
# 11. DISPLAY OUTPUTS
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 6 RESIDUE-LEVEL ENRICHMENT")
print("=" * 78)

display(
    residue_enrichment[
        [
            "dataset",
            "minimum_model_support",
            "CPP_supported_fraction",
            "nonCPP_supported_fraction",
            "odds_ratio_corrected",
            "p_value",
        ]
    ]
)


print("\n" + "=" * 78)
print("FIGURE 6 SEQUENCE-LEVEL COMPARISON")
print("=" * 78)

display(
    class_comparison[
        [
            "dataset",
            "metric",
            "CPP_mean",
            "nonCPP_mean",
            "p_value",
        ]
    ]
)


print("\n" + "=" * 78)
print("FIGURE 6 COMPLETED")
print("=" * 78)

for file_path in saved_files:
    print(
        file_path
    )

In [ ]:
# ============================================================
# STEP 28: FIGURE 7 — PHYSICOCHEMICAL AND POSITIONAL GRAMMAR
#
# Panels:
# A. Replicated physicochemical class enrichment
# B. Sequence-level charge and hydropathy
# C. Regional positional enrichment
# D. Detailed 10-bin CPP hotspot positional profile
#
# Exports:
# PNG 600 dpi
# PDF vector
# SVG vector
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# ============================================================
# 1. DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

RESULTS_DIR = (
    PROJECT_DIR / "07_results"
)

TABLE_MAIN_DIR = (
    RESULTS_DIR / "tables_main"
)

TABLE_SI_DIR = (
    RESULTS_DIR / "tables_SI"
)

FIGURE_MAIN_DIR = (
    RESULTS_DIR / "figures_main"
)

FINAL_FIGURE_DIR = (
    RESULTS_DIR
    / "final_manuscript_package"
    / "03_main_figures"
)

CHECKPOINT_DIR = (
    PROJECT_DIR / "08_checkpoints"
)

for directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 2. PUBLICATION STYLE
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "font.weight": "bold",

    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.2,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,

    "legend.fontsize": 8.5,
    "legend.frameon": False,

    "lines.linewidth": 2.0,
    "lines.markersize": 6,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})


# ============================================================
# 3. INPUT FILES
# ============================================================

PHYSICOCHEMICAL_FILE = (
    TABLE_MAIN_DIR
    / "physicochemical_category_enrichment.csv"
)

PHYSICOCHEMICAL_REPLICATION_FILE = (
    TABLE_MAIN_DIR
    / "physicochemical_enrichment_replication.csv"
)

PHYSICOCHEMICAL_SEQUENCE_FILE = (
    TABLE_SI_DIR
    / "physicochemical_properties_per_sequence.csv"
)

PHYSICOCHEMICAL_TEST_FILE = (
    TABLE_MAIN_DIR
    / "physicochemical_sequence_level_tests.csv"
)

POSITIONAL_REGION_FILE = (
    TABLE_MAIN_DIR
    / "hotspot_positional_region_enrichment.csv"
)

POSITIONAL_REPLICATION_FILE = (
    TABLE_MAIN_DIR
    / "hotspot_positional_replication.csv"
)

POSITIONAL_BIN_FILE = (
    TABLE_MAIN_DIR
    / "hotspot_position_10bin_distribution.csv"
)

for required_file in [
    PHYSICOCHEMICAL_FILE,
    PHYSICOCHEMICAL_REPLICATION_FILE,
    PHYSICOCHEMICAL_SEQUENCE_FILE,
    PHYSICOCHEMICAL_TEST_FILE,
    POSITIONAL_REGION_FILE,
    POSITIONAL_REPLICATION_FILE,
    POSITIONAL_BIN_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file missing:\n{required_file}"
        )


physicochemical = pd.read_csv(
    PHYSICOCHEMICAL_FILE
)

physicochemical_replication = pd.read_csv(
    PHYSICOCHEMICAL_REPLICATION_FILE
)

physicochemical_sequence = pd.read_csv(
    PHYSICOCHEMICAL_SEQUENCE_FILE
)

physicochemical_tests = pd.read_csv(
    PHYSICOCHEMICAL_TEST_FILE
)

positional_region = pd.read_csv(
    POSITIONAL_REGION_FILE
)

positional_replication = pd.read_csv(
    POSITIONAL_REPLICATION_FILE
)

positional_bins = pd.read_csv(
    POSITIONAL_BIN_FILE
)


# ============================================================
# 4. CONFIGURATION
# ============================================================

DATASET_DISPLAY = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}

PROPERTY_ORDER = [
    "basic_positive",
    "acidic_negative",
    "polar_uncharged",
    "aromatic",
    "aliphatic_hydrophobic",
    "structure_special",
]

PROPERTY_DISPLAY = {
    "basic_positive": "Basic\npositive",
    "acidic_negative": "Acidic\nnegative",
    "polar_uncharged": "Polar\nuncharged",
    "aromatic": "Aromatic",
    "aliphatic_hydrophobic": "Aliphatic\nhydrophobic",
    "structure_special": "Structure\nspecial",
}

REGION_ORDER = [
    "N_terminal",
    "Middle",
    "C_terminal",
]

REGION_DISPLAY = {
    "N_terminal": "N-terminal",
    "Middle": "Middle",
    "C_terminal": "C-terminal",
}


# ============================================================
# 5. HELPERS
# ============================================================

def add_panel_label(
    axis,
    label,
):
    axis.text(
        -0.13,
        1.08,
        label,
        transform=axis.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )


def style_axis(
    axis,
):
    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        width=1.1,
        length=4,
    )


def significance_symbol(
    p_value,
):
    p_value = float(
        p_value
    )

    if p_value < 0.001:
        return "***"

    if p_value < 0.01:
        return "**"

    if p_value < 0.05:
        return "*"

    return "ns"


# ============================================================
# 6. PREPARE PANEL A DATA
# ============================================================

phys_plot = physicochemical[
    physicochemical[
        "analysis"
    ] == "CPP_hotspot_vs_CPP_nonhotspot"
].copy()

phys_plot[
    "property_class"
] = pd.Categorical(
    phys_plot[
        "property_class"
    ],
    categories=PROPERTY_ORDER,
    ordered=True,
)

phys_plot = phys_plot.sort_values(
    [
        "property_class",
        "dataset",
    ]
)


# ============================================================
# 7. PREPARE PANEL B DATA
# ============================================================

cpp_sequence = physicochemical_sequence[
    physicochemical_sequence[
        "label"
    ] == 1
].copy()

continuous_properties = [
    (
        "charge_score",
        "Mean charge score",
    ),
    (
        "hydropathy_KD",
        "Mean hydropathy",
    ),
]


# ============================================================
# 8. PREPARE PANEL C DATA
# ============================================================

region_plot = positional_region[
    positional_region[
        "class"
    ] == "CPP"
].copy()

region_plot[
    "region"
] = pd.Categorical(
    region_plot[
        "region"
    ],
    categories=REGION_ORDER,
    ordered=True,
)

region_plot = region_plot.sort_values(
    [
        "region",
        "dataset",
    ]
)


# ============================================================
# 9. PREPARE PANEL D DATA
# ============================================================

bin_plot = positional_bins[
    positional_bins[
        "class"
    ] == "CPP"
].copy()

bin_plot = bin_plot.sort_values(
    [
        "dataset",
        "position_bin",
    ]
)


# ============================================================
# 10. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(15.0, 9.5)
)

grid = fig.add_gridspec(
    nrows=2,
    ncols=2,
    width_ratios=[
        1.0,
        1.0,
    ],
    height_ratios=[
        1.0,
        1.0,
    ],
    wspace=0.30,
    hspace=0.37,
)

ax_a = fig.add_subplot(
    grid[0, 0]
)

ax_b = fig.add_subplot(
    grid[0, 1]
)

ax_c = fig.add_subplot(
    grid[1, 0]
)

ax_d = fig.add_subplot(
    grid[1, 1]
)


# ============================================================
# PANEL A — PHYSICOCHEMICAL CLASS ENRICHMENT
# ============================================================

x = np.arange(
    len(
        PROPERTY_ORDER
    )
)

width = 0.36

internal_values = []
kelm_values = []

internal_significance = []
kelm_significance = []

for property_name in PROPERTY_ORDER:

    internal_row = phys_plot[
        (
            phys_plot[
                "dataset"
            ] == "internal_test"
        )
        & (
            phys_plot[
                "property_class"
            ] == property_name
        )
    ].iloc[0]

    kelm_row = phys_plot[
        (
            phys_plot[
                "dataset"
            ] == "kelm_external"
        )
        & (
            phys_plot[
                "property_class"
            ] == property_name
        )
    ].iloc[0]

    internal_values.append(
        float(
            internal_row[
                "log2_enrichment"
            ]
        )
    )

    kelm_values.append(
        float(
            kelm_row[
                "log2_enrichment"
            ]
        )
    )

    internal_significance.append(
        significance_symbol(
            internal_row[
                "fdr_bh"
            ]
        )
    )

    kelm_significance.append(
        significance_symbol(
            kelm_row[
                "fdr_bh"
            ]
        )
    )


bars_internal = ax_a.bar(
    x - width / 2,
    internal_values,
    width=width,
    label="Internal test",
    edgecolor="black",
    linewidth=0.7,
)

bars_kelm = ax_a.bar(
    x + width / 2,
    kelm_values,
    width=width,
    label="KELM external",
    edgecolor="black",
    linewidth=0.7,
)

ax_a.axhline(
    0,
    linestyle="--",
    linewidth=1.1,
)

ax_a.set_xticks(
    x
)

ax_a.set_xticklabels(
    [
        PROPERTY_DISPLAY[
            property_name
        ]
        for property_name
        in PROPERTY_ORDER
    ],
    fontsize=8,
    fontweight="bold",
)

ax_a.set_ylabel(
    "log$_2$ enrichment in CPP hotspots"
)

ax_a.set_title(
    "Physicochemical composition of CPP hotspots"
)

ax_a.legend(
    loc="lower left"
)

for bar, symbol in zip(
    bars_internal,
    internal_significance,
):

    height = bar.get_height()

    ax_a.text(
        bar.get_x()
        + bar.get_width() / 2,
        height
        + (
            0.10
            if height >= 0
            else -0.18
        ),
        symbol,
        ha="center",
        va=(
            "bottom"
            if height >= 0
            else "top"
        ),
        fontsize=8,
        fontweight="bold",
    )


for bar, symbol in zip(
    bars_kelm,
    kelm_significance,
):

    height = bar.get_height()

    ax_a.text(
        bar.get_x()
        + bar.get_width() / 2,
        height
        + (
            0.10
            if height >= 0
            else -0.18
        ),
        symbol,
        ha="center",
        va=(
            "bottom"
            if height >= 0
            else "top"
        ),
        fontsize=8,
        fontweight="bold",
    )


style_axis(
    ax_a
)

add_panel_label(
    ax_a,
    "A",
)


# ============================================================
# PANEL B — CHARGE AND HYDROPATHY
# ============================================================

box_values = []
box_positions = []
box_labels = []

property_centers = []

for property_index, (
    property_name,
    property_label,
) in enumerate(
    continuous_properties
):

    base_position = (
        property_index * 5
    )

    property_centers.append(
        base_position + 1.35
    )

    groups = [
        (
            "internal_test",
            f"hotspot_{property_name}",
            "Internal hotspot",
        ),
        (
            "internal_test",
            f"nonhotspot_{property_name}",
            "Internal non-hotspot",
        ),
        (
            "kelm_external",
            f"hotspot_{property_name}",
            "KELM hotspot",
        ),
        (
            "kelm_external",
            f"nonhotspot_{property_name}",
            "KELM non-hotspot",
        ),
    ]

    for group_index, (
        dataset_name,
        column_name,
        group_label,
    ) in enumerate(
        groups
    ):

        values = cpp_sequence[
            cpp_sequence[
                "dataset"
            ] == dataset_name
        ][
            column_name
        ].dropna().to_numpy()

        position = (
            base_position
            + group_index * 0.9
        )

        box_values.append(
            values
        )

        box_positions.append(
            position
        )

        box_labels.append(
            group_label
        )


boxplot = ax_b.boxplot(
    box_values,
    positions=box_positions,
    widths=0.65,
    patch_artist=True,
    showfliers=False,
    medianprops={
        "linewidth": 1.5,
    },
    boxprops={
        "linewidth": 0.9,
    },
    whiskerprops={
        "linewidth": 0.9,
    },
    capprops={
        "linewidth": 0.9,
    },
)

box_facecolors = [
    "#A6A6A6",
    "#E0E0E0",
    "#737373",
    "white",
]

for box_index, box in enumerate(
    boxplot[
        "boxes"
    ]
):
    box.set_facecolor(
        box_facecolors[
            box_index % 4
        ]
    )

    box.set_edgecolor(
        "black"
    )


ax_b.axhline(
    0,
    linestyle="--",
    linewidth=1.0,
)

ax_b.set_xticks(
    property_centers
)

ax_b.set_xticklabels(
    [
        "Mean charge score",
        "Mean hydropathy",
    ],
    fontweight="bold",
)

ax_b.set_ylabel(
    "Sequence-level property value"
)

ax_b.set_title(
    "CPP hotspots are more charged and hydrophilic"
)

legend_handles = [
    mpl.patches.Patch(
        facecolor=box_facecolors[index],
        edgecolor="black",
        label=label,
    )
    for index, label in enumerate([
        "Internal hotspot",
        "Internal non-hotspot",
        "KELM hotspot",
        "KELM non-hotspot",
    ])
]

ax_b.legend(
    handles=legend_handles,
    loc="upper right",
    fontsize=7.5,
    ncol=2,
)

style_axis(
    ax_b
)

add_panel_label(
    ax_b,
    "B",
)


# ============================================================
# PANEL C — REGIONAL POSITIONAL ENRICHMENT
# ============================================================

x = np.arange(
    len(
        REGION_ORDER
    )
)

width = 0.36

internal_region_values = []
kelm_region_values = []

internal_region_significance = []
kelm_region_significance = []

for region_name in REGION_ORDER:

    internal_row = region_plot[
        (
            region_plot[
                "dataset"
            ] == "internal_test"
        )
        & (
            region_plot[
                "region"
            ] == region_name
        )
    ].iloc[0]

    kelm_row = region_plot[
        (
            region_plot[
                "dataset"
            ] == "kelm_external"
        )
        & (
            region_plot[
                "region"
            ] == region_name
        )
    ].iloc[0]

    internal_region_values.append(
        float(
            internal_row[
                "log2_enrichment"
            ]
        )
    )

    kelm_region_values.append(
        float(
            kelm_row[
                "log2_enrichment"
            ]
        )
    )

    internal_region_significance.append(
        significance_symbol(
            internal_row[
                "fdr_bh"
            ]
        )
    )

    kelm_region_significance.append(
        significance_symbol(
            kelm_row[
                "fdr_bh"
            ]
        )
    )


bars_internal_region = ax_c.bar(
    x - width / 2,
    internal_region_values,
    width=width,
    label="Internal test",
    edgecolor="black",
    linewidth=0.7,
)

bars_kelm_region = ax_c.bar(
    x + width / 2,
    kelm_region_values,
    width=width,
    label="KELM external",
    edgecolor="black",
    linewidth=0.7,
)

ax_c.axhline(
    0,
    linestyle="--",
    linewidth=1.1,
)

ax_c.set_xticks(
    x
)

ax_c.set_xticklabels(
    [
        REGION_DISPLAY[
            region_name
        ]
        for region_name
        in REGION_ORDER
    ],
    fontweight="bold",
)

ax_c.set_ylabel(
    "log$_2$ enrichment in CPP hotspots"
)

ax_c.set_title(
    "Regional organization of CPP hotspots"
)

ax_c.legend(
    loc="lower left"
)


for bar, symbol in zip(
    bars_internal_region,
    internal_region_significance,
):

    height = bar.get_height()

    ax_c.text(
        bar.get_x()
        + bar.get_width() / 2,
        height
        + (
            0.035
            if height >= 0
            else -0.06
        ),
        symbol,
        ha="center",
        va=(
            "bottom"
            if height >= 0
            else "top"
        ),
        fontsize=8,
        fontweight="bold",
    )


for bar, symbol in zip(
    bars_kelm_region,
    kelm_region_significance,
):

    height = bar.get_height()

    ax_c.text(
        bar.get_x()
        + bar.get_width() / 2,
        height
        + (
            0.035
            if height >= 0
            else -0.06
        ),
        symbol,
        ha="center",
        va=(
            "bottom"
            if height >= 0
            else "top"
        ),
        fontsize=8,
        fontweight="bold",
    )


style_axis(
    ax_c
)

add_panel_label(
    ax_c,
    "C",
)


# ============================================================
# PANEL D — DETAILED POSITIONAL PROFILE
# ============================================================

for dataset_name, marker_style in [
    (
        "internal_test",
        "o",
    ),
    (
        "kelm_external",
        "s",
    ),
]:

    subset = bin_plot[
        bin_plot[
            "dataset"
        ] == dataset_name
    ].sort_values(
        "position_bin"
    )

    ax_d.plot(
        subset[
            "position_bin"
        ],
        subset[
            "hotspot_to_nonhotspot_ratio"
        ],
        marker=marker_style,
        linewidth=2.0,
        label=DATASET_DISPLAY[
            dataset_name
        ],
    )


ax_d.axhline(
    1,
    linestyle="--",
    linewidth=1.1,
)

ax_d.axvspan(
    3.0,
    7.5,
    alpha=0.08,
)

ax_d.set_xticks(
    np.arange(
        1,
        11,
    )
)

ax_d.set_xticklabels(
    [
        "0–10",
        "10–20",
        "20–30",
        "30–40",
        "40–50",
        "50–60",
        "60–70",
        "70–80",
        "80–90",
        "90–100",
    ],
    rotation=35,
    ha="right",
    fontsize=8,
    fontweight="bold",
)

ax_d.set_xlabel(
    "Relative sequence position (%)"
)

ax_d.set_ylabel(
    "Hotspot/non-hotspot frequency ratio"
)

ax_d.set_title(
    "Detailed positional profile of CPP hotspots"
)

ax_d.legend(
    loc="upper right"
)

ax_d.text(
    5.25,
    ax_d.get_ylim()[1] * 0.95,
    "Central region",
    ha="center",
    va="top",
    fontsize=8.5,
    fontweight="bold",
)

style_axis(
    ax_d
)

add_panel_label(
    ax_d,
    "D",
)


# ============================================================
# 11. FINAL LAYOUT
# ============================================================

fig.suptitle(
    "Physicochemical and Positional Organization "
    "of Consensus CPP Hotspots",
    fontsize=16,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    left=0.075,
    right=0.98,
    top=0.92,
    bottom=0.09,
)


# ============================================================
# 12. SAVE FIGURE
# ============================================================

figure_stem = (
    "Figure_7_Physicochemical_and_Positional_Grammar"
)

saved_files = []

for output_directory in [
    FIGURE_MAIN_DIR,
    FINAL_FIGURE_DIR,
]:

    base_file = (
        output_directory
        / figure_stem
    )

    for extension in [
        ".png",
        ".pdf",
        ".svg",
    ]:

        output_file = (
            base_file.with_suffix(
                extension
            )
        )

        if extension == ".png":

            fig.savefig(
                output_file,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        else:

            fig.savefig(
                output_file,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        saved_files.append(
            output_file
        )


plt.show()
plt.close(fig)


# ============================================================
# 13. SAVE FIGURE AUDIT WORKBOOK
# ============================================================

figure_audit_file = (
    CHECKPOINT_DIR
    / "figure_7_physicochemical_positional_audit.xlsx"
)

with pd.ExcelWriter(
    figure_audit_file,
    engine="openpyxl",
) as writer:

    physicochemical.to_excel(
        writer,
        sheet_name="Physicochemical Enrichment",
        index=False,
    )

    physicochemical_tests.to_excel(
        writer,
        sheet_name="Sequence Tests",
        index=False,
    )

    positional_region.to_excel(
        writer,
        sheet_name="Regional Enrichment",
        index=False,
    )

    positional_bins.to_excel(
        writer,
        sheet_name="Position Bins",
        index=False,
    )


# ============================================================
# 14. CHECKPOINT
# ============================================================

checkpoint_details = {
    "figure": "Figure 7",

    "title": (
        "Physicochemical and positional "
        "organization of consensus CPP hotspots"
    ),

    "panels": {
        "A": (
            "Physicochemical class enrichment"
        ),
        "B": (
            "Sequence-level charge and hydropathy"
        ),
        "C": (
            "N-terminal, middle, and C-terminal "
            "regional enrichment"
        ),
        "D": (
            "Detailed 10-bin positional profile"
        ),
    },

    "main_findings": [
        (
            "Basic positively charged residues were "
            "enriched in both datasets."
        ),
        (
            "CPP hotspots were more hydrophilic than "
            "non-hotspot CPP residues."
        ),
        (
            "Middle-region enrichment and C-terminal "
            "depletion replicated externally."
        ),
    ],

    "formats": [
        "PNG 600 dpi",
        "PDF",
        "SVG",
    ],
}


try:

    mark_step_complete(
        "28_figure_7_physicochemical_positional_grammar",
        output_files=(
            saved_files
            + [
                figure_audit_file,
            ]
        ),
        details=checkpoint_details,
    )

except NameError:

    print(
        "mark_step_complete() was not found, "
        "but Figure 7 was saved successfully."
    )


# ============================================================
# 15. DISPLAY OUTPUTS
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 7 PHYSICOCHEMICAL REPLICATION")
print("=" * 78)

display(
    physicochemical_replication[
        [
            "property_class",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_fdr",
            "same_direction",
            "significant_in_both",
        ]
    ]
)


print("\n" + "=" * 78)
print("FIGURE 7 POSITIONAL REPLICATION")
print("=" * 78)

display(
    positional_replication[
        [
            "region",
            "internal_odds_ratio",
            "internal_fdr",
            "kelm_odds_ratio",
            "kelm_fdr",
            "same_direction",
            "significant_in_both",
        ]
    ]
)


print("\n" + "=" * 78)
print("FIGURE 7 COMPLETED")
print("=" * 78)

for file_path in saved_files:
    print(
        file_path
    )

In [ ]:
from google.colab import files
import shutil

project = "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/final_manuscript_package"

zip_name = "/content/pLM4CPP_XAI_Final_Manuscript_Package"

shutil.make_archive(zip_name, "zip", project)

files.download(zip_name + ".zip")

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive and activate the project
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys
import platform

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

if not PROJECT.exists():
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT}\n"
        "Check the folder name in Google Drive."
    )

os.chdir(PROJECT)

print("Project activated")
print("Working directory:", Path.cwd())
print("Python:", sys.version)
print("Platform:", platform.platform())

print("\nTop-level project folders:")
for item in sorted(PROJECT.iterdir()):
    print(" -", item.name)

In [ ]:
# ======================================================================
# COMPLETE pLM4CPP-XAI SUPPLEMENTARY INFORMATION GENERATOR
#
# Paste this complete script into ONE Google Colab cell and run.
#
# Output:
# /content/drive/MyDrive/pLM4CPP_XAI_2026/
#     10_supplementary_information/
#         tables/
#         figures/
#         source_data/
#         SI_generation_manifest.xlsx
# ======================================================================

# ----------------------------------------------------------------------
# 0. INSTALL REQUIRED PACKAGES
# ----------------------------------------------------------------------

import sys
import subprocess
import importlib.util

required_packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "openpyxl",
    "xlsxwriter",
    "scipy"
]

for package in required_packages:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

# ----------------------------------------------------------------------
# 1. IMPORTS
# ----------------------------------------------------------------------

from google.colab import drive
from pathlib import Path

import os
import re
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# 2. MOUNT GOOGLE DRIVE AND ACTIVATE PROJECT
# ----------------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

if not PROJECT.exists():
    raise FileNotFoundError(
        f"\nProject directory was not found:\n{PROJECT}\n"
        "Check that your Google Drive folder is named exactly "
        "'pLM4CPP_XAI_2026'."
    )

os.chdir(PROJECT)

SI_DIR = PROJECT / "10_supplementary_information"
SI_TABLE_DIR = SI_DIR / "tables"
SI_FIGURE_DIR = SI_DIR / "figures"
SI_SOURCE_DIR = SI_DIR / "source_data"

for directory in [
    SI_DIR,
    SI_TABLE_DIR,
    SI_FIGURE_DIR,
    SI_SOURCE_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PROJECT ACTIVATED")
print("=" * 80)
print("Project:", PROJECT)
print("SI output:", SI_DIR)

# ----------------------------------------------------------------------
# 3. GENERAL HELPER FUNCTIONS
# ----------------------------------------------------------------------

manifest = []


def record_manifest(item, category, status, source="", note=""):
    manifest.append({
        "Item": item,
        "Category": category,
        "Status": status,
        "Source": str(source),
        "Note": note
    })


def normalize_name(text):
    """Convert column/file names into a standardized lowercase form."""
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_").lower()


def clean_columns(df):
    df = df.copy()
    df.columns = [normalize_name(c) for c in df.columns]
    return df


def original_project_files(extensions=None):
    """
    Return original project files while excluding newly generated SI files.
    """

    if extensions is None:
        extensions = {
            ".csv", ".xlsx", ".xls", ".json",
            ".png", ".pdf", ".npy", ".npz"
        }

    results = []

    for file in PROJECT.rglob("*"):
        if not file.is_file():
            continue

        if SI_DIR in file.parents:
            continue

        if file.suffix.lower() not in extensions:
            continue

        results.append(file)

    return sorted(results)


ALL_ORIGINAL_FILES = original_project_files()


def find_files(keywords, extensions=None):
    """
    Find files containing all provided keywords in filename or relative path.
    """

    if isinstance(keywords, str):
        keywords = [keywords]

    keywords = [normalize_name(k) for k in keywords]

    if extensions is None:
        extensions = {".csv", ".xlsx", ".xls", ".json"}

    matches = []

    for file in ALL_ORIGINAL_FILES:
        if file.suffix.lower() not in extensions:
            continue

        searchable = normalize_name(str(file.relative_to(PROJECT)))

        if all(keyword in searchable for keyword in keywords):
            matches.append(file)

    return matches


def find_files_any_keyword(keyword_groups, extensions=None):
    """
    Find files matching any keyword group.

    Example:
    [
        ["residue", "enrichment"],
        ["amino", "enrichment"]
    ]
    """

    found = []

    for keywords in keyword_groups:
        found.extend(find_files(keywords, extensions=extensions))

    # Remove duplicate paths
    return list(dict.fromkeys(found))


def read_data_file(file):
    """Read CSV, Excel or JSON data."""

    suffix = file.suffix.lower()

    if suffix == ".csv":
        attempts = [
            {},
            {"sep": "\t"},
            {"encoding": "latin1"}
        ]

        for kwargs in attempts:
            try:
                df = pd.read_csv(file, **kwargs)
                if df.shape[1] > 1 or kwargs == attempts[-1]:
                    return clean_columns(df)
            except Exception:
                continue

        raise ValueError(f"Unable to read CSV: {file}")

    if suffix in [".xlsx", ".xls"]:
        return clean_columns(pd.read_excel(file))

    if suffix == ".json":
        with open(file, "r", encoding="utf-8") as handle:
            data = json.load(handle)

        if isinstance(data, list):
            return clean_columns(pd.DataFrame(data))

        if isinstance(data, dict):
            try:
                return clean_columns(pd.DataFrame(data))
            except Exception:
                return clean_columns(
                    pd.json_normalize(data)
                )

    raise ValueError(f"Unsupported file type: {file}")


def safe_sheet_name(name, used_names):
    """Create a valid and unique Excel worksheet name."""

    name = re.sub(r"[\[\]\:\*\?\/\\]", "_", str(name))
    name = name[:31] or "Data"

    original = name
    counter = 2

    while name in used_names:
        suffix = f"_{counter}"
        name = original[:31-len(suffix)] + suffix
        counter += 1

    used_names.add(name)
    return name


def write_excel_workbook(
    tables,
    output_file,
    item_name,
    source_files=None
):
    """
    Write one or more DataFrames into a formatted Excel workbook.
    """

    if not tables:
        record_manifest(
            item_name,
            "Table",
            "Not generated",
            note="No suitable source data found."
        )
        return None

    used_names = set()

    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:

        workbook = writer.book

        header_format = workbook.add_format({
            "bold": True,
            "text_wrap": True,
            "valign": "top",
            "border": 1
        })

        number_format = workbook.add_format({
            "num_format": "0.0000"
        })

        scientific_format = workbook.add_format({
            "num_format": "0.00E+00"
        })

        for name, df in tables.items():

            if df is None or df.empty:
                continue

            df = df.copy()

            sheet_name = safe_sheet_name(name, used_names)
            df.to_excel(
                writer,
                index=False,
                sheet_name=sheet_name
            )

            worksheet = writer.sheets[sheet_name]

            for col_num, column in enumerate(df.columns):
                worksheet.write(
                    0,
                    col_num,
                    str(column),
                    header_format
                )

                values = df[column].astype(str)
                max_length = max(
                    len(str(column)),
                    values.map(len).max()
                )

                worksheet.set_column(
                    col_num,
                    col_num,
                    min(max_length + 2, 45)
                )

                if pd.api.types.is_numeric_dtype(df[column]):
                    worksheet.set_column(
                        col_num,
                        col_num,
                        14,
                        number_format
                    )

            worksheet.freeze_panes(1, 0)
            worksheet.autofilter(
                0,
                0,
                len(df),
                len(df.columns) - 1
            )

    source_text = ""

    if source_files:
        source_text = "; ".join(
            str(Path(f).relative_to(PROJECT))
            for f in source_files
        )

    record_manifest(
        item_name,
        "Table",
        "Generated",
        source=source_text,
        note=str(output_file.relative_to(PROJECT))
    )

    print("Generated:", output_file.name)
    return output_file


def find_column(df, exact=None, contains=None, excludes=None):
    """
    Find a column flexibly.

    exact:
        List of accepted exact normalized column names.

    contains:
        List of terms that must all occur in the column name.

    excludes:
        List of terms that must not occur.
    """

    exact = exact or []
    contains = contains or []
    excludes = excludes or []

    columns = list(df.columns)

    for candidate in exact:
        candidate = normalize_name(candidate)

        if candidate in columns:
            return candidate

    for column in columns:
        if all(term in column for term in contains):
            if not any(term in column for term in excludes):
                return column

    return None


def load_matching_tables(files, maximum=30):
    """Load matching files into a dictionary suitable for Excel export."""

    tables = {}

    for index, file in enumerate(files[:maximum], start=1):
        try:
            df = read_data_file(file)

            key = normalize_name(file.stem)
            key = f"{index:02d}_{key}"

            tables[key] = df

        except Exception as error:
            print("Could not read:", file)
            print("Reason:", error)

    return tables


def save_figure(fig, filename, item_name, source=""):
    output = SI_FIGURE_DIR / filename

    fig.tight_layout()
    fig.savefig(output, dpi=600, bbox_inches="tight")
    fig.savefig(output.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)

    record_manifest(
        item_name,
        "Figure",
        "Generated",
        source=source,
        note=str(output.relative_to(PROJECT))
    )

    print("Generated:", filename)
    return output


def skip_figure(item_name, note):
    record_manifest(
        item_name,
        "Figure",
        "Not generated",
        note=note
    )

    print(f"{item_name}: not generated — {note}")


# ----------------------------------------------------------------------
# 4. CREATE COMPLETE ORIGINAL FILE INVENTORY
# ----------------------------------------------------------------------

inventory_rows = []

for file in ALL_ORIGINAL_FILES:
    inventory_rows.append({
        "Filename": file.name,
        "Relative path": str(file.relative_to(PROJECT)),
        "Extension": file.suffix.lower(),
        "Size KB": round(file.stat().st_size / 1024, 3)
    })

inventory_df = pd.DataFrame(inventory_rows)

inventory_file = SI_DIR / "Original_project_file_inventory.xlsx"

write_excel_workbook(
    {"Original_files": inventory_df},
    inventory_file,
    "Original project file inventory"
)

# ----------------------------------------------------------------------
# 5. TABLE S1: DATASET QUALITY CONTROL AND PARTITIONS
# ----------------------------------------------------------------------

table_s1 = pd.DataFrame({
    "Dataset or quality-control step": [
        "Original internal dataset",
        "Removed: non-standard amino acids",
        "Removed: exact duplicate sequences",
        "Removed: conflicting labels",
        "Final internal dataset",
        "Training set",
        "Validation set",
        "Internal test set",
        "KELM external dataset",
        "Internal–KELM exact overlap"
    ],
    "Total sequences": [
        5479,
        20,
        0,
        0,
        5459,
        3821,
        819,
        819,
        192,
        0
    ],
    "CPP sequences": [
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        1370,
        959,
        206,
        205,
        96,
        np.nan
    ],
    "Non-CPP sequences": [
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        4089,
        2862,
        613,
        614,
        96,
        np.nan
    ],
    "Minimum length": [
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        3,
        3,
        4,
        3,
        11,
        np.nan
    ],
    "Maximum length": [
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        61,
        61,
        41,
        60,
        49,
        np.nan
    ],
    "Mean length": [
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        19.87,
        19.57,
        19.68,
        24.12,
        np.nan
    ],
    "Purpose or outcome": [
        "Internal dataset before removal of non-standard sequences",
        "Excluded during sequence cleaning",
        "No exact duplicates detected after preprocessing",
        "No identical sequences with conflicting labels detected",
        "Dataset retained for model development",
        "Classifier training",
        "Early stopping and decision-threshold optimization",
        "Held-out internal performance evaluation",
        "Independent external validation",
        "No exact sequence overlap detected"
    ]
})

write_excel_workbook(
    {"Dataset_QC": table_s1},
    SI_TABLE_DIR / "Table_S1_Dataset_quality_control.xlsx",
    "Table S1"
)

# ----------------------------------------------------------------------
# 6. TABLE S2: PLM AND CLASSIFIER INFORMATION
# ----------------------------------------------------------------------

table_s2_models = pd.DataFrame({
    "PLM": [
        "ESM2-320",
        "ESM2-640",
        "ESM2-1280",
        "ProtT5"
    ],
    "Pretrained model identifier": [
        "esm2_t6_8M_UR50D",
        "esm2_t30_150M_UR50D",
        "esm2_t33_650M_UR50D",
        "ProtT5-XL-UniRef50"
    ],
    "Residue embedding dimension": [
        320,
        640,
        1280,
        1024
    ],
    "PLM fine-tuning": [
        "No",
        "No",
        "No",
        "No"
    ],
    "Classifier": [
        "Projection, batch normalization, ReLU, dropout, masked attention pooling, dense classifier",
        "Projection, batch normalization, ReLU, dropout, masked attention pooling, dense classifier",
        "Projection, batch normalization, ReLU, dropout, masked attention pooling, dense classifier",
        "Projection, batch normalization, ReLU, dropout, masked attention pooling, dense classifier"
    ],
    "Loss": [
        "Binary cross-entropy",
        "Binary cross-entropy",
        "Binary cross-entropy",
        "Binary cross-entropy"
    ],
    "Optimizer": [
        "Adam",
        "Adam",
        "Adam",
        "Adam"
    ],
    "Threshold-selection criterion": [
        "Maximum validation MCC",
        "Maximum validation MCC",
        "Maximum validation MCC",
        "Maximum validation MCC"
    ],
    "Selected threshold": [
        0.830,
        0.925,
        0.615,
        0.765
    ]
})

table_s2_ensembles = pd.DataFrame({
    "Prediction model": [
        "Mean ensemble",
        "Median ensemble"
    ],
    "Aggregation method": [
        "Arithmetic mean of four PLM probabilities",
        "Median of four PLM probabilities"
    ],
    "Threshold-selection criterion": [
        "Maximum validation MCC",
        "Maximum validation MCC"
    ],
    "Selected threshold": [
        0.695,
        0.750
    ]
})

write_excel_workbook(
    {
        "Individual_models": table_s2_models,
        "Ensembles": table_s2_ensembles
    },
    SI_TABLE_DIR / "Table_S2_Model_and_classifier_information.xlsx",
    "Table S2"
)

# ----------------------------------------------------------------------
# 7. LOCATE ANALYSIS OUTPUT FILES
# ----------------------------------------------------------------------

training_files = find_files_any_keyword([
    ["history"],
    ["training", "loss"],
    ["epoch"],
    ["train", "history"]
])

threshold_files = find_files_any_keyword([
    ["threshold"],
    ["validation", "mcc"]
])

attribution_files = find_files_any_keyword([
    ["attribution", "agreement"],
    ["spearman"],
    ["correlation", "attribution"],
    ["method", "agreement"]
])

residue_files = find_files_any_keyword([
    ["residue", "enrichment"],
    ["amino", "enrichment"],
    ["aa", "enrichment"]
])

motif_files = find_files_any_keyword([
    ["motif", "enrichment"],
    ["motif", "statistics"],
    ["ngram"],
    ["kmer"]
])

perturbation_files = find_files_any_keyword([
    ["perturb"],
    ["faithfulness"],
    ["masking", "random"]
])

conservation_files = find_files_any_keyword([
    ["conservation"],
    ["jaccard"],
    ["plm", "support"],
    ["cross", "plm"],
    ["hotspot", "support"]
])

physicochemical_files = find_files_any_keyword([
    ["physicochemical"],
    ["hydropathy"],
    ["charge"],
    ["residue", "class"],
    ["physchem"]
])

positional_files = find_files_any_keyword([
    ["positional"],
    ["position", "bin"],
    ["normalized", "position"],
    ["region", "enrichment"],
    ["ten", "bin"]
])

print("\n" + "=" * 80)
print("SOURCE FILE SEARCH COMPLETED")
print("=" * 80)

source_groups = {
    "Training": training_files,
    "Threshold": threshold_files,
    "Attribution": attribution_files,
    "Residue enrichment": residue_files,
    "Motif enrichment": motif_files,
    "Perturbation": perturbation_files,
    "Conservation": conservation_files,
    "Physicochemical": physicochemical_files,
    "Positional": positional_files
}

for group, files in source_groups.items():
    print(f"\n{group}: {len(files)} file(s)")

    for file in files[:15]:
        print("  -", file.relative_to(PROJECT))

# ----------------------------------------------------------------------
# 8. TABLE S3: COMPLETE ATTRIBUTION AGREEMENT
# ----------------------------------------------------------------------

attribution_tables = load_matching_tables(attribution_files)

write_excel_workbook(
    attribution_tables,
    SI_TABLE_DIR / "Table_S3_Attribution_method_agreement.xlsx",
    "Table S3",
    attribution_files
)

# ----------------------------------------------------------------------
# 9. TABLE S4: COMPLETE RESIDUE ENRICHMENT
# ----------------------------------------------------------------------

residue_tables = load_matching_tables(residue_files)

write_excel_workbook(
    residue_tables,
    SI_TABLE_DIR / "Table_S4_Complete_residue_enrichment.xlsx",
    "Table S4",
    residue_files
)

# ----------------------------------------------------------------------
# 10. TABLE S5: COMPLETE MOTIF ENRICHMENT
# ----------------------------------------------------------------------

motif_tables = load_matching_tables(motif_files)

write_excel_workbook(
    motif_tables,
    SI_TABLE_DIR / "Table_S5_Complete_motif_enrichment.xlsx",
    "Table S5",
    motif_files
)

# Also copy complete motif files to source_data
for file in motif_files:
    destination = SI_SOURCE_DIR / (
        "Supplementary_Data_1_" + file.name
    )

    try:
        shutil.copy2(file, destination)
    except Exception:
        pass

# ----------------------------------------------------------------------
# 11. TABLE S6: COMPLETE PERTURBATION ANALYSIS
# ----------------------------------------------------------------------

perturbation_tables = load_matching_tables(perturbation_files)

write_excel_workbook(
    perturbation_tables,
    SI_TABLE_DIR / "Table_S6_Complete_perturbation_analysis.xlsx",
    "Table S6",
    perturbation_files
)

for file in perturbation_files:
    destination = SI_SOURCE_DIR / (
        "Supplementary_Data_2_" + file.name
    )

    try:
        shutil.copy2(file, destination)
    except Exception:
        pass

# ----------------------------------------------------------------------
# 12. TABLE S7: COMPLETE CROSS-PLM CONSERVATION
# ----------------------------------------------------------------------

conservation_tables = load_matching_tables(conservation_files)

write_excel_workbook(
    conservation_tables,
    SI_TABLE_DIR / "Table_S7_Cross_PLM_conservation.xlsx",
    "Table S7",
    conservation_files
)

# ----------------------------------------------------------------------
# 13. TABLE S8: COMPLETE PHYSICOCHEMICAL ANALYSIS
# ----------------------------------------------------------------------

physicochemical_tables = load_matching_tables(
    physicochemical_files
)

write_excel_workbook(
    physicochemical_tables,
    SI_TABLE_DIR / "Table_S8_Physicochemical_analysis.xlsx",
    "Table S8",
    physicochemical_files
)

# ----------------------------------------------------------------------
# 14. TABLE S9: COMPLETE POSITIONAL ANALYSIS
# ----------------------------------------------------------------------

positional_tables = load_matching_tables(positional_files)

write_excel_workbook(
    positional_tables,
    SI_TABLE_DIR / "Table_S9_Positional_analysis.xlsx",
    "Table S9",
    positional_files
)

# ----------------------------------------------------------------------
# 15. FIGURE S1: TRAINING AND VALIDATION HISTORIES
# ----------------------------------------------------------------------

figure_created = False

fig, ax = plt.subplots(figsize=(8, 5.5))

for file in training_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    epoch_col = find_column(
        df,
        exact=["epoch", "epochs"]
    )

    train_loss_col = find_column(
        df,
        exact=["loss", "train_loss", "training_loss"],
        contains=["loss"],
        excludes=["val", "validation"]
    )

    val_loss_col = find_column(
        df,
        exact=["val_loss", "validation_loss"],
        contains=["loss", "val"]
    )

    if val_loss_col is None:
        val_loss_col = find_column(
            df,
            contains=["validation", "loss"]
        )

    if epoch_col is None:
        df = df.reset_index()
        epoch_col = "index"

    label_base = file.stem

    if train_loss_col:
        ax.plot(
            df[epoch_col],
            df[train_loss_col],
            label=f"{label_base}: training"
        )
        figure_created = True

    if val_loss_col:
        ax.plot(
            df[epoch_col],
            df[val_loss_col],
            linestyle="--",
            label=f"{label_base}: validation"
        )
        figure_created = True

if figure_created:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(frameon=False, fontsize=7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S1_Training_and_validation_history.png",
        "Figure S1",
        "; ".join(str(f.relative_to(PROJECT)) for f in training_files)
    )
else:
    plt.close(fig)
    skip_figure(
        "Figure S1",
        "No readable epoch/loss history file was found."
    )

# ----------------------------------------------------------------------
# 16. FIGURE S2: THRESHOLD OPTIMIZATION
# ----------------------------------------------------------------------

figure_created = False

fig, ax = plt.subplots(figsize=(8, 5.5))

for file in threshold_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    threshold_col = find_column(
        df,
        exact=[
            "threshold",
            "decision_threshold",
            "probability_threshold"
        ],
        contains=["threshold"]
    )

    mcc_col = find_column(
        df,
        exact=[
            "mcc",
            "validation_mcc",
            "val_mcc"
        ],
        contains=["mcc"]
    )

    model_col = find_column(
        df,
        exact=["model", "plm", "model_name"]
    )

    if threshold_col is None or mcc_col is None:
        continue

    if model_col:
        for model, group in df.groupby(model_col):
            group = group.sort_values(threshold_col)

            ax.plot(
                group[threshold_col],
                group[mcc_col],
                label=str(model)
            )
            figure_created = True
    else:
        group = df.sort_values(threshold_col)

        ax.plot(
            group[threshold_col],
            group[mcc_col],
            label=file.stem
        )
        figure_created = True

if figure_created:
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Validation MCC")
    ax.legend(frameon=False, fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S2_Threshold_optimization.png",
        "Figure S2",
        "; ".join(str(f.relative_to(PROJECT)) for f in threshold_files)
    )
else:
    plt.close(fig)
    skip_figure(
        "Figure S2",
        "No file containing both threshold and validation MCC was found."
    )

# ----------------------------------------------------------------------
# 17. FIGURE S3: ATTRIBUTION AGREEMENT
# ----------------------------------------------------------------------

figure_created = False

for file in attribution_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    numeric_df = df.select_dtypes(include=np.number)

    if numeric_df.shape[1] < 2:
        continue

    corr = numeric_df.corr(method="spearman")

    if corr.shape[0] < 2:
        continue

    fig, ax = plt.subplots(figsize=(7, 6))

    image = ax.imshow(
        corr.values,
        aspect="auto",
        vmin=-1,
        vmax=1
    )

    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(
        corr.columns,
        rotation=45,
        ha="right",
        fontsize=8
    )

    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(
        corr.index,
        fontsize=8
    )

    if corr.shape[0] <= 12:
        for i in range(corr.shape[0]):
            for j in range(corr.shape[1]):
                ax.text(
                    j,
                    i,
                    f"{corr.iloc[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7
                )

    fig.colorbar(
        image,
        ax=ax,
        label="Spearman correlation"
    )

    save_figure(
        fig,
        "Figure_S3_Attribution_agreement_matrix.png",
        "Figure S3",
        str(file.relative_to(PROJECT))
    )

    figure_created = True
    break

if not figure_created:
    skip_figure(
        "Figure S3",
        "No suitable numeric attribution-agreement table was found."
    )

# ----------------------------------------------------------------------
# 18. FIGURE S4: COMPLETE RESIDUE ENRICHMENT
# ----------------------------------------------------------------------

figure_created = False

for file in residue_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    residue_col = find_column(
        df,
        exact=[
            "residue",
            "amino_acid",
            "aminoacid",
            "aa"
        ]
    )

    internal_or_col = find_column(
        df,
        contains=["internal", "or"]
    )

    external_or_col = find_column(
        df,
        contains=["kelm", "or"]
    )

    if external_or_col is None:
        external_or_col = find_column(
            df,
            contains=["external", "or"]
        )

    if not residue_col or not internal_or_col or not external_or_col:
        continue

    plot_df = df[
        [residue_col, internal_or_col, external_or_col]
    ].copy()

    plot_df[internal_or_col] = pd.to_numeric(
        plot_df[internal_or_col],
        errors="coerce"
    )

    plot_df[external_or_col] = pd.to_numeric(
        plot_df[external_or_col],
        errors="coerce"
    )

    plot_df = plot_df.dropna()

    plot_df = plot_df[
        (plot_df[internal_or_col] > 0) &
        (plot_df[external_or_col] > 0)
    ]

    amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

    plot_df = plot_df[
        plot_df[residue_col].astype(str).isin(amino_acids)
    ]

    if plot_df.empty:
        continue

    plot_df["internal_log2_or"] = np.log2(
        plot_df[internal_or_col]
    )

    plot_df["external_log2_or"] = np.log2(
        plot_df[external_or_col]
    )

    x = np.arange(len(plot_df))
    width = 0.38

    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.bar(
        x - width / 2,
        plot_df["internal_log2_or"],
        width,
        label="Internal"
    )

    ax.bar(
        x + width / 2,
        plot_df["external_log2_or"],
        width,
        label="KELM"
    )

    ax.axhline(0, linewidth=1)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df[residue_col])
    ax.set_xlabel("Amino acid")
    ax.set_ylabel("log₂ odds ratio")
    ax.legend(frameon=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S4_Complete_residue_enrichment.png",
        "Figure S4",
        str(file.relative_to(PROJECT))
    )

    figure_created = True
    break

if not figure_created:
    skip_figure(
        "Figure S4",
        "No residue table containing residue, internal OR and KELM OR was found."
    )

# ----------------------------------------------------------------------
# 19. FIGURE S5: TOP REPLICATED MOTIFS
# ----------------------------------------------------------------------

figure_created = False

for file in motif_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    motif_col = find_column(
        df,
        exact=[
            "motif",
            "sequence_motif",
            "kmer",
            "ngram"
        ]
    )

    internal_or_col = find_column(
        df,
        contains=["internal", "or"]
    )

    external_or_col = find_column(
        df,
        contains=["kelm", "or"]
    )

    if external_or_col is None:
        external_or_col = find_column(
            df,
            contains=["external", "or"]
        )

    if not motif_col or not internal_or_col or not external_or_col:
        continue

    plot_df = df[
        [motif_col, internal_or_col, external_or_col]
    ].copy()

    plot_df[internal_or_col] = pd.to_numeric(
        plot_df[internal_or_col],
        errors="coerce"
    )

    plot_df[external_or_col] = pd.to_numeric(
        plot_df[external_or_col],
        errors="coerce"
    )

    plot_df = plot_df.dropna()

    plot_df = plot_df[
        (plot_df[internal_or_col] > 0) &
        (plot_df[external_or_col] > 0)
    ]

    if plot_df.empty:
        continue

    plot_df["replicated_strength"] = np.minimum(
        plot_df[internal_or_col],
        plot_df[external_or_col]
    )

    plot_df = (
        plot_df
        .sort_values("replicated_strength", ascending=False)
        .head(25)
        .sort_values("replicated_strength")
    )

    y = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.scatter(
        np.log2(plot_df[internal_or_col]),
        y,
        label="Internal"
    )

    ax.scatter(
        np.log2(plot_df[external_or_col]),
        y,
        label="KELM"
    )

    ax.axvline(0, linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(plot_df[motif_col])
    ax.set_xlabel("log₂ odds ratio")
    ax.set_ylabel("Motif")
    ax.legend(frameon=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S5_Top_replicated_motifs.png",
        "Figure S5",
        str(file.relative_to(PROJECT))
    )

    figure_created = True
    break

if not figure_created:
    skip_figure(
        "Figure S5",
        "No motif table containing motif, internal OR and KELM OR was found."
    )

# ----------------------------------------------------------------------
# 20. FIGURE S6: PERTURBATION EFFECTS
# ----------------------------------------------------------------------

figure_created = False

for file in perturbation_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    hotspot_col = find_column(
        df,
        contains=["hotspot", "drop"]
    )

    random_col = find_column(
        df,
        contains=["random", "drop"]
    )

    model_col = find_column(
        df,
        exact=["model", "plm", "model_name"]
    )

    dataset_col = find_column(
        df,
        exact=["dataset", "split", "data"]
    )

    if not hotspot_col or not random_col:
        continue

    df[hotspot_col] = pd.to_numeric(
        df[hotspot_col],
        errors="coerce"
    )

    df[random_col] = pd.to_numeric(
        df[random_col],
        errors="coerce"
    )

    plot_df = df.dropna(
        subset=[hotspot_col, random_col]
    )

    if plot_df.empty:
        continue

    if model_col and dataset_col:
        summary = (
            plot_df
            .groupby([dataset_col, model_col])[
                [hotspot_col, random_col]
            ]
            .mean()
            .reset_index()
        )

        labels = (
            summary[dataset_col].astype(str) +
            " | " +
            summary[model_col].astype(str)
        )
    elif model_col:
        summary = (
            plot_df
            .groupby(model_col)[
                [hotspot_col, random_col]
            ]
            .mean()
            .reset_index()
        )

        labels = summary[model_col].astype(str)
    else:
        summary = pd.DataFrame({
            hotspot_col: [plot_df[hotspot_col].mean()],
            random_col: [plot_df[random_col].mean()]
        })

        labels = pd.Series(["All data"])

    x = np.arange(len(summary))
    width = 0.38

    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.bar(
        x - width/2,
        summary[hotspot_col],
        width,
        label="Hotspot perturbation"
    )

    ax.bar(
        x + width/2,
        summary[random_col],
        width,
        label="Random perturbation"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        rotation=45,
        ha="right"
    )

    ax.set_ylabel("Mean decrease in CPP probability")
    ax.legend(frameon=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S6_Perturbation_effects.png",
        "Figure S6",
        str(file.relative_to(PROJECT))
    )

    figure_created = True
    break

if not figure_created:
    skip_figure(
        "Figure S6",
        "No perturbation table containing hotspot and random probability drops was found."
    )

# ----------------------------------------------------------------------
# 21. FIGURE S7: CROSS-PLM SUPPORT DISTRIBUTION
# ----------------------------------------------------------------------

figure_created = False

for file in conservation_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    support_col = find_column(
        df,
        exact=[
            "support",
            "support_count",
            "plm_support",
            "n_plms",
            "number_of_plms"
        ],
        contains=["support"]
    )

    value_col = find_column(
        df,
        exact=[
            "fraction",
            "proportion",
            "percentage",
            "percent",
            "count"
        ]
    )

    class_col = find_column(
        df,
        exact=[
            "class",
            "label",
            "sequence_class",
            "peptide_class"
        ]
    )

    dataset_col = find_column(
        df,
        exact=["dataset", "split"]
    )

    if not support_col or not value_col:
        continue

    df[support_col] = pd.to_numeric(
        df[support_col],
        errors="coerce"
    )

    df[value_col] = pd.to_numeric(
        df[value_col],
        errors="coerce"
    )

    plot_df = df.dropna(
        subset=[support_col, value_col]
    )

    if plot_df.empty:
        continue

    fig, ax = plt.subplots(figsize=(8, 5.5))

    if class_col and dataset_col:
        group_columns = [dataset_col, class_col]
    elif class_col:
        group_columns = [class_col]
    elif dataset_col:
        group_columns = [dataset_col]
    else:
        group_columns = []

    if group_columns:
        for group_name, group in plot_df.groupby(group_columns):

            group = group.sort_values(support_col)

            if isinstance(group_name, tuple):
                label = " | ".join(map(str, group_name))
            else:
                label = str(group_name)

            ax.plot(
                group[support_col],
                group[value_col],
                marker="o",
                label=label
            )
    else:
        plot_df = plot_df.sort_values(support_col)

        ax.plot(
            plot_df[support_col],
            plot_df[value_col],
            marker="o"
        )

    ax.set_xlabel("Number of supporting PLMs")
    ax.set_ylabel(value_col.replace("_", " ").title())

    if group_columns:
        ax.legend(frameon=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S7_Cross_PLM_support_distribution.png",
        "Figure S7",
        str(file.relative_to(PROJECT))
    )

    figure_created = True
    break

if not figure_created:
    skip_figure(
        "Figure S7",
        "No table containing PLM support count and frequency/fraction was found."
    )

# ----------------------------------------------------------------------
# 22. FIGURE S8: PHYSICOCHEMICAL ANALYSIS
# ----------------------------------------------------------------------

figure_created = False

for file in physicochemical_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    feature_col = find_column(
        df,
        exact=[
            "feature",
            "property",
            "physicochemical_class",
            "residue_class",
            "descriptor"
        ]
    )

    internal_or_col = find_column(
        df,
        contains=["internal", "or"]
    )

    external_or_col = find_column(
        df,
        contains=["kelm", "or"]
    )

    if external_or_col is None:
        external_or_col = find_column(
            df,
            contains=["external", "or"]
        )

    if not feature_col or not internal_or_col or not external_or_col:
        continue

    plot_df = df[
        [feature_col, internal_or_col, external_or_col]
    ].copy()

    plot_df[internal_or_col] = pd.to_numeric(
        plot_df[internal_or_col],
        errors="coerce"
    )

    plot_df[external_or_col] = pd.to_numeric(
        plot_df[external_or_col],
        errors="coerce"
    )

    plot_df = plot_df.dropna()

    plot_df = plot_df[
        (plot_df[internal_or_col] > 0) &
        (plot_df[external_or_col] > 0)
    ]

    if plot_df.empty:
        continue

    x = np.arange(len(plot_df))
    width = 0.38

    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.bar(
        x - width/2,
        np.log2(plot_df[internal_or_col]),
        width,
        label="Internal"
    )

    ax.bar(
        x + width/2,
        np.log2(plot_df[external_or_col]),
        width,
        label="KELM"
    )

    ax.axhline(0, linewidth=1)

    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_df[feature_col],
        rotation=45,
        ha="right"
    )

    ax.set_ylabel("log₂ odds ratio")
    ax.set_xlabel("Physicochemical feature")
    ax.legend(frameon=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S8_Physicochemical_enrichment.png",
        "Figure S8",
        str(file.relative_to(PROJECT))
    )

    figure_created = True
    break

if not figure_created:
    skip_figure(
        "Figure S8",
        "No physicochemical table containing feature, internal OR and KELM OR was found."
    )

# ----------------------------------------------------------------------
# 23. FIGURE S9: TEN-BIN POSITIONAL PROFILE
# ----------------------------------------------------------------------

figure_created = False

for file in positional_files:

    try:
        df = read_data_file(file)
    except Exception:
        continue

    bin_col = find_column(
        df,
        exact=[
            "bin",
            "position_bin",
            "normalized_bin",
            "sequence_bin",
            "decile"
        ]
    )

    value_col = find_column(
        df,
        exact=[
            "frequency",
            "fraction",
            "proportion",
            "hotspot_frequency",
            "hotspot_fraction"
        ]
    )

    class_col = find_column(
        df,
        exact=[
            "class",
            "label",
            "sequence_class",
            "region_type"
        ]
    )

    dataset_col = find_column(
        df,
        exact=["dataset", "split"]
    )

    if not bin_col or not value_col:
        continue

    df[bin_col] = pd.to_numeric(
        df[bin_col],
        errors="coerce"
    )

    df[value_col] = pd.to_numeric(
        df[value_col],
        errors="coerce"
    )

    plot_df = df.dropna(
        subset=[bin_col, value_col]
    )

    if plot_df.empty:
        continue

    fig, ax = plt.subplots(figsize=(8, 5.5))

    if class_col and dataset_col:
        group_columns = [dataset_col, class_col]
    elif class_col:
        group_columns = [class_col]
    elif dataset_col:
        group_columns = [dataset_col]
    else:
        group_columns = []

    if group_columns:
        for group_name, group in plot_df.groupby(group_columns):

            group = group.sort_values(bin_col)

            if isinstance(group_name, tuple):
                label = " | ".join(map(str, group_name))
            else:
                label = str(group_name)

            ax.plot(
                group[bin_col],
                group[value_col],
                marker="o",
                label=label
            )
    else:
        plot_df = plot_df.sort_values(bin_col)

        ax.plot(
            plot_df[bin_col],
            plot_df[value_col],
            marker="o"
        )

    ax.set_xlabel("Normalized sequence bin")
    ax.set_ylabel(value_col.replace("_", " ").title())

    if group_columns:
        ax.legend(frameon=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_figure(
        fig,
        "Figure_S9_Ten_bin_positional_profile.png",
        "Figure S9",
        str(file.relative_to(PROJECT))
    )

    figure_created = True
    break

if not figure_created:
    skip_figure(
        "Figure S9",
        "No positional table containing sequence bins and hotspot frequency was found."
    )

# ----------------------------------------------------------------------
# 24. CREATE SI CITATION GUIDE
# ----------------------------------------------------------------------

citation_guide = pd.DataFrame({
    "SI item": [
        "Table S1",
        "Table S2",
        "Figure S1",
        "Figure S2",
        "Table S3",
        "Figure S3",
        "Table S4",
        "Figure S4",
        "Table S5",
        "Figure S5",
        "Table S6",
        "Figure S6",
        "Table S7",
        "Figure S7",
        "Table S8",
        "Figure S8",
        "Table S9",
        "Figure S9"
    ],
    "Manuscript location": [
        "Methods 2.1 and Results 3.1",
        "Methods 2.2–2.4 and Results 3.2",
        "Methods 2.3 or Results 3.2",
        "Methods 2.3 or Results 3.2",
        "Methods 2.5–2.6 and Results 3.4",
        "Results 3.4",
        "Methods 2.8 and Results 3.5",
        "Results 3.5",
        "Methods 2.8 and Results 3.5",
        "Results 3.5",
        "Methods 2.9 and Results 3.6",
        "Results 3.6",
        "Methods 2.7 and Results 3.7",
        "Results 3.7",
        "Methods 2.10 and Results 3.8",
        "Results 3.8",
        "Methods 2.10 and Results 3.9",
        "Results 3.9"
    ],
    "Suggested citation sentence": [
        "Detailed quality-control outcomes and dataset partition statistics are provided in Table S1.",
        "Complete model and classifier information is provided in Table S2.",
        "Training and validation histories are shown in Figure S1.",
        "Validation-based decision-threshold optimization is shown in Figure S2.",
        "Complete attribution-method agreement statistics are provided in Table S3.",
        "The complete attribution-agreement structure is shown in Figure S3.",
        "Complete enrichment results for all amino acids are provided in Table S4.",
        "Complete amino-acid enrichment profiles are shown in Figure S4.",
        "Complete motif enrichment statistics are provided in Table S5 and Supplementary Data 1.",
        "The highest-ranking replicated motifs are shown in Figure S5.",
        "Complete perturbation statistics are provided in Table S6 and Supplementary Data 2.",
        "Additional perturbation-effect distributions are shown in Figure S6.",
        "Complete cross-PLM conservation statistics are provided in Table S7.",
        "Hotspot support distributions across the four PLMs are shown in Figure S7.",
        "Complete physicochemical comparisons are provided in Table S8.",
        "Additional physicochemical enrichment results are shown in Figure S8.",
        "Complete regional and normalized-bin positional statistics are provided in Table S9.",
        "The complete ten-bin positional profile is shown in Figure S9."
    ]
})

write_excel_workbook(
    {"SI_citation_guide": citation_guide},
    SI_DIR / "SI_citation_guide.xlsx",
    "SI citation guide"
)

# ----------------------------------------------------------------------
# 25. SAVE FINAL GENERATION MANIFEST
# ----------------------------------------------------------------------

manifest_df = pd.DataFrame(manifest)

manifest_output = SI_DIR / "SI_generation_manifest.xlsx"

with pd.ExcelWriter(
    manifest_output,
    engine="xlsxwriter"
) as writer:

    manifest_df.to_excel(
        writer,
        index=False,
        sheet_name="Manifest"
    )

    citation_guide.to_excel(
        writer,
        index=False,
        sheet_name="Citation_guide"
    )

    source_summary_rows = []

    for group, files in source_groups.items():
        if files:
            for file in files:
                source_summary_rows.append({
                    "Analysis group": group,
                    "Source file": str(file.relative_to(PROJECT))
                })
        else:
            source_summary_rows.append({
                "Analysis group": group,
                "Source file": "No source file detected"
            })

    pd.DataFrame(source_summary_rows).to_excel(
        writer,
        index=False,
        sheet_name="Detected_sources"
    )

print("\n" + "=" * 80)
print("SUPPLEMENTARY INFORMATION GENERATION COMPLETE")
print("=" * 80)

print("\nOutput directory:")
print(SI_DIR)

print("\nGenerated tables:")
for file in sorted(SI_TABLE_DIR.glob("*")):
    print(" -", file.name)

print("\nGenerated figures:")
for file in sorted(SI_FIGURE_DIR.glob("*.png")):
    print(" -", file.name)

print("\nGeneration status:")
display(manifest_df)

print("\nFinal manifest:")
print(manifest_output)

print("\nIMPORTANT:")
print(
    "An item marked 'Not generated' means the script could not find "
    "a source file with the required columns. All available source "
    "tables were still collected into the corresponding Excel workbook."
)

In [ ]:
# ======================================================================
# FIXED FIGURE S4: COMPLETE AMINO-ACID ENRICHMENT
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

SI_FIGURE_DIR = (
    PROJECT
    / "10_supplementary_information"
    / "figures"
)

SI_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def clean_columns(df):
    df = df.copy()

    df.columns = [
        re.sub(r"[^a-zA-Z0-9]+", "_", str(column))
        .strip("_")
        .lower()
        for column in df.columns
    ]

    # Remove duplicate columns created after name cleaning
    df = df.loc[:, ~df.columns.duplicated()].copy()

    return df


# Use the replication table first because it directly compares datasets
candidate_files = [
    PROJECT
    / "07_results/tables_main/"
      "residue_enrichment_replication.csv",

    PROJECT
    / "07_results/final_manuscript_package/02_SI_tables/"
      "Table_S5_residue_enrichment_all_thresholds.csv",

    PROJECT
    / "07_results/tables_SI/"
      "residue_enrichment_all_thresholds.csv"
]

residue_file = next(
    (file for file in candidate_files if file.exists()),
    None
)

if residue_file is None:
    raise FileNotFoundError(
        "No residue-enrichment file was found."
    )

residue_df = clean_columns(
    pd.read_csv(residue_file)
)

print("Source file:")
print(residue_file)

print("\nAvailable columns:")
print(residue_df.columns.tolist())

display(residue_df.head())


# ----------------------------------------------------------------------
# Detect residue column
# ----------------------------------------------------------------------

residue_col = next(
    (
        column
        for column in [
            "residue",
            "amino_acid",
            "aminoacid",
            "aa"
        ]
        if column in residue_df.columns
    ),
    None
)

if residue_col is None:
    raise KeyError(
        "No residue column was detected.\n"
        f"Columns: {residue_df.columns.tolist()}"
    )


# ----------------------------------------------------------------------
# Detect whether data are long format or wide format
# ----------------------------------------------------------------------

dataset_col = next(
    (
        column
        for column in [
            "dataset",
            "data_set",
            "evaluation_dataset"
        ]
        if column in residue_df.columns
    ),
    None
)

odds_col = next(
    (
        column
        for column in [
            "odds_ratio",
            "or"
        ]
        if column in residue_df.columns
    ),
    None
)


# ----------------------------------------------------------------------
# Case 1: Long format
# residue | dataset | odds_ratio
# ----------------------------------------------------------------------

if dataset_col is not None and odds_col is not None:

    temp = residue_df[
        [residue_col, dataset_col, odds_col]
    ].copy()

    temp[odds_col] = pd.to_numeric(
        temp[odds_col],
        errors="coerce"
    )

    temp[dataset_col] = (
        temp[dataset_col]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    temp = temp.dropna(
        subset=[residue_col, dataset_col, odds_col]
    )

    pivot = temp.pivot_table(
        index=residue_col,
        columns=dataset_col,
        values=odds_col,
        aggfunc="first"
    ).reset_index()

    pivot.columns = [
        str(column).lower().replace(" ", "_")
        for column in pivot.columns
    ]

    internal_col = next(
        (
            column
            for column in pivot.columns
            if "internal" in column
        ),
        None
    )

    kelm_col = next(
        (
            column
            for column in pivot.columns
            if "kelm" in column or "external" in column
        ),
        None
    )

    if internal_col is None or kelm_col is None:
        raise KeyError(
            "Internal and KELM dataset columns were not identified "
            "after pivoting.\n"
            f"Pivot columns: {pivot.columns.tolist()}"
        )

    plot_df = pd.DataFrame({
        "residue": pivot[residue_col],
        "internal_or": pd.to_numeric(
            pivot[internal_col],
            errors="coerce"
        ),
        "kelm_or": pd.to_numeric(
            pivot[kelm_col],
            errors="coerce"
        )
    })


# ----------------------------------------------------------------------
# Case 2: Wide format
# residue | internal_or | kelm_or
# ----------------------------------------------------------------------

else:

    internal_col = next(
        (
            column
            for column in residue_df.columns
            if (
                "internal" in column
                and (
                    "odds_ratio" in column
                    or column.endswith("_or")
                )
            )
        ),
        None
    )

    kelm_col = next(
        (
            column
            for column in residue_df.columns
            if (
                ("kelm" in column or "external" in column)
                and (
                    "odds_ratio" in column
                    or column.endswith("_or")
                )
            )
        ),
        None
    )

    if internal_col is None or kelm_col is None:
        raise KeyError(
            "Internal and KELM odds-ratio columns were not detected.\n"
            f"Columns: {residue_df.columns.tolist()}"
        )

    plot_df = pd.DataFrame({
        "residue": residue_df[residue_col],
        "internal_or": pd.to_numeric(
            residue_df[internal_col],
            errors="coerce"
        ),
        "kelm_or": pd.to_numeric(
            residue_df[kelm_col],
            errors="coerce"
        )
    })


# ----------------------------------------------------------------------
# Clean and organize the 20 amino acids
# ----------------------------------------------------------------------

amino_acid_order = list("ACDEFGHIKLMNPQRSTVWY")

plot_df["residue"] = (
    plot_df["residue"]
    .astype(str)
    .str.upper()
    .str.strip()
)

plot_df = plot_df[
    plot_df["residue"].isin(amino_acid_order)
].copy()

plot_df = plot_df.dropna(
    subset=["internal_or", "kelm_or"]
)

plot_df = plot_df[
    (plot_df["internal_or"] > 0)
    & (plot_df["kelm_or"] > 0)
]

# Keep one row per amino acid
plot_df = (
    plot_df
    .drop_duplicates(subset="residue", keep="first")
)

plot_df["residue"] = pd.Categorical(
    plot_df["residue"],
    categories=amino_acid_order,
    ordered=True
)

plot_df = plot_df.sort_values("residue")

plot_df["internal_log2_or"] = np.log2(
    plot_df["internal_or"]
)

plot_df["kelm_log2_or"] = np.log2(
    plot_df["kelm_or"]
)

print("\nPlotting data:")
display(plot_df)


# ----------------------------------------------------------------------
# Generate Figure S4
# ----------------------------------------------------------------------

x = np.arange(len(plot_df))
width = 0.38

fig, ax = plt.subplots(figsize=(10, 5.8))

ax.bar(
    x - width / 2,
    plot_df["internal_log2_or"],
    width,
    label="Internal test"
)

ax.bar(
    x + width / 2,
    plot_df["kelm_log2_or"],
    width,
    label="KELM external"
)

ax.axhline(0, linewidth=1)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["residue"])

ax.set_xlabel("Amino acid")
ax.set_ylabel("log₂ odds ratio")

ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

png_output = (
    SI_FIGURE_DIR
    / "Figure_S4_Complete_residue_enrichment.png"
)

pdf_output = (
    SI_FIGURE_DIR
    / "Figure_S4_Complete_residue_enrichment.pdf"
)

fig.savefig(
    png_output,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_output,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nFigure S4 generated successfully:")
print(png_output)
print(pdf_output)

In [ ]:
# ======================================================================
# FIXED FIGURE S5: REPLICATED HOTSPOT MOTIF ENRICHMENT
# Uses dataset + log2_enrichment directly
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

SI_FIGURE_DIR = (
    PROJECT
    / "10_supplementary_information"
    / "figures"
)

SI_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def clean_columns(df):
    df = df.copy()

    df.columns = [
        re.sub(r"[^a-zA-Z0-9]+", "_", str(column))
        .strip("_")
        .lower()
        for column in df.columns
    ]

    df = df.loc[:, ~df.columns.duplicated()].copy()

    return df


motif_file = (
    PROJECT
    / "07_results/final_manuscript_package/02_SI_tables/"
      "Table_S6_complete_motif_enrichment.csv"
)

if not motif_file.exists():
    raise FileNotFoundError(
        f"Motif file not found:\n{motif_file}"
    )

motif_df = clean_columns(
    pd.read_csv(motif_file)
)

print("Source file:")
print(motif_file)

print("\nAvailable columns:")
print(motif_df.columns.tolist())

display(motif_df.head())


# ----------------------------------------------------------------------
# Required columns
# ----------------------------------------------------------------------

required_columns = [
    "dataset",
    "motif",
    "log2_enrichment",
    "fdr_bh"
]

missing_columns = [
    column
    for column in required_columns
    if column not in motif_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )


# ----------------------------------------------------------------------
# Clean data
# ----------------------------------------------------------------------

motif_df["dataset"] = (
    motif_df["dataset"]
    .astype(str)
    .str.lower()
    .str.strip()
)

motif_df["motif"] = (
    motif_df["motif"]
    .astype(str)
    .str.upper()
    .str.strip()
)

motif_df["log2_enrichment"] = pd.to_numeric(
    motif_df["log2_enrichment"],
    errors="coerce"
)

motif_df["fdr_bh"] = pd.to_numeric(
    motif_df["fdr_bh"],
    errors="coerce"
)

motif_df = motif_df.dropna(
    subset=[
        "dataset",
        "motif",
        "log2_enrichment",
        "fdr_bh"
    ]
)


# ----------------------------------------------------------------------
# Standardize dataset names
# ----------------------------------------------------------------------

def standardize_dataset_name(value):
    value = str(value).lower()

    if "internal" in value:
        return "internal_test"

    if "kelm" in value or "external" in value:
        return "kelm_external"

    return value


motif_df["dataset_standard"] = (
    motif_df["dataset"]
    .map(standardize_dataset_name)
)

print("\nDetected datasets:")
print(
    motif_df["dataset_standard"]
    .value_counts()
)


# ----------------------------------------------------------------------
# Keep best row per motif within each dataset
# In case multiple thresholds or duplicate records exist
# ----------------------------------------------------------------------

motif_df = (
    motif_df
    .sort_values(
        ["dataset_standard", "motif", "fdr_bh"]
    )
    .drop_duplicates(
        subset=["dataset_standard", "motif"],
        keep="first"
    )
)


# ----------------------------------------------------------------------
# Pivot internal and KELM values
# ----------------------------------------------------------------------

enrichment_pivot = motif_df.pivot_table(
    index="motif",
    columns="dataset_standard",
    values="log2_enrichment",
    aggfunc="first"
).reset_index()

fdr_pivot = motif_df.pivot_table(
    index="motif",
    columns="dataset_standard",
    values="fdr_bh",
    aggfunc="first"
).reset_index()

enrichment_pivot.columns.name = None
fdr_pivot.columns.name = None

print("\nEnrichment pivot columns:")
print(enrichment_pivot.columns.tolist())


if "internal_test" not in enrichment_pivot.columns:
    raise KeyError(
        "No internal_test motif results were detected."
    )

if "kelm_external" not in enrichment_pivot.columns:
    raise KeyError(
        "No KELM external motif results were detected.\n"
        f"Detected datasets: "
        f"{motif_df['dataset_standard'].unique().tolist()}"
    )


# ----------------------------------------------------------------------
# Combine enrichment and FDR values
# ----------------------------------------------------------------------

plot_df = enrichment_pivot[
    ["motif", "internal_test", "kelm_external"]
].copy()

plot_df = plot_df.rename(columns={
    "internal_test": "internal_log2_enrichment",
    "kelm_external": "kelm_log2_enrichment"
})

if (
    "internal_test" in fdr_pivot.columns
    and "kelm_external" in fdr_pivot.columns
):
    fdr_data = fdr_pivot[
        ["motif", "internal_test", "kelm_external"]
    ].copy()

    fdr_data = fdr_data.rename(columns={
        "internal_test": "internal_fdr",
        "kelm_external": "kelm_fdr"
    })

    plot_df = plot_df.merge(
        fdr_data,
        on="motif",
        how="left"
    )


# ----------------------------------------------------------------------
# Keep motifs replicated in the same direction
# ----------------------------------------------------------------------

plot_df = plot_df.dropna(
    subset=[
        "internal_log2_enrichment",
        "kelm_log2_enrichment"
    ]
)

same_positive_direction = (
    (plot_df["internal_log2_enrichment"] > 0)
    & (plot_df["kelm_log2_enrichment"] > 0)
)

plot_df = plot_df[
    same_positive_direction
].copy()


# Prefer motifs significant in both datasets
if (
    "internal_fdr" in plot_df.columns
    and "kelm_fdr" in plot_df.columns
):

    significant_both = plot_df[
        (plot_df["internal_fdr"] < 0.05)
        & (plot_df["kelm_fdr"] < 0.05)
    ].copy()

    if len(significant_both) >= 5:
        plot_df = significant_both


# ----------------------------------------------------------------------
# Rank motifs by weaker replicated enrichment
# ----------------------------------------------------------------------

plot_df["replicated_strength"] = np.minimum(
    plot_df["internal_log2_enrichment"],
    plot_df["kelm_log2_enrichment"]
)

plot_df = (
    plot_df
    .sort_values(
        "replicated_strength",
        ascending=False
    )
    .head(25)
    .sort_values("replicated_strength")
)

if plot_df.empty:
    raise RuntimeError(
        "No positively replicated motifs were found "
        "in both internal and KELM datasets."
    )

print("\nMotifs used in Figure S5:")
display(plot_df)


# ----------------------------------------------------------------------
# Generate Figure S5
# ----------------------------------------------------------------------

y = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    plot_df["internal_log2_enrichment"],
    y,
    s=55,
    label="Internal test"
)

ax.scatter(
    plot_df["kelm_log2_enrichment"],
    y,
    s=55,
    label="KELM external"
)

for i in range(len(plot_df)):
    ax.plot(
        [
            plot_df.iloc[i]["internal_log2_enrichment"],
            plot_df.iloc[i]["kelm_log2_enrichment"]
        ],
        [i, i],
        linewidth=0.8
    )

ax.axvline(0, linewidth=1)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["motif"])

ax.set_xlabel("log₂ enrichment")
ax.set_ylabel("Motif")

ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

png_output = (
    SI_FIGURE_DIR
    / "Figure_S5_Replicated_hotspot_motifs.png"
)

pdf_output = (
    SI_FIGURE_DIR
    / "Figure_S5_Replicated_hotspot_motifs.pdf"
)

fig.savefig(
    png_output,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_output,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nFigure S5 generated successfully:")
print(png_output)
print(pdf_output)

In [ ]:
# ======================================================================
# FIGURE S7
# Cross-PLM hotspot support distribution
# ======================================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import re

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

FIG_DIR = PROJECT/"10_supplementary_information"/"figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

internal_file = PROJECT/"06_xai/hotspot_conservation/internal_test/cross_plm_residue_support.csv"
kelm_file     = PROJECT/"06_xai/hotspot_conservation/kelm_external/cross_plm_residue_support.csv"


def clean(df):
    df.columns=[
        re.sub(r"[^A-Za-z0-9]+","_",c).strip("_").lower()
        for c in df.columns
    ]
    return df.loc[:,~df.columns.duplicated()]


def summarize(file,dataset):

    df=clean(pd.read_csv(file))

    print("\n========================")
    print(dataset)
    print(df.columns.tolist())
    print(df.head())

    support=None

    for c in df.columns:
        if "support" in c:
            support=c
            break

    if support is None:
        raise Exception("Support column not found.")

    df[support]=pd.to_numeric(df[support],errors="coerce")
    df=df.dropna(subset=[support])

    summary=df.groupby(support).size().reset_index(name="count")
    summary["fraction"]=summary["count"]/summary["count"].sum()
    summary["dataset"]=dataset

    summary=summary.rename(columns={support:"support"})

    return summary


internal=summarize(internal_file,"Internal test")
kelm=summarize(kelm_file,"KELM external")

plot=pd.concat([internal,kelm],ignore_index=True)

fig,ax=plt.subplots(figsize=(7,5))

for dataset,g in plot.groupby("dataset"):
    g=g.sort_values("support")
    ax.plot(
        g["support"],
        g["fraction"],
        marker="o",
        linewidth=2,
        label=dataset
    )

ax.set_xlabel("Number of PLMs supporting hotspot")
ax.set_ylabel("Fraction of hotspot residues")
ax.set_xticks(sorted(plot["support"].unique()))
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

png=FIG_DIR/"Figure_S7_Cross_PLM_support_distribution.png"
pdf=FIG_DIR/"Figure_S7_Cross_PLM_support_distribution.pdf"

plt.savefig(png,dpi=600,bbox_inches="tight")
plt.savefig(pdf,bbox_inches="tight")
plt.show()

print("\nSaved:")
print(png)
print(pdf)

In [ ]:
# ======================================================================
# FIXED FIGURE S8: PHYSICOCHEMICAL ENRICHMENT
# ======================================================================

from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

SI_FIGURE_DIR = (
    PROJECT
    / "10_supplementary_information"
    / "figures"
)

SI_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

physchem_file = (
    PROJECT
    / "07_results/tables_main/"
      "physicochemical_enrichment_replication.csv"
)

if not physchem_file.exists():
    raise FileNotFoundError(
        f"File not found:\n{physchem_file}"
    )

physchem_df = pd.read_csv(physchem_file)

physchem_df.columns = [
    re.sub(r"[^a-zA-Z0-9]+", "_", str(column))
    .strip("_")
    .lower()
    for column in physchem_df.columns
]

physchem_df = (
    physchem_df
    .loc[:, ~physchem_df.columns.duplicated()]
    .copy()
)

print("=" * 80)
print("FIGURE S8: PHYSICOCHEMICAL ENRICHMENT")
print("=" * 80)

print("\nSource file:")
print(physchem_file)

print("\nAvailable columns:")
print(physchem_df.columns.tolist())

display(physchem_df)


required_columns = [
    "property_class",
    "internal_log2_enrichment",
    "kelm_log2_enrichment"
]

missing_columns = [
    column
    for column in required_columns
    if column not in physchem_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )


# ----------------------------------------------------------------------
# Prepare plotting table
# ----------------------------------------------------------------------

plot_df = physchem_df[
    [
        "property_class",
        "internal_log2_enrichment",
        "kelm_log2_enrichment",
        "internal_fdr",
        "kelm_fdr",
        "same_direction",
        "significant_in_both"
    ]
].copy()

plot_df["internal_log2_enrichment"] = pd.to_numeric(
    plot_df["internal_log2_enrichment"],
    errors="coerce"
)

plot_df["kelm_log2_enrichment"] = pd.to_numeric(
    plot_df["kelm_log2_enrichment"],
    errors="coerce"
)

plot_df["internal_fdr"] = pd.to_numeric(
    plot_df["internal_fdr"],
    errors="coerce"
)

plot_df["kelm_fdr"] = pd.to_numeric(
    plot_df["kelm_fdr"],
    errors="coerce"
)

plot_df = plot_df.dropna(
    subset=[
        "property_class",
        "internal_log2_enrichment",
        "kelm_log2_enrichment"
    ]
)

plot_df["property_label"] = (
    plot_df["property_class"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.title()
)

plot_df["mean_log2_enrichment"] = (
    plot_df[
        [
            "internal_log2_enrichment",
            "kelm_log2_enrichment"
        ]
    ]
    .mean(axis=1)
)

plot_df = plot_df.sort_values(
    "mean_log2_enrichment"
).reset_index(drop=True)

print("\nPhysicochemical classes used in Figure S8:")
display(plot_df)


# ----------------------------------------------------------------------
# Generate Figure S8
# ----------------------------------------------------------------------

y = range(len(plot_df))
bar_height = 0.36

fig_height = max(
    5.2,
    0.7 * len(plot_df) + 1.5
)

fig, ax = plt.subplots(
    figsize=(8.5, fig_height)
)

ax.barh(
    [value - bar_height / 2 for value in y],
    plot_df["internal_log2_enrichment"],
    height=bar_height,
    label="Internal test"
)

ax.barh(
    [value + bar_height / 2 for value in y],
    plot_df["kelm_log2_enrichment"],
    height=bar_height,
    label="KELM external"
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_yticks(list(y))

ax.set_yticklabels(
    plot_df["property_label"]
)

ax.set_xlabel(
    "log₂ enrichment"
)

ax.set_ylabel(
    "Physicochemical residue class"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()


# ----------------------------------------------------------------------
# Save PNG and PDF
# ----------------------------------------------------------------------

png_output = (
    SI_FIGURE_DIR
    / "Figure_S8_Physicochemical_enrichment.png"
)

pdf_output = (
    SI_FIGURE_DIR
    / "Figure_S8_Physicochemical_enrichment.pdf"
)

fig.savefig(
    png_output,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_output,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nFigure S8 generated successfully:")
print(png_output)
print(pdf_output)

print("\n" + "=" * 80)
print("SUPPLEMENTARY INFORMATION ANALYSES COMPLETE")
print("=" * 80)

print("\nTables completed: S1-S9")
print("Figures completed: S1-S9")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

os.chdir(PROJECT)

print("Project activated.")
print("Working directory:", PROJECT)

In [ ]:
from pathlib import Path
import pandas as pd

SI = PROJECT / "10_supplementary_information"

print("="*80)
print("SUPPLEMENTARY INFORMATION")
print("="*80)

print("\nTables:")
for f in sorted((SI/"tables").glob("*")):
    print(f.name)

print("\nFigures:")
for f in sorted((SI/"figures").glob("*")):
    print(f.name)

In [ ]:
import shutil

zip_path = "/content/Supplementary_Information.zip"

shutil.make_archive(
    "/content/Supplementary_Information",
    "zip",
    SI
)

print("ZIP created:")
print(zip_path)

In [ ]:
from google.colab import files

files.download("/content/Supplementary_Information.zip")

**fslgvsdghsedkjv**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

if not PROJECT.exists():
    raise FileNotFoundError(
        f"Project not found: {PROJECT}"
    )

os.chdir(PROJECT)

print("✓ Google Drive connected")
print("✓ Project activated")
print("Working directory:", Path.cwd())
print("Python:", sys.version.split()[0])

In [ ]:
!find /content/drive/MyDrive/pLM4CPP_XAI_2026 -type f | grep -Ei "prediction|prob|performance|metric|roc|ensemble" | head -100

In [ ]:
# ======================================================================
# MANUSCRIPT FIGURE 2 — COMPLETE REGENERATION FROM SAVED PREDICTIONS
# ======================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score
)

# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

PRED_DIR = PROJECT / "05_predictions"

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# MODELS
# ======================================================================

MODEL_DIRS = {
    "ESM2-320": "ESM2_320",
    "ESM2-640": "ESM2_640",
    "ESM2-1280": "ESM2_1280",
    "ProtT5": "ProtT5",
}

MODEL_ORDER = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean ensemble",
    "Median ensemble"
]


# ======================================================================
# COLUMN DETECTION
# ======================================================================

def find_column(df, candidates):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    # exact match
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    # partial match
    for c in df.columns:
        lc = c.lower()

        for candidate in candidates:
            if candidate.lower() in lc:
                return c

    return None


def detect_columns(df):

    label_col = find_column(
        df,
        [
            "label",
            "true_label",
            "y_true",
            "target",
            "class",
            "actual"
        ]
    )

    prob_col = find_column(
        df,
        [
            "probability",
            "prob",
            "pred_prob",
            "prediction_probability",
            "positive_probability",
            "cpp_probability",
            "score",
            "y_score"
        ]
    )

    pred_col = find_column(
        df,
        [
            "predicted_label",
            "prediction",
            "pred_label",
            "y_pred"
        ]
    )

    seq_col = find_column(
        df,
        [
            "sequence",
            "peptide",
            "seq"
        ]
    )

    id_col = find_column(
        df,
        [
            "sequence_id",
            "id",
            "peptide_id"
        ]
    )

    return {
        "label": label_col,
        "prob": prob_col,
        "pred": pred_col,
        "sequence": seq_col,
        "id": id_col
    }


# ======================================================================
# LOAD FILE
# ======================================================================

def load_prediction_file(path):

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )

    df = pd.read_csv(path)

    cols = detect_columns(df)

    if cols["label"] is None:
        raise KeyError(
            f"\nCould not detect true-label column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    if cols["prob"] is None:
        raise KeyError(
            f"\nCould not detect probability column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    return df, cols


# ======================================================================
# LOAD ALL PREDICTIONS
# ======================================================================

datasets = {
    "validation": {},
    "internal": {},
    "external": {}
}

dataset_files = {
    "validation": "validation_predictions.csv",
    "internal": "internal_test_predictions.csv",
    "external": "kelm_external_predictions.csv"
}


print("=" * 90)
print("LOADING SAVED MODEL PREDICTIONS")
print("=" * 90)

for model_name, folder_name in MODEL_DIRS.items():

    print(f"\n{model_name}")

    for dataset_name, filename in dataset_files.items():

        path = (
            PRED_DIR
            / folder_name
            / filename
        )

        df, cols = load_prediction_file(path)

        datasets[dataset_name][model_name] = {
            "df": df,
            "cols": cols
        }

        print(
            f"  {dataset_name:10s}: "
            f"{len(df):4d} rows | "
            f"label={cols['label']} | "
            f"prob={cols['prob']} | "
            f"pred={cols['pred']}"
        )


# ======================================================================
# EXTRACT TRUE LABELS AND SCORES
# ======================================================================

def extract_dataset(dataset_name):

    scores = {}

    first_model = list(MODEL_DIRS.keys())[0]

    first = datasets[dataset_name][first_model]

    y_true = pd.to_numeric(
        first["df"][first["cols"]["label"]],
        errors="raise"
    ).astype(int).to_numpy()

    for model in MODEL_DIRS:

        entry = datasets[dataset_name][model]

        model_y = pd.to_numeric(
            entry["df"][entry["cols"]["label"]],
            errors="raise"
        ).astype(int).to_numpy()

        if not np.array_equal(y_true, model_y):

            raise ValueError(
                f"Label ordering mismatch detected for "
                f"{dataset_name}: {model}"
            )

        scores[model] = pd.to_numeric(
            entry["df"][entry["cols"]["prob"]],
            errors="raise"
        ).to_numpy()

    # Mean ensemble
    score_matrix = np.column_stack(
        [
            scores["ESM2-320"],
            scores["ESM2-640"],
            scores["ESM2-1280"],
            scores["ProtT5"]
        ]
    )

    scores["Mean ensemble"] = np.mean(
        score_matrix,
        axis=1
    )

    scores["Median ensemble"] = np.median(
        score_matrix,
        axis=1
    )

    return y_true, scores


validation_y_true, validation_scores = extract_dataset(
    "validation"
)

internal_y_true, internal_scores = extract_dataset(
    "internal"
)

external_y_true, external_scores = extract_dataset(
    "external"
)


print("\n✓ Scores reconstructed successfully")

print("\nDataset sizes:")
print("Validation      :", len(validation_y_true))
print("Internal test   :", len(internal_y_true))
print("KELM external   :", len(external_y_true))


# ======================================================================
# DETERMINE VALIDATION-OPTIMAL THRESHOLDS
#
# We optimize MCC ONLY on the validation set.
# The same threshold is then applied to internal test and KELM.
# ======================================================================

def optimize_threshold_mcc(y_true, probabilities):

    thresholds = np.linspace(
        0.01,
        0.99,
        981
    )

    best_threshold = 0.5
    best_mcc = -999

    for threshold in thresholds:

        pred = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            pred
        )

        if mcc > best_mcc:

            best_mcc = mcc
            best_threshold = threshold

    return best_threshold, best_mcc


thresholds = {}

print("\n" + "=" * 90)
print("VALIDATION-OPTIMAL THRESHOLDS")
print("=" * 90)

for model in MODEL_ORDER:

    threshold, validation_mcc = optimize_threshold_mcc(
        validation_y_true,
        validation_scores[model]
    )

    thresholds[model] = threshold

    print(
        f"{model:18s} "
        f"threshold = {threshold:.3f} | "
        f"validation MCC = {validation_mcc:.3f}"
    )


# ======================================================================
# CALCULATE METRICS
# ======================================================================

def calculate_metrics(
    y_true,
    scores,
    thresholds
):

    metrics = {}

    for model in MODEL_ORDER:

        prob = scores[model]

        threshold = thresholds[model]

        pred = (
            prob >= threshold
        ).astype(int)

        metrics[model] = {

            "roc_auc":
                roc_auc_score(
                    y_true,
                    prob
                ),

            "pr_auc":
                average_precision_score(
                    y_true,
                    prob
                ),

            "mcc":
                matthews_corrcoef(
                    y_true,
                    pred
                ),

            "bacc":
                balanced_accuracy_score(
                    y_true,
                    pred
                )
        }

    return metrics


internal_metrics = calculate_metrics(
    internal_y_true,
    internal_scores,
    thresholds
)

external_metrics = calculate_metrics(
    external_y_true,
    external_scores,
    thresholds
)


# ======================================================================
# PRINT METRICS BEFORE PLOTTING
# ======================================================================

summary_rows = []

for model in MODEL_ORDER:

    summary_rows.append({

        "Model": model,

        "Internal ROC-AUC":
            internal_metrics[model]["roc_auc"],

        "Internal PR-AUC":
            internal_metrics[model]["pr_auc"],

        "Internal MCC":
            internal_metrics[model]["mcc"],

        "Internal bACC":
            internal_metrics[model]["bacc"],

        "KELM ROC-AUC":
            external_metrics[model]["roc_auc"],

        "KELM PR-AUC":
            external_metrics[model]["pr_auc"],

        "KELM MCC":
            external_metrics[model]["mcc"],

        "KELM bACC":
            external_metrics[model]["bacc"]
    })


summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 90)
print("FIGURE 2 METRICS")
print("=" * 90)

display(
    summary_df.round(3)
)


# ======================================================================
# PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 12,

    "axes.titlesize": 15,
    "axes.titleweight": "bold",

    "axes.labelsize": 13,
    "axes.labelweight": "bold",

    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,

    "legend.fontsize": 9.2,

    "axes.linewidth": 1.2,

    "lines.linewidth": 2.0,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(19, 11.5)
)

gs = fig.add_gridspec(
    2,
    3,
    width_ratios=[
        1.12,
        1.12,
        1.08
    ],
    wspace=0.28,
    hspace=0.35
)


axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axE = fig.add_subplot(gs[0, 2])

axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axF = fig.add_subplot(gs[1, 2])


# ======================================================================
# A — INTERNAL ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        internal_y_true,
        internal_scores[model]
    )

    auc_value = (
        internal_metrics[model]["roc_auc"]
    )

    axA.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axA.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.2
)

axA.set(
    xlim=(0, 1),
    ylim=(0, 1.02),

    xlabel="False-positive rate",
    ylabel="True-positive rate"
)

axA.set_title(
    "Internal test ROC",
    pad=12
)

leg = axA.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.35
)

leg.get_title().set_fontweight("bold")


# ======================================================================
# B — INTERNAL PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = (
        precision_recall_curve(
            internal_y_true,
            internal_scores[model]
        )
    )

    auc_value = (
        internal_metrics[model]["pr_auc"]
    )

    axB.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


internal_prevalence = np.mean(
    internal_y_true
)

axB.axhline(
    internal_prevalence,
    linestyle=":",
    linewidth=1.2,
    label=f"Prevalence ({internal_prevalence:.3f})"
)

axB.set(
    xlim=(0, 1),
    ylim=(0, 1.02),

    xlabel="Recall",
    ylabel="Precision"
)

axB.set_title(
    "Internal test precision–recall",
    pad=12
)

leg = axB.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.35
)

leg.get_title().set_fontweight("bold")


# ======================================================================
# C — KELM ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        external_y_true,
        external_scores[model]
    )

    auc_value = (
        external_metrics[model]["roc_auc"]
    )

    axC.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axC.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.2
)

axC.set(
    xlim=(0, 1),
    ylim=(0, 1.02),

    xlabel="False-positive rate",
    ylabel="True-positive rate"
)

axC.set_title(
    "KELM external ROC",
    pad=12
)

leg = axC.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.35
)

leg.get_title().set_fontweight("bold")


# ======================================================================
# D — KELM PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = (
        precision_recall_curve(
            external_y_true,
            external_scores[model]
        )
    )

    auc_value = (
        external_metrics[model]["pr_auc"]
    )

    axD.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


external_prevalence = np.mean(
    external_y_true
)

axD.axhline(
    external_prevalence,
    linestyle=":",
    linewidth=1.2,
    label=f"Prevalence ({external_prevalence:.3f})"
)

axD.set(
    xlim=(0, 1),
    ylim=(0, 1.02),

    xlabel="Recall",
    ylabel="Precision"
)

axD.set_title(
    "KELM external precision–recall",
    pad=12
)

leg = axD.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.35
)

leg.get_title().set_fontweight("bold")


# ======================================================================
# BAR CHART SETTINGS
# ======================================================================

x = np.arange(
    len(MODEL_ORDER)
)

width = 0.34

bar_labels = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean\nensemble",
    "Median\nensemble"
]


# ======================================================================
# E — MCC
# ======================================================================

int_mcc = [
    internal_metrics[m]["mcc"]
    for m in MODEL_ORDER
]

ext_mcc = [
    external_metrics[m]["mcc"]
    for m in MODEL_ORDER
]


bars_int = axE.bar(
    x - width/2,
    int_mcc,
    width,
    label="Internal test"
)

bars_ext = axE.bar(
    x + width/2,
    ext_mcc,
    width,
    label="KELM external"
)


axE.set_ylabel(
    "Matthews correlation coefficient"
)

axE.set_title(
    "Matthews correlation coefficient",
    pad=12
)

axE.set_xticks(x)

axE.set_xticklabels(
    bar_labels,
    rotation=28,
    ha="right"
)

axE.set_ylim(
    0.50,
    max(
        max(int_mcc),
        max(ext_mcc)
    ) + 0.10
)

axE.legend(
    frameon=False,
    loc="upper right"
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axE.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.008,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=10,
            fontweight="bold"
        )


# ======================================================================
# F — BALANCED ACCURACY
# ======================================================================

int_bacc = [
    internal_metrics[m]["bacc"]
    for m in MODEL_ORDER
]

ext_bacc = [
    external_metrics[m]["bacc"]
    for m in MODEL_ORDER
]


bars_int = axF.bar(
    x - width/2,
    int_bacc,
    width,
    label="Internal test"
)

bars_ext = axF.bar(
    x + width/2,
    ext_bacc,
    width,
    label="KELM external"
)


axF.set_ylabel(
    "Balanced accuracy"
)

axF.set_title(
    "Balanced accuracy",
    pad=12
)

axF.set_xticks(x)

axF.set_xticklabels(
    bar_labels,
    rotation=28,
    ha="right"
)

axF.set_ylim(
    0.70,
    min(
        1.00,
        max(
            max(int_bacc),
            max(ext_bacc)
        ) + 0.08
    )
)

axF.legend(
    frameon=False,
    loc="upper right"
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axF.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.006,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=10,
            fontweight="bold"
        )


# ======================================================================
# CLEAN SPINES
# ======================================================================

for ax in [
    axA,
    axB,
    axC,
    axD,
    axE,
    axF
]:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        width=1.1,
        length=4
    )


# ======================================================================
# PANEL LABELS
# ======================================================================

panel_labels = [
    (axA, "A"),
    (axB, "B"),
    (axC, "C"),
    (axD, "D"),
    (axE, "E"),
    (axF, "F")
]


for ax, label in panel_labels:

    ax.text(
        -0.16,
        1.08,

        label,

        transform=ax.transAxes,

        fontsize=21,
        fontweight="bold",

        va="top",
        ha="left"
    )


# ======================================================================
# FINAL SPACING
# ======================================================================

fig.subplots_adjust(
    left=0.06,
    right=0.985,
    bottom=0.11,
    top=0.94,
    wspace=0.30,
    hspace=0.36
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_2_Model_performance_improved.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_2_Model_performance_improved.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_2_Model_performance_improved.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

fig.savefig(
    svg_file,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 2 REGENERATED SUCCESSFULLY")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# MANUSCRIPT FIGURE 2 — FINAL IMPROVED VERSION
# Larger + bold X/Y tick values and clearer publication formatting
# ======================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score
)

# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

PRED_DIR = PROJECT / "05_predictions"

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# MODELS
# ======================================================================

MODEL_DIRS = {
    "ESM2-320": "ESM2_320",
    "ESM2-640": "ESM2_640",
    "ESM2-1280": "ESM2_1280",
    "ProtT5": "ProtT5",
}

MODEL_ORDER = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean ensemble",
    "Median ensemble"
]


# ======================================================================
# COLUMN DETECTION
# ======================================================================

def find_column(df, candidates):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    for c in df.columns:
        lc = c.lower()

        for candidate in candidates:
            if candidate.lower() in lc:
                return c

    return None


def detect_columns(df):

    label_col = find_column(
        df,
        [
            "label",
            "true_label",
            "y_true",
            "target",
            "class",
            "actual"
        ]
    )

    prob_col = find_column(
        df,
        [
            "probability",
            "prob",
            "pred_prob",
            "prediction_probability",
            "positive_probability",
            "cpp_probability",
            "score",
            "y_score"
        ]
    )

    pred_col = find_column(
        df,
        [
            "predicted_label",
            "prediction",
            "pred_label",
            "y_pred"
        ]
    )

    seq_col = find_column(
        df,
        [
            "sequence",
            "peptide",
            "seq"
        ]
    )

    id_col = find_column(
        df,
        [
            "sequence_id",
            "id",
            "peptide_id"
        ]
    )

    return {
        "label": label_col,
        "prob": prob_col,
        "pred": pred_col,
        "sequence": seq_col,
        "id": id_col
    }


# ======================================================================
# LOAD PREDICTION FILE
# ======================================================================

def load_prediction_file(path):

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )

    df = pd.read_csv(path)

    cols = detect_columns(df)

    if cols["label"] is None:
        raise KeyError(
            f"\nCould not detect true-label column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    if cols["prob"] is None:
        raise KeyError(
            f"\nCould not detect probability column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    return df, cols


# ======================================================================
# LOAD ALL SAVED PREDICTIONS
# ======================================================================

datasets = {
    "validation": {},
    "internal": {},
    "external": {}
}

dataset_files = {
    "validation": "validation_predictions.csv",
    "internal": "internal_test_predictions.csv",
    "external": "kelm_external_predictions.csv"
}


print("=" * 90)
print("LOADING SAVED MODEL PREDICTIONS")
print("=" * 90)

for model_name, folder_name in MODEL_DIRS.items():

    print(f"\n{model_name}")

    for dataset_name, filename in dataset_files.items():

        path = (
            PRED_DIR
            / folder_name
            / filename
        )

        df, cols = load_prediction_file(path)

        datasets[dataset_name][model_name] = {
            "df": df,
            "cols": cols
        }

        print(
            f"  {dataset_name:10s}: "
            f"{len(df):4d} rows | "
            f"label={cols['label']} | "
            f"prob={cols['prob']} | "
            f"pred={cols['pred']}"
        )


# ======================================================================
# EXTRACT TRUE LABELS AND MODEL SCORES
# ======================================================================

def extract_dataset(dataset_name):

    scores = {}

    first_model = list(MODEL_DIRS.keys())[0]

    first = datasets[dataset_name][first_model]

    y_true = pd.to_numeric(
        first["df"][first["cols"]["label"]],
        errors="raise"
    ).astype(int).to_numpy()

    for model in MODEL_DIRS:

        entry = datasets[dataset_name][model]

        model_y = pd.to_numeric(
            entry["df"][entry["cols"]["label"]],
            errors="raise"
        ).astype(int).to_numpy()

        if not np.array_equal(y_true, model_y):

            raise ValueError(
                f"Label ordering mismatch detected for "
                f"{dataset_name}: {model}"
            )

        scores[model] = pd.to_numeric(
            entry["df"][entry["cols"]["prob"]],
            errors="raise"
        ).to_numpy()

    score_matrix = np.column_stack(
        [
            scores["ESM2-320"],
            scores["ESM2-640"],
            scores["ESM2-1280"],
            scores["ProtT5"]
        ]
    )

    scores["Mean ensemble"] = np.mean(
        score_matrix,
        axis=1
    )

    scores["Median ensemble"] = np.median(
        score_matrix,
        axis=1
    )

    return y_true, scores


validation_y_true, validation_scores = extract_dataset(
    "validation"
)

internal_y_true, internal_scores = extract_dataset(
    "internal"
)

external_y_true, external_scores = extract_dataset(
    "external"
)


print("\n✓ Scores reconstructed successfully")

print("\nDataset sizes:")
print("Validation      :", len(validation_y_true))
print("Internal test   :", len(internal_y_true))
print("KELM external   :", len(external_y_true))


# ======================================================================
# VALIDATION-OPTIMAL THRESHOLDS
# ======================================================================

def optimize_threshold_mcc(y_true, probabilities):

    thresholds_scan = np.linspace(
        0.01,
        0.99,
        981
    )

    best_threshold = 0.5
    best_mcc = -999

    for threshold in thresholds_scan:

        pred = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            pred
        )

        if mcc > best_mcc:

            best_mcc = mcc
            best_threshold = threshold

    return best_threshold, best_mcc


thresholds = {}

print("\n" + "=" * 90)
print("VALIDATION-OPTIMAL THRESHOLDS")
print("=" * 90)

for model in MODEL_ORDER:

    threshold, validation_mcc = optimize_threshold_mcc(
        validation_y_true,
        validation_scores[model]
    )

    thresholds[model] = threshold

    print(
        f"{model:18s} "
        f"threshold = {threshold:.3f} | "
        f"validation MCC = {validation_mcc:.3f}"
    )


# ======================================================================
# CALCULATE METRICS
# ======================================================================

def calculate_metrics(
    y_true,
    scores,
    thresholds
):

    metrics = {}

    for model in MODEL_ORDER:

        prob = scores[model]

        threshold = thresholds[model]

        pred = (
            prob >= threshold
        ).astype(int)

        metrics[model] = {

            "roc_auc":
                roc_auc_score(
                    y_true,
                    prob
                ),

            "pr_auc":
                average_precision_score(
                    y_true,
                    prob
                ),

            "mcc":
                matthews_corrcoef(
                    y_true,
                    pred
                ),

            "bacc":
                balanced_accuracy_score(
                    y_true,
                    pred
                )
        }

    return metrics


internal_metrics = calculate_metrics(
    internal_y_true,
    internal_scores,
    thresholds
)

external_metrics = calculate_metrics(
    external_y_true,
    external_scores,
    thresholds
)


# ======================================================================
# PRINT METRICS
# ======================================================================

summary_rows = []

for model in MODEL_ORDER:

    summary_rows.append({

        "Model": model,

        "Internal ROC-AUC":
            internal_metrics[model]["roc_auc"],

        "Internal PR-AUC":
            internal_metrics[model]["pr_auc"],

        "Internal MCC":
            internal_metrics[model]["mcc"],

        "Internal bACC":
            internal_metrics[model]["bacc"],

        "KELM ROC-AUC":
            external_metrics[model]["roc_auc"],

        "KELM PR-AUC":
            external_metrics[model]["pr_auc"],

        "KELM MCC":
            external_metrics[model]["mcc"],

        "KELM bACC":
            external_metrics[model]["bacc"]
    })


summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("FIGURE 2 METRICS")
print("=" * 90)

display(
    summary_df.round(3)
)


# ======================================================================
# PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 13,

    # Panel titles
    "axes.titlesize": 16,
    "axes.titleweight": "bold",

    # X/Y axis titles
    "axes.labelsize": 15,
    "axes.labelweight": "bold",

    # X/Y tick values
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,

    # Tick thickness
    "xtick.major.width": 1.4,
    "ytick.major.width": 1.4,

    # Legend
    "legend.fontsize": 10,

    # Axes
    "axes.linewidth": 1.4,

    # Curves
    "lines.linewidth": 2.2,

    # Save
    "savefig.dpi": 600
})


# ======================================================================
# FIGURE LAYOUT
# ======================================================================

fig = plt.figure(
    figsize=(19.5, 12)
)

gs = fig.add_gridspec(
    2,
    3,
    width_ratios=[
        1.12,
        1.12,
        1.08
    ],
    wspace=0.30,
    hspace=0.37
)


axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axE = fig.add_subplot(gs[0, 2])

axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axF = fig.add_subplot(gs[1, 2])


# ======================================================================
# PANEL A — INTERNAL ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        internal_y_true,
        internal_scores[model]
    )

    auc_value = (
        internal_metrics[model]["roc_auc"]
    )

    axA.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axA.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.4
)

axA.set_xlim(0, 1)
axA.set_ylim(0, 1.02)

axA.set_xlabel(
    "False-positive rate"
)

axA.set_ylabel(
    "True-positive rate"
)

axA.set_title(
    "Internal test ROC",
    pad=12
)

leg = axA.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL B — INTERNAL PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = precision_recall_curve(
        internal_y_true,
        internal_scores[model]
    )

    auc_value = (
        internal_metrics[model]["pr_auc"]
    )

    axB.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


internal_prevalence = np.mean(
    internal_y_true
)

axB.axhline(
    internal_prevalence,
    linestyle=":",
    linewidth=1.4,
    label=f"Prevalence ({internal_prevalence:.3f})"
)

axB.set_xlim(0, 1)
axB.set_ylim(0, 1.02)

axB.set_xlabel(
    "Recall"
)

axB.set_ylabel(
    "Precision"
)

axB.set_title(
    "Internal test precision–recall",
    pad=12
)

leg = axB.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL C — KELM EXTERNAL ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        external_y_true,
        external_scores[model]
    )

    auc_value = (
        external_metrics[model]["roc_auc"]
    )

    axC.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axC.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.4
)

axC.set_xlim(0, 1)
axC.set_ylim(0, 1.02)

axC.set_xlabel(
    "False-positive rate"
)

axC.set_ylabel(
    "True-positive rate"
)

axC.set_title(
    "KELM external ROC",
    pad=12
)

leg = axC.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL D — KELM EXTERNAL PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = precision_recall_curve(
        external_y_true,
        external_scores[model]
    )

    auc_value = (
        external_metrics[model]["pr_auc"]
    )

    axD.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


external_prevalence = np.mean(
    external_y_true
)

axD.axhline(
    external_prevalence,
    linestyle=":",
    linewidth=1.4,
    label=f"Prevalence ({external_prevalence:.3f})"
)

axD.set_xlim(0, 1)
axD.set_ylim(0, 1.02)

axD.set_xlabel(
    "Recall"
)

axD.set_ylabel(
    "Precision"
)

axD.set_title(
    "KELM external precision–recall",
    pad=12
)

leg = axD.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# BAR CHART SETTINGS
# ======================================================================

x = np.arange(
    len(MODEL_ORDER)
)

width = 0.34

bar_labels = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean\nensemble",
    "Median\nensemble"
]


# ======================================================================
# PANEL E — MCC
# ======================================================================

int_mcc = [
    internal_metrics[m]["mcc"]
    for m in MODEL_ORDER
]

ext_mcc = [
    external_metrics[m]["mcc"]
    for m in MODEL_ORDER
]


bars_int = axE.bar(
    x - width/2,
    int_mcc,
    width,
    label="Internal test"
)

bars_ext = axE.bar(
    x + width/2,
    ext_mcc,
    width,
    label="KELM external"
)


axE.set_ylabel(
    "Matthews correlation coefficient"
)

axE.set_title(
    "Matthews correlation coefficient",
    pad=12
)

axE.set_xticks(x)

axE.set_xticklabels(
    bar_labels,
    rotation=27,
    ha="right"
)

axE.set_ylim(
    0.50,
    max(
        max(int_mcc),
        max(ext_mcc)
    ) + 0.10
)

axE.legend(
    frameon=False,
    loc="upper right",
    fontsize=10.5
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axE.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.008,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold"
        )


# ======================================================================
# PANEL F — BALANCED ACCURACY
# ======================================================================

int_bacc = [
    internal_metrics[m]["bacc"]
    for m in MODEL_ORDER
]

ext_bacc = [
    external_metrics[m]["bacc"]
    for m in MODEL_ORDER
]


bars_int = axF.bar(
    x - width/2,
    int_bacc,
    width,
    label="Internal test"
)

bars_ext = axF.bar(
    x + width/2,
    ext_bacc,
    width,
    label="KELM external"
)


axF.set_ylabel(
    "Balanced accuracy"
)

axF.set_title(
    "Balanced accuracy",
    pad=12
)

axF.set_xticks(x)

axF.set_xticklabels(
    bar_labels,
    rotation=27,
    ha="right"
)

axF.set_ylim(
    0.70,
    min(
        1.00,
        max(
            max(int_bacc),
            max(ext_bacc)
        ) + 0.08
    )
)

axF.legend(
    frameon=False,
    loc="upper right",
    fontsize=10.5
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axF.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.006,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold"
        )


# ======================================================================
# AXIS STYLE
# Larger + bold numeric X/Y tick values
# ======================================================================

for ax in [
    axA,
    axB,
    axC,
    axD,
    axE,
    axF
]:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(1.4)
    ax.spines["bottom"].set_linewidth(1.4)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=13,
        width=1.4,
        length=5
    )

    # Bold all X-axis tick labels
    for label in ax.get_xticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            13
        )

    # Bold all Y-axis tick labels
    for label in ax.get_yticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            13
        )


# ======================================================================
# Slightly smaller X labels for E/F model names
# ======================================================================

for label in axE.get_xticklabels():

    label.set_fontsize(12)
    label.set_fontweight("bold")


for label in axF.get_xticklabels():

    label.set_fontsize(12)
    label.set_fontweight("bold")


# ======================================================================
# PANEL LABELS
# ======================================================================

panel_labels = [
    (axA, "A"),
    (axB, "B"),
    (axC, "C"),
    (axD, "D"),
    (axE, "E"),
    (axF, "F")
]


for ax, label in panel_labels:

    ax.text(
        -0.16,
        1.08,

        label,

        transform=ax.transAxes,

        fontsize=22,
        fontweight="bold",

        va="top",
        ha="left"
    )


# ======================================================================
# FINAL SPACING
# ======================================================================

fig.subplots_adjust(
    left=0.062,
    right=0.985,
    bottom=0.115,
    top=0.94,
    wspace=0.31,
    hspace=0.38
)


# ======================================================================
# SAVE FINAL FIGURE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

fig.savefig(
    svg_file,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FINAL FIGURE 2 GENERATED SUCCESSFULLY")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# MANUSCRIPT FIGURE 2 — FINAL IMPROVED VERSION
# Larger + bold X/Y tick values and clearer publication formatting
# ======================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score
)

# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

PRED_DIR = PROJECT / "05_predictions"

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# MODELS
# ======================================================================

MODEL_DIRS = {
    "ESM2-320": "ESM2_320",
    "ESM2-640": "ESM2_640",
    "ESM2-1280": "ESM2_1280",
    "ProtT5": "ProtT5",
}

MODEL_ORDER = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean ensemble",
    "Median ensemble"
]


# ======================================================================
# COLUMN DETECTION
# ======================================================================

def find_column(df, candidates):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    for c in df.columns:
        lc = c.lower()

        for candidate in candidates:
            if candidate.lower() in lc:
                return c

    return None


def detect_columns(df):

    label_col = find_column(
        df,
        [
            "label",
            "true_label",
            "y_true",
            "target",
            "class",
            "actual"
        ]
    )

    prob_col = find_column(
        df,
        [
            "probability",
            "prob",
            "pred_prob",
            "prediction_probability",
            "positive_probability",
            "cpp_probability",
            "score",
            "y_score"
        ]
    )

    pred_col = find_column(
        df,
        [
            "predicted_label",
            "prediction",
            "pred_label",
            "y_pred"
        ]
    )

    seq_col = find_column(
        df,
        [
            "sequence",
            "peptide",
            "seq"
        ]
    )

    id_col = find_column(
        df,
        [
            "sequence_id",
            "id",
            "peptide_id"
        ]
    )

    return {
        "label": label_col,
        "prob": prob_col,
        "pred": pred_col,
        "sequence": seq_col,
        "id": id_col
    }


# ======================================================================
# LOAD PREDICTION FILE
# ======================================================================

def load_prediction_file(path):

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )

    df = pd.read_csv(path)

    cols = detect_columns(df)

    if cols["label"] is None:
        raise KeyError(
            f"\nCould not detect true-label column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    if cols["prob"] is None:
        raise KeyError(
            f"\nCould not detect probability column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    return df, cols


# ======================================================================
# LOAD ALL SAVED PREDICTIONS
# ======================================================================

datasets = {
    "validation": {},
    "internal": {},
    "external": {}
}

dataset_files = {
    "validation": "validation_predictions.csv",
    "internal": "internal_test_predictions.csv",
    "external": "kelm_external_predictions.csv"
}


print("=" * 90)
print("LOADING SAVED MODEL PREDICTIONS")
print("=" * 90)

for model_name, folder_name in MODEL_DIRS.items():

    print(f"\n{model_name}")

    for dataset_name, filename in dataset_files.items():

        path = (
            PRED_DIR
            / folder_name
            / filename
        )

        df, cols = load_prediction_file(path)

        datasets[dataset_name][model_name] = {
            "df": df,
            "cols": cols
        }

        print(
            f"  {dataset_name:10s}: "
            f"{len(df):4d} rows | "
            f"label={cols['label']} | "
            f"prob={cols['prob']} | "
            f"pred={cols['pred']}"
        )


# ======================================================================
# EXTRACT TRUE LABELS AND MODEL SCORES
# ======================================================================

def extract_dataset(dataset_name):

    scores = {}

    first_model = list(MODEL_DIRS.keys())[0]

    first = datasets[dataset_name][first_model]

    y_true = pd.to_numeric(
        first["df"][first["cols"]["label"]],
        errors="raise"
    ).astype(int).to_numpy()

    for model in MODEL_DIRS:

        entry = datasets[dataset_name][model]

        model_y = pd.to_numeric(
            entry["df"][entry["cols"]["label"]],
            errors="raise"
        ).astype(int).to_numpy()

        if not np.array_equal(y_true, model_y):

            raise ValueError(
                f"Label ordering mismatch detected for "
                f"{dataset_name}: {model}"
            )

        scores[model] = pd.to_numeric(
            entry["df"][entry["cols"]["prob"]],
            errors="raise"
        ).to_numpy()

    score_matrix = np.column_stack(
        [
            scores["ESM2-320"],
            scores["ESM2-640"],
            scores["ESM2-1280"],
            scores["ProtT5"]
        ]
    )

    scores["Mean ensemble"] = np.mean(
        score_matrix,
        axis=1
    )

    scores["Median ensemble"] = np.median(
        score_matrix,
        axis=1
    )

    return y_true, scores


validation_y_true, validation_scores = extract_dataset(
    "validation"
)

internal_y_true, internal_scores = extract_dataset(
    "internal"
)

external_y_true, external_scores = extract_dataset(
    "external"
)


print("\n✓ Scores reconstructed successfully")

print("\nDataset sizes:")
print("Validation      :", len(validation_y_true))
print("Internal test   :", len(internal_y_true))
print("KELM external   :", len(external_y_true))


# ======================================================================
# VALIDATION-OPTIMAL THRESHOLDS
# ======================================================================

def optimize_threshold_mcc(y_true, probabilities):

    thresholds_scan = np.linspace(
        0.01,
        0.99,
        981
    )

    best_threshold = 0.5
    best_mcc = -999

    for threshold in thresholds_scan:

        pred = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            pred
        )

        if mcc > best_mcc:

            best_mcc = mcc
            best_threshold = threshold

    return best_threshold, best_mcc


thresholds = {}

print("\n" + "=" * 90)
print("VALIDATION-OPTIMAL THRESHOLDS")
print("=" * 90)

for model in MODEL_ORDER:

    threshold, validation_mcc = optimize_threshold_mcc(
        validation_y_true,
        validation_scores[model]
    )

    thresholds[model] = threshold

    print(
        f"{model:18s} "
        f"threshold = {threshold:.3f} | "
        f"validation MCC = {validation_mcc:.3f}"
    )


# ======================================================================
# CALCULATE METRICS
# ======================================================================

def calculate_metrics(
    y_true,
    scores,
    thresholds
):

    metrics = {}

    for model in MODEL_ORDER:

        prob = scores[model]

        threshold = thresholds[model]

        pred = (
            prob >= threshold
        ).astype(int)

        metrics[model] = {

            "roc_auc":
                roc_auc_score(
                    y_true,
                    prob
                ),

            "pr_auc":
                average_precision_score(
                    y_true,
                    prob
                ),

            "mcc":
                matthews_corrcoef(
                    y_true,
                    pred
                ),

            "bacc":
                balanced_accuracy_score(
                    y_true,
                    pred
                )
        }

    return metrics


internal_metrics = calculate_metrics(
    internal_y_true,
    internal_scores,
    thresholds
)

external_metrics = calculate_metrics(
    external_y_true,
    external_scores,
    thresholds
)


# ======================================================================
# PRINT METRICS
# ======================================================================

summary_rows = []

for model in MODEL_ORDER:

    summary_rows.append({

        "Model": model,

        "Internal ROC-AUC":
            internal_metrics[model]["roc_auc"],

        "Internal PR-AUC":
            internal_metrics[model]["pr_auc"],

        "Internal MCC":
            internal_metrics[model]["mcc"],

        "Internal bACC":
            internal_metrics[model]["bacc"],

        "KELM ROC-AUC":
            external_metrics[model]["roc_auc"],

        "KELM PR-AUC":
            external_metrics[model]["pr_auc"],

        "KELM MCC":
            external_metrics[model]["mcc"],

        "KELM bACC":
            external_metrics[model]["bacc"]
    })


summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("FIGURE 2 METRICS")
print("=" * 90)

display(
    summary_df.round(3)
)


# ======================================================================
# PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 13,

    # Panel titles
    "axes.titlesize": 16,
    "axes.titleweight": "bold",

    # X/Y axis titles
    "axes.labelsize": 15,
    "axes.labelweight": "bold",

    # X/Y tick values
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,

    # Tick thickness
    "xtick.major.width": 1.4,
    "ytick.major.width": 1.4,

    # Legend
    "legend.fontsize": 10,

    # Axes
    "axes.linewidth": 1.4,

    # Curves
    "lines.linewidth": 2.2,

    # Save
    "savefig.dpi": 600
})


# ======================================================================
# FIGURE LAYOUT
# ======================================================================

fig = plt.figure(
    figsize=(19.5, 12)
)

gs = fig.add_gridspec(
    2,
    3,
    width_ratios=[
        1.12,
        1.12,
        1.08
    ],
    wspace=0.30,
    hspace=0.37
)


axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axE = fig.add_subplot(gs[0, 2])

axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axF = fig.add_subplot(gs[1, 2])


# ======================================================================
# PANEL A — INTERNAL ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        internal_y_true,
        internal_scores[model]
    )

    auc_value = (
        internal_metrics[model]["roc_auc"]
    )

    axA.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axA.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.4
)

axA.set_xlim(0, 1)
axA.set_ylim(0, 1.02)

axA.set_xlabel(
    "False-positive rate"
)

axA.set_ylabel(
    "True-positive rate"
)

leg = axA.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL B — INTERNAL PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = precision_recall_curve(
        internal_y_true,
        internal_scores[model]
    )

    auc_value = (
        internal_metrics[model]["pr_auc"]
    )

    axB.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


internal_prevalence = np.mean(
    internal_y_true
)

axB.axhline(
    internal_prevalence,
    linestyle=":",
    linewidth=1.4,
    label=f"Prevalence ({internal_prevalence:.3f})"
)

axB.set_xlim(0, 1)
axB.set_ylim(0, 1.02)

axB.set_xlabel(
    "Recall"
)

axB.set_ylabel(
    "Precision"
)

leg = axB.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL C — KELM EXTERNAL ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        external_y_true,
        external_scores[model]
    )

    auc_value = (
        external_metrics[model]["roc_auc"]
    )

    axC.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axC.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.4
)

axC.set_xlim(0, 1)
axC.set_ylim(0, 1.02)

axC.set_xlabel(
    "False-positive rate"
)

axC.set_ylabel(
    "True-positive rate"
)

leg = axC.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL D — KELM EXTERNAL PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = precision_recall_curve(
        external_y_true,
        external_scores[model]
    )

    auc_value = (
        external_metrics[model]["pr_auc"]
    )

    axD.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


external_prevalence = np.mean(
    external_y_true
)

axD.axhline(
    external_prevalence,
    linestyle=":",
    linewidth=1.4,
    label=f"Prevalence ({external_prevalence:.3f})"
)

axD.set_xlim(0, 1)
axD.set_ylim(0, 1.02)

axD.set_xlabel(
    "Recall"
)

axD.set_ylabel(
    "Precision"
)

leg = axD.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# BAR CHART SETTINGS
# ======================================================================

x = np.arange(
    len(MODEL_ORDER)
)

width = 0.34

bar_labels = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean\nensemble",
    "Median\nensemble"
]


# ======================================================================
# PANEL E — MCC
# ======================================================================

int_mcc = [
    internal_metrics[m]["mcc"]
    for m in MODEL_ORDER
]

ext_mcc = [
    external_metrics[m]["mcc"]
    for m in MODEL_ORDER
]


bars_int = axE.bar(
    x - width/2,
    int_mcc,
    width,
    label="Internal test"
)

bars_ext = axE.bar(
    x + width/2,
    ext_mcc,
    width,
    label="KELM external"
)


axE.set_ylabel(
    "Matthews correlation coefficient"
)

axE.set_xticks(x)

axE.set_xticklabels(
    bar_labels,
    rotation=27,
    ha="right"
)

axE.set_ylim(
    0.50,
    max(
        max(int_mcc),
        max(ext_mcc)
    ) + 0.10
)

axE.legend(
    frameon=False,
    loc="upper right",
    fontsize=10.5
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axE.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.008,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold"
        )


# ======================================================================
# PANEL F — BALANCED ACCURACY
# ======================================================================

int_bacc = [
    internal_metrics[m]["bacc"]
    for m in MODEL_ORDER
]

ext_bacc = [
    external_metrics[m]["bacc"]
    for m in MODEL_ORDER
]


bars_int = axF.bar(
    x - width/2,
    int_bacc,
    width,
    label="Internal test"
)

bars_ext = axF.bar(
    x + width/2,
    ext_bacc,
    width,
    label="KELM external"
)


axF.set_ylabel(
    "Balanced accuracy"
)

axF.set_xticks(x)

axF.set_xticklabels(
    bar_labels,
    rotation=27,
    ha="right"
)

axF.set_ylim(
    0.70,
    min(
        1.00,
        max(
            max(int_bacc),
            max(ext_bacc)
        ) + 0.08
    )
)

axF.legend(
    frameon=False,
    loc="upper right",
    fontsize=10.5
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axF.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.006,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold"
        )


# ======================================================================
# AXIS STYLE
# Larger + bold numeric X/Y tick values
# ======================================================================

for ax in [
    axA,
    axB,
    axC,
    axD,
    axE,
    axF
]:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(1.4)
    ax.spines["bottom"].set_linewidth(1.4)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=13,
        width=1.4,
        length=5
    )

    # Bold all X-axis tick labels
    for label in ax.get_xticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            13
        )

    # Bold all Y-axis tick labels
    for label in ax.get_yticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            13
        )


# ======================================================================
# Slightly smaller X labels for E/F model names
# ======================================================================

for label in axE.get_xticklabels():

    label.set_fontsize(12)
    label.set_fontweight("bold")


for label in axF.get_xticklabels():

    label.set_fontsize(12)
    label.set_fontweight("bold")


# ======================================================================
# PANEL LABELS
# ======================================================================

panel_labels = [
    (axA, "A"),
    (axB, "B"),
    (axC, "C"),
    (axD, "D"),
    (axE, "E"),
    (axF, "F")
]


for ax, label in panel_labels:

    ax.text(
        -0.16,
        1.08,

        label,

        transform=ax.transAxes,

        fontsize=22,
        fontweight="bold",

        va="top",
        ha="left"
    )


# ======================================================================
# FINAL SPACING
# ======================================================================

fig.subplots_adjust(
    left=0.062,
    right=0.985,
    bottom=0.115,
    top=0.94,
    wspace=0.31,
    hspace=0.38
)


# ======================================================================
# SAVE FINAL FIGURE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

fig.savefig(
    svg_file,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FINAL FIGURE 2 GENERATED SUCCESSFULLY")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/final_manuscript_package/03_main_figures/Figure_2_Model_performance_FINAL.png"
)

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

print("=" * 100)
print("POSSIBLE FIGURE 3 SOURCE FILES")
print("=" * 100)

keywords = [
    "agreement",
    "attribution",
    "consensus",
    "stability",
    "spearman",
    "jaccard",
    "representative"
]

files_found = []

for f in PROJECT.rglob("*.csv"):
    name = str(f).lower()

    if any(k in name for k in keywords):
        files_found.append(f)

for f in sorted(files_found):
    print(f)

In [ ]:
# ======================================================================
# MANUSCRIPT FIGURE 3 — FINAL IMPROVED VERSION
# Construction, Agreement, and Stability of Multi-PLM Explainable AI
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle

# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

TABLE_DIR = PROJECT / "07_results" / "tables_main"

XAI_DIR = PROJECT / "06_xai"

CHECKPOINT_DIR = PROJECT / "08_checkpoints"

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# INPUT FILES
# ======================================================================

METHOD_AGREEMENT_FILE = (
    TABLE_DIR / "xai_method_agreement.csv"
)

JACCARD_INTERNAL_FILE = (
    TABLE_DIR / "hotspot_jaccard_matrix_internal_test_all.csv"
)

JACCARD_KELM_FILE = (
    TABLE_DIR / "hotspot_jaccard_matrix_kelm_external_all.csv"
)

ORIGINAL_ADJUSTED_FILE = (
    TABLE_DIR / "original_vs_adjusted_consensus.csv"
)

REPRESENTATIVE_FILE = (
    CHECKPOINT_DIR / "figure_4_representative_sequences.csv"
)

GLOBAL_INTERNAL_FILE = (
    XAI_DIR
    / "consensus_adjusted"
    / "global"
    / "internal_test"
    / "adjusted_global_consensus_residue_scores.csv"
)


# ======================================================================
# GLOBAL PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 13,

    # Titles
    "axes.titlesize": 16,
    "axes.titleweight": "bold",

    # Axis titles
    "axes.labelsize": 15,
    "axes.labelweight": "bold",

    # Tick labels
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,

    # Legend
    "legend.fontsize": 11,

    # Axis thickness
    "axes.linewidth": 1.4,

    # Saving
    "savefig.dpi": 600
})


# ======================================================================
# HELPER FUNCTIONS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def find_col(df, candidates):

    cols = list(df.columns)

    # Exact match first
    for candidate in candidates:
        if candidate in cols:
            return candidate

    # Partial matching second
    for candidate in candidates:
        for c in cols:
            if candidate in c:
                return c

    return None


def bold_ticks(ax, xsize=13, ysize=13):

    ax.tick_params(
        axis="both",
        which="major",
        width=1.4,
        length=5,
        labelsize=max(xsize, ysize)
    )

    for label in ax.get_xticklabels():
        label.set_fontsize(xsize)
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontsize(ysize)
        label.set_fontweight("bold")


def clean_spines(ax):

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(1.4)
    ax.spines["bottom"].set_linewidth(1.4)


def annotate_heatmap(ax, matrix, fontsize=11):

    arr = np.asarray(matrix, dtype=float)

    midpoint = (
        np.nanmin(arr)
        + np.nanmax(arr)
    ) / 2

    for i in range(arr.shape[0]):

        for j in range(arr.shape[1]):

            value = arr[i, j]

            if np.isnan(value):
                continue

            text_color = (
                "white"
                if value > midpoint
                else "black"
            )

            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=fontsize,
                fontweight="bold",
                color=text_color
            )


# ======================================================================
# LOAD DATA
# ======================================================================

print("=" * 90)
print("LOADING FIGURE 3 SOURCE FILES")
print("=" * 90)

required_files = [
    METHOD_AGREEMENT_FILE,
    JACCARD_INTERNAL_FILE,
    JACCARD_KELM_FILE,
    ORIGINAL_ADJUSTED_FILE,
    REPRESENTATIVE_FILE,
    GLOBAL_INTERNAL_FILE
]

for f in required_files:

    if not f.exists():

        raise FileNotFoundError(
            f"Missing required file:\n{f}"
        )

    print("✓", f)


method_df = clean_columns(
    pd.read_csv(METHOD_AGREEMENT_FILE)
)

internal_jaccard_raw = pd.read_csv(
    JACCARD_INTERNAL_FILE,
    index_col=0
)

kelm_jaccard_raw = pd.read_csv(
    JACCARD_KELM_FILE,
    index_col=0
)

adjusted_df = clean_columns(
    pd.read_csv(ORIGINAL_ADJUSTED_FILE)
)

representative_df = clean_columns(
    pd.read_csv(REPRESENTATIVE_FILE)
)

global_df = clean_columns(
    pd.read_csv(GLOBAL_INTERNAL_FILE)
)


print("\nMethod agreement columns:")
print(method_df.columns.tolist())

print("\nOriginal vs adjusted columns:")
print(adjusted_df.columns.tolist())

print("\nRepresentative sequence columns:")
print(representative_df.columns.tolist())

print("\nGlobal consensus columns:")
print(global_df.columns.tolist())


# ======================================================================
# PANEL A — REPRESENTATIVE SEQUENCE
# ======================================================================

# ----------------------------------------------------------------------
# Identify representative sequence
# ----------------------------------------------------------------------

sequence_col = find_col(
    representative_df,
    [
        "sequence",
        "peptide_sequence",
        "seq"
    ]
)

id_col = find_col(
    representative_df,
    [
        "sequence_id",
        "seq_id",
        "id"
    ]
)

if sequence_col is None:

    raise KeyError(
        "Could not identify representative sequence column.\n"
        f"Columns: {representative_df.columns.tolist()}"
    )


representative_sequence = str(
    representative_df.iloc[0][sequence_col]
)

representative_id = (
    str(representative_df.iloc[0][id_col])
    if id_col is not None
    else "Representative CPP"
)


print("\nRepresentative sequence:")
print(representative_id)
print(representative_sequence)


# ----------------------------------------------------------------------
# Match representative sequence in global consensus file
# ----------------------------------------------------------------------

global_sequence_col = find_col(
    global_df,
    [
        "sequence",
        "peptide_sequence",
        "seq"
    ]
)

global_id_col = find_col(
    global_df,
    [
        "sequence_id",
        "seq_id",
        "id"
    ]
)

position_col = find_col(
    global_df,
    [
        "position",
        "residue_position",
        "pos"
    ]
)

residue_col = find_col(
    global_df,
    [
        "residue",
        "amino_acid",
        "aa"
    ]
)


if global_sequence_col is not None:

    rep_global = global_df[
        global_df[global_sequence_col].astype(str)
        == representative_sequence
    ].copy()

elif (
    global_id_col is not None
    and id_col is not None
):

    rep_global = global_df[
        global_df[global_id_col].astype(str)
        == representative_id
    ].copy()

else:

    raise KeyError(
        "Unable to match representative sequence "
        "to global consensus file."
    )


if rep_global.empty:

    raise ValueError(
        "Representative sequence was not found "
        "in adjusted global consensus data."
    )


if position_col is not None:

    rep_global = rep_global.sort_values(
        position_col
    )


# ======================================================================
# DETECT PANEL A SCORE COLUMNS
# ======================================================================

attention_col = find_col(
    rep_global,
    [
        "attention_score",
        "attention",
        "mean_attention"
    ]
)

grad_col = find_col(
    rep_global,
    [
        "gradient_x_input",
        "gradient_input",
        "grad_x_input",
        "gradient_score"
    ]
)

ig_col = find_col(
    rep_global,
    [
        "integrated_gradients",
        "integrated_gradient",
        "ig_score"
    ]
)

consensus_col = find_col(
    rep_global,
    [
        "adjusted_consensus_score",
        "global_adjusted_score",
        "adjusted_score",
        "consensus_score",
        "normalized_score"
    ]
)


# ----------------------------------------------------------------------
# Fallback:
# if global table does not contain method-level columns,
# load one model's adjusted model consensus.
# ----------------------------------------------------------------------

if (
    attention_col is None
    or grad_col is None
    or ig_col is None
):

    MODEL_FOR_PANEL_A = "ESM2_1280"

    model_file = (
        XAI_DIR
        / "consensus_adjusted"
        / MODEL_FOR_PANEL_A
        / "internal_test"
        / "adjusted_model_consensus.csv"
    )

    model_df = clean_columns(
        pd.read_csv(model_file)
    )

    model_sequence_col = find_col(
        model_df,
        ["sequence", "peptide_sequence", "seq"]
    )

    model_id_col = find_col(
        model_df,
        ["sequence_id", "seq_id", "id"]
    )

    model_position_col = find_col(
        model_df,
        ["position", "residue_position", "pos"]
    )

    if model_sequence_col is not None:

        rep_model = model_df[
            model_df[model_sequence_col].astype(str)
            == representative_sequence
        ].copy()

    elif (
        model_id_col is not None
        and id_col is not None
    ):

        rep_model = model_df[
            model_df[model_id_col].astype(str)
            == representative_id
        ].copy()

    else:

        raise KeyError(
            "Unable to match representative sequence "
            "in adjusted model consensus file."
        )


    if model_position_col is not None:

        rep_model = rep_model.sort_values(
            model_position_col
        )


    attention_col = find_col(
        rep_model,
        [
            "attention_score",
            "attention",
            "mean_attention"
        ]
    )

    grad_col = find_col(
        rep_model,
        [
            "gradient_x_input",
            "gradient_input",
            "grad_x_input",
            "gradient_score"
        ]
    )

    ig_col = find_col(
        rep_model,
        [
            "integrated_gradients",
            "integrated_gradient",
            "ig_score"
        ]
    )

    if (
        attention_col is None
        or grad_col is None
        or ig_col is None
    ):

        raise KeyError(
            "\nCould not identify the three attribution columns.\n"
            f"Adjusted model columns:\n"
            f"{rep_model.columns.tolist()}"
        )

else:

    rep_model = rep_global


# ----------------------------------------------------------------------
# Detect consensus score separately
# ----------------------------------------------------------------------

if consensus_col is None:

    consensus_candidates = [
        c
        for c in rep_global.columns
        if (
            "score" in c
            and (
                "adjust" in c
                or "consensus" in c
            )
        )
    ]

    if len(consensus_candidates) == 0:

        raise KeyError(
            "Could not identify adjusted consensus score.\n"
            f"Columns:\n{rep_global.columns.tolist()}"
        )

    consensus_col = consensus_candidates[0]


# ----------------------------------------------------------------------
# Build panel-A matrix
# ----------------------------------------------------------------------

attention_values = pd.to_numeric(
    rep_model[attention_col],
    errors="coerce"
).to_numpy()

grad_values = pd.to_numeric(
    rep_model[grad_col],
    errors="coerce"
).to_numpy()

ig_values = pd.to_numeric(
    rep_model[ig_col],
    errors="coerce"
).to_numpy()

consensus_values = pd.to_numeric(
    rep_global[consensus_col],
    errors="coerce"
).to_numpy()


# Ensure lengths match
L = min(
    len(attention_values),
    len(grad_values),
    len(ig_values),
    len(consensus_values),
    len(representative_sequence)
)

attention_values = attention_values[:L]
grad_values = grad_values[:L]
ig_values = ig_values[:L]
consensus_values = consensus_values[:L]

representative_sequence = representative_sequence[:L]


def normalize_vector(x):

    x = np.asarray(x, dtype=float)

    finite = np.isfinite(x)

    if not np.any(finite):
        return np.zeros_like(x)

    mn = np.nanmin(x)
    mx = np.nanmax(x)

    if mx == mn:
        return np.zeros_like(x)

    return (
        (x - mn)
        / (mx - mn)
    )


panelA_matrix = np.vstack([
    normalize_vector(attention_values),
    normalize_vector(grad_values),
    normalize_vector(ig_values),
    normalize_vector(consensus_values)
])


# ======================================================================
# PANEL B — METHOD AGREEMENT
# ======================================================================

dataset_col = find_col(
    method_df,
    [
        "dataset",
        "data_set"
    ]
)

pair_col = find_col(
    method_df,
    [
        "method_pair",
        "comparison",
        "pair"
    ]
)

corr_col = find_col(
    method_df,
    [
        "mean_spearman",
        "spearman",
        "correlation"
    ]
)


if dataset_col is None or corr_col is None:

    raise KeyError(
        "Could not identify columns in "
        "xai_method_agreement.csv\n"
        f"{method_df.columns.tolist()}"
    )


# If pair column absent, construct it from method columns
if pair_col is None:

    method1_col = find_col(
        method_df,
        ["method_1", "method1"]
    )

    method2_col = find_col(
        method_df,
        ["method_2", "method2"]
    )

    if (
        method1_col is None
        or method2_col is None
    ):

        raise KeyError(
            "Unable to identify method comparisons."
        )

    method_df["method_pair"] = (
        method_df[method1_col].astype(str)
        + " vs "
        + method_df[method2_col].astype(str)
    )

    pair_col = "method_pair"


method_df[dataset_col] = (
    method_df[dataset_col]
    .astype(str)
    .str.lower()
)


internal_method = method_df[
    method_df[dataset_col]
    .str.contains("internal")
].copy()

kelm_method = method_df[
    method_df[dataset_col]
    .str.contains("kelm|external", regex=True)
].copy()


pairs = list(
    dict.fromkeys(
        internal_method[pair_col]
        .astype(str)
        .tolist()
    )
)


# ======================================================================
# PANEL C — JACCARD MATRICES
# ======================================================================

def clean_matrix(df):

    df = df.copy()

    df.index = [
        str(x)
        .replace("_", "-")
        for x in df.index
    ]

    df.columns = [
        str(x)
        .replace("_", "-")
        for x in df.columns
    ]

    # force numeric
    df = df.apply(
        pd.to_numeric,
        errors="coerce"
    )

    return df


internal_matrix = clean_matrix(
    internal_jaccard_raw
)

kelm_matrix = clean_matrix(
    kelm_jaccard_raw
)


# ======================================================================
# PANEL D — ORIGINAL VS ADJUSTED
# ======================================================================

adjust_dataset_col = find_col(
    adjusted_df,
    [
        "dataset",
        "data_set"
    ]
)

spearman_col = find_col(
    adjusted_df,
    [
        "overall_rank_spearman",
        "rank_spearman",
        "spearman"
    ]
)

jaccard_col = find_col(
    adjusted_df,
    [
        "top20_hotspot_jaccard",
        "top_20_hotspot_jaccard",
        "top20_jaccard",
        "jaccard"
    ]
)


if (
    adjust_dataset_col is None
    or spearman_col is None
    or jaccard_col is None
):

    raise KeyError(
        "Could not identify original-vs-adjusted columns.\n"
        f"{adjusted_df.columns.tolist()}"
    )


adjusted_df[adjust_dataset_col] = (
    adjusted_df[adjust_dataset_col]
    .astype(str)
    .str.lower()
)


internal_adjust = adjusted_df[
    adjusted_df[adjust_dataset_col]
    .str.contains("internal")
].iloc[0]

kelm_adjust = adjusted_df[
    adjusted_df[adjust_dataset_col]
    .str.contains("kelm|external", regex=True)
].iloc[0]


# ======================================================================
# CREATE FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(20, 15)
)

outer = gridspec.GridSpec(
    3,
    1,
    height_ratios=[
        1.55,
        1.15,
        0.95
    ],
    hspace=0.48
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)

imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)


axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)


axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa
        in enumerate(
            representative_sequence
        )
    ]
)


axA.set_xlabel(
    "Residue and sequence position",
    labelpad=12
)

axA.set_title(
    "Construction of multi-method consensus XAI",
    pad=32
)


# Larger residue labels
for label in axA.get_xticklabels():

    label.set_fontsize(9.5)
    label.set_fontweight("bold")


for label in axA.get_yticklabels():

    label.set_fontsize(13)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# Representative sequence information
# ----------------------------------------------------------------------

axA.text(
    0.5,
    1.035,

    f"Representative CPP: {representative_id}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=11.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Consensus hotspot borders
# Top 20% residues highlighted
# ----------------------------------------------------------------------

n_hotspots = max(
    1,
    int(np.ceil(0.20 * L))
)

top_indices = np.argsort(
    consensus_values[:L]
)[-n_hotspots:]


for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                3 - 0.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=2.0
        )
    )


# ----------------------------------------------------------------------
# Colorbar
# ----------------------------------------------------------------------

cbar = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.018,
    pad=0.012
)

cbar.set_label(
    "Normalized residue importance",
    fontsize=13,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=12,
    width=1.3
)

for label in cbar.ax.get_yticklabels():

    label.set_fontweight("bold")


# Panel A letter
axA.text(
    -0.055,
    1.07,
    "A",
    transform=axA.transAxes,
    fontsize=24,
    fontweight="bold",
    va="top"
)


# ======================================================================
# SECOND ROW: B + C
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(
    1,
    2,
    subplot_spec=outer[1],
    width_ratios=[
        1.05,
        1.35
    ],
    wspace=0.32
)


# ======================================================================
# PANEL B — XAI METHOD AGREEMENT
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)

x = np.arange(
    len(pairs)
)

width = 0.34


def values_for_pairs(df, pairs):

    values = []

    for pair in pairs:

        row = df[
            df[pair_col].astype(str)
            == str(pair)
        ]

        if len(row) == 0:
            values.append(np.nan)

        else:
            values.append(
                float(
                    row.iloc[0][corr_col]
                )
            )

    return values


internal_vals = values_for_pairs(
    internal_method,
    pairs
)

kelm_vals = values_for_pairs(
    kelm_method,
    pairs
)


bars1 = axB.bar(
    x - width/2,
    internal_vals,
    width,
    label="Internal test"
)

bars2 = axB.bar(
    x + width/2,
    kelm_vals,
    width,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level Spearman correlation"
)

axB.set_title(
    "Agreement among XAI methods",
    pad=14
)


# ----------------------------------------------------------------------
# Cleaner method-pair labels
# ----------------------------------------------------------------------

pretty_pairs = []

for p in pairs:

    p = str(p)

    p = (
        p.replace("_", " ")
        .replace("gradient x input", "Gradient × Input")
        .replace("integrated gradients", "Integrated Gradients")
        .replace("attention", "Attention")
    )

    if " vs " in p.lower():

        pieces = re.split(
            r"\s+vs\s+",
            p,
            flags=re.IGNORECASE
        )

        p = "\nvs\n".join(
            pieces
        )

    pretty_pairs.append(p)


axB.set_xticks(x)

axB.set_xticklabels(
    pretty_pairs,
    rotation=0
)


for label in axB.get_xticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


axB.legend(
    frameon=False,
    loc="lower right"
)


# Value annotations
for bars in [bars1, bars2]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.004,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold",

            rotation=90
        )


valid_B = [
    v
    for v in (
        internal_vals
        + kelm_vals
    )
    if np.isfinite(v)
]

if valid_B:

    axB.set_ylim(
        max(0, min(valid_B) - 0.08),
        min(1.04, max(valid_B) + 0.07)
    )


bold_ticks(
    axB,
    xsize=11,
    ysize=13
)

clean_spines(
    axB
)


axB.text(
    -0.11,
    1.08,
    "B",
    transform=axB.transAxes,
    fontsize=24,
    fontweight="bold",
    va="top"
)


# ======================================================================
# PANEL C — CROSS-PLM JACCARD HEATMAPS
# ======================================================================

C_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    2,
    subplot_spec=middle[1],
    wspace=0.28
)

axC1 = fig.add_subplot(
    C_grid[0]
)

axC2 = fig.add_subplot(
    C_grid[1]
)


# ----------------------------------------------------------------------
# Internal heatmap
# ----------------------------------------------------------------------

imC1 = axC1.imshow(
    internal_matrix.values,
    vmin=0,
    vmax=1,
    aspect="equal"
)

axC1.set_title(
    "Internal test",
    fontsize=14,
    fontweight="bold",
    pad=10
)


axC1.set_xticks(
    np.arange(
        len(internal_matrix.columns)
    )
)

axC1.set_yticks(
    np.arange(
        len(internal_matrix.index)
    )
)

axC1.set_xticklabels(
    internal_matrix.columns,
    rotation=45,
    ha="right"
)

axC1.set_yticklabels(
    internal_matrix.index
)


annotate_heatmap(
    axC1,
    internal_matrix.values,
    fontsize=11
)


# ----------------------------------------------------------------------
# KELM heatmap
# ----------------------------------------------------------------------

imC2 = axC2.imshow(
    kelm_matrix.values,
    vmin=0,
    vmax=1,
    aspect="equal"
)

axC2.set_title(
    "KELM external",
    fontsize=14,
    fontweight="bold",
    pad=10
)


axC2.set_xticks(
    np.arange(
        len(kelm_matrix.columns)
    )
)

axC2.set_yticks(
    np.arange(
        len(kelm_matrix.index)
    )
)

axC2.set_xticklabels(
    kelm_matrix.columns,
    rotation=45,
    ha="right"
)

axC2.set_yticklabels(
    kelm_matrix.index
)


annotate_heatmap(
    axC2,
    kelm_matrix.values,
    fontsize=11
)


for ax in [
    axC1,
    axC2
]:

    bold_ticks(
        ax,
        xsize=10.5,
        ysize=10.5
    )


# Main C title positioned above both matrices
bbox1 = axC1.get_position()
bbox2 = axC2.get_position()

center_C = (
    bbox1.x0
    + bbox2.x1
) / 2

fig.text(
    center_C,
    bbox1.y1 + 0.047,

    "Cross-PLM agreement and adjusted-consensus stability",

    ha="center",
    va="bottom",

    fontsize=16,
    fontweight="bold"
)


axC1.text(
    -0.30,
    1.22,
    "C",
    transform=axC1.transAxes,
    fontsize=24,
    fontweight="bold",
    va="top"
)


# ======================================================================
# PANEL D — ORIGINAL VS REDUNDANCY-ADJUSTED CONSENSUS
# ======================================================================

axD = fig.add_subplot(
    outer[2]
)


categories = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard"
]


internal_D = [
    float(
        internal_adjust[spearman_col]
    ),
    float(
        internal_adjust[jaccard_col]
    )
]

kelm_D = [
    float(
        kelm_adjust[spearman_col]
    ),
    float(
        kelm_adjust[jaccard_col]
    )
]


xD = np.arange(2)

widthD = 0.30


barsD1 = axD.bar(
    xD - widthD/2,
    internal_D,
    widthD,
    label="Internal test"
)

barsD2 = axD.bar(
    xD + widthD/2,
    kelm_D,
    widthD,
    label="KELM external"
)


axD.set_xticks(
    xD
)

axD.set_xticklabels(
    categories
)

axD.set_ylabel(
    "Agreement"
)

axD.set_title(
    "Original vs redundancy-adjusted consensus",
    pad=15
)


for label in axD.get_xticklabels():

    label.set_fontsize(13)
    label.set_fontweight("bold")


axD.legend(
    frameon=False,
    loc="lower left"
)


for bars in [
    barsD1,
    barsD2
]:

    for bar in bars:

        value = bar.get_height()

        axD.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.008,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=12,
            fontweight="bold"
        )


all_D = internal_D + kelm_D

axD.set_ylim(
    max(
        0,
        min(all_D) - 0.10
    ),
    min(
        1.06,
        max(all_D) + 0.08
    )
)


bold_ticks(
    axD,
    xsize=13,
    ysize=13
)

clean_spines(
    axD
)


axD.text(
    -0.055,
    1.08,
    "D",
    transform=axD.transAxes,
    fontsize=24,
    fontweight="bold",
    va="top"
)


# ======================================================================
# MAIN FIGURE TITLE
# ======================================================================

fig.suptitle(
    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",
    fontsize=19,
    fontweight="bold",
    y=0.995
)


# ======================================================================
# FINAL LAYOUT
# ======================================================================

fig.subplots_adjust(
    left=0.075,
    right=0.975,
    bottom=0.065,
    top=0.945
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

fig.savefig(
    svg_file,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FINAL FIGURE 3 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# MANUSCRIPT FIGURE 3 — UPDATED FINAL VERSION
#
# A = Representative sequence INTQC_00032
# B = Agreement among XAI methods
# C = Original vs redundancy-adjusted consensus
# D = Cross-PLM Jaccard agreement matrices
#
# Individual panel titles removed
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Rectangle


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

XAI_DIR = (
    PROJECT
    / "06_xai"
)

CHECKPOINT_DIR = (
    PROJECT
    / "08_checkpoints"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# INPUT FILES
# ======================================================================

METHOD_AGREEMENT_FILE = (
    TABLE_DIR
    / "xai_method_agreement.csv"
)

JACCARD_INTERNAL_FILE = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_internal_test_all.csv"
)

JACCARD_KELM_FILE = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_kelm_external_all.csv"
)

ORIGINAL_ADJUSTED_FILE = (
    TABLE_DIR
    / "original_vs_adjusted_consensus.csv"
)

REPRESENTATIVE_FILE = (
    CHECKPOINT_DIR
    / "figure_4_representative_sequences.csv"
)

GLOBAL_INTERNAL_FILE = (
    XAI_DIR
    / "consensus_adjusted"
    / "global"
    / "internal_test"
    / "adjusted_global_consensus_residue_scores.csv"
)


# ======================================================================
# REPRESENTATIVE SEQUENCE
# ======================================================================

TARGET_ID = "INTQC_00651"


# ======================================================================
# GLOBAL PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 13,

    # Axis labels
    "axes.labelsize": 15,
    "axes.labelweight": "bold",

    # Tick values
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,

    # Legend
    "legend.fontsize": 11,

    # Axes
    "axes.linewidth": 1.4,

    # Save
    "savefig.dpi": 600
})


# ======================================================================
# HELPER FUNCTIONS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def find_col(df, candidates):

    cols = list(df.columns)

    # Exact match
    for candidate in candidates:

        if candidate in cols:
            return candidate

    # Partial match
    for candidate in candidates:

        for c in cols:

            if candidate in c:
                return c

    return None


def bold_ticks(
    ax,
    xsize=13,
    ysize=13
):

    ax.tick_params(
        axis="both",
        which="major",
        width=1.4,
        length=5
    )

    for label in ax.get_xticklabels():

        label.set_fontsize(
            xsize
        )

        label.set_fontweight(
            "bold"
        )

    for label in ax.get_yticklabels():

        label.set_fontsize(
            ysize
        )

        label.set_fontweight(
            "bold"
        )


def clean_spines(ax):

    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    ax.spines["left"].set_linewidth(
        1.4
    )

    ax.spines["bottom"].set_linewidth(
        1.4
    )


def annotate_heatmap(
    ax,
    matrix,
    fontsize=12
):

    arr = np.asarray(
        matrix,
        dtype=float
    )

    midpoint = 0.50

    for i in range(
        arr.shape[0]
    ):

        for j in range(
            arr.shape[1]
        ):

            value = arr[i, j]

            if np.isnan(value):
                continue

            color = (
                "white"
                if value >= midpoint
                else "black"
            )

            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=fontsize,
                fontweight="bold",
                color=color
            )


def normalize_vector(x):

    x = np.asarray(
        x,
        dtype=float
    )

    finite = np.isfinite(x)

    if not np.any(finite):

        return np.zeros_like(x)

    mn = np.nanmin(x)
    mx = np.nanmax(x)

    if mx == mn:

        return np.zeros_like(x)

    return (
        (x - mn)
        / (mx - mn)
    )


# ======================================================================
# VERIFY FILES
# ======================================================================

required_files = [

    METHOD_AGREEMENT_FILE,
    JACCARD_INTERNAL_FILE,
    JACCARD_KELM_FILE,
    ORIGINAL_ADJUSTED_FILE,
    REPRESENTATIVE_FILE,
    GLOBAL_INTERNAL_FILE
]


print("=" * 95)
print("FIGURE 3 SOURCE FILES")
print("=" * 95)


for f in required_files:

    if not f.exists():

        raise FileNotFoundError(
            f"Missing file:\n{f}"
        )

    print("✓", f)


# ======================================================================
# LOAD DATA
# ======================================================================

method_df = clean_columns(
    pd.read_csv(
        METHOD_AGREEMENT_FILE
    )
)

internal_jaccard_raw = pd.read_csv(
    JACCARD_INTERNAL_FILE,
    index_col=0
)

kelm_jaccard_raw = pd.read_csv(
    JACCARD_KELM_FILE,
    index_col=0
)

adjusted_df = clean_columns(
    pd.read_csv(
        ORIGINAL_ADJUSTED_FILE
    )
)

representative_df = clean_columns(
    pd.read_csv(
        REPRESENTATIVE_FILE
    )
)

global_df = clean_columns(
    pd.read_csv(
        GLOBAL_INTERNAL_FILE
    )
)


# ======================================================================
# PANEL A
# FORCE REPRESENTATIVE = INTQC_00651
# ======================================================================

id_col = find_col(
    representative_df,
    [
        "sequence_id",
        "seq_id",
        "id"
    ]
)

sequence_col = find_col(
    representative_df,
    [
        "sequence",
        "peptide_sequence",
        "seq"
    ]
)


if id_col is None:

    raise KeyError(
        "No sequence ID column found in "
        "representative-sequence table."
    )


if sequence_col is None:

    raise KeyError(
        "No sequence column found in "
        "representative-sequence table."
    )


target_row = representative_df[
    representative_df[id_col]
    .astype(str)
    .str.strip()
    == TARGET_ID
]


if target_row.empty:

    raise ValueError(
        f"{TARGET_ID} was not found in:\n"
        f"{REPRESENTATIVE_FILE}\n\n"
        f"Available IDs include:\n"
        f"{representative_df[id_col].head(20).tolist()}"
    )


representative_id = TARGET_ID

representative_sequence = str(
    target_row.iloc[0][sequence_col]
).strip()


print("\n" + "=" * 95)
print("PANEL A REPRESENTATIVE")
print("=" * 95)

print(
    "Sequence ID:",
    representative_id
)

print(
    "Sequence:",
    representative_sequence
)

print(
    "Length:",
    len(representative_sequence)
)


# ======================================================================
# MATCH TARGET IN GLOBAL CONSENSUS DATA
# ======================================================================

global_id_col = find_col(
    global_df,
    [
        "sequence_id",
        "seq_id",
        "id"
    ]
)

global_sequence_col = find_col(
    global_df,
    [
        "sequence",
        "peptide_sequence",
        "seq"
    ]
)

position_col = find_col(
    global_df,
    [
        "position",
        "residue_position",
        "pos"
    ]
)


if global_id_col is not None:

    rep_global = global_df[
        global_df[global_id_col]
        .astype(str)
        .str.strip()
        == TARGET_ID
    ].copy()

else:

    rep_global = global_df[
        global_df[global_sequence_col]
        .astype(str)
        .str.strip()
        == representative_sequence
    ].copy()


if rep_global.empty:

    raise ValueError(
        f"{TARGET_ID} was not found in "
        "adjusted global consensus data."
    )


if position_col is not None:

    rep_global = (
        rep_global
        .sort_values(position_col)
        .reset_index(drop=True)
    )


# ======================================================================
# LOAD METHOD-SPECIFIC ATTRIBUTIONS FOR A
# ======================================================================

MODEL_FOR_PANEL_A = "ESM2_1280"

MODEL_A_FILE = (
    XAI_DIR
    / "consensus_adjusted"
    / MODEL_FOR_PANEL_A
    / "internal_test"
    / "adjusted_model_consensus.csv"
)


if not MODEL_A_FILE.exists():

    raise FileNotFoundError(
        MODEL_A_FILE
    )


model_df = clean_columns(
    pd.read_csv(
        MODEL_A_FILE
    )
)


model_id_col = find_col(
    model_df,
    [
        "sequence_id",
        "seq_id",
        "id"
    ]
)

model_sequence_col = find_col(
    model_df,
    [
        "sequence",
        "peptide_sequence",
        "seq"
    ]
)

model_position_col = find_col(
    model_df,
    [
        "position",
        "residue_position",
        "pos"
    ]
)


if model_id_col is not None:

    rep_model = model_df[
        model_df[model_id_col]
        .astype(str)
        .str.strip()
        == TARGET_ID
    ].copy()

else:

    rep_model = model_df[
        model_df[model_sequence_col]
        .astype(str)
        .str.strip()
        == representative_sequence
    ].copy()


if rep_model.empty:

    raise ValueError(
        f"{TARGET_ID} not found in "
        f"{MODEL_A_FILE}"
    )


if model_position_col is not None:

    rep_model = (
        rep_model
        .sort_values(
            model_position_col
        )
        .reset_index(drop=True)
    )


# ======================================================================
# DETECT ATTRIBUTION COLUMNS
# ======================================================================

attention_col = find_col(
    rep_model,
    [
        "attention_score",
        "attention",
        "mean_attention"
    ]
)

grad_col = find_col(
    rep_model,
    [
        "gradient_x_input",
        "gradient_input",
        "grad_x_input",
        "gradient_score"
    ]
)

ig_col = find_col(
    rep_model,
    [
        "integrated_gradients",
        "integrated_gradient",
        "ig_score"
    ]
)

consensus_col = find_col(
    rep_global,
    [
        "adjusted_consensus_score",
        "global_adjusted_score",
        "adjusted_score",
        "consensus_score",
        "normalized_score"
    ]
)


if any(
    x is None
    for x in [
        attention_col,
        grad_col,
        ig_col,
        consensus_col
    ]
):

    print(
        "\nAdjusted model columns:"
    )

    print(
        rep_model.columns.tolist()
    )

    print(
        "\nGlobal columns:"
    )

    print(
        rep_global.columns.tolist()
    )

    raise KeyError(
        "One or more required attribution "
        "columns were not detected."
    )


# ======================================================================
# BUILD PANEL A MATRIX
# ======================================================================

attention_values = pd.to_numeric(
    rep_model[attention_col],
    errors="coerce"
).to_numpy()

grad_values = pd.to_numeric(
    rep_model[grad_col],
    errors="coerce"
).to_numpy()

ig_values = pd.to_numeric(
    rep_model[ig_col],
    errors="coerce"
).to_numpy()

consensus_values = pd.to_numeric(
    rep_global[consensus_col],
    errors="coerce"
).to_numpy()


L = min(
    len(representative_sequence),
    len(attention_values),
    len(grad_values),
    len(ig_values),
    len(consensus_values)
)


representative_sequence = (
    representative_sequence[:L]
)

attention_values = (
    attention_values[:L]
)

grad_values = (
    grad_values[:L]
)

ig_values = (
    ig_values[:L]
)

consensus_values = (
    consensus_values[:L]
)


panelA_matrix = np.vstack(
    [
        normalize_vector(
            attention_values
        ),

        normalize_vector(
            grad_values
        ),

        normalize_vector(
            ig_values
        ),

        normalize_vector(
            consensus_values
        )
    ]
)


# ======================================================================
# PANEL B — XAI METHOD AGREEMENT
# ======================================================================

dataset_col = find_col(
    method_df,
    [
        "dataset",
        "data_set"
    ]
)

pair_col = find_col(
    method_df,
    [
        "method_pair",
        "comparison",
        "pair"
    ]
)

corr_col = find_col(
    method_df,
    [
        "mean_spearman",
        "spearman",
        "correlation"
    ]
)


if pair_col is None:

    method1_col = find_col(
        method_df,
        [
            "method_1",
            "method1"
        ]
    )

    method2_col = find_col(
        method_df,
        [
            "method_2",
            "method2"
        ]
    )

    if (
        method1_col is None
        or method2_col is None
    ):

        raise KeyError(
            "Unable to detect method pairs."
        )

    method_df["method_pair"] = (
        method_df[method1_col]
        .astype(str)
        +
        " vs "
        +
        method_df[method2_col]
        .astype(str)
    )

    pair_col = "method_pair"


method_df[dataset_col] = (
    method_df[dataset_col]
    .astype(str)
    .str.lower()
)


internal_method = method_df[
    method_df[dataset_col]
    .str.contains(
        "internal"
    )
].copy()


kelm_method = method_df[
    method_df[dataset_col]
    .str.contains(
        "kelm|external",
        regex=True
    )
].copy()


pairs = list(
    dict.fromkeys(
        internal_method[pair_col]
        .astype(str)
        .tolist()
    )
)


def values_for_pairs(
    df,
    pair_list
):

    values = []

    for pair in pair_list:

        row = df[
            df[pair_col]
            .astype(str)
            == str(pair)
        ]

        if row.empty:

            values.append(
                np.nan
            )

        else:

            values.append(
                float(
                    row.iloc[0][corr_col]
                )
            )

    return values


internal_B = values_for_pairs(
    internal_method,
    pairs
)

kelm_B = values_for_pairs(
    kelm_method,
    pairs
)


# ======================================================================
# PANEL C — ORIGINAL VS ADJUSTED
# ======================================================================

adjust_dataset_col = find_col(
    adjusted_df,
    [
        "dataset",
        "data_set"
    ]
)

spearman_col = find_col(
    adjusted_df,
    [
        "overall_rank_spearman",
        "rank_spearman",
        "spearman"
    ]
)

jaccard_col = find_col(
    adjusted_df,
    [
        "top20_hotspot_jaccard",
        "top_20_hotspot_jaccard",
        "top20_jaccard",
        "jaccard"
    ]
)


if (
    adjust_dataset_col is None
    or spearman_col is None
    or jaccard_col is None
):

    raise KeyError(
        "Unable to identify columns in "
        "original_vs_adjusted_consensus.csv"
    )


adjusted_df[
    adjust_dataset_col
] = (
    adjusted_df[
        adjust_dataset_col
    ]
    .astype(str)
    .str.lower()
)


internal_adjust = adjusted_df[
    adjusted_df[
        adjust_dataset_col
    ]
    .str.contains(
        "internal"
    )
].iloc[0]


kelm_adjust = adjusted_df[
    adjusted_df[
        adjust_dataset_col
    ]
    .str.contains(
        "kelm|external",
        regex=True
    )
].iloc[0]


internal_C = [

    float(
        internal_adjust[
            spearman_col
        ]
    ),

    float(
        internal_adjust[
            jaccard_col
        ]
    )
]


kelm_C = [

    float(
        kelm_adjust[
            spearman_col
        ]
    ),

    float(
        kelm_adjust[
            jaccard_col
        ]
    )
]


# ======================================================================
# PANEL D — JACCARD MATRICES
# ======================================================================

def clean_matrix(df):

    df = df.copy()

    df.index = [
        str(x)
        .replace("_", "-")
        for x in df.index
    ]

    df.columns = [
        str(x)
        .replace("_", "-")
        for x in df.columns
    ]

    return df.apply(
        pd.to_numeric,
        errors="coerce"
    )


internal_matrix = clean_matrix(
    internal_jaccard_raw
)

kelm_matrix = clean_matrix(
    kelm_jaccard_raw
)


# ======================================================================
# CREATE FIGURE
#
# A = full width
#
# B | C
#
# D = full width with two heatmaps
# ======================================================================

fig = plt.figure(
    figsize=(20, 15)
)


outer = gridspec.GridSpec(

    3,
    1,

    height_ratios=[
        1.55,
        1.05,
        1.30
    ],

    hspace=0.50
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)


imA = axA.imshow(

    panelA_matrix,

    aspect="auto",

    interpolation="nearest",

    vmin=0,
    vmax=1
)


axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)


axA.set_xticks(
    np.arange(L)
)


axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa
        in enumerate(
            representative_sequence
        )
    ]
)


axA.set_xlabel(
    "Residue and sequence position",
    labelpad=12
)


# ----------------------------------------------------------------------
# Bold residue values
# ----------------------------------------------------------------------

for label in axA.get_xticklabels():

    label.set_fontsize(
        10
    )

    label.set_fontweight(
        "bold"
    )


for label in axA.get_yticklabels():

    label.set_fontsize(
        14
    )

    label.set_fontweight(
        "bold"
    )


# ----------------------------------------------------------------------
# Representative sequence text
# ----------------------------------------------------------------------

axA.text(

    0.5,
    1.035,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=12.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Top-20% adjusted-consensus hotspots
# ----------------------------------------------------------------------

n_hotspots = max(
    1,
    int(
        np.ceil(
            0.20 * L
        )
    )
)


top_indices = np.argsort(
    consensus_values
)[
    -n_hotspots:
]


for idx in top_indices:

    axA.add_patch(

        Rectangle(

            (
                idx - 0.5,
                3 - 0.5
            ),

            1,
            1,

            fill=False,

            edgecolor="white",

            linewidth=2.2
        )
    )


# ----------------------------------------------------------------------
# Colorbar
# ----------------------------------------------------------------------

cbar = fig.colorbar(

    imA,

    ax=axA,

    fraction=0.018,

    pad=0.012
)


cbar.set_label(

    "Normalized residue importance",

    fontsize=14,

    fontweight="bold"
)


cbar.ax.tick_params(
    labelsize=12,
    width=1.3
)


for label in cbar.ax.get_yticklabels():

    label.set_fontweight(
        "bold"
    )


# ----------------------------------------------------------------------
# A label
# ----------------------------------------------------------------------

axA.text(

    -0.055,
    1.08,

    "A",

    transform=axA.transAxes,

    fontsize=24,

    fontweight="bold",

    va="top"
)


# ======================================================================
# SECOND ROW
# B = method agreement
# C = original vs adjusted
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(

    1,
    2,

    subplot_spec=outer[1],

    width_ratios=[
        1.25,
        1.0
    ],

    wspace=0.30
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)


xB = np.arange(
    len(pairs)
)

widthB = 0.34


barsB1 = axB.bar(

    xB - widthB/2,

    internal_B,

    widthB,

    label="Internal test"
)


barsB2 = axB.bar(

    xB + widthB/2,

    kelm_B,

    widthB,

    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level Spearman correlation"
)


# ----------------------------------------------------------------------
# Method labels
# ----------------------------------------------------------------------

pretty_pairs = []


for p in pairs:

    p = str(p)

    p = p.replace(
        "_",
        " "
    )

    p = p.replace(
        "gradient x input",
        "Gradient × Input"
    )

    p = p.replace(
        "integrated gradients",
        "Integrated Gradients"
    )

    p = p.replace(
        "attention",
        "Attention"
    )

    pieces = re.split(
        r"\s+vs\s+",
        p,
        flags=re.IGNORECASE
    )

    if len(pieces) == 2:

        p = (
            pieces[0]
            + "\nvs\n"
            + pieces[1]
        )

    pretty_pairs.append(
        p
    )


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)


for label in axB.get_xticklabels():

    label.set_fontsize(
        11.5
    )

    label.set_fontweight(
        "bold"
    )


axB.legend(
    frameon=False,
    loc="lower right",
    fontsize=11
)


# ----------------------------------------------------------------------
# Values above bars
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.004,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=11.5,

            fontweight="bold",

            rotation=90
        )


valid_B = [
    v
    for v in (
        internal_B
        + kelm_B
    )
    if np.isfinite(v)
]


if valid_B:

    axB.set_ylim(

        max(
            0,
            min(valid_B) - 0.08
        ),

        min(
            1.04,
            max(valid_B) + 0.07
        )
    )


bold_ticks(
    axB,
    xsize=11.5,
    ysize=13
)

clean_spines(
    axB
)


axB.text(

    -0.12,
    1.08,

    "B",

    transform=axB.transAxes,

    fontsize=24,

    fontweight="bold",

    va="top"
)


# ======================================================================
# PANEL C — ORIGINAL VS ADJUSTED
# ======================================================================

axC = fig.add_subplot(
    middle[1]
)


categories_C = [

    "Overall rank\nSpearman",

    "Top-20% hotspot\nJaccard"
]


xC = np.arange(
    2
)

widthC = 0.30


barsC1 = axC.bar(

    xC - widthC/2,

    internal_C,

    widthC,

    label="Internal test"
)


barsC2 = axC.bar(

    xC + widthC/2,

    kelm_C,

    widthC,

    label="KELM external"
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)


axC.set_ylabel(
    "Agreement"
)


for label in axC.get_xticklabels():

    label.set_fontsize(
        13
    )

    label.set_fontweight(
        "bold"
    )


axC.legend(
    frameon=False,
    loc="lower left",
    fontsize=11
)


for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.008,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=12,

            fontweight="bold"
        )


all_C = (
    internal_C
    + kelm_C
)


axC.set_ylim(

    max(
        0,
        min(all_C) - 0.10
    ),

    min(
        1.06,
        max(all_C) + 0.08
    )
)


bold_ticks(
    axC,
    xsize=13,
    ysize=13
)

clean_spines(
    axC
)


axC.text(

    -0.12,
    1.08,

    "C",

    transform=axC.transAxes,

    fontsize=24,

    fontweight="bold",

    va="top"
)


# ======================================================================
# PANEL D — CROSS-PLM HEATMAPS
# FULL WIDTH BOTTOM
# ======================================================================

D_grid = gridspec.GridSpecFromSubplotSpec(

    1,
    2,

    subplot_spec=outer[2],

    wspace=0.22
)


axD1 = fig.add_subplot(
    D_grid[0]
)

axD2 = fig.add_subplot(
    D_grid[1]
)


# ======================================================================
# D1 — INTERNAL TEST
# ======================================================================

imD1 = axD1.imshow(

    internal_matrix.values,

    vmin=0,
    vmax=1,

    aspect="equal"
)


axD1.set_xticks(
    np.arange(
        len(
            internal_matrix.columns
        )
    )
)

axD1.set_yticks(
    np.arange(
        len(
            internal_matrix.index
        )
    )
)


axD1.set_xticklabels(
    internal_matrix.columns,
    rotation=35,
    ha="right"
)

axD1.set_yticklabels(
    internal_matrix.index
)


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=13
)


bold_ticks(
    axD1,
    xsize=12,
    ysize=12
)


# Internal test label, not panel title
axD1.text(

    0.5,
    1.035,

    "Internal test",

    transform=axD1.transAxes,

    ha="center",

    fontsize=13,

    fontweight="bold"
)


# ======================================================================
# D2 — KELM EXTERNAL
# ======================================================================

imD2 = axD2.imshow(

    kelm_matrix.values,

    vmin=0,
    vmax=1,

    aspect="equal"
)


axD2.set_xticks(
    np.arange(
        len(
            kelm_matrix.columns
        )
    )
)

axD2.set_yticks(
    np.arange(
        len(
            kelm_matrix.index
        )
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns,
    rotation=35,
    ha="right"
)

axD2.set_yticklabels(
    kelm_matrix.index
)


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=13
)


bold_ticks(
    axD2,
    xsize=12,
    ysize=12
)


axD2.text(

    0.5,
    1.035,

    "KELM external",

    transform=axD2.transAxes,

    ha="center",

    fontsize=13,

    fontweight="bold"
)


# ----------------------------------------------------------------------
# Shared colorbar for D
# ----------------------------------------------------------------------

cbarD = fig.colorbar(

    imD2,

    ax=[
        axD1,
        axD2
    ],

    fraction=0.020,

    pad=0.025,

    shrink=0.88
)


cbarD.set_label(

    "Hotspot Jaccard",

    fontsize=14,

    fontweight="bold"
)


cbarD.ax.tick_params(
    labelsize=12
)


for label in cbarD.ax.get_yticklabels():

    label.set_fontweight(
        "bold"
    )


# ----------------------------------------------------------------------
# Panel D letter
# ----------------------------------------------------------------------

axD1.text(

    -0.16,
    1.08,

    "D",

    transform=axD1.transAxes,

    fontsize=24,

    fontweight="bold",

    va="top"
)


# ======================================================================
# MAIN FIGURE TITLE ONLY
# ======================================================================

fig.suptitle(

    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",

    fontsize=20,

    fontweight="bold",

    y=0.995
)


# ======================================================================
# FINAL LAYOUT
# ======================================================================

fig.subplots_adjust(

    left=0.075,

    right=0.965,

    bottom=0.060,

    top=0.945
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_v2.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_v2.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_v2.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    pdf_file,
    bbox_inches="tight"
)


fig.savefig(
    svg_file,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print(
    "\n" + "=" * 95
)

print(
    "UPDATED FIGURE 3 GENERATED SUCCESSFULLY"
)

print(
    "=" * 95
)


print(
    "\nPanel A representative:",
    TARGET_ID
)

print(
    "\nPNG:"
)

print(
    png_file
)

print(
    "\nPDF:"
)

print(
    pdf_file
)

print(
    "\nSVG:"
)

print(
    svg_file
)

In [ ]:
# ======================================================================
# FIGURE 3 — COMPACT PUBLICATION VERSION
#
# A = Representative CPP
# B = XAI method agreement
# C = Original vs redundancy-adjusted consensus
# D = Cross-PLM hotspot Jaccard
#
# Compact layout + no overlaps
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
import numpy as np


# ======================================================================
# GLOBAL STYLE
# ======================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.labelweight": "bold",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 1.1,
    "savefig.dpi": 600
})


# ======================================================================
# CREATE COMPACT FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(14.5, 9.0)
)

outer = gridspec.GridSpec(
    3,
    1,
    figure=fig,
    height_ratios=[
        1.25,   # A
        0.90,   # B + C
        1.05    # D
    ],
    hspace=0.43
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)

imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)


# ----------------------------------------------------------------------
# Y axis
# ----------------------------------------------------------------------

axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ],
    fontsize=10,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# X axis: residues
# ----------------------------------------------------------------------

axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa
        in enumerate(representative_sequence)
    ],
    fontsize=6.7,
    fontweight="bold"
)

axA.tick_params(
    axis="x",
    pad=2,
    length=2
)

axA.tick_params(
    axis="y",
    pad=3,
    length=3
)


axA.set_xlabel(
    "Residue and sequence position",
    fontsize=11,
    fontweight="bold",
    labelpad=5
)


# ----------------------------------------------------------------------
# Representative sequence
# ----------------------------------------------------------------------

axA.text(
    0.5,
    1.035,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Top 20% hotspot borders
# ----------------------------------------------------------------------

n_hotspots = max(
    1,
    int(np.ceil(0.20 * L))
)

top_indices = np.argsort(
    consensus_values
)[-n_hotspots:]


for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                2.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.4
        )
    )


# ----------------------------------------------------------------------
# Colorbar A
# ----------------------------------------------------------------------

cbarA = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.015,
    pad=0.010,
    aspect=25
)

cbarA.set_label(
    "Normalized residue importance",
    fontsize=9,
    fontweight="bold",
    labelpad=5
)

cbarA.ax.tick_params(
    labelsize=8,
    width=1
)

for label in cbarA.ax.get_yticklabels():
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# Panel A
# ----------------------------------------------------------------------

axA.text(
    -0.055,
    1.08,
    "A",
    transform=axA.transAxes,
    fontsize=18,
    fontweight="bold",
    va="top"
)


# ======================================================================
# SECOND ROW: B + C
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(
    1,
    2,
    subplot_spec=outer[1],
    width_ratios=[
        1.15,
        1.0
    ],
    wspace=0.28
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)

xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)

barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level Spearman correlation",
    fontsize=10,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Shorter labels
# ----------------------------------------------------------------------

pretty_pairs = [
    "Attention\nvs\nGradient × Input",
    "Attention\nvs\nIntegrated Gradients",
    "Gradient × Input\nvs\nIntegrated Gradients"
]


# Safety if number/order differs
if len(pretty_pairs) != len(pairs):

    pretty_pairs = []

    for p in pairs:

        p = str(p).replace(
            "_",
            " "
        )

        p = p.replace(
            "gradient x input",
            "Gradient × Input"
        )

        p = p.replace(
            "integrated gradients",
            "Integrated Gradients"
        )

        p = p.replace(
            "attention",
            "Attention"
        )

        pieces = re.split(
            r"\s+vs\s+",
            p,
            flags=re.IGNORECASE
        )

        if len(pieces) == 2:
            p = pieces[0] + "\nvs\n" + pieces[1]

        pretty_pairs.append(p)


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs,
    fontsize=8,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Bar values
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.004,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            rotation=90
        )


valid_B = [
    v
    for v in internal_B + kelm_B
    if np.isfinite(v)
]

axB.set_ylim(
    max(0, min(valid_B) - 0.07),
    min(1.035, max(valid_B) + 0.035)
)


# Legend inside unused lower-right space
axB.legend(
    frameon=False,
    loc="lower right",
    fontsize=8,
    handlelength=1.2
)


axB.tick_params(
    axis="y",
    labelsize=9,
    width=1.1
)

for label in axB.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


axB.text(
    -0.10,
    1.07,
    "B",
    transform=axB.transAxes,
    fontsize=18,
    fontweight="bold",
    va="top"
)


# ======================================================================
# PANEL C
# ======================================================================

axC = fig.add_subplot(
    middle[1]
)


categories_C = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(
    xC - widthC/2,
    internal_C,
    widthC,
    label="Internal test"
)

barsC2 = axC.bar(
    xC + widthC/2,
    kelm_C,
    widthC,
    label="KELM external"
)


axC.set_ylabel(
    "Agreement",
    fontsize=10,
    fontweight="bold"
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C,
    fontsize=8.5,
    fontweight="bold"
)


for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.006,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8.5,
            fontweight="bold"
        )


all_C = internal_C + kelm_C

axC.set_ylim(
    max(0, min(all_C) - 0.08),
    min(1.04, max(all_C) + 0.045)
)


axC.legend(
    frameon=False,
    loc="lower left",
    fontsize=8,
    handlelength=1.2
)


axC.tick_params(
    axis="y",
    labelsize=9,
    width=1.1
)

for label in axC.get_yticklabels():
    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


axC.text(
    -0.10,
    1.07,
    "C",
    transform=axC.transAxes,
    fontsize=18,
    fontweight="bold",
    va="top"
)


# ======================================================================
# PANEL D
#
# Important change:
# Give the colorbar its OWN column.
# This completely removes the overlap.
# ======================================================================

D_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,
    subplot_spec=outer[2],
    width_ratios=[
        1,
        1,
        0.035
    ],
    wspace=0.22
)


axD1 = fig.add_subplot(
    D_grid[0]
)

axD2 = fig.add_subplot(
    D_grid[1]
)

caxD = fig.add_subplot(
    D_grid[2]
)


# ======================================================================
# D1 — INTERNAL TEST
# ======================================================================

imD1 = axD1.imshow(
    internal_matrix.values,
    vmin=0,
    vmax=1,
    aspect="equal"
)


axD1.set_xticks(
    np.arange(
        len(internal_matrix.columns)
    )
)

axD1.set_yticks(
    np.arange(
        len(internal_matrix.index)
    )
)


axD1.set_xticklabels(
    internal_matrix.columns,
    rotation=30,
    ha="right",
    fontsize=8.5,
    fontweight="bold"
)

axD1.set_yticklabels(
    internal_matrix.index,
    fontsize=8.5,
    fontweight="bold"
)


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=9
)


axD1.text(
    0.5,
    1.04,
    "Internal test",
    transform=axD1.transAxes,
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold"
)


# ======================================================================
# D2 — KELM
# ======================================================================

imD2 = axD2.imshow(
    kelm_matrix.values,
    vmin=0,
    vmax=1,
    aspect="equal"
)


axD2.set_xticks(
    np.arange(
        len(kelm_matrix.columns)
    )
)

axD2.set_yticks(
    np.arange(
        len(kelm_matrix.index)
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns,
    rotation=30,
    ha="right",
    fontsize=8.5,
    fontweight="bold"
)

axD2.set_yticklabels(
    kelm_matrix.index,
    fontsize=8.5,
    fontweight="bold"
)


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=9
)


axD2.text(
    0.5,
    1.04,
    "KELM external",
    transform=axD2.transAxes,
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold"
)


# ======================================================================
# D COLORBAR — DEDICATED AXIS
# ======================================================================

cbarD = fig.colorbar(
    imD2,
    cax=caxD
)

cbarD.set_label(
    "Hotspot Jaccard",
    fontsize=9,
    fontweight="bold",
    labelpad=5
)

cbarD.ax.tick_params(
    labelsize=8,
    width=1
)

for label in cbarD.ax.get_yticklabels():
    label.set_fontweight("bold")


# ======================================================================
# PANEL D LABEL
# ======================================================================

axD1.text(
    -0.17,
    1.08,
    "D",
    transform=axD1.transAxes,
    fontsize=18,
    fontweight="bold",
    va="top"
)


# ======================================================================
# MAIN TITLE
# ======================================================================

fig.suptitle(
    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",
    fontsize=15,
    fontweight="bold",
    y=0.985
)


# ======================================================================
# COMPACT MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.085,
    right=0.955,
    bottom=0.075,
    top=0.925
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_COMPACT.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_COMPACT.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_COMPACT.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()

plt.close(fig)


print("\n" + "=" * 80)
print("COMPACT FIGURE 3 GENERATED")
print("=" * 80)

print("PNG:", png_file)
print("PDF:", pdf_file)
print("SVG:", svg_file)

In [ ]:
# ======================================================================
# FIGURE 3 — COMPACT FINAL LAYOUT v2
#
# A = Representative CPP
# B = XAI-method agreement
# C = Original vs adjusted consensus
# D = Cross-PLM hotspot Jaccard matrices
#
# Improvements:
# - A/B/C/D moved away from plots
# - Legends above B and C
# - Panel D uses full available width
# - No overlap with colorbar
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
import numpy as np
import re


# ======================================================================
# PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 9,

    "axes.linewidth": 1.1,

    "savefig.dpi": 600
})


# ======================================================================
# CREATE FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(15.5, 9.5)
)


outer = gridspec.GridSpec(

    3,
    1,

    figure=fig,

    height_ratios=[
        1.16,    # A
        0.82,    # B + C
        1.18     # D
    ],

    hspace=0.48
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)


imA = axA.imshow(

    panelA_matrix,

    aspect="auto",

    interpolation="nearest",

    vmin=0,
    vmax=1
)


# ----------------------------------------------------------------------
# Y labels
# ----------------------------------------------------------------------

axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)


for label in axA.get_yticklabels():

    label.set_fontsize(10.5)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# Residue labels
# ----------------------------------------------------------------------

axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa
        in enumerate(
            representative_sequence
        )
    ]
)


for label in axA.get_xticklabels():

    label.set_fontsize(6.8)
    label.set_fontweight("bold")


axA.tick_params(
    axis="x",
    length=2,
    width=1,
    pad=2
)

axA.tick_params(
    axis="y",
    length=3,
    width=1,
    pad=4
)


axA.set_xlabel(

    "Residue and sequence position",

    fontsize=11,
    fontweight="bold",

    labelpad=5
)


# ----------------------------------------------------------------------
# Representative CPP identifier
# ----------------------------------------------------------------------

axA.text(

    0.5,
    1.04,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Top 20% adjusted-consensus hotspot borders
# ----------------------------------------------------------------------

n_hotspots = max(
    1,
    int(
        np.ceil(
            0.20 * L
        )
    )
)


top_indices = np.argsort(
    consensus_values
)[
    -n_hotspots:
]


for idx in top_indices:

    axA.add_patch(

        Rectangle(

            (
                idx - 0.5,
                2.5
            ),

            1,
            1,

            fill=False,

            edgecolor="white",

            linewidth=1.5
        )
    )


# ----------------------------------------------------------------------
# Panel A colorbar
# ----------------------------------------------------------------------

cbarA = fig.colorbar(

    imA,

    ax=axA,

    fraction=0.014,

    pad=0.010,

    aspect=25
)


cbarA.set_label(

    "Normalized residue importance",

    fontsize=9,
    fontweight="bold",

    labelpad=5
)


cbarA.ax.tick_params(
    labelsize=8,
    width=1
)


for label in cbarA.ax.get_yticklabels():

    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# PANEL LETTER A
# moved farther outside
# ----------------------------------------------------------------------

axA.text(

    -0.075,
    1.10,

    "A",

    transform=axA.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# SECOND ROW
# B + C
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(

    1,
    2,

    subplot_spec=outer[1],

    width_ratios=[
        1.12,
        1.0
    ],

    wspace=0.30
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)


xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(

    xB - widthB/2,

    internal_B,

    widthB,

    label="Internal test"
)


barsB2 = axB.bar(

    xB + widthB/2,

    kelm_B,

    widthB,

    label="KELM external"
)


axB.set_ylabel(

    "Mean sequence-level\nSpearman correlation",

    fontsize=10.5,
    fontweight="bold",

    labelpad=6
)


# ----------------------------------------------------------------------
# B x-axis labels
# ----------------------------------------------------------------------

pretty_pairs = [

    "Attention\nvs\nGradient × Input",

    "Attention\nvs\nIntegrated Gradients",

    "Gradient × Input\nvs\nIntegrated Gradients"
]


if len(pretty_pairs) != len(pairs):

    pretty_pairs = []

    for p in pairs:

        p = str(p).replace(
            "_",
            " "
        )

        p = p.replace(
            "gradient x input",
            "Gradient × Input"
        )

        p = p.replace(
            "integrated gradients",
            "Integrated Gradients"
        )

        p = p.replace(
            "attention",
            "Attention"
        )

        pieces = re.split(
            r"\s+vs\s+",
            p,
            flags=re.IGNORECASE
        )

        if len(pieces) == 2:

            p = (
                pieces[0]
                + "\nvs\n"
                + pieces[1]
            )

        pretty_pairs.append(
            p
        )


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)


for label in axB.get_xticklabels():

    label.set_fontsize(8.2)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# B values
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.0035,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8.2,
            fontweight="bold",

            rotation=90
        )


valid_B = [

    v
    for v in (
        internal_B
        + kelm_B
    )

    if np.isfinite(v)
]


axB.set_ylim(

    max(
        0,
        min(valid_B) - 0.07
    ),

    min(
        1.035,
        max(valid_B) + 0.035
    )
)


# ----------------------------------------------------------------------
# B LEGEND — TOP
# ----------------------------------------------------------------------

axB.legend(

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.04
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.8,

    borderaxespad=0
)


# ----------------------------------------------------------------------
# B y ticks
# ----------------------------------------------------------------------

axB.tick_params(
    axis="y",
    labelsize=9,
    width=1.1,
    length=4
)


for label in axB.get_yticklabels():

    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# PANEL LETTER B
# ----------------------------------------------------------------------

axB.text(

    -0.13,
    1.10,

    "B",

    transform=axB.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

axC = fig.add_subplot(
    middle[1]
)


categories_C = [

    "Overall rank\nSpearman",

    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(

    xC - widthC/2,

    internal_C,

    widthC,

    label="Internal test"
)


barsC2 = axC.bar(

    xC + widthC/2,

    kelm_C,

    widthC,

    label="KELM external"
)


axC.set_ylabel(

    "Agreement",

    fontsize=10.5,
    fontweight="bold",

    labelpad=6
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)


for label in axC.get_xticklabels():

    label.set_fontsize(8.5)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# C values
# ----------------------------------------------------------------------

for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.006,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8.5,
            fontweight="bold"
        )


all_C = (
    internal_C
    + kelm_C
)


axC.set_ylim(

    max(
        0,
        min(all_C) - 0.08
    ),

    min(
        1.04,
        max(all_C) + 0.045
    )
)


# ----------------------------------------------------------------------
# C LEGEND — TOP
# ----------------------------------------------------------------------

axC.legend(

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.04
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.8,

    borderaxespad=0
)


axC.tick_params(
    axis="y",
    labelsize=9,
    width=1.1,
    length=4
)


for label in axC.get_yticklabels():

    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# PANEL LETTER C
# ----------------------------------------------------------------------

axC.text(

    -0.13,
    1.10,

    "C",

    transform=axC.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D
#
# Uses almost complete available width.
#
# Structure:
#
# [ Internal matrix ][ KELM matrix ][ colorbar ]
#
# ======================================================================

D_grid = gridspec.GridSpecFromSubplotSpec(

    1,
    3,

    subplot_spec=outer[2],

    width_ratios=[
        1,
        1,
        0.028
    ],

    wspace=0.12
)


axD1 = fig.add_subplot(
    D_grid[0]
)

axD2 = fig.add_subplot(
    D_grid[1]
)

caxD = fig.add_subplot(
    D_grid[2]
)


# ======================================================================
# D1 — INTERNAL TEST
# ======================================================================

imD1 = axD1.imshow(

    internal_matrix.values,

    vmin=0,
    vmax=1,

    aspect="equal"
)


axD1.set_xticks(
    np.arange(
        len(
            internal_matrix.columns
        )
    )
)

axD1.set_yticks(
    np.arange(
        len(
            internal_matrix.index
        )
    )
)


axD1.set_xticklabels(

    internal_matrix.columns,

    rotation=25,

    ha="right"
)


axD1.set_yticklabels(
    internal_matrix.index
)


for label in axD1.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD1.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(

    axD1,

    internal_matrix.values,

    fontsize=10
)


axD1.text(

    0.5,
    1.035,

    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D2 — KELM EXTERNAL
# ======================================================================

imD2 = axD2.imshow(

    kelm_matrix.values,

    vmin=0,
    vmax=1,

    aspect="equal"
)


axD2.set_xticks(
    np.arange(
        len(
            kelm_matrix.columns
        )
    )
)

axD2.set_yticks(
    np.arange(
        len(
            kelm_matrix.index
        )
    )
)


axD2.set_xticklabels(

    kelm_matrix.columns,

    rotation=25,

    ha="right"
)


axD2.set_yticklabels(
    kelm_matrix.index
)


for label in axD2.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD2.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(

    axD2,

    kelm_matrix.values,

    fontsize=10
)


axD2.text(

    0.5,
    1.035,

    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D COLORBAR
# ======================================================================

cbarD = fig.colorbar(

    imD2,

    cax=caxD
)


cbarD.set_label(

    "Hotspot Jaccard",

    fontsize=9.5,
    fontweight="bold",

    labelpad=6
)


cbarD.ax.tick_params(
    labelsize=8.5,
    width=1
)


for label in cbarD.ax.get_yticklabels():

    label.set_fontweight("bold")


# ======================================================================
# PANEL LETTER D
# moved well outside
# ======================================================================

axD1.text(

    -0.18,
    1.10,

    "D",

    transform=axD1.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# MAIN FIGURE TITLE
# ======================================================================

fig.suptitle(

    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",

    fontsize=15,

    fontweight="bold",

    y=0.985
)


# ======================================================================
# FINAL MARGINS
# ======================================================================

fig.subplots_adjust(

    left=0.095,

    right=0.955,

    bottom=0.075,

    top=0.925
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_COMPACT_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_COMPACT_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_COMPACT_FINAL.svg"
)


fig.savefig(

    png_file,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.08
)


fig.savefig(

    pdf_file,

    bbox_inches="tight",

    pad_inches=0.08
)


fig.savefig(

    svg_file,

    bbox_inches="tight",

    pad_inches=0.08
)


plt.show()

plt.close(fig)


print("\n" + "=" * 85)

print(
    "FINAL COMPACT FIGURE 3 GENERATED"
)

print(
    "=" * 85
)

print(
    "\nPNG:",
    png_file
)

print(
    "\nPDF:",
    pdf_file
)

print(
    "\nSVG:",
    svg_file
)

In [ ]:
# ======================================================================
# FIGURE 3 — FINAL COMPACT LAYOUT v3
#
# A = Representative CPP
# B = XAI-method agreement
# C = Original vs adjusted consensus
# D = Cross-PLM hotspot Jaccard matrices
#
# Main changes:
#   - B and C substantially more compact
#   - D fills almost entire available width
#   - D heatmaps no longer constrained to square panels
#   - minimal white space
#   - legends above B and C
#   - panel letters moved outward
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
import numpy as np
import re


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "savefig.dpi": 600
})


# ======================================================================
# MAIN FIGURE
#
# Reduced height compared with previous version
# ======================================================================

fig = plt.figure(
    figsize=(15.0, 8.3)
)


outer = gridspec.GridSpec(
    3,
    1,
    figure=fig,

    # A medium, B/C compact, D larger
    height_ratios=[
        1.18,   # A
        0.66,   # B + C
        1.20    # D
    ],

    hspace=0.48
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)


imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)


# ----------------------------------------------------------------------
# A — Y labels
# ----------------------------------------------------------------------

axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)


for label in axA.get_yticklabels():

    label.set_fontsize(10)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# A — residue labels
# ----------------------------------------------------------------------

axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa in enumerate(
            representative_sequence
        )
    ]
)


for label in axA.get_xticklabels():

    label.set_fontsize(6.5)
    label.set_fontweight("bold")


axA.tick_params(
    axis="x",
    length=2,
    width=1,
    pad=1.5
)

axA.tick_params(
    axis="y",
    length=3,
    width=1,
    pad=3
)


axA.set_xlabel(
    "Residue and sequence position",
    fontsize=11,
    fontweight="bold",
    labelpad=4
)


# ----------------------------------------------------------------------
# Representative sequence
# ----------------------------------------------------------------------

axA.text(
    0.5,
    1.04,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.2,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Top-20% hotspot borders
# ----------------------------------------------------------------------

n_hotspots = max(
    1,
    int(
        np.ceil(
            0.20 * L
        )
    )
)


top_indices = np.argsort(
    consensus_values
)[-n_hotspots:]


for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                2.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.4
        )
    )


# ----------------------------------------------------------------------
# A colorbar
# ----------------------------------------------------------------------

cbarA = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.013,
    pad=0.009,
    aspect=28
)


cbarA.set_label(
    "Normalized residue importance",
    fontsize=8.8,
    fontweight="bold",
    labelpad=4
)


cbarA.ax.tick_params(
    labelsize=7.8,
    width=1
)


for label in cbarA.ax.get_yticklabels():

    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# PANEL A LETTER
# ----------------------------------------------------------------------

axA.text(
    -0.072,
    1.10,
    "A",

    transform=axA.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# SECOND ROW — B + C
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(
    1,
    2,

    subplot_spec=outer[1],

    width_ratios=[
        1.08,
        1.00
    ],

    wspace=0.26
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)


xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)


barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level\nSpearman correlation",
    fontsize=9.8,
    fontweight="bold",
    labelpad=5
)


# ----------------------------------------------------------------------
# B x labels
# ----------------------------------------------------------------------

pretty_pairs = [

    "Attention\nvs\nGradient × Input",

    "Attention\nvs\nIntegrated Gradients",

    "Gradient × Input\nvs\nIntegrated Gradients"
]


if len(pretty_pairs) != len(pairs):

    pretty_pairs = []

    for p in pairs:

        p = str(p).replace(
            "_",
            " "
        )

        p = p.replace(
            "gradient x input",
            "Gradient × Input"
        )

        p = p.replace(
            "integrated gradients",
            "Integrated Gradients"
        )

        p = p.replace(
            "attention",
            "Attention"
        )

        pieces = re.split(
            r"\s+vs\s+",
            p,
            flags=re.IGNORECASE
        )

        if len(pieces) == 2:

            p = (
                pieces[0]
                + "\nvs\n"
                + pieces[1]
            )

        pretty_pairs.append(p)


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)


for label in axB.get_xticklabels():

    label.set_fontsize(7.8)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# B bar annotations
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.003,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=7.8,
            fontweight="bold",

            rotation=90
        )


valid_B = [
    v
    for v in (
        internal_B
        + kelm_B
    )
    if np.isfinite(v)
]


axB.set_ylim(
    max(
        0,
        min(valid_B) - 0.055
    ),
    min(
        1.025,
        max(valid_B) + 0.025
    )
)


# ----------------------------------------------------------------------
# B legend — ABOVE
# ----------------------------------------------------------------------

axB.legend(
    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.035
    ),

    ncol=2,

    fontsize=8.0,

    handlelength=1.3,

    columnspacing=1.6,

    borderaxespad=0
)


axB.tick_params(
    axis="y",
    labelsize=8.5,
    width=1.1,
    length=3.5
)


for label in axB.get_yticklabels():

    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# B LETTER
# ----------------------------------------------------------------------

axB.text(
    -0.12,
    1.12,
    "B",

    transform=axB.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

axC = fig.add_subplot(
    middle[1]
)


categories_C = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(
    xC - widthC/2,
    internal_C,
    widthC,
    label="Internal test"
)


barsC2 = axC.bar(
    xC + widthC/2,
    kelm_C,
    widthC,
    label="KELM external"
)


axC.set_ylabel(
    "Agreement",
    fontsize=9.8,
    fontweight="bold",
    labelpad=5
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)


for label in axC.get_xticklabels():

    label.set_fontsize(8)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# C annotations
# ----------------------------------------------------------------------

for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.005,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8,
            fontweight="bold"
        )


all_C = (
    internal_C
    + kelm_C
)


axC.set_ylim(
    max(
        0,
        min(all_C) - 0.06
    ),
    min(
        1.025,
        max(all_C) + 0.03
    )
)


# ----------------------------------------------------------------------
# C legend — ABOVE
# ----------------------------------------------------------------------

axC.legend(
    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.035
    ),

    ncol=2,

    fontsize=8.0,

    handlelength=1.3,

    columnspacing=1.6,

    borderaxespad=0
)


axC.tick_params(
    axis="y",
    labelsize=8.5,
    width=1.1,
    length=3.5
)


for label in axC.get_yticklabels():

    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# C LETTER
# ----------------------------------------------------------------------

axC.text(
    -0.12,
    1.12,
    "C",

    transform=axC.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D
#
# FULL-WIDTH DESIGN
#
# The important change is aspect="auto".
# This allows the heatmaps to use the complete subplot area.
# ======================================================================

D_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[2],

    width_ratios=[
        1,
        1,
        0.025
    ],

    # Very small space between heatmaps
    wspace=0.10
)


axD1 = fig.add_subplot(
    D_grid[0]
)

axD2 = fig.add_subplot(
    D_grid[1]
)

caxD = fig.add_subplot(
    D_grid[2]
)


# ======================================================================
# D1 — INTERNAL TEST
# ======================================================================

imD1 = axD1.imshow(
    internal_matrix.values,

    vmin=0,
    vmax=1,

    # IMPORTANT:
    # allow matrix to fill complete panel
    aspect="auto",

    interpolation="nearest"
)


axD1.set_xticks(
    np.arange(
        len(
            internal_matrix.columns
        )
    )
)

axD1.set_yticks(
    np.arange(
        len(
            internal_matrix.index
        )
    )
)


axD1.set_xticklabels(
    internal_matrix.columns,
    rotation=20,
    ha="right"
)

axD1.set_yticklabels(
    internal_matrix.index
)


for label in axD1.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD1.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=10
)


axD1.text(
    0.5,
    1.025,

    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D2 — KELM EXTERNAL
# ======================================================================

imD2 = axD2.imshow(
    kelm_matrix.values,

    vmin=0,
    vmax=1,

    aspect="auto",

    interpolation="nearest"
)


axD2.set_xticks(
    np.arange(
        len(
            kelm_matrix.columns
        )
    )
)

axD2.set_yticks(
    np.arange(
        len(
            kelm_matrix.index
        )
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns,
    rotation=20,
    ha="right"
)

axD2.set_yticklabels(
    kelm_matrix.index
)


for label in axD2.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD2.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=10
)


axD2.text(
    0.5,
    1.025,

    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D COLORBAR
# ======================================================================

cbarD = fig.colorbar(
    imD2,
    cax=caxD
)


cbarD.set_label(
    "Hotspot Jaccard",
    fontsize=9.2,
    fontweight="bold",
    labelpad=5
)


cbarD.ax.tick_params(
    labelsize=8.2,
    width=1
)


for label in cbarD.ax.get_yticklabels():

    label.set_fontweight("bold")


# ======================================================================
# D LETTER
# ======================================================================

axD1.text(
    -0.10,
    1.09,
    "D",

    transform=axD1.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# MAIN FIGURE TITLE
# ======================================================================

fig.suptitle(
    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",

    fontsize=14.5,

    fontweight="bold",

    y=0.987
)


# ======================================================================
# FINAL COMPACT MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.095,
    right=0.955,
    bottom=0.075,
    top=0.925
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_COMPACT_v3.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_COMPACT_v3.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_COMPACT_v3.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.06
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 3 COMPACT v3 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 3 — FINAL COMPACT LAYOUT v3
#
# A = Representative CPP
# B = XAI-method agreement
# C = Original vs adjusted consensus
# D = Cross-PLM hotspot Jaccard matrices
#
# Main changes:
#   - B and C substantially more compact
#   - D fills almost entire available width
#   - D heatmaps no longer constrained to square panels
#   - minimal white space
#   - legends above B and C
#   - panel letters moved outward
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
import numpy as np
import re


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "savefig.dpi": 600
})


# ======================================================================
# MAIN FIGURE
#
# Reduced height compared with previous version
# ======================================================================

fig = plt.figure(
    figsize=(15.0, 8.3)
)


outer = gridspec.GridSpec(
    3,
    1,
    figure=fig,

    # A medium, B/C compact, D larger
    height_ratios=[
        1.18,   # A
        0.66,   # B + C
        1.40    # D
    ],

    hspace=0.48
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)


imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)


# ----------------------------------------------------------------------
# A — Y labels
# ----------------------------------------------------------------------

axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)


for label in axA.get_yticklabels():

    label.set_fontsize(10)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# A — residue labels
# ----------------------------------------------------------------------

axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa in enumerate(
            representative_sequence
        )
    ]
)


for label in axA.get_xticklabels():

    label.set_fontsize(6.5)
    label.set_fontweight("bold")


axA.tick_params(
    axis="x",
    length=2,
    width=1,
    pad=1.5
)

axA.tick_params(
    axis="y",
    length=3,
    width=1,
    pad=3
)


axA.set_xlabel(
    "Residue and sequence position",
    fontsize=11,
    fontweight="bold",
    labelpad=4
)


# ----------------------------------------------------------------------
# Representative sequence
# ----------------------------------------------------------------------

axA.text(
    0.5,
    1.04,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.2,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Top-20% hotspot borders
# ----------------------------------------------------------------------

n_hotspots = max(
    1,
    int(
        np.ceil(
            0.20 * L
        )
    )
)


top_indices = np.argsort(
    consensus_values
)[-n_hotspots:]


for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                2.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.4
        )
    )


# ----------------------------------------------------------------------
# A colorbar
# ----------------------------------------------------------------------

cbarA = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.013,
    pad=0.009,
    aspect=28
)


cbarA.set_label(
    "Normalized residue importance",
    fontsize=8.8,
    fontweight="bold",
    labelpad=4
)


cbarA.ax.tick_params(
    labelsize=7.8,
    width=1
)


for label in cbarA.ax.get_yticklabels():

    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# PANEL A LETTER
# ----------------------------------------------------------------------

axA.text(
    -0.072,
    1.10,
    "A",

    transform=axA.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# SECOND ROW — B + C
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(
    1,
    2,

    subplot_spec=outer[1],

    width_ratios=[
        1.08,
        1.00
    ],

    wspace=0.26
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)


xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)


barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level\nSpearman correlation",
    fontsize=9.8,
    fontweight="bold",
    labelpad=5
)


# ----------------------------------------------------------------------
# B x labels
# ----------------------------------------------------------------------

pretty_pairs = [

    "Attention\nvs\nGradient × Input",

    "Attention\nvs\nIntegrated Gradients",

    "Gradient × Input\nvs\nIntegrated Gradients"
]


if len(pretty_pairs) != len(pairs):

    pretty_pairs = []

    for p in pairs:

        p = str(p).replace(
            "_",
            " "
        )

        p = p.replace(
            "gradient x input",
            "Gradient × Input"
        )

        p = p.replace(
            "integrated gradients",
            "Integrated Gradients"
        )

        p = p.replace(
            "attention",
            "Attention"
        )

        pieces = re.split(
            r"\s+vs\s+",
            p,
            flags=re.IGNORECASE
        )

        if len(pieces) == 2:

            p = (
                pieces[0]
                + "\nvs\n"
                + pieces[1]
            )

        pretty_pairs.append(p)


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)


for label in axB.get_xticklabels():

    label.set_fontsize(7.8)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# B bar annotations
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.003,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=7.8,
            fontweight="bold",

            rotation=90
        )


valid_B = [
    v
    for v in (
        internal_B
        + kelm_B
    )
    if np.isfinite(v)
]


axB.set_ylim(
    max(
        0,
        min(valid_B) - 0.055
    ),
    min(
        1.025,
        max(valid_B) + 0.025
    )
)


# ----------------------------------------------------------------------
# B legend — ABOVE
# ----------------------------------------------------------------------

axB.legend(
    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.035
    ),

    ncol=2,

    fontsize=8.0,

    handlelength=1.3,

    columnspacing=1.6,

    borderaxespad=0
)


axB.tick_params(
    axis="y",
    labelsize=8.5,
    width=1.1,
    length=3.5
)


for label in axB.get_yticklabels():

    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# B LETTER
# ----------------------------------------------------------------------

axB.text(
    -0.12,
    1.12,
    "B",

    transform=axB.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

axC = fig.add_subplot(
    middle[1]
)


categories_C = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(
    xC - widthC/2,
    internal_C,
    widthC,
    label="Internal test"
)


barsC2 = axC.bar(
    xC + widthC/2,
    kelm_C,
    widthC,
    label="KELM external"
)


axC.set_ylabel(
    "Agreement",
    fontsize=9.8,
    fontweight="bold",
    labelpad=5
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)


for label in axC.get_xticklabels():

    label.set_fontsize(8)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# C annotations
# ----------------------------------------------------------------------

for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.005,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8,
            fontweight="bold"
        )


all_C = (
    internal_C
    + kelm_C
)


axC.set_ylim(
    max(
        0,
        min(all_C) - 0.06
    ),
    min(
        1.025,
        max(all_C) + 0.03
    )
)


# ----------------------------------------------------------------------
# C legend — ABOVE
# ----------------------------------------------------------------------

axC.legend(
    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.035
    ),

    ncol=2,

    fontsize=8.0,

    handlelength=1.3,

    columnspacing=1.6,

    borderaxespad=0
)


axC.tick_params(
    axis="y",
    labelsize=8.5,
    width=1.1,
    length=3.5
)


for label in axC.get_yticklabels():

    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# C LETTER
# ----------------------------------------------------------------------

axC.text(
    -0.12,
    1.12,
    "C",

    transform=axC.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D
#
# FULL-WIDTH DESIGN
#
# The important change is aspect="auto".
# This allows the heatmaps to use the complete subplot area.
# ======================================================================

D_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[2],

    width_ratios=[
        1,
        1,
        0.025
    ],

    # Very small space between heatmaps
    wspace=0.10
)


axD1 = fig.add_subplot(
    D_grid[0]
)

axD2 = fig.add_subplot(
    D_grid[1]
)

caxD = fig.add_subplot(
    D_grid[2]
)


# ======================================================================
# D1 — INTERNAL TEST
# ======================================================================

imD1 = axD1.imshow(
    internal_matrix.values,

    vmin=0,
    vmax=1,

    # IMPORTANT:
    # allow matrix to fill complete panel
    aspect="auto",

    interpolation="nearest"
)


axD1.set_xticks(
    np.arange(
        len(
            internal_matrix.columns
        )
    )
)

axD1.set_yticks(
    np.arange(
        len(
            internal_matrix.index
        )
    )
)


axD1.set_xticklabels(
    internal_matrix.columns,
    rotation=20,
    ha="right"
)

axD1.set_yticklabels(
    internal_matrix.index
)


for label in axD1.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD1.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=10
)


axD1.text(
    0.5,
    1.025,

    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D2 — KELM EXTERNAL
# ======================================================================

imD2 = axD2.imshow(
    kelm_matrix.values,

    vmin=0,
    vmax=1,

    aspect="auto",

    interpolation="nearest"
)


axD2.set_xticks(
    np.arange(
        len(
            kelm_matrix.columns
        )
    )
)

axD2.set_yticks(
    np.arange(
        len(
            kelm_matrix.index
        )
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns,
    rotation=20,
    ha="right"
)

axD2.set_yticklabels(
    kelm_matrix.index
)


for label in axD2.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD2.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=10
)


axD2.text(
    0.5,
    1.025,

    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D COLORBAR
# ======================================================================

cbarD = fig.colorbar(
    imD2,
    cax=caxD
)


cbarD.set_label(
    "Hotspot Jaccard",
    fontsize=9.2,
    fontweight="bold",
    labelpad=5
)


cbarD.ax.tick_params(
    labelsize=8.2,
    width=1
)


for label in cbarD.ax.get_yticklabels():

    label.set_fontweight("bold")


# ======================================================================
# D LETTER
# ======================================================================

axD1.text(
    -0.10,
    1.09,
    "D",

    transform=axD1.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# MAIN FIGURE TITLE
# ======================================================================

fig.suptitle(
    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",

    fontsize=14.5,

    fontweight="bold",

    y=0.987
)


# ======================================================================
# FINAL COMPACT MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.095,
    right=0.955,
    bottom=0.075,
    top=0.925
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_COMPACT_v3.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_COMPACT_v3.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_FINAL_COMPACT_v3.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.06
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 3 COMPACT v3 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# MAIN FIGURE — BALANCED, COMPACT, NO OVERLAP
# ======================================================================

fig = plt.figure(
    figsize=(14.5, 8.5)
)

outer = gridspec.GridSpec(
    3,
    1,
    figure=fig,

    height_ratios=[
        1.12,   # A
        0.62,   # B + C compact
        1.50    # D larger but balanced
    ],

    hspace=0.46
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)

imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)

axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)

for label in axA.get_yticklabels():
    label.set_fontsize(10)
    label.set_fontweight("bold")

axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa in enumerate(representative_sequence)
    ]
)

for label in axA.get_xticklabels():
    label.set_fontsize(6.5)
    label.set_fontweight("bold")

axA.tick_params(
    axis="x",
    length=2,
    width=1,
    pad=1.5
)

axA.tick_params(
    axis="y",
    length=3,
    width=1,
    pad=3
)

axA.set_xlabel(
    "Residue and sequence position",
    fontsize=11,
    fontweight="bold",
    labelpad=4
)

axA.text(
    0.5,
    1.04,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.2,
    fontweight="bold"
)

n_hotspots = max(
    1,
    int(np.ceil(0.20 * L))
)

top_indices = np.argsort(
    consensus_values
)[-n_hotspots:]

for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                2.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.4
        )
    )

cbarA = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.013,
    pad=0.009,
    aspect=28
)

cbarA.set_label(
    "Normalized residue importance",
    fontsize=8.8,
    fontweight="bold",
    labelpad=4
)

cbarA.ax.tick_params(
    labelsize=7.8,
    width=1
)

for label in cbarA.ax.get_yticklabels():
    label.set_fontweight("bold")

axA.text(
    -0.075,
    1.10,
    "A",

    transform=axA.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# SECOND ROW — B + C
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(
    1,
    2,

    subplot_spec=outer[1],

    width_ratios=[
        1.08,
        1.00
    ],

    wspace=0.24
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)

xB = np.arange(
    len(pairs)
)

widthB = 0.32

barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)

barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)

axB.set_ylabel(
    "Mean sequence-level\nSpearman correlation",
    fontsize=9.5,
    fontweight="bold",
    labelpad=4
)

pretty_pairs = [
    "Attention\nvs\nGradient × Input",
    "Attention\nvs\nIntegrated Gradients",
    "Gradient × Input\nvs\nIntegrated Gradients"
]

axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)

for label in axB.get_xticklabels():
    label.set_fontsize(7.6)
    label.set_fontweight("bold")

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.003,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=7.6,
            fontweight="bold",
            rotation=90
        )

valid_B = [
    v
    for v in internal_B + kelm_B
    if np.isfinite(v)
]

axB.set_ylim(
    max(0, min(valid_B) - 0.055),
    min(1.025, max(valid_B) + 0.025)
)

axB.legend(
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.50, 1.035),
    ncol=2,
    fontsize=8.0,
    handlelength=1.3,
    columnspacing=1.5,
    borderaxespad=0
)

axB.tick_params(
    axis="y",
    labelsize=8.4,
    width=1.1,
    length=3.5
)

for label in axB.get_yticklabels():
    label.set_fontweight("bold")

axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)

axB.text(
    -0.13,
    1.14,
    "B",

    transform=axB.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

axC = fig.add_subplot(
    middle[1]
)

categories_C = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard"
]

xC = np.arange(2)

widthC = 0.30

barsC1 = axC.bar(
    xC - widthC/2,
    internal_C,
    widthC,
    label="Internal test"
)

barsC2 = axC.bar(
    xC + widthC/2,
    kelm_C,
    widthC,
    label="KELM external"
)

axC.set_ylabel(
    "Agreement",
    fontsize=9.5,
    fontweight="bold",
    labelpad=4
)

axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)

for label in axC.get_xticklabels():
    label.set_fontsize(8)
    label.set_fontweight("bold")

for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.005,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold"
        )

all_C = internal_C + kelm_C

axC.set_ylim(
    max(0, min(all_C) - 0.06),
    min(1.025, max(all_C) + 0.03)
)

axC.legend(
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.50, 1.035),
    ncol=2,
    fontsize=8.0,
    handlelength=1.3,
    columnspacing=1.5,
    borderaxespad=0
)

axC.tick_params(
    axis="y",
    labelsize=8.4,
    width=1.1,
    length=3.5
)

for label in axC.get_yticklabels():
    label.set_fontweight("bold")

axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)

axC.text(
    -0.13,
    1.14,
    "C",

    transform=axC.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D — BALANCED LARGE HEATMAPS, NO STRETCHING
# ======================================================================

D_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[2],

    width_ratios=[
        1,
        1,
        0.035
    ],

    wspace=0.14
)

axD1 = fig.add_subplot(
    D_grid[0]
)

axD2 = fig.add_subplot(
    D_grid[1]
)

caxD = fig.add_subplot(
    D_grid[2]
)


# ----------------------------------------------------------------------
# D1 Internal
# ----------------------------------------------------------------------

imD1 = axD1.imshow(
    internal_matrix.values,

    vmin=0,
    vmax=1,

    aspect="equal",

    interpolation="nearest"
)

axD1.set_xticks(
    np.arange(
        len(internal_matrix.columns)
    )
)

axD1.set_yticks(
    np.arange(
        len(internal_matrix.index)
    )
)

axD1.set_xticklabels(
    internal_matrix.columns,
    rotation=25,
    ha="right"
)

axD1.set_yticklabels(
    internal_matrix.index
)

for label in axD1.get_xticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")

for label in axD1.get_yticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")

annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=9.5
)

axD1.text(
    0.5,
    1.03,
    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# D2 KELM
# ----------------------------------------------------------------------

imD2 = axD2.imshow(
    kelm_matrix.values,

    vmin=0,
    vmax=1,

    aspect="equal",

    interpolation="nearest"
)

axD2.set_xticks(
    np.arange(
        len(kelm_matrix.columns)
    )
)

axD2.set_yticks(
    np.arange(
        len(kelm_matrix.index)
    )
)

axD2.set_xticklabels(
    kelm_matrix.columns,
    rotation=25,
    ha="right"
)

axD2.set_yticklabels(
    kelm_matrix.index
)

for label in axD2.get_xticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")

for label in axD2.get_yticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")

annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=9.5
)

axD2.text(
    0.5,
    1.03,
    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# D colorbar
# ----------------------------------------------------------------------

cbarD = fig.colorbar(
    imD2,
    cax=caxD
)

cbarD.set_label(
    "Hotspot Jaccard",
    fontsize=9.2,
    fontweight="bold",
    labelpad=5
)

cbarD.ax.tick_params(
    labelsize=8.2,
    width=1
)

for label in cbarD.ax.get_yticklabels():
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# D letter
# ----------------------------------------------------------------------

axD1.text(
    -0.15,
    1.10,
    "D",

    transform=axD1.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# MAIN TITLE
# ======================================================================

fig.suptitle(
    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",

    fontsize=14.5,
    fontweight="bold",

    y=0.988
)


# ======================================================================
# MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.095,
    right=0.955,
    bottom=0.075,
    top=0.925
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_BALANCED_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_BALANCED_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_BALANCED_FINAL.svg"
)

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.06
)

plt.show()

plt.close(fig)

print("\n" + "=" * 90)
print("BALANCED FIGURE 3 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 3 — PUBLICATION-STYLE LAYOUT
#
# A = Representative CPP
# B = XAI-method agreement
# C = Original vs adjusted consensus
# D = Cross-PLM hotspot Jaccard matrices
#
# B and C y-axes fixed at 0–1.05
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
import numpy as np
import re


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 9,

    "axes.linewidth": 1.1,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(15.5, 10.0)
)

outer = gridspec.GridSpec(
    3,
    1,
    figure=fig,

    height_ratios=[
        1.10,   # A
        0.95,   # B + C
        1.20    # D
    ],

    hspace=0.42
)


# ======================================================================
# PANEL A
# ======================================================================

axA = fig.add_subplot(
    outer[0]
)

imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)


axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)

for label in axA.get_yticklabels():
    label.set_fontsize(10.5)
    label.set_fontweight("bold")


axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa in enumerate(
            representative_sequence
        )
    ]
)

for label in axA.get_xticklabels():
    label.set_fontsize(6.8)
    label.set_fontweight("bold")


axA.tick_params(
    axis="x",
    length=2,
    width=1,
    pad=2
)

axA.tick_params(
    axis="y",
    length=3,
    width=1,
    pad=4
)


axA.set_xlabel(
    "Residue and sequence position",
    fontsize=11,
    fontweight="bold",
    labelpad=5
)


axA.text(
    0.5,
    1.04,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.7,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# Top-20% hotspot borders
# ----------------------------------------------------------------------

n_hotspots = max(
    1,
    int(np.ceil(0.20 * L))
)

top_indices = np.argsort(
    consensus_values
)[-n_hotspots:]


for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                2.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.5
        )
    )


# ----------------------------------------------------------------------
# Colorbar A
# ----------------------------------------------------------------------

cbarA = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.014,
    pad=0.010,
    aspect=26
)

cbarA.set_label(
    "Normalized residue importance",
    fontsize=9,
    fontweight="bold",
    labelpad=5
)

cbarA.ax.tick_params(
    labelsize=8,
    width=1
)

for label in cbarA.ax.get_yticklabels():
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# Panel letter A
# ----------------------------------------------------------------------

axA.text(
    -0.075,
    1.10,
    "A",

    transform=axA.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# SECOND ROW — B + C
# ======================================================================

middle = gridspec.GridSpecFromSubplotSpec(
    1,
    2,

    subplot_spec=outer[1],

    width_ratios=[
        1.10,
        1.00
    ],

    wspace=0.24
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    middle[0]
)

xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)

barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level\nSpearman correlation",
    fontsize=10.5,
    fontweight="bold",
    labelpad=5
)


pretty_pairs = [
    "Attention\nvs\nGradient × Input",
    "Attention\nvs\nIntegrated Gradients",
    "Gradient × Input\nvs\nIntegrated Gradients"
]


if len(pretty_pairs) != len(pairs):

    pretty_pairs = []

    for p in pairs:

        p = str(p).replace(
            "_",
            " "
        )

        p = p.replace(
            "gradient x input",
            "Gradient × Input"
        )

        p = p.replace(
            "integrated gradients",
            "Integrated Gradients"
        )

        p = p.replace(
            "attention",
            "Attention"
        )

        pieces = re.split(
            r"\s+vs\s+",
            p,
            flags=re.IGNORECASE
        )

        if len(pieces) == 2:
            p = pieces[0] + "\nvs\n" + pieces[1]

        pretty_pairs.append(p)


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)

for label in axB.get_xticklabels():
    label.set_fontsize(8.5)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# BAR VALUES
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.018,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8.5,
            fontweight="bold"
        )


# ----------------------------------------------------------------------
# IMPORTANT: FIXED Y AXIS
# ----------------------------------------------------------------------

axB.set_ylim(
    0,
    1.05
)

axB.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


axB.legend(
    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.04
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.7,

    borderaxespad=0
)


axB.tick_params(
    axis="y",
    labelsize=9,
    width=1.1,
    length=4
)

for label in axB.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


axB.text(
    -0.13,
    1.10,
    "B",

    transform=axB.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

axC = fig.add_subplot(
    middle[1]
)


categories_C = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(
    xC - widthC/2,
    internal_C,
    widthC,
    label="Internal test"
)

barsC2 = axC.bar(
    xC + widthC/2,
    kelm_C,
    widthC,
    label="KELM external"
)


axC.set_ylabel(
    "Agreement",
    fontsize=10.5,
    fontweight="bold",
    labelpad=5
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)

for label in axC.get_xticklabels():
    label.set_fontsize(8.5)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# C value annotations
# ----------------------------------------------------------------------

for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.018,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8.5,
            fontweight="bold"
        )


# ----------------------------------------------------------------------
# IMPORTANT: FIXED Y AXIS
# ----------------------------------------------------------------------

axC.set_ylim(
    0,
    1.05
)

axC.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


axC.legend(
    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.04
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.7,

    borderaxespad=0
)


axC.tick_params(
    axis="y",
    labelsize=9,
    width=1.1,
    length=4
)

for label in axC.get_yticklabels():
    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


axC.text(
    -0.13,
    1.10,
    "C",

    transform=axC.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D — LARGE, BALANCED, NOT STRETCHED
# ======================================================================

D_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[2],

    width_ratios=[
        1,
        1,
        0.028
    ],

    wspace=0.13
)


axD1 = fig.add_subplot(
    D_grid[0]
)

axD2 = fig.add_subplot(
    D_grid[1]
)

caxD = fig.add_subplot(
    D_grid[2]
)


# ======================================================================
# D1 — INTERNAL TEST
# ======================================================================

imD1 = axD1.imshow(
    internal_matrix.values,
    vmin=0,
    vmax=1,
    aspect="equal",
    interpolation="nearest"
)


axD1.set_xticks(
    np.arange(
        len(
            internal_matrix.columns
        )
    )
)

axD1.set_yticks(
    np.arange(
        len(
            internal_matrix.index
        )
    )
)


axD1.set_xticklabels(
    internal_matrix.columns,
    rotation=0,
    ha="center"
)

axD1.set_yticklabels(
    internal_matrix.index
)


for label in axD1.get_xticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")

for label in axD1.get_yticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=10
)


axD1.text(
    0.5,
    1.035,
    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D2 — KELM EXTERNAL
# ======================================================================

imD2 = axD2.imshow(
    kelm_matrix.values,
    vmin=0,
    vmax=1,
    aspect="equal",
    interpolation="nearest"
)


axD2.set_xticks(
    np.arange(
        len(
            kelm_matrix.columns
        )
    )
)

axD2.set_yticks(
    np.arange(
        len(
            kelm_matrix.index
        )
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns,
    rotation=0,
    ha="center"
)

axD2.set_yticklabels(
    kelm_matrix.index
)


for label in axD2.get_xticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")

for label in axD2.get_yticklabels():
    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=10
)


axD2.text(
    0.5,
    1.035,
    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


# ======================================================================
# D COLORBAR
# ======================================================================

cbarD = fig.colorbar(
    imD2,
    cax=caxD
)

cbarD.set_label(
    "Hotspot Jaccard",
    fontsize=9.5,
    fontweight="bold",
    labelpad=6
)

cbarD.ax.tick_params(
    labelsize=8.5,
    width=1
)

for label in cbarD.ax.get_yticklabels():
    label.set_fontweight("bold")


# ======================================================================
# PANEL D LETTER
# ======================================================================

axD1.text(
    -0.17,
    1.08,
    "D",

    transform=axD1.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# MAIN TITLE
# ======================================================================

fig.suptitle(
    "Construction, Agreement, and Stability of Multi-PLM Explainable AI",
    fontsize=15,
    fontweight="bold",
    y=0.985
)


# ======================================================================
# FINAL MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.09,
    right=0.96,
    bottom=0.07,
    top=0.92
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_MS_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_MS_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_3_XAI_consensus_MS_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.06
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("MANUSCRIPT FIGURE 3 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 3A — REPRESENTATIVE CPP XAI MAP
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.labelweight": "bold",
    "xtick.labelsize": 8,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.1,
    "savefig.dpi": 600
})


fig, axA = plt.subplots(
    figsize=(14.5, 3.4)
)


imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)


# Y labels
axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)

for label in axA.get_yticklabels():
    label.set_fontsize(10)
    label.set_fontweight("bold")


# X labels
axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa in enumerate(
            representative_sequence
        )
    ]
)

for label in axA.get_xticklabels():
    label.set_fontsize(6.8)
    label.set_fontweight("bold")


axA.tick_params(
    axis="x",
    length=2,
    width=1,
    pad=2
)

axA.tick_params(
    axis="y",
    length=3,
    width=1,
    pad=4
)


axA.set_xlabel(
    "Residue and sequence position",
    fontsize=11,
    fontweight="bold",
    labelpad=5
)


# Representative sequence line
axA.text(
    0.5,
    1.045,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.7,
    fontweight="bold"
)


# Top-20% hotspot borders
n_hotspots = max(
    1,
    int(np.ceil(0.20 * L))
)

top_indices = np.argsort(
    consensus_values
)[-n_hotspots:]


for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                2.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.5
        )
    )


# Colorbar
cbarA = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.014,
    pad=0.010,
    aspect=28
)

cbarA.set_label(
    "Normalized residue importance",
    fontsize=9,
    fontweight="bold",
    labelpad=5
)

cbarA.ax.tick_params(
    labelsize=8,
    width=1
)

for label in cbarA.ax.get_yticklabels():
    label.set_fontweight("bold")


# Panel letter
axA.text(
    -0.075,
    1.10,
    "A",

    transform=axA.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


fig.subplots_adjust(
    left=0.11,
    right=0.965,
    bottom=0.22,
    top=0.82
)


# Save
png_A = (
    OUT_DIR
    / "Figure_3A_Representative_XAI_map.png"
)

pdf_A = (
    OUT_DIR
    / "Figure_3A_Representative_XAI_map.pdf"
)

svg_A = (
    OUT_DIR
    / "Figure_3A_Representative_XAI_map.svg"
)


fig.savefig(
    png_A,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_A,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_A,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()
plt.close(fig)


print("Figure 3A saved:")
print(png_A)
print(pdf_A)
print(svg_A)

In [ ]:
# ======================================================================
# FIGURE 3B–D — COMPACT MANUSCRIPT VERSION
#
# B = XAI-method agreement
# C = Original vs adjusted consensus
# D = Cross-PLM hotspot Jaccard matrices
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
import numpy as np


plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.labelsize": 10,
    "axes.labelweight": "bold",
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 1.0,
    "savefig.dpi": 600
})


# ======================================================================
# FIGURE SIZE
# ======================================================================

fig = plt.figure(
    figsize=(13.5, 6.1)
)


outer = gridspec.GridSpec(
    2,
    1,
    figure=fig,

    height_ratios=[
        0.80,   # B + C
        1.20    # D
    ],

    hspace=0.38
)


# ======================================================================
# TOP ROW — B + C
# ======================================================================

top = gridspec.GridSpecFromSubplotSpec(
    1,
    2,

    subplot_spec=outer[0],

    width_ratios=[
        1.10,
        1.00
    ],

    wspace=0.25
)


# ======================================================================
# PANEL B
# ======================================================================

axB = fig.add_subplot(
    top[0]
)

xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)

barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level\nSpearman correlation",
    fontsize=9.5,
    fontweight="bold",
    labelpad=4
)


pretty_pairs = [
    "Attention\nvs\nGradient × Input",
    "Attention\nvs\nIntegrated Gradients",
    "Gradient × Input\nvs\nIntegrated Gradients"
]


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)

for label in axB.get_xticklabels():
    label.set_fontsize(7.5)
    label.set_fontweight("bold")


# Fixed 0–1.05
axB.set_ylim(
    0,
    1.05
)

axB.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.015,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=7.5,
            fontweight="bold"
        )


axB.legend(
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.50, 1.03),
    ncol=2,
    fontsize=7.8,
    handlelength=1.3,
    columnspacing=1.5
)


for label in axB.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


axB.text(
    -0.12,
    1.10,
    "B",

    transform=axB.transAxes,

    fontsize=18,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

axC = fig.add_subplot(
    top[1]
)


categories_C = [
    "Overall rank\nSpearman",
    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(
    xC - widthC/2,
    internal_C,
    widthC,
    label="Internal test"
)

barsC2 = axC.bar(
    xC + widthC/2,
    kelm_C,
    widthC,
    label="KELM external"
)


axC.set_ylabel(
    "Agreement",
    fontsize=9.5,
    fontweight="bold",
    labelpad=4
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)

for label in axC.get_xticklabels():
    label.set_fontsize(7.8)
    label.set_fontweight("bold")


# Fixed 0–1.05
axC.set_ylim(
    0,
    1.05
)

axC.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.015,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=7.8,
            fontweight="bold"
        )


axC.legend(
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.50, 1.03),
    ncol=2,
    fontsize=7.8,
    handlelength=1.3,
    columnspacing=1.5
)


for label in axC.get_yticklabels():
    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


axC.text(
    -0.12,
    1.10,
    "C",

    transform=axC.transAxes,

    fontsize=18,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D
# ======================================================================

bottom = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[1],

    width_ratios=[
        1,
        1,
        0.028
    ],

    wspace=0.12
)


axD1 = fig.add_subplot(
    bottom[0]
)

axD2 = fig.add_subplot(
    bottom[1]
)

caxD = fig.add_subplot(
    bottom[2]
)


# ----------------------------------------------------------------------
# D1
# ----------------------------------------------------------------------

imD1 = axD1.imshow(
    internal_matrix.values,
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest"
)


axD1.set_xticks(
    np.arange(
        len(internal_matrix.columns)
    )
)

axD1.set_yticks(
    np.arange(
        len(internal_matrix.index)
    )
)


axD1.set_xticklabels(
    internal_matrix.columns,
    fontsize=8.5,
    fontweight="bold"
)

axD1.set_yticklabels(
    internal_matrix.index,
    fontsize=8.5,
    fontweight="bold"
)


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=9
)


axD1.text(
    0.5,
    1.025,

    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# D2
# ----------------------------------------------------------------------

imD2 = axD2.imshow(
    kelm_matrix.values,
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest"
)


axD2.set_xticks(
    np.arange(
        len(kelm_matrix.columns)
    )
)

axD2.set_yticks(
    np.arange(
        len(kelm_matrix.index)
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns,
    fontsize=8.5,
    fontweight="bold"
)

axD2.set_yticklabels(
    kelm_matrix.index,
    fontsize=8.5,
    fontweight="bold"
)


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=9
)


axD2.text(
    0.5,
    1.025,

    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.5,
    fontweight="bold"
)


# ----------------------------------------------------------------------
# D colorbar
# ----------------------------------------------------------------------

cbarD = fig.colorbar(
    imD2,
    cax=caxD
)

cbarD.set_label(
    "Hotspot Jaccard",
    fontsize=8.8,
    fontweight="bold",
    labelpad=5
)

cbarD.ax.tick_params(
    labelsize=7.8
)

for label in cbarD.ax.get_yticklabels():
    label.set_fontweight("bold")


# D label
axD1.text(
    -0.10,
    1.08,
    "D",

    transform=axD1.transAxes,

    fontsize=18,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.095,
    right=0.955,
    bottom=0.09,
    top=0.92
)


# ======================================================================
# SAVE
# ======================================================================

png_BD = (
    OUT_DIR
    / "Figure_3B-D_XAI_agreement_COMPACT.png"
)

pdf_BD = (
    OUT_DIR
    / "Figure_3B-D_XAI_agreement_COMPACT.pdf"
)

svg_BD = (
    OUT_DIR
    / "Figure_3B-D_XAI_agreement_COMPACT.svg"
)


fig.savefig(
    png_BD,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_BD,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_BD,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()
plt.close(fig)


print("Figure 3B-D saved:")
print(png_BD)
print(pdf_BD)
print(svg_BD)

In [ ]:
from google.colab import files

files.download(str(png_A))

In [ ]:
# ======================================================================
# FIGURE 3B–D — BALANCED 2 × 2 MANUSCRIPT LAYOUT
#
#        B                         C
# XAI agreement           consensus stability
#
#        D                         D
# Internal heatmap          KELM heatmap
#
# All four plotting regions use equal width and equal row height
# ======================================================================

import matplotlib.pyplot as plt
import numpy as np


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
#
# 2 × 2 grid with equal row/column dimensions
# ======================================================================

fig = plt.figure(
    figsize=(13.5, 7.2)
)


gs = fig.add_gridspec(

    2,
    2,

    width_ratios=[
        1,
        1
    ],

    height_ratios=[
        1,
        1
    ],

    left=0.085,
    right=0.94,

    bottom=0.10,
    top=0.93,

    wspace=0.24,
    hspace=0.34
)


axB  = fig.add_subplot(gs[0, 0])
axC  = fig.add_subplot(gs[0, 1])

axD1 = fig.add_subplot(gs[1, 0])
axD2 = fig.add_subplot(gs[1, 1])


# ======================================================================
# PANEL B
# ======================================================================

xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)

barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level\nSpearman correlation",
    fontsize=10.5,
    fontweight="bold",
    labelpad=5
)


pretty_pairs = [

    "Attention\nvs\nGradient × Input",

    "Attention\nvs\nIntegrated Gradients",

    "Gradient × Input\nvs\nIntegrated Gradients"
]


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)


for label in axB.get_xticklabels():

    label.set_fontsize(8.0)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# B axis: 0–1.05
# ----------------------------------------------------------------------

axB.set_ylim(
    0,
    1.05
)

axB.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


# ----------------------------------------------------------------------
# B annotations
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.018,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8.5,
            fontweight="bold"
        )


# ----------------------------------------------------------------------
# B legend directly above axis
# ----------------------------------------------------------------------

axB.legend(

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.015
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.6,

    borderaxespad=0
)


axB.tick_params(
    axis="both",
    labelsize=9,
    width=1.1,
    length=4
)


for label in axB.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# B letter — outside
# ----------------------------------------------------------------------

axB.text(

    -0.13,
    1.07,

    "B",

    transform=axB.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

categories_C = [

    "Overall rank\nSpearman",

    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(

    xC - widthC/2,

    internal_C,

    widthC,

    label="Internal test"
)


barsC2 = axC.bar(

    xC + widthC/2,

    kelm_C,

    widthC,

    label="KELM external"
)


axC.set_ylabel(
    "Agreement",
    fontsize=10.5,
    fontweight="bold",
    labelpad=5
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)


for label in axC.get_xticklabels():

    label.set_fontsize(8.2)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# C axis: exactly same scale/height as B
# ----------------------------------------------------------------------

axC.set_ylim(
    0,
    1.05
)

axC.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


# ----------------------------------------------------------------------
# C annotations
# ----------------------------------------------------------------------

for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.018,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8.5,
            fontweight="bold"
        )


# ----------------------------------------------------------------------
# C legend same location as B
# ----------------------------------------------------------------------

axC.legend(

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.015
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.6,

    borderaxespad=0
)


axC.tick_params(
    axis="both",
    labelsize=9,
    width=1.1,
    length=4
)


for label in axC.get_yticklabels():
    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# C letter
# ----------------------------------------------------------------------

axC.text(

    -0.13,
    1.07,

    "C",

    transform=axC.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D1 — INTERNAL TEST
#
# Same physical panel width/height as B
# ======================================================================

imD1 = axD1.imshow(

    internal_matrix.values,

    vmin=0,
    vmax=1,

    interpolation="nearest",

    # Fill exactly the subplot area
    aspect="auto"
)


axD1.set_xticks(
    np.arange(
        len(
            internal_matrix.columns
        )
    )
)

axD1.set_yticks(
    np.arange(
        len(
            internal_matrix.index
        )
    )
)


axD1.set_xticklabels(
    internal_matrix.columns
)

axD1.set_yticklabels(
    internal_matrix.index
)


for label in axD1.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD1.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=10
)


axD1.text(

    0.5,
    1.025,

    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


axD1.tick_params(
    axis="both",
    width=1.0,
    length=3
)


# ======================================================================
# PANEL D2 — KELM EXTERNAL
#
# Same physical panel width/height as C
# ======================================================================

imD2 = axD2.imshow(

    kelm_matrix.values,

    vmin=0,
    vmax=1,

    interpolation="nearest",

    aspect="auto"
)


axD2.set_xticks(
    np.arange(
        len(
            kelm_matrix.columns
        )
    )
)

axD2.set_yticks(
    np.arange(
        len(
            kelm_matrix.index
        )
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns
)

axD2.set_yticklabels(
    kelm_matrix.index
)


for label in axD2.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD2.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=10
)


axD2.text(

    0.5,
    1.025,

    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


axD2.tick_params(
    axis="both",
    width=1.0,
    length=3
)


# ======================================================================
# D LABEL
# ======================================================================

axD1.text(

    -0.13,
    1.07,

    "D",

    transform=axD1.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# SHARED D COLORBAR
#
# Add manually OUTSIDE the 2 × 2 grid.
# Therefore it does not shrink or distort D2.
# ======================================================================

posD2 = axD2.get_position()


cbar_ax = fig.add_axes(
    [
        posD2.x1 + 0.015,    # x
        posD2.y0,            # y
        0.012,               # width
        posD2.height         # EXACT same height as D
    ]
)


cbarD = fig.colorbar(
    imD2,
    cax=cbar_ax
)


cbarD.set_label(

    "Hotspot Jaccard",

    fontsize=9.5,
    fontweight="bold",

    labelpad=5
)


cbarD.ax.tick_params(
    labelsize=8.5,
    width=1
)


for label in cbarD.ax.get_yticklabels():

    label.set_fontweight("bold")


# ======================================================================
# SAVE
# ======================================================================

png_BD = (
    OUT_DIR
    / "Figure_3B-D_BALANCED_2x2.png"
)

pdf_BD = (
    OUT_DIR
    / "Figure_3B-D_BALANCED_2x2.pdf"
)

svg_BD = (
    OUT_DIR
    / "Figure_3B-D_BALANCED_2x2.svg"
)


fig.savefig(
    png_BD,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_BD,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_BD,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()

plt.close(fig)


print("\nFigure 3B-D generated:")
print(png_BD)
print(pdf_BD)
print(svg_BD)

In [ ]:
from google.colab import files

files.download(str(png_BD))

In [ ]:
!find /content/drive/MyDrive/pLM4CPP_XAI_2026 -type f | \
grep -Ei "residue.*enrichment|motif.*enrichment|replication|representative|figure_4" | \
sort

In [ ]:
# ======================================================================
# FIGURE 4 — PUBLICATION-QUALITY REGENERATION
#
# A = Replicated residue enrichment
# B = External replication of residue effects
# C = Replicated hotspot motifs
# D = Representative motif-centered hotspot maps
#
# Balanced 2×2 layout
# Larger fonts
# Compact whitespace
# Panel B uses broken-scale logic so extreme M does not dominate
# ======================================================================

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

CHECKPOINT_DIR = (
    PROJECT
    / "08_checkpoints"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RESIDUE_FILE = (
    TABLE_DIR
    / "residue_enrichment_replication.csv"
)

MOTIF_FILE = (
    TABLE_DIR
    / "hotspot_motif_replication.csv"
)

REP_RESIDUES_FILE = (
    CHECKPOINT_DIR
    / "figure_4_replicated_residues.csv"
)

REP_MOTIFS_FILE = (
    CHECKPOINT_DIR
    / "figure_4_replicated_motifs.csv"
)

REP_SEQS_FILE = (
    CHECKPOINT_DIR
    / "figure_4_representative_sequences.csv"
)


# ======================================================================
# VERIFY FILES
# ======================================================================

required_files = [
    RESIDUE_FILE,
    MOTIF_FILE,
    REP_RESIDUES_FILE,
    REP_MOTIFS_FILE,
    REP_SEQS_FILE
]

print("=" * 90)
print("FIGURE 4 SOURCE FILES")
print("=" * 90)

for f in required_files:

    if not f.exists():

        raise FileNotFoundError(
            f"Missing file:\n{f}"
        )

    print("✓", f)


# ======================================================================
# HELPERS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def find_col(df, candidates):

    cols = list(df.columns)

    # exact first
    for candidate in candidates:

        if candidate in cols:
            return candidate

    # partial second
    for candidate in candidates:

        for c in cols:

            if candidate in c:
                return c

    return None


def bold_ticks(
    ax,
    xsize=9,
    ysize=9
):

    ax.tick_params(
        axis="both",
        labelsize=max(
            xsize,
            ysize
        ),
        width=1.1,
        length=4
    )

    for label in ax.get_xticklabels():

        label.set_fontsize(
            xsize
        )

        label.set_fontweight(
            "bold"
        )

    for label in ax.get_yticklabels():

        label.set_fontsize(
            ysize
        )

        label.set_fontweight(
            "bold"
        )


def clean_spines(ax):

    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    ax.spines["left"].set_linewidth(
        1.1
    )

    ax.spines["bottom"].set_linewidth(
        1.1
    )


# ======================================================================
# LOAD DATA
# ======================================================================

residue_df = clean_columns(
    pd.read_csv(
        RESIDUE_FILE
    )
)

motif_df = clean_columns(
    pd.read_csv(
        MOTIF_FILE
    )
)

rep_residue_df = clean_columns(
    pd.read_csv(
        REP_RESIDUES_FILE
    )
)

rep_motif_df = clean_columns(
    pd.read_csv(
        REP_MOTIFS_FILE
    )
)

rep_seq_df = clean_columns(
    pd.read_csv(
        REP_SEQS_FILE
    )
)


print("\nResidue replication columns:")
print(
    residue_df.columns.tolist()
)

print("\nMotif replication columns:")
print(
    motif_df.columns.tolist()
)

print("\nRepresentative sequence columns:")
print(
    rep_seq_df.columns.tolist()
)


# ======================================================================
# RESIDUE TABLE COLUMN DETECTION
# ======================================================================

residue_col = find_col(
    residue_df,
    [
        "residue",
        "amino_acid",
        "aa"
    ]
)

internal_log2_col = find_col(
    residue_df,
    [
        "internal_log2_enrichment",
        "internal_log2_odds_ratio",
        "internal_log2"
    ]
)

external_log2_col = find_col(
    residue_df,
    [
        "kelm_log2_enrichment",
        "external_log2_enrichment",
        "kelm_log2_odds_ratio",
        "external_log2"
    ]
)

same_direction_col = find_col(
    residue_df,
    [
        "same_direction",
        "replicated_direction"
    ]
)

sig_both_col = find_col(
    residue_df,
    [
        "significant_in_both",
        "replicated",
        "significant_both"
    ]
)


if any(
    x is None
    for x in [
        residue_col,
        internal_log2_col,
        external_log2_col
    ]
):

    raise KeyError(
        "Could not detect residue enrichment columns.\n"
        f"Available:\n{residue_df.columns.tolist()}"
    )


# ======================================================================
# FILTER TO REPLICATED RESIDUES
# ======================================================================

res_plot = residue_df.copy()

if sig_both_col is not None:

    res_plot = res_plot[
        res_plot[sig_both_col]
        .astype(bool)
    ].copy()

elif same_direction_col is not None:

    res_plot = res_plot[
        res_plot[same_direction_col]
        .astype(bool)
    ].copy()


res_plot[internal_log2_col] = pd.to_numeric(
    res_plot[internal_log2_col],
    errors="coerce"
)

res_plot[external_log2_col] = pd.to_numeric(
    res_plot[external_log2_col],
    errors="coerce"
)

res_plot = res_plot.dropna(
    subset=[
        residue_col,
        internal_log2_col,
        external_log2_col
    ]
)


# Sort by internal effect
res_plot = (
    res_plot
    .sort_values(
        internal_log2_col
    )
    .reset_index(drop=True)
)


# ======================================================================
# MOTIF TABLE COLUMN DETECTION
# ======================================================================

motif_col = find_col(
    motif_df,
    [
        "motif",
        "hotspot_motif"
    ]
)

motif_internal_col = find_col(
    motif_df,
    [
        "internal_log2_enrichment",
        "internal_log2_odds_ratio",
        "internal_log2"
    ]
)

motif_external_col = find_col(
    motif_df,
    [
        "kelm_log2_enrichment",
        "external_log2_enrichment",
        "kelm_log2_odds_ratio",
        "external_log2"
    ]
)

motif_sig_col = find_col(
    motif_df,
    [
        "significant_in_both",
        "replicated",
        "significant_both"
    ]
)

internal_n_col = find_col(
    motif_df,
    [
        "internal_n",
        "internal_count",
        "n_internal"
    ]
)

external_n_col = find_col(
    motif_df,
    [
        "kelm_n",
        "external_n",
        "external_count",
        "n_external"
    ]
)


if any(
    x is None
    for x in [
        motif_col,
        motif_internal_col,
        motif_external_col
    ]
):

    raise KeyError(
        "Could not detect motif enrichment columns.\n"
        f"Available:\n{motif_df.columns.tolist()}"
    )


motif_plot = motif_df.copy()

if motif_sig_col is not None:

    motif_plot = motif_plot[
        motif_plot[motif_sig_col]
        .astype(bool)
    ].copy()


motif_plot[motif_internal_col] = pd.to_numeric(
    motif_plot[motif_internal_col],
    errors="coerce"
)

motif_plot[motif_external_col] = pd.to_numeric(
    motif_plot[motif_external_col],
    errors="coerce"
)

motif_plot = motif_plot.dropna(
    subset=[
        motif_col,
        motif_internal_col,
        motif_external_col
    ]
)

motif_plot = (
    motif_plot
    .sort_values(
        motif_internal_col
    )
    .reset_index(drop=True)
)


# ======================================================================
# REPRESENTATIVE SEQUENCES FOR PANEL D
# ======================================================================

seq_id_col = find_col(
    rep_seq_df,
    [
        "sequence_id",
        "id",
        "seq_id"
    ]
)

seq_col = find_col(
    rep_seq_df,
    [
        "sequence",
        "peptide_sequence",
        "seq"
    ]
)

seq_motif_col = find_col(
    rep_seq_df,
    [
        "motif",
        "representative_motif"
    ]
)


if seq_id_col is None or seq_col is None:

    raise KeyError(
        "Could not detect representative sequence columns.\n"
        f"{rep_seq_df.columns.tolist()}"
    )


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE LAYOUT
#
# Balanced 2 × 2
# ======================================================================

fig = plt.figure(
    figsize=(14.5, 9.0)
)

gs = fig.add_gridspec(
    2,
    2,

    width_ratios=[
        1,
        1
    ],

    height_ratios=[
        1,
        1
    ],

    left=0.08,
    right=0.97,

    bottom=0.08,
    top=0.94,

    wspace=0.28,
    hspace=0.35
)


axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANEL A — REPLICATED RESIDUE ENRICHMENT
# ======================================================================

yA = np.arange(
    len(res_plot)
)


for i, row in res_plot.iterrows():

    x1 = row[internal_log2_col]
    x2 = row[external_log2_col]

    axA.plot(
        [
            x1,
            x2
        ],
        [
            i,
            i
        ],
        linewidth=1.2,
        alpha=0.75
    )

    axA.scatter(
        x1,
        i,
        s=38,
        marker="o",
        edgecolor="black",
        linewidth=0.5,
        zorder=3
    )

    axA.scatter(
        x2,
        i,
        s=38,
        marker="s",
        edgecolor="black",
        linewidth=0.5,
        zorder=3
    )


axA.axvline(
    0,
    linestyle="--",
    linewidth=1,
    alpha=0.7
)


axA.set_yticks(
    yA
)

axA.set_yticklabels(
    res_plot[residue_col]
    .astype(str)
)


axA.set_xlabel(
    "log$_2$(enrichment)"
)

axA.set_ylabel(
    "Amino-acid residue"
)


# Direction labels
xmin, xmax = axA.get_xlim()

axA.text(
    xmin,
    -0.65,
    "Depleted",
    ha="left",
    va="top",
    fontsize=8,
    fontweight="bold"
)

axA.text(
    xmax,
    -0.65,
    "Enriched",
    ha="right",
    va="top",
    fontsize=8,
    fontweight="bold"
)


legend_handles = [

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markeredgecolor="black",
        label="Internal test"
    ),

    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="none",
        markeredgecolor="black",
        label="KELM external"
    )
]


axA.legend(
    handles=legend_handles,
    frameon=False,
    loc="lower right"
)


bold_ticks(
    axA,
    xsize=9,
    ysize=10
)

clean_spines(
    axA
)


axA.text(
    -0.12,
    1.05,
    "A",
    transform=axA.transAxes,
    fontsize=20,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL B — EXTERNAL REPLICATION
#
# Main cluster zoomed; extreme values retained as annotated outliers
# ======================================================================

xB = res_plot[
    internal_log2_col
].to_numpy()

yB = res_plot[
    external_log2_col
].to_numpy()

labelsB = res_plot[
    residue_col
].astype(str).to_numpy()


# ----------------------------------------------------------------------
# Define display range from non-extreme residues
# ----------------------------------------------------------------------

finite_mask = (
    np.isfinite(xB)
    &
    np.isfinite(yB)
)

xB = xB[finite_mask]
yB = yB[finite_mask]
labelsB = labelsB[finite_mask]


# robust range
x_low = max(
    np.percentile(
        xB,
        5
    ) - 1,
    -8
)

x_high = min(
    np.percentile(
        xB,
        95
    ) + 1,
    3
)

y_low = max(
    np.percentile(
        yB,
        10
    ) - 1.5,
    -8
)

y_high = min(
    np.percentile(
        yB,
        95
    ) + 1,
    3
)


# Ensure zero visible
x_high = max(
    x_high,
    1
)

y_high = max(
    y_high,
    1
)


# identify extreme points
extreme_mask = (
    (xB < x_low)
    |
    (xB > x_high)
    |
    (yB < y_low)
    |
    (yB > y_high)
)


normal_mask = ~extreme_mask


# ----------------------------------------------------------------------
# Plot normal points
# ----------------------------------------------------------------------

axB.scatter(
    xB[normal_mask],
    yB[normal_mask],
    s=48,
    edgecolor="black",
    linewidth=0.6,
    zorder=3
)


# ----------------------------------------------------------------------
# Identity line
# ----------------------------------------------------------------------

diag_low = min(
    x_low,
    y_low
)

diag_high = max(
    x_high,
    y_high
)

axB.plot(
    [
        diag_low,
        diag_high
    ],
    [
        diag_low,
        diag_high
    ],
    linestyle="--",
    linewidth=1.2
)


axB.axhline(
    0,
    linestyle=":",
    linewidth=1
)

axB.axvline(
    0,
    linestyle=":",
    linewidth=1
)


# labels
for x, y, aa in zip(
    xB[normal_mask],
    yB[normal_mask],
    labelsB[normal_mask]
):

    axB.text(
        x + 0.10,
        y + 0.10,
        aa,
        fontsize=8.5,
        fontweight="bold"
    )


# ----------------------------------------------------------------------
# Add extreme residues as compact annotation
# ----------------------------------------------------------------------

if np.any(
    extreme_mask
):

    extreme_text = []

    for x, y, aa in zip(
        xB[extreme_mask],
        yB[extreme_mask],
        labelsB[extreme_mask]
    ):

        extreme_text.append(
            f"{aa}: ({x:.1f}, {y:.1f})"
        )

    axB.text(
        0.03,
        0.04,

        "Extreme replicated effect:\n"
        + "\n".join(
            extreme_text
        ),

        transform=axB.transAxes,

        ha="left",
        va="bottom",

        fontsize=8,
        fontweight="bold",

        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="0.6",
            alpha=0.9
        )
    )


axB.set_xlim(
    x_low,
    x_high
)

axB.set_ylim(
    y_low,
    y_high
)


axB.set_xlabel(
    "Internal log$_2$ enrichment"
)

axB.set_ylabel(
    "KELM log$_2$ enrichment"
)


bold_ticks(
    axB,
    xsize=9,
    ysize=9
)

clean_spines(
    axB
)


axB.text(
    -0.12,
    1.05,
    "B",
    transform=axB.transAxes,
    fontsize=20,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL C — REPLICATED MOTIFS
# ======================================================================

yC = np.arange(
    len(motif_plot)
)


for i, row in motif_plot.iterrows():

    x1 = row[
        motif_internal_col
    ]

    x2 = row[
        motif_external_col
    ]

    axC.plot(
        [
            x1,
            x2
        ],
        [
            i,
            i
        ],
        linewidth=1.2,
        alpha=0.75
    )

    axC.scatter(
        x1,
        i,
        s=40,
        marker="o",
        edgecolor="black",
        linewidth=0.5,
        zorder=3
    )

    axC.scatter(
        x2,
        i,
        s=40,
        marker="s",
        edgecolor="black",
        linewidth=0.5,
        zorder=3
    )


axC.set_yticks(
    yC
)

axC.set_yticklabels(
    motif_plot[
        motif_col
    ].astype(str)
)


axC.set_xlabel(
    "log$_2$(enrichment)"
)

axC.set_ylabel(
    "Replicated hotspot motif"
)


# counts at right if available
if (
    internal_n_col is not None
    and external_n_col is not None
):

    x_right = axC.get_xlim()[1]

    for i, row in motif_plot.iterrows():

        try:

            ni = int(
                row[internal_n_col]
            )

            ne = int(
                row[external_n_col]
            )

            text = (
                f"n={ni}/{ne}"
            )

        except:

            continue

        axC.text(
            x_right,
            i,
            text,
            fontsize=7.5,
            fontweight="bold",
            ha="right",
            va="center"
        )


axC.legend(
    handles=legend_handles,
    frameon=False,
    loc="lower right"
)


bold_ticks(
    axC,
    xsize=9,
    ysize=10
)

clean_spines(
    axC
)


axC.text(
    -0.12,
    1.05,
    "C",
    transform=axC.transAxes,
    fontsize=20,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL D — REPRESENTATIVE MOTIF-CENTERED HOTSPOTS
#
# Enlarged residue boxes
# ======================================================================

axD.set_axis_off()


# ----------------------------------------------------------------------
# Panel D letter
# ----------------------------------------------------------------------

axD.text(
    -0.06,
    1.03,
    "D",
    transform=axD.transAxes,
    fontsize=20,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ----------------------------------------------------------------------
# Use first three representative sequences
# ----------------------------------------------------------------------

rep_show = (
    rep_seq_df
    .head(3)
    .copy()
)


# vertical positions
y_positions = [
    0.78,
    0.50,
    0.22
]


for row_idx, (_, row) in enumerate(
    rep_show.iterrows()
):

    seq_id = str(
        row[
            seq_id_col
        ]
    )

    seq = str(
        row[
            seq_col
        ]
    )

    motif = (
        str(
            row[
                seq_motif_col
            ]
        )
        if seq_motif_col is not None
        else ""
    )


    y0 = y_positions[
        row_idx
    ]


    # --------------------------------------------------------------
    # Header
    # --------------------------------------------------------------

    axD.text(
        0.02,
        y0 + 0.10,

        f"{seq_id} — motif {motif}",

        transform=axD.transAxes,

        fontsize=9,
        fontweight="bold",

        ha="left",
        va="center"
    )


    # --------------------------------------------------------------
    # Determine available box width dynamically
    # --------------------------------------------------------------

    n = len(seq)

    total_width = 0.92

    box_w = min(
        0.034,
        total_width / max(
            n,
            1
        )
    )

    box_h = 0.060

    x_start = 0.02


    # motif locations
    motif_positions = []

    if motif and motif != "nan":

        start = 0

        while True:

            idx = seq.find(
                motif,
                start
            )

            if idx == -1:
                break

            motif_positions.extend(
                range(
                    idx,
                    idx + len(motif)
                )
            )

            start = idx + 1


    motif_positions = set(
        motif_positions
    )


    # --------------------------------------------------------------
    # Residue boxes
    # --------------------------------------------------------------

    for j, aa in enumerate(seq):

        x0 = (
            x_start
            + j * box_w
        )

        is_motif = (
            j in motif_positions
        )

        face = (
            "#e57373"
            if is_motif
            else "#dddddd"
        )

        rect = Rectangle(
            (
                x0,
                y0
            ),
            box_w * 0.95,
            box_h,

            transform=axD.transAxes,

            facecolor=face,

            edgecolor="black",

            linewidth=0.6
        )

        axD.add_patch(
            rect
        )


        axD.text(
            x0 + box_w * 0.475,
            y0 + box_h/2,

            aa,

            transform=axD.transAxes,

            fontsize=6.5,
            fontweight="bold",

            ha="center",
            va="center"
        )


    # --------------------------------------------------------------
    # Motif marker underneath
    # --------------------------------------------------------------

    if motif_positions:

        for j in motif_positions:

            x0 = (
                x_start
                + j * box_w
            )

            axD.text(
                x0 + box_w*0.475,
                y0 - 0.025,
                "★",

                transform=axD.transAxes,

                fontsize=7,
                ha="center",
                va="center"
            )


# ----------------------------------------------------------------------
# Panel D legend
# ----------------------------------------------------------------------

legend_D = [

    Rectangle(
        (0, 0),
        1,
        1,
        facecolor="#dddddd",
        edgecolor="black",
        label="Other residue"
    ),

    Rectangle(
        (0, 0),
        1,
        1,
        facecolor="#e57373",
        edgecolor="black",
        label="Replicated motif"
    ),

    Line2D(
        [0],
        [0],
        marker="*",
        linestyle="none",
        color="black",
        markersize=8,
        label="Motif position"
    )
]


axD.legend(
    handles=legend_D,

    frameon=False,

    loc="lower left",

    bbox_to_anchor=(
        0.02,
        0.00
    ),

    ncol=3,

    fontsize=8
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_4_Replicated_Residue_and_Motif_Grammar_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_4_Replicated_Residue_and_Motif_Grammar_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_4_Replicated_Residue_and_Motif_Grammar_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.06
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.06
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 4 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 4 — MANUSCRIPT-QUALITY FINAL LAYOUT
#
# TOP ROW:
# A = replicated residue effects
# B = internal vs external replication
# C = replicated motif effects
#
# BOTTOM:
# D = representative motif-centered hotspot maps
#
# A, B, C in one line
# D spans full width
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
import pandas as pd


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "lines.linewidth": 1.4,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(16.0, 8.3)
)


outer = gridspec.GridSpec(
    2,
    1,
    figure=fig,

    height_ratios=[
        1.00,   # A B C
        0.95    # D
    ],

    hspace=0.38
)


# ======================================================================
# TOP ROW — A B C
# ======================================================================

top = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[0],

    width_ratios=[
        1.05,
        1.00,
        1.05
    ],

    wspace=0.34
)


axA = fig.add_subplot(
    top[0]
)

axB = fig.add_subplot(
    top[1]
)

axC = fig.add_subplot(
    top[2]
)


# ======================================================================
# COMMON LEGEND
# ======================================================================

legend_handles = [

    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="none",

        markersize=6,

        markerfacecolor="tab:blue",
        markeredgecolor="black",

        label="Internal test"
    ),

    Line2D(
        [0],
        [0],

        marker="s",
        linestyle="none",

        markersize=6,

        markerfacecolor="tab:orange",
        markeredgecolor="black",

        label="KELM external"
    )
]


# ======================================================================
# PANEL A — RESIDUE EFFECT REPLICATION
#
# Publication-style paired forest/dumbbell plot
# ======================================================================

# Sort from strongest enrichment to strongest depletion
res_A = (
    res_plot
    .sort_values(
        internal_log2_col,
        ascending=True
    )
    .reset_index(drop=True)
)


yA = np.arange(
    len(res_A)
)


for i, row in res_A.iterrows():

    internal_value = float(
        row[internal_log2_col]
    )

    external_value = float(
        row[external_log2_col]
    )


    # connector
    axA.plot(
        [
            internal_value,
            external_value
        ],
        [
            i,
            i
        ],

        color="0.65",

        linewidth=1.2,

        zorder=1
    )


    # internal
    axA.scatter(
        internal_value,
        i,

        s=48,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.6,

        zorder=3
    )


    # KELM
    axA.scatter(
        external_value,
        i,

        s=48,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.6,

        zorder=3
    )


# zero reference
axA.axvline(
    0,

    color="0.45",

    linestyle="--",

    linewidth=1.0
)


axA.set_yticks(
    yA
)

axA.set_yticklabels(
    res_A[residue_col]
    .astype(str)
)


axA.set_xlabel(
    "log$_2$ enrichment"
)

axA.set_ylabel(
    "Amino-acid residue"
)


# bold ticks
for label in axA.get_xticklabels():
    label.set_fontweight("bold")

for label in axA.get_yticklabels():
    label.set_fontweight("bold")


axA.tick_params(
    width=1.1,
    length=4
)


# clean frame
axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


# legend
axA.legend(
    handles=legend_handles,

    frameon=False,

    loc="upper left",

    fontsize=8,

    handletextpad=0.4,

    borderaxespad=0.2
)


# Direction text
xmin_A, xmax_A = axA.get_xlim()

axA.text(
    0.02,
    0.02,

    "Depleted",

    transform=axA.transAxes,

    fontsize=8,
    fontweight="bold",

    ha="left",
    va="bottom"
)

axA.text(
    0.98,
    0.02,

    "Enriched",

    transform=axA.transAxes,

    fontsize=8,
    fontweight="bold",

    ha="right",
    va="bottom"
)


# Panel letter
axA.text(
    -0.16,
    1.06,

    "A",

    transform=axA.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL B — EXTERNAL REPLICATION
#
# Main scatter focuses on replicated cluster.
# Extreme point retained in inset.
# ======================================================================

xB = res_plot[
    internal_log2_col
].astype(float).to_numpy()

yB = res_plot[
    external_log2_col
].astype(float).to_numpy()

labelsB = (
    res_plot[
        residue_col
    ]
    .astype(str)
    .to_numpy()
)


# ----------------------------------------------------------------------
# Identify most extreme observation
# ----------------------------------------------------------------------

distance_from_center = (
    np.abs(xB)
    + np.abs(yB)
)

extreme_index = np.argmax(
    distance_from_center
)

normal_mask = np.ones(
    len(xB),
    dtype=bool
)

normal_mask[
    extreme_index
] = False


# ----------------------------------------------------------------------
# Plot normal replicated residues
# ----------------------------------------------------------------------

axB.scatter(
    xB[normal_mask],
    yB[normal_mask],

    s=52,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.6,

    zorder=3
)


# ----------------------------------------------------------------------
# Labels
# ----------------------------------------------------------------------

for x, y, aa in zip(
    xB[normal_mask],
    yB[normal_mask],
    labelsB[normal_mask]
):

    axB.annotate(
        aa,

        xy=(
            x,
            y
        ),

        xytext=(
            4,
            4
        ),

        textcoords="offset points",

        fontsize=8.5,

        fontweight="bold"
    )


# ----------------------------------------------------------------------
# Determine sensible zoom range
# ----------------------------------------------------------------------

normal_x = xB[
    normal_mask
]

normal_y = yB[
    normal_mask
]


xmin = min(
    normal_x.min(),
    normal_y.min()
)

xmax = max(
    normal_x.max(),
    normal_y.max()
)


padding = max(
    0.6,
    0.12 * (
        xmax - xmin
    )
)


plot_min = xmin - padding
plot_max = xmax + padding


# Ensure zero visible
plot_max = max(
    plot_max,
    0.8
)


axB.set_xlim(
    plot_min,
    plot_max
)

axB.set_ylim(
    plot_min,
    plot_max
)


# ----------------------------------------------------------------------
# Identity line
# ----------------------------------------------------------------------

axB.plot(
    [
        plot_min,
        plot_max
    ],
    [
        plot_min,
        plot_max
    ],

    linestyle="--",

    color="0.45",

    linewidth=1.2,

    zorder=1
)


axB.axhline(
    0,

    linestyle=":",

    color="0.65",

    linewidth=1
)

axB.axvline(
    0,

    linestyle=":",

    color="0.65",

    linewidth=1
)


axB.set_xlabel(
    "Internal log$_2$ enrichment"
)

axB.set_ylabel(
    "KELM log$_2$ enrichment"
)


for label in axB.get_xticklabels():
    label.set_fontweight("bold")

for label in axB.get_yticklabels():
    label.set_fontweight("bold")


axB.tick_params(
    width=1.1,
    length=4
)


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# Extreme-effect inset
# ----------------------------------------------------------------------

inset = inset_axes(
    axB,

    width="34%",
    height="34%",

    loc="lower left",

    borderpad=1.2
)


extreme_x = xB[
    extreme_index
]

extreme_y = yB[
    extreme_index
]

extreme_label = labelsB[
    extreme_index
]


inset.scatter(
    extreme_x,
    extreme_y,

    s=42,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.6
)


inset.annotate(
    extreme_label,

    xy=(
        extreme_x,
        extreme_y
    ),

    xytext=(
        5,
        4
    ),

    textcoords="offset points",

    fontsize=8,

    fontweight="bold"
)


extreme_min = min(
    extreme_x,
    extreme_y
)

extreme_max = max(
    extreme_x,
    extreme_y
)


pad_extreme = max(
    1,
    (
        extreme_max
        - extreme_min
    )
    * 0.3
)


inset.set_xlim(
    extreme_min - pad_extreme,
    extreme_max + pad_extreme
)

inset.set_ylim(
    extreme_min - pad_extreme,
    extreme_max + pad_extreme
)


inset.plot(
    [
        extreme_min - pad_extreme,
        extreme_max + pad_extreme
    ],
    [
        extreme_min - pad_extreme,
        extreme_max + pad_extreme
    ],

    linestyle="--",

    color="0.5",

    linewidth=0.8
)


inset.tick_params(
    labelsize=6.5,
    width=0.8,
    length=2.5
)


for label in inset.get_xticklabels():
    label.set_fontweight("bold")

for label in inset.get_yticklabels():
    label.set_fontweight("bold")


inset.set_title(
    "Extreme effect",

    fontsize=7.5,

    fontweight="bold",

    pad=2
)


# Panel letter
axB.text(
    -0.16,
    1.06,

    "B",

    transform=axB.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C — REPLICATED MOTIF EFFECTS
#
# Same visual grammar as A
# ======================================================================

motif_C = (
    motif_plot
    .sort_values(
        motif_internal_col,
        ascending=True
    )
    .reset_index(drop=True)
)


yC = np.arange(
    len(motif_C)
)


for i, row in motif_C.iterrows():

    internal_value = float(
        row[motif_internal_col]
    )

    external_value = float(
        row[motif_external_col]
    )


    axC.plot(
        [
            internal_value,
            external_value
        ],
        [
            i,
            i
        ],

        color="0.65",

        linewidth=1.2,

        zorder=1
    )


    axC.scatter(
        internal_value,
        i,

        s=48,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.6,

        zorder=3
    )


    axC.scatter(
        external_value,
        i,

        s=48,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.6,

        zorder=3
    )


axC.set_yticks(
    yC
)

axC.set_yticklabels(
    motif_C[
        motif_col
    ].astype(str)
)


axC.set_xlabel(
    "log$_2$ enrichment"
)

axC.set_ylabel(
    "Replicated hotspot motif"
)


for label in axC.get_xticklabels():
    label.set_fontweight("bold")

for label in axC.get_yticklabels():
    label.set_fontweight("bold")


axC.tick_params(
    width=1.1,
    length=4
)


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# Add n internal / KELM
# ----------------------------------------------------------------------

if (
    internal_n_col is not None
    and external_n_col is not None
):

    right_lim = axC.get_xlim()[1]

    span = (
        axC.get_xlim()[1]
        - axC.get_xlim()[0]
    )


    axC.set_xlim(
        axC.get_xlim()[0],
        right_lim + 0.18 * span
    )


    text_x = (
        right_lim
        + 0.04 * span
    )


    for i, row in motif_C.iterrows():

        try:

            ni = int(
                row[internal_n_col]
            )

            ne = int(
                row[external_n_col]
            )

        except:
            continue


        axC.text(
            text_x,
            i,

            f"n={ni}/{ne}",

            fontsize=7.5,

            fontweight="bold",

            ha="left",
            va="center"
        )


axC.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower right",

    fontsize=8
)


# Panel letter
axC.text(
    -0.16,
    1.06,

    "C",

    transform=axC.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D — FULL-WIDTH REPRESENTATIVE SEQUENCE MAPS
# ======================================================================

axD = fig.add_subplot(
    outer[1]
)

axD.set_axis_off()


# Panel letter
axD.text(
    -0.035,
    1.03,

    "D",

    transform=axD.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# REPRESENTATIVE SEQUENCES
# ======================================================================

rep_show = (
    rep_seq_df
    .head(3)
    .copy()
)


# More vertical separation
y_positions = [
    0.72,
    0.43,
    0.14
]


for row_idx, (_, row) in enumerate(
    rep_show.iterrows()
):

    seq_id = str(
        row[
            seq_id_col
        ]
    )

    seq = str(
        row[
            seq_col
        ]
    )


    motif = (
        str(
            row[
                seq_motif_col
            ]
        )

        if seq_motif_col is not None

        else ""
    )


    y0 = y_positions[
        row_idx
    ]


    # --------------------------------------------------------------
    # Sequence title
    # --------------------------------------------------------------

    axD.text(
        0.04,
        y0 + 0.085,

        f"{seq_id}  —  motif {motif}",

        transform=axD.transAxes,

        fontsize=9.5,

        fontweight="bold",

        ha="left",
        va="center"
    )


    # --------------------------------------------------------------
    # Dynamic box width
    # --------------------------------------------------------------

    n = len(seq)

    available_width = 0.90

    box_w = min(
        0.030,
        available_width / max(
            n,
            1
        )
    )

    box_h = 0.075

    x_start = 0.04


    # --------------------------------------------------------------
    # Find motif residues
    # --------------------------------------------------------------

    motif_positions = []


    if (
        motif
        and motif.lower() != "nan"
    ):

        start = 0

        while True:

            idx = seq.find(
                motif,
                start
            )

            if idx == -1:
                break


            motif_positions.extend(
                range(
                    idx,
                    idx + len(motif)
                )
            )


            start = idx + 1


    motif_positions = set(
        motif_positions
    )


    # --------------------------------------------------------------
    # Draw residues
    # --------------------------------------------------------------

    for j, aa in enumerate(seq):

        x0 = (
            x_start
            + j * box_w
        )


        is_motif = (
            j in motif_positions
        )


        face = (
            "#e57373"
            if is_motif
            else "#e6e6e6"
        )


        rect = Rectangle(
            (
                x0,
                y0
            ),

            box_w * 0.94,
            box_h,

            transform=axD.transAxes,

            facecolor=face,

            edgecolor="black",

            linewidth=0.65
        )


        axD.add_patch(
            rect
        )


        axD.text(
            x0
            + box_w * 0.47,

            y0
            + box_h / 2,

            aa,

            transform=axD.transAxes,

            fontsize=7,

            fontweight="bold",

            ha="center",
            va="center"
        )


    # --------------------------------------------------------------
    # Motif star markers
    # --------------------------------------------------------------

    for j in motif_positions:

        x0 = (
            x_start
            + j * box_w
        )


        axD.text(
            x0
            + box_w * 0.47,

            y0 - 0.025,

            "★",

            transform=axD.transAxes,

            fontsize=6.5,

            ha="center",
            va="center"
        )


# ======================================================================
# D LEGEND
# ======================================================================

legend_D = [

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e6e6e6",

        edgecolor="black",

        label="Other residue"
    ),

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e57373",

        edgecolor="black",

        label="Replicated motif"
    ),

    Line2D(
        [0],
        [0],

        marker="*",

        linestyle="none",

        color="black",

        markersize=8,

        label="Motif position"
    )
]


axD.legend(
    handles=legend_D,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.5,
        -0.03
    ),

    ncol=3,

    fontsize=8.5,

    columnspacing=2
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_MS_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_MS_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_MS_FINAL.svg"
)


fig.savefig(
    png_file,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.06
)


fig.savefig(
    pdf_file,

    bbox_inches="tight",

    pad_inches=0.06
)


fig.savefig(
    svg_file,

    bbox_inches="tight",

    pad_inches=0.06
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 4 — MANUSCRIPT FINAL GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 4 — IMPROVED MANUSCRIPT VERSION
#
# A, B, C = one row
# D = full-width below
#
# Larger fonts
# No overlapping labels
# No n=0/0 text
# Enlarged D residue maps
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np


# ======================================================================
# GLOBAL STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 12,

    "axes.labelsize": 13,
    "axes.labelweight": "bold",

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,

    "legend.fontsize": 10,

    "axes.linewidth": 1.3,

    "lines.linewidth": 1.5,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(18, 10.5)
)

outer = gridspec.GridSpec(
    2,
    1,

    figure=fig,

    height_ratios=[
        1.0,    # A B C
        1.05    # D
    ],

    hspace=0.42
)


# ======================================================================
# TOP ROW — A B C
# ======================================================================

top = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[0],

    width_ratios=[
        1.05,
        1.05,
        1.15
    ],

    wspace=0.34
)


axA = fig.add_subplot(
    top[0]
)

axB = fig.add_subplot(
    top[1]
)

axC = fig.add_subplot(
    top[2]
)


# ======================================================================
# COMMON LEGEND
# ======================================================================

legend_handles = [

    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="none",

        markerfacecolor="tab:blue",
        markeredgecolor="black",

        markersize=7,

        label="Internal test"
    ),

    Line2D(
        [0],
        [0],

        marker="s",
        linestyle="none",

        markerfacecolor="tab:orange",
        markeredgecolor="black",

        markersize=7,

        label="KELM external"
    )
]


# ======================================================================
# PANEL A — REPLICATED RESIDUE ENRICHMENT
# ======================================================================

res_A = (
    res_plot
    .sort_values(
        internal_log2_col,
        ascending=True
    )
    .reset_index(drop=True)
)


yA = np.arange(
    len(res_A)
)


for i, row in res_A.iterrows():

    internal_value = float(
        row[internal_log2_col]
    )

    external_value = float(
        row[external_log2_col]
    )


    # connector
    axA.plot(
        [
            internal_value,
            external_value
        ],
        [
            i,
            i
        ],

        color="0.68",

        linewidth=1.4,

        zorder=1
    )


    axA.scatter(
        internal_value,
        i,

        s=58,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


    axA.scatter(
        external_value,
        i,

        s=58,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


# zero line
axA.axvline(
    0,

    linestyle="--",

    color="0.45",

    linewidth=1.2
)


axA.set_yticks(
    yA
)

axA.set_yticklabels(
    res_A[
        residue_col
    ].astype(str)
)


axA.set_xlabel(
    "log$_2$ enrichment"
)

axA.set_ylabel(
    "Amino-acid residue"
)


# larger bold ticks
for label in axA.get_xticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


for label in axA.get_yticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


axA.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# Legend ABOVE A
# ----------------------------------------------------------------------

axA.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.02
    ),

    ncol=2,

    fontsize=10,

    handletextpad=0.4,

    columnspacing=1.2
)


# ----------------------------------------------------------------------
# Direction labels
# ----------------------------------------------------------------------

axA.text(
    0.02,
    0.015,

    "Depleted",

    transform=axA.transAxes,

    fontsize=9.5,
    fontweight="bold",

    ha="left",
    va="bottom"
)

axA.text(
    0.98,
    0.015,

    "Enriched",

    transform=axA.transAxes,

    fontsize=9.5,
    fontweight="bold",

    ha="right",
    va="bottom"
)


axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


# Panel A
axA.text(
    -0.14,
    1.08,

    "A",

    transform=axA.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL B — EXTERNAL REPLICATION
# ======================================================================

xB = (
    res_plot[
        internal_log2_col
    ]
    .astype(float)
    .to_numpy()
)

yB = (
    res_plot[
        external_log2_col
    ]
    .astype(float)
    .to_numpy()
)

labelsB = (
    res_plot[
        residue_col
    ]
    .astype(str)
    .to_numpy()
)


# ----------------------------------------------------------------------
# Identify extreme observation
# ----------------------------------------------------------------------

distance_from_center = (
    np.abs(xB)
    + np.abs(yB)
)

extreme_index = np.argmax(
    distance_from_center
)

normal_mask = np.ones(
    len(xB),
    dtype=bool
)

normal_mask[
    extreme_index
] = False


normal_x = xB[
    normal_mask
]

normal_y = yB[
    normal_mask
]


# ----------------------------------------------------------------------
# Main points
# ----------------------------------------------------------------------

axB.scatter(
    normal_x,
    normal_y,

    s=64,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7,

    zorder=3
)


# ----------------------------------------------------------------------
# Labels with offset to avoid point overlap
# ----------------------------------------------------------------------

for x, y, aa in zip(
    normal_x,
    normal_y,
    labelsB[normal_mask]
):

    axB.annotate(
        aa,

        xy=(
            x,
            y
        ),

        xytext=(
            7,
            5
        ),

        textcoords="offset points",

        fontsize=10,
        fontweight="bold"
    )


# ----------------------------------------------------------------------
# Balanced plotting limits
# ----------------------------------------------------------------------

all_normal = np.concatenate(
    [
        normal_x,
        normal_y
    ]
)

plot_min = (
    np.min(all_normal)
    - 0.8
)

plot_max = (
    np.max(all_normal)
    + 0.8
)

plot_max = max(
    plot_max,
    0.8
)


axB.set_xlim(
    plot_min,
    plot_max
)

axB.set_ylim(
    plot_min,
    plot_max
)


# identity
axB.plot(
    [
        plot_min,
        plot_max
    ],
    [
        plot_min,
        plot_max
    ],

    linestyle="--",

    color="0.45",

    linewidth=1.3
)


axB.axhline(
    0,

    linestyle=":",

    color="0.65",

    linewidth=1
)

axB.axvline(
    0,

    linestyle=":",

    color="0.65",

    linewidth=1
)


axB.set_xlabel(
    "Internal log$_2$ enrichment"
)

axB.set_ylabel(
    "KELM log$_2$ enrichment"
)


for label in axB.get_xticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


for label in axB.get_yticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


axB.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# Extreme-effect inset — moved to upper left
# ----------------------------------------------------------------------

inset = inset_axes(
    axB,

    width="34%",
    height="34%",

    loc="upper left",

    borderpad=1.3
)


extreme_x = xB[
    extreme_index
]

extreme_y = yB[
    extreme_index
]

extreme_label = labelsB[
    extreme_index
]


inset.scatter(
    extreme_x,
    extreme_y,

    s=48,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7
)


inset.annotate(
    extreme_label,

    xy=(
        extreme_x,
        extreme_y
    ),

    xytext=(
        6,
        5
    ),

    textcoords="offset points",

    fontsize=9,

    fontweight="bold"
)


extreme_min = min(
    extreme_x,
    extreme_y
)

extreme_max = max(
    extreme_x,
    extreme_y
)

extreme_pad = max(
    1.5,
    0.20
    * (
        extreme_max
        - extreme_min
    )
)


inset.set_xlim(
    extreme_min - extreme_pad,
    extreme_max + extreme_pad
)

inset.set_ylim(
    extreme_min - extreme_pad,
    extreme_max + extreme_pad
)


inset.plot(
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],

    linestyle="--",

    color="0.5",

    linewidth=0.8
)


inset.set_title(
    "Extreme effect",

    fontsize=9,
    fontweight="bold",

    pad=3
)


inset.tick_params(
    labelsize=7.5,
    width=0.8,
    length=3
)


for label in inset.get_xticklabels():

    label.set_fontweight("bold")


for label in inset.get_yticklabels():

    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# Panel B
axB.text(
    -0.14,
    1.08,

    "B",

    transform=axB.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C — REPLICATED MOTIFS
# ======================================================================

motif_C = (
    motif_plot
    .sort_values(
        motif_internal_col,
        ascending=True
    )
    .reset_index(drop=True)
)


yC = np.arange(
    len(motif_C)
)


for i, row in motif_C.iterrows():

    internal_value = float(
        row[motif_internal_col]
    )

    external_value = float(
        row[motif_external_col]
    )


    axC.plot(
        [
            internal_value,
            external_value
        ],
        [
            i,
            i
        ],

        color="0.68",

        linewidth=1.4,

        zorder=1
    )


    axC.scatter(
        internal_value,
        i,

        s=58,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


    axC.scatter(
        external_value,
        i,

        s=58,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axC.set_yticks(
    yC
)

axC.set_yticklabels(
    motif_C[
        motif_col
    ].astype(str)
)


axC.set_xlabel(
    "log$_2$ enrichment"
)

axC.set_ylabel(
    "Replicated hotspot motif"
)


# no n=0/0 labels
# intentionally removed


for label in axC.get_xticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


for label in axC.get_yticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


axC.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# Legend ABOVE C
# ----------------------------------------------------------------------

axC.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.02
    ),

    ncol=2,

    fontsize=10,

    handletextpad=0.4,

    columnspacing=1.2
)


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# Panel C
axC.text(
    -0.14,
    1.08,

    "C",

    transform=axC.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D — FULL WIDTH, LARGE RESIDUE MAPS
# ======================================================================

axD = fig.add_subplot(
    outer[1]
)

axD.set_axis_off()


# ----------------------------------------------------------------------
# Panel D label
# ----------------------------------------------------------------------

axD.text(
    -0.025,
    1.02,

    "D",

    transform=axD.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# REPRESENTATIVE SEQUENCES
# ======================================================================

rep_show = (
    rep_seq_df
    .head(3)
    .copy()
)


# More separation vertically
y_positions = [
    0.72,
    0.43,
    0.14
]


for row_idx, (_, row) in enumerate(
    rep_show.iterrows()
):

    seq_id = str(
        row[
            seq_id_col
        ]
    )

    seq = str(
        row[
            seq_col
        ]
    )

    motif = (
        str(
            row[
                seq_motif_col
            ]
        )

        if seq_motif_col is not None

        else ""
    )


    y0 = y_positions[
        row_idx
    ]


    # --------------------------------------------------------------
    # Sequence heading
    # --------------------------------------------------------------

    axD.text(
        0.055,
        y0 + 0.095,

        f"{seq_id}   —   motif {motif}",

        transform=axD.transAxes,

        fontsize=11,

        fontweight="bold",

        ha="left",
        va="center"
    )


    # --------------------------------------------------------------
    # Larger boxes
    # --------------------------------------------------------------

    n = len(seq)

    available_width = 0.88

    box_w = min(
        0.034,
        available_width
        / max(
            n,
            1
        )
    )

    box_h = 0.085

    x_start = 0.055


    # --------------------------------------------------------------
    # Identify motif positions
    # --------------------------------------------------------------

    motif_positions = []


    if (
        motif
        and motif.lower() != "nan"
    ):

        start = 0

        while True:

            idx = seq.find(
                motif,
                start
            )

            if idx == -1:
                break


            motif_positions.extend(
                range(
                    idx,
                    idx
                    + len(motif)
                )
            )


            start = idx + 1


    motif_positions = set(
        motif_positions
    )


    # --------------------------------------------------------------
    # Draw sequence
    # --------------------------------------------------------------

    for j, aa in enumerate(
        seq
    ):

        x0 = (
            x_start
            + j
            * box_w
        )

        is_motif = (
            j
            in motif_positions
        )


        face = (
            "#e57373"
            if is_motif
            else "#e6e6e6"
        )


        rect = Rectangle(
            (
                x0,
                y0
            ),

            box_w
            * 0.94,

            box_h,

            transform=axD.transAxes,

            facecolor=face,

            edgecolor="black",

            linewidth=0.8
        )


        axD.add_patch(
            rect
        )


        axD.text(
            x0
            + box_w
            * 0.47,

            y0
            + box_h
            / 2,

            aa,

            transform=axD.transAxes,

            fontsize=8.2,

            fontweight="bold",

            ha="center",
            va="center"
        )


    # --------------------------------------------------------------
    # Motif markers
    # --------------------------------------------------------------

    for j in motif_positions:

        x0 = (
            x_start
            + j
            * box_w
        )

        axD.text(
            x0
            + box_w
            * 0.47,

            y0
            - 0.030,

            "★",

            transform=axD.transAxes,

            fontsize=8,

            ha="center",
            va="center"
        )


# ======================================================================
# D LEGEND
# ======================================================================

legend_D = [

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e6e6e6",

        edgecolor="black",

        label="Other residue"
    ),

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e57373",

        edgecolor="black",

        label="Replicated motif"
    ),

    Line2D(
        [0],
        [0],

        marker="*",

        linestyle="none",

        color="black",

        markersize=9,

        label="Motif position"
    )
]


axD.legend(
    handles=legend_D,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.5,
        -0.08
    ),

    ncol=3,

    fontsize=10,

    columnspacing=2.5
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_MS_FINAL_v2.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_MS_FINAL_v2.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_MS_FINAL_v2.svg"
)


fig.savefig(
    png_file,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.08
)


fig.savefig(
    pdf_file,

    bbox_inches="tight",

    pad_inches=0.08
)


fig.savefig(
    svg_file,

    bbox_inches="tight",

    pad_inches=0.08
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("IMPROVED FIGURE 4 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 4 — FINAL MANUSCRIPT VERSION
#
# TOP ROW:
# A = replicated residue enrichment
# B = internal vs external replication
# C = replicated hotspot motifs
#
# BOTTOM:
# D = representative motif-centered hotspot maps
#
# Improvements:
# - A/B/C in one row
# - D full-width below
# - larger fonts
# - A/B/C letters moved outward
# - A legend moved higher
# - B inset moved inward/right
# - more spacing between A/B/C
# - D residue boxes enlarged
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

CHECKPOINT_DIR = (
    PROJECT
    / "08_checkpoints"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RESIDUE_FILE = (
    TABLE_DIR
    / "residue_enrichment_replication.csv"
)

MOTIF_FILE = (
    TABLE_DIR
    / "hotspot_motif_replication.csv"
)

REP_SEQS_FILE = (
    CHECKPOINT_DIR
    / "figure_4_representative_sequences.csv"
)


# ======================================================================
# HELPERS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def find_col(df, candidates):

    cols = list(df.columns)

    for candidate in candidates:
        if candidate in cols:
            return candidate

    for candidate in candidates:
        for c in cols:
            if candidate in c:
                return c

    return None


# ======================================================================
# LOAD DATA
# ======================================================================

for f in [
    RESIDUE_FILE,
    MOTIF_FILE,
    REP_SEQS_FILE
]:

    if not f.exists():

        raise FileNotFoundError(
            f"Missing file:\n{f}"
        )


residue_df = clean_columns(
    pd.read_csv(
        RESIDUE_FILE
    )
)

motif_df = clean_columns(
    pd.read_csv(
        MOTIF_FILE
    )
)

rep_seq_df = clean_columns(
    pd.read_csv(
        REP_SEQS_FILE
    )
)


# ======================================================================
# DETECT RESIDUE COLUMNS
# ======================================================================

residue_col = find_col(
    residue_df,
    [
        "residue",
        "amino_acid",
        "aa"
    ]
)

internal_log2_col = find_col(
    residue_df,
    [
        "internal_log2_enrichment",
        "internal_log2_odds_ratio",
        "internal_log2"
    ]
)

external_log2_col = find_col(
    residue_df,
    [
        "kelm_log2_enrichment",
        "external_log2_enrichment",
        "kelm_log2_odds_ratio",
        "external_log2"
    ]
)

sig_both_col = find_col(
    residue_df,
    [
        "significant_in_both",
        "replicated",
        "significant_both"
    ]
)

same_direction_col = find_col(
    residue_df,
    [
        "same_direction",
        "replicated_direction"
    ]
)


if any(
    x is None
    for x in [
        residue_col,
        internal_log2_col,
        external_log2_col
    ]
):

    raise KeyError(
        "Could not detect required residue columns.\n"
        f"{residue_df.columns.tolist()}"
    )


res_plot = residue_df.copy()

if sig_both_col is not None:

    res_plot = res_plot[
        res_plot[
            sig_both_col
        ].astype(bool)
    ].copy()

elif same_direction_col is not None:

    res_plot = res_plot[
        res_plot[
            same_direction_col
        ].astype(bool)
    ].copy()


res_plot[
    internal_log2_col
] = pd.to_numeric(
    res_plot[
        internal_log2_col
    ],
    errors="coerce"
)

res_plot[
    external_log2_col
] = pd.to_numeric(
    res_plot[
        external_log2_col
    ],
    errors="coerce"
)


res_plot = res_plot.dropna(
    subset=[
        residue_col,
        internal_log2_col,
        external_log2_col
    ]
)


# ======================================================================
# DETECT MOTIF COLUMNS
# ======================================================================

motif_col = find_col(
    motif_df,
    [
        "motif",
        "hotspot_motif"
    ]
)

motif_internal_col = find_col(
    motif_df,
    [
        "internal_log2_enrichment",
        "internal_log2_odds_ratio",
        "internal_log2"
    ]
)

motif_external_col = find_col(
    motif_df,
    [
        "kelm_log2_enrichment",
        "external_log2_enrichment",
        "kelm_log2_odds_ratio",
        "external_log2"
    ]
)

motif_sig_col = find_col(
    motif_df,
    [
        "significant_in_both",
        "replicated",
        "significant_both"
    ]
)


if any(
    x is None
    for x in [
        motif_col,
        motif_internal_col,
        motif_external_col
    ]
):

    raise KeyError(
        "Could not detect required motif columns.\n"
        f"{motif_df.columns.tolist()}"
    )


motif_plot = motif_df.copy()

if motif_sig_col is not None:

    motif_plot = motif_plot[
        motif_plot[
            motif_sig_col
        ].astype(bool)
    ].copy()


motif_plot[
    motif_internal_col
] = pd.to_numeric(
    motif_plot[
        motif_internal_col
    ],
    errors="coerce"
)

motif_plot[
    motif_external_col
] = pd.to_numeric(
    motif_plot[
        motif_external_col
    ],
    errors="coerce"
)


motif_plot = motif_plot.dropna(
    subset=[
        motif_col,
        motif_internal_col,
        motif_external_col
    ]
)


# ======================================================================
# REPRESENTATIVE SEQUENCE COLUMNS
# ======================================================================

seq_id_col = find_col(
    rep_seq_df,
    [
        "sequence_id",
        "id",
        "seq_id"
    ]
)

seq_col = find_col(
    rep_seq_df,
    [
        "sequence",
        "peptide_sequence",
        "seq"
    ]
)

seq_motif_col = find_col(
    rep_seq_df,
    [
        "motif",
        "representative_motif"
    ]
)


if (
    seq_id_col is None
    or seq_col is None
):

    raise KeyError(
        "Could not detect representative sequence columns.\n"
        f"{rep_seq_df.columns.tolist()}"
    )


# ======================================================================
# PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 12,

    "axes.labelsize": 13,
    "axes.labelweight": "bold",

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,

    "legend.fontsize": 10,

    "axes.linewidth": 1.3,

    "lines.linewidth": 1.5,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE LAYOUT
# ======================================================================

fig = plt.figure(
    figsize=(18.5, 10.5)
)


outer = gridspec.GridSpec(
    2,
    1,

    figure=fig,

    height_ratios=[
        1.0,
        1.05
    ],

    hspace=0.43
)


# ======================================================================
# TOP ROW A/B/C
# ======================================================================

top = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[0],

    width_ratios=[
        1.05,
        1.05,
        1.15
    ],

    # More room between A/B/C
    wspace=0.40
)


axA = fig.add_subplot(
    top[0]
)

axB = fig.add_subplot(
    top[1]
)

axC = fig.add_subplot(
    top[2]
)


# ======================================================================
# COMMON LEGEND
# ======================================================================

legend_handles = [

    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="none",

        markerfacecolor="tab:blue",
        markeredgecolor="black",

        markersize=7,

        label="Internal test"
    ),

    Line2D(
        [0],
        [0],

        marker="s",
        linestyle="none",

        markerfacecolor="tab:orange",
        markeredgecolor="black",

        markersize=7,

        label="KELM external"
    )
]


# ======================================================================
# PANEL A — RESIDUE ENRICHMENT
# ======================================================================

res_A = (
    res_plot
    .sort_values(
        internal_log2_col,
        ascending=True
    )
    .reset_index(drop=True)
)


yA = np.arange(
    len(res_A)
)


for i, row in res_A.iterrows():

    internal_value = float(
        row[
            internal_log2_col
        ]
    )

    external_value = float(
        row[
            external_log2_col
        ]
    )


    axA.plot(
        [
            internal_value,
            external_value
        ],
        [
            i,
            i
        ],

        color="0.68",

        linewidth=1.4,

        zorder=1
    )


    axA.scatter(
        internal_value,
        i,

        s=58,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


    axA.scatter(
        external_value,
        i,

        s=58,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


# zero reference
axA.axvline(
    0,

    linestyle="--",

    color="0.45",

    linewidth=1.2
)


axA.set_yticks(
    yA
)

axA.set_yticklabels(
    res_A[
        residue_col
    ].astype(str)
)


axA.set_xlabel(
    "log$_2$ enrichment",
    fontsize=13,
    fontweight="bold"
)

axA.set_ylabel(
    "Amino-acid residue",
    fontsize=13,
    fontweight="bold",
    labelpad=8
)


for label in axA.get_xticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


for label in axA.get_yticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


axA.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# Legend moved higher
# ----------------------------------------------------------------------

axA.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.56,
        1.075
    ),

    ncol=2,

    fontsize=10,

    handletextpad=0.4,

    columnspacing=1.4,

    borderaxespad=0
)


# Direction labels
axA.text(
    0.03,
    0.025,

    "Depleted",

    transform=axA.transAxes,

    fontsize=9.5,
    fontweight="bold",

    ha="left",
    va="bottom"
)

axA.text(
    0.97,
    0.025,

    "Enriched",

    transform=axA.transAxes,

    fontsize=9.5,
    fontweight="bold",

    ha="right",
    va="bottom"
)


axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# A moved farther outside
# ----------------------------------------------------------------------

axA.text(
    -0.19,
    1.085,

    "A",

    transform=axA.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL B — INTERNAL vs KELM REPLICATION
# ======================================================================

xB = (
    res_plot[
        internal_log2_col
    ]
    .astype(float)
    .to_numpy()
)

yB = (
    res_plot[
        external_log2_col
    ]
    .astype(float)
    .to_numpy()
)

labelsB = (
    res_plot[
        residue_col
    ]
    .astype(str)
    .to_numpy()
)


# ----------------------------------------------------------------------
# Extreme observation
# ----------------------------------------------------------------------

distance_from_center = (
    np.abs(xB)
    + np.abs(yB)
)

extreme_index = np.argmax(
    distance_from_center
)

normal_mask = np.ones(
    len(xB),
    dtype=bool
)

normal_mask[
    extreme_index
] = False


normal_x = xB[
    normal_mask
]

normal_y = yB[
    normal_mask
]


# ----------------------------------------------------------------------
# Main scatter
# ----------------------------------------------------------------------

axB.scatter(
    normal_x,
    normal_y,

    s=64,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7,

    zorder=3
)


# labels
for x, y, aa in zip(
    normal_x,
    normal_y,
    labelsB[
        normal_mask
    ]
):

    axB.annotate(
        aa,

        xy=(
            x,
            y
        ),

        xytext=(
            7,
            5
        ),

        textcoords="offset points",

        fontsize=10,

        fontweight="bold"
    )


# ----------------------------------------------------------------------
# Main range
# ----------------------------------------------------------------------

all_normal = np.concatenate(
    [
        normal_x,
        normal_y
    ]
)


plot_min = (
    np.min(
        all_normal
    )
    - 0.8
)

plot_max = (
    np.max(
        all_normal
    )
    + 0.8
)

plot_max = max(
    plot_max,
    0.8
)


axB.set_xlim(
    plot_min,
    plot_max
)

axB.set_ylim(
    plot_min,
    plot_max
)


# identity
axB.plot(
    [
        plot_min,
        plot_max
    ],
    [
        plot_min,
        plot_max
    ],

    linestyle="--",

    color="0.45",

    linewidth=1.3
)


axB.axhline(
    0,
    linestyle=":",
    color="0.65",
    linewidth=1
)

axB.axvline(
    0,
    linestyle=":",
    color="0.65",
    linewidth=1
)


axB.set_xlabel(
    "Internal log$_2$ enrichment",
    fontsize=13,
    fontweight="bold"
)

axB.set_ylabel(
    "KELM log$_2$ enrichment",
    fontsize=13,
    fontweight="bold",
    labelpad=8
)


for label in axB.get_xticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


for label in axB.get_yticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


axB.tick_params(
    width=1.2,
    length=5
)


# ======================================================================
# B INSET — moved inward/right
# ======================================================================

inset = inset_axes(
    axB,

    width="31%",
    height="31%",

    loc="upper left",

    bbox_to_anchor=(
        0.10,
        0.00,
        1,
        1
    ),

    bbox_transform=axB.transAxes,

    borderpad=0.8
)


extreme_x = xB[
    extreme_index
]

extreme_y = yB[
    extreme_index
]

extreme_label = labelsB[
    extreme_index
]


inset.scatter(
    extreme_x,
    extreme_y,

    s=48,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7
)


inset.annotate(
    extreme_label,

    xy=(
        extreme_x,
        extreme_y
    ),

    xytext=(
        6,
        5
    ),

    textcoords="offset points",

    fontsize=9,

    fontweight="bold"
)


extreme_min = min(
    extreme_x,
    extreme_y
)

extreme_max = max(
    extreme_x,
    extreme_y
)

extreme_pad = max(
    1.5,

    0.20
    * (
        extreme_max
        - extreme_min
    )
)


inset.set_xlim(
    extreme_min
    - extreme_pad,

    extreme_max
    + extreme_pad
)

inset.set_ylim(
    extreme_min
    - extreme_pad,

    extreme_max
    + extreme_pad
)


inset.plot(
    [
        extreme_min
        - extreme_pad,

        extreme_max
        + extreme_pad
    ],

    [
        extreme_min
        - extreme_pad,

        extreme_max
        + extreme_pad
    ],

    linestyle="--",

    color="0.5",

    linewidth=0.8
)


inset.set_title(
    "Extreme effect",

    fontsize=9,

    fontweight="bold",

    pad=3
)


inset.tick_params(
    labelsize=7.2,
    width=0.8,
    length=3
)


for label in inset.get_xticklabels():

    label.set_fontweight(
        "bold"
    )


for label in inset.get_yticklabels():

    label.set_fontweight(
        "bold"
    )


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# B moved farther outside
# ----------------------------------------------------------------------

axB.text(
    -0.20,
    1.085,

    "B",

    transform=axB.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C — REPLICATED MOTIFS
# ======================================================================

motif_C = (
    motif_plot
    .sort_values(
        motif_internal_col,
        ascending=True
    )
    .reset_index(drop=True)
)


yC = np.arange(
    len(motif_C)
)


for i, row in motif_C.iterrows():

    internal_value = float(
        row[
            motif_internal_col
        ]
    )

    external_value = float(
        row[
            motif_external_col
        ]
    )


    axC.plot(
        [
            internal_value,
            external_value
        ],
        [
            i,
            i
        ],

        color="0.68",

        linewidth=1.4,

        zorder=1
    )


    axC.scatter(
        internal_value,
        i,

        s=58,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


    axC.scatter(
        external_value,
        i,

        s=58,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axC.set_yticks(
    yC
)

axC.set_yticklabels(
    motif_C[
        motif_col
    ].astype(str)
)


axC.set_xlabel(
    "log$_2$ enrichment",
    fontsize=13,
    fontweight="bold"
)

axC.set_ylabel(
    "Replicated hotspot motif",
    fontsize=13,
    fontweight="bold",
    labelpad=8
)


for label in axC.get_xticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


for label in axC.get_yticklabels():

    label.set_fontsize(11)
    label.set_fontweight("bold")


axC.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# Legend ABOVE C
# ----------------------------------------------------------------------

axC.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.075
    ),

    ncol=2,

    fontsize=10,

    handletextpad=0.4,

    columnspacing=1.4,

    borderaxespad=0
)


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# C outside
# ----------------------------------------------------------------------

axC.text(
    -0.17,
    1.085,

    "C",

    transform=axC.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D — FULL WIDTH
# ======================================================================

axD = fig.add_subplot(
    outer[1]
)

axD.set_axis_off()


# ----------------------------------------------------------------------
# D label
# ----------------------------------------------------------------------

axD.text(
    -0.025,
    1.02,

    "D",

    transform=axD.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# REPRESENTATIVE SEQUENCES
# ======================================================================

rep_show = (
    rep_seq_df
    .head(3)
    .copy()
)


y_positions = [
    0.72,
    0.43,
    0.14
]


for row_idx, (_, row) in enumerate(
    rep_show.iterrows()
):

    seq_id = str(
        row[
            seq_id_col
        ]
    )

    seq = str(
        row[
            seq_col
        ]
    )

    motif = (
        str(
            row[
                seq_motif_col
            ]
        )

        if seq_motif_col is not None

        else ""
    )


    y0 = y_positions[
        row_idx
    ]


    # heading
    axD.text(
        0.055,
        y0 + 0.095,

        f"{seq_id}   —   motif {motif}",

        transform=axD.transAxes,

        fontsize=11,

        fontweight="bold",

        ha="left",
        va="center"
    )


    # box geometry
    n = len(seq)

    available_width = 0.88

    box_w = min(
        0.034,
        available_width
        / max(
            n,
            1
        )
    )

    box_h = 0.085

    x_start = 0.055


    # motif positions
    motif_positions = []

    if (
        motif
        and motif.lower() != "nan"
    ):

        start = 0

        while True:

            idx = seq.find(
                motif,
                start
            )

            if idx == -1:
                break

            motif_positions.extend(
                range(
                    idx,
                    idx
                    + len(motif)
                )
            )

            start = idx + 1


    motif_positions = set(
        motif_positions
    )


    # draw boxes
    for j, aa in enumerate(
        seq
    ):

        x0 = (
            x_start
            + j
            * box_w
        )


        is_motif = (
            j
            in motif_positions
        )


        face = (
            "#e57373"
            if is_motif
            else "#e6e6e6"
        )


        rect = Rectangle(
            (
                x0,
                y0
            ),

            box_w
            * 0.94,

            box_h,

            transform=axD.transAxes,

            facecolor=face,

            edgecolor="black",

            linewidth=0.8
        )

        axD.add_patch(
            rect
        )


        axD.text(
            x0
            + box_w
            * 0.47,

            y0
            + box_h
            / 2,

            aa,

            transform=axD.transAxes,

            fontsize=8.2,

            fontweight="bold",

            ha="center",
            va="center"
        )


    # motif markers
    for j in motif_positions:

        x0 = (
            x_start
            + j
            * box_w
        )

        axD.text(
            x0
            + box_w
            * 0.47,

            y0
            - 0.030,

            "★",

            transform=axD.transAxes,

            fontsize=8,

            ha="center",
            va="center"
        )


# ======================================================================
# PANEL D LEGEND
# ======================================================================

legend_D = [

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e6e6e6",

        edgecolor="black",

        label="Other residue"
    ),

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e57373",

        edgecolor="black",

        label="Replicated motif"
    ),

    Line2D(
        [0],
        [0],

        marker="*",

        linestyle="none",

        color="black",

        markersize=9,

        label="Motif position"
    )
]


axD.legend(
    handles=legend_D,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.5,
        -0.08
    ),

    ncol=3,

    fontsize=10,

    columnspacing=2.5
)


# ======================================================================
# FINAL MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.075,
    right=0.975,
    bottom=0.08,
    top=0.93
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_FINAL_v3.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_FINAL_v3.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_FINAL_v3.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.08
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.08
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.08
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FINAL FIGURE 4 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/final_manuscript_package/03_main_figures/Figure_4_Residue_Motif_Grammar_FINAL_v3.png"
)

In [ ]:
# ======================================================================
# FIGURE 4 — COMPACT HIGH-QUALITY MANUSCRIPT VERSION
#
# A, B, C = single compact top row
# D = full-width bottom row
#
# Improvements:
# - much less white space
# - larger text
# - larger D sequence maps
# - tighter vertical layout
# - balanced panel spacing
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 12,

    "axes.labelsize": 14,
    "axes.labelweight": "bold",

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,

    "legend.fontsize": 10,

    "axes.linewidth": 1.3,

    "lines.linewidth": 1.5,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
#
# Shorter figure = much less white space
# ======================================================================

fig = plt.figure(
    figsize=(17.5, 7.4)
)


outer = gridspec.GridSpec(
    2,
    1,

    figure=fig,

    height_ratios=[
        1.00,   # A B C
        0.85    # D
    ],

    # much tighter vertical spacing
    hspace=0.22
)


# ======================================================================
# TOP ROW — A B C
# ======================================================================

top = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[0],

    width_ratios=[
        1.00,
        1.05,
        1.10
    ],

    wspace=0.32
)


axA = fig.add_subplot(
    top[0]
)

axB = fig.add_subplot(
    top[1]
)

axC = fig.add_subplot(
    top[2]
)


# ======================================================================
# COMMON LEGEND
# ======================================================================

legend_handles = [

    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="none",

        markerfacecolor="tab:blue",
        markeredgecolor="black",

        markersize=7,

        label="Internal test"
    ),

    Line2D(
        [0],
        [0],

        marker="s",
        linestyle="none",

        markerfacecolor="tab:orange",
        markeredgecolor="black",

        markersize=7,

        label="KELM external"
    )
]


# ======================================================================
# PANEL A
# ======================================================================

res_A = (
    res_plot
    .sort_values(
        internal_log2_col,
        ascending=True
    )
    .reset_index(drop=True)
)

yA = np.arange(
    len(res_A)
)


for i, row in res_A.iterrows():

    internal_value = float(
        row[internal_log2_col]
    )

    external_value = float(
        row[external_log2_col]
    )

    axA.plot(
        [internal_value, external_value],
        [i, i],

        color="0.70",
        linewidth=1.4,

        zorder=1
    )

    axA.scatter(
        internal_value,
        i,

        s=64,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )

    axA.scatter(
        external_value,
        i,

        s=64,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axA.axvline(
    0,

    linestyle="--",

    color="0.45",

    linewidth=1.2
)


axA.set_yticks(
    yA
)

axA.set_yticklabels(
    res_A[residue_col].astype(str)
)


axA.set_xlabel(
    "log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axA.set_ylabel(
    "Amino-acid residue",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axA.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axA.get_yticklabels():
    label.set_fontsize(12)
    label.set_fontweight("bold")


axA.tick_params(
    width=1.2,
    length=5
)


# Legend above A
axA.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.55,
        1.025
    ),

    ncol=2,

    fontsize=10,

    handletextpad=0.4,

    columnspacing=1.3,

    borderaxespad=0
)


# Direction labels
axA.text(
    0.03,
    0.025,

    "Depleted",

    transform=axA.transAxes,

    fontsize=10,
    fontweight="bold",

    ha="left",
    va="bottom"
)

axA.text(
    0.97,
    0.025,

    "Enriched",

    transform=axA.transAxes,

    fontsize=10,
    fontweight="bold",

    ha="right",
    va="bottom"
)


axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


axA.text(
    -0.17,
    1.07,

    "A",

    transform=axA.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL B
# ======================================================================

xB = (
    res_plot[internal_log2_col]
    .astype(float)
    .to_numpy()
)

yB = (
    res_plot[external_log2_col]
    .astype(float)
    .to_numpy()
)

labelsB = (
    res_plot[residue_col]
    .astype(str)
    .to_numpy()
)


distance_from_center = (
    np.abs(xB)
    + np.abs(yB)
)

extreme_index = np.argmax(
    distance_from_center
)

normal_mask = np.ones(
    len(xB),
    dtype=bool
)

normal_mask[
    extreme_index
] = False


normal_x = xB[
    normal_mask
]

normal_y = yB[
    normal_mask
]


axB.scatter(
    normal_x,
    normal_y,

    s=68,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7,

    zorder=3
)


for x, y, aa in zip(
    normal_x,
    normal_y,
    labelsB[normal_mask]
):

    axB.annotate(
        aa,

        xy=(x, y),

        xytext=(7, 5),

        textcoords="offset points",

        fontsize=10.5,

        fontweight="bold"
    )


all_normal = np.concatenate(
    [
        normal_x,
        normal_y
    ]
)

plot_min = (
    np.min(all_normal)
    - 0.7
)

plot_max = (
    np.max(all_normal)
    + 0.7
)

plot_max = max(
    plot_max,
    0.8
)


axB.set_xlim(
    plot_min,
    plot_max
)

axB.set_ylim(
    plot_min,
    plot_max
)


axB.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],

    linestyle="--",

    color="0.45",

    linewidth=1.3
)


axB.axhline(
    0,

    linestyle=":",

    color="0.65",

    linewidth=1
)

axB.axvline(
    0,

    linestyle=":",

    color="0.65",

    linewidth=1
)


axB.set_xlabel(
    "Internal log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axB.set_ylabel(
    "KELM log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axB.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axB.get_yticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")


axB.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# Inset — compact and positioned inward
# ----------------------------------------------------------------------

inset = inset_axes(
    axB,

    width="29%",
    height="29%",

    loc="upper left",

    bbox_to_anchor=(
        0.11,
        0.00,
        1,
        1
    ),

    bbox_transform=axB.transAxes,

    borderpad=0.7
)


extreme_x = xB[
    extreme_index
]

extreme_y = yB[
    extreme_index
]

extreme_label = labelsB[
    extreme_index
]


inset.scatter(
    extreme_x,
    extreme_y,

    s=48,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7
)


inset.annotate(
    extreme_label,

    xy=(
        extreme_x,
        extreme_y
    ),

    xytext=(
        5,
        4
    ),

    textcoords="offset points",

    fontsize=8.5,

    fontweight="bold"
)


extreme_min = min(
    extreme_x,
    extreme_y
)

extreme_max = max(
    extreme_x,
    extreme_y
)


extreme_pad = max(
    1.5,

    0.20
    * (
        extreme_max
        - extreme_min
    )
)


inset.set_xlim(
    extreme_min
    - extreme_pad,

    extreme_max
    + extreme_pad
)

inset.set_ylim(
    extreme_min
    - extreme_pad,

    extreme_max
    + extreme_pad
)


inset.plot(
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],

    linestyle="--",

    color="0.5",

    linewidth=0.8
)


inset.set_title(
    "Extreme effect",

    fontsize=8.5,

    fontweight="bold",

    pad=2
)


inset.tick_params(
    labelsize=7,
    width=0.8,
    length=2.5
)


for label in inset.get_xticklabels():
    label.set_fontweight("bold")

for label in inset.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


axB.text(
    -0.17,
    1.07,

    "B",

    transform=axB.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

motif_C = (
    motif_plot
    .sort_values(
        motif_internal_col,
        ascending=True
    )
    .reset_index(drop=True)
)


yC = np.arange(
    len(motif_C)
)


for i, row in motif_C.iterrows():

    internal_value = float(
        row[motif_internal_col]
    )

    external_value = float(
        row[motif_external_col]
    )

    axC.plot(
        [internal_value, external_value],
        [i, i],

        color="0.70",
        linewidth=1.4,

        zorder=1
    )

    axC.scatter(
        internal_value,
        i,

        s=64,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )

    axC.scatter(
        external_value,
        i,

        s=64,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axC.set_yticks(
    yC
)

axC.set_yticklabels(
    motif_C[motif_col]
    .astype(str)
)


axC.set_xlabel(
    "log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axC.set_ylabel(
    "Replicated hotspot motif",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axC.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axC.get_yticklabels():
    label.set_fontsize(12)
    label.set_fontweight("bold")


axC.tick_params(
    width=1.2,
    length=5
)


axC.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.52,
        1.025
    ),

    ncol=2,

    fontsize=10,

    handletextpad=0.4,

    columnspacing=1.3,

    borderaxespad=0
)


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


axC.text(
    -0.16,
    1.07,

    "C",

    transform=axC.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D — FULL WIDTH, ENLARGED
# ======================================================================

axD = fig.add_subplot(
    outer[1]
)

axD.set_axis_off()


axD.text(
    -0.018,
    1.015,

    "D",

    transform=axD.transAxes,

    fontsize=23,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


rep_show = (
    rep_seq_df
    .head(3)
    .copy()
)


# closer together = less blank space
y_positions = [
    0.70,
    0.40,
    0.10
]


for row_idx, (_, row) in enumerate(
    rep_show.iterrows()
):

    seq_id = str(
        row[seq_id_col]
    )

    seq = str(
        row[seq_col]
    )

    motif = (
        str(
            row[seq_motif_col]
        )

        if seq_motif_col is not None

        else ""
    )

    y0 = y_positions[
        row_idx
    ]


    # sequence heading
    axD.text(
        0.045,
        y0 + 0.105,

        f"{seq_id}   —   motif {motif}",

        transform=axD.transAxes,

        fontsize=12,

        fontweight="bold",

        ha="left",
        va="center"
    )


    n = len(seq)

    available_width = 0.92

    box_w = min(
        0.036,

        available_width
        / max(
            n,
            1
        )
    )

    box_h = 0.105

    x_start = 0.045


    motif_positions = []


    if (
        motif
        and motif.lower() != "nan"
    ):

        start = 0

        while True:

            idx = seq.find(
                motif,
                start
            )

            if idx == -1:
                break

            motif_positions.extend(
                range(
                    idx,
                    idx + len(motif)
                )
            )

            start = idx + 1


    motif_positions = set(
        motif_positions
    )


    for j, aa in enumerate(
        seq
    ):

        x0 = (
            x_start
            + j * box_w
        )


        is_motif = (
            j in motif_positions
        )


        face = (
            "#e57373"
            if is_motif
            else "#e6e6e6"
        )


        rect = Rectangle(
            (
                x0,
                y0
            ),

            box_w * 0.94,
            box_h,

            transform=axD.transAxes,

            facecolor=face,

            edgecolor="black",

            linewidth=0.9
        )

        axD.add_patch(
            rect
        )


        axD.text(
            x0 + box_w * 0.47,
            y0 + box_h / 2,

            aa,

            transform=axD.transAxes,

            fontsize=9,

            fontweight="bold",

            ha="center",
            va="center"
        )


    for j in motif_positions:

        x0 = (
            x_start
            + j * box_w
        )

        axD.text(
            x0 + box_w * 0.47,
            y0 - 0.035,

            "★",

            transform=axD.transAxes,

            fontsize=8.5,

            ha="center",
            va="center"
        )


# ======================================================================
# D LEGEND
# ======================================================================

legend_D = [

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e6e6e6",

        edgecolor="black",

        label="Other residue"
    ),

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e57373",

        edgecolor="black",

        label="Replicated motif"
    ),

    Line2D(
        [0],
        [0],

        marker="*",

        linestyle="none",

        color="black",

        markersize=10,

        label="Motif position"
    )
]


axD.legend(
    handles=legend_D,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        -0.055
    ),

    ncol=3,

    fontsize=11,

    columnspacing=2.5
)


# ======================================================================
# FINAL MARGINS — MUCH TIGHTER
# ======================================================================

fig.subplots_adjust(
    left=0.065,
    right=0.985,
    bottom=0.075,
    top=0.93
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_COMPACT_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_COMPACT_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_COMPACT_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.04
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("COMPACT FIGURE 4 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 4 — FINAL POLISHED VERSION
#
# A, B, C = top row
# D = full-width bottom
#
# Updates:
# - Panel D overlap fixed
# - D headings moved above boxes
# - More vertical spacing between sequence rows
# - D legend moved lower
# - A and C legends larger + bold
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 12,

    "axes.labelsize": 14,
    "axes.labelweight": "bold",

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,

    "legend.fontsize": 11,

    "axes.linewidth": 1.3,

    "lines.linewidth": 1.5,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(17.5, 7.7)
)


outer = gridspec.GridSpec(
    2,
    1,

    figure=fig,

    height_ratios=[
        1.00,
        0.92
    ],

    hspace=0.22
)


# ======================================================================
# TOP ROW — A B C
# ======================================================================

top = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[0],

    width_ratios=[
        1.00,
        1.05,
        1.10
    ],

    wspace=0.32
)


axA = fig.add_subplot(top[0])
axB = fig.add_subplot(top[1])
axC = fig.add_subplot(top[2])


# ======================================================================
# COMMON LEGEND
# ======================================================================

legend_handles = [

    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="none",

        markerfacecolor="tab:blue",
        markeredgecolor="black",

        markersize=7.5,

        label="Internal test"
    ),

    Line2D(
        [0],
        [0],

        marker="s",
        linestyle="none",

        markerfacecolor="tab:orange",
        markeredgecolor="black",

        markersize=7.5,

        label="KELM external"
    )
]


# ======================================================================
# PANEL A
# ======================================================================

res_A = (
    res_plot
    .sort_values(
        internal_log2_col,
        ascending=True
    )
    .reset_index(drop=True)
)

yA = np.arange(len(res_A))


for i, row in res_A.iterrows():

    internal_value = float(
        row[internal_log2_col]
    )

    external_value = float(
        row[external_log2_col]
    )

    axA.plot(
        [internal_value, external_value],
        [i, i],

        color="0.70",
        linewidth=1.4,

        zorder=1
    )

    axA.scatter(
        internal_value,
        i,

        s=64,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )

    axA.scatter(
        external_value,
        i,

        s=64,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axA.axvline(
    0,
    linestyle="--",
    color="0.45",
    linewidth=1.2
)


axA.set_yticks(yA)

axA.set_yticklabels(
    res_A[residue_col].astype(str)
)


axA.set_xlabel(
    "log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axA.set_ylabel(
    "Amino-acid residue",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axA.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axA.get_yticklabels():
    label.set_fontsize(12)
    label.set_fontweight("bold")


axA.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# A legend — larger and bold
# ----------------------------------------------------------------------

legA = axA.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(0.55, 1.025),

    ncol=2,

    fontsize=11,

    handletextpad=0.45,

    columnspacing=1.4,

    borderaxespad=0
)

for txt in legA.get_texts():
    txt.set_fontweight("bold")


axA.text(
    0.03,
    0.025,
    "Depleted",
    transform=axA.transAxes,
    fontsize=10,
    fontweight="bold",
    ha="left",
    va="bottom"
)

axA.text(
    0.97,
    0.025,
    "Enriched",
    transform=axA.transAxes,
    fontsize=10,
    fontweight="bold",
    ha="right",
    va="bottom"
)


axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


axA.text(
    -0.17,
    1.07,
    "A",
    transform=axA.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL B
# ======================================================================

xB = (
    res_plot[internal_log2_col]
    .astype(float)
    .to_numpy()
)

yB = (
    res_plot[external_log2_col]
    .astype(float)
    .to_numpy()
)

labelsB = (
    res_plot[residue_col]
    .astype(str)
    .to_numpy()
)


distance_from_center = (
    np.abs(xB)
    + np.abs(yB)
)

extreme_index = np.argmax(
    distance_from_center
)

normal_mask = np.ones(
    len(xB),
    dtype=bool
)

normal_mask[extreme_index] = False

normal_x = xB[normal_mask]
normal_y = yB[normal_mask]


axB.scatter(
    normal_x,
    normal_y,

    s=68,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7,

    zorder=3
)


for x, y, aa in zip(
    normal_x,
    normal_y,
    labelsB[normal_mask]
):

    axB.annotate(
        aa,
        xy=(x, y),
        xytext=(7, 5),
        textcoords="offset points",
        fontsize=10.5,
        fontweight="bold"
    )


all_normal = np.concatenate(
    [normal_x, normal_y]
)

plot_min = (
    np.min(all_normal)
    - 0.7
)

plot_max = (
    np.max(all_normal)
    + 0.7
)

plot_max = max(
    plot_max,
    0.8
)


axB.set_xlim(
    plot_min,
    plot_max
)

axB.set_ylim(
    plot_min,
    plot_max
)


axB.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],

    linestyle="--",

    color="0.45",

    linewidth=1.3
)


axB.axhline(
    0,
    linestyle=":",
    color="0.65",
    linewidth=1
)

axB.axvline(
    0,
    linestyle=":",
    color="0.65",
    linewidth=1
)


axB.set_xlabel(
    "Internal log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axB.set_ylabel(
    "KELM log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axB.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axB.get_yticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")


axB.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# B inset
# ----------------------------------------------------------------------

inset = inset_axes(
    axB,

    width="29%",
    height="29%",

    loc="upper left",

    bbox_to_anchor=(
        0.11,
        0.00,
        1,
        1
    ),

    bbox_transform=axB.transAxes,

    borderpad=0.7
)


extreme_x = xB[extreme_index]
extreme_y = yB[extreme_index]
extreme_label = labelsB[extreme_index]


inset.scatter(
    extreme_x,
    extreme_y,

    s=48,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7
)


inset.annotate(
    extreme_label,

    xy=(extreme_x, extreme_y),

    xytext=(5, 4),

    textcoords="offset points",

    fontsize=8.5,

    fontweight="bold"
)


extreme_min = min(
    extreme_x,
    extreme_y
)

extreme_max = max(
    extreme_x,
    extreme_y
)

extreme_pad = max(
    1.5,

    0.20
    * (
        extreme_max
        - extreme_min
    )
)


inset.set_xlim(
    extreme_min - extreme_pad,
    extreme_max + extreme_pad
)

inset.set_ylim(
    extreme_min - extreme_pad,
    extreme_max + extreme_pad
)


inset.plot(
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],

    linestyle="--",

    color="0.5",

    linewidth=0.8
)


inset.set_title(
    "Extreme effect",

    fontsize=8.5,

    fontweight="bold",

    pad=2
)


inset.tick_params(
    labelsize=7,
    width=0.8,
    length=2.5
)


for label in inset.get_xticklabels():
    label.set_fontweight("bold")

for label in inset.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


axB.text(
    -0.17,
    1.07,
    "B",
    transform=axB.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

motif_C = (
    motif_plot
    .sort_values(
        motif_internal_col,
        ascending=True
    )
    .reset_index(drop=True)
)

yC = np.arange(
    len(motif_C)
)


for i, row in motif_C.iterrows():

    internal_value = float(
        row[motif_internal_col]
    )

    external_value = float(
        row[motif_external_col]
    )

    axC.plot(
        [internal_value, external_value],
        [i, i],

        color="0.70",
        linewidth=1.4,

        zorder=1
    )

    axC.scatter(
        internal_value,
        i,

        s=64,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )

    axC.scatter(
        external_value,
        i,

        s=64,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axC.set_yticks(yC)

axC.set_yticklabels(
    motif_C[motif_col]
    .astype(str)
)


axC.set_xlabel(
    "log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axC.set_ylabel(
    "Replicated hotspot motif",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axC.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axC.get_yticklabels():
    label.set_fontsize(12)
    label.set_fontweight("bold")


axC.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# C legend — larger and bold
# ----------------------------------------------------------------------

legC = axC.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(0.52, 1.025),

    ncol=2,

    fontsize=11,

    handletextpad=0.45,

    columnspacing=1.4,

    borderaxespad=0
)

for txt in legC.get_texts():
    txt.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


axC.text(
    -0.16,
    1.07,
    "C",
    transform=axC.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL D — OVERLAP FIXED
# ======================================================================

axD = fig.add_subplot(
    outer[1]
)

axD.set_axis_off()


axD.text(
    -0.018,
    1.015,
    "D",
    transform=axD.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


rep_show = (
    rep_seq_df
    .head(3)
    .copy()
)


# ----------------------------------------------------------------------
# More vertical room between sequence rows
# ----------------------------------------------------------------------

y_positions = [
    0.68,
    0.37,
    0.06
]


for row_idx, (_, row) in enumerate(
    rep_show.iterrows()
):

    seq_id = str(
        row[seq_id_col]
    )

    seq = str(
        row[seq_col]
    )

    motif = (
        str(
            row[seq_motif_col]
        )
        if seq_motif_col is not None
        else ""
    )

    y0 = y_positions[row_idx]


    # --------------------------------------------------------------
    # Heading moved further above residue boxes
    # --------------------------------------------------------------

    axD.text(
        0.045,
        y0 + 0.135,

        f"{seq_id}   —   motif {motif}",

        transform=axD.transAxes,

        fontsize=12,

        fontweight="bold",

        ha="left",
        va="center"
    )


    n = len(seq)

    available_width = 0.92

    box_w = min(
        0.036,

        available_width
        / max(n, 1)
    )

    box_h = 0.10

    x_start = 0.045


    # --------------------------------------------------------------
    # Motif positions
    # --------------------------------------------------------------

    motif_positions = []

    if (
        motif
        and motif.lower() != "nan"
    ):

        start = 0

        while True:

            idx = seq.find(
                motif,
                start
            )

            if idx == -1:
                break

            motif_positions.extend(
                range(
                    idx,
                    idx + len(motif)
                )
            )

            start = idx + 1


    motif_positions = set(
        motif_positions
    )


    # --------------------------------------------------------------
    # Residue boxes
    # --------------------------------------------------------------

    for j, aa in enumerate(seq):

        x0 = (
            x_start
            + j * box_w
        )

        is_motif = (
            j in motif_positions
        )

        face = (
            "#e57373"
            if is_motif
            else "#e6e6e6"
        )


        rect = Rectangle(
            (
                x0,
                y0
            ),

            box_w * 0.94,
            box_h,

            transform=axD.transAxes,

            facecolor=face,

            edgecolor="black",

            linewidth=0.9
        )

        axD.add_patch(
            rect
        )


        axD.text(
            x0 + box_w * 0.47,
            y0 + box_h / 2,

            aa,

            transform=axD.transAxes,

            fontsize=9,

            fontweight="bold",

            ha="center",
            va="center"
        )


    # --------------------------------------------------------------
    # Motif stars
    # --------------------------------------------------------------

    for j in motif_positions:

        x0 = (
            x_start
            + j * box_w
        )

        axD.text(
            x0 + box_w * 0.47,
            y0 - 0.040,

            "★",

            transform=axD.transAxes,

            fontsize=8.5,

            ha="center",
            va="center"
        )


# ======================================================================
# PANEL D LEGEND — moved lower
# ======================================================================

legend_D = [

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e6e6e6",

        edgecolor="black",

        label="Other residue"
    ),

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e57373",

        edgecolor="black",

        label="Replicated motif"
    ),

    Line2D(
        [0],
        [0],

        marker="*",

        linestyle="none",

        color="black",

        markersize=10,

        label="Motif position"
    )
]


axD.legend(
    handles=legend_D,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        -0.10
    ),

    ncol=3,

    fontsize=11,

    columnspacing=2.5
)


# ======================================================================
# MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.065,
    right=0.985,
    bottom=0.09,
    top=0.93
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.04
)


plt.show()
plt.close(fig)


print("\n" + "=" * 90)
print("POLISHED FIGURE 4 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
from google.colab import files
import shutil

FIG_DIR = "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/figures_main"

shutil.make_archive(
    "/content/Figure_4_FINAL",
    "zip",
    FIG_DIR,
    base_dir="."
)

files.download("/content/Figure_4_FINAL.zip")

In [ ]:
!find /content/drive/MyDrive/pLM4CPP_XAI_2026 -type f | \
grep -Ei "perturb|ablation|faithful|sufficien|hotspot.*drop|random.*drop|figure_5" | \
sort

In [ ]:
# ======================================================================
# FIGURE 5 — MANUSCRIPT-QUALITY FINAL VERSION
# Perturbation-Based Faithfulness of Consensus CPP Hotspots
# ======================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import warnings

warnings.filterwarnings("ignore")

# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

FAITH_DIR = PROJECT / "06_xai" / "faithfulness"
RESULTS = PROJECT / "07_results"
TABLE_DIR = RESULTS / "tables_main"
OUT_DIR = RESULTS / "figures_main"

OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5"
]

MODEL_LABELS = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5"
]

DATASETS = [
    "internal_test",
    "kelm_external"
]


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,

    "axes.labelsize": 13,
    "axes.labelweight": "bold",

    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,

    "legend.fontsize": 10,

    "axes.linewidth": 1.2,

    "xtick.major.width": 1.1,
    "ytick.major.width": 1.1,

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "savefig.dpi": 600
})


# ======================================================================
# HELPER FUNCTIONS
# ======================================================================

def find_col(df, candidates, contains=None):
    """Robust column detector."""

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    if contains:
        for c in df.columns:
            lc = c.lower()
            if all(x.lower() in lc for x in contains):
                return c

    return None


def read_faithfulness(model, dataset):

    file = (
        FAITH_DIR
        / model
        / dataset
        / "per_sequence_faithfulness.csv"
    )

    if not file.exists():
        raise FileNotFoundError(file)

    df = pd.read_csv(file)

    df["model_plot"] = model
    df["dataset_plot"] = dataset

    return df


# ======================================================================
# LOAD ALL PER-SEQUENCE DATA
# ======================================================================

frames = []

for model in MODELS:
    for dataset in DATASETS:

        df = read_faithfulness(
            model,
            dataset
        )

        frames.append(df)


faith = pd.concat(
    frames,
    ignore_index=True
)


print("=" * 90)
print("PER-SEQUENCE FAITHFULNESS DATA")
print("=" * 90)

print("\nColumns:")
print(faith.columns.tolist())

print("\nShape:", faith.shape)

display(faith.head())


# ======================================================================
# DETECT COLUMNS
# ======================================================================

hotspot_drop_col = find_col(
    faith,
    [
        "hotspot_drop",
        "consensus_hotspot_drop",
        "hotspot_probability_drop",
        "consensus_drop",
        "hotspot_ablation_drop"
    ],
    contains=["hotspot", "drop"]
)


random_drop_col = find_col(
    faith,
    [
        "random_drop",
        "matched_random_drop",
        "random_probability_drop",
        "random_ablation_drop",
        "mean_random_drop"
    ],
    contains=["random", "drop"]
)


original_prob_col = find_col(
    faith,
    [
        "original_probability",
        "original_prob",
        "original_cpp_probability",
        "original_predicted_probability",
        "original_prediction"
    ],
    contains=["original", "prob"]
)


sufficiency_prob_col = find_col(
    faith,
    [
        "hotspot_only_probability",
        "hotspot_only_prob",
        "sufficiency_probability",
        "hotspot_probability",
        "hotspot_only_prediction"
    ],
    contains=["hotspot", "only"]
)


print("\nDetected columns:")
print("Hotspot drop :", hotspot_drop_col)
print("Random drop  :", random_drop_col)
print("Original prob:", original_prob_col)
print("Hotspot-only :", sufficiency_prob_col)


if hotspot_drop_col is None:
    raise KeyError(
        "Could not detect hotspot-drop column.\n"
        f"Available columns:\n{faith.columns.tolist()}"
    )

if random_drop_col is None:
    raise KeyError(
        "Could not detect random-drop column.\n"
        f"Available columns:\n{faith.columns.tolist()}"
    )


# ======================================================================
# CALCULATE FAITHFULNESS EFFECT
#
# effect = hotspot ablation drop - matched random ablation drop
# ======================================================================

faith["faithfulness_effect"] = (
    faith[hotspot_drop_col].astype(float)
    -
    faith[random_drop_col].astype(float)
)


# ======================================================================
# SUMMARY FOR PANEL C
# ======================================================================

summary_rows = []

for model in MODELS:

    for dataset in DATASETS:

        subset = faith[
            (faith["model_plot"] == model)
            &
            (faith["dataset_plot"] == dataset)
        ]["faithfulness_effect"].dropna()


        n = len(subset)

        mean = subset.mean()

        sd = subset.std(ddof=1)

        se = (
            sd / np.sqrt(n)
            if n > 1
            else np.nan
        )

        ci95 = (
            1.96 * se
            if n > 1
            else np.nan
        )


        summary_rows.append({
            "model": model,
            "dataset": dataset,
            "n": n,
            "mean": mean,
            "ci95": ci95
        })


summary = pd.DataFrame(
    summary_rows
)

print("\nFaithfulness-effect summary:")
display(summary)


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(14.8, 9.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.075,
    right=0.985,
    bottom=0.085,
    top=0.94,

    wspace=0.27,
    hspace=0.38
)


axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])


# ======================================================================
# FUNCTION FOR PANELS A/B
# ======================================================================

def perturbation_panel(
    ax,
    dataset,
    panel_letter
):

    positions = np.arange(
        len(MODELS)
    )

    offset = 0.16

    box_width = 0.25


    for i, model in enumerate(MODELS):

        sub = faith[
            (faith["model_plot"] == model)
            &
            (faith["dataset_plot"] == dataset)
        ]


        hotspot = (
            sub[hotspot_drop_col]
            .dropna()
            .astype(float)
            .to_numpy()
        )

        random = (
            sub[random_drop_col]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        # --------------------------------------------------------------
        # hotspot ablation
        # --------------------------------------------------------------

        bp1 = ax.boxplot(
            hotspot,

            positions=[i - offset],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.7,
                color="tab:orange"
            ),

            boxprops=dict(
                linewidth=1.0,
                facecolor="0.82",
                edgecolor="black"
            ),

            whiskerprops=dict(
                linewidth=1.0,
                color="black"
            ),

            capprops=dict(
                linewidth=1.0,
                color="black"
            )
        )


        # --------------------------------------------------------------
        # random ablation
        # --------------------------------------------------------------

        bp2 = ax.boxplot(
            random,

            positions=[i + offset],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.7,
                color="tab:orange"
            ),

            boxprops=dict(
                linewidth=1.0,
                facecolor="white",
                edgecolor="black"
            ),

            whiskerprops=dict(
                linewidth=1.0,
                color="black"
            ),

            capprops=dict(
                linewidth=1.0,
                color="black"
            )
        )


        # --------------------------------------------------------------
        # light raw points
        # --------------------------------------------------------------

        rng = np.random.default_rng(
            100 + i
        )


        if len(hotspot) > 0:

            jitter = rng.normal(
                i - offset,
                0.025,
                len(hotspot)
            )

            ax.scatter(
                jitter,
                hotspot,

                s=8,

                alpha=0.16,

                color="tab:blue",

                linewidths=0,

                zorder=1
            )


        if len(random) > 0:

            jitter = rng.normal(
                i + offset,
                0.025,
                len(random)
            )

            ax.scatter(
                jitter,
                random,

                s=8,

                alpha=0.16,

                color="tab:orange",

                linewidths=0,

                zorder=1
            )


    # --------------------------------------------------------------
    # zero reference
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",

        linewidth=1.1,

        color="tab:blue",

        alpha=0.8
    )


    ax.set_xticks(
        positions
    )

    ax.set_xticklabels(
        MODEL_LABELS,
        rotation=18,
        ha="right"
    )


    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")


    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")


    ax.set_ylabel(
        "Decrease in predicted CPP probability",
        fontsize=12.5,
        fontweight="bold"
    )


    # --------------------------------------------------------------
    # title
    # --------------------------------------------------------------

    title = (
        "Internal test CPPs"
        if dataset == "internal_test"
        else "KELM external CPPs"
    )

    ax.set_title(
        title,

        fontsize=13,

        fontweight="bold",

        pad=9
    )


    # --------------------------------------------------------------
    # legend ABOVE panel
    # --------------------------------------------------------------

    legend_handles = [

        Patch(
            facecolor="0.82",
            edgecolor="black",
            label="Consensus-hotspot ablation"
        ),

        Patch(
            facecolor="white",
            edgecolor="black",
            label="Matched random ablation"
        )
    ]


    leg = ax.legend(
        handles=legend_handles,

        frameon=False,

        loc="upper right",

        bbox_to_anchor=(1.00, 1.02),

        fontsize=9.5,

        handlelength=1.7
    )


    for text in leg.get_texts():
        text.set_fontweight("bold")


    # --------------------------------------------------------------
    # panel letter
    # --------------------------------------------------------------

    ax.text(
        -0.13,
        1.06,

        panel_letter,

        transform=ax.transAxes,

        fontsize=22,

        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False
    )


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ======================================================================
# PANEL A
# ======================================================================

perturbation_panel(
    axA,
    "internal_test",
    "A"
)


# ======================================================================
# PANEL B
# ======================================================================

perturbation_panel(
    axB,
    "kelm_external",
    "B"
)


# ======================================================================
# MATCH A/B Y-LIMITS
# ======================================================================

combined_values = faith[
    [
        hotspot_drop_col,
        random_drop_col
    ]
].astype(float).values.flatten()

combined_values = combined_values[
    np.isfinite(combined_values)
]


if len(combined_values):

    q_low = np.quantile(
        combined_values,
        0.005
    )

    q_high = np.quantile(
        combined_values,
        0.995
    )

    span = q_high - q_low

    ymin = min(
        -0.10,
        q_low - 0.12 * span
    )

    ymax = max(
        0.20,
        q_high + 0.12 * span
    )

    axA.set_ylim(
        ymin,
        ymax
    )

    axB.set_ylim(
        ymin,
        ymax
    )


# ======================================================================
# PANEL C — EFFECT + 95% CI
# ======================================================================

x = np.arange(
    len(MODELS)
)

offset = 0.12


internal_summary = (
    summary[
        summary["dataset"] == "internal_test"
    ]
    .set_index("model")
    .loc[MODELS]
)


external_summary = (
    summary[
        summary["dataset"] == "kelm_external"
    ]
    .set_index("model")
    .loc[MODELS]
)


axC.errorbar(
    x - offset,

    internal_summary["mean"],

    yerr=internal_summary["ci95"],

    fmt="o",

    markersize=7,

    capsize=4,

    elinewidth=1.5,

    markeredgewidth=0.8,

    label="Internal test"
)


axC.errorbar(
    x + offset,

    external_summary["mean"],

    yerr=external_summary["ci95"],

    fmt="s",

    markersize=7,

    capsize=4,

    elinewidth=1.5,

    markeredgewidth=0.8,

    label="KELM external"
)


axC.axhline(
    0,

    linestyle="--",

    linewidth=1.1,

    color="0.45"
)


axC.set_xticks(
    x
)

axC.set_xticklabels(
    MODEL_LABELS,
    rotation=18,
    ha="right"
)


for tick in axC.get_xticklabels():
    tick.set_fontweight("bold")

for tick in axC.get_yticklabels():
    tick.set_fontweight("bold")


axC.set_ylabel(
    "Hotspot drop − random drop",
    fontsize=12.5,
    fontweight="bold"
)


axC.set_title(
    "Faithfulness effect with 95% confidence intervals",

    fontsize=13,

    fontweight="bold",

    pad=10
)


legC = axC.legend(
    frameon=False,

    loc="upper left",

    fontsize=9.5
)

for text in legC.get_texts():
    text.set_fontweight("bold")


axC.text(
    -0.13,
    1.06,

    "C",

    transform=axC.transAxes,

    fontsize=22,

    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ======================================================================
# PANEL D — PREDICTIVE SUFFICIENCY
# ======================================================================

if (
    original_prob_col is not None
    and
    sufficiency_prob_col is not None
):

    marker_map = {
        "ESM2_320": "o",
        "ESM2_640": "s",
        "ESM2_1280": "^",
        "ProtT5": "D"
    }


    dataset_color = {
        "internal_test": "tab:blue",
        "kelm_external": "tab:orange"
    }


    for dataset in DATASETS:

        for model in MODELS:

            sub = faith[
                (faith["dataset_plot"] == dataset)
                &
                (faith["model_plot"] == model)
            ]


            axD.scatter(
                sub[original_prob_col],
                sub[sufficiency_prob_col],

                marker=marker_map[model],

                s=21,

                facecolors="none",

                edgecolors=dataset_color[dataset],

                linewidths=0.65,

                alpha=0.43
            )


    # identity line
    axD.plot(
        [0, 1],
        [0, 1],

        linestyle="--",

        linewidth=1.4,

        color="0.45"
    )


    axD.set_xlim(
        0,
        1.01
    )

    axD.set_ylim(
        0,
        1.01
    )


    axD.set_xlabel(
        "Original predicted CPP probability",
        fontsize=12.5,
        fontweight="bold"
    )

    axD.set_ylabel(
        "Hotspot-only predicted CPP probability",
        fontsize=12.5,
        fontweight="bold"
    )


    for tick in axD.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in axD.get_yticklabels():
        tick.set_fontweight("bold")


    axD.set_title(
        "Predictive sufficiency of consensus hotspots",

        fontsize=13,

        fontweight="bold",

        pad=10
    )


    # --------------------------------------------------------------
    # Dataset legend
    # --------------------------------------------------------------

    dataset_handles = [

        Line2D(
            [0],
            [0],

            marker="o",

            linestyle="none",

            markerfacecolor="none",

            markeredgecolor="tab:blue",

            markersize=7,

            label="Internal test"
        ),

        Line2D(
            [0],
            [0],

            marker="o",

            linestyle="none",

            markerfacecolor="none",

            markeredgecolor="tab:orange",

            markersize=7,

            label="KELM external"
        )
    ]


    leg_dataset = axD.legend(
        handles=dataset_handles,

        title="Dataset",

        loc="upper left",

        frameon=False,

        fontsize=9,

        title_fontsize=9.5
    )


    leg_dataset.get_title().set_fontweight(
        "bold"
    )

    for text in leg_dataset.get_texts():
        text.set_fontweight("bold")


    axD.add_artist(
        leg_dataset
    )


    # --------------------------------------------------------------
    # Model legend
    # --------------------------------------------------------------

    model_handles = []

    for model, label in zip(
        MODELS,
        MODEL_LABELS
    ):

        model_handles.append(

            Line2D(
                [0],
                [0],

                marker=marker_map[model],

                linestyle="none",

                color="black",

                markerfacecolor="black",

                markersize=6,

                label=label
            )

        )


    leg_model = axD.legend(
        handles=model_handles,

        title="Model",

        loc="lower right",

        frameon=False,

        fontsize=8.5,

        title_fontsize=9.5
    )


    leg_model.get_title().set_fontweight(
        "bold"
    )

    for text in leg_model.get_texts():
        text.set_fontweight("bold")


else:

    axD.text(
        0.5,
        0.5,

        "Sufficiency probability columns\nnot detected",

        ha="center",
        va="center",

        fontsize=13,

        fontweight="bold",

        transform=axD.transAxes
    )


axD.text(
    -0.13,
    1.06,

    "D",

    transform=axD.transAxes,

    fontsize=22,

    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


axD.spines["top"].set_visible(False)
axD.spines["right"].set_visible(False)


# ======================================================================
# OPTIONAL OVERALL TITLE
#
# For the manuscript I recommend leaving this OFF because the figure
# caption already provides the title.
# ======================================================================

# fig.suptitle(
#     "Perturbation-Based Faithfulness of Consensus CPP Hotspots",
#     fontsize=16,
#     fontweight="bold",
#     y=0.992
# )


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_POLISHED.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_POLISHED.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_POLISHED.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.04
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 5 GENERATED SUCCESSFULLY")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/figures_main/Figure_5_Consensus_Hotspot_Faithfulness_POLISHED.png"
)

In [ ]:
# ======================================================================
# FIGURE 5 — COMPACT MANUSCRIPT VERSION
# Panel letters fully visible
#
# A = Internal perturbation
# B = KELM perturbation
# C = Faithfulness effect
# D = Predictive sufficiency
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10.5,

    "axes.labelsize": 12,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(13.8, 7.2)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    # slightly larger left margin so A/C are not clipped
    left=0.095,
    right=0.985,

    bottom=0.085,
    top=0.965,

    wspace=0.22,
    hspace=0.30
)


axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])


# ======================================================================
# FUNCTION FOR PANELS A/B
# ======================================================================

def perturbation_panel(
    ax,
    dataset,
    panel_letter
):

    positions = np.arange(
        len(MODELS)
    )

    offset = 0.15
    box_width = 0.24


    for i, model in enumerate(MODELS):

        sub = faith[
            (faith["model_plot"] == model)
            &
            (faith["dataset_plot"] == dataset)
        ]


        hotspot = (
            sub[hotspot_drop_col]
            .dropna()
            .astype(float)
            .to_numpy()
        )

        random = (
            sub[random_drop_col]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        # --------------------------------------------------------------
        # Consensus-hotspot ablation
        # --------------------------------------------------------------

        ax.boxplot(
            hotspot,

            positions=[i - offset],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
                color="tab:orange"
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="0.82",
                edgecolor="black"
            ),

            whiskerprops=dict(
                linewidth=0.9,
                color="black"
            ),

            capprops=dict(
                linewidth=0.9,
                color="black"
            )
        )


        # --------------------------------------------------------------
        # Matched random ablation
        # --------------------------------------------------------------

        ax.boxplot(
            random,

            positions=[i + offset],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
                color="tab:orange"
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="white",
                edgecolor="black"
            ),

            whiskerprops=dict(
                linewidth=0.9,
                color="black"
            ),

            capprops=dict(
                linewidth=0.9,
                color="black"
            )
        )


        # --------------------------------------------------------------
        # Light raw points
        # --------------------------------------------------------------

        rng = np.random.default_rng(
            100 + i
        )


        if len(hotspot) > 0:

            jitter = rng.normal(
                i - offset,
                0.022,
                len(hotspot)
            )

            ax.scatter(
                jitter,
                hotspot,

                s=7,
                alpha=0.13,
                color="tab:blue",
                linewidths=0,
                zorder=1
            )


        if len(random) > 0:

            jitter = rng.normal(
                i + offset,
                0.022,
                len(random)
            )

            ax.scatter(
                jitter,
                random,

                s=7,
                alpha=0.13,
                color="tab:orange",
                linewidths=0,
                zorder=1
            )


    # --------------------------------------------------------------
    # Zero reference
    # --------------------------------------------------------------

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1.0,
        color="tab:blue",
        alpha=0.75
    )


    # --------------------------------------------------------------
    # X axis
    # --------------------------------------------------------------

    ax.set_xticks(
        positions
    )

    ax.set_xticklabels(
        MODEL_LABELS,
        rotation=17,
        ha="right"
    )


    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")


    # --------------------------------------------------------------
    # Y label
    # --------------------------------------------------------------

    ax.set_ylabel(
        "Decrease in predicted CPP probability",
        fontsize=11.5,
        fontweight="bold",
        labelpad=4
    )


    # --------------------------------------------------------------
    # Legend
    # --------------------------------------------------------------

    legend_handles = [

        Patch(
            facecolor="0.82",
            edgecolor="black",
            label="Consensus-hotspot ablation"
        ),

        Patch(
            facecolor="white",
            edgecolor="black",
            label="Matched random ablation"
        )
    ]


    leg = ax.legend(
        handles=legend_handles,

        frameon=False,

        loc="upper right",

        bbox_to_anchor=(0.99, 0.995),

        fontsize=8.3,

        handlelength=1.6,

        borderaxespad=0.1
    )


    for text in leg.get_texts():
        text.set_fontweight("bold")


    # --------------------------------------------------------------
    # PANEL LETTER
    # moved inward so it is fully visible
    # --------------------------------------------------------------

    ax.text(
        -0.09,
        1.06,

        panel_letter,

        transform=ax.transAxes,

        fontsize=21,
        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False
    )


    # --------------------------------------------------------------
    # Clean spines
    # --------------------------------------------------------------

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ======================================================================
# PANEL A
# ======================================================================

perturbation_panel(
    axA,
    "internal_test",
    "A"
)


# ======================================================================
# PANEL B
# ======================================================================

perturbation_panel(
    axB,
    "kelm_external",
    "B"
)


# ======================================================================
# SAME Y LIMITS FOR A/B
# ======================================================================

combined_values = faith[
    [
        hotspot_drop_col,
        random_drop_col
    ]
].astype(float).values.flatten()


combined_values = combined_values[
    np.isfinite(combined_values)
]


if len(combined_values):

    q_low = np.quantile(
        combined_values,
        0.005
    )

    q_high = np.quantile(
        combined_values,
        0.995
    )

    span = q_high - q_low


    ymin = min(
        -0.10,
        q_low - 0.10 * span
    )

    ymax = max(
        0.20,
        q_high + 0.10 * span
    )


    axA.set_ylim(
        ymin,
        ymax
    )

    axB.set_ylim(
        ymin,
        ymax
    )


# ======================================================================
# PANEL C — FAITHFULNESS EFFECT
# ======================================================================

x = np.arange(
    len(MODELS)
)

offset = 0.11


internal_summary = (
    summary[
        summary["dataset"] == "internal_test"
    ]
    .set_index("model")
    .loc[MODELS]
)


external_summary = (
    summary[
        summary["dataset"] == "kelm_external"
    ]
    .set_index("model")
    .loc[MODELS]
)


axC.errorbar(
    x - offset,

    internal_summary["mean"],

    yerr=internal_summary["ci95"],

    fmt="o",

    markersize=6.5,

    capsize=3.5,

    elinewidth=1.3,

    markeredgewidth=0.8,

    label="Internal test"
)


axC.errorbar(
    x + offset,

    external_summary["mean"],

    yerr=external_summary["ci95"],

    fmt="s",

    markersize=6.5,

    capsize=3.5,

    elinewidth=1.3,

    markeredgewidth=0.8,

    label="KELM external"
)


axC.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


axC.set_xticks(
    x
)

axC.set_xticklabels(
    MODEL_LABELS,
    rotation=17,
    ha="right"
)


for tick in axC.get_xticklabels():
    tick.set_fontweight("bold")

for tick in axC.get_yticklabels():
    tick.set_fontweight("bold")


axC.set_ylabel(
    "Hotspot drop − random drop",
    fontsize=11.5,
    fontweight="bold",
    labelpad=4
)


# --------------------------------------------------------------
# C legend
# --------------------------------------------------------------

legC = axC.legend(
    frameon=False,

    loc="upper left",

    bbox_to_anchor=(0.01, 0.995),

    fontsize=8.5,

    borderaxespad=0.1
)


for text in legC.get_texts():
    text.set_fontweight("bold")


# --------------------------------------------------------------
# C letter
# --------------------------------------------------------------

axC.text(
    -0.09,
    1.06,

    "C",

    transform=axC.transAxes,

    fontsize=21,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ======================================================================
# PANEL D — PREDICTIVE SUFFICIENCY
# ======================================================================

if (
    original_prob_col is not None
    and
    sufficiency_prob_col is not None
):

    marker_map = {
        "ESM2_320": "o",
        "ESM2_640": "s",
        "ESM2_1280": "^",
        "ProtT5": "D"
    }


    dataset_color = {
        "internal_test": "tab:blue",
        "kelm_external": "tab:orange"
    }


    for dataset in DATASETS:

        for model in MODELS:

            sub = faith[
                (faith["dataset_plot"] == dataset)
                &
                (faith["model_plot"] == model)
            ]


            axD.scatter(
                sub[original_prob_col],
                sub[sufficiency_prob_col],

                marker=marker_map[model],

                s=18,

                facecolors="none",

                edgecolors=dataset_color[dataset],

                linewidths=0.55,

                alpha=0.38
            )


    # identity line
    axD.plot(
        [0, 1],
        [0, 1],

        linestyle="--",

        linewidth=1.2,

        color="0.45"
    )


    axD.set_xlim(
        0,
        1.01
    )

    axD.set_ylim(
        0,
        1.01
    )


    axD.set_xlabel(
        "Original predicted CPP probability",
        fontsize=11.5,
        fontweight="bold",
        labelpad=4
    )

    axD.set_ylabel(
        "Hotspot-only predicted CPP probability",
        fontsize=11.5,
        fontweight="bold",
        labelpad=4
    )


    for tick in axD.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in axD.get_yticklabels():
        tick.set_fontweight("bold")


    # --------------------------------------------------------------
    # Dataset legend
    # --------------------------------------------------------------

    dataset_handles = [

        Line2D(
            [0],
            [0],

            marker="o",

            linestyle="none",

            markerfacecolor="none",

            markeredgecolor="tab:blue",

            markersize=6,

            label="Internal test"
        ),

        Line2D(
            [0],
            [0],

            marker="o",

            linestyle="none",

            markerfacecolor="none",

            markeredgecolor="tab:orange",

            markersize=6,

            label="KELM external"
        )
    ]


    leg_dataset = axD.legend(
        handles=dataset_handles,

        title="Dataset",

        loc="upper left",

        bbox_to_anchor=(0.01, 0.99),

        frameon=False,

        fontsize=8.2,

        title_fontsize=8.5,

        borderaxespad=0
    )


    leg_dataset.get_title().set_fontweight(
        "bold"
    )


    for text in leg_dataset.get_texts():
        text.set_fontweight("bold")


    axD.add_artist(
        leg_dataset
    )


    # --------------------------------------------------------------
    # Model legend
    # --------------------------------------------------------------

    model_handles = []


    for model, label in zip(
        MODELS,
        MODEL_LABELS
    ):

        model_handles.append(

            Line2D(
                [0],
                [0],

                marker=marker_map[model],

                linestyle="none",

                color="black",

                markerfacecolor="black",

                markersize=5.5,

                label=label
            )
        )


    leg_model = axD.legend(
        handles=model_handles,

        title="Model",

        loc="lower right",

        frameon=False,

        fontsize=8,

        title_fontsize=8.5
    )


    leg_model.get_title().set_fontweight(
        "bold"
    )


    for text in leg_model.get_texts():
        text.set_fontweight("bold")


else:

    axD.text(
        0.5,
        0.5,

        "Sufficiency probability columns\nnot detected",

        ha="center",
        va="center",

        fontsize=12,

        fontweight="bold",

        transform=axD.transAxes
    )


# --------------------------------------------------------------
# D letter
# --------------------------------------------------------------

axD.text(
    -0.09,
    1.06,

    "D",

    transform=axD.transAxes,

    fontsize=21,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


axD.spines["top"].set_visible(False)
axD.spines["right"].set_visible(False)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_COMPACT_v2.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_COMPACT_v2.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_COMPACT_v2.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.04
)


plt.show()
plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 5 COMPACT v2 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 5 — FINAL COMPACT MANUSCRIPT VERSION
#
# A/B y-labels split into two lines
# D y-label split into two lines
# A/B/C/D outside plotting regions
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10.5,

    "axes.labelsize": 12,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,

    "xtick.major.size": 4,
    "ytick.major.size": 4,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(10, 6)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    # enough margin for outside A/C
    left=0.105,
    right=0.985,

    bottom=0.085,
    top=0.965,

    wspace=0.24,
    hspace=0.30
)


axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])


# ======================================================================
# OUTSIDE PANEL LETTER
# ======================================================================

def add_panel_letter(ax, letter):

    ax.text(
        -0.105,
        1.035,

        letter,

        transform=ax.transAxes,

        fontsize=21,
        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,
        zorder=100
    )


# ======================================================================
# FUNCTION FOR PANELS A/B
# ======================================================================

def perturbation_panel(
    ax,
    dataset
):

    positions = np.arange(
        len(MODELS)
    )

    offset = 0.15
    box_width = 0.24


    for i, model in enumerate(MODELS):

        sub = faith[
            (faith["model_plot"] == model)
            &
            (faith["dataset_plot"] == dataset)
        ]


        hotspot = (
            sub[hotspot_drop_col]
            .dropna()
            .astype(float)
            .to_numpy()
        )

        random = (
            sub[random_drop_col]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        # --------------------------------------------------------------
        # Consensus-hotspot ablation
        # --------------------------------------------------------------

        ax.boxplot(
            hotspot,

            positions=[i - offset],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
                color="tab:orange"
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="0.82",
                edgecolor="black"
            ),

            whiskerprops=dict(
                linewidth=0.9,
                color="black"
            ),

            capprops=dict(
                linewidth=0.9,
                color="black"
            )
        )


        # --------------------------------------------------------------
        # Matched random ablation
        # --------------------------------------------------------------

        ax.boxplot(
            random,

            positions=[i + offset],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
                color="tab:orange"
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="white",
                edgecolor="black"
            ),

            whiskerprops=dict(
                linewidth=0.9,
                color="black"
            ),

            capprops=dict(
                linewidth=0.9,
                color="black"
            )
        )


        # --------------------------------------------------------------
        # Raw points
        # --------------------------------------------------------------

        rng = np.random.default_rng(
            100 + i
        )


        if len(hotspot) > 0:

            jitter = rng.normal(
                i - offset,
                0.022,
                len(hotspot)
            )

            ax.scatter(
                jitter,
                hotspot,

                s=7,
                alpha=0.13,

                color="tab:blue",

                linewidths=0,
                zorder=1
            )


        if len(random) > 0:

            jitter = rng.normal(
                i + offset,
                0.022,
                len(random)
            )

            ax.scatter(
                jitter,
                random,

                s=7,
                alpha=0.13,

                color="tab:orange",

                linewidths=0,
                zorder=1
            )


    # --------------------------------------------------------------
    # Zero line
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",

        linewidth=1.0,

        color="tab:blue",

        alpha=0.75
    )


    # --------------------------------------------------------------
    # X axis
    # --------------------------------------------------------------

    ax.set_xticks(
        positions
    )

    ax.set_xticklabels(
        MODEL_LABELS,

        rotation=17,

        ha="right"
    )


    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")


    # --------------------------------------------------------------
    # IMPORTANT:
    # Two-line y-axis label
    # --------------------------------------------------------------

    ax.set_ylabel(
        "Decrease in predicted\nCPP probability",

        fontsize=11.5,

        fontweight="bold",

        labelpad=7,

        linespacing=1.15
    )


    # --------------------------------------------------------------
    # Legend
    # --------------------------------------------------------------

    legend_handles = [

        Patch(
            facecolor="0.82",
            edgecolor="black",
            label="Consensus-hotspot ablation"
        ),

        Patch(
            facecolor="white",
            edgecolor="black",
            label="Matched random ablation"
        )
    ]


    leg = ax.legend(
        handles=legend_handles,

        frameon=False,

        loc="upper right",

        bbox_to_anchor=(
            0.99,
            0.995
        ),

        fontsize=8.3,

        handlelength=1.6,

        borderaxespad=0.1
    )


    for text in leg.get_texts():
        text.set_fontweight("bold")


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ======================================================================
# PANEL A
# ======================================================================

perturbation_panel(
    axA,
    "internal_test"
)


# ======================================================================
# PANEL B
# ======================================================================

perturbation_panel(
    axB,
    "kelm_external"
)


# ======================================================================
# COMMON Y RANGE FOR A/B
# ======================================================================

combined_values = faith[
    [
        hotspot_drop_col,
        random_drop_col
    ]
].astype(float).values.flatten()


combined_values = combined_values[
    np.isfinite(combined_values)
]


if len(combined_values):

    q_low = np.quantile(
        combined_values,
        0.005
    )

    q_high = np.quantile(
        combined_values,
        0.995
    )

    span = (
        q_high
        - q_low
    )


    ymin = min(
        -0.10,
        q_low - 0.10 * span
    )

    ymax = max(
        0.20,
        q_high + 0.10 * span
    )


    axA.set_ylim(
        ymin,
        ymax
    )

    axB.set_ylim(
        ymin,
        ymax
    )


# ======================================================================
# PANEL C — FAITHFULNESS EFFECT
# ======================================================================

x = np.arange(
    len(MODELS)
)

offset = 0.11


internal_summary = (
    summary[
        summary["dataset"]
        == "internal_test"
    ]
    .set_index("model")
    .loc[MODELS]
)


external_summary = (
    summary[
        summary["dataset"]
        == "kelm_external"
    ]
    .set_index("model")
    .loc[MODELS]
)


axC.errorbar(
    x - offset,

    internal_summary["mean"],

    yerr=internal_summary["ci95"],

    fmt="o",

    markersize=6.5,

    capsize=3.5,

    elinewidth=1.3,

    markeredgewidth=0.8,

    label="Internal test"
)


axC.errorbar(
    x + offset,

    external_summary["mean"],

    yerr=external_summary["ci95"],

    fmt="s",

    markersize=6.5,

    capsize=3.5,

    elinewidth=1.3,

    markeredgewidth=0.8,

    label="KELM external"
)


axC.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


axC.set_xticks(
    x
)

axC.set_xticklabels(
    MODEL_LABELS,

    rotation=17,

    ha="right"
)


for tick in axC.get_xticklabels():
    tick.set_fontweight("bold")

for tick in axC.get_yticklabels():
    tick.set_fontweight("bold")


axC.set_ylabel(
    "Hotspot drop \n− random drop",

    fontsize=11.5,

    fontweight="bold",

    labelpad=6
)


# C legend
legC = axC.legend(
    frameon=False,

    loc="upper left",

    bbox_to_anchor=(
        0.01,
        0.995
    ),

    fontsize=8.5,

    borderaxespad=0.1
)


for text in legC.get_texts():
    text.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ======================================================================
# PANEL D — PREDICTIVE SUFFICIENCY
# ======================================================================

if (
    original_prob_col is not None
    and
    sufficiency_prob_col is not None
):

    marker_map = {
        "ESM2_320": "o",
        "ESM2_640": "s",
        "ESM2_1280": "^",
        "ProtT5": "D"
    }


    dataset_color = {
        "internal_test": "tab:blue",
        "kelm_external": "tab:orange"
    }


    for dataset in DATASETS:

        for model in MODELS:

            sub = faith[
                (faith["dataset_plot"] == dataset)
                &
                (faith["model_plot"] == model)
            ]


            axD.scatter(
                sub[original_prob_col],
                sub[sufficiency_prob_col],

                marker=marker_map[model],

                s=18,

                facecolors="none",

                edgecolors=dataset_color[
                    dataset
                ],

                linewidths=0.55,

                alpha=0.38
            )


    # --------------------------------------------------------------
    # Identity line
    # --------------------------------------------------------------

    axD.plot(
        [0, 1],
        [0, 1],

        linestyle="--",

        linewidth=1.2,

        color="0.45"
    )


    axD.set_xlim(
        0,
        1.01
    )

    axD.set_ylim(
        0,
        1.01
    )


    axD.set_xlabel(
        "Original predicted CPP probability",

        fontsize=11.5,

        fontweight="bold",

        labelpad=4
    )


    # --------------------------------------------------------------
    # Two-line D y-label
    # --------------------------------------------------------------

    axD.set_ylabel(
        "Hotspot-only predicted\nCPP probability",

        fontsize=11.5,

        fontweight="bold",

        labelpad=7,

        linespacing=1.15
    )


    for tick in axD.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in axD.get_yticklabels():
        tick.set_fontweight("bold")


    # --------------------------------------------------------------
    # Dataset legend
    # --------------------------------------------------------------

    dataset_handles = [

        Line2D(
            [0],
            [0],

            marker="o",

            linestyle="none",

            markerfacecolor="none",

            markeredgecolor="tab:blue",

            markersize=6,

            label="Internal test"
        ),

        Line2D(
            [0],
            [0],

            marker="o",

            linestyle="none",

            markerfacecolor="none",

            markeredgecolor="tab:orange",

            markersize=6,

            label="KELM external"
        )
    ]


    leg_dataset = axD.legend(
        handles=dataset_handles,

        title="Dataset",

        loc="upper left",

        bbox_to_anchor=(
            0.01,
            0.99
        ),

        frameon=False,

        fontsize=8.2,

        title_fontsize=8.5,

        borderaxespad=0
    )


    leg_dataset.get_title().set_fontweight(
        "bold"
    )


    for text in leg_dataset.get_texts():
        text.set_fontweight("bold")


    axD.add_artist(
        leg_dataset
    )


    # --------------------------------------------------------------
    # Model legend
    # --------------------------------------------------------------

    model_handles = []


    for model, label in zip(
        MODELS,
        MODEL_LABELS
    ):

        model_handles.append(

            Line2D(
                [0],
                [0],

                marker=marker_map[model],

                linestyle="none",

                color="black",

                markerfacecolor="black",

                markersize=5.5,

                label=label
            )
        )


    leg_model = axD.legend(
        handles=model_handles,

        title="Model",

        loc="lower right",

        frameon=False,

        fontsize=8,

        title_fontsize=8.5
    )


    leg_model.get_title().set_fontweight(
        "bold"
    )


    for text in leg_model.get_texts():
        text.set_fontweight("bold")


else:

    axD.text(
        0.5,
        0.5,

        "Sufficiency probability columns\nnot detected",

        ha="center",
        va="center",

        fontsize=12,

        fontweight="bold",

        transform=axD.transAxes
    )


axD.spines["top"].set_visible(False)
axD.spines["right"].set_visible(False)


# ======================================================================
# PANEL LETTERS — OUTSIDE
# ======================================================================

add_panel_letter(
    axA,
    "A"
)

add_panel_letter(
    axB,
    "B"
)

add_panel_letter(
    axC,
    "C"
)

add_panel_letter(
    axD,
    "D"
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_FINAL_OUTSIDE.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_FINAL_OUTSIDE.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_5_Consensus_Hotspot_Faithfulness_FINAL_OUTSIDE.svg"
)


fig.savefig(
    png_file,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    pdf_file,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    svg_file,

    bbox_inches="tight",

    pad_inches=0.05
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FINAL FIGURE 5 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
!find /content/drive/MyDrive/pLM4CPP_XAI_2026 -type f | \
grep -Ei "conservation|cross_plm|jaccard|support|enrichment|figure_6|hotspot.*conservation|pairwise" | \
sort

In [ ]:
!find /content/drive/MyDrive/pLM4CPP_XAI_2026/07_results -type f | \
grep -Ei "cross_plm|jaccard|support|conservation|figure_6|enrichment" | \
sort

In [ ]:
# ======================================================================
# FIGURE 6 — COMPACT MANUSCRIPT VERSION
# Cross-PLM Conservation of Consensus CPP Hotspots
#
# A = Pairwise CPP hotspot Jaccard matrices
# B = Cross-PLM residue-support distribution
# C = CPP enrichment with increasing PLM agreement
# D = Sequence-level hotspot conservation
#
# Style matched to final Figure 5:
# - compact 2 × 2
# - no panel titles
# - bold readable text
# - A/B/C/D outside panels
# - minimal white space
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

SI_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Panel A
A_INTERNAL = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_internal_test_CPP.csv"
)

A_KELM = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_kelm_external_CPP.csv"
)


# Panel B
B_FILE = (
    TABLE_DIR
    / "cross_plm_hotspot_support_distribution.csv"
)


# Panel C
C_FILE = (
    TABLE_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)


# Panel D
D_FILE = (
    SI_DIR
    / "cross_plm_hotspot_conservation_per_sequence.csv"
)


# ======================================================================
# VERIFY
# ======================================================================

for f in [
    A_INTERNAL,
    A_KELM,
    B_FILE,
    C_FILE,
    D_FILE
]:

    if not f.exists():

        raise FileNotFoundError(
            f"Missing source file:\n{f}"
        )

    print("✓", f)


# ======================================================================
# HELPERS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def find_col(
    df,
    candidates,
    contains=None
):

    cols = list(
        df.columns
    )

    for x in candidates:

        if x in cols:
            return x

    if contains:

        for c in cols:

            if all(
                k.lower() in c.lower()
                for k in contains
            ):

                return c

    return None


def boolify(series):

    return (
        series
        .astype(str)
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes"
            ]
        )
    )


def style_ticks(ax):

    ax.tick_params(
        axis="both",
        width=1.0,
        length=4
    )

    for lab in ax.get_xticklabels():

        lab.set_fontweight(
            "bold"
        )

    for lab in ax.get_yticklabels():

        lab.set_fontweight(
            "bold"
        )


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        10.5,

    "axes.labelsize":
        12,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        9.5,

    "ytick.labelsize":
        9.5,

    "legend.fontsize":
        8.5,

    "axes.linewidth":
        1.1,

    "savefig.dpi":
        600
})


# ======================================================================
# LOAD PANEL A MATRICES
# ======================================================================

mat_internal = pd.read_csv(
    A_INTERNAL,
    index_col=0
)

mat_kelm = pd.read_csv(
    A_KELM,
    index_col=0
)


# standard labels
pretty_model = {

    "ESM2_320":
        "ESM2-320",

    "ESM2_640":
        "ESM2-640",

    "ESM2_1280":
        "ESM2-1280",

    "ProtT5":
        "ProtT5"
}


mat_internal.index = [
    pretty_model.get(
        str(x),
        str(x)
    )
    for x in mat_internal.index
]

mat_internal.columns = [
    pretty_model.get(
        str(x),
        str(x)
    )
    for x in mat_internal.columns
]


mat_kelm.index = [
    pretty_model.get(
        str(x),
        str(x)
    )
    for x in mat_kelm.index
]

mat_kelm.columns = [
    pretty_model.get(
        str(x),
        str(x)
    )
    for x in mat_kelm.columns
]


# ======================================================================
# LOAD B/C/D
# ======================================================================

B = clean_columns(
    pd.read_csv(
        B_FILE
    )
)

C = clean_columns(
    pd.read_csv(
        C_FILE
    )
)

D = clean_columns(
    pd.read_csv(
        D_FILE
    )
)


print("\nPanel B columns:")
print(B.columns.tolist())

print("\nPanel C columns:")
print(C.columns.tolist())

print("\nPanel D columns:")
print(D.columns.tolist())


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(13.8, 7.4)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.085,
    right=0.975,

    bottom=0.095,
    top=0.965,

    wspace=0.25,
    hspace=0.31
)


axA_container = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# container only for A
axA_container.set_axis_off()


# ======================================================================
# PANEL A — TWO JACCARD MATRICES
# ======================================================================

A_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=gs[0, 0],

    width_ratios=[
        1,
        1,
        0.045
    ],

    wspace=0.09
)


axA1 = fig.add_subplot(
    A_grid[0]
)

axA2 = fig.add_subplot(
    A_grid[1]
)

caxA = fig.add_subplot(
    A_grid[2]
)


im1 = axA1.imshow(
    mat_internal.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


axA2.imshow(
    mat_kelm.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


# ----------------------------------------------------------------------
# Heatmap helper
# ----------------------------------------------------------------------

def format_heatmap(
    ax,
    matrix,
    show_y=True
):

    n = len(
        matrix.index
    )


    ax.set_xticks(
        np.arange(n)
    )

    ax.set_yticks(
        np.arange(n)
    )


    ax.set_xticklabels(
        matrix.columns,

        rotation=35,

        ha="right",

        fontsize=7.8,

        fontweight="bold"
    )


    if show_y:

        ax.set_yticklabels(
            matrix.index,

            fontsize=8,

            fontweight="bold"
        )

    else:

        ax.set_yticklabels([])

        ax.tick_params(
            axis="y",
            length=0
        )


    for i in range(
        matrix.shape[0]
    ):

        for j in range(
            matrix.shape[1]
        ):

            val = float(
                matrix.iloc[
                    i,
                    j
                ]
            )

            ax.text(
                j,
                i,

                f"{val:.2f}",

                ha="center",
                va="center",

                fontsize=7.5,

                fontweight="bold",

                color=(
                    "white"
                    if val > 0.55
                    else "black"
                )
            )


format_heatmap(
    axA1,
    mat_internal,
    True
)

format_heatmap(
    axA2,
    mat_kelm,
    False
)


# small dataset labels only
axA1.text(
    0.5,
    1.03,

    "Internal test",

    transform=axA1.transAxes,

    ha="center",

    fontsize=9.5,

    fontweight="bold"
)


axA2.text(
    0.5,
    1.03,

    "KELM external",

    transform=axA2.transAxes,

    ha="center",

    fontsize=9.5,

    fontweight="bold"
)


# colorbar
cbA = fig.colorbar(
    im1,
    cax=caxA
)

cbA.set_label(
    "Mean per-sequence Jaccard",

    fontsize=9.2,

    fontweight="bold",

    labelpad=5
)

cbA.ax.tick_params(
    labelsize=8
)

for t in cbA.ax.get_yticklabels():
    t.set_fontweight("bold")


# ======================================================================
# PANEL B — CROSS-PLM SUPPORT DISTRIBUTION
# ======================================================================

support_col = find_col(
    B,
    [
        "model_support_count",
        "support_count",
        "n_models",
        "plm_support"
    ],
    contains=[
        "support",
        "count"
    ]
)


dataset_col_B = find_col(
    B,
    [
        "dataset",
        "dataset_name"
    ],
    contains=[
        "dataset"
    ]
)


label_col_B = find_col(
    B,
    [
        "label",
        "class",
        "cpp_label"
    ]
)


fraction_col_B = find_col(
    B,
    [
        "fraction",
        "fraction_residues",
        "fraction_hotspot_union_residues",
        "proportion"
    ],
    contains=[
        "fraction"
    ]
)


if support_col is None:

    raise KeyError(
        "Could not detect PLM support-count column in Panel B.\n"
        f"{B.columns.tolist()}"
    )


# ----------------------------------------------------------------------
# If table already contains fractions
# ----------------------------------------------------------------------

if fraction_col_B is not None:

    Bplot = B.copy()

else:

    # reconstruct fractions from rows/counts

    group_cols = [
        support_col
    ]

    if dataset_col_B:
        group_cols.insert(
            0,
            dataset_col_B
        )

    if label_col_B:
        group_cols.insert(
            1 if dataset_col_B else 0,
            label_col_B
        )


    tmp = (
        B.groupby(
            group_cols
        )
        .size()
        .reset_index(
            name="n"
        )
    )


    denom_cols = [
        x
        for x in [
            dataset_col_B,
            label_col_B
        ]
        if x is not None
    ]


    if denom_cols:

        tmp["fraction_auto"] = (

            tmp["n"]
            /
            tmp.groupby(
                denom_cols
            )["n"]
            .transform("sum")
        )

    else:

        tmp["fraction_auto"] = (
            tmp["n"]
            / tmp["n"].sum()
        )


    Bplot = tmp

    fraction_col_B = (
        "fraction_auto"
    )


# ----------------------------------------------------------------------
# make groups
# ----------------------------------------------------------------------

def normalize_dataset(v):

    s = str(v).lower()

    if "internal" in s:
        return "Internal"

    if "kelm" in s or "external" in s:
        return "KELM"

    return str(v)


def normalize_class(v):

    s = str(v).lower()

    if s in [
        "1",
        "true",
        "cpp",
        "positive"
    ]:
        return "CPP"

    if s in [
        "0",
        "false",
        "noncpp",
        "non-cpp",
        "negative"
    ]:
        return "non-CPP"

    return str(v)


if dataset_col_B is not None:

    Bplot["dataset_clean"] = (
        Bplot[
            dataset_col_B
        ]
        .map(
            normalize_dataset
        )
    )

else:

    Bplot["dataset_clean"] = (
        "Internal"
    )


if label_col_B is not None:

    Bplot["class_clean"] = (
        Bplot[
            label_col_B
        ]
        .map(
            normalize_class
        )
    )

else:

    Bplot["class_clean"] = (
        "CPP"
    )


group_order = [
    (
        "Internal",
        "CPP"
    ),
    (
        "Internal",
        "non-CPP"
    ),
    (
        "KELM",
        "CPP"
    ),
    (
        "KELM",
        "non-CPP"
    )
]


support_values = [
    1,
    2,
    3,
    4
]


xB = np.arange(
    4
)

width = 0.18


for k, (
    dataset_name,
    class_name
) in enumerate(
    group_order
):

    vals = []

    for support in support_values:

        rows = Bplot[
            (Bplot["dataset_clean"] == dataset_name)
            &
            (Bplot["class_clean"] == class_name)
            &
            (
                pd.to_numeric(
                    Bplot[support_col],
                    errors="coerce"
                )
                == support
            )
        ]

        vals.append(
            float(
                rows[
                    fraction_col_B
                ].iloc[0]
            )
            if len(rows)
            else 0
        )


    axB.bar(
        xB
        + (
            k - 1.5
        )
        * width,

        vals,

        width,

        label=(
            f"{dataset_name} "
            f"{class_name}"
        )
    )


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    [
        "1 PLM",
        "2 PLMs",
        "3 PLMs",
        "4 PLMs"
    ],

    fontweight="bold"
)


axB.set_ylabel(
    "Fraction of hotspot-union\nresidues",

    fontsize=11.5,

    fontweight="bold",

    labelpad=6
)


style_ticks(
    axB
)


legB = axB.legend(
    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=8.2,

    columnspacing=1.0,

    handlelength=1.3
)

for text in legB.get_texts():
    text.set_fontweight("bold")


axB.spines[
    "top"
].set_visible(False)

axB.spines[
    "right"
].set_visible(False)


# ======================================================================
# PANEL C — CPP ENRICHMENT WITH PLM AGREEMENT
# ======================================================================

threshold_col = find_col(
    C,
    [
        "threshold",
        "support_threshold",
        "plm_support_threshold",
        "min_support"
    ]
)


dataset_col_C = find_col(
    C,
    [
        "dataset",
        "dataset_name"
    ]
)


odds_col = find_col(
    C,
    [
        "odds_ratio",
        "cpp_odds_ratio",
        "enrichment_odds_ratio"
    ],
    contains=[
        "odds",
        "ratio"
    ]
)


fdr_col = find_col(
    C,
    [
        "fdr",
        "fdr_bh",
        "qvalue",
        "q_value"
    ]
)


if odds_col is None:

    raise KeyError(
        "Could not detect odds ratio column in Panel C.\n"
        f"{C.columns.tolist()}"
    )


# ----------------------------------------------------------------------
# detect support level from available columns
# ----------------------------------------------------------------------

if threshold_col is None:

    threshold_col = find_col(
        C,
        [
            "support_count",
            "model_support_count"
        ],
        contains=[
            "support"
        ]
    )


Cplot = C.copy()


if dataset_col_C:

    Cplot["dataset_clean"] = (
        Cplot[
            dataset_col_C
        ]
        .map(
            normalize_dataset
        )
    )

else:

    Cplot[
        "dataset_clean"
    ] = "Internal"


# normalize support labels
def support_label(v):

    s = str(v).lower()

    if "all" in s or s in ["4", "4.0"]:
        return "All 4 PLMs"

    nums = re.findall(
        r"\d+",
        s
    )

    if nums:

        n = int(
            nums[0]
        )

        if n == 2:
            return "≥2 PLMs"

        if n == 3:
            return "≥3 PLMs"

        if n == 4:
            return "All 4 PLMs"

    return str(v)


Cplot[
    "support_clean"
] = Cplot[
    threshold_col
].map(
    support_label
)


c_levels = [
    "≥2 PLMs",
    "≥3 PLMs",
    "All 4 PLMs"
]


xC = np.arange(
    3
)

widthC = 0.32


for offset_i, dataset_name in enumerate(
    [
        "Internal",
        "KELM"
    ]
):

    vals = []

    fdrs = []

    for lev in c_levels:

        row = Cplot[
            (
                Cplot[
                    "dataset_clean"
                ]
                == dataset_name
            )
            &
            (
                Cplot[
                    "support_clean"
                ]
                == lev
            )
        ]


        if len(row):

            vals.append(
                float(
                    row[
                        odds_col
                    ].iloc[0]
                )
            )

            if fdr_col:

                fdrs.append(
                    float(
                        row[
                            fdr_col
                        ].iloc[0]
                    )
                )

            else:

                fdrs.append(
                    np.nan
                )

        else:

            vals.append(
                np.nan
            )

            fdrs.append(
                np.nan
            )


    xpos = (
        xC
        + (
            -widthC / 2
            if offset_i == 0
            else widthC / 2
        )
    )


    bars = axC.bar(
        xpos,
        vals,
        widthC,
        label=(
            "Internal test"
            if dataset_name
            == "Internal"
            else "KELM external"
        )
    )


    # significance
    for bar, p in zip(
        bars,
        fdrs
    ):

        if np.isnan(p):
            text = ""

        elif p < 0.001:
            text = "***"

        elif p < 0.01:
            text = "**"

        elif p < 0.05:
            text = "*"

        else:
            text = "ns"


        if text:

            axC.text(
                bar.get_x()
                + bar.get_width() / 2,

                bar.get_height()
                + 0.06,

                text,

                ha="center",
                va="bottom",

                fontsize=9,

                fontweight="bold"
            )


axC.axhline(
    1,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    c_levels,

    fontweight="bold"
)


axC.set_ylabel(
    "CPP enrichment\nodds ratio",

    fontsize=11.5,

    fontweight="bold",

    labelpad=6
)


style_ticks(
    axC
)


legC = axC.legend(
    frameon=False,

    loc="upper left",

    fontsize=8.5
)

for text in legC.get_texts():
    text.set_fontweight("bold")


axC.spines[
    "top"
].set_visible(False)

axC.spines[
    "right"
].set_visible(False)


# ======================================================================
# PANEL D — SEQUENCE-LEVEL CONSERVATION
# ======================================================================

dataset_col_D = find_col(
    D,
    [
        "dataset",
        "dataset_name"
    ]
)


label_col_D = find_col(
    D,
    [
        "label",
        "class",
        "cpp_label"
    ]
)


jaccard_col = find_col(
    D,
    [
        "mean_pairwise_jaccard",
        "mean_jaccard",
        "pairwise_jaccard"
    ],
    contains=[
        "jaccard"
    ]
)


support3_col = find_col(
    D,
    [
        "fraction_supported_by_at_least_3",
        "fraction_supported_by_3",
        "fraction_at_least_3"
    ],
    contains=[
        "fraction",
        "3"
    ]
)


support4_col = find_col(
    D,
    [
        "fraction_supported_by_all_4",
        "fraction_supported_by_4",
        "fraction_all_4"
    ],
    contains=[
        "fraction",
        "4"
    ]
)


if any(
    x is None
    for x in [
        jaccard_col,
        support3_col,
        support4_col
    ]
):

    raise KeyError(
        "Could not detect Panel D conservation metrics.\n"
        f"{D.columns.tolist()}"
    )


if dataset_col_D:

    D["dataset_clean"] = (
        D[
            dataset_col_D
        ]
        .map(
            normalize_dataset
        )
    )

else:

    D["dataset_clean"] = (
        "Internal"
    )


if label_col_D:

    D["class_clean"] = (
        D[
            label_col_D
        ]
        .map(
            normalize_class
        )
    )

else:

    D["class_clean"] = (
        "CPP"
    )


metrics = [
    (
        jaccard_col,
        "Mean pairwise\nJaccard"
    ),
    (
        support3_col,
        "Supported by\n≥3 PLMs"
    ),
    (
        support4_col,
        "Supported by\nall 4 PLMs"
    )
]


D_groups = [
    (
        "Internal",
        "non-CPP"
    ),
    (
        "Internal",
        "CPP"
    ),
    (
        "KELM",
        "non-CPP"
    ),
    (
        "KELM",
        "CPP"
    )
]


base_positions = np.arange(
    3
)

offsets = [
    -0.27,
    -0.09,
    0.09,
    0.27
]


legend_D_handles = []


for gi, (
    dataset_name,
    class_name
) in enumerate(
    D_groups
):

    group_data = D[
        (
            D[
                "dataset_clean"
            ]
            == dataset_name
        )
        &
        (
            D[
                "class_clean"
            ]
            == class_name
        )
    ]


    values = [

        pd.to_numeric(
            group_data[col],
            errors="coerce"
        )
        .dropna()
        .to_numpy()

        for col, _
        in metrics
    ]


    positions = (
        base_positions
        + offsets[gi]
    )


    bp = axD.boxplot(
        values,

        positions=positions,

        widths=0.16,

        patch_artist=True,

        showfliers=False,

        medianprops=dict(
            color="tab:orange",
            linewidth=1.4
        ),

        boxprops=dict(
            facecolor=(
                "0.82"
                if class_name
                == "CPP"
                else "white"
            ),

            edgecolor="black",

            linewidth=0.8
        ),

        whiskerprops=dict(
            color="black",
            linewidth=0.8
        ),

        capprops=dict(
            color="black",
            linewidth=0.8
        )
    )


    legend_D_handles.append(

        Patch(
            facecolor=(
                "0.82"
                if class_name
                == "CPP"
                else "white"
            ),

            edgecolor="black",

            label=(
                f"{dataset_name} "
                f"{class_name}"
            )
        )
    )


axD.set_xticks(
    base_positions
)

axD.set_xticklabels(
    [
        label
        for _,
        label
        in metrics
    ],

    fontweight="bold"
)


axD.set_ylabel(
    "Sequence-level\nconservation",

    fontsize=11.5,

    fontweight="bold",

    labelpad=6
)


style_ticks(
    axD
)


# avoid duplicate-looking legend patches by adding hatch
# rebuild clearer legend
legend_D = [

    Patch(
        facecolor="white",
        edgecolor="black",
        label="Internal non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="black",
        label="Internal CPP"
    ),

    Patch(
        facecolor="white",
        edgecolor="0.4",
        label="KELM non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="0.4",
        label="KELM CPP"
    )
]


legD = axD.legend(
    handles=legend_D,

    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=8,

    columnspacing=1.0
)

for text in legD.get_texts():
    text.set_fontweight("bold")


axD.spines[
    "top"
].set_visible(False)

axD.spines[
    "right"
].set_visible(False)


# ======================================================================
# PANEL LETTERS — OUTSIDE
# ======================================================================

def add_letter(
    ax,
    letter,
    x=-0.105,
    y=1.035
):

    ax.text(
        x,
        y,

        letter,

        transform=ax.transAxes,

        fontsize=21,

        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,

        zorder=100
    )


# A letter belongs to left heatmap
add_letter(
    axA1,
    "A",
    x=-0.17
)

add_letter(
    axB,
    "B"
)

add_letter(
    axC,
    "C"
)

add_letter(
    axD,
    "D"
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_COMPACT.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_COMPACT.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_COMPACT.svg"
)


fig.savefig(
    png_file,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    pdf_file,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    svg_file,

    bbox_inches="tight",

    pad_inches=0.05
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 6 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 6 — CORRECTED FINAL VERSION
#
# Fixes:
# 1. Panel B uses fraction_hotspot_union (correct quantity)
# 2. All four B groups retained
# 3. More gap between A and B
# 4. Robust Panel C support-threshold detection
# 5. Robust Panel D detection
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

SI_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


A_INTERNAL = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_internal_test_CPP.csv"
)

A_KELM = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_kelm_external_CPP.csv"
)

B_FILE = (
    TABLE_DIR
    / "cross_plm_hotspot_support_distribution.csv"
)

C_FILE = (
    TABLE_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

D_FILE = (
    SI_DIR
    / "cross_plm_hotspot_conservation_per_sequence.csv"
)


# ======================================================================
# HELPERS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def normalize_dataset(value):

    s = str(value).strip().lower()

    if "internal" in s:
        return "Internal"

    if (
        "kelm" in s
        or "external" in s
    ):
        return "KELM"

    return str(value)


def normalize_class(value):

    s = str(value).strip().lower()

    # IMPORTANT: test non-CPP first
    if (
        s in ["0", "0.0", "false", "negative"]
        or "non_cpp" in s
        or "non-cpp" in s
        or "noncpp" in s
        or "non cpp" in s
    ):
        return "non-CPP"

    if (
        s in ["1", "1.0", "true", "positive"]
        or s == "cpp"
    ):
        return "CPP"

    if s == "all":
        return "all"

    return str(value)


def style_ticks(ax):

    ax.tick_params(
        axis="both",
        width=1.0,
        length=4
    )

    for lab in ax.get_xticklabels():
        lab.set_fontweight("bold")

    for lab in ax.get_yticklabels():
        lab.set_fontweight("bold")


def first_existing(df, names):

    for name in names:

        if name in df.columns:
            return name

    return None


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        10.5,

    "axes.labelsize":
        12,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        9.5,

    "ytick.labelsize":
        9.5,

    "legend.fontsize":
        8.5,

    "axes.linewidth":
        1.1,

    "savefig.dpi":
        600
})


# ======================================================================
# LOAD
# ======================================================================

mat_internal = pd.read_csv(
    A_INTERNAL,
    index_col=0
)

mat_kelm = pd.read_csv(
    A_KELM,
    index_col=0
)

B = clean_columns(
    pd.read_csv(B_FILE)
)

C = clean_columns(
    pd.read_csv(C_FILE)
)

D = clean_columns(
    pd.read_csv(D_FILE)
)


print("=" * 90)
print("SOURCE TABLE COLUMNS")
print("=" * 90)

print("\nB:")
print(B.columns.tolist())

print("\nC:")
print(C.columns.tolist())

print("\nD:")
print(D.columns.tolist())


# ======================================================================
# PRETTY MODEL LABELS
# ======================================================================

pretty_model = {

    "ESM2_320":
        "ESM2-320",

    "ESM2_640":
        "ESM2-640",

    "ESM2_1280":
        "ESM2-1280",

    "ProtT5":
        "ProtT5"
}


for matrix in [
    mat_internal,
    mat_kelm
]:

    matrix.index = [
        pretty_model.get(
            str(x),
            str(x)
        )
        for x in matrix.index
    ]

    matrix.columns = [
        pretty_model.get(
            str(x),
            str(x)
        )
        for x in matrix.columns
    ]


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(14.3, 7.5)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.075,
    right=0.975,

    bottom=0.09,
    top=0.97,

    # --------------------------------------------------------------
    # MORE GAP BETWEEN A AND B
    # --------------------------------------------------------------
    wspace=0.39,

    hspace=0.31,

    width_ratios=[
        1.04,
        1.00
    ]
)


axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANEL A
# ======================================================================

A_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=gs[0, 0],

    width_ratios=[
        1,
        1,
        0.045
    ],

    wspace=0.07
)


axA1 = fig.add_subplot(
    A_grid[0]
)

axA2 = fig.add_subplot(
    A_grid[1]
)

caxA = fig.add_subplot(
    A_grid[2]
)


imA = axA1.imshow(
    mat_internal.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


axA2.imshow(
    mat_kelm.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


def format_heatmap(
    ax,
    matrix,
    show_y=True
):

    n = matrix.shape[0]

    ax.set_xticks(
        np.arange(n)
    )

    ax.set_yticks(
        np.arange(n)
    )


    ax.set_xticklabels(
        matrix.columns,

        rotation=35,

        ha="right",

        fontsize=8,

        fontweight="bold"
    )


    if show_y:

        ax.set_yticklabels(
            matrix.index,

            fontsize=8.5,

            fontweight="bold"
        )

    else:

        ax.set_yticklabels([])

        ax.tick_params(
            axis="y",
            length=0
        )


    for i in range(
        matrix.shape[0]
    ):

        for j in range(
            matrix.shape[1]
        ):

            value = float(
                matrix.iloc[i, j]
            )

            ax.text(
                j,
                i,

                f"{value:.2f}",

                ha="center",
                va="center",

                fontsize=7.6,

                fontweight="bold",

                color=(
                    "white"
                    if value > 0.55
                    else "black"
                )
            )


format_heatmap(
    axA1,
    mat_internal,
    True
)

format_heatmap(
    axA2,
    mat_kelm,
    False
)


axA1.text(
    0.5,
    1.035,

    "Internal test",

    transform=axA1.transAxes,

    ha="center",

    fontsize=10,

    fontweight="bold"
)


axA2.text(
    0.5,
    1.035,

    "KELM external",

    transform=axA2.transAxes,

    ha="center",

    fontsize=10,

    fontweight="bold"
)


cbA = fig.colorbar(
    imA,
    cax=caxA
)

cbA.set_label(
    "Mean per-sequence Jaccard",

    fontsize=9,

    fontweight="bold",

    labelpad=5
)

cbA.ax.tick_params(
    labelsize=8
)

for lab in cbA.ax.get_yticklabels():
    lab.set_fontweight("bold")


# ======================================================================
# PANEL B — CORRECT DATA
#
# Explicitly use fraction_hotspot_union
# ======================================================================

required_B = [
    "dataset",
    "class",
    "model_support_count",
    "fraction_hotspot_union"
]


missing_B = [
    c for c in required_B
    if c not in B.columns
]


if missing_B:

    raise KeyError(
        "Panel B missing columns:\n"
        + "\n".join(missing_B)
    )


B["dataset_clean"] = (
    B["dataset"]
    .map(normalize_dataset)
)

B["class_clean"] = (
    B["class"]
    .map(normalize_class)
)

B["support_clean"] = pd.to_numeric(
    B["model_support_count"],
    errors="coerce"
)

B["fraction_plot"] = pd.to_numeric(
    B["fraction_hotspot_union"],
    errors="coerce"
)


# Ignore support=0 because this is NOT hotspot union
Bplot = B[
    B["support_clean"].isin(
        [1, 2, 3, 4]
    )
].copy()


# Ignore "all" class
Bplot = Bplot[
    Bplot["class_clean"].isin(
        [
            "CPP",
            "non-CPP"
        ]
    )
].copy()


print("\n" + "=" * 90)
print("PANEL B VALUES ACTUALLY PLOTTED")
print("=" * 90)

display(
    Bplot[
        [
            "dataset_clean",
            "class_clean",
            "support_clean",
            "fraction_plot"
        ]
    ]
    .sort_values(
        [
            "dataset_clean",
            "class_clean",
            "support_clean"
        ]
    )
)


group_order = [

    (
        "Internal",
        "CPP",
        "Internal CPP"
    ),

    (
        "Internal",
        "non-CPP",
        "Internal non-CPP"
    ),

    (
        "KELM",
        "CPP",
        "KELM CPP"
    ),

    (
        "KELM",
        "non-CPP",
        "KELM non-CPP"
    )
]


support_values = [
    1,
    2,
    3,
    4
]


xB = np.arange(
    4
)

widthB = 0.18


for group_index, (
    dataset_name,
    class_name,
    legend_name
) in enumerate(
    group_order
):

    values = []


    for support in support_values:

        row = Bplot[
            (
                Bplot["dataset_clean"]
                == dataset_name
            )
            &
            (
                Bplot["class_clean"]
                == class_name
            )
            &
            (
                Bplot["support_clean"]
                == support
            )
        ]


        if len(row):

            value = float(
                row["fraction_plot"]
                .iloc[0]
            )

        else:

            value = 0.0


        values.append(
            value
        )


    xpos = (
        xB
        + (
            group_index
            - 1.5
        )
        * widthB
    )


    bars = axB.bar(
        xpos,
        values,
        widthB,
        label=legend_name
    )


    # value labels
    for bar, value in zip(
        bars,
        values
    ):

        axB.text(
            bar.get_x()
            + bar.get_width() / 2,

            value + 0.009,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=7.7,

            fontweight="bold"
        )


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    [
        "1 PLM",
        "2 PLMs",
        "3 PLMs",
        "4 PLMs"
    ],

    fontsize=10,

    fontweight="bold"
)


axB.set_ylabel(
    "Fraction of hotspot-union\nresidues",

    fontsize=11.5,

    fontweight="bold",

    labelpad=6
)


style_ticks(
    axB
)


# enough space for bar annotations
max_B = Bplot[
    "fraction_plot"
].max()

axB.set_ylim(
    0,
    max_B * 1.19
)


legB = axB.legend(
    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=8.1,

    columnspacing=1.0,

    handlelength=1.3
)


for text in legB.get_texts():
    text.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ======================================================================
# PANEL C — ROBUST COLUMN DETECTION
# ======================================================================

print("\nPanel C preview:")
display(C)


# ----------------------------------------------------------------------
# Explicit likely names first
# ----------------------------------------------------------------------

dataset_col_C = first_existing(
    C,
    [
        "dataset",
        "dataset_name"
    ]
)


odds_col = first_existing(
    C,
    [
        "odds_ratio",
        "cpp_odds_ratio",
        "enrichment_odds_ratio",
        "or"
    ]
)


fdr_col = first_existing(
    C,
    [
        "fdr",
        "fdr_bh",
        "qvalue",
        "q_value",
        "adjusted_p",
        "adjusted_p_value"
    ]
)


# ----------------------------------------------------------------------
# Search threshold/support column more flexibly
# ----------------------------------------------------------------------

threshold_col = first_existing(
    C,
    [
        "threshold",
        "support_threshold",
        "plm_support_threshold",
        "min_support",
        "minimum_support",
        "model_support_count",
        "support_count",
        "n_models",
        "support"
    ]
)


if threshold_col is None:

    for col in C.columns:

        lc = col.lower()

        if (
            "support" in lc
            or "threshold" in lc
            or "model" in lc
        ):

            # Does this column look like 2/3/4?
            test_vals = (
                C[col]
                .astype(str)
                .str.extract(
                    r"(\d+)"
                )[0]
            )

            nums = set(
                pd.to_numeric(
                    test_vals,
                    errors="coerce"
                )
                .dropna()
                .astype(int)
                .tolist()
            )


            if len(
                nums.intersection(
                    {2, 3, 4}
                )
            ) >= 2:

                threshold_col = col
                break


# ----------------------------------------------------------------------
# Robust odds-ratio search
# ----------------------------------------------------------------------

if odds_col is None:

    for col in C.columns:

        if (
            "odds" in col
            and "ratio" in col
        ):

            odds_col = col
            break


print("\nDetected Panel C columns:")
print("Dataset  :", dataset_col_C)
print("Support  :", threshold_col)
print("Odds     :", odds_col)
print("FDR      :", fdr_col)


if odds_col is None:

    raise KeyError(
        "Panel C odds-ratio column not detected.\n"
        f"Columns: {C.columns.tolist()}"
    )


Cplot = C.copy()


# Dataset
if dataset_col_C is not None:

    Cplot["dataset_clean"] = (
        Cplot[dataset_col_C]
        .map(normalize_dataset)
    )

else:

    # If exactly six rows, infer internal first 3, KELM next 3
    if len(Cplot) == 6:

        Cplot["dataset_clean"] = (
            ["Internal"] * 3
            + ["KELM"] * 3
        )

    else:

        raise KeyError(
            "Could not identify Panel C dataset."
        )


# ----------------------------------------------------------------------
# Support level
# ----------------------------------------------------------------------

def convert_support(value):

    s = str(value).lower()

    if "all" in s:
        return 4

    nums = re.findall(
        r"\d+",
        s
    )

    if nums:

        return int(
            nums[-1]
        )

    return np.nan


if threshold_col is not None:

    Cplot["support_num"] = (
        Cplot[threshold_col]
        .map(convert_support)
    )

else:

    # If three rows per dataset, assign 2/3/4 in row order.
    Cplot["support_num"] = np.nan

    for dataset_name in [
        "Internal",
        "KELM"
    ]:

        idx = Cplot[
            Cplot["dataset_clean"]
            == dataset_name
        ].index.tolist()


        if len(idx) == 3:

            Cplot.loc[
                idx,
                "support_num"
            ] = [
                2,
                3,
                4
            ]

        else:

            raise KeyError(
                "Could not infer Panel C support levels."
            )


# numeric OR
Cplot["odds_plot"] = pd.to_numeric(
    Cplot[odds_col],
    errors="coerce"
)


levels_C = [
    2,
    3,
    4
]

labels_C = [
    "≥2 PLMs",
    "≥3 PLMs",
    "All 4 PLMs"
]


xC = np.arange(
    3
)

widthC = 0.32


for di, dataset_name in enumerate(
    [
        "Internal",
        "KELM"
    ]
):

    vals = []
    ps = []


    for support in levels_C:

        row = Cplot[
            (
                Cplot["dataset_clean"]
                == dataset_name
            )
            &
            (
                Cplot["support_num"]
                == support
            )
        ]


        if len(row):

            vals.append(
                float(
                    row["odds_plot"]
                    .iloc[0]
                )
            )


            if fdr_col is not None:

                ps.append(
                    float(
                        row[fdr_col]
                        .iloc[0]
                    )
                )

            else:

                ps.append(
                    np.nan
                )

        else:

            vals.append(
                np.nan
            )

            ps.append(
                np.nan
            )


    xpos = (
        xC
        + (
            -widthC / 2
            if di == 0
            else widthC / 2
        )
    )


    bars = axC.bar(
        xpos,

        vals,

        widthC,

        label=(
            "Internal test"
            if dataset_name == "Internal"
            else "KELM external"
        )
    )


    for bar, val, p in zip(
        bars,
        vals,
        ps
    ):

        if np.isnan(val):
            continue


        axC.text(
            bar.get_x()
            + bar.get_width() / 2,

            val + 0.05,

            f"{val:.2f}",

            ha="center",
            va="bottom",

            fontsize=8,

            fontweight="bold"
        )


        if not np.isnan(p):

            if p < 0.001:
                sig = "***"

            elif p < 0.01:
                sig = "**"

            elif p < 0.05:
                sig = "*"

            else:
                sig = "ns"


            axC.text(
                bar.get_x()
                + bar.get_width() / 2,

                val + 0.20,

                sig,

                ha="center",
                va="bottom",

                fontsize=8.5,

                fontweight="bold"
            )


axC.axhline(
    1,

    linestyle="--",

    linewidth=1,

    color="0.45"
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    labels_C,

    fontsize=10,

    fontweight="bold"
)


axC.set_ylabel(
    "CPP enrichment\nodds ratio",

    fontsize=11.5,

    fontweight="bold",

    labelpad=6
)


style_ticks(
    axC
)


legC = axC.legend(
    frameon=False,

    loc="upper left",

    fontsize=8.5
)


for text in legC.get_texts():
    text.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ======================================================================
# PANEL D
# ======================================================================

dataset_col_D = first_existing(
    D,
    [
        "dataset",
        "dataset_name"
    ]
)

label_col_D = first_existing(
    D,
    [
        "label",
        "class",
        "cpp_label"
    ]
)


# detect metrics
jaccard_col = first_existing(
    D,
    [
        "mean_pairwise_jaccard",
        "mean_jaccard",
        "pairwise_jaccard"
    ]
)

support3_col = first_existing(
    D,
    [
        "fraction_supported_by_at_least_3",
        "fraction_supported_by_3",
        "fraction_at_least_3"
    ]
)

support4_col = first_existing(
    D,
    [
        "fraction_supported_by_all_4",
        "fraction_supported_by_4",
        "fraction_all_4"
    ]
)


# fallback
if jaccard_col is None:

    for col in D.columns:

        if "jaccard" in col:

            jaccard_col = col
            break


if support3_col is None:

    for col in D.columns:

        if (
            "fraction" in col
            and "3" in col
        ):

            support3_col = col
            break


if support4_col is None:

    for col in D.columns:

        if (
            "fraction" in col
            and "4" in col
        ):

            support4_col = col
            break


if any(
    x is None
    for x in [
        dataset_col_D,
        label_col_D,
        jaccard_col,
        support3_col,
        support4_col
    ]
):

    raise KeyError(
        "Panel D column detection failed.\n"
        f"{D.columns.tolist()}"
    )


D["dataset_clean"] = (
    D[dataset_col_D]
    .map(normalize_dataset)
)

D["class_clean"] = (
    D[label_col_D]
    .map(normalize_class)
)


metrics_D = [

    (
        jaccard_col,
        "Mean pairwise\nJaccard"
    ),

    (
        support3_col,
        "Supported by\n≥3 PLMs"
    ),

    (
        support4_col,
        "Supported by\nall 4 PLMs"
    )
]


groups_D = [

    (
        "Internal",
        "non-CPP"
    ),

    (
        "Internal",
        "CPP"
    ),

    (
        "KELM",
        "non-CPP"
    ),

    (
        "KELM",
        "CPP"
    )
]


base_positions = np.arange(
    3
)

offsets_D = [
    -0.27,
    -0.09,
    0.09,
    0.27
]


for gi, (
    dataset_name,
    class_name
) in enumerate(
    groups_D
):

    group = D[
        (
            D["dataset_clean"]
            == dataset_name
        )
        &
        (
            D["class_clean"]
            == class_name
        )
    ]


    vals = [

        pd.to_numeric(
            group[col],
            errors="coerce"
        )
        .dropna()
        .to_numpy()

        for col, _
        in metrics_D
    ]


    axD.boxplot(
        vals,

        positions=(
            base_positions
            + offsets_D[gi]
        ),

        widths=0.16,

        patch_artist=True,

        showfliers=False,

        medianprops=dict(
            color="tab:orange",
            linewidth=1.4
        ),

        boxprops=dict(
            facecolor=(
                "0.82"
                if class_name == "CPP"
                else "white"
            ),

            edgecolor="black",

            linewidth=0.8
        ),

        whiskerprops=dict(
            color="black",
            linewidth=0.8
        ),

        capprops=dict(
            color="black",
            linewidth=0.8
        )
    )


axD.set_xticks(
    base_positions
)

axD.set_xticklabels(
    [
        label
        for _,
        label in metrics_D
    ],

    fontsize=9.5,

    fontweight="bold"
)


axD.set_ylabel(
    "Sequence-level\nconservation",

    fontsize=11.5,

    fontweight="bold",

    labelpad=6
)


style_ticks(
    axD
)


legend_D = [

    Patch(
        facecolor="white",
        edgecolor="black",
        label="Internal non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="black",
        label="Internal CPP"
    ),

    Patch(
        facecolor="white",
        edgecolor="0.45",
        label="KELM non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="0.45",
        label="KELM CPP"
    )
]


legD = axD.legend(
    handles=legend_D,

    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=8
)


for text in legD.get_texts():
    text.set_fontweight("bold")


axD.spines["top"].set_visible(False)
axD.spines["right"].set_visible(False)


# ======================================================================
# PANEL LETTERS
# ======================================================================

def add_letter(
    ax,
    letter,
    x=-0.11,
    y=1.03
):

    ax.text(
        x,
        y,

        letter,

        transform=ax.transAxes,

        fontsize=21,

        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,

        zorder=100
    )


add_letter(
    axA1,
    "A",
    x=-0.18
)

add_letter(
    axB,
    "B",
    x=-0.13
)

add_letter(
    axC,
    "C"
)

add_letter(
    axD,
    "D"
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_CORRECTED.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_CORRECTED.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_CORRECTED.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()
plt.close(fig)


print("\n" + "=" * 90)
print("CORRECTED FIGURE 6 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 6 — FINAL MANUSCRIPT VERSION WITH LARGER FONTS
#
# A = Pairwise CPP hotspot Jaccard matrices
# B = Cross-PLM hotspot-union support distribution
# C = CPP enrichment with increasing PLM agreement
# D = Sequence-level hotspot conservation
#
# Improvements:
# - larger fonts throughout
# - larger heatmap numbers
# - larger legends
# - larger bar-value labels
# - larger significance labels
# - larger panel letters
# - preserved gap between A and B
# - Panel B uses fraction_hotspot_union
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

SI_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


A_INTERNAL = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_internal_test_CPP.csv"
)

A_KELM = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_kelm_external_CPP.csv"
)

B_FILE = (
    TABLE_DIR
    / "cross_plm_hotspot_support_distribution.csv"
)

C_FILE = (
    TABLE_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

D_FILE = (
    SI_DIR
    / "cross_plm_hotspot_conservation_per_sequence.csv"
)


# ======================================================================
# HELPERS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def normalize_dataset(value):

    s = str(value).strip().lower()

    if "internal" in s:
        return "Internal"

    if (
        "kelm" in s
        or "external" in s
    ):
        return "KELM"

    return str(value)


def normalize_class(value):

    s = str(value).strip().lower()

    if (
        s in [
            "0",
            "0.0",
            "false",
            "negative"
        ]
        or "non_cpp" in s
        or "non-cpp" in s
        or "noncpp" in s
        or "non cpp" in s
    ):
        return "non-CPP"

    if (
        s in [
            "1",
            "1.0",
            "true",
            "positive"
        ]
        or s == "cpp"
    ):
        return "CPP"

    if s == "all":
        return "all"

    return str(value)


def style_ticks(ax):

    ax.tick_params(
        axis="both",
        width=1.1,
        length=4.5
    )

    for lab in ax.get_xticklabels():
        lab.set_fontweight("bold")

    for lab in ax.get_yticklabels():
        lab.set_fontweight("bold")


def first_existing(df, names):

    for name in names:

        if name in df.columns:
            return name

    return None


# ======================================================================
# STYLE — ENLARGED
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        12,

    "axes.labelsize":
        14,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        11,

    "ytick.labelsize":
        11,

    "legend.fontsize":
        10,

    "axes.linewidth":
        1.2,

    "savefig.dpi":
        600
})


# ======================================================================
# LOAD DATA
# ======================================================================

mat_internal = pd.read_csv(
    A_INTERNAL,
    index_col=0
)

mat_kelm = pd.read_csv(
    A_KELM,
    index_col=0
)

B = clean_columns(
    pd.read_csv(B_FILE)
)

C = clean_columns(
    pd.read_csv(C_FILE)
)

D = clean_columns(
    pd.read_csv(D_FILE)
)


# ======================================================================
# PRETTY MODEL LABELS
# ======================================================================

pretty_model = {

    "ESM2_320":
        "ESM2-320",

    "ESM2_640":
        "ESM2-640",

    "ESM2_1280":
        "ESM2-1280",

    "ProtT5":
        "ProtT5"
}


for matrix in [
    mat_internal,
    mat_kelm
]:

    matrix.index = [
        pretty_model.get(
            str(x),
            str(x)
        )
        for x in matrix.index
    ]

    matrix.columns = [
        pretty_model.get(
            str(x),
            str(x)
        )
        for x in matrix.columns
    ]


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(14.8, 8.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.075,
    right=0.975,

    bottom=0.09,
    top=0.97,

    # keep visible A–B separation
    wspace=0.39,

    hspace=0.31,

    width_ratios=[
        1.04,
        1.00
    ]
)


axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANEL A — HEATMAPS
# ======================================================================

A_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=gs[0, 0],

    width_ratios=[
        1,
        1,
        0.045
    ],

    wspace=0.07
)


axA1 = fig.add_subplot(
    A_grid[0]
)

axA2 = fig.add_subplot(
    A_grid[1]
)

caxA = fig.add_subplot(
    A_grid[2]
)


imA = axA1.imshow(
    mat_internal.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


axA2.imshow(
    mat_kelm.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


def format_heatmap(
    ax,
    matrix,
    show_y=True
):

    n = matrix.shape[0]

    ax.set_xticks(
        np.arange(n)
    )

    ax.set_yticks(
        np.arange(n)
    )


    ax.set_xticklabels(
        matrix.columns,

        rotation=35,

        ha="right",

        fontsize=10,

        fontweight="bold"
    )


    if show_y:

        ax.set_yticklabels(
            matrix.index,

            fontsize=10,

            fontweight="bold"
        )

    else:

        ax.set_yticklabels([])

        ax.tick_params(
            axis="y",
            length=0
        )


    for i in range(
        matrix.shape[0]
    ):

        for j in range(
            matrix.shape[1]
        ):

            value = float(
                matrix.iloc[i, j]
            )

            ax.text(
                j,
                i,

                f"{value:.2f}",

                ha="center",
                va="center",

                fontsize=9,

                fontweight="bold",

                color=(
                    "white"
                    if value > 0.55
                    else "black"
                )
            )


format_heatmap(
    axA1,
    mat_internal,
    True
)

format_heatmap(
    axA2,
    mat_kelm,
    False
)


axA1.text(
    0.5,
    1.035,

    "Internal test",

    transform=axA1.transAxes,

    ha="center",

    fontsize=12,

    fontweight="bold"
)


axA2.text(
    0.5,
    1.035,

    "KELM external",

    transform=axA2.transAxes,

    ha="center",

    fontsize=12,

    fontweight="bold"
)


cbA = fig.colorbar(
    imA,
    cax=caxA
)

cbA.set_label(
    "Mean per-sequence Jaccard",

    fontsize=11,

    fontweight="bold",

    labelpad=6
)

cbA.ax.tick_params(
    labelsize=9.5
)

for lab in cbA.ax.get_yticklabels():
    lab.set_fontweight("bold")


# ======================================================================
# PANEL B — HOTSPOT SUPPORT DISTRIBUTION
# ======================================================================

required_B = [
    "dataset",
    "class",
    "model_support_count",
    "fraction_hotspot_union"
]


missing_B = [
    c
    for c in required_B
    if c not in B.columns
]


if missing_B:

    raise KeyError(
        "Panel B missing columns:\n"
        + "\n".join(missing_B)
    )


B["dataset_clean"] = (
    B["dataset"]
    .map(normalize_dataset)
)

B["class_clean"] = (
    B["class"]
    .map(normalize_class)
)

B["support_clean"] = pd.to_numeric(
    B["model_support_count"],
    errors="coerce"
)

B["fraction_plot"] = pd.to_numeric(
    B["fraction_hotspot_union"],
    errors="coerce"
)


Bplot = B[
    B["support_clean"].isin(
        [1, 2, 3, 4]
    )
].copy()


Bplot = Bplot[
    Bplot["class_clean"].isin(
        [
            "CPP",
            "non-CPP"
        ]
    )
].copy()


group_order = [

    (
        "Internal",
        "CPP",
        "Internal CPP"
    ),

    (
        "Internal",
        "non-CPP",
        "Internal non-CPP"
    ),

    (
        "KELM",
        "CPP",
        "KELM CPP"
    ),

    (
        "KELM",
        "non-CPP",
        "KELM non-CPP"
    )
]


support_values = [
    1,
    2,
    3,
    4
]


xB = np.arange(
    4
)

widthB = 0.18


for group_index, (
    dataset_name,
    class_name,
    legend_name
) in enumerate(
    group_order
):

    values = []


    for support in support_values:

        row = Bplot[
            (
                Bplot["dataset_clean"]
                == dataset_name
            )
            &
            (
                Bplot["class_clean"]
                == class_name
            )
            &
            (
                Bplot["support_clean"]
                == support
            )
        ]


        if len(row):

            value = float(
                row["fraction_plot"]
                .iloc[0]
            )

        else:

            value = 0.0


        values.append(
            value
        )


    xpos = (
        xB
        + (
            group_index
            - 1.5
        )
        * widthB
    )


    bars = axB.bar(
        xpos,
        values,
        widthB,
        label=legend_name
    )


    # larger annotations
    for bar, value in zip(
        bars,
        values
    ):

        axB.text(
            bar.get_x()
            + bar.get_width() / 2,

            value + 0.010,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=9,

            fontweight="bold"
        )


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    [
        "1 PLM",
        "2 PLMs",
        "3 PLMs",
        "4 PLMs"
    ],

    fontsize=11,

    fontweight="bold"
)


axB.set_ylabel(
    "Fraction of hotspot-union\nresidues",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


style_ticks(
    axB
)


max_B = Bplot[
    "fraction_plot"
].max()

axB.set_ylim(
    0,
    max_B * 1.20
)


legB = axB.legend(
    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=9.5,

    columnspacing=1.1,

    handlelength=1.4
)


for text in legB.get_texts():
    text.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ======================================================================
# PANEL C — CPP ENRICHMENT
# ======================================================================

dataset_col_C = first_existing(
    C,
    [
        "dataset",
        "dataset_name"
    ]
)


odds_col = first_existing(
    C,
    [
        "odds_ratio",
        "cpp_odds_ratio",
        "enrichment_odds_ratio",
        "or"
    ]
)


fdr_col = first_existing(
    C,
    [
        "fdr",
        "fdr_bh",
        "qvalue",
        "q_value",
        "adjusted_p",
        "adjusted_p_value"
    ]
)


threshold_col = first_existing(
    C,
    [
        "threshold",
        "support_threshold",
        "plm_support_threshold",
        "min_support",
        "minimum_support",
        "model_support_count",
        "support_count",
        "n_models",
        "support"
    ]
)


if threshold_col is None:

    for col in C.columns:

        lc = col.lower()

        if (
            "support" in lc
            or "threshold" in lc
            or "model" in lc
        ):

            test_vals = (
                C[col]
                .astype(str)
                .str.extract(
                    r"(\d+)"
                )[0]
            )

            nums = set(
                pd.to_numeric(
                    test_vals,
                    errors="coerce"
                )
                .dropna()
                .astype(int)
                .tolist()
            )

            if len(
                nums.intersection(
                    {2, 3, 4}
                )
            ) >= 2:

                threshold_col = col
                break


if odds_col is None:

    for col in C.columns:

        if (
            "odds" in col
            and "ratio" in col
        ):
            odds_col = col
            break


if odds_col is None:

    raise KeyError(
        "Panel C odds-ratio column not detected."
    )


Cplot = C.copy()


if dataset_col_C is not None:

    Cplot["dataset_clean"] = (
        Cplot[dataset_col_C]
        .map(normalize_dataset)
    )

else:

    if len(Cplot) == 6:

        Cplot["dataset_clean"] = (
            ["Internal"] * 3
            + ["KELM"] * 3
        )

    else:

        raise KeyError(
            "Could not identify Panel C dataset."
        )


def convert_support(value):

    s = str(value).lower()

    if "all" in s:
        return 4

    nums = re.findall(
        r"\d+",
        s
    )

    if nums:
        return int(
            nums[-1]
        )

    return np.nan


if threshold_col is not None:

    Cplot["support_num"] = (
        Cplot[threshold_col]
        .map(convert_support)
    )

else:

    Cplot["support_num"] = np.nan


    for dataset_name in [
        "Internal",
        "KELM"
    ]:

        idx = Cplot[
            Cplot["dataset_clean"]
            == dataset_name
        ].index.tolist()


        if len(idx) == 3:

            Cplot.loc[
                idx,
                "support_num"
            ] = [
                2,
                3,
                4
            ]

        else:

            raise KeyError(
                "Could not infer Panel C support levels."
            )


Cplot["odds_plot"] = pd.to_numeric(
    Cplot[odds_col],
    errors="coerce"
)


levels_C = [
    2,
    3,
    4
]

labels_C = [
    "≥2 PLMs",
    "≥3 PLMs",
    "All 4 PLMs"
]


xC = np.arange(
    3
)

widthC = 0.32


for di, dataset_name in enumerate(
    [
        "Internal",
        "KELM"
    ]
):

    vals = []
    ps = []


    for support in levels_C:

        row = Cplot[
            (
                Cplot["dataset_clean"]
                == dataset_name
            )
            &
            (
                Cplot["support_num"]
                == support
            )
        ]


        if len(row):

            vals.append(
                float(
                    row["odds_plot"]
                    .iloc[0]
                )
            )


            if fdr_col is not None:

                ps.append(
                    float(
                        row[fdr_col]
                        .iloc[0]
                    )
                )

            else:

                ps.append(
                    np.nan
                )

        else:

            vals.append(
                np.nan
            )

            ps.append(
                np.nan
            )


    xpos = (
        xC
        + (
            -widthC / 2
            if di == 0
            else widthC / 2
        )
    )


    bars = axC.bar(
        xpos,
        vals,
        widthC,

        label=(
            "Internal test"
            if dataset_name == "Internal"
            else "KELM external"
        )
    )


    for bar, val, p in zip(
        bars,
        vals,
        ps
    ):

        if np.isnan(val):
            continue


        # larger value label
        axC.text(
            bar.get_x()
            + bar.get_width() / 2,

            val + 0.05,

            f"{val:.2f}",

            ha="center",
            va="bottom",

            fontsize=9.5,

            fontweight="bold"
        )


        if not np.isnan(p):

            if p < 0.001:
                sig = "***"

            elif p < 0.01:
                sig = "**"

            elif p < 0.05:
                sig = "*"

            else:
                sig = "ns"


            axC.text(
                bar.get_x()
                + bar.get_width() / 2,

                val + 0.22,

                sig,

                ha="center",
                va="bottom",

                fontsize=10,

                fontweight="bold"
            )


axC.axhline(
    1,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    labels_C,

    fontsize=11,

    fontweight="bold"
)


axC.set_ylabel(
    "CPP enrichment\nodds ratio",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


style_ticks(
    axC
)


legC = axC.legend(
    frameon=False,

    loc="upper left",

    fontsize=9.5
)


for text in legC.get_texts():
    text.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ======================================================================
# PANEL D — SEQUENCE-LEVEL CONSERVATION
# ======================================================================

dataset_col_D = first_existing(
    D,
    [
        "dataset",
        "dataset_name"
    ]
)

label_col_D = first_existing(
    D,
    [
        "label",
        "class",
        "cpp_label"
    ]
)


jaccard_col = first_existing(
    D,
    [
        "mean_pairwise_jaccard",
        "mean_jaccard",
        "pairwise_jaccard"
    ]
)

support3_col = first_existing(
    D,
    [
        "fraction_supported_by_at_least_3",
        "fraction_supported_by_3",
        "fraction_at_least_3"
    ]
)

support4_col = first_existing(
    D,
    [
        "fraction_supported_by_all_4",
        "fraction_supported_by_4",
        "fraction_all_4"
    ]
)


if jaccard_col is None:

    for col in D.columns:

        if "jaccard" in col:
            jaccard_col = col
            break


if support3_col is None:

    for col in D.columns:

        if (
            "fraction" in col
            and "3" in col
        ):
            support3_col = col
            break


if support4_col is None:

    for col in D.columns:

        if (
            "fraction" in col
            and "4" in col
        ):
            support4_col = col
            break


if any(
    x is None
    for x in [
        dataset_col_D,
        label_col_D,
        jaccard_col,
        support3_col,
        support4_col
    ]
):

    raise KeyError(
        "Panel D column detection failed."
    )


D["dataset_clean"] = (
    D[dataset_col_D]
    .map(normalize_dataset)
)

D["class_clean"] = (
    D[label_col_D]
    .map(normalize_class)
)


metrics_D = [

    (
        jaccard_col,
        "Mean pairwise\nJaccard"
    ),

    (
        support3_col,
        "Supported by\n≥3 PLMs"
    ),

    (
        support4_col,
        "Supported by\nall 4 PLMs"
    )
]


groups_D = [

    (
        "Internal",
        "non-CPP"
    ),

    (
        "Internal",
        "CPP"
    ),

    (
        "KELM",
        "non-CPP"
    ),

    (
        "KELM",
        "CPP"
    )
]


base_positions = np.arange(
    3
)

offsets_D = [
    -0.27,
    -0.09,
    0.09,
    0.27
]


for gi, (
    dataset_name,
    class_name
) in enumerate(
    groups_D
):

    group = D[
        (
            D["dataset_clean"]
            == dataset_name
        )
        &
        (
            D["class_clean"]
            == class_name
        )
    ]


    vals = [

        pd.to_numeric(
            group[col],
            errors="coerce"
        )
        .dropna()
        .to_numpy()

        for col, _
        in metrics_D
    ]


    axD.boxplot(
        vals,

        positions=(
            base_positions
            + offsets_D[gi]
        ),

        widths=0.16,

        patch_artist=True,

        showfliers=False,

        medianprops=dict(
            color="tab:orange",
            linewidth=1.5
        ),

        boxprops=dict(
            facecolor=(
                "0.82"
                if class_name == "CPP"
                else "white"
            ),

            edgecolor="black",

            linewidth=0.9
        ),

        whiskerprops=dict(
            color="black",
            linewidth=0.9
        ),

        capprops=dict(
            color="black",
            linewidth=0.9
        )
    )


axD.set_xticks(
    base_positions
)

axD.set_xticklabels(
    [
        label
        for _,
        label in metrics_D
    ],

    fontsize=10.5,

    fontweight="bold"
)


axD.set_ylabel(
    "Sequence-level\nconservation",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


style_ticks(
    axD
)


legend_D = [

    Patch(
        facecolor="white",
        edgecolor="black",
        label="Internal non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="black",
        label="Internal CPP"
    ),

    Patch(
        facecolor="white",
        edgecolor="0.45",
        label="KELM non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="0.45",
        label="KELM CPP"
    )
]


legD = axD.legend(
    handles=legend_D,

    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=9.2
)


for text in legD.get_texts():
    text.set_fontweight("bold")


axD.spines["top"].set_visible(False)
axD.spines["right"].set_visible(False)


# ======================================================================
# PANEL LETTERS — LARGER
# ======================================================================

def add_letter(
    ax,
    letter,
    x=-0.11,
    y=1.03
):

    ax.text(
        x,
        y,

        letter,

        transform=ax.transAxes,

        fontsize=23,

        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,

        zorder=100
    )


add_letter(
    axA1,
    "A",
    x=-0.18
)

add_letter(
    axB,
    "B",
    x=-0.13
)

add_letter(
    axC,
    "C"
)

add_letter(
    axD,
    "D"
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()
plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 6 WITH LARGER FONTS GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
!find /content/drive/MyDrive/pLM4CPP_XAI_2026 -type f | \
grep -Ei "physicochemical|hydropathy|charge|positional|regional|region|position|figure_7" | \
sort

In [ ]:
for f in [
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/tables_main/physicochemical_enrichment_replication.csv",
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/tables_SI/physicochemical_properties_per_sequence.csv",
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/tables_main/hotspot_positional_region_enrichment.csv",
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/tables_main/hotspot_position_10bin_distribution.csv",
]:
    import pandas as pd
    print("\n" + "="*100)
    print(f)
    df = pd.read_csv(f)
    print(df.columns.tolist())
    display(df.head(12))

In [ ]:
# ======================================================================
# FIGURE 7 — FINAL MANUSCRIPT VERSION
# Physicochemical and Positional Organization of Consensus CPP Hotspots
#
# A = Replicated physicochemical enrichment
# B = Sequence-level charge and hydropathy of hotspots vs non-hotspots
# C = Regional positional enrichment of CPP hotspots
# D = Detailed 10-bin positional profile
#
# Uses exact source-table columns provided above.
# ======================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

SI_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


A_FILE = (
    TABLE_DIR
    / "physicochemical_enrichment_replication.csv"
)

B_FILE = (
    SI_DIR
    / "physicochemical_properties_per_sequence.csv"
)

C_FILE = (
    TABLE_DIR
    / "hotspot_positional_region_enrichment.csv"
)

D_FILE = (
    TABLE_DIR
    / "hotspot_position_10bin_distribution.csv"
)


# ======================================================================
# LOAD DATA
# ======================================================================

A = pd.read_csv(A_FILE)
B = pd.read_csv(B_FILE)
C = pd.read_csv(C_FILE)
D = pd.read_csv(D_FILE)


print("=" * 90)
print("FIGURE 7 SOURCE TABLES LOADED")
print("=" * 90)

print("\nA:", A.shape)
print("B:", B.shape)
print("C:", C.shape)
print("D:", D.shape)


# ======================================================================
# HELPERS
# ======================================================================

def normalize_dataset(value):

    s = str(value).lower()

    if "internal" in s:
        return "Internal"

    if (
        "kelm" in s
        or "external" in s
    ):
        return "KELM"

    return str(value)


def significance_symbol(p):

    if pd.isna(p):
        return ""

    if p < 0.001:
        return "***"

    if p < 0.01:
        return "**"

    if p < 0.05:
        return "*"

    return "ns"


def bold_ticks(ax):

    ax.tick_params(
        axis="both",
        width=1.1,
        length=4.5
    )

    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")


def clean_axes(ax):

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def add_panel_letter(
    ax,
    letter,
    x=-0.11,
    y=1.03
):

    ax.text(
        x,
        y,
        letter,

        transform=ax.transAxes,

        fontsize=23,
        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,
        zorder=100
    )


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        12,

    "axes.labelsize":
        13,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        10.5,

    "ytick.labelsize":
        10.5,

    "legend.fontsize":
        9.5,

    "axes.linewidth":
        1.2,

    "savefig.dpi":
        600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(14.5, 8.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.080,
    right=0.975,

    bottom=0.095,
    top=0.970,

    wspace=0.30,
    hspace=0.34
)


axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANEL A
# PHYSICOCHEMICAL ENRICHMENT
# ======================================================================

Aplot = A.copy()


property_order = [
    "basic_positive",
    "acidic_negative",
    "polar_uncharged",
    "aromatic",
    "aliphatic_hydrophobic",
    "structure_special"
]


property_labels = {
    "basic_positive":
        "Basic\npositive",

    "acidic_negative":
        "Acidic\nnegative",

    "polar_uncharged":
        "Polar\nuncharged",

    "aromatic":
        "Aromatic",

    "aliphatic_hydrophobic":
        "Aliphatic\nhydrophobic",

    "structure_special":
        "Structure\nspecial"
}


Aplot = (
    Aplot
    .set_index("property_class")
    .loc[property_order]
    .reset_index()
)


xA = np.arange(
    len(property_order)
)

widthA = 0.34


bars_internal = axA.bar(
    xA - widthA/2,

    Aplot[
        "internal_log2_enrichment"
    ],

    widthA,

    label="Internal test"
)


bars_kelm = axA.bar(
    xA + widthA/2,

    Aplot[
        "kelm_log2_enrichment"
    ],

    widthA,

    label="KELM external"
)


# zero reference
axA.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


# ----------------------------------------------------------------------
# Significance labels
# Put above positive bars / below negative bars
# ----------------------------------------------------------------------

def annotate_significance(
    ax,
    bars,
    pvalues
):

    for bar, p in zip(
        bars,
        pvalues
    ):

        h = bar.get_height()

        sig = significance_symbol(
            p
        )


        if h >= 0:

            y = h + 0.10

            va = "bottom"

        else:

            y = h - 0.11

            va = "top"


        ax.text(
            bar.get_x()
            + bar.get_width()/2,

            y,

            sig,

            ha="center",
            va=va,

            fontsize=9.5,

            fontweight="bold"
        )


annotate_significance(
    axA,
    bars_internal,
    Aplot["internal_fdr"]
)

annotate_significance(
    axA,
    bars_kelm,
    Aplot["kelm_fdr"]
)


axA.set_xticks(
    xA
)

axA.set_xticklabels(
    [
        property_labels[x]
        for x in property_order
    ],

    fontsize=9.8,

    fontweight="bold"
)


axA.set_ylabel(
    "log$_2$ enrichment\nin CPP hotspots",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


legA = axA.legend(
    frameon=False,

    loc="lower left",

    fontsize=9.5
)

for text in legA.get_texts():
    text.set_fontweight("bold")


bold_ticks(axA)
clean_axes(axA)

add_panel_letter(
    axA,
    "A"
)


# ======================================================================
# PANEL B
# CHARGE SCORE + HYDROPATHY
#
# CPP sequences only
# ======================================================================

Bplot = B.copy()


# CPP only
Bplot = Bplot[
    Bplot["label"] == 1
].copy()


Bplot["dataset_clean"] = (
    Bplot["dataset"]
    .map(normalize_dataset)
)


# ----------------------------------------------------------------------
# Boxplot groups:
#
# Metric 1: charge
#   Internal hotspot
#   Internal non-hotspot
#   KELM hotspot
#   KELM non-hotspot
#
# Metric 2: hydropathy
#   same four groups
# ----------------------------------------------------------------------

metrics_B = [

    (
        "hotspot_charge_score",
        "nonhotspot_charge_score",
        "Mean charge\nscore"
    ),

    (
        "hotspot_hydropathy_KD",
        "nonhotspot_hydropathy_KD",
        "Mean hydropathy"
    )
]


base_B = np.array([
    0,
    1
])


offsets_B = [
    -0.27,
    -0.09,
    0.09,
    0.27
]


group_specs_B = [

    (
        "Internal",
        "hotspot",
        "Internal hotspot",
        "0.55"
    ),

    (
        "Internal",
        "nonhotspot",
        "Internal non-hotspot",
        "white"
    ),

    (
        "KELM",
        "hotspot",
        "KELM hotspot",
        "0.72"
    ),

    (
        "KELM",
        "nonhotspot",
        "KELM non-hotspot",
        "white"
    )
]


for gi, (
    dataset_name,
    region_type,
    legend_name,
    face
) in enumerate(
    group_specs_B
):

    metric_values = []


    for hotspot_col, nonhot_col, _ in metrics_B:

        col = (
            hotspot_col
            if region_type == "hotspot"
            else nonhot_col
        )


        values = (
            Bplot[
                Bplot["dataset_clean"]
                == dataset_name
            ][col]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        metric_values.append(
            values
        )


    axB.boxplot(
        metric_values,

        positions=(
            base_B
            + offsets_B[gi]
        ),

        widths=0.16,

        patch_artist=True,

        showfliers=False,

        medianprops=dict(
            color="tab:orange",
            linewidth=1.5
        ),

        boxprops=dict(
            facecolor=face,
            edgecolor="black",
            linewidth=0.9
        ),

        whiskerprops=dict(
            color="black",
            linewidth=0.9
        ),

        capprops=dict(
            color="black",
            linewidth=0.9
        )
    )


axB.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


axB.set_xticks(
    base_B
)

axB.set_xticklabels(
    [
        "Mean charge\nscore",
        "Mean hydropathy"
    ],

    fontsize=10,

    fontweight="bold"
)


axB.set_ylabel(
    "Sequence-level\nproperty value",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


legend_B = [

    Patch(
        facecolor="0.55",
        edgecolor="black",
        label="Internal hotspot"
    ),

    Patch(
        facecolor="white",
        edgecolor="black",
        label="Internal non-hotspot"
    ),

    Patch(
        facecolor="0.72",
        edgecolor="black",
        label="KELM hotspot"
    ),

    Patch(
        facecolor="white",
        edgecolor="0.45",
        label="KELM non-hotspot"
    )
]


legB = axB.legend(
    handles=legend_B,

    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=8.7,

    columnspacing=1.0,

    handlelength=1.5
)


for text in legB.get_texts():
    text.set_fontweight("bold")


bold_ticks(axB)
clean_axes(axB)

add_panel_letter(
    axB,
    "B"
)


# ======================================================================
# PANEL C
# REGIONAL ORGANIZATION
#
# CPP only
# ======================================================================

Cplot = C[
    C["class"] == "CPP"
].copy()


Cplot["dataset_clean"] = (
    Cplot["dataset"]
    .map(normalize_dataset)
)


region_order = [
    "N_terminal",
    "Middle",
    "C_terminal"
]


region_labels = {
    "N_terminal":
        "N-terminal",

    "Middle":
        "Middle",

    "C_terminal":
        "C-terminal"
}


xC = np.arange(
    3
)

widthC = 0.34


values_internal = []
values_kelm = []

fdr_internal = []
fdr_kelm = []


for region in region_order:

    row_i = Cplot[
        (
            Cplot["dataset_clean"]
            == "Internal"
        )
        &
        (
            Cplot["region"]
            == region
        )
    ]


    row_k = Cplot[
        (
            Cplot["dataset_clean"]
            == "KELM"
        )
        &
        (
            Cplot["region"]
            == region
        )
    ]


    values_internal.append(
        float(
            row_i[
                "log2_enrichment"
            ].iloc[0]
        )
    )

    values_kelm.append(
        float(
            row_k[
                "log2_enrichment"
            ].iloc[0]
        )
    )


    fdr_internal.append(
        float(
            row_i[
                "fdr_bh"
            ].iloc[0]
        )
    )

    fdr_kelm.append(
        float(
            row_k[
                "fdr_bh"
            ].iloc[0]
        )
    )


bars_C_i = axC.bar(
    xC - widthC/2,

    values_internal,

    widthC,

    label="Internal test"
)


bars_C_k = axC.bar(
    xC + widthC/2,

    values_kelm,

    widthC,

    label="KELM external"
)


axC.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


annotate_significance(
    axC,
    bars_C_i,
    fdr_internal
)

annotate_significance(
    axC,
    bars_C_k,
    fdr_kelm
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    [
        region_labels[x]
        for x in region_order
    ],

    fontsize=10.5,

    fontweight="bold"
)


axC.set_ylabel(
    "log$_2$ enrichment\nin CPP hotspots",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


legC = axC.legend(
    frameon=False,

    loc="lower left",

    fontsize=9.5
)


for text in legC.get_texts():
    text.set_fontweight("bold")


bold_ticks(axC)
clean_axes(axC)

add_panel_letter(
    axC,
    "C"
)


# ======================================================================
# PANEL D
# DETAILED TEN-BIN POSITIONAL PROFILE
#
# CPP only
# ======================================================================

Dplot = D[
    D["class"] == "CPP"
].copy()


Dplot["dataset_clean"] = (
    Dplot["dataset"]
    .map(normalize_dataset)
)


internal_D = (
    Dplot[
        Dplot["dataset_clean"]
        == "Internal"
    ]
    .sort_values(
        "position_bin"
    )
)


kelm_D = (
    Dplot[
        Dplot["dataset_clean"]
        == "KELM"
    ]
    .sort_values(
        "position_bin"
    )
)


xD = np.arange(
    1,
    11
)


# ----------------------------------------------------------------------
# Central region shading: 20–70%
# corresponds approximately bins 3–7
# ----------------------------------------------------------------------

axD.axvspan(
    2.5,
    7.5,

    alpha=0.12,

    color="tab:blue",

    zorder=0
)


axD.plot(
    xD,

    internal_D[
        "hotspot_to_nonhotspot_ratio"
    ].to_numpy(),

    marker="o",

    linewidth=1.8,

    markersize=5.5,

    label="Internal test"
)


axD.plot(
    xD,

    kelm_D[
        "hotspot_to_nonhotspot_ratio"
    ].to_numpy(),

    marker="s",

    linewidth=1.8,

    markersize=5.5,

    label="KELM external"
)


# neutral reference
axD.axhline(
    1,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


position_labels = [
    "0–10",
    "10–20",
    "20–30",
    "30–40",
    "40–50",
    "50–60",
    "60–70",
    "70–80",
    "80–90",
    "90–100"
]


axD.set_xticks(
    xD
)

axD.set_xticklabels(
    position_labels,

    rotation=35,

    ha="right",

    fontsize=9.5,

    fontweight="bold"
)


axD.set_xlabel(
    "Relative sequence position (%)",

    fontsize=12.5,

    fontweight="bold",

    labelpad=6
)


axD.set_ylabel(
    "Hotspot/non-hotspot\nfrequency ratio",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


# small central-region annotation
ymax_D = max(
    internal_D[
        "hotspot_to_nonhotspot_ratio"
    ].max(),

    kelm_D[
        "hotspot_to_nonhotspot_ratio"
    ].max()
)


axD.text(
    5.0,
    ymax_D * 0.97,

    "Central region",

    ha="center",
    va="top",

    fontsize=9.5,

    fontweight="bold"
)


legD = axD.legend(
    frameon=False,

    loc="upper right",

    fontsize=9.5
)


for text in legD.get_texts():
    text.set_fontweight("bold")


bold_ticks(axD)
clean_axes(axD)

add_panel_letter(
    axD,
    "D"
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_7_Physicochemical_and_Positional_Grammar_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_7_Physicochemical_and_Positional_Grammar_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_7_Physicochemical_and_Positional_Grammar_FINAL.svg"
)


fig.savefig(
    png_file,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    pdf_file,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    svg_file,

    bbox_inches="tight",

    pad_inches=0.05
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 7 GENERATED SUCCESSFULLY")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/figures_main/Figure_7_Physicochemical_and_Positional_Grammar_FINAL.png"
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob
import os

files = glob.glob("/content/drive/MyDrive/**/*.ipynb", recursive=True)

print(f"Found {len(files)} notebooks:\n")

for f in files:
    print(f)

In [ ]:
# ============================================================
# PROJECT PATHS — matching the original notebook
# ============================================================

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

DIRS = {
    "code": PROJECT_DIR / "00_code",
    "data_original": PROJECT_DIR / "01_data_original",
    "data_processed": PROJECT_DIR / "02_data_processed",
    "embeddings": PROJECT_DIR / "03_embeddings",
    "models": PROJECT_DIR / "04_models",
    "predictions": PROJECT_DIR / "05_predictions",
    "xai": PROJECT_DIR / "06_xai",
    "results": PROJECT_DIR / "07_results",
    "checkpoints": PROJECT_DIR / "08_checkpoints",
    "logs": PROJECT_DIR / "09_logs",
}

print("PROJECT:", PROJECT_DIR)
print("RESULTS:", DIRS["results"])
print("XAI:", DIRS["xai"])

assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}"
assert DIRS["xai"].exists(), f"XAI folder not found: {DIRS['xai']}"
assert DIRS["results"].exists(), f"Results folder not found: {DIRS['results']}"

print("\n✓ Correct project paths initialized")

In [ ]:
for dataset_name in ["internal_test", "kelm_external"]:

    f = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    print("\nDataset:", dataset_name)
    print("File:", f)
    print("Exists:", f.exists())

In [ ]:
# ============================================================
# STEP 20B: HOTSPOT-THRESHOLD ROBUSTNESS
# Tests top 10%, 15%, 20%, 25%, and 30%
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

DATASETS = [
    "internal_test",
    "kelm_external",
]

THRESHOLDS = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]

AA20 = list("ACDEFGHIKLMNPQRSTVWY")

OUT = DIRS["results"] / "tables_SI"
OUT.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT)


# ------------------------------------------------------------
# BENJAMINI-HOCHBERG FDR
# ------------------------------------------------------------

def bh_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)

    n = len(p_values)

    order = np.argsort(p_values)

    ranked = p_values[order]

    adjusted = (
        ranked
        * n
        / np.arange(1, n + 1)
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1,
    )

    output = np.empty(
        n,
        dtype=float,
    )

    output[order] = adjusted

    return output


# ------------------------------------------------------------
# ODDS RATIO WITH 0.5 CORRECTION
# ------------------------------------------------------------

def corrected_odds_ratio(a, b, c, d):

    return (
        (a + 0.5)
        * (d + 0.5)
    ) / (
        (b + 0.5)
        * (c + 0.5)
    )


# ------------------------------------------------------------
# RUN ANALYSIS
# ------------------------------------------------------------

rows = []

for dataset_name in DATASETS:

    input_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    print("\n" + "=" * 80)
    print("Loading:", dataset_name)
    print(input_file)

    df = pd.read_csv(input_file)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    for fraction in THRESHOLDS:

        cutoff = 1.0 - fraction

        work = df.copy()

        # Hotspot based on percentile rank
        work["hotspot"] = (
            work["adjusted_global_consensus_rank"]
            >= cutoff
        )

        # CPP residues only
        cpp = work[
            work["label"].astype(int) == 1
        ].copy()

        print(
            f"{dataset_name} | "
            f"top {int(fraction * 100)}% | "
            f"hotspots = {cpp['hotspot'].sum()} / {len(cpp)}"
        )

        for aa in AA20:

            # ----------------------------
            # contingency table
            #
            #                AA   Other
            # hotspot         a     b
            # non-hotspot     c     d
            # ----------------------------

            a = int(
                (
                    (cpp["residue"] == aa)
                    &
                    cpp["hotspot"]
                ).sum()
            )

            b = int(
                (
                    (cpp["residue"] != aa)
                    &
                    cpp["hotspot"]
                ).sum()
            )

            c = int(
                (
                    (cpp["residue"] == aa)
                    &
                    (~cpp["hotspot"])
                ).sum()
            )

            d = int(
                (
                    (cpp["residue"] != aa)
                    &
                    (~cpp["hotspot"])
                ).sum()
            )

            fisher_or, p_value = fisher_exact(
                [
                    [a, b],
                    [c, d],
                ],
                alternative="two-sided",
            )

            corrected_or = corrected_odds_ratio(
                a, b, c, d
            )

            rows.append(
                {
                    "dataset": dataset_name,
                    "threshold_pct": int(fraction * 100),
                    "residue": aa,
                    "hotspot_aa": a,
                    "hotspot_other": b,
                    "nonhotspot_aa": c,
                    "nonhotspot_other": d,
                    "odds_ratio": corrected_or,
                    "fisher_OR_raw": fisher_or,
                    "p_value": p_value,
                }
            )


# ------------------------------------------------------------
# COMBINE RESULTS
# ------------------------------------------------------------

results = pd.DataFrame(rows)


# ------------------------------------------------------------
# FDR CORRECTION WITHIN EACH DATASET + THRESHOLD
# ------------------------------------------------------------

results["fdr"] = np.nan

for (dataset_name, threshold), indices in results.groupby(
    ["dataset", "threshold_pct"]
).groups.items():

    pvals = results.loc[
        indices,
        "p_value"
    ].values

    results.loc[
        indices,
        "fdr"
    ] = bh_adjust(pvals)


results["significant_fdr_0_05"] = (
    results["fdr"] < 0.05
)


results["direction"] = np.where(
    results["odds_ratio"] > 1,
    "enriched",
    np.where(
        results["odds_ratio"] < 1,
        "depleted",
        "neutral",
    ),
)


# ------------------------------------------------------------
# CROSS-DATASET ROBUSTNESS
# ------------------------------------------------------------

stability_rows = []

for (threshold, residue), group in results.groupby(
    ["threshold_pct", "residue"]
):

    temp = group.set_index("dataset")

    if (
        "internal_test" not in temp.index
        or
        "kelm_external" not in temp.index
    ):
        continue

    internal_or = float(
        temp.loc[
            "internal_test",
            "odds_ratio"
        ]
    )

    kelm_or = float(
        temp.loc[
            "kelm_external",
            "odds_ratio"
        ]
    )

    internal_fdr = float(
        temp.loc[
            "internal_test",
            "fdr"
        ]
    )

    kelm_fdr = float(
        temp.loc[
            "kelm_external",
            "fdr"
        ]
    )

    internal_direction = (
        "enriched"
        if internal_or > 1
        else "depleted"
    )

    kelm_direction = (
        "enriched"
        if kelm_or > 1
        else "depleted"
    )

    same_direction = (
        internal_direction
        == kelm_direction
    )

    significant_both = (
        internal_fdr < 0.05
        and
        kelm_fdr < 0.05
    )

    stability_rows.append(
        {
            "threshold_pct": threshold,
            "residue": residue,
            "internal_OR": internal_or,
            "kelm_OR": kelm_or,
            "internal_FDR": internal_fdr,
            "kelm_FDR": kelm_fdr,
            "internal_direction": internal_direction,
            "kelm_direction": kelm_direction,
            "same_direction": same_direction,
            "significant_both": significant_both,
        }
    )


stability = pd.DataFrame(
    stability_rows
)


# ------------------------------------------------------------
# SAVE FULL TABLES
# ------------------------------------------------------------

results_file = (
    OUT
    / "hotspot_threshold_robustness_all_residues.csv"
)

stability_file = (
    OUT
    / "hotspot_threshold_robustness_cross_dataset.csv"
)

results.to_csv(
    results_file,
    index=False,
)

stability.to_csv(
    stability_file,
    index=False,
)


# ------------------------------------------------------------
# DISPLAY RESIDUES CURRENTLY DISCUSSED IN MANUSCRIPT
# ------------------------------------------------------------

key_residues = [
    "K",
    "R",
    "L",
    "Q",
    "M",
    "Y",
    "G",
    "F",
    "N",
]

key = stability[
    stability["residue"].isin(
        key_residues
    )
].copy()


print("\n" + "=" * 100)
print("KEY RESIDUES ACROSS ALL HOTSPOT THRESHOLDS")
print("=" * 100)

display(
    key.sort_values(
        [
            "residue",
            "threshold_pct",
        ]
    )
)


# ------------------------------------------------------------
# ROBUSTNESS SUMMARY
# ------------------------------------------------------------

summary_rows = []

for residue in key_residues:

    g = key[
        key["residue"] == residue
    ].copy()

    if len(g) == 0:
        continue

    summary_rows.append(
        {
            "residue": residue,

            "n_thresholds":
                g["threshold_pct"].nunique(),

            "same_direction_all":
                bool(
                    g["same_direction"].all()
                ),

            "significant_both_all":
                bool(
                    g["significant_both"].all()
                ),

            "internal_OR_min":
                g["internal_OR"].min(),

            "internal_OR_max":
                g["internal_OR"].max(),

            "kelm_OR_min":
                g["kelm_OR"].min(),

            "kelm_OR_max":
                g["kelm_OR"].max(),
        }
    )


summary = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 100)
print("ROBUSTNESS SUMMARY")
print("=" * 100)

display(summary)


# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

summary_file = (
    OUT
    / "hotspot_threshold_robustness_key_residues_summary.csv"
)

summary.to_csv(
    summary_file,
    index=False,
)


print("\n" + "=" * 100)
print("DONE")
print("=" * 100)

print("Saved:")
print(results_file)
print(stability_file)
print(summary_file)

In [ ]:
# ============================================================
# CHECK WHY RANK-BASED HOTSPOT FRACTIONS EXCEED TARGET %
# ============================================================

import pandas as pd
import numpy as np

for dataset_name in [
    "internal_test",
    "kelm_external",
]:

    f = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    df = pd.read_csv(f)

    cpp = df[
        df["label"].astype(int) == 1
    ].copy()

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    print(
        "Number CPP sequences:",
        cpp["sequence_id"].nunique()
    )

    print(
        "Number CPP residues:",
        len(cpp)
    )

    print(
        "\nUnique global consensus ranks:",
        cpp["adjusted_global_consensus_rank"].nunique()
    )

    print("\nExisting hotspot columns:")

    for col in [
        "adjusted_hotspot_top10",
        "adjusted_hotspot_top15",
        "adjusted_hotspot_top20",
    ]:

        n = cpp[col].astype(bool).sum()

        print(
            col,
            ":",
            n,
            "/",
            len(cpp),
            "=",
            round(
                100 * n / len(cpp),
                2
            ),
            "%"
        )

    print("\nRank-derived thresholds:")

    for fraction in [
        0.10,
        0.15,
        0.20,
        0.25,
        0.30,
    ]:

        cutoff = 1 - fraction

        mask = (
            cpp[
                "adjusted_global_consensus_rank"
            ]
            >= cutoff
        )

        print(
            f"Target top {int(fraction*100)}%:",
            mask.sum(),
            "/",
            len(cpp),
            "=",
            round(
                100 * mask.mean(),
                2
            ),
            "%"
        )

In [ ]:
# ============================================================
# STEP 20C: COMPOSITION-CONTROLLED MOTIF VALIDATION
#
# Question:
# Are RR, WK, WR, RW, RL, LR enriched because of their
# local ordering, or simply because R/K/etc. are already
# abundant in hotspot regions?
#
# Null model:
# Within every CPP separately:
#   - preserve peptide length
#   - preserve hotspot positions
#   - preserve hotspot amino-acid composition
#   - preserve non-hotspot amino-acid composition
#   - shuffle residues within hotspot/non-hotspot positions
#
# Thus, residue composition is held constant while local
# sequence ordering is randomized.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

DATASETS = [
    "internal_test",
    "kelm_external",
]

# Motifs already replicated in your manuscript
CANDIDATES = [
    "RR",
    "WK",
    "WR",
    "RW",
    "RL",
    "LR",
]

N_SHUFFLES = 2000
SEED = 42

OUT = DIRS["results"] / "tables_SI"
OUT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# BENJAMINI-HOCHBERG
# ------------------------------------------------------------

def bh_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(p_values)

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1,
    )

    output = np.empty(
        n,
        dtype=float,
    )

    output[
        order
    ] = adjusted

    return output


# ------------------------------------------------------------
# DETECT MOTIF OCCURRENCE OVERLAPPING ≥1 HOTSPOT
#
# Returns True/False at peptide level.
# ------------------------------------------------------------

def motif_present_overlap(
    sequence,
    hotspot_mask,
    motif,
):

    m = len(motif)

    for start in range(
        len(sequence) - m + 1
    ):

        end = start + m

        if sequence[start:end] == motif:

            if np.any(
                hotspot_mask[start:end]
            ):

                return True

    return False


# ------------------------------------------------------------
# COMPOSITION-PRESERVING SHUFFLE
# ------------------------------------------------------------

def shuffle_sequence(
    sequence,
    hotspot_mask,
    rng,
):

    residues = np.array(
        list(sequence),
        dtype="U1",
    )

    hotspot_indices = np.flatnonzero(
        hotspot_mask
    )

    nonhotspot_indices = np.flatnonzero(
        ~hotspot_mask
    )

    # Shuffle hotspot residues only among hotspot positions
    if len(hotspot_indices) > 1:

        residues[
            hotspot_indices
        ] = rng.permutation(
            residues[
                hotspot_indices
            ]
        )

    # Shuffle non-hotspot residues only among non-hotspot positions
    if len(nonhotspot_indices) > 1:

        residues[
            nonhotspot_indices
        ] = rng.permutation(
            residues[
                nonhotspot_indices
            ]
        )

    return "".join(
        residues.tolist()
    )


# ------------------------------------------------------------
# RUN ANALYSIS
# ------------------------------------------------------------

all_rows = []


for dataset_name in DATASETS:

    print("\n" + "=" * 100)
    print("DATASET:", dataset_name)
    print("=" * 100)

    input_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    df = pd.read_csv(
        input_file
    )

    # CPPs only
    cpp = df[
        df["label"].astype(int) == 1
    ].copy()

    records = []

    # --------------------------------------------------------
    # Reconstruct individual peptide sequences + hotspot masks
    # --------------------------------------------------------

    for sequence_id, g in cpp.groupby(
        "sequence_id",
        sort=False,
    ):

        g = g.sort_values(
            "position"
        )

        sequence = str(
            g["sequence"].iloc[0]
        )

        hotspot_mask = (
            g[
                "adjusted_hotspot_top20"
            ]
            .astype(bool)
            .to_numpy()
        )

        if len(sequence) != len(hotspot_mask):

            raise ValueError(
                f"Length mismatch for {sequence_id}: "
                f"{len(sequence)} vs {len(hotspot_mask)}"
            )

        records.append(
            (
                str(sequence_id),
                sequence,
                hotspot_mask,
            )
        )

    print(
        "CPP sequences:",
        len(records)
    )


    # --------------------------------------------------------
    # OBSERVED COUNTS
    # number of CPP peptides containing motif overlapping hotspot
    # --------------------------------------------------------

    observed = {}

    for motif in CANDIDATES:

        observed[motif] = sum(

            motif_present_overlap(
                sequence,
                hotspot_mask,
                motif,
            )

            for _, sequence, hotspot_mask
            in records
        )

    print("\nObserved motif-containing CPP sequences:")

    for motif in CANDIDATES:

        print(
            motif,
            ":",
            observed[motif]
        )


    # --------------------------------------------------------
    # NULL DISTRIBUTIONS
    # --------------------------------------------------------

    null_counts = {

        motif: np.zeros(
            N_SHUFFLES,
            dtype=int,
        )

        for motif in CANDIDATES
    }


    rng = np.random.default_rng(
        SEED
        if dataset_name == "internal_test"
        else SEED + 10000
    )


    for shuffle_index in range(
        N_SHUFFLES
    ):

        current_counts = {
            motif: 0
            for motif in CANDIDATES
        }

        for (
            sequence_id,
            sequence,
            hotspot_mask,
        ) in records:

            shuffled = shuffle_sequence(
                sequence,
                hotspot_mask,
                rng,
            )

            for motif in CANDIDATES:

                if motif_present_overlap(
                    shuffled,
                    hotspot_mask,
                    motif,
                ):

                    current_counts[
                        motif
                    ] += 1


        for motif in CANDIDATES:

            null_counts[
                motif
            ][shuffle_index] = (
                current_counts[
                    motif
                ]
            )


        if (
            shuffle_index + 1
        ) % 200 == 0:

            print(
                f"Completed "
                f"{shuffle_index + 1}/"
                f"{N_SHUFFLES}"
            )


    # --------------------------------------------------------
    # EMPIRICAL STATISTICS
    # --------------------------------------------------------

    for motif in CANDIDATES:

        null_values = (
            null_counts[motif]
        )

        obs = observed[motif]

        null_mean = (
            null_values.mean()
        )

        # empirical one-sided P for enrichment
        p_empirical = (
            1
            + np.sum(
                null_values >= obs
            )
        ) / (
            N_SHUFFLES + 1
        )

        fold_over_null = (
            obs + 0.5
        ) / (
            null_mean + 0.5
        )

        all_rows.append(
            {
                "dataset":
                    dataset_name,

                "motif":
                    motif,

                "n_cpp":
                    len(records),

                "observed_sequences":
                    obs,

                "null_mean":
                    null_mean,

                "null_sd":
                    null_values.std(
                        ddof=1
                    ),

                "null_2.5pct":
                    np.quantile(
                        null_values,
                        0.025,
                    ),

                "null_97.5pct":
                    np.quantile(
                        null_values,
                        0.975,
                    ),

                "fold_over_null":
                    fold_over_null,

                "empirical_p":
                    p_empirical,
            }
        )


# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

results = pd.DataFrame(
    all_rows
)


# ------------------------------------------------------------
# FDR WITHIN EACH DATASET
# ------------------------------------------------------------

results[
    "fdr"
] = np.nan


for dataset_name, indices in results.groupby(
    "dataset"
).groups.items():

    results.loc[
        indices,
        "fdr"
    ] = bh_adjust(

        results.loc[
            indices,
            "empirical_p"
        ].values

    )


results[
    "significant_after_composition_control"
] = (
    (results["fold_over_null"] > 1)
    &
    (results["fdr"] < 0.05)
)


# ------------------------------------------------------------
# CROSS-DATASET REPLICATION
# ------------------------------------------------------------

rep_rows = []


for motif in CANDIDATES:

    g = results[
        results["motif"] == motif
    ].set_index(
        "dataset"
    )

    internal = g.loc[
        "internal_test"
    ]

    kelm = g.loc[
        "kelm_external"
    ]

    rep_rows.append(
        {
            "motif":
                motif,

            "internal_observed":
                internal[
                    "observed_sequences"
                ],

            "internal_null_mean":
                internal[
                    "null_mean"
                ],

            "internal_fold":
                internal[
                    "fold_over_null"
                ],

            "internal_FDR":
                internal[
                    "fdr"
                ],

            "kelm_observed":
                kelm[
                    "observed_sequences"
                ],

            "kelm_null_mean":
                kelm[
                    "null_mean"
                ],

            "kelm_fold":
                kelm[
                    "fold_over_null"
                ],

            "kelm_FDR":
                kelm[
                    "fdr"
                ],

            "replicated_after_composition_control":
                bool(
                    internal[
                        "significant_after_composition_control"
                    ]
                    and
                    kelm[
                        "significant_after_composition_control"
                    ]
                ),
        }
    )


replication = pd.DataFrame(
    rep_rows
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

results_file = (
    OUT
    / "motif_composition_controlled_null.csv"
)

replication_file = (
    OUT
    / "motif_composition_controlled_replication.csv"
)


results.to_csv(
    results_file,
    index=False,
)

replication.to_csv(
    replication_file,
    index=False,
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("COMPOSITION-CONTROLLED MOTIF RESULTS")
print("=" * 100)

display(
    results.sort_values(
        [
            "motif",
            "dataset",
        ]
    )
)


print("\n" + "=" * 100)
print("CROSS-DATASET REPLICATION")
print("=" * 100)

display(
    replication
)


print("\nSaved:")
print(results_file)
print(replication_file)

In [ ]:
# ============================================================
# STEP 20D-A
# SEQUENCE-LEVEL MUTATION FAITHFULNESS
# SETUP + FILE VERIFICATION
# ============================================================

from pathlib import Path
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd

import torch
import tensorflow as tf

from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# BASIC SETTINGS
# ------------------------------------------------------------

SEED = 42
MAX_LEN = 61
RANDOM_REPEATS = 20
N_BOOTSTRAP = 2000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Torch device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# PROJECT PATHS
# DIRS should already exist from previous cells,
# but redefine safely if needed.
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

DIRS = {
    "data_processed":
        PROJECT_DIR / "02_data_processed",

    "embeddings":
        PROJECT_DIR / "03_embeddings",

    "models":
        PROJECT_DIR / "04_models",

    "predictions":
        PROJECT_DIR / "05_predictions",

    "xai":
        PROJECT_DIR / "06_xai",

    "results":
        PROJECT_DIR / "07_results",

    "checkpoints":
        PROJECT_DIR / "08_checkpoints",
}


# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

OUT = (
    DIRS["results"]
    / "tables_SI"
    / "sequence_mutation_faithfulness"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Output:", OUT)


# ------------------------------------------------------------
# PLM SETTINGS
# EXACT MODELS FROM ORIGINAL PIPELINE
# ------------------------------------------------------------

MODEL_CONFIGS = {

    "ESM2_320": {
        "type":
            "esm",

        "loader":
            "esm2_t6_8M_UR50D",

        "layer":
            6,

        "embedding_dimension":
            320,

        "batch_size":
            64,
    },

    "ESM2_640": {
        "type":
            "esm",

        "loader":
            "esm2_t30_150M_UR50D",

        "layer":
            30,

        "embedding_dimension":
            640,

        "batch_size":
            24,
    },

    "ESM2_1280": {
        "type":
            "esm",

        "loader":
            "esm2_t33_650M_UR50D",

        "layer":
            33,

        "embedding_dimension":
            1280,

        "batch_size":
            8,
    },

    "ProtT5": {
        "type":
            "prott5",

        "loader":
            "Rostlab/prot_t5_xl_uniref50",

        "layer":
            None,

        "embedding_dimension":
            1024,

        "batch_size":
            8,
    },
}


DATASETS = [
    "internal_test",
    "kelm_external",
]


PREDICTION_FILENAMES = {
    "internal_test":
        "internal_test_predictions.csv",

    "kelm_external":
        "kelm_external_predictions.csv",
}


# ------------------------------------------------------------
# VERIFY ALL REQUIRED FILES
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VERIFYING REQUIRED FILES")
print("=" * 100)

all_ok = True

for model_name in MODEL_CONFIGS:

    model_file = (
        DIRS["models"]
        / model_name
        / "final_attention_classifier.keras"
    )

    print(
        f"\n{model_name}"
    )

    print(
        "Classifier:",
        model_file.exists(),
        model_file
    )

    if not model_file.exists():
        all_ok = False


    for dataset_name in DATASETS:

        prediction_file = (
            DIRS["predictions"]
            / model_name
            / PREDICTION_FILENAMES[
                dataset_name
            ]
        )

        print(
            dataset_name,
            "predictions:",
            prediction_file.exists()
        )

        if not prediction_file.exists():
            all_ok = False


for dataset_name in DATASETS:

    consensus_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    print(
        "\nConsensus:",
        dataset_name,
        consensus_file.exists()
    )

    if not consensus_file.exists():
        all_ok = False


print("\n" + "=" * 100)

if all_ok:
    print("✓ ALL REQUIRED FILES FOUND")
else:
    print("✗ SOME REQUIRED FILES ARE MISSING")

print("=" * 100)

In [ ]:
# ============================================================
# STEP 20D-B
# FUNCTIONS FOR ACTUAL SEQUENCE MUTATION FAITHFULNESS
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch
import tensorflow as tf

from scipy.stats import wilcoxon


# ============================================================
# 1. CUSTOM ATTENTION LAYER
# EXACTLY MATCHES ORIGINAL CLASSIFIER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="pLM4CPP"
)
class MaskedAttentionPooling(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        **kwargs,
    ):

        super().__init__(
            **kwargs
        )

        self.attention_dense = (
            tf.keras.layers.Dense(
                1,
                use_bias=True,
                name="residue_attention_score",
            )
        )


    def build(
        self,
        input_shape,
    ):

        residue_shape, _ = (
            input_shape
        )

        self.attention_dense.build(
            residue_shape
        )

        super().build(
            input_shape
        )


    def call(
        self,
        inputs,
    ):

        residue_features, residue_mask = (
            inputs
        )

        logits = self.attention_dense(
            residue_features
        )

        logits = tf.squeeze(
            logits,
            axis=-1,
        )

        residue_mask = tf.cast(
            residue_mask,
            logits.dtype,
        )

        masked_logits = (
            logits
            +
            (
                1.0
                - residue_mask
            )
            *
            tf.cast(
                -1e4,
                logits.dtype,
            )
        )

        attention_weights = (
            tf.nn.softmax(
                masked_logits,
                axis=1,
            )
        )

        pooled_vector = (
            tf.reduce_sum(
                residue_features
                *
                tf.expand_dims(
                    attention_weights,
                    axis=-1,
                ),
                axis=1,
            )
        )

        return pooled_vector


    def get_config(
        self,
    ):

        return super().get_config()


# ============================================================
# 2. CLEAR MEMORY
# ============================================================

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


# ============================================================
# 3. ALANINE MUTATION
#
# positions are ZERO-BASED here
# ============================================================

def mutate_to_alanine(
    sequence,
    positions,
):

    residues = list(
        sequence
    )

    for position in positions:

        if residues[position] != "A":
            residues[position] = "A"

    return "".join(
        residues
    )


# ============================================================
# 4. LOAD CONSENSUS HOTSPOTS FOR CPPs
# ============================================================

def load_consensus_cpp_records(
    dataset_name,
):

    consensus_file = (
        DIRS["xai"]
        / "consensus_adjusted"
        / "global"
        / dataset_name
        / "adjusted_global_consensus_residue_scores.csv"
    )

    df = pd.read_csv(
        consensus_file
    )

    # CPP only
    df = df[
        df["label"].astype(int)
        == 1
    ].copy()


    records = {}


    for sequence_id, group in df.groupby(
        "sequence_id",
        sort=False,
    ):

        group = group.sort_values(
            "position"
        )

        sequence = str(
            group["sequence"].iloc[0]
        )

        hotspot_mask = (
            group[
                "adjusted_hotspot_top20"
            ]
            .astype(bool)
            .to_numpy()
        )


        if len(sequence) != len(
            hotspot_mask
        ):

            raise ValueError(
                f"Length mismatch for "
                f"{sequence_id}"
            )


        # ----------------------------------------------------
        # IMPORTANT:
        # Existing alanines are excluded because A -> A is
        # not an actual mutation.
        # ----------------------------------------------------

        hotspot_nonA = np.array(
            [
                i
                for i, is_hotspot
                in enumerate(
                    hotspot_mask
                )
                if (
                    is_hotspot
                    and sequence[i] != "A"
                )
            ],
            dtype=int,
        )


        nonhotspot_nonA = np.array(
            [
                i
                for i, is_hotspot
                in enumerate(
                    hotspot_mask
                )
                if (
                    (not is_hotspot)
                    and sequence[i] != "A"
                )
            ],
            dtype=int,
        )


        records[
            str(sequence_id)
        ] = {

            "sequence":
                sequence,

            "hotspot_mask":
                hotspot_mask,

            "hotspot_nonA":
                hotspot_nonA,

            "nonhotspot_nonA":
                nonhotspot_nonA,
        }


    return records


# ============================================================
# 5. LOAD CORRECTLY CLASSIFIED CPP IDs FOR ONE MODEL
# ============================================================

def load_correct_cpp_predictions(
    model_name,
    dataset_name,
):

    prediction_file = (
        DIRS["predictions"]
        / model_name
        / PREDICTION_FILENAMES[
            dataset_name
        ]
    )

    df = pd.read_csv(
        prediction_file
    )


    required = [
        "sequence_id",
        "sequence",
        "label",
        "probability_CPP",
        "predicted_label",
        "correct_prediction",
    ]


    missing = [
        column
        for column in required
        if column not in df.columns
    ]


    if missing:

        raise KeyError(
            f"{prediction_file}\n"
            f"Missing columns: {missing}\n"
            f"Available: {df.columns.tolist()}"
        )


    cpp = df[
        (df["label"].astype(int) == 1)
        &
        (df["correct_prediction"].astype(bool))
        &
        (df["predicted_label"].astype(int) == 1)
    ].copy()


    cpp[
        "sequence_id"
    ] = (
        cpp[
            "sequence_id"
        ].astype(str)
    )


    return cpp


# ============================================================
# 6. BUILD MUTATION PANEL
#
# One hotspot mutant + 20 matched random mutants per peptide.
# Random mutations have EXACTLY the same number of actual
# non-A substitutions as hotspot mutation.
# ============================================================

def build_mutation_panel(
    model_name,
    dataset_name,
    random_repeats=20,
    seed=42,
):

    consensus_records = (
        load_consensus_cpp_records(
            dataset_name
        )
    )


    prediction_cpp = (
        load_correct_cpp_predictions(
            model_name,
            dataset_name,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    panel_rows = []
    sequence_info = []


    for row in prediction_cpp.itertuples(
        index=False
    ):

        sequence_id = str(
            row.sequence_id
        )

        if sequence_id not in (
            consensus_records
        ):
            continue


        rec = consensus_records[
            sequence_id
        ]


        sequence = rec[
            "sequence"
        ]


        hotspot_positions = rec[
            "hotspot_nonA"
        ]


        control_positions = rec[
            "nonhotspot_nonA"
        ]


        n_mutations = len(
            hotspot_positions
        )


        # Need at least one true hotspot mutation.
        if n_mutations == 0:
            continue


        # Need sufficient non-hotspot positions for
        # matched random control.
        if len(control_positions) < (
            n_mutations
        ):
            continue


        original_probability = float(
            row.probability_CPP
        )


        # ----------------------------------------------------
        # Hotspot alanine mutant
        # ----------------------------------------------------

        hotspot_sequence = (
            mutate_to_alanine(
                sequence,
                hotspot_positions,
            )
        )


        panel_rows.append(
            {
                "sequence_id":
                    sequence_id,

                "variant_type":
                    "hotspot",

                "repeat":
                    -1,

                "sequence":
                    hotspot_sequence,

                "n_mutations":
                    n_mutations,
            }
        )


        # ----------------------------------------------------
        # Matched random mutants
        # ----------------------------------------------------

        for repeat in range(
            random_repeats
        ):

            selected = rng.choice(
                control_positions,
                size=n_mutations,
                replace=False,
            )


            random_sequence = (
                mutate_to_alanine(
                    sequence,
                    selected,
                )
            )


            panel_rows.append(
                {
                    "sequence_id":
                        sequence_id,

                    "variant_type":
                        "random",

                    "repeat":
                        repeat,

                    "sequence":
                        random_sequence,

                    "n_mutations":
                        n_mutations,
                }
            )


        sequence_info.append(
            {
                "sequence_id":
                    sequence_id,

                "original_sequence":
                    sequence,

                "original_probability":
                    original_probability,

                "n_hotspot_nonA":
                    n_mutations,

                "n_available_random_nonA":
                    len(
                        control_positions
                    ),
            }
        )


    panel = pd.DataFrame(
        panel_rows
    )

    info = pd.DataFrame(
        sequence_info
    )


    return (
        panel,
        info,
    )


# ============================================================
# 7. LOAD ESM MODEL
# ============================================================

def load_esm_model(
    config,
):

    import esm

    loader = getattr(
        esm.pretrained,
        config["loader"],
    )

    model, alphabet = loader()

    model = (
        model
        .to(
            DEVICE
        )
        .eval()
    )

    batch_converter = (
        alphabet
        .get_batch_converter()
    )


    return (
        model,
        batch_converter,
    )


# ============================================================
# 8. LOAD PROTT5
# ============================================================

def load_prott5_model(
    config,
):

    from transformers import (
        T5Tokenizer,
        T5EncoderModel,
    )


    tokenizer = (
        T5Tokenizer
        .from_pretrained(
            config["loader"],
            do_lower_case=False,
        )
    )


    dtype = (
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )


    model = (
        T5EncoderModel
        .from_pretrained(
            config["loader"],
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
        )
        .to(
            DEVICE
        )
        .eval()
    )


    return (
        model,
        tokenizer,
    )


# ============================================================
# 9. EMBED ESM SEQUENCES
# ============================================================

@torch.no_grad()
def embed_esm_sequences(
    sequences,
    model,
    batch_converter,
    layer,
    embedding_dimension,
):

    items = [
        (
            f"seq_{i}",
            sequence,
        )
        for i, sequence
        in enumerate(
            sequences
        )
    ]


    _, raw_sequences, tokens = (
        batch_converter(
            items
        )
    )


    tokens = tokens.to(
        DEVICE
    )


    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=torch.cuda.is_available(),
    ):

        outputs = model(
            tokens,
            repr_layers=[
                layer
            ],
            return_contacts=False,
        )


    representations = (
        outputs[
            "representations"
        ][
            layer
        ]
    )


    X = np.zeros(
        (
            len(sequences),
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )


    M = np.zeros(
        (
            len(sequences),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )


    for i, sequence in enumerate(
        raw_sequences
    ):

        length = min(
            len(sequence),
            MAX_LEN,
        )


        X[
            i,
            :length,
            :
        ] = (
            representations[
                i,
                1:length + 1,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(
                np.float16
            )
        )


        M[
            i,
            :length
        ] = 1


    del outputs
    del representations
    del tokens

    clear_memory()


    return (
        X,
        M,
    )


# ============================================================
# 10. EMBED PROTT5 SEQUENCES
# ============================================================

@torch.no_grad()
def embed_prott5_sequences(
    sequences,
    model,
    tokenizer,
    embedding_dimension,
):

    # ProtT5 expects spaces between amino acids.
    spaced = [
        " ".join(
            list(sequence)
        )
        for sequence
        in sequences
    ]


    encoded = tokenizer(
        spaced,
        add_special_tokens=True,
        padding=True,
        return_tensors="pt",
    )


    input_ids = (
        encoded[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    attention_mask = (
        encoded[
            "attention_mask"
        ]
        .to(
            DEVICE
        )
    )


    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=torch.cuda.is_available(),
    ):

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


    representations = (
        outputs.last_hidden_state
    )


    X = np.zeros(
        (
            len(sequences),
            MAX_LEN,
            embedding_dimension,
        ),
        dtype=np.float16,
    )


    M = np.zeros(
        (
            len(sequences),
            MAX_LEN,
        ),
        dtype=np.uint8,
    )


    for i, sequence in enumerate(
        sequences
    ):

        length = min(
            len(sequence),
            MAX_LEN,
        )


        # ProtT5 output starts directly with residue 1.
        X[
            i,
            :length,
            :
        ] = (
            representations[
                i,
                :length,
                :
            ]
            .float()
            .cpu()
            .numpy()
            .astype(
                np.float16
            )
        )


        M[
            i,
            :length
        ] = 1


    del outputs
    del representations
    del input_ids
    del attention_mask

    clear_memory()


    return (
        X,
        M,
    )


# ============================================================
# 11. PREDICT MUTATED SEQUENCES IN BATCHES
# ============================================================

def predict_mutation_panel(
    panel,
    model_name,
    config,
    plm,
    auxiliary,
    classifier,
):

    probabilities = np.zeros(
        len(panel),
        dtype=float,
    )


    batch_size = (
        config[
            "batch_size"
        ]
    )


    for start in range(
        0,
        len(panel),
        batch_size,
    ):

        stop = min(
            start + batch_size,
            len(panel),
        )


        batch_sequences = (
            panel.iloc[
                start:stop
            ][
                "sequence"
            ]
            .astype(str)
            .tolist()
        )


        if config[
            "type"
        ] == "esm":

            X, M = (
                embed_esm_sequences(
                    sequences=batch_sequences,
                    model=plm,
                    batch_converter=auxiliary,
                    layer=config[
                        "layer"
                    ],
                    embedding_dimension=config[
                        "embedding_dimension"
                    ],
                )
            )

        else:

            X, M = (
                embed_prott5_sequences(
                    sequences=batch_sequences,
                    model=plm,
                    tokenizer=auxiliary,
                    embedding_dimension=config[
                        "embedding_dimension"
                    ],
                )
            )


        # Classifier is tiny relative to PLM.
        batch_probability = (
            classifier(
                [
                    X,
                    M,
                ],
                training=False,
            )
            .numpy()
            .reshape(-1)
        )


        probabilities[
            start:stop
        ] = batch_probability


        del X
        del M
        del batch_probability

        clear_memory()


        if (
            stop % 500 == 0
            or stop == len(panel)
        ):

            print(
                f"  predicted "
                f"{stop}/{len(panel)} variants"
            )


    return probabilities


# ============================================================
# 12. SUMMARIZE MUTATION RESULTS PER SEQUENCE
# ============================================================

def summarize_mutation_panel(
    panel,
    info,
):

    merged = panel.merge(
        info,
        on="sequence_id",
        how="left",
        validate="many_to_one",
    )


    rows = []


    for sequence_id, g in merged.groupby(
        "sequence_id",
        sort=False,
    ):

        original_probability = float(
            g[
                "original_probability"
            ].iloc[0]
        )


        hotspot_values = (
            g[
                g["variant_type"]
                == "hotspot"
            ][
                "mutated_probability"
            ]
            .to_numpy(
                dtype=float
            )
        )


        random_values = (
            g[
                g["variant_type"]
                == "random"
            ][
                "mutated_probability"
            ]
            .to_numpy(
                dtype=float
            )
        )


        if (
            len(hotspot_values) != 1
            or len(random_values) == 0
        ):
            continue


        hotspot_probability = float(
            hotspot_values[0]
        )


        random_probability_mean = float(
            np.mean(
                random_values
            )
        )


        delta_hotspot = (
            original_probability
            - hotspot_probability
        )


        random_drops = (
            original_probability
            - random_values
        )


        delta_random = float(
            np.mean(
                random_drops
            )
        )


        faithfulness = (
            delta_hotspot
            - delta_random
        )


        rows.append(
            {
                "sequence_id":
                    sequence_id,

                "original_probability":
                    original_probability,

                "hotspot_mutant_probability":
                    hotspot_probability,

                "random_mutant_probability_mean":
                    random_probability_mean,

                "delta_hotspot":
                    delta_hotspot,

                "delta_random":
                    delta_random,

                "faithfulness_effect":
                    faithfulness,

                "positive_faithfulness":
                    faithfulness > 0,

                "n_mutations":
                    int(
                        g[
                            "n_mutations"
                        ].iloc[0]
                    ),

                "n_random_repeats":
                    len(
                        random_values
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 13. COHEN'S dz
# ============================================================

def paired_cohens_dz(
    hotspot,
    random,
):

    diff = (
        np.asarray(
            hotspot,
            dtype=float
        )
        -
        np.asarray(
            random,
            dtype=float
        )
    )


    if len(diff) < 2:
        return np.nan


    sd = np.std(
        diff,
        ddof=1,
    )


    if sd == 0:
        return np.nan


    return float(
        np.mean(diff)
        / sd
    )


# ============================================================
# 14. MATCHED-PAIRS RANK-BISERIAL CORRELATION
#
# Based on signed ranks of paired differences.
# ============================================================

def paired_rank_biserial(
    hotspot,
    random,
):

    diff = (
        np.asarray(
            hotspot,
            dtype=float
        )
        -
        np.asarray(
            random,
            dtype=float
        )
    )


    diff = diff[
        diff != 0
    ]


    if len(diff) == 0:
        return np.nan


    abs_diff = np.abs(
        diff
    )


    from scipy.stats import rankdata

    ranks = rankdata(
        abs_diff
    )


    positive = np.sum(
        ranks[
            diff > 0
        ]
    )


    negative = np.sum(
        ranks[
            diff < 0
        ]
    )


    total = (
        positive
        + negative
    )


    return float(
        (
            positive
            - negative
        )
        / total
    )


# ============================================================
# 15. BOOTSTRAP CI
# ============================================================

def bootstrap_mean_ci(
    values,
    n_bootstrap=2000,
    seed=42,
):

    values = np.asarray(
        values,
        dtype=float,
    )


    rng = np.random.default_rng(
        seed
    )


    means = np.empty(
        n_bootstrap,
        dtype=float,
    )


    n = len(
        values
    )


    for i in range(
        n_bootstrap
    ):

        sample = rng.choice(
            values,
            size=n,
            replace=True,
        )

        means[i] = np.mean(
            sample
        )


    return (
        float(
            np.quantile(
                means,
                0.025
            )
        ),
        float(
            np.quantile(
                means,
                0.975
            )
        ),
    )


# ============================================================
# 16. FINAL STATISTICAL SUMMARY
# ============================================================

def calculate_mutation_statistics(
    per_sequence,
    model_name,
    dataset_name,
):

    hotspot = (
        per_sequence[
            "delta_hotspot"
        ]
        .to_numpy(
            dtype=float
        )
    )


    random_control = (
        per_sequence[
            "delta_random"
        ]
        .to_numpy(
            dtype=float
        )
    )


    faithfulness = (
        hotspot
        - random_control
    )


    if len(
        per_sequence
    ) == 0:

        raise ValueError(
            "No sequences available "
            "for statistical analysis."
        )


    try:

        wilcoxon_result = (
            wilcoxon(
                hotspot,
                random_control,
                alternative="two-sided",
                zero_method="wilcox",
            )
        )

        statistic = float(
            wilcoxon_result.statistic
        )

        p_value = float(
            wilcoxon_result.pvalue
        )

    except ValueError:

        statistic = np.nan
        p_value = np.nan


    dz = paired_cohens_dz(
        hotspot,
        random_control,
    )


    rbc = paired_rank_biserial(
        hotspot,
        random_control,
    )


    hotspot_ci = bootstrap_mean_ci(
        hotspot,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
    )


    random_ci = bootstrap_mean_ci(
        random_control,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED + 1,
    )


    faith_ci = bootstrap_mean_ci(
        faithfulness,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED + 2,
    )


    return {

        "model":
            model_name,

        "dataset":
            dataset_name,

        "n_sequences":
            len(
                per_sequence
            ),

        "mean_delta_hotspot":
            float(
                np.mean(
                    hotspot
                )
            ),

        "hotspot_CI_low":
            hotspot_ci[0],

        "hotspot_CI_high":
            hotspot_ci[1],

        "mean_delta_random":
            float(
                np.mean(
                    random_control
                )
            ),

        "random_CI_low":
            random_ci[0],

        "random_CI_high":
            random_ci[1],

        "mean_faithfulness_effect":
            float(
                np.mean(
                    faithfulness
                )
            ),

        "faithfulness_CI_low":
            faith_ci[0],

        "faithfulness_CI_high":
            faith_ci[1],

        "median_faithfulness_effect":
            float(
                np.median(
                    faithfulness
                )
            ),

        "positive_faithfulness_fraction":
            float(
                np.mean(
                    faithfulness > 0
                )
            ),

        "wilcoxon_statistic":
            statistic,

        "wilcoxon_p":
            p_value,

        "paired_cohens_dz":
            dz,

        "rank_biserial":
            rbc,
    }


print(
    "✓ Sequence-mutation functions loaded"
)

In [ ]:
# ============================================================
# INSTALL ESM DEPENDENCY
# ============================================================

!pip -q install fair-esm==2.0.0

import esm

print("fair-esm version:", esm.__version__)
print("✓ ESM successfully installed")

In [ ]:
# ============================================================
# STEP 20D-C
# RUN ACTUAL SEQUENCE-LEVEL MUTATION FAITHFULNESS
#
# IMPORTANT:
# This can take substantially longer than the previous analyses.
#
# Each completed model/dataset pair is saved immediately.
# ============================================================

import gc
import json
import numpy as np
import pandas as pd
import torch
import tensorflow as tf


# ------------------------------------------------------------
# RESULT COLLECTION
# ------------------------------------------------------------

summary_rows = []


# ------------------------------------------------------------
# RUN ONE PLM AT A TIME
# ------------------------------------------------------------

for model_index, (
    model_name,
    config,
) in enumerate(
    MODEL_CONFIGS.items()
):

    print("\n\n" + "#" * 100)
    print(
        f"MODEL {model_index + 1}/"
        f"{len(MODEL_CONFIGS)}: "
        f"{model_name}"
    )
    print("#" * 100)


    # ========================================================
    # LOAD SAVED CLASSIFIER
    # ========================================================

    classifier_file = (
        DIRS["models"]
        / model_name
        / "final_attention_classifier.keras"
    )


    print(
        "Loading classifier:",
        classifier_file
    )


    classifier = (
        tf.keras.models.load_model(
            classifier_file,
            custom_objects={
                "MaskedAttentionPooling":
                    MaskedAttentionPooling,
            },
            compile=False,
        )
    )


    print(
        "✓ Classifier loaded"
    )


    # ========================================================
    # LOAD PLM
    # ========================================================

    print(
        "Loading pretrained PLM..."
    )


    if config[
        "type"
    ] == "esm":

        plm, auxiliary = (
            load_esm_model(
                config
            )
        )

    else:

        plm, auxiliary = (
            load_prott5_model(
                config
            )
        )


    print(
        "✓ PLM loaded"
    )


    # ========================================================
    # DATASETS
    # ========================================================

    for dataset_index, dataset_name in enumerate(
        DATASETS
    ):

        print("\n" + "=" * 100)

        print(
            f"{model_name} | "
            f"{dataset_name}"
        )

        print("=" * 100)


        # ----------------------------------------------------
        # RESTART-SAFE FILES
        # ----------------------------------------------------

        prefix = (
            f"{model_name}_"
            f"{dataset_name}"
        )


        per_sequence_file = (
            OUT
            / f"{prefix}_per_sequence.csv"
        )


        panel_file = (
            OUT
            / f"{prefix}_all_mutants.csv"
        )


        summary_file = (
            OUT
            / f"{prefix}_summary.json"
        )


        # ----------------------------------------------------
        # SKIP COMPLETED
        # ----------------------------------------------------

        if (
            per_sequence_file.exists()
            and summary_file.exists()
        ):

            print(
                "✓ Already completed — skipping"
            )


            with open(
                summary_file,
                "r",
                encoding="utf-8",
            ) as handle:

                existing_summary = (
                    json.load(
                        handle
                    )
                )


            summary_rows.append(
                existing_summary
            )

            continue


        # ----------------------------------------------------
        # BUILD ACTUAL MUTATION PANEL
        # ----------------------------------------------------

        panel, info = (
            build_mutation_panel(
                model_name=model_name,
                dataset_name=dataset_name,
                random_repeats=RANDOM_REPEATS,
                seed=(
                    SEED
                    + model_index * 1000
                    + dataset_index * 100
                ),
            )
        )


        print(
            "Correctly classified CPPs "
            "retained after mutation matching:",
            len(info)
        )


        print(
            "Total mutant sequences to evaluate:",
            len(panel)
        )


        if len(
            info
        ) == 0:

            raise ValueError(
                f"No eligible sequences for "
                f"{model_name}/{dataset_name}"
            )


        print(
            "\nMutation count distribution:"
        )

        display(
            info[
                "n_hotspot_nonA"
            ].describe()
        )


        # ----------------------------------------------------
        # PREDICT ALL MUTANTS
        # ----------------------------------------------------

        probabilities = (
            predict_mutation_panel(
                panel=panel,
                model_name=model_name,
                config=config,
                plm=plm,
                auxiliary=auxiliary,
                classifier=classifier,
            )
        )


        panel[
            "mutated_probability"
        ] = probabilities


        # ----------------------------------------------------
        # PER-SEQUENCE SUMMARY
        # ----------------------------------------------------

        per_sequence = (
            summarize_mutation_panel(
                panel=panel,
                info=info,
            )
        )


        # Add identifying information
        per_sequence[
            "model"
        ] = model_name

        per_sequence[
            "dataset"
        ] = dataset_name


        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        summary = (
            calculate_mutation_statistics(
                per_sequence=per_sequence,
                model_name=model_name,
                dataset_name=dataset_name,
            )
        )


        # ----------------------------------------------------
        # SAVE IMMEDIATELY
        # ----------------------------------------------------

        panel.to_csv(
            panel_file,
            index=False,
        )


        per_sequence.to_csv(
            per_sequence_file,
            index=False,
        )


        with open(
            summary_file,
            "w",
            encoding="utf-8",
        ) as handle:

            json.dump(
                summary,
                handle,
                indent=2,
            )


        summary_rows.append(
            summary
        )


        # ----------------------------------------------------
        # DISPLAY THIS RESULT
        # ----------------------------------------------------

        print("\n" + "-" * 100)

        print(
            "RESULT:",
            model_name,
            dataset_name
        )

        print("-" * 100)


        print(
            "N sequences:",
            summary[
                "n_sequences"
            ]
        )


        print(
            "Mean hotspot mutation Δp:",
            round(
                summary[
                    "mean_delta_hotspot"
                ],
                4,
            )
        )


        print(
            "Mean random mutation Δp:",
            round(
                summary[
                    "mean_delta_random"
                ],
                4,
            )
        )


        print(
            "Mean faithfulness:",
            round(
                summary[
                    "mean_faithfulness_effect"
                ],
                4,
            )
        )


        print(
            "95% CI faithfulness:",
            (
                round(
                    summary[
                        "faithfulness_CI_low"
                    ],
                    4,
                ),
                round(
                    summary[
                        "faithfulness_CI_high"
                    ],
                    4,
                ),
            )
        )


        print(
            "Positive faithfulness:",
            round(
                100
                *
                summary[
                    "positive_faithfulness_fraction"
                ],
                1,
            ),
            "%"
        )


        print(
            "Wilcoxon P:",
            summary[
                "wilcoxon_p"
            ]
        )


        print(
            "Cohen's dz:",
            round(
                summary[
                    "paired_cohens_dz"
                ],
                3,
            )
        )


        print(
            "Rank-biserial:",
            round(
                summary[
                    "rank_biserial"
                ],
                3,
            )
        )


        # ----------------------------------------------------
        # RELEASE TEMPORARY DATA
        # ----------------------------------------------------

        del panel
        del info
        del probabilities
        del per_sequence

        clear_memory()


    # ========================================================
    # RELEASE THIS PLM BEFORE LOADING NEXT ONE
    # ========================================================

    print(
        f"\nReleasing {model_name}..."
    )


    del plm
    del auxiliary
    del classifier

    tf.keras.backend.clear_session()

    clear_memory()


# ============================================================
# COMBINE SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)


summary_df = (
    summary_df
    .drop_duplicates(
        subset=[
            "model",
            "dataset",
        ],
        keep="last",
    )
    .sort_values(
        [
            "dataset",
            "model",
        ]
    )
    .reset_index(
        drop=True
    )
)


combined_summary_file = (
    OUT
    / "sequence_level_mutation_faithfulness_summary.csv"
)


summary_df.to_csv(
    combined_summary_file,
    index=False,
)


print("\n\n" + "=" * 120)
print("FINAL SEQUENCE-LEVEL MUTATION FAITHFULNESS")
print("=" * 120)


display(
    summary_df[
        [
            "model",
            "dataset",
            "n_sequences",
            "mean_delta_hotspot",
            "mean_delta_random",
            "mean_faithfulness_effect",
            "faithfulness_CI_low",
            "faithfulness_CI_high",
            "positive_faithfulness_fraction",
            "wilcoxon_p",
            "paired_cohens_dz",
            "rank_biserial",
        ]
    ]
)


print(
    "\nSaved combined summary:"
)

print(
    combined_summary_file
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# FIGURE 4E
# Composition-controlled motif enrichment
#
# Grouped bars:
#   Internal vs KELM
# Y-axis:
#   Fold over composition-preserving null
#
# Significance:
#   *  FDR < 0.05
#
# Output:
#   Figure4E_Composition_Controlled_Motifs.png
#   Figure4E_Composition_Controlled_Motifs.pdf
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

INPUT_FILE = (
    PROJECT_DIR
    / "07_results"
    / "tables_SI"
    / "motif_composition_controlled_null.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "07_results"
    / "figures_main"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PNG_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.png"
)

PDF_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.pdf"
)

print("Input exists:", INPUT_FILE.exists())
print("Input:", INPUT_FILE)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

motif_order = [
    "RR",
    "WK",
    "WR",
    "RW",
    "RL",
    "LR",
]

df["motif"] = pd.Categorical(
    df["motif"],
    categories=motif_order,
    ordered=True
)

df = df.sort_values(
    ["motif", "dataset"]
)

# ------------------------------------------------------------
# SPLIT DATASETS
# ------------------------------------------------------------

internal = (
    df[df["dataset"] == "internal_test"]
    .set_index("motif")
    .loc[motif_order]
)

kelm = (
    df[df["dataset"] == "kelm_external"]
    .set_index("motif")
    .loc[motif_order]
)

internal_fold = internal["fold_over_null"].to_numpy()
kelm_fold = kelm["fold_over_null"].to_numpy()

internal_fdr = internal["fdr"].to_numpy()
kelm_fdr = kelm["fdr"].to_numpy()

# ------------------------------------------------------------
# PLOT SETTINGS
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.2,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, ax = plt.subplots(
    figsize=(6.2, 4.2)
)

x = np.arange(
    len(motif_order)
)

bar_width = 0.36

# ------------------------------------------------------------
# BARS
# ------------------------------------------------------------

bars_internal = ax.bar(
    x - bar_width / 2,
    internal_fold,
    width=bar_width,
    label="Internal",
    edgecolor="black",
    linewidth=1.0,
)

bars_kelm = ax.bar(
    x + bar_width / 2,
    kelm_fold,
    width=bar_width,
    label="KELM",
    edgecolor="black",
    linewidth=1.0,
)

# ------------------------------------------------------------
# REFERENCE LINE
# ------------------------------------------------------------

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1.2,
    color="black",
)

# ------------------------------------------------------------
# SIGNIFICANCE STARS
# ------------------------------------------------------------

def add_sig_stars(
    bars,
    fdr_values,
):

    for bar, fdr in zip(
        bars,
        fdr_values
    ):

        if fdr < 0.001:
            star = "***"
        elif fdr < 0.01:
            star = "**"
        elif fdr < 0.05:
            star = "*"
        else:
            star = ""

        if star:

            height = (
                bar.get_height()
            )

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height + 0.07,
                star,
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold",
            )

add_sig_stars(
    bars_internal,
    internal_fdr
)

add_sig_stars(
    bars_kelm,
    kelm_fdr
)

# ------------------------------------------------------------
# VALUE LABELS
# ------------------------------------------------------------

def add_value_labels(
    bars,
):

    for bar in bars:

        height = (
            bar.get_height()
        )

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.015,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            rotation=90,
        )

add_value_labels(
    bars_internal
)

add_value_labels(
    bars_kelm
)

# ------------------------------------------------------------
# AXES
# ------------------------------------------------------------

ax.set_xticks(
    x
)

ax.set_xticklabels(
    motif_order,
    fontweight="bold"
)

ax.set_ylabel(
    "Fold over composition-preserving null",
    fontweight="bold"
)

ax.set_xlabel(
    "Motif",
    fontweight="bold"
)

ax.set_ylim(
    0,
    max(
        internal_fold.max(),
        kelm_fold.max()
    ) + 0.55
)

# ------------------------------------------------------------
# LEGEND
# ------------------------------------------------------------

legend = ax.legend(
    frameon=False,
    loc="upper right",
    prop={
        "weight": "bold",
        "size": 9,
    },
)

# ------------------------------------------------------------
# CLEAN STYLE
# ------------------------------------------------------------

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

ax.tick_params(
    width=1.2
)

ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.6,
    alpha=0.5
)

# ------------------------------------------------------------
# PANEL LABEL
# ------------------------------------------------------------

ax.text(
    -0.11,
    1.03,
    "E",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    va="top",
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

plt.tight_layout()

plt.savefig(
    PNG_FILE,
    dpi=600,
    bbox_inches="tight",
)

plt.savefig(
    PDF_FILE,
    bbox_inches="tight",
)

plt.show()

print("\nSaved:")
print(PNG_FILE)
print(PDF_FILE)

In [ ]:
# ============================================================
# FIGURE 4E — FINAL COMPACT VERSION
# Composition-controlled motif enrichment
#
# Plot:
#   Internal vs KELM
#
# Y-axis:
#   Fold over composition-preserving null
#
# Significance:
#   *   FDR < 0.05
#   **  FDR < 0.01
#   *** FDR < 0.001
#
# Saves:
#   PNG (600 dpi)
#   PDF
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

INPUT_FILE = (
    PROJECT_DIR
    / "07_results"
    / "tables_SI"
    / "motif_composition_controlled_null.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "07_results"
    / "figures_main"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PNG_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.png"
)

PDF_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.pdf"
)


print("Input file:", INPUT_FILE)
print("Exists:", INPUT_FILE.exists())

assert INPUT_FILE.exists(), (
    f"Input file not found: {INPUT_FILE}"
)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    INPUT_FILE
)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# MOTIF ORDER
# ============================================================

motif_order = [
    "RR",
    "WK",
    "WR",
    "RW",
    "RL",
    "LR",
]


# ============================================================
# PREPARE DATA
# ============================================================

internal = (
    df[
        df["dataset"] == "internal_test"
    ]
    .set_index("motif")
    .loc[motif_order]
)

kelm = (
    df[
        df["dataset"] == "kelm_external"
    ]
    .set_index("motif")
    .loc[motif_order]
)


internal_fold = (
    internal[
        "fold_over_null"
    ]
    .to_numpy(
        dtype=float
    )
)

kelm_fold = (
    kelm[
        "fold_over_null"
    ]
    .to_numpy(
        dtype=float
    )
)

internal_fdr = (
    internal[
        "fdr"
    ]
    .to_numpy(
        dtype=float
    )
)

kelm_fdr = (
    kelm[
        "fdr"
    ]
    .to_numpy(
        dtype=float
    )
)


# ============================================================
# OPTIONAL CHECK
# ============================================================

check_table = pd.DataFrame(
    {
        "Motif": motif_order,
        "Internal_fold": internal_fold,
        "Internal_FDR": internal_fdr,
        "KELM_fold": kelm_fold,
        "KELM_FDR": kelm_fdr,
    }
)

print("\nData used for Figure 4E:")
display(
    check_table
)


# ============================================================
# GLOBAL STYLE
# ============================================================

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "font.weight": "bold",

        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "axes.linewidth": 1.2,

        "xtick.labelsize": 10,
        "ytick.labelsize": 10,

        "legend.fontsize": 9,
    }
)


# ============================================================
# CREATE FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(6.8, 4.8)
)


x = np.arange(
    len(motif_order)
)

bar_width = 0.35


# ============================================================
# BARS
# ============================================================

bars_internal = ax.bar(
    x - bar_width / 2,
    internal_fold,
    width=bar_width,
    label="Internal",
    edgecolor="black",
    linewidth=1.0,
)

bars_kelm = ax.bar(
    x + bar_width / 2,
    kelm_fold,
    width=bar_width,
    label="KELM",
    edgecolor="black",
    linewidth=1.0,
)


# ============================================================
# NULL EXPECTATION
# ============================================================

ax.axhline(
    1.0,
    color="black",
    linestyle="--",
    linewidth=1.2,
    zorder=0,
)


# ============================================================
# SIGNIFICANCE FUNCTION
# ============================================================

def get_significance_label(
    fdr
):

    if fdr < 0.001:
        return "***"

    elif fdr < 0.01:
        return "**"

    elif fdr < 0.05:
        return "*"

    else:
        return ""


# ============================================================
# ADD SIGNIFICANCE STARS
# ============================================================

for bar, fdr in zip(
    bars_internal,
    internal_fdr
):

    sig = (
        get_significance_label(
            fdr
        )
    )

    if sig:

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            bar.get_height()
            + 0.055,

            sig,

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold",
        )


for bar, fdr in zip(
    bars_kelm,
    kelm_fdr
):

    sig = (
        get_significance_label(
            fdr
        )
    )

    if sig:

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            bar.get_height()
            + 0.055,

            sig,

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold",
        )


# ============================================================
# AXES
# ============================================================

ax.set_xticks(
    x
)

ax.set_xticklabels(
    motif_order,
    fontweight="bold",
)


ax.set_xlabel(
    "Motif",
    fontweight="bold",
    fontsize=10,
)


ax.set_ylabel(
    "Fold over composition-preserving null",
    fontweight="bold",
    fontsize=10,
)


ax.set_ylim(
    0,
    2.55
)


# ============================================================
# FORCE TICK LABELS BOLD
# ============================================================

for label in ax.get_xticklabels():
    label.set_fontweight(
        "bold"
    )

for label in ax.get_yticklabels():
    label.set_fontweight(
        "bold"
    )


# ============================================================
# LEGEND
# ============================================================

legend = ax.legend(
    frameon=False,
    loc="upper right",
)

for text in legend.get_texts():
    text.set_fontweight(
        "bold"
    )


# ============================================================
# CLEAN ACS-LIKE STYLE
# ============================================================

ax.spines[
    "top"
].set_visible(
    False
)

ax.spines[
    "right"
].set_visible(
    False
)

ax.spines[
    "left"
].set_linewidth(
    1.2
)

ax.spines[
    "bottom"
].set_linewidth(
    1.2
)


ax.tick_params(
    axis="both",
    width=1.2,
    length=4,
)


# very subtle horizontal grid
ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.5,
    alpha=0.25,
)


# ============================================================
# PANEL LABEL
# ============================================================

ax.text(
    -0.105,
    1.03,
    "E",

    transform=ax.transAxes,

    fontsize=16,
    fontweight="bold",

    ha="left",
    va="top",
)


# ============================================================
# LAYOUT
# ============================================================

plt.tight_layout(
    pad=0.6
)


# ============================================================
# SAVE
# ============================================================

plt.savefig(
    PNG_FILE,
    dpi=600,
    bbox_inches="tight",
)

plt.savefig(
    PDF_FILE,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# DONE
# ============================================================

print("\nSaved:")
print(PNG_FILE)
print(PDF_FILE)

In [ ]:
# ======================================================================
# FIGURE 5 — UPDATED FINAL COMPACT MANUSCRIPT VERSION
#
# Complementary Perturbation-Based Faithfulness of Consensus CPP Hotspots
#
# A = Internal embedding-level ablation
# B = KELM embedding-level ablation
# C = Internal sequence-level alanine-scanning faithfulness
# D = KELM sequence-level alanine-scanning faithfulness
#
# A/B:
#   Consensus-hotspot embedding ablation vs matched random ablation
#
# C/D:
#   Mean sequence-level faithfulness effect
#   F = hotspot mutation drop − matched random mutation drop
#   Error bars = bootstrap 95% CI
#
# Style:
#   - compact 2 × 2
#   - all text bold
#   - no panel titles
#   - panel labels outside plotting region
#   - ACS-like clean layout
#   - 600 dpi PNG + PDF + SVG
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

FAITH_DIR = (
    PROJECT
    / "06_xai"
    / "faithfulness"
)

RESULTS = (
    PROJECT
    / "07_results"
)

OUT_DIR = (
    RESULTS
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# NEW alanine-scanning summary
MUTATION_SUMMARY_FILE = (
    RESULTS
    / "tables_SI"
    / "sequence_mutation_faithfulness"
    / "sequence_level_mutation_faithfulness_summary.csv"
)


# ======================================================================
# MODELS / DATASETS
# ======================================================================

MODELS = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_LABELS = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]


# ======================================================================
# VERIFY NEW FILE
# ======================================================================

if not MUTATION_SUMMARY_FILE.exists():

    raise FileNotFoundError(
        f"Mutation summary not found:\n"
        f"{MUTATION_SUMMARY_FILE}"
    )

print(
    "✓ Mutation summary:",
    MUTATION_SUMMARY_FILE
)


# ======================================================================
# STYLE
# matched to your latest compact Figure 5
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        10.5,

    "font.weight":
        "bold",

    "axes.labelsize":
        11.5,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        9.3,

    "ytick.labelsize":
        9.3,

    "legend.fontsize":
        8.3,

    "axes.linewidth":
        1.1,

    "xtick.major.width":
        1.0,

    "ytick.major.width":
        1.0,

    "xtick.major.size":
        4,

    "ytick.major.size":
        4,

    "savefig.dpi":
        600,
})


# ======================================================================
# HELPERS
# ======================================================================

def find_col(
    df,
    candidates,
    contains=None,
):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    if contains:

        for c in df.columns:

            lc = c.lower()

            if all(
                x.lower() in lc
                for x in contains
            ):

                return c

    return None


def bold_ticks(
    ax
):

    for tick in ax.get_xticklabels():

        tick.set_fontweight(
            "bold"
        )

    for tick in ax.get_yticklabels():

        tick.set_fontweight(
            "bold"
        )


def add_panel_letter(
    ax,
    letter
):

    ax.text(
        -0.105,
        1.035,

        letter,

        transform=ax.transAxes,

        fontsize=18,
        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,
        zorder=100,
    )


# ======================================================================
# LOAD EXISTING EMBEDDING-ABLATION DATA
# ======================================================================

def read_faithfulness(
    model,
    dataset
):

    file = (
        FAITH_DIR
        / model
        / dataset
        / "per_sequence_faithfulness.csv"
    )

    if not file.exists():

        raise FileNotFoundError(
            f"Missing faithfulness file:\n"
            f"{file}"
        )

    df = pd.read_csv(
        file
    )

    df[
        "model_plot"
    ] = model

    df[
        "dataset_plot"
    ] = dataset

    return df


frames = []

for model in MODELS:

    for dataset in DATASETS:

        frames.append(
            read_faithfulness(
                model,
                dataset
            )
        )


faith = pd.concat(
    frames,
    ignore_index=True
)


# ======================================================================
# DETECT EXISTING ABLATION COLUMNS
# ======================================================================

hotspot_drop_col = find_col(
    faith,

    [
        "hotspot_drop",
        "consensus_hotspot_drop",
        "hotspot_probability_drop",
        "consensus_drop",
        "hotspot_ablation_drop",
    ],

    contains=[
        "hotspot",
        "drop",
    ],
)


random_drop_col = find_col(
    faith,

    [
        "random_drop",
        "matched_random_drop",
        "random_probability_drop",
        "random_ablation_drop",
        "mean_random_drop",
    ],

    contains=[
        "random",
        "drop",
    ],
)


if hotspot_drop_col is None:

    raise KeyError(
        "Could not detect hotspot-drop column.\n"
        f"Columns:\n{faith.columns.tolist()}"
    )


if random_drop_col is None:

    raise KeyError(
        "Could not detect random-drop column.\n"
        f"Columns:\n{faith.columns.tolist()}"
    )


print(
    "\nExisting embedding-ablation columns:"
)

print(
    "Hotspot:",
    hotspot_drop_col
)

print(
    "Random :",
    random_drop_col
)


# ======================================================================
# LOAD NEW SEQUENCE-LEVEL MUTATION SUMMARY
# ======================================================================

mutation = pd.read_csv(
    MUTATION_SUMMARY_FILE
)


print(
    "\nMutation summary columns:"
)

print(
    mutation.columns.tolist()
)


# ensure desired model order
mutation[
    "model"
] = pd.Categorical(
    mutation[
        "model"
    ],

    categories=MODELS,

    ordered=True,
)


mutation = mutation.sort_values(
    [
        "dataset",
        "model",
    ]
)


print(
    "\nSequence-level mutation data:"
)

display(
    mutation[
        [
            "model",
            "dataset",
            "n_sequences",
            "mean_delta_hotspot",
            "mean_delta_random",
            "mean_faithfulness_effect",
            "faithfulness_CI_low",
            "faithfulness_CI_high",
            "positive_faithfulness_fraction",
            "wilcoxon_p",
        ]
    ]
)


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(11.0, 7.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.105,
    right=0.985,

    bottom=0.095,
    top=0.965,

    wspace=0.24,
    hspace=0.31,
)


axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANELS A/B — EXISTING EMBEDDING ABLATION
# ======================================================================

def perturbation_panel(
    ax,
    dataset,
):

    positions = np.arange(
        len(MODELS)
    )

    offset = 0.15

    box_width = 0.24


    for i, model in enumerate(
        MODELS
    ):

        sub = faith[
            (
                faith[
                    "model_plot"
                ]
                == model
            )
            &
            (
                faith[
                    "dataset_plot"
                ]
                == dataset
            )
        ]


        hotspot = (
            sub[
                hotspot_drop_col
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        random_control = (
            sub[
                random_drop_col
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        # --------------------------------------------------------------
        # HOTSPOT ABLATION
        # --------------------------------------------------------------

        ax.boxplot(
            hotspot,

            positions=[
                i - offset
            ],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="0.82",
                edgecolor="black",
            ),

            whiskerprops=dict(
                linewidth=0.9,
            ),

            capprops=dict(
                linewidth=0.9,
            ),
        )


        # --------------------------------------------------------------
        # RANDOM ABLATION
        # --------------------------------------------------------------

        ax.boxplot(
            random_control,

            positions=[
                i + offset
            ],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="white",
                edgecolor="black",
            ),

            whiskerprops=dict(
                linewidth=0.9,
            ),

            capprops=dict(
                linewidth=0.9,
            ),
        )


        # --------------------------------------------------------------
        # RAW POINTS
        # --------------------------------------------------------------

        rng = np.random.default_rng(
            100 + i
        )


        if len(
            hotspot
        ):

            jitter = rng.normal(
                i - offset,
                0.022,
                len(hotspot),
            )

            ax.scatter(
                jitter,
                hotspot,

                s=7,
                alpha=0.13,

                linewidths=0,

                zorder=1,
            )


        if len(
            random_control
        ):

            jitter = rng.normal(
                i + offset,
                0.022,
                len(random_control),
            )

            ax.scatter(
                jitter,
                random_control,

                s=7,
                alpha=0.13,

                linewidths=0,

                zorder=1,
            )


    # --------------------------------------------------------------
    # ZERO LINE
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",
        linewidth=1.0,

        alpha=0.75,
    )


    # --------------------------------------------------------------
    # AXES
    # --------------------------------------------------------------

    ax.set_xticks(
        positions
    )

    ax.set_xticklabels(
        MODEL_LABELS,

        rotation=17,
        ha="right",
    )


    ax.set_ylabel(
        "Decrease in predicted\nCPP probability",

        fontsize=11.5,
        fontweight="bold",

        labelpad=7,

        linespacing=1.15,
    )


    bold_ticks(
        ax
    )


    # --------------------------------------------------------------
    # LEGEND
    # --------------------------------------------------------------

    legend_handles = [

        Patch(
            facecolor="0.82",
            edgecolor="black",
            label="Consensus-hotspot ablation",
        ),

        Patch(
            facecolor="white",
            edgecolor="black",
            label="Matched random ablation",
        ),
    ]


    leg = ax.legend(
        handles=legend_handles,

        frameon=False,

        loc="upper right",

        bbox_to_anchor=(
            0.99,
            0.995,
        ),

        fontsize=8.1,

        handlelength=1.5,

        borderaxespad=0.1,
    )


    for text in leg.get_texts():

        text.set_fontweight(
            "bold"
        )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ======================================================================
# PANEL A
# ======================================================================

perturbation_panel(
    axA,
    "internal_test",
)


# ======================================================================
# PANEL B
# ======================================================================

perturbation_panel(
    axB,
    "kelm_external",
)


# ======================================================================
# COMMON Y RANGE FOR A/B
# ======================================================================

combined_values = faith[
    [
        hotspot_drop_col,
        random_drop_col,
    ]
].astype(
    float
).values.flatten()


combined_values = (
    combined_values[
        np.isfinite(
            combined_values
        )
    ]
)


if len(
    combined_values
):

    q_low = np.quantile(
        combined_values,
        0.005,
    )

    q_high = np.quantile(
        combined_values,
        0.995,
    )

    span = (
        q_high
        - q_low
    )


    ymin = min(
        -0.10,
        q_low
        - 0.10 * span,
    )

    ymax = max(
        0.20,
        q_high
        + 0.10 * span,
    )


    axA.set_ylim(
        ymin,
        ymax,
    )

    axB.set_ylim(
        ymin,
        ymax,
    )


# ======================================================================
# PANELS C/D
# SEQUENCE-LEVEL ALANINE-SCANNING FAITHFULNESS
# ======================================================================

def mutation_faithfulness_panel(
    ax,
    dataset,
):

    sub = (
        mutation[
            mutation[
                "dataset"
            ]
            == dataset
        ]
        .copy()
        .set_index(
            "model"
        )
        .loc[
            MODELS
        ]
    )


    x = np.arange(
        len(MODELS)
    )


    means = (
        sub[
            "mean_faithfulness_effect"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_low = (
        sub[
            "faithfulness_CI_low"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_high = (
        sub[
            "faithfulness_CI_high"
        ]
        .astype(float)
        .to_numpy()
    )


    lower_error = (
        means
        - ci_low
    )

    upper_error = (
        ci_high
        - means
    )


    yerr = np.vstack(
        [
            lower_error,
            upper_error,
        ]
    )


    # --------------------------------------------------------------
    # EFFECT + 95% CI
    # --------------------------------------------------------------

    ax.errorbar(
        x,
        means,

        yerr=yerr,

        fmt="o",

        markersize=7,

        markeredgecolor="black",
        markeredgewidth=0.9,

        capsize=4,
        capthick=1.2,

        elinewidth=1.3,

        linewidth=0,

        zorder=3,
    )


    # --------------------------------------------------------------
    # ZERO REFERENCE
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",

        linewidth=1.0,

        alpha=0.75,

        zorder=1,
    )


    # --------------------------------------------------------------
    # X AXIS
    # --------------------------------------------------------------

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        MODEL_LABELS,

        rotation=17,
        ha="right",
    )


    # --------------------------------------------------------------
    # Y AXIS
    # --------------------------------------------------------------

    ax.set_ylabel(
        "Sequence-level faithfulness\n"
        "(hotspot − random Δp)",

        fontsize=11.5,

        fontweight="bold",

        labelpad=7,

        linespacing=1.15,
    )


    bold_ticks(
        ax
    )


    # --------------------------------------------------------------
    # REMOVE TOP/RIGHT SPINES
    # --------------------------------------------------------------

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ======================================================================
# PANEL C — INTERNAL MUTATION
# ======================================================================

mutation_faithfulness_panel(
    axC,
    "internal_test",
)


# ======================================================================
# PANEL D — KELM MUTATION
# ======================================================================

mutation_faithfulness_panel(
    axD,
    "kelm_external",
)


# ======================================================================
# COMMON Y SCALE FOR C/D
# ======================================================================

mutation_ci_min = (
    mutation[
        "faithfulness_CI_low"
    ]
    .astype(float)
    .min()
)

mutation_ci_max = (
    mutation[
        "faithfulness_CI_high"
    ]
    .astype(float)
    .max()
)


mutation_span = (
    mutation_ci_max
    - min(
        0,
        mutation_ci_min
    )
)


mutation_ymin = min(
    -0.025,
    mutation_ci_min
    - 0.08 * mutation_span,
)

mutation_ymax = (
    mutation_ci_max
    + 0.12 * mutation_span
)


axC.set_ylim(
    mutation_ymin,
    mutation_ymax,
)

axD.set_ylim(
    mutation_ymin,
    mutation_ymax,
)


# ======================================================================
# OPTIONAL SIGNIFICANCE INDICATORS
#
# All eight are significant in your data.
# Stars are placed above CI, not directly over points.
# ======================================================================

def p_to_star(
    p
):

    if p < 0.001:
        return "***"

    elif p < 0.01:
        return "**"

    elif p < 0.05:
        return "*"

    return ""


def add_mutation_significance(
    ax,
    dataset,
):

    sub = (
        mutation[
            mutation[
                "dataset"
            ]
            == dataset
        ]
        .copy()
        .set_index(
            "model"
        )
        .loc[
            MODELS
        ]
    )


    for i, model in enumerate(
        MODELS
    ):

        p = float(
            sub.loc[
                model,
                "wilcoxon_p",
            ]
        )

        upper = float(
            sub.loc[
                model,
                "faithfulness_CI_high",
            ]
        )

        star = p_to_star(
            p
        )


        if star:

            ax.text(
                i,
                upper
                + 0.018,

                star,

                ha="center",
                va="bottom",

                fontsize=10.5,

                fontweight="bold",

                clip_on=False,
            )


add_mutation_significance(
    axC,
    "internal_test",
)

add_mutation_significance(
    axD,
    "kelm_external",
)


# ======================================================================
# PANEL LETTERS
# ======================================================================

add_panel_letter(
    axA,
    "A",
)

add_panel_letter(
    axB,
    "B",
)

add_panel_letter(
    axC,
    "C",
)

add_panel_letter(
    axD,
    "D",
)


# ======================================================================
# FINAL FORMATTING
# ======================================================================

for ax in [
    axA,
    axB,
    axC,
    axD,
]:

    ax.tick_params(
        axis="both",

        width=1.0,

        length=4,
    )

    bold_ticks(
        ax
    )


# ======================================================================
# SAVE
# ======================================================================

FIG_BASE = (
    OUT_DIR
    / "Figure5_Updated_Dual_Faithfulness"
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".png"
    ),

    dpi=600,

    bbox_inches="tight",
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".pdf"
    ),

    bbox_inches="tight",
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".svg"
    ),

    bbox_inches="tight",
)


plt.show()


print(
    "\nSaved:"
)

print(
    FIG_BASE.with_suffix(
        ".png"
    )
)

print(
    FIG_BASE.with_suffix(
        ".pdf"
    )
)

print(
    FIG_BASE.with_suffix(
        ".svg"
    )
)

In [ ]:
# ======================================================================
# FIGURE 5 — FINAL UPDATED MANUSCRIPT VERSION
#
# A = Internal embedding-level ablation
# B = KELM embedding-level ablation
# C = Internal sequence-level alanine-scanning faithfulness
# D = KELM sequence-level alanine-scanning faithfulness
#
# A/B:
#   Consensus-hotspot embedding ablation vs matched random ablation
#
# C/D:
#   Mean sequence-level faithfulness effect, F
#   Error bars = bootstrap 95% CI
#
# Style:
#   - compact 2 × 2
#   - all text bold
#   - lighter raw points in A/B
#   - reduced x-label rotation
#   - common scales within A/B and C/D
#   - 600 dpi PNG + PDF + SVG
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

FAITH_DIR = (
    PROJECT
    / "06_xai"
    / "faithfulness"
)

RESULTS = (
    PROJECT
    / "07_results"
)

OUT_DIR = (
    RESULTS
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MUTATION_SUMMARY_FILE = (
    RESULTS
    / "tables_SI"
    / "sequence_mutation_faithfulness"
    / "sequence_level_mutation_faithfulness_summary.csv"
)


# ======================================================================
# MODELS / DATASETS
# ======================================================================

MODELS = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_LABELS = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]


# ======================================================================
# VERIFY FILE
# ======================================================================

if not MUTATION_SUMMARY_FILE.exists():

    raise FileNotFoundError(
        f"Mutation summary not found:\n"
        f"{MUTATION_SUMMARY_FILE}"
    )

print(
    "✓ Mutation summary:",
    MUTATION_SUMMARY_FILE
)


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        10.5,

    "font.weight":
        "bold",

    "axes.labelsize":
        11.2,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        9.2,

    "ytick.labelsize":
        9.2,

    "legend.fontsize":
        8.2,

    "axes.linewidth":
        1.1,

    "xtick.major.width":
        1.0,

    "ytick.major.width":
        1.0,

    "xtick.major.size":
        4,

    "ytick.major.size":
        4,

    "savefig.dpi":
        600,
})


# ======================================================================
# HELPERS
# ======================================================================

def find_col(
    df,
    candidates,
    contains=None,
):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    if contains:

        for c in df.columns:

            lc = c.lower()

            if all(
                x.lower() in lc
                for x in contains
            ):

                return c

    return None


def bold_ticks(
    ax
):

    for tick in ax.get_xticklabels():

        tick.set_fontweight(
            "bold"
        )

    for tick in ax.get_yticklabels():

        tick.set_fontweight(
            "bold"
        )


def add_panel_letter(
    ax,
    letter
):

    ax.text(
        -0.105,
        1.035,

        letter,

        transform=ax.transAxes,

        fontsize=18,
        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,
        zorder=100,
    )


# ======================================================================
# LOAD EXISTING EMBEDDING-ABLATION DATA
# ======================================================================

def read_faithfulness(
    model,
    dataset
):

    file = (
        FAITH_DIR
        / model
        / dataset
        / "per_sequence_faithfulness.csv"
    )

    if not file.exists():

        raise FileNotFoundError(
            f"Missing faithfulness file:\n"
            f"{file}"
        )

    df = pd.read_csv(
        file
    )

    df[
        "model_plot"
    ] = model

    df[
        "dataset_plot"
    ] = dataset

    return df


frames = []

for model in MODELS:

    for dataset in DATASETS:

        frames.append(
            read_faithfulness(
                model,
                dataset
            )
        )


faith = pd.concat(
    frames,
    ignore_index=True
)


# ======================================================================
# DETECT EXISTING ABLATION COLUMNS
# ======================================================================

hotspot_drop_col = find_col(
    faith,

    [
        "hotspot_drop",
        "consensus_hotspot_drop",
        "hotspot_probability_drop",
        "consensus_drop",
        "hotspot_ablation_drop",
    ],

    contains=[
        "hotspot",
        "drop",
    ],
)


random_drop_col = find_col(
    faith,

    [
        "random_drop",
        "matched_random_drop",
        "random_probability_drop",
        "random_ablation_drop",
        "mean_random_drop",
    ],

    contains=[
        "random",
        "drop",
    ],
)


if hotspot_drop_col is None:

    raise KeyError(
        "Could not detect hotspot-drop column.\n"
        f"Columns:\n{faith.columns.tolist()}"
    )


if random_drop_col is None:

    raise KeyError(
        "Could not detect random-drop column.\n"
        f"Columns:\n{faith.columns.tolist()}"
    )


print(
    "\nEmbedding-level columns:"
)

print(
    "Hotspot:",
    hotspot_drop_col
)

print(
    "Random :",
    random_drop_col
)


# ======================================================================
# LOAD SEQUENCE-MUTATION SUMMARY
# ======================================================================

mutation = pd.read_csv(
    MUTATION_SUMMARY_FILE
)


mutation[
    "model"
] = pd.Categorical(
    mutation[
        "model"
    ],

    categories=MODELS,

    ordered=True,
)


mutation = mutation.sort_values(
    [
        "dataset",
        "model",
    ]
)


# ======================================================================
# FIGURE CANVAS
# ======================================================================

fig = plt.figure(
    figsize=(12.0, 8.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.105,
    right=0.985,

    bottom=0.095,
    top=0.965,

    wspace=0.24,
    hspace=0.31,
)


axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANELS A/B — EMBEDDING ABLATION
# ======================================================================

def perturbation_panel(
    ax,
    dataset,
):

    positions = np.arange(
        len(MODELS)
    )

    offset = 0.15

    box_width = 0.24


    for i, model in enumerate(
        MODELS
    ):

        sub = faith[
            (
                faith[
                    "model_plot"
                ]
                == model
            )
            &
            (
                faith[
                    "dataset_plot"
                ]
                == dataset
            )
        ]


        hotspot = (
            sub[
                hotspot_drop_col
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        random_control = (
            sub[
                random_drop_col
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        # --------------------------------------------------------------
        # HOTSPOT ABLATION BOXPLOT
        # --------------------------------------------------------------

        ax.boxplot(
            hotspot,

            positions=[
                i - offset
            ],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="0.82",
                edgecolor="black",
            ),

            whiskerprops=dict(
                linewidth=0.9,
            ),

            capprops=dict(
                linewidth=0.9,
            ),
        )


        # --------------------------------------------------------------
        # RANDOM ABLATION BOXPLOT
        # --------------------------------------------------------------

        ax.boxplot(
            random_control,

            positions=[
                i + offset
            ],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="white",
                edgecolor="black",
            ),

            whiskerprops=dict(
                linewidth=0.9,
            ),

            capprops=dict(
                linewidth=0.9,
            ),
        )


        # --------------------------------------------------------------
        # LIGHT RAW POINTS
        # --------------------------------------------------------------

        rng = np.random.default_rng(
            100 + i
        )


        if len(
            hotspot
        ):

            jitter = rng.normal(
                i - offset,
                0.020,
                len(hotspot),
            )

            ax.scatter(
                jitter,
                hotspot,

                s=5,
                alpha=0.075,

                linewidths=0,

                zorder=1,
            )


        if len(
            random_control
        ):

            jitter = rng.normal(
                i + offset,
                0.020,
                len(random_control),
            )

            ax.scatter(
                jitter,
                random_control,

                s=5,
                alpha=0.075,

                linewidths=0,

                zorder=1,
            )


    # --------------------------------------------------------------
    # ZERO REFERENCE
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",
        linewidth=1.0,

        alpha=0.75,
    )


    # --------------------------------------------------------------
    # X AXIS
    # --------------------------------------------------------------

    ax.set_xticks(
        positions
    )

    ax.set_xticklabels(
        MODEL_LABELS,

        rotation=13,
        ha="right",
    )


    # --------------------------------------------------------------
    # Y AXIS
    # --------------------------------------------------------------

    ax.set_ylabel(
        "Prediction change, Δp",

        fontsize=11.2,
        fontweight="bold",

        labelpad=7,
    )


    bold_ticks(
        ax
    )


    # --------------------------------------------------------------
    # LEGEND
    # --------------------------------------------------------------

    legend_handles = [

        Patch(
            facecolor="0.82",
            edgecolor="black",
            label="Consensus-hotspot ablation",
        ),

        Patch(
            facecolor="white",
            edgecolor="black",
            label="Matched random ablation",
        ),
    ]


    leg = ax.legend(
        handles=legend_handles,

        frameon=False,

        loc="upper right",

        bbox_to_anchor=(
            0.99,
            0.995,
        ),

        fontsize=8.0,

        handlelength=1.5,

        borderaxespad=0.1,
    )


    for text in leg.get_texts():

        text.set_fontweight(
            "bold"
        )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ======================================================================
# PANEL A
# ======================================================================

perturbation_panel(
    axA,
    "internal_test",
)


# ======================================================================
# PANEL B
# ======================================================================

perturbation_panel(
    axB,
    "kelm_external",
)


# ======================================================================
# COMMON Y RANGE FOR A/B
# ======================================================================

combined_values = faith[
    [
        hotspot_drop_col,
        random_drop_col,
    ]
].astype(
    float
).values.flatten()


combined_values = (
    combined_values[
        np.isfinite(
            combined_values
        )
    ]
)


if len(
    combined_values
):

    q_low = np.quantile(
        combined_values,
        0.005,
    )

    q_high = np.quantile(
        combined_values,
        0.995,
    )

    span = (
        q_high
        - q_low
    )


    ymin = min(
        -0.10,
        q_low
        - 0.10 * span,
    )

    ymax = max(
        0.20,
        q_high
        + 0.10 * span,
    )


    axA.set_ylim(
        ymin,
        ymax,
    )

    axB.set_ylim(
        ymin,
        ymax,
    )


# ======================================================================
# PANELS C/D — SEQUENCE-LEVEL FAITHFULNESS
# ======================================================================

def mutation_faithfulness_panel(
    ax,
    dataset,
):

    sub = (
        mutation[
            mutation[
                "dataset"
            ]
            == dataset
        ]
        .copy()
        .set_index(
            "model"
        )
        .loc[
            MODELS
        ]
    )


    x = np.arange(
        len(MODELS)
    )


    means = (
        sub[
            "mean_faithfulness_effect"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_low = (
        sub[
            "faithfulness_CI_low"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_high = (
        sub[
            "faithfulness_CI_high"
        ]
        .astype(float)
        .to_numpy()
    )


    lower_error = (
        means
        - ci_low
    )

    upper_error = (
        ci_high
        - means
    )


    yerr = np.vstack(
        [
            lower_error,
            upper_error,
        ]
    )


    # --------------------------------------------------------------
    # POINT + 95% CI
    # --------------------------------------------------------------

    ax.errorbar(
        x,
        means,

        yerr=yerr,

        fmt="o",

        markersize=7,

        markeredgecolor="black",
        markeredgewidth=0.9,

        capsize=4,
        capthick=1.2,

        elinewidth=1.3,

        linewidth=0,

        zorder=3,
    )


    # --------------------------------------------------------------
    # ZERO REFERENCE
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",

        linewidth=1.0,

        alpha=0.75,

        zorder=1,
    )


    # --------------------------------------------------------------
    # X AXIS
    # --------------------------------------------------------------

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        MODEL_LABELS,

        rotation=13,
        ha="right",
    )


    # --------------------------------------------------------------
    # Y AXIS
    # --------------------------------------------------------------

    ax.set_ylabel(
        "Sequence-level faithfulness, F",

        fontsize=11.2,

        fontweight="bold",

        labelpad=7,
    )


    bold_ticks(
        ax
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ======================================================================
# PANEL C
# ======================================================================

mutation_faithfulness_panel(
    axC,
    "internal_test",
)


# ======================================================================
# PANEL D
# ======================================================================

mutation_faithfulness_panel(
    axD,
    "kelm_external",
)


# ======================================================================
# COMMON Y RANGE FOR C/D
# ======================================================================

mutation_ci_min = (
    mutation[
        "faithfulness_CI_low"
    ]
    .astype(float)
    .min()
)

mutation_ci_max = (
    mutation[
        "faithfulness_CI_high"
    ]
    .astype(float)
    .max()
)


mutation_span = (
    mutation_ci_max
    - min(
        0,
        mutation_ci_min
    )
)


mutation_ymin = min(
    -0.025,
    mutation_ci_min
    - 0.08 * mutation_span,
)

mutation_ymax = (
    mutation_ci_max
    + 0.12 * mutation_span
)


axC.set_ylim(
    mutation_ymin,
    mutation_ymax,
)

axD.set_ylim(
    mutation_ymin,
    mutation_ymax,
)


# ======================================================================
# SIGNIFICANCE STARS FROM ACTUAL WILCOXON P VALUES
# ======================================================================

def p_to_star(
    p
):

    if p < 0.001:
        return "***"

    elif p < 0.01:
        return "**"

    elif p < 0.05:
        return "*"

    return ""


def add_mutation_significance(
    ax,
    dataset,
):

    sub = (
        mutation[
            mutation[
                "dataset"
            ]
            == dataset
        ]
        .copy()
        .set_index(
            "model"
        )
        .loc[
            MODELS
        ]
    )


    for i, model in enumerate(
        MODELS
    ):

        p = float(
            sub.loc[
                model,
                "wilcoxon_p",
            ]
        )

        upper = float(
            sub.loc[
                model,
                "faithfulness_CI_high",
            ]
        )

        star = p_to_star(
            p
        )


        if star:

            ax.text(
                i,
                upper
                + 0.015,

                star,

                ha="center",
                va="bottom",

                fontsize=10.5,

                fontweight="bold",

                clip_on=False,
            )


add_mutation_significance(
    axC,
    "internal_test",
)

add_mutation_significance(
    axD,
    "kelm_external",
)


# ======================================================================
# PANEL LETTERS
# ======================================================================

add_panel_letter(
    axA,
    "A",
)

add_panel_letter(
    axB,
    "B",
)

add_panel_letter(
    axC,
    "C",
)

add_panel_letter(
    axD,
    "D",
)


# ======================================================================
# FINAL FORMATTING
# ======================================================================

for ax in [
    axA,
    axB,
    axC,
    axD,
]:

    ax.tick_params(
        axis="both",

        width=1.0,

        length=4,
    )

    bold_ticks(
        ax
    )


# ======================================================================
# SAVE
# ======================================================================

FIG_BASE = (
    OUT_DIR
    / "Figure5_Final_Dual_Faithfulness"
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".png"
    ),

    dpi=600,

    bbox_inches="tight",
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".pdf"
    ),

    bbox_inches="tight",
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".svg"
    ),

    bbox_inches="tight",
)


plt.show()


print(
    "\nSaved:"
)

print(
    FIG_BASE.with_suffix(
        ".png"
    )
)

print(
    FIG_BASE.with_suffix(
        ".pdf"
    )
)

print(
    FIG_BASE.with_suffix(
        ".svg"
    )
)

In [ ]:
# ============================================================
# DOWNLOAD FINAL FIGURE
# ============================================================

from google.colab import files

# Download high-resolution PNG
files.download(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/figures_main/Figure5_Final_Dual_Faithfulness.png"
)

In [ ]:
# ============================================================
# FIGURE Sx
# Robustness of residue enrichment across hotspot thresholds
#
# A = Internal test
# B = KELM external
#
# Heatmap value:
#   log2(Odds Ratio)
#
# Significance:
#   *  FDR < 0.05
#
# Rows:
#   K, R, L, Q, G, M, Y, F, N
#
# Columns:
#   10%, 15%, 20%, 25%, 30%
#
# Saves:
#   PNG 600 dpi
#   PDF
#   SVG
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

INPUT_FILE = (
    PROJECT_DIR
    / "07_results"
    / "tables_SI"
    / "hotspot_threshold_robustness_all_residues.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "07_results"
    / "figures_SI"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIG_BASE = (
    OUTPUT_DIR
    / "FigureS_Threshold_Robustness_Heatmap"
)

assert INPUT_FILE.exists(), (
    f"Input file not found:\n{INPUT_FILE}"
)

print("Input:", INPUT_FILE)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    INPUT_FILE
)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# SETTINGS
# ============================================================

RESIDUE_ORDER = [
    "K",
    "R",
    "L",
    "Q",
    "G",
    "M",
    "Y",
    "F",
    "N",
]

THRESHOLD_ORDER = [
    10,
    15,
    20,
    25,
    30,
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

DATASET_LABELS = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}


# ============================================================
# CALCULATE log2(OR)
# ============================================================

df = df.copy()

df["log2_OR"] = np.log2(
    df["odds_ratio"].astype(float)
)


# ============================================================
# BUILD MATRICES
# ============================================================

def build_matrix(
    dataset_name,
    value_col,
):

    sub = df[
        df["dataset"] == dataset_name
    ].copy()

    matrix = (
        sub
        .pivot(
            index="residue",
            columns="threshold_pct",
            values=value_col,
        )
        .reindex(
            index=RESIDUE_ORDER,
            columns=THRESHOLD_ORDER,
        )
    )

    return matrix


internal_log2 = build_matrix(
    "internal_test",
    "log2_OR",
)

kelm_log2 = build_matrix(
    "kelm_external",
    "log2_OR",
)

internal_fdr = build_matrix(
    "internal_test",
    "fdr",
)

kelm_fdr = build_matrix(
    "kelm_external",
    "fdr",
)


print("\nInternal log2(OR):")
display(internal_log2)

print("\nKELM log2(OR):")
display(kelm_log2)


# ============================================================
# SHARED COLOR SCALE
# ============================================================

all_values = np.concatenate(
    [
        internal_log2.to_numpy().flatten(),
        kelm_log2.to_numpy().flatten(),
    ]
)

all_values = all_values[
    np.isfinite(all_values)
]

max_abs = np.max(
    np.abs(all_values)
)

# symmetric range around zero
vmin = -max_abs
vmax = max_abs

print(
    "\nShared color scale:",
    round(vmin, 2),
    "to",
    round(vmax, 2),
)


# ============================================================
# STYLE
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
})


# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(8.2, 4.2),
    constrained_layout=True,
)


# ============================================================
# HELPER TO DRAW ONE HEATMAP
# ============================================================

def draw_heatmap(
    ax,
    value_matrix,
    fdr_matrix,
    dataset_label,
    panel_letter,
):

    im = ax.imshow(
        value_matrix.to_numpy(),
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap="coolwarm",
        interpolation="nearest",
    )

    # --------------------------------------------------------
    # X ticks
    # --------------------------------------------------------

    ax.set_xticks(
        np.arange(
            len(THRESHOLD_ORDER)
        )
    )

    ax.set_xticklabels(
        [
            f"{x}%"
            for x in THRESHOLD_ORDER
        ],
        fontweight="bold",
    )

    # --------------------------------------------------------
    # Y ticks
    # --------------------------------------------------------

    ax.set_yticks(
        np.arange(
            len(RESIDUE_ORDER)
        )
    )

    ax.set_yticklabels(
        RESIDUE_ORDER,
        fontweight="bold",
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "Hotspot threshold",
        fontweight="bold",
    )

    ax.set_title(
        dataset_label,
        fontweight="bold",
        pad=8,
    )

    # --------------------------------------------------------
    # Cell annotations
    # --------------------------------------------------------

    values = value_matrix.to_numpy()
    fdrs = fdr_matrix.to_numpy()

    for i in range(
        values.shape[0]
    ):

        for j in range(
            values.shape[1]
        ):

            value = values[i, j]
            fdr = fdrs[i, j]

            if not np.isfinite(value):
                continue

            # text contrast
            normalized = abs(value) / max_abs

            text_color = (
                "white"
                if normalized > 0.55
                else "black"
            )

            sig = (
                "*"
                if fdr < 0.05
                else ""
            )

            ax.text(
                j,
                i,
                f"{value:.2f}{sig}",
                ha="center",
                va="center",
                fontsize=8.3,
                fontweight="bold",
                color=text_color,
            )

    # --------------------------------------------------------
    # Thin cell borders
    # --------------------------------------------------------

    ax.set_xticks(
        np.arange(
            -0.5,
            len(THRESHOLD_ORDER),
            1
        ),
        minor=True,
    )

    ax.set_yticks(
        np.arange(
            -0.5,
            len(RESIDUE_ORDER),
            1
        ),
        minor=True,
    )

    ax.grid(
        which="minor",
        linewidth=0.7,
        color="white",
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    # --------------------------------------------------------
    # Panel letter
    # --------------------------------------------------------

    ax.text(
        -0.16,
        1.06,
        panel_letter,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )

    # bold ticks
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

    return im


# ============================================================
# PANEL A
# ============================================================

im = draw_heatmap(
    axes[0],
    internal_log2,
    internal_fdr,
    "Internal test",
    "A",
)


# ============================================================
# PANEL B
# ============================================================

draw_heatmap(
    axes[1],
    kelm_log2,
    kelm_fdr,
    "KELM external",
    "B",
)


# ============================================================
# SHARED Y LABEL
# ============================================================

axes[0].set_ylabel(
    "Residue",
    fontweight="bold",
)


# ============================================================
# COLORBAR
# ============================================================

cbar = fig.colorbar(
    im,
    ax=axes,
    shrink=0.90,
    pad=0.03,
)

cbar.set_label(
    "log₂(Odds ratio)",
    fontweight="bold",
)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight(
        "bold"
    )


# ============================================================
# SAVE
# ============================================================

fig.savefig(
    FIG_BASE.with_suffix(".png"),
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    FIG_BASE.with_suffix(".pdf"),
    bbox_inches="tight",
)

fig.savefig(
    FIG_BASE.with_suffix(".svg"),
    bbox_inches="tight",
)

plt.show()


print("\nSaved:")
print(FIG_BASE.with_suffix(".png"))
print(FIG_BASE.with_suffix(".pdf"))
print(FIG_BASE.with_suffix(".svg"))

In [ ]:
# ======================================================================
# TABLE Sz
# Statistical summary of embedding- and sequence-level
# perturbation-based faithfulness analyses
#
# Part A = Embedding-level ablation
# Part B = Sequence-level alanine scanning
#
# Output:
#   Table_Sz_Perturbation_Faithfulness.xlsx
#   Table_SzA_Embedding_Ablation.csv
#   Table_SzB_Sequence_Alanine_Scanning.csv
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    wilcoxon,
    rankdata,
)


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

FAITH_DIR = (
    PROJECT
    / "06_xai"
    / "faithfulness"
)

MUTATION_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
    / "sequence_mutation_faithfulness"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MUTATION_SUMMARY_FILE = (
    MUTATION_DIR
    / "sequence_level_mutation_faithfulness_summary.csv"
)


# ======================================================================
# MODELS / DATASETS
# ======================================================================

MODELS = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_LABELS = {
    "ESM2_320":
        "ESM2-320",

    "ESM2_640":
        "ESM2-640",

    "ESM2_1280":
        "ESM2-1280",

    "ProtT5":
        "ProtT5",
}


DATASETS = [
    "internal_test",
    "kelm_external",
]

DATASET_LABELS = {
    "internal_test":
        "Internal",

    "kelm_external":
        "KELM",
}


# ======================================================================
# HELPER: FIND COLUMN
# ======================================================================

def find_col(
    df,
    candidates,
    contains=None,
):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }


    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]


    if contains:

        for c in df.columns:

            lc = c.lower()

            if all(
                item.lower() in lc
                for item in contains
            ):

                return c


    return None


# ======================================================================
# PART A
# EMBEDDING-LEVEL ABLATION
# ======================================================================

embedding_rows = []


for model in MODELS:

    for dataset in DATASETS:

        file = (
            FAITH_DIR
            / model
            / dataset
            / "per_sequence_faithfulness.csv"
        )


        assert file.exists(), (
            f"Missing file:\n{file}"
        )


        df = pd.read_csv(
            file
        )


        # --------------------------------------------------------------
        # DETECT COLUMNS
        # --------------------------------------------------------------

        hotspot_col = find_col(
            df,

            [
                "hotspot_drop",
                "consensus_hotspot_drop",
                "hotspot_probability_drop",
                "hotspot_ablation_drop",
                "delta_hotspot",
            ],

            contains=[
                "hotspot",
                "drop",
            ],
        )


        random_col = find_col(
            df,

            [
                "random_drop",
                "matched_random_drop",
                "random_probability_drop",
                "random_ablation_drop",
                "mean_random_drop",
                "delta_random",
            ],

            contains=[
                "random",
                "drop",
            ],
        )


        faith_col = find_col(
            df,

            [
                "faithfulness",
                "faithfulness_effect",
                "paired_faithfulness",
            ],
        )


        if hotspot_col is None:

            raise KeyError(
                f"Could not detect hotspot-drop column in:\n"
                f"{file}\n\n"
                f"Columns:\n{df.columns.tolist()}"
            )


        if random_col is None:

            raise KeyError(
                f"Could not detect random-drop column in:\n"
                f"{file}\n\n"
                f"Columns:\n{df.columns.tolist()}"
            )


        print(
            f"\n{model} | {dataset}"
        )

        print(
            "Hotspot column:",
            hotspot_col
        )

        print(
            "Random column :",
            random_col
        )


        # --------------------------------------------------------------
        # NUMERIC DATA
        # --------------------------------------------------------------

        hotspot_series = pd.to_numeric(
            df[
                hotspot_col
            ],
            errors="coerce",
        )


        random_series = pd.to_numeric(
            df[
                random_col
            ],
            errors="coerce",
        )


        valid = (
            hotspot_series.notna()
            &
            random_series.notna()
        )


        hotspot = (
            hotspot_series[
                valid
            ]
            .to_numpy(
                dtype=float
            )
        )


        random_control = (
            random_series[
                valid
            ]
            .to_numpy(
                dtype=float
            )
        )


        # --------------------------------------------------------------
        # FAITHFULNESS
        # --------------------------------------------------------------

        differences = (
            hotspot
            - random_control
        )


        if faith_col is not None:

            faithfulness = (
                pd.to_numeric(
                    df.loc[
                        valid,
                        faith_col,
                    ],
                    errors="coerce",
                )
                .to_numpy(
                    dtype=float
                )
            )


            # if any NaNs remain, use directly calculated difference
            if np.any(
                ~np.isfinite(
                    faithfulness
                )
            ):

                faithfulness = (
                    differences.copy()
                )

        else:

            faithfulness = (
                differences.copy()
            )


        # --------------------------------------------------------------
        # WILCOXON
        # two-sided to match manuscript methods
        # --------------------------------------------------------------

        try:

            wilcox = wilcoxon(
                hotspot,
                random_control,

                alternative="two-sided",
                zero_method="wilcox",
            )

            p_value = float(
                wilcox.pvalue
            )

        except ValueError:

            p_value = np.nan


        # --------------------------------------------------------------
        # PAIRED COHEN'S dz
        # --------------------------------------------------------------

        if len(
            differences
        ) > 1:

            diff_sd = np.std(
                differences,
                ddof=1,
            )

        else:

            diff_sd = np.nan


        if (
            np.isfinite(
                diff_sd
            )
            and diff_sd > 0
        ):

            cohens_dz = (
                np.mean(
                    differences
                )
                / diff_sd
            )

        else:

            cohens_dz = np.nan


        # --------------------------------------------------------------
        # MATCHED-PAIRS RANK-BISERIAL
        # --------------------------------------------------------------

        nonzero = differences[
            differences != 0
        ]


        if len(
            nonzero
        ) > 0:

            ranks = rankdata(
                np.abs(
                    nonzero
                )
            )


            W_pos = np.sum(
                ranks[
                    nonzero > 0
                ]
            )


            W_neg = np.sum(
                ranks[
                    nonzero < 0
                ]
            )


            denominator = (
                W_pos
                + W_neg
            )


            if denominator > 0:

                rank_biserial = (
                    W_pos
                    - W_neg
                ) / denominator

            else:

                rank_biserial = np.nan

        else:

            rank_biserial = np.nan


        # --------------------------------------------------------------
        # ROW
        # --------------------------------------------------------------

        embedding_rows.append(
            {

                "PLM":
                    MODEL_LABELS[
                        model
                    ],

                "Dataset":
                    DATASET_LABELS[
                        dataset
                    ],

                "n":
                    len(
                        differences
                    ),

                "Mean Δp hotspot":
                    float(
                        np.mean(
                            hotspot
                        )
                    ),

                "Mean Δp random":
                    float(
                        np.mean(
                            random_control
                        )
                    ),

                "Mean F":
                    float(
                        np.mean(
                            faithfulness
                        )
                    ),

                "Wilcoxon P":
                    p_value,

                "Cohen's dz":
                    cohens_dz,

                "Rank-biserial r":
                    rank_biserial,
            }
        )


table_A = pd.DataFrame(
    embedding_rows
)


# ======================================================================
# PART B
# SEQUENCE-LEVEL ALANINE SCANNING
# ======================================================================

assert MUTATION_SUMMARY_FILE.exists(), (
    f"Mutation summary not found:\n"
    f"{MUTATION_SUMMARY_FILE}"
)


mutation = pd.read_csv(
    MUTATION_SUMMARY_FILE
)


print(
    "\n"
    + "=" * 100
)

print(
    "MUTATION SUMMARY COLUMNS"
)

print(
    "=" * 100
)

print(
    mutation.columns.tolist()
)


# ======================================================================
# BUILD PART B
# ======================================================================

sequence_rows = []


for _, row in mutation.iterrows():

    model = row[
        "model"
    ]

    dataset = row[
        "dataset"
    ]


    sequence_rows.append(
        {

            "PLM":
                MODEL_LABELS.get(
                    model,
                    model,
                ),

            "Dataset":
                DATASET_LABELS.get(
                    dataset,
                    dataset,
                ),

            "n":
                int(
                    row[
                        "n_sequences"
                    ]
                ),

            "Mean Δp hotspot":
                float(
                    row[
                        "mean_delta_hotspot"
                    ]
                ),

            "Mean Δp random":
                float(
                    row[
                        "mean_delta_random"
                    ]
                ),

            "Mean F":
                float(
                    row[
                        "mean_faithfulness_effect"
                    ]
                ),

            "95% CI":
                (
                    f"{row['faithfulness_CI_low']:.3f}"
                    f"–"
                    f"{row['faithfulness_CI_high']:.3f}"
                ),

            "Positive F (%)":
                (
                    100
                    * float(
                        row[
                            "positive_faithfulness_fraction"
                        ]
                    )
                ),

            "Wilcoxon P":
                float(
                    row[
                        "wilcoxon_p"
                    ]
                ),

            # CORRECT COLUMN NAME
            "Cohen's dz":
                float(
                    row[
                        "paired_cohens_dz"
                    ]
                ),

            "Rank-biserial r":
                float(
                    row[
                        "rank_biserial"
                    ]
                ),
        }
    )


table_B = pd.DataFrame(
    sequence_rows
)


# ======================================================================
# ROW ORDER
# ======================================================================

plm_order = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
]

dataset_order = [
    "Internal",
    "KELM",
]


for table in [
    table_A,
    table_B,
]:

    table[
        "PLM"
    ] = pd.Categorical(
        table[
            "PLM"
        ],

        categories=plm_order,

        ordered=True,
    )


    table[
        "Dataset"
    ] = pd.Categorical(
        table[
            "Dataset"
        ],

        categories=dataset_order,

        ordered=True,
    )


table_A = (
    table_A
    .sort_values(
        [
            "Dataset",
            "PLM",
        ]
    )
    .reset_index(
        drop=True
    )
)


table_B = (
    table_B
    .sort_values(
        [
            "Dataset",
            "PLM",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ======================================================================
# ROUND NUMERIC VALUES
# ======================================================================

numeric_cols_A = [
    "Mean Δp hotspot",
    "Mean Δp random",
    "Mean F",
    "Cohen's dz",
    "Rank-biserial r",
]


for col in numeric_cols_A:

    table_A[
        col
    ] = (
        table_A[
            col
        ]
        .astype(float)
        .round(3)
    )


numeric_cols_B = [
    "Mean Δp hotspot",
    "Mean Δp random",
    "Mean F",
    "Positive F (%)",
    "Cohen's dz",
    "Rank-biserial r",
]


for col in numeric_cols_B:

    table_B[
        col
    ] = (
        table_B[
            col
        ]
        .astype(float)
        .round(3)
    )


# ======================================================================
# FORMAT P VALUES
# ======================================================================

def format_p(
    p
):

    if pd.isna(
        p
    ):

        return ""


    p = float(
        p
    )


    if p < 0.001:

        return (
            f"{p:.2e}"
        )


    return (
        f"{p:.3f}"
    )


table_A[
    "Wilcoxon P"
] = table_A[
    "Wilcoxon P"
].apply(
    format_p
)


table_B[
    "Wilcoxon P"
] = table_B[
    "Wilcoxon P"
].apply(
    format_p
)


# ======================================================================
# DISPLAY
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TABLE SzA — EMBEDDING-LEVEL ABLATION"
)

print(
    "=" * 110
)

display(
    table_A
)


print(
    "\n"
    + "=" * 110
)

print(
    "TABLE SzB — SEQUENCE-LEVEL ALANINE SCANNING"
)

print(
    "=" * 110
)

display(
    table_B
)


# ======================================================================
# SAVE CSV FILES
# ======================================================================

CSV_A = (
    OUT_DIR
    / "Table_SzA_Embedding_Ablation.csv"
)

CSV_B = (
    OUT_DIR
    / "Table_SzB_Sequence_Alanine_Scanning.csv"
)


table_A.to_csv(
    CSV_A,
    index=False,
)


table_B.to_csv(
    CSV_B,
    index=False,
)


# ======================================================================
# SAVE COMBINED EXCEL FILE
# ======================================================================

XLSX_FILE = (
    OUT_DIR
    / "Table_Sz_Perturbation_Faithfulness.xlsx"
)


with pd.ExcelWriter(
    XLSX_FILE,
    engine="openpyxl",
) as writer:

    table_A.to_excel(
        writer,

        sheet_name="A_Embedding_Ablation",

        index=False,
    )


    table_B.to_excel(
        writer,

        sheet_name="B_Alanine_Scanning",

        index=False,
    )


# ======================================================================
# DONE
# ======================================================================

print(
    "\nSaved:"
)

print(
    CSV_A
)

print(
    CSV_B
)

print(
    XLSX_FILE
)

In [ ]:
# ============================================================
# DOWNLOAD TABLE Sz
# ============================================================

from google.colab import files

files.download(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/tables_SI/Table_Sz_Perturbation_Faithfulness.xlsx"
)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

PROJECT = "/content/drive/MyDrive/pLM4CPP_XAI_2026"

os.chdir(PROJECT)

print("Current directory:")
print(os.getcwd())

print("\nProject folders:")
for x in sorted(os.listdir(PROJECT)):
    print(x)

In [ ]:
RESULTS = os.path.join(PROJECT, "07_results")

print("07_results exists:", os.path.exists(RESULTS))

if os.path.exists(RESULTS):
    print("\nContents:")
    for x in sorted(os.listdir(RESULTS)):
        print(x)

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

search_dirs = [
    PROJECT / "07_results" / "figures_main",
    PROJECT / "07_results" / "final_manuscript_package" / "03_main_figures",
]

for fig_num in range(2, 8):
    print("\n" + "=" * 100)
    print(f"FIGURE {fig_num}")
    print("=" * 100)

    found = []

    for d in search_dirs:
        if d.exists():
            for f in sorted(d.glob(f"Figure_{fig_num}*")):
                if f.suffix.lower() in [".png", ".pdf", ".svg", ".tif", ".tiff"]:
                    found.append(f)

    if not found:
        print("No matching files found.")
    else:
        for f in found:
            print(f)

In [ ]:
table_dirs = [
    PROJECT / "07_results" / "tables_main",
    PROJECT / "07_results" / "final_manuscript_package" / "01_main_tables",
]

print("\n" + "=" * 100)
print("MAIN TABLE FILES")
print("=" * 100)

for d in table_dirs:
    if d.exists():
        print(f"\nDirectory: {d}")

        for f in sorted(d.iterdir()):
            if f.suffix.lower() in [".csv", ".xlsx", ".xls"]:
                print(f)

In [ ]:
from pathlib import Path
from datetime import datetime

FIGDIR1 = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/figures_main")
FIGDIR2 = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/final_manuscript_package/03_main_figures")

for n in range(2, 8):

    print("\n" + "="*110)
    print(f"FIGURE {n} — PNG VERSIONS BY MODIFICATION TIME")
    print("="*110)

    files = []

    for d in [FIGDIR1, FIGDIR2]:
        files.extend(d.glob(f"Figure_{n}*.png"))

    files = sorted(
        files,
        key=lambda x: x.stat().st_mtime,
        reverse=True
    )

    for f in files:
        t = datetime.fromtimestamp(f.stat().st_mtime)
        size = f.stat().st_size / 1024

        print(
            t.strftime("%Y-%m-%d %H:%M:%S"),
            f"{size:8.1f} KB",
            f.name
        )

In [ ]:
from pathlib import Path
import shutil
import zipfile

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

FINAL_DIR = PROJECT / "07_results" / "MANUSCRIPT_FINAL_PACKAGE"
FIG_OUT = FINAL_DIR / "Figures"
TAB_OUT = FINAL_DIR / "Tables"

FIG_OUT.mkdir(parents=True, exist_ok=True)
TAB_OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# FINAL FIGURES
# ------------------------------------------------------------

figure_files = [
    # Figure 2
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_2_Model_performance_FINAL.png",
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_2_Model_performance_FINAL.pdf",

    # Figure 3A
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_3A_Representative_XAI_map.png",
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_3A_Representative_XAI_map.pdf",

    # Figure 3B-D
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_3B-D_BALANCED_2x2.png",
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_3B-D_BALANCED_2x2.pdf",

    # Figure 4
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.png",
    PROJECT / "07_results/final_manuscript_package/03_main_figures/Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.pdf",

    # Figure 5
    PROJECT / "07_results/figures_main/Figure_5_Consensus_Hotspot_Faithfulness_FINAL_OUTSIDE.png",
    PROJECT / "07_results/figures_main/Figure_5_Consensus_Hotspot_Faithfulness_FINAL_OUTSIDE.pdf",

    # Figure 6
    PROJECT / "07_results/figures_main/Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.png",
    PROJECT / "07_results/figures_main/Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.pdf",

    # Figure 7
    PROJECT / "07_results/figures_main/Figure_7_Physicochemical_and_Positional_Grammar_FINAL.png",
    PROJECT / "07_results/figures_main/Figure_7_Physicochemical_and_Positional_Grammar_FINAL.pdf",
]

# ------------------------------------------------------------
# FINAL TABLES
# ------------------------------------------------------------

TABLE_DIR = PROJECT / "07_results/final_manuscript_package/01_main_tables"

table_files = [
    TABLE_DIR / "Table_1_Datasets_and_splits.xlsx",
    TABLE_DIR / "Table_2_Predictive_performance.xlsx",
    TABLE_DIR / "Table_3A_Replicated_residue_determinants.xlsx",
    TABLE_DIR / "Table_3B_Replicated_hotspot_motifs.xlsx",
    TABLE_DIR / "Table_4A_Hotspot_faithfulness.xlsx",
    TABLE_DIR / "Table_4B_Replicated_biological_validation.xlsx",
]

# ------------------------------------------------------------
# COPY
# ------------------------------------------------------------

print("Copying figures...")

for f in figure_files:
    if f.exists():
        shutil.copy2(f, FIG_OUT / f.name)
        print("✓", f.name)
    else:
        print("MISSING:", f)

print("\nCopying tables...")

for f in table_files:
    if f.exists():
        shutil.copy2(f, TAB_OUT / f.name)
        print("✓", f.name)
    else:
        print("MISSING:", f)

# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

zip_path = PROJECT / "07_results" / "pLM4CPP_XAI_FINAL_MS_FIGURES_TABLES.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in FINAL_DIR.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(FINAL_DIR))

print("\n" + "=" * 90)
print("FINAL PACKAGE READY")
print("=" * 90)
print(zip_path)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026/07_results/pLM4CPP_XAI_FINAL_MS_FIGURES_TABLES.zip"
)